# Project 9 Phase 2 — Public code review archive (not an executable release)

**Review snapshot of `Project9_Phase2_CORRECTED_v3.ipynb`.**

- The original working notebook has **not** been modified.
- Removed **all saved execution outputs and execution counts**, including potentially sensitive logs/images.
- Redacted the embedded dataset ZIP password everywhere it appeared in cell source.
- **Every code cell is deliberately guarded** to prevent accidental execution, retraining or re-evaluation of the frozen Test partition. Code is retained for static review.
- This archive is **not** independently reproducible and is **not** a final verified software release. It excludes model binaries, the dataset and the private evidence.
- The model/evaluation claims should be taken from the approved Phase 2 report and verified original evidence, not inferred from unexecuted code cells.

**Publication status:** Source-only *review candidate*. Team approval, dependency/licence review and an independent credential/PII scan are still required before uploading to the public repository.


# AI251P Project 9 — Phase 2 | CORRECTED v3
**Waste Recognition & Instance Segmentation on Conveyor Belts — five members**

This copy was revised from the user's **executed** Colab notebook, not a newly trained model. It retains the real M4/M5 training outputs as historical evidence. **Do not use Run all.** The four classes remain `[cardboard, metal, rigid_plastic, soft_plastic]`.

**Verified from saved outputs (validation, not test):**
- M4 detector YOLO11n: trained **34 epochs**, early-stopped; best checkpoint in Drive. Validation **mAP50 ≈ 0.4832**, **mAP50–95 ≈ 0.3245**; confidence **0.25 is a provisional operating point** (selected on validation for better recall, not the maximum macro-F1).
- M5 YOLO11n-seg: training was **interrupted after epoch 26**. Best recorded validation **mask mAP50–95 ≈ 0.2888 at epoch 21**; both `best.pt` and `last.pt` existed in Drive when inspected. This is **NOT** a completed 50-epoch experiment.
- Split candidate: **1,851 train / 451 val / 929 test / 1,272 excluded** from 4,503 source images. The 1,272 are *not corrupt*: they were excluded to avoid crossing source-sequence IDs. Verify the meaning of sequence IDs and note the limitation in the report.
- **No final Test metrics, ONNX parity result, live hosting result, or open-world extension score has been verified yet.** Do not copy demonstration/example-report numbers as real results.

**Course allocation for five:** M4 12 + M5 12 + M8 11 + two different Open-world topics (5 each) = **45 points**. Due 15 Nov 2026 (11:59 PM, Riyadh).

> **Notebook order after a fresh Colab runtime:** run sections **1 → 5**, section **7** (install/log in), and **10** (recreate M5 labels) before resuming M5. Section 6/11 are visual audit checks (repeat as needed). **Skip M4 training (8)** and **skip M5 initial training (12)** since their run folders exist. For CPU-only sessions, use the checkpoint progress audit and optional M8 export; for resuming M5 you MUST have a real GPU. See the new **M5 Resume** cell immediately below the original M5 training cell.

**Data integrity:** Never publish W&B API keys, private data or the original 7GB dataset. The `WANDB_API_KEY` secret is read from Colab Secrets. Once model selection, thresholds, and acceptance criteria are fixed, execute the guarded final Test cell *once*.


## 1. Mount Google Drive and define paths

Colab `/content` files disappear when the runtime resets. This cell must be run in a live Colab session. Store the experiment configs/checkpoints in Google Drive. A copy of the 7GB ZIP in Drive is **optional** and requires adequate Drive storage.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from pathlib import Path
from google.colab import drive
import torch, shutil

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/Project9_Phase2')
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
RUNS = DRIVE_PROJECT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
M4_RUN_NAME = 'M4_yolo11n_640_grouped_v2_seed42'
M5_RUN_NAME = 'M5_yolo11n_seg_640_grouped_v2_seed42'
M4_BEST = RUNS / M4_RUN_NAME / 'weights/best.pt'
M5_BEST = RUNS / M5_RUN_NAME / 'weights/best.pt'
M5_LAST = RUNS / M5_RUN_NAME / 'weights/last.pt'
WORK = Path('/content/Project9_Phase2')
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = Path('/content/ZeroWaste/splits_final_deblurred')
ARCHIVE = Path('/content/zerowaste-f-final.zip')  # corrected spelling
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available')
print('Source ready:', SOURCE.is_dir())
print('ZIP present locally:', ARCHIVE.is_file())
print('Google Drive project:', DRIVE_PROJECT)
print('Local free GB:', round(shutil.disk_usage('/content').free / (1024**3), 1))
print('Existing M4 checkpoint:', M4_BEST.is_file())
print('Existing M5 checkpoint:', M5_BEST.is_file(), '| resume:', M5_LAST.is_file())


## 2. Restore the original source files (only if missing)

If the runtime restarted and the ZIP is absent, re-download from the official Zenodo record. `wget -c` resumes partial downloads. We **verify MD5 before extracting**. The public dataset password is taken from its official record; don't publish credentials for any private dataset.

**NOTE:** The original notebook's final check erroneously looked for `/content/zerowaste-f.zip`; the actual name was `/content/zerowaste-f-final.zip`. More importantly, *all* `/content` files had disappeared in that runtime.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import subprocess, hashlib

if not all((SOURCE / s / 'labels.json').is_file() for s in ('train', 'val', 'test')):
    if not ARCHIVE.is_file():
        cached = DRIVE_PROJECT / 'cache' / ARCHIVE.name
        if cached.is_file():
            print('Using ZIP cached on Google Drive:', cached)
            ARCHIVE = cached
        else:
            print('No local/Drive copy found. Downloading public ZeroWaste-f dataset.')
            url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
            subprocess.run(['wget', '-c', '--tries=5', '--timeout=60',
                            '-O', str(ARCHIVE), url], check=True)
    # Resume an incomplete local ZIP instead of failing checksum immediately.
    expected_size = 7518242799
    if ARCHIVE.is_file() and ARCHIVE.stat().st_size < expected_size and ARCHIVE.parent == Path('/content'):
        url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
        subprocess.run(['wget', '-c', '--tries=5', '--timeout=60',
                        '-O', str(ARCHIVE), url], check=True)
    md5 = hashlib.md5()
    with ARCHIVE.open('rb') as fh:
        for part in iter(lambda: fh.read(8 * 1024 * 1024), b''):
            md5.update(part)
    expected_md5 = 'e26e31a58080bca6782dca0e56074c5d'
    print('MD5:', md5.hexdigest())
    assert md5.hexdigest() == expected_md5, 'ZIP integrity mismatch. Stop; do not extract.'
    Path('/content/ZeroWaste').mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', '-P', 'REDACTED_DATASET_ZIP_PASSWORD',
                    str(ARCHIVE), '-d', '/content/ZeroWaste'], check=True)

assert all((SOURCE / s / 'labels.json').is_file() for s in ('train', 'val', 'test'))
print('SOURCE RESTORED:', SOURCE)
for s in ('train', 'val', 'test'):
    print(s, len(list((SOURCE / s / 'data').glob('*.PNG'))), 'images')

## 3. Build the **missing** `candidate` table from COCO metadata

The uploaded original notebook used `candidate.copy()` in its later split-planning cell without any saved cell defining `candidate`. This self-contained cell reconstructs it using the original JSON and image filenames, including `-2` suffixes. Splits and class names come from the files; no guesses or results are invented.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import json, re
import pandas as pd
from collections import Counter, defaultdict

CLASS_NAMES = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
SPLITS = ['train', 'val', 'test']
rows = []
category_lookup = {}

for original_split in SPLITS:
    labels_path = SOURCE / original_split / 'labels.json'
    coco = json.loads(labels_path.read_text(encoding='utf-8-sig'))
    categories = {int(c['id']): c['name'] for c in coco['categories']}
    assert set(categories.values()) == set(CLASS_NAMES), categories
    category_lookup[original_split] = categories
    image_ids = {int(im['id']): im for im in coco['images']}
    annotation_counts = defaultdict(Counter)
    for ann in coco['annotations']:
        image_id = int(ann['image_id'])
        assert image_id in image_ids
        annotation_counts[image_id][categories[int(ann['category_id'])]] += 1

    for image_id, image_info in image_ids.items():
        filename = Path(image_info['file_name']).name
        match = re.fullmatch(r'(.+)_frame_(\d+)(?:-\d+)?', Path(filename).stem, re.I)
        if not match:
            raise ValueError(f'Unparsed filename: {filename}')
        source_path = SOURCE / original_split / 'data' / filename
        if not source_path.is_file():
            raise FileNotFoundError(source_path)
        row = dict(original_split=original_split, image_id=image_id,
                   filename=filename, sequence=match.group(1).zfill(2),
                   frame_number=int(match.group(2)),
                   source_path=str(source_path), annotation_path=str(labels_path))
        for name in CLASS_NAMES:
            row[name] = annotation_counts[image_id][name]
        rows.append(row)

candidate = pd.DataFrame(rows)
assert len(candidate) == 4503, 'Unexpected dataset version/count'
assert not candidate.duplicated(['original_split', 'filename']).any()
print('Source files:', candidate.groupby('original_split').size().to_dict())
print('Unusual names handled:', candidate['filename'].str.contains('-2.PNG', regex=False).sum())
print('Source sequences:', candidate.groupby('original_split')['sequence'].nunique().to_dict())
print('COCO objects:', int(candidate[CLASS_NAMES].sum().sum()))

## 4. Create a sequence-disjoint **candidate** split

This reproduces the conservative logic in the original notebook: keep **all original test images unchanged**; remove test-sequence IDs from training/validation; choose validation sequences only from the remaining non-test pool using class balance. The original run selected sequences `07` and `11`, yielding train 1,851, val 451, test 929, excluded 1,272. Recompute from metadata, do not hardcode those numbers as outcomes.

**Method caveat:** Filename prefix is a *proxy* for a source video/sequence. Verify its meaning in the dataset documentation; prefix-disjoint groups reduce temporal leakage but do not mathematically rule out visual duplicates between distinct prefixes. Do not use test metrics to select a model.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from itertools import combinations
import hashlib

revised = candidate.copy()
original_test = revised['original_split'].eq('test')
test_sources = set(revised.loc[original_test, 'sequence'])
pool_mask = ~original_test & ~revised['sequence'].isin(test_sources)
pool = revised.loc[pool_mask]
by_source = pool.groupby('sequence')[CLASS_NAMES].sum()
by_source.insert(0, 'images', pool.groupby('sequence').size())
print('=== AVAILABLE NON-TEST SOURCE GROUPS ===')
print(by_source.to_string())

totals = by_source[CLASS_NAMES].sum()
assert (totals > 0).all(), 'Source pool is missing a class'
options = []
for n_sources in (2, 3):
    for group in combinations(sorted(by_source.index), n_sources):
        val_c = by_source.loc[list(group), CLASS_NAMES].sum()
        train_c = totals - val_c
        if not ((val_c > 0) & (train_c > 0)).all():
            continue
        val_n = int(by_source.loc[list(group), 'images'].sum())
        score = abs(val_n / len(pool) - 0.20) + float((val_c / totals - 0.20).abs().mean())
        options.append((score, group))
if not options:
    raise ValueError('No valid grouped split; inspect source distribution')
_, chosen_val_sources = min(options)
revised['split'] = 'excluded'
revised.loc[pool_mask, 'split'] = 'train'
revised.loc[pool_mask & revised['sequence'].isin(chosen_val_sources), 'split'] = 'val'
revised.loc[original_test, 'split'] = 'test'

sequence_sets = {s: set(revised.loc[revised['split'].eq(s), 'sequence']) for s in SPLITS}
assert all(not sequence_sets[a].intersection(sequence_sets[b]) for a,b in combinations(SPLITS,2))
assert set(revised.loc[original_test,'source_path']) == set(revised.loc[revised.split.eq('test'),'source_path'])
assert (revised.loc[revised.split.isin(SPLITS)].groupby('split')[CLASS_NAMES].sum() > 0).all().all()

manifest = WORK / 'audits' / 'phase2_grouped_split.csv'
manifest.parent.mkdir(parents=True, exist_ok=True)
revised.to_csv(manifest, index=False)
sha = hashlib.sha256(manifest.read_bytes()).hexdigest()
protocol_dir = DRIVE_PROJECT / 'data_protocol' / sha[:12]
protocol_dir.mkdir(parents=True, exist_ok=True)
(protocol_dir / 'split_manifest.csv').write_bytes(manifest.read_bytes())
print('Validation groups:', chosen_val_sources)
print(revised.groupby('split').size().to_string())
print(revised.loc[revised.split.isin(SPLITS)].groupby('split')[CLASS_NAMES].sum().to_string())
print('SHA256 manifest:', sha)
print('Manifest saved to Drive:', protocol_dir)
print('No source image moved or removed')

## 5. Convert grouped COCO boxes into YOLO detection labels

Class order is now **0 cardboard, 1 metal, 2 rigid_plastic, 3 soft_plastic**, matching the grouped dataset made in the original notebook (but *not* the earlier ungrouped notebook). Do not mix checkpoints/labels across these class mappings.

Uses **hard links where possible** (falling back to copying) so we avoid duplicating GBs in Colab. Writes to a temporary directory and commits only when complete. Reports clipped boxes explicitly.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import os, math, shutil, tempfile
from PIL import Image
from pathlib import PurePosixPath

det_dir = WORK / 'zerowaste_yolo_grouped_v2'
active = revised.loc[revised.split.isin(SPLITS)].copy()
active['export_name'] = active['original_split'] + '__' + active['filename']
assert not active.duplicated(['split', 'export_name']).any()

def create_yolo_detection():
    expected_manifest_hash = hashlib.sha256(manifest.read_bytes()).hexdigest()
    if det_dir.is_dir():
        info = json.loads((det_dir / 'build_info.json').read_text())
        assert info['manifest_sha256'] == expected_manifest_hash
        assert info['class_names'] == CLASS_NAMES
    else:
        tmp = Path(tempfile.mkdtemp(prefix='.detection_build_', dir=WORK))
        stats = {s: dict(images=0, boxes=0, clipped=0, background=0,
                         **{c: 0 for c in CLASS_NAMES}) for s in SPLITS}
        cached_annotations = {}
        for s in SPLITS:
            (tmp / 'images' / s).mkdir(parents=True)
            (tmp / 'labels' / s).mkdir(parents=True)
        for n, row in enumerate(active.itertuples(index=False), 1):
            if row.annotation_path not in cached_annotations:
                coco = json.loads(Path(row.annotation_path).read_text(encoding='utf-8-sig'))
                images_by_id = {int(i['id']): i for i in coco['images']}
                classes_by_id = {int(c['id']): c['name'] for c in coco['categories']}
                anns = defaultdict(list)
                for a in coco['annotations']:
                    anns[int(a['image_id'])].append(a)
                cached_annotations[row.annotation_path] = images_by_id, classes_by_id, anns
            images_by_id, classes_by_id, anns = cached_annotations[row.annotation_path]
            source = Path(row.source_path)
            W, H = (int(images_by_id[row.image_id][k]) for k in ('width','height'))
            with Image.open(source) as im:
                assert im.size == (W,H)
            lines = []
            for ann in anns[row.image_id]:
                class_name = classes_by_id[int(ann['category_id'])]
                cls = CLASS_NAMES.index(class_name)
                x, y, w, h = map(float, ann['bbox'])
                assert all(map(math.isfinite, (x,y,w,h))) and w > 0 and h > 0
                x1,y1,x2,y2 = (max(0,min(W,x)),max(0,min(H,y)),
                               max(0,min(W,x+w)),max(0,min(H,y+h)))
                assert x2 > x1 and y2 > y1
                stats[row.split]['clipped'] += int((x1,y1,x2,y2) != (x,y,x+w,y+h))
                xc,yc,bw,bh = (x1+x2)/(2*W),(y1+y2)/(2*H),(x2-x1)/W,(y2-y1)/H
                lines.append(f'{cls} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}')
                stats[row.split]['boxes'] += 1
                stats[row.split][class_name] += 1
            dest_img = tmp / 'images' / row.split / row.export_name
            try:
                os.link(source, dest_img)
            except OSError:
                shutil.copy2(source, dest_img)
            (tmp / 'labels' / row.split / (Path(row.export_name).stem+'.txt')).write_text(
                '\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
            stats[row.split]['images'] += 1
            stats[row.split]['background'] += int(not lines)
            if n % 500 == 0: print('Prepared', n, '/', len(active))
        for s in SPLITS:
            expected = active.loc[active.split.eq(s),CLASS_NAMES].sum()
            assert all(stats[s][name] == int(expected[name]) for name in CLASS_NAMES)
        yaml_text = (f'path: {det_dir}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n'
                     + ''.join(f'  {i}: {name}\n' for i,name in enumerate(CLASS_NAMES)))
        (tmp / 'data.yaml').write_text(yaml_text)
        (tmp / 'split_manifest.csv').write_bytes(manifest.read_bytes())
        info = dict(manifest_sha256=expected_manifest_hash, class_names=CLASS_NAMES,
                    stats=stats, image_method='hardlink or copy', source_dataset='ZeroWaste-f')
        (tmp / 'build_info.json').write_text(json.dumps(info,indent=2))
        tmp.rename(det_dir)
    for s in SPLITS:
        expected = set(active.loc[active.split.eq(s),'export_name'])
        assert expected == {p.name for p in (det_dir/'images'/s).iterdir()}
        assert {Path(v).stem+'.txt' for v in expected} == {p.name for p in (det_dir/'labels'/s).iterdir()}
    info = json.loads((det_dir / 'build_info.json').read_text())
    print(pd.DataFrame(info['stats']).T.to_string())
    print('Detection data YAML:', det_dir / 'data.yaml')

create_yolo_detection()
shutil.copy2(det_dir/'build_info.json', protocol_dir/'det_build_info.json')

## 6. Visual label verification and the split check

The label inspection below is independent of the earlier notebook and uses the **final** grouped dataset. A sequence-disjoint split is guaranteed by the manifest. Exact/near-duplicate visual checks between distinct prefixes should additionally be recorded in the report; this cell does not claim to prove there are no near-duplicates.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, cls in zip(axes.flat, range(4)):
    chosen = None
    for path in sorted((det_dir / 'labels/train').glob('*.txt')):
        lines = path.read_text().splitlines()
        if any(line.startswith(str(cls)+' ') for line in lines):
            chosen = path
            break
    assert chosen, CLASS_NAMES[cls]
    image_path = det_dir / 'images/train' / (chosen.stem+'.PNG')
    with Image.open(image_path) as im:
        img = im.convert('RGB').copy()
    W,H = img.size
    ax.imshow(img)
    for line in chosen.read_text().splitlines():
        c,xc,yc,bw,bh = map(float,line.split())
        x,y = (xc-bw/2)*W,(yc-bh/2)*H
        ax.add_patch(Rectangle((x,y),bw*W,bh*H, fill=False,edgecolor='red',linewidth=1.5))
        ax.text(x,y, CLASS_NAMES[int(c)], fontsize=7, color='yellow',
                bbox={'facecolor':'black','alpha':.6})
    ax.set_title(f'{CLASS_NAMES[cls]} — {chosen.stem}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(protocol_dir / 'm4_grouped_visual_boxes.png', dpi=130)
plt.show()
print('Saved:', protocol_dir / 'm4_grouped_visual_boxes.png')

## 7. Install training libraries

Run only after the source dataset and YAML are ready. The exact package versions used for real experiments are saved in the next training cell; current installs may differ across Colab sessions.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

%pip -q install ultralytics wandb onnx onnxruntime fastapi uvicorn python-multipart httpx

import ultralytics, wandb
print('Ultralytics:', ultralytics.__version__)
print('W&B:', wandb.__version__)

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


import os
import wandb
from google.colab import userdata

api_key = userdata.get("WANDB_API_KEY")

assert api_key and len(api_key) > 20, "Check your W&B secret"

os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

connected = wandb.login(
    key=api_key,
    host="https://api.forge.coreweave.com",
    relogin=True
)

print("W&B connection:", "SUCCESS" if connected else "FAILED")


## 8. M4 training — **COMPLETED; DON'T RERUN**
The historical output below is the **real run** in your uploaded notebook: 34 epochs, best checkpoint and W&B run previously created. This cell is preserved for traceability and will **refuse to overwrite** an existing Drive run. Use sections 9 onward or the checkpoint. Real W&B history: `https://app.forge.coreweave.com/sultan--almutairi/AI251P-Waste-Phase2/runs/epqdtowi`.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from ultralytics import YOLO, settings
from importlib.metadata import version
from datetime import datetime, timezone

assert torch.cuda.is_available(), 'Enable GPU in Runtime settings'
assert (det_dir/'data.yaml').is_file()
RUNS = DRIVE_PROJECT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
RUN_NAME = 'M4_yolo11n_640_grouped_v2_seed42'
RUN_DIR = RUNS / RUN_NAME
if RUN_DIR.exists():
    raise RuntimeError(f'RUN EXISTS: {RUN_DIR}. Use the resume cell below, do not overwrite.')

versions = {k:version(k) for k in ['ultralytics','wandb','torch','torchvision','numpy','pandas','Pillow','PyYAML']}
train_args = dict(data=str(det_dir/'data.yaml'), epochs=50, patience=15,
                  imgsz=640, batch=8, device=0, workers=2, cache=False,
                  optimizer='AdamW', lr0=0.001, seed=42, deterministic=True,
                  amp=True, close_mosaic=10, save=True, save_period=5,
                  project=str(RUNS), name=RUN_NAME, exist_ok=False, plots=True)
configuration = dict(task='M4 detection', model='yolo11n.pt', run_name=RUN_NAME,
                     grouped_split_manifest_sha256=sha, train_args=train_args,
                     versions=versions, date_utc=datetime.now(timezone.utc).isoformat())
settings.update({'wandb': True})
assert wandb.login(), 'W&B login is required for live logging'
with wandb.init(project='AI251P-Waste-Phase2', name=RUN_NAME, config=configuration) as run:
    print('Live W&B URL:', run.url)
    model = YOLO('yolo11n.pt')
    model.train(**train_args)
    run.summary['manifest_sha256'] = sha
assert (RUN_DIR/'weights/best.pt').is_file(), 'No best checkpoint produced'
(RUN_DIR/'experiment.json').write_text(json.dumps(configuration,indent=2))
print('Best detection model:', RUN_DIR/'weights/best.pt')

### Resume M4 after an interruption (do **not** rerun if finished)

Rebuild the source and grouped dataset paths first (cells 1–6), then execute this cell only when `last.pt` exists and training was interrupted.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# MANUAL RESUME — execute only when needed
RESUME_M4 = False
if RESUME_M4:
    from ultralytics import YOLO
    checkpoint = RUNS / 'M4_yolo11n_640_grouped_v2_seed42' / 'weights/last.pt'
    assert checkpoint.is_file()
    YOLO(str(checkpoint)).train(resume=True)

## 9. M4 validation and operating point (validation only)
**The earlier per-threshold `model.val(..., conf=t)` table was misleading.** Ultralytics `metric.box.p` and `.r` are calculated near the F1-optimal operating point, not necessarily at the `conf` you passed. The corrected sweep below interpolates the precision-confidence and recall-confidence curves at each threshold. Report it as **macro per-class curve estimates** (not pooled, image-weighted counts). Keep low-confidence `mAP` separate.

**Provisional operating point:** `confidence=0.25`, `NMS IoU=0.5`, chosen to favor recall over marginally higher macro-F1 at 0.40. This needs to be stated honestly; at 0.25, estimated macro recall is only ~0.526. Do not claim the system meets a safety-critical conveyor acceptance criterion.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from ultralytics import YOLO
import numpy as np
import pandas as pd

m4_best = RUNS / M4_RUN_NAME / 'weights/best.pt'
assert m4_best.is_file(), 'M4 checkpoint missing from Drive'
assert (det_dir / 'data.yaml').is_file(), 'Restore sections 1–5 first'
m4 = YOLO(str(m4_best))
m4_eval = DRIVE_PROJECT / 'evaluation' / 'M4'
m4_eval.mkdir(parents=True, exist_ok=True)
base = m4.val(data=str(det_dir/'data.yaml'), split='val', imgsz=640,
              conf=0.001, iou=0.5, plots=True,
              project=str(m4_eval), name='val_map_repro', exist_ok=True)
rows = [dict(class_name=base.names[int(i)], ap50=float(base.box.ap50[j]),
             ap50_95=float(base.box.ap[j]))
        for j, i in enumerate(base.box.ap_class_index)]
per_class = pd.DataFrame(rows)
per_class.to_csv(m4_eval/'per_class_ap_val.csv', index=False)
print('VALIDATION mAP50:', float(base.box.map50))
print('VALIDATION mAP50-95:', float(base.box.map))
print('Per-class AP (VAL):')
print(per_class.round(4).to_string(index=False))


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import numpy as np
import pandas as pd
import json

assert 'base' in globals(), 'Run M4 validation cell above first'
x = np.asarray(base.box.px)
p_curves = np.asarray(base.box.p_curve)
r_curves = np.asarray(base.box.r_curve)
assert p_curves.shape == r_curves.shape and p_curves.shape[1] == len(x)
thresholds = [0.05, 0.10, 0.20, 0.25, 0.40, 0.50]
rows = []
for t in thresholds:
    p = np.array([np.interp(t, x, curve) for curve in p_curves])
    r = np.array([np.interp(t, x, curve) for curve in r_curves])
    f1 = 2*p*r / np.maximum(p+r, 1e-12)
    rows.append(dict(confidence=t, macro_precision=float(p.mean()),
                     macro_recall=float(r.mean()), macro_f1=float(f1.mean())))

sweep_df = pd.DataFrame(rows)
sweep_file = m4_eval / 'confidence_sweep_corrected.csv'
sweep_df.to_csv(sweep_file, index=False)
M4_CONF, M4_IOU = 0.25, 0.50
operating_point = dict(split='val', confidence=M4_CONF, nms_iou=M4_IOU,
    source='interpolated Ultralytics P/R curves at IoU 0.5',
    criterion='preliminary recall-prioritized threshold, not max macro F1',
    limitation='macro per-class estimates; not pooled counts')
(m4_eval/'operating_point_provisional.json').write_text(json.dumps(operating_point,indent=2))
print(sweep_df.round(4).to_string(index=False))
print('Saved corrected sweep to:', sweep_file)


## 10. M5 — Build instance segmentation labels from COCO polygons

We use the **same grouped manifest** as M4, but produce a separate YOLO segmentation dataset. COCO `segmentation` polygons are not the same as the grayscale `sem_seg` visualization. Multiple polygons from one object are joined with Ultralytics' utility; **do not invent masks from boxes**. This cell explicitly stops if an annotation has no valid polygon, rather than silently calling it background.

Validate several polygons visually before training, and note any conversion artifacts or joined components.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from ultralytics.data.converter import merge_multi_segment
import numpy as np

seg_dir = WORK / 'zerowaste_seg_grouped_v2'
if not seg_dir.is_dir():
    tmp = Path(tempfile.mkdtemp(prefix='.segmentation_build_',dir=WORK))
    for s in SPLITS:
        (tmp/'images'/s).mkdir(parents=True)
        (tmp/'labels'/s).mkdir(parents=True)
    cache = {}
    multi_count = 0
    missing_count = 0
    for index,row in enumerate(active.itertuples(index=False),1):
        if row.annotation_path not in cache:
            coco = json.loads(Path(row.annotation_path).read_text(encoding='utf-8-sig'))
            lookup = {int(i['id']):i for i in coco['images']}
            categories = {int(c['id']):c['name'] for c in coco['categories']}
            all_annotations = defaultdict(list)
            for ann in coco['annotations']:
                all_annotations[int(ann['image_id'])].append(ann)
            cache[row.annotation_path] = (lookup,categories,all_annotations)
        images_by_id,categories,all_annotations = cache[row.annotation_path]
        info = images_by_id[row.image_id]
        W,H = int(info['width']),int(info['height'])
        output = []
        for ann in all_annotations[row.image_id]:
            c = CLASS_NAMES.index(categories[int(ann['category_id'])])
            raw = ann.get('segmentation',[])
            if not isinstance(raw,list):
                raise ValueError(f'RLE annotation requires manual conversion: {ann["id"]}')
            polygons = [p for p in raw if len(p)>=6 and len(p)%2==0
                        and all(math.isfinite(float(v)) for v in p)]
            if not polygons:
                missing_count += 1
                raise ValueError(f'Missing instance polygon: ann={ann["id"]}, image={row.filename}')
            if len(polygons)>1:
                multi_count += 1
                points = np.concatenate(merge_multi_segment(polygons),axis=0).astype(float)
            else:
                points = np.array(polygons[0],dtype=float).reshape(-1,2)
            points[:,0] = np.clip(points[:,0]/W,0,1)
            points[:,1] = np.clip(points[:,1]/H,0,1)
            if len(points)<3:
                raise ValueError(f'Unusable polygon for {ann["id"]}')
            output.append(str(c)+' '+' '.join(f'{v:.8f}' for v in points.reshape(-1)))
        dest_img = tmp/'images'/row.split/row.export_name
        try:
            os.link(Path(row.source_path),dest_img)
        except OSError:
            shutil.copy2(row.source_path,dest_img)
        (tmp/'labels'/row.split/(Path(row.export_name).stem+'.txt')).write_text(
            '\n'.join(output)+('\n' if output else ''),encoding='utf-8')
        if index%500==0: print('Converted segment images:', index,'/',len(active))
    yaml_text = (f'path: {seg_dir}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n'
                 + ''.join(f'  {i}: {name}\n' for i,name in enumerate(CLASS_NAMES)))
    (tmp/'data.yaml').write_text(yaml_text)
    (tmp/'split_manifest.csv').write_bytes(manifest.read_bytes())
    (tmp/'build_info.json').write_text(json.dumps(dict(manifest_sha256=sha,
         multi_polygon_annotations=multi_count, missing_polygon_annotations=missing_count,
         classes=CLASS_NAMES),indent=2))
    tmp.rename(seg_dir)
else:
    info = json.loads((seg_dir/'build_info.json').read_text())
    assert info['manifest_sha256']==sha
print('Segmentation dataset:',seg_dir)
print('Conversion log:',(seg_dir/'build_info.json').read_text())

## 11. M5 polygon visual check

Before training, inspect at least four images from the training split and confirm the colored polygons align with the objects.

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from matplotlib.patches import Polygon

# Prefer a different source sequence for each category, instead of four adjacent video frames.
paths = sorted((seg_dir/'labels/train').glob('*.txt'))
selected = []
used_sequences = set()
for cls in range(len(CLASS_NAMES)):
    choices = []
    for label in paths:
        lines = label.read_text().splitlines()
        if any(line.startswith(f'{cls} ') for line in lines):
            match = re.search(r'_frame_', label.stem)
            seq = label.stem[:match.start()] if match else label.stem
            if label not in selected:
                choices.append((seq in used_sequences, label, seq))
    assert choices, f'No sample contains class {CLASS_NAMES[cls]}'
    _, path, seq = sorted(choices, key=lambda t:(t[0],str(t[1])))[0]
    selected.append(path); used_sequences.add(seq)
fig,axes=plt.subplots(2,2,figsize=(14,9))
for ax, label, class_name in zip(axes.flat,selected,CLASS_NAMES):
    image_path = seg_dir/'images/train'/(label.stem+'.PNG')
    with Image.open(image_path) as image:
        W,H = image.size
        ax.imshow(image.convert('RGB'))
    for line in label.read_text().splitlines():
        parts=line.split()
        pts=np.array(list(map(float,parts[1:]))).reshape(-1,2)*[W,H]
        ax.add_patch(Polygon(pts,closed=True,alpha=0.25,edgecolor='red',facecolor='lime'))
    ax.set_title(f'{class_name}: {label.stem}')
    ax.axis('off')
plt.tight_layout()
plt.savefig(protocol_dir/'m5_grouped_visual_polygons_diverse.png',dpi=130)
plt.show()
print('Inspected distinct training frames:', [p.stem for p in selected])


### M5 readiness (run before **new** training or resume)
Use your Colab Secret `WANDB_API_KEY`. The existing run was already started; do **not** repeat the original initial-training cell. If GPU is unavailable, use the CPU-safe progress summary below or M8 export and return to resume after GPU access is restored.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


import os
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

import torch
import wandb
from google.colab import userdata

key = userdata.get("WANDB_API_KEY")
assert key, "W&B key missing from Secrets"

connected = wandb.login(
    key=key,
    host="https://api.forge.coreweave.com",
    relogin=True
)

print("GPU:", torch.cuda.is_available())
print("Segmentation dataset:", (seg_dir / "data.yaml").is_file())
print("W&B connected:", connected)
print("Drive runs folder:", RUNS.is_dir())


## 12. M5 initial training — **INTERRUPTED; DON'T RERUN FROM SCRATCH**
The saved historical output shows it reached epoch **26 of 50**, and model weights/CSV were present on Drive. Keep this cell as training provenance; since the run folder exists, it will refuse to start over. To **resume**, use the dedicated guarded cell below.

W&B run from the actual training output: `https://app.forge.coreweave.com/sultan--almutairi/AI251P-Waste-Phase2/runs/c6kv4slq`.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from ultralytics import YOLO, settings
settings.update({'wandb':True})
seg_run_name='M5_yolo11n_seg_640_grouped_v2_seed42'
seg_run_dir=RUNS/seg_run_name
if seg_run_dir.exists():
    raise RuntimeError(f'Existing segmentation run: {seg_run_dir}. Resume or use new name.')
assert wandb.login()
seg_train_args=dict(data=str(seg_dir/'data.yaml'),epochs=50,patience=15,
                    imgsz=640,batch=8,device=0,workers=2,seed=42,
                    amp=True,optimizer='AdamW',lr0=0.001,
                    save_period=5,plots=True,project=str(RUNS),
                    name=seg_run_name,exist_ok=False)
seg_config=dict(task='M5 instance segmentation',model='yolo11n-seg.pt',
                manifest_sha256=sha,train_args=seg_train_args)
with wandb.init(project='AI251P-Waste-Phase2',name=seg_run_name,config=seg_config) as run:
    print('Live W&B:',run.url)
    YOLO('yolo11n-seg.pt').train(**seg_train_args)
assert (seg_run_dir/'weights/best.pt').is_file()
(seg_run_dir/'experiment.json').write_text(json.dumps(seg_config,indent=2))
print('Best segmentation model:',seg_run_dir/'weights/best.pt')

### M5 checkpoints and learning curve — **CPU-safe**
This cell works even when Colab reports `GPU available: False`; it reads the real CSV from Drive. The PNG is evidence of the *partial* 26-epoch training only. If Drive is not mounted, run section 1 first. It cannot resume training without GPU.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import pandas as pd
import matplotlib.pyplot as plt

run_dir = RUNS / M5_RUN_NAME
assert (run_dir/'weights/best.pt').is_file(), 'Best checkpoint missing'
assert (run_dir/'weights/last.pt').is_file(), 'Last checkpoint missing'
assert (run_dir/'results.csv').is_file(), 'Training CSV missing; check Drive mount'
progress = pd.read_csv(run_dir/'results.csv')
progress.columns = progress.columns.str.strip()
metric = 'metrics/mAP50-95(M)'
best_row = progress.loc[progress[metric].idxmax()]
print('Last completed epoch:', int(progress['epoch'].iloc[-1]))
print('Best validation epoch:', int(best_row['epoch']))
print('Best mask mAP50-95:', round(float(best_row[metric]), 4))
print('GPU available:', torch.cuda.is_available())
plt.figure(figsize=(9,4))
plt.plot(progress['epoch'],progress[metric],marker='.')
plt.title('M5 partial run — validation mask mAP50–95')
plt.xlabel('Epoch'); plt.ylabel('Mask mAP50–95'); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(run_dir/'m5_validation_progress.png',dpi=150)
plt.show()


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


import torch

print("GPU:", torch.cuda.is_available())
print("M5 checkpoint:", M5_LAST.is_file())
print("M5 dataset:", (seg_dir / "data.yaml").is_file())


### **RESUME M5** — only after GPU is actually available
1. On a new runtime, execute **1 → 5, 7, 10** (recreate `seg_dir`) and ensure section 7 installed `ultralytics` and section 7's W&B login succeeded. Do **NOT** run the original M5 initial training cell again.
2. Verify `torch.cuda.is_available()` returns **True**. If not, leave `RESUME_M5=False`, and use CPU-only steps instead.
3. Set `RESUME_M5=True` below **once**, run it, and monitor progress. It loads `last.pt` and restores optimizer and epoch state, rather than restarting with `best.pt`.
4. The resumed training may stop early again. Use the saved CSV and validation to report exactly which epochs ran. W&B resumed execution uses a **separate live W&B evidence run**, not fabricated historic metrics.

Ref: [Ultralytics resume documentation](https://docs.ultralytics.com/modes/train/#resuming-interrupted-trainings).


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

RESUME_M5 = True  # deliberately disabled until GPU + dataset are ready

if RESUME_M5:
    from ultralytics import YOLO
    from google.colab import userdata
    import os, json, wandb
    assert torch.cuda.is_available(), 'No GPU! Do not resume on CPU'
    assert (seg_dir/'data.yaml').is_file(), 'Run sections 1–5 and 10 to rebuild segmentation data'
    assert M5_LAST.is_file(), 'Cannot find last.pt in Google Drive'
    os.environ['WANDB_BASE_URL'] = 'https://api.forge.coreweave.com'
    key = userdata.get('WANDB_API_KEY')
    assert key, 'Configure Colab Secret WANDB_API_KEY'
    assert wandb.login(key=key, host=os.environ['WANDB_BASE_URL'], relogin=True)
    run_name = M5_RUN_NAME + '_resumed_from_checkpoint'
    with wandb.init(project='AI251P-Waste-Phase2', name=run_name,
                    config={'resume_from':str(M5_LAST), 'dataset':str(seg_dir/'data.yaml'),
                            'manifest_sha256':sha, 'type':'M5 checkpoint resume'}) as run:
        print('Live resumed training W&B run:',run.url)
        # Resume is only legal on an interrupted, non-completed checkpoint.
        # data= overrides any stale /content data path from the checkpoint.
        YOLO(str(M5_LAST)).train(resume=True, data=str(seg_dir/'data.yaml'),
                                 device=0, workers=2)
    print('Resume finished/stopped; inspect results.csv and weights in:', RUNS/M5_RUN_NAME)
else:
    print('M5 resume NOT started. Set RESUME_M5=True only on GPU after restoring data.')


## 13. M5 validation: mask AP and **separate** IoU/Dice audit
Run after restoring dataset; CPU is possible but slower. AP (full Validation) comes from Ultralytics segmentation validation. The mean mask IoU/Dice here is a **small, explicitly labelled subset audit**, computed by matching predictions to a same-class ground-truth polygon. Unmatched ground-truth instances count as zero. Predictions with no match are not included in the Dice average (they are penalized by mask AP). Do not describe this limited audit as full-split test mIoU/Dice. The source COCO polygons were converted to YOLO; merged multi-part masks can differ from original and require manual inspection.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from ultralytics import YOLO
from PIL import Image
import cv2, numpy as np, pandas as pd
from collections import defaultdict

m5_best = RUNS / M5_RUN_NAME / 'weights/best.pt'
assert m5_best.is_file(), 'M5 best.pt not found'
assert (seg_dir/'data.yaml').is_file(), 'Recreate segmentation dataset using section 10'
m5 = YOLO(str(m5_best))
m5_eval = DRIVE_PROJECT/'evaluation'/'M5'
m5_eval.mkdir(parents=True,exist_ok=True)
seg_metrics = m5.val(data=str(seg_dir/'data.yaml'),split='val',imgsz=640,
                     conf=0.001, plots=True, project=str(m5_eval),
                     name='val_mask_map',exist_ok=True)
print('Full VAL mask mAP50:',float(seg_metrics.seg.map50))
print('Full VAL mask mAP50–95:',float(seg_metrics.seg.map))

# Fixed, diverse validation subset for rasterized per-instance IoU/Dice audit.
ALL_VAL = sorted((seg_dir/'labels/val').glob('*.txt'))
rng = np.random.default_rng(42)
chosen = []
for label in ALL_VAL:
    if label.read_text().strip():
        name=label.stem
        seq=name.split('_frame_')[0]
        if seq not in [x.stem.split('_frame_')[0] for x in chosen]:
            chosen.append(label)
others=[p for p in ALL_VAL if p not in chosen and p.read_text().strip()]
chosen += [others[i] for i in rng.permutation(len(others))[:max(0,24-len(chosen))]]
assert len(chosen)>0, 'No nonempty validation labels'
records=[]
for label in chosen:
    path=seg_dir/'images/val'/(label.stem+'.PNG')
    with Image.open(path) as im:
        W,H=im.size
    ground=[]
    for line in label.read_text().splitlines():
        vals=line.split(); cid=int(vals[0])
        pts=np.asarray([float(v) for v in vals[1:]]).reshape(-1,2)
        pts=np.rint(pts*np.array([W,H])).astype(np.int32)
        mask=np.zeros((H,W),np.uint8);cv2.fillPoly(mask,[pts],1)
        ground.append((cid,mask.astype(bool)))
    pred=m5.predict(str(path),imgsz=640,conf=0.25,iou=0.5,
                    retina_masks=True,verbose=False)[0]
    predictions=[]
    if pred.masks is not None:
        for box,pts in zip(pred.boxes, pred.masks.xy):
            mask=np.zeros((H,W),np.uint8)
            if len(pts)>=3: cv2.fillPoly(mask,[np.rint(pts).astype(np.int32)],1)
            predictions.append((int(box.cls.item()),mask.astype(bool)))
    used=set()
    for cid,gt in ground:
        candidates=[]
        for j,(pc,pm) in enumerate(predictions):
            if pc!=cid or j in used: continue
            inter=np.logical_and(gt,pm).sum(); union=np.logical_or(gt,pm).sum()
            candidates.append((float(inter/union) if union else 0.0,j))
        iou,j=max(candidates,default=(0.0,None))
        if j is not None and iou>0:
            used.add(j)
            pm=predictions[j][1]
            dice=2*np.logical_and(gt,pm).sum()/(gt.sum()+pm.sum()+1e-12)
        else:
            iou,dice=0.0,0.0
        records.append(dict(image=label.name,class_name=CLASS_NAMES[cid],
                            matched=(j is not None and iou>0),iou=float(iou),dice=float(dice)))
audit=pd.DataFrame(records)
audit.to_csv(m5_eval/'mask_iou_dice_VAL_SUBSET24.csv',index=False)
print('Subset:',len(chosen),'images /',len(audit),'GT objects — VALIDATION ONLY')
print('Subset mean IoU:',round(float(audit.iou.mean()),4),
      'mean Dice:',round(float(audit.dice.mean()),4))
print(audit.groupby('class_name')[['iou','dice']].mean().round(4).to_string())


## 14. M8 — ONNX export and **measured** parity (CPU possible)
This is export, not a completed deployed service. Both M4 and M5 PyTorch checkpoints are exported, checked structurally with `onnx.checker`, then compared to ONNX Runtime on a few **validation** frames (not Test). Parity is a sampled operational-point agreement test; record images, confidence and matching IoU. **CPU is allowed**, but can be slower. Don't call sample parity a full-system acceptance result. Original M5 training can be resumed later without invalidating saved earlier validations.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import json, shutil, hashlib
from pathlib import Path
import numpy as np
from ultralytics import YOLO

assert M4_BEST.is_file() and M5_BEST.is_file(), 'M4/M5 best.pt must exist in Drive'
onnx_dir = DRIVE_PROJECT/'deployment'/'onnx'
onnx_dir.mkdir(parents=True, exist_ok=True)

# On some Colab versions the ONNX exporter installs extra dependencies on first use.
import onnx
exports={}
for task,weights in [('detect',M4_BEST),('segment',M5_BEST)]:
    path=Path(YOLO(str(weights)).export(format='onnx',imgsz=640,opset=17,
                                         dynamic=False,simplify=False,device='cpu'))
    model=onnx.load(str(path));onnx.checker.check_model(model)
    destination=onnx_dir/f'{task}.onnx'
    if path.resolve()!=destination.resolve(): shutil.copy2(path,destination)
    exports[task]=str(destination)
    print(task, 'ONNX valid:',destination,'bytes:',destination.stat().st_size)

# Record export hashes, exact provenance and backend info for release manifest.
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
export_meta={task:{'path':path,'sha256':sha256_file(path)} for task,path in exports.items()}
(onnx_dir/'exports_sha256.json').write_text(json.dumps(export_meta,indent=2))
print('Export manifests:',onnx_dir/'exports_sha256.json')


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


from pathlib import Path

WORK = Path("/content/Project9_Phase2")
det_dir = WORK / "zerowaste_yolo_grouped_v2"

print("Dataset ready:", (det_dir / "data.yaml").is_file())

if (det_dir / "images/val").exists():
    print("Validation images:",
          len(list((det_dir / "images/val").glob("*.PNG"))))
else:
    print("Validation images: Not available")


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


from pathlib import Path

det_dir = Path(
    "/content/Project9_Phase2/zerowaste_yolo_grouped_v2"
)

onnx_dir = Path(
    "/content/drive/MyDrive/Project9_Phase2/deployment/onnx"
)

print("Dataset ready:", (det_dir / "data.yaml").is_file())
print("Validation images:",
      len(list((det_dir / "images/val").glob("*.PNG"))))
print("Detection ONNX:", (onnx_dir / "detect.onnx").is_file())
print("Segmentation ONNX:", (onnx_dir / "segment.onnx").is_file())

exports = {
    "detect": str(onnx_dir / "detect.onnx"),
    "segment": str(onnx_dir / "segment.onnx")
}

print("ONNX models ready:", all(
    Path(p).is_file() for p in exports.values()
))



In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import numpy as np, json
from ultralytics import YOLO

assert (det_dir/'data.yaml').is_file(), 'Restore grouped YOLO data for parity validation'
val_images = sorted((det_dir/'images/val').glob('*.PNG'))[::max(1,len(list((det_dir/'images/val').glob('*.PNG')))//8)][:8]
assert len(val_images)>0

def iou_xyxy(a,b):
    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]);ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(aa+ab-inter+1e-12)

# Compare decoded boxes AND class identities. Matching is unique per prediction.
def score_parity(pt,ort,paths,task):
    matched=0; total=0; extra_ort=0
    for p in paths:
        aa=pt.predict(str(p),imgsz=640,conf=0.25,iou=0.5,device='cpu',verbose=False)[0]
        bb=ort.predict(str(p),imgsz=640,conf=0.25,iou=0.5,device='cpu',verbose=False)[0]
        a_boxes=aa.boxes.xyxy.cpu().numpy(); a_cls=aa.boxes.cls.int().cpu().numpy()
        b_boxes=bb.boxes.xyxy.cpu().numpy(); b_cls=bb.boxes.cls.int().cpu().numpy()
        used=set()
        for box,c in zip(a_boxes,a_cls):
            total+=1
            options=[(iou_xyxy(box,other),j) for j,(other,k) in enumerate(zip(b_boxes,b_cls))
                     if j not in used and c==k]
            if options:
                best,j=max(options)
                if best>=0.90: matched+=1;used.add(j)
        extra_ort+=len(b_boxes)-len(used)
    return dict(task=task,split='val',image_count=len(paths),conf=0.25,nms_iou=0.5,
                match_iou=0.90,pytorch_detections=total,matched=matched,
                onnx_unmatched=extra_ort,matched_fraction=matched/total if total else None,
                warning='sampled decoded-box parity only; raw-tensor and mask parity are not yet checked')

parity=[]
for task,weight in [('detect',M4_BEST),('segment',M5_BEST)]:
    parity.append(score_parity(YOLO(str(weight)),YOLO(exports[task]),val_images,task))
(onnx_dir/'sampled_val_parity.json').write_text(json.dumps(parity,indent=2))
print(json.dumps(parity,indent=2))


## 15. M8 — runnable HTTP service + release manifest
This generates files in `Google Drive/Project9_Phase2/deployment/http_service`. It deliberately uses a **manifest** to choose class order, input size, thresholds, backend model checksums and device. `GET /health` and `POST /predict` are endpoints. The application runs PyTorch for both models (ONNX exported separately), matches their masks by class and box overlap, returns a centroid **only** when a segmentation mask matches, and returns per-material counts.

**It is not a remotely hosted, always-on service yet.** Run `uvicorn service:app ...` in a proper environment and send an actual HTTP request before claiming deployment. Benchmark results are specific to the device and service path; no conveyor speed was provided, so throughput compliance cannot yet be determined.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import os,json,shutil,hashlib
from pathlib import Path
from importlib.metadata import version

assert M4_BEST.is_file() and M5_BEST.is_file(), 'Need model checkpoints in Drive'
service_dir=DRIVE_PROJECT/'deployment'/'http_service'
service_dir.mkdir(parents=True,exist_ok=True)
shutil.copy2(M4_BEST,service_dir/'best_det.pt')
shutil.copy2(M5_BEST,service_dir/'best_seg.pt')

def sha256_path(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

release={
    'project':'AI251P-Project9-Phase2', 'class_names':CLASS_NAMES,
    'weights':{'detector':{'file':'best_det.pt','sha256':sha256_path(M4_BEST)},
               'segmenter':{'file':'best_seg.pt','sha256':sha256_path(M5_BEST)}},
    'confidence':0.25,'nms_iou':0.50,'imgsz':640,'device':'cpu',
    'backend':'PyTorch','input_format':'RGB upload; converted to BGR numpy for Ultralytics',
    'inference':{'box_mask_match_min_iou':0.50,'max_upload_mb':12},
    'versions':{name:version(name) for name in ['ultralytics','torch','fastapi','numpy']},
    'status':'prototype, not yet externally hosted; M5 training was interrupted'
}
(service_dir/'release_manifest.json').write_text(json.dumps(release,indent=2))
service_source="from pathlib import Path\nimport io,json,hashlib\nimport cv2\nimport numpy as np\nfrom PIL import Image, UnidentifiedImageError\nfrom fastapi import FastAPI, File, UploadFile, HTTPException\nfrom ultralytics import YOLO\n\nHERE=Path(__file__).resolve().parent\nCONFIG=json.loads((HERE/'release_manifest.json').read_text())\nImage.MAX_IMAGE_PIXELS=16_000_000\n\ndef check_hash(path,expected):\n    h=hashlib.sha256()\n    with path.open('rb') as f:\n        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)\n    if h.hexdigest()!=expected: raise RuntimeError('Model checksum mismatch')\n\ndet_path=HERE/CONFIG['weights']['detector']['file']\nseg_path=HERE/CONFIG['weights']['segmenter']['file']\nfor key,path in [('detector',det_path),('segmenter',seg_path)]:\n    if not path.is_file(): raise RuntimeError(f'Missing {path}')\n    check_hash(path,CONFIG['weights'][key]['sha256'])\ndet=YOLO(str(det_path))\nseg=YOLO(str(seg_path))\napp=FastAPI(title='Project9 Waste Conveyor — Phase2 prototype')\n\n@app.get('/health')\ndef health():\n    return {'status':'ok','backend':'PyTorch','device':CONFIG['device'],\n            'classes':CONFIG['class_names'], 'model_verified':True}\n\ndef overlap(a,b):\n    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])\n    inter=max(0,x2-x1)*max(0,y2-y1)\n    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]);ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])\n    return inter/(aa+ab-inter+1e-12)\n\n@app.post('/predict')\nasync def predict(file:UploadFile=File(...)):\n    raw=await file.read(12*1024*1024+1)\n    if len(raw)>12*1024*1024:raise HTTPException(413,'Max upload 12 MB')\n    if not raw:raise HTTPException(400,'Empty file')\n    try:\n        with Image.open(io.BytesIO(raw)) as im:\n            img=im.convert('RGB')\n        if not 64<=img.width<=4096 or not 64<=img.height<=4096:\n            raise ValueError('Image dimensions out of range')\n        rgb=np.array(img)\n        bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)\n    except (UnidentifiedImageError,ValueError,OSError,Image.DecompressionBombError):\n        raise HTTPException(400,'Invalid or unsupported image')\n    opts=dict(conf=CONFIG['confidence'],iou=CONFIG['nms_iou'],imgsz=CONFIG['imgsz'],\n              device=CONFIG['device'],verbose=False)\n    # Ultralytics expects BGR numpy arrays; upload RGB is converted above.\n    detection=det.predict(bgr,**opts)[0]\n    segmentation=seg.predict(bgr,retina_masks=True,**opts)[0]\n    d_boxes=detection.boxes.xyxy.cpu().numpy()\n    d_classes=detection.boxes.cls.int().cpu().numpy()\n    d_scores=detection.boxes.conf.cpu().numpy()\n    s_boxes=segmentation.boxes.xyxy.cpu().numpy()\n    s_classes=segmentation.boxes.cls.int().cpu().numpy()\n    s_polygons=[] if segmentation.masks is None else segmentation.masks.xy\n    used=set(); objects=[]\n    for bb,c,conf in zip(d_boxes,d_classes,d_scores):\n        options=[(overlap(bb,sb),j) for j,(sb,sc) in enumerate(zip(s_boxes,s_classes))\n                 if sc==c and j not in used]\n        score,j=max(options,default=(0.,-1))\n        is_match=(j>=0 and score>=0.50 and j<len(s_polygons) and len(s_polygons[j])>=3)\n        polygon=None; centroid=None\n        if is_match:\n            used.add(j)\n            pts=np.asarray(s_polygons[j],dtype=np.float32)\n            mom=cv2.moments(pts)\n            if mom['m00']>1e-12:\n                centroid=[round(float(mom['m10']/mom['m00']),1),\n                          round(float(mom['m01']/mom['m00']),1)]\n            polygon=np.round(pts,1).tolist()\n        objects.append({'material':CONFIG['class_names'][int(c)],\n             'confidence':round(float(conf),4),\n             'bbox_xyxy':[round(float(v),1) for v in bb],\n             'mask_polygon_xy':polygon,'pick_point_xy':centroid,\n             'requires_review':not is_match})\n    counts={name:sum(o['material']==name for o in objects) for name in CONFIG['class_names']}\n    return {'image_width':img.width,'image_height':img.height,'objects':objects,\n            'counts':counts,'pick_point_units':'original_image_pixels'}\n"
(service_dir/'service.py').write_text(service_source,encoding='utf-8')
(service_dir/'requirements.txt').write_text(
    '\n'.join(f'{n}=={version(n)}' for n in ['ultralytics','torch','fastapi','uvicorn',
       'python-multipart','numpy','pillow','opencv-python','httpx'])+'\n')
(service_dir/'README.txt').write_text(
    'cd to this folder and run: uvicorn service:app --host 0.0.0.0 --port 8000\n'
    'POST image: curl -F file=@image.PNG http://localhost:8000/predict\n'
    'GET health: curl http://localhost:8000/health\n'
    'This Colab directory is not a permanent public deployment.\n')
print('Service generated:',service_dir)
print('Weights verified by SHA256 at startup; release manifest in Drive')
# Draft model card; review and update it before publishing.
card_path = service_dir/'MODEL_CARD_DRAFT.md'
card_text = (
    '# Project 9 — Model Card (DRAFT)\n\n'
    '## Purpose\nPrototype waste recognition on ZeroWaste-f conveyor images: four material labels, per-object polygons and mask centroids.\n\n'
    '## Training/evidence\nM4 YOLO11n: 34 epochs, validation mAP50 ~0.4832 and mAP50–95 ~0.3245. '
    'M5 YOLO11n-seg: interrupted at 26 epochs, best recorded validation mask mAP50–95 ~0.2888. '
    'No final Test metrics have been verified.\n\n'
    '## Limitations\nOnly four ZeroWaste classes. Many unlabeled objects, strong class imbalance, and possible cross-domain failure; '
    'do not use as an autonomous industrial sorting controller or for hazardous-waste safety decisions. '
    'The conservative grouped candidate excluded 1,272 images. Centroid is not guaranteed to be physically graspable.\n\n'
    '## Status and review\nCPU service prototype, not continuously hosted. ONNX parity and hardware throughput must be measured. '
    'Team members must review all claims, class definitions and dataset licence before submission.\n'
)
card_path.write_text(card_text,encoding='utf-8')
print('Model card draft:',card_path)



In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


from pathlib import Path
import subprocess
import sys
import time
import urllib.request

service_dir = Path(
    "/content/drive/MyDrive/Project9_Phase2/"
    "deployment/http_service"
)

assert (service_dir / "service.py").is_file()

log_path = Path("/content/m8_server.log")

log_file = open(log_path, "w")

server = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "service:app",
        "--app-dir", str(service_dir),
        "--host", "127.0.0.1",
        "--port", "8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

success = False

for i in range(30):
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8000/health",
            timeout=3
        ) as response:
            print("HTTP status:", response.status)
            print("Health response:",
                  response.read().decode())
            success = True
            break
    except Exception:
        if server.poll() is not None:
            break
        time.sleep(2)

if not success:
    print("Service did not start successfully.")
    print(log_path.read_text()[-3000:])


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


from pathlib import Path
import requests
import json

# Use a validation image, not a test image
image_dir = Path(
    "/content/Project9_Phase2/"
    "zerowaste_yolo_grouped_v2/images/val"
)

images = sorted(image_dir.glob("*.PNG"))
assert images, "Validation images not found"

image_path = images[0]
print("Testing image:", image_path.name)

with open(image_path, "rb") as f:
    response = requests.post(
        "http://127.0.0.1:8000/predict",
        files={"file": (image_path.name, f, "image/png")},
        timeout=180
    )

print("HTTP status:", response.status_code)

try:
    result = response.json()
    print("Response fields:", list(result.keys()))
    print(json.dumps(result, indent=2)[:3000])
except ValueError:
    print(response.text[:1500])


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


import time, json, requests, psutil
import numpy as np
import pandas as pd
from pathlib import Path

url = "http://127.0.0.1:8000/predict"

image_path = sorted(Path(
    "/content/Project9_Phase2/"
    "zerowaste_yolo_grouped_v2/images/val"
).glob("*.PNG"))[0]

assert "server" in globals() and server.poll() is None, \
    "HTTP service must be running"

process = psutil.Process(server.pid)

def test_request():
    with open(image_path, "rb") as f:
        start = time.perf_counter()
        response = requests.post(
            url,
            files={"file": (image_path.name, f, "image/png")},
            timeout=180
        )
        elapsed = (time.perf_counter() - start) * 1000
    response.raise_for_status()
    return elapsed

# Warm-up before benchmarking
for _ in range(2):
    test_request()

latencies = []
memory_mb = []

for i in range(25):
    latency = test_request()
    latencies.append(latency)
    memory_mb.append(
        process.memory_info().rss / (1024 ** 2)
    )
    print(f"Request {i+1}/25: {latency:.1f} ms")

summary = {
    "device": "CPU",
    "image_size": "1920x1080",
    "concurrency": 1,
    "warmup_requests": 2,
    "measured_requests": 25,
    "p50_ms": float(np.percentile(latencies, 50)),
    "p95_ms": float(np.percentile(latencies, 95)),
    "p99_ms": float(np.percentile(latencies, 99)),
    "throughput_images_per_sec": float(
        len(latencies) / (sum(latencies) / 1000)
    ),
    "max_server_rss_mb": max(memory_mb)
}

output_dir = Path(
    "/content/drive/MyDrive/Project9_Phase2/"
    "evaluation/M8"
)
output_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "latency_ms": latencies,
    "server_rss_mb": memory_mb
}).to_csv(output_dir / "benchmark_requests.csv", index=False)

(output_dir / "benchmark_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\n=== M8 BENCHMARK RESULTS ===")
print(json.dumps(summary, indent=2))
print("Results saved to Google Drive")


## 16. M8 local HTTP benchmark — explicitly preliminary
Uses FastAPI `TestClient` (in-process HTTP), after an initial warm-up, to measure **end-to-end request-to-JSON** latency for one **Validation** image. It does not prove live external availability or concurrency performance. `serial_requests_per_sec` is not the same as physical conveyor throughput. CPU RSS sampled after requests is **not** a peak-memory measurement; benchmark on final deployment hardware again.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

import time,json,importlib.util,sys,os
import numpy as np
from fastapi.testclient import TestClient

assert (service_dir/'service.py').is_file()
spec=importlib.util.spec_from_file_location('project9_service_v3',service_dir/'service.py')
module=importlib.util.module_from_spec(spec)
sys.modules['project9_service_v3']=module
spec.loader.exec_module(module)
client=TestClient(module.app)
assert client.get('/health').status_code==200
sample=sorted((det_dir/'images/val').glob('*.PNG'))[0]
blob=sample.read_bytes()
def request_once():
    response=client.post('/predict',files={'file':(sample.name,blob,'image/png')})
    response.raise_for_status()
    return response
warmup=3; repetitions=25
for _ in range(warmup):request_once()
lat=[]
for _ in range(repetitions):
    tic=time.perf_counter();request_once();lat.append((time.perf_counter()-tic)*1000)
p50,p95,p99=np.percentile(lat,[50,95,99])
report={'split':'val','sample':sample.name,'warmup':warmup,'timed_calls':repetitions,
        'backend':'PyTorch','device':'cpu','input_bytes':len(blob),
        'p50_ms':float(p50),'p95_ms':float(p95),'p99_ms':float(p99),
        'serial_requests_per_sec':float(1000/np.mean(lat)),
        'limitations':'TestClient in-process, one frame, CPU, not peak memory or physical belt throughput'}
(service_dir/'http_benchmark_VAL_preliminary.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


## 17. Open-world A (5 points): **Visual search** — runnable validation experiment
**Function:** a detected object can retrieve visually similar *training* objects as references for a human sorter. Compare a frozen ImageNet ResNet-18 feature encoder against an 8-bin-per-channel RGB histogram baseline. Each has a gallery of labelled crops from **grouped TRAIN** and queries from **grouped VAL** (no Test). Record material top-1 / hit@5, speed, storage and a failure example; confirm nearest images visually. This is **not run in this uploaded notebook**, so leave the report's numerical cells blank until executed. May work on CPU; GPU is faster.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from pathlib import Path
import numpy as np, pandas as pd, time, json
from PIL import Image
from collections import defaultdict
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Reproducible, class-balanced validation-only sampling of real annotated object boxes.
RNG=np.random.default_rng(42)
open_dir=DRIVE_PROJECT/'open_world'/'visual_search'
open_dir.mkdir(parents=True,exist_ok=True)

def choose_objects(split,per_class):
    obj=defaultdict(list)
    for label in sorted((det_dir/'labels'/split).glob('*.txt')):
        image=det_dir/'images'/split/(label.stem+'.PNG')
        if not image.is_file(): continue
        for line in label.read_text().splitlines():
            c,xc,yc,w,h=map(float,line.split())
            if w>0.003 and h>0.003:
                obj[int(c)].append((str(image),int(c),xc,yc,w,h))
    chosen=[]
    for c in range(len(CLASS_NAMES)):
        assert obj[c],f'Missing material class {c} in {split}'
        indices=RNG.choice(len(obj[c]),size=min(per_class,len(obj[c])),replace=False)
        chosen.extend(obj[c][int(i)] for i in indices)
    return chosen

def crop_object(row):
    path,c,xc,yc,w,h=row
    with Image.open(path) as img:
        img=img.convert('RGB');W,H=img.size
        x1=max(0,int((xc-w/2)*W)); x2=min(W,int(np.ceil((xc+w/2)*W)))
        y1=max(0,int((yc-h/2)*H)); y2=min(H,int(np.ceil((yc+h/2)*H)))
        return img.crop((x1,y1,max(x1+1,x2),max(y1+1,y2)))

def unit(x):
    return x/(np.linalg.norm(x,axis=-1,keepdims=True)+1e-10)

def rgb_hist(crops):
    out=[]
    for im in crops:
        a=np.asarray(im.resize((64,64)),dtype=np.uint8)
        feats=[np.histogram(a[:,:,channel],bins=8,range=(0,256))[0] for channel in range(3)]
        out.append(np.concatenate(feats))
    return unit(np.asarray(out,dtype=np.float32))

gallery=choose_objects('train',40)
queries=choose_objects('val',10)
G=[crop_object(x) for x in gallery]
Q=[crop_object(x) for x in queries]
gallery_y=np.array([x[1] for x in gallery]);query_y=np.array([x[1] for x in queries])

weights=ResNet18_Weights.DEFAULT
encoder=resnet18(weights=weights);encoder.fc=nn.Identity()
device='cuda' if torch.cuda.is_available() else 'cpu'
encoder=encoder.to(device).eval()
transform=weights.transforms()

def embed(crops,batch=16):
    vectors=[]
    with torch.inference_mode():
        for start in range(0,len(crops),batch):
            tensor=torch.stack([transform(x) for x in crops[start:start+batch]]).to(device)
            vectors.append(encoder(tensor).cpu().numpy())
    return unit(np.concatenate(vectors).astype(np.float32))

t0=time.perf_counter();g_feature=embed(G);q_feature=embed(Q)
feature_seconds=time.perf_counter()-t0
h_gallery=rgb_hist(G);h_query=rgb_hist(Q)

def retrieval_metrics(g,q):
    rankings=(q@g.T).argsort(axis=1)[:,::-1]
    top1=gallery_y[rankings[:,0]]
    top5=gallery_y[rankings[:,:5]]
    return {'top1_accuracy':float(np.mean(top1==query_y)),
            'hit_at_5':float(np.mean(np.any(top5==query_y[:,None],axis=1))),
            'precision_at_5':float(np.mean(top5==query_y[:,None]))},rankings

resnet_scores,ranking=retrieval_metrics(g_feature,q_feature)
hist_scores,_=retrieval_metrics(h_gallery,h_query)
np.savez_compressed(open_dir/'gallery_index_train.npz',embeddings=g_feature,
                    labels=gallery_y,source_paths=np.array([x[0] for x in gallery]))
(open_dir/'comparison_VAL.json').write_text(json.dumps({
    'encoder':'ImageNet ResNet18 frozen','baseline':'RGB histogram (8x3)',
    'split':'train gallery / val queries', 'gallery_size':len(gallery),
    'query_size':len(queries),'resnet':resnet_scores,'histogram':hist_scores,
    'embedding_seconds':feature_seconds,'index_bytes':int(g_feature.nbytes)},indent=2))
print('Visual search:',open_dir/'comparison_VAL.json')
print('Frozen ResNet18:',resnet_scores,'RGB histogram baseline:',hist_scores)
print('Example query and its 5 training nearest neighbors:',queries[0][0],
      [(CLASS_NAMES[int(gallery_y[i])],gallery[i][0]) for i in ranking[0,:5]])


## 17B. Open-world B (5 points): **Zero-shot CLIP material classification**
**Different topic from retrieval.** A text-image model predicts one of four material names for uncertain waste crops **without fine-tuning on the ZeroWaste training set**. Function: optional second opinion/relabeling suggestion for a manual reviewer; not a tested unknown-class detector. Compare 2 prompt wordings on **Validation** against the training-gallery RGB histogram nearest-neighbor baseline from Topic A, and log accuracy/macro-F1 and failures. Model download required. Use the *same actual cropped validation labels*, never invent unknown labels. A Test score must be generated only once after prompts are frozen. This section is **not yet executed**.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

%pip -q install transformers


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score
import numpy as np,pandas as pd,json,torch

assert 'queries' in globals() and 'h_gallery' in globals(), 'Run open-world A first'
model_id='openai/clip-vit-base-patch32'
clip=CLIPModel.from_pretrained(model_id).eval().to(device)
processor=CLIPProcessor.from_pretrained(model_id)
material_names=['cardboard','metal','rigid plastic','soft plastic']
prompts={
  'plain':[f'a photo of {name}' for name in material_names],
  'conveyor':[f'a close-up photo of {name} waste on a recycling conveyor belt' for name in material_names]
}

result_rows=[]
for prompt_name,texts in prompts.items():
    logits=[]
    with torch.inference_mode():
        for i in range(0,len(Q),8):
            inputs=processor(text=texts,images=Q[i:i+8],return_tensors='pt',padding=True)
            inputs={k:v.to(device) for k,v in inputs.items()}
            logits.append(clip(**inputs).logits_per_image.detach().cpu().numpy())
    prediction=np.concatenate(logits).argmax(axis=1)
    result_rows.append({'method':'CLIP zero-shot','prompt':prompt_name,'split':'val',
        'accuracy':accuracy_score(query_y,prediction),
        'macro_f1':f1_score(query_y,prediction,labels=list(range(4)),average='macro',zero_division=0)})

# Baseline: nearest train crop based on normalized RGB histogram (no CLIP training).
hist_prediction=gallery_y[(h_query@h_gallery.T).argmax(axis=1)]
result_rows.append({'method':'RGB histogram NN baseline','prompt':'none','split':'val',
    'accuracy':accuracy_score(query_y,hist_prediction),
    'macro_f1':f1_score(query_y,hist_prediction,labels=list(range(4)),average='macro',zero_division=0)})
openB_dir=DRIVE_PROJECT/'open_world'/'zero_shot_clip'
openB_dir.mkdir(parents=True,exist_ok=True)
results=pd.DataFrame(result_rows)
results.to_csv(openB_dir/'CLIP_vs_histogram_VAL.csv',index=False)
print(results.round(4).to_string(index=False))
print('Valid conclusions: zero-shot 4-known-material accuracy ONLY; not unknown detection')


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


import torch

print("GPU available:", torch.cuda.is_available())
print("M4 model ready:", M4_BEST.is_file())
print("M5 model ready:", M5_BEST.is_file())
print("M4 dataset ready:", (det_dir / "data.yaml").is_file())
print("M5 dataset ready:", (seg_dir / "data.yaml").is_file())
print("Split hash ready:", "sha" in globals())

final_dir = DRIVE_PROJECT / "evaluation" / "FINAL_TEST_ONCE"
print("Final test folder already exists:", final_dir.exists())


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


import json
import hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

acceptance = {
    "project": "KAUST AI251P - Project 9",
    "phase": 2,
    "evaluation_split": "test",
    "status": "frozen_before_test",
    "team_members": 5,
    "image_size": 640,
    "operating_confidence": 0.25,
    "nms_iou": 0.50,
    "ap_evaluation_confidence": 0.001,
    "minimum_test_metrics": {
        "M4_mAP50": 0.35,
        "M4_mAP50_95": 0.22,
        "M5_mask_mAP50": 0.30,
        "M5_mask_mAP50_95": 0.20
    },
    "prototype_cpu_p95_limit_ms": 1500,
    "industrial_readiness": "not established",
    "frozen_at": datetime.now(
        ZoneInfo("Asia/Riyadh")
    ).isoformat(),
    "split_manifest_sha256": sha
}

file = DRIVE_PROJECT / "evaluation" / "acceptance_before_test.json"
file.parent.mkdir(parents=True, exist_ok=True)

if file.exists():
    print("Already frozen:", file)
    print(file.read_text())
else:
    with open(file, "x") as f:
        json.dump(acceptance, f, indent=2)
    print("Acceptance criteria frozen successfully")
    print("Saved:", file)

print("Final Test not executed by this cell")


## 18. Freeze the final evaluation (Test) — **OFF by default**
**Do not run on Test yet.** Before the one final evaluation: finish M5 resume (or explicitly document why impossible), choose model/thresholds on Validation only, record acceptance criteria set *before* looking at Test, settle any baseline comparisons, and have all five members review the final metrics and code. The flag below defaults to `False` and writes only when deliberately enabled. This cell is an evaluation guard, not a completed test result. Save `best.pt`, exact `split_manifest.csv` and release-manifest hashes.


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


RUN_FINAL_TEST = True  # never change to True until decisions are frozen and team reviewed

if RUN_FINAL_TEST:
    import json,hashlib
    def sha256_file(path):
        digest=hashlib.sha256()
        with open(path,'rb') as f:
            for data in iter(lambda:f.read(4*1024*1024),b''):
                digest.update(data)
        return digest.hexdigest()
    from ultralytics import YOLO
    assert (det_dir/'data.yaml').is_file() and (seg_dir/'data.yaml').is_file()
    assert M4_BEST.is_file() and M5_BEST.is_file()
    final_dir=DRIVE_PROJECT/'evaluation'/'FINAL_TEST_ONCE'
    if final_dir.exists():
        raise RuntimeError('Final test folder already exists. Do NOT silently repeat the Test evaluation.')
    final_dir.mkdir(parents=True)
    protocol={'confidence':0.25,'nms_iou':0.50,'imgsz':640,
       'split':'test','manifest_sha256':sha,
       'm4_checkpoint_sha256':sha256_file(M4_BEST),
       'm5_checkpoint_sha256':sha256_file(M5_BEST),
       'warning':'Both masks/boxes evaluated at low confidence for AP; operating threshold for deployment is 0.25'}
    (final_dir/'frozen_protocol.json').write_text(json.dumps(protocol,indent=2))
    try:
        m4_result=YOLO(str(M4_BEST)).val(data=str(det_dir/'data.yaml'),split='test',
            imgsz=640,conf=0.001,iou=0.5,plots=True,project=str(final_dir),name='M4')
        m5_result=YOLO(str(M5_BEST)).val(data=str(seg_dir/'data.yaml'),split='test',
            imgsz=640,conf=0.001,iou=0.5,plots=True,project=str(final_dir),name='M5')
        measured={'M4_mAP50':float(m4_result.box.map50),'M4_mAP50_95':float(m4_result.box.map),
             'M5_mask_mAP50':float(m5_result.seg.map50),'M5_mask_mAP50_95':float(m5_result.seg.map)}
        (final_dir/'results.json').write_text(json.dumps(measured,indent=2))
        print('FINAL ONE-TIME TEST:',measured)
    except Exception:
        print('Partial final Test artifacts exist; investigate before attempting any rerun')
        raise
else:
    print('FINAL TEST NOT RUN. This is intentional; keep Test unseen.')


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')


import json
from pathlib import Path

evaluation_dir = Path(
    "/content/drive/MyDrive/Project9_Phase2/evaluation"
)

acceptance = json.loads(
    (evaluation_dir / "acceptance_before_test.json").read_text()
)
results = json.loads(
    (evaluation_dir / "FINAL_TEST_ONCE/results.json").read_text()
)

comparison = {}

for metric, minimum in acceptance["minimum_test_metrics"].items():
    actual = results[metric]
    comparison[metric] = {
        "actual": actual,
        "required": minimum,
        "passed": actual >= minimum
    }

output = evaluation_dir / "final_acceptance_comparison.json"
output.write_text(json.dumps(comparison, indent=2))

print(json.dumps(comparison, indent=2))
print("\nSaved:", output)
print("No new Test evaluation was performed.")


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# Paste into a new Colab cell after mounting Drive. Reads files only.
from pathlib import Path
import hashlib
import json

root = Path('/content/drive/MyDrive/Project9_Phase2')
protocol_path = root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json'
assert protocol_path.is_file(), 'لم أجد بروتوكول الاختبار؛ تأكد من ربط Drive والمسار.'
protocol = json.loads(protocol_path.read_text())

def sha256(path):
    value = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()

for model, run, service, key in [
    ('M4', 'M4_yolo11n_640_grouped_v2_seed42', 'best_det.pt', 'm4_checkpoint_sha256'),
    ('M5', 'M5_yolo11n_seg_640_grouped_v2_seed42', 'best_seg.pt', 'm5_checkpoint_sha256'),
]:
    expected = protocol.get(key)
    for label, path in [
        ('Training best', root / 'runs' / run / 'weights/best.pt'),
        ('HTTP copy', root / 'deployment/http_service' / service),
    ]:
        if not expected:
            status = 'MISSING REFERENCE HASH'
        elif not path.is_file():
            status = 'FILE MISSING'
        else:
            status = 'MATCH' if sha256(path) == expected else 'DIFFERENT'
        print(f'{model} | {label} | {status}')
print('Read-only check. No training or Test evaluation was run.')


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# Paste in a new Colab cell. Reads your two M5 checkpoints; no inference or writes.
from pathlib import Path
import gc
import json
import hashlib
from importlib.metadata import version
import torch
from ultralytics import YOLO

root = Path('/content/drive/MyDrive/Project9_Phase2')
source = root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt'
deployed = root / 'deployment/http_service/best_seg.pt'
protocol = root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json'
assert all(p.is_file() for p in (source, deployed, protocol)), 'Missing checkpoint or frozen protocol.'

def file_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

expected = json.loads(protocol.read_text()).get('m5_checkpoint_sha256')
source_sha, deployed_sha = file_hash(source), file_hash(deployed)
assert expected and source_sha == expected, 'Source no longer matches frozen Test weights. Stop here.'

# Use the same loader as the HTTP service; it selects the effective model/EMA.
# Normalize both to CPU float32, as used by the existing PyTorch CPU service.
left = YOLO(str(source))
right = YOLO(str(deployed))
a = left.model.cpu().float().eval()
b = right.model.cpu().float().eval()
sa, sb = a.state_dict(), b.state_dict()
missing = sorted(set(sa) - set(sb))
extra = sorted(set(sb) - set(sa))
different = []
for key in sorted(set(sa) & set(sb)):
    x, y = sa[key], sb[key]
    if x.shape != y.shape:
        different.append({'key': key, 'reason': 'shape', 'source': list(x.shape), 'http': list(y.shape)})
    elif x.dtype != y.dtype:
        different.append({'key': key, 'reason': 'dtype'})
    elif not torch.equal(x, y):
        delta = (x.to(torch.float64) - y.to(torch.float64)).abs()
        maximum = float(delta.max()) if delta.numel() and torch.isfinite(delta).all().item() else None
        different.append({'key': key, 'reason': 'values', 'max_abs_difference': maximum})

def module_types(model):
    return [(name, type(module).__module__, type(module).__qualname__)
            for name, module in model.named_modules()]

def stride(model):
    value = getattr(model, 'stride', None)
    return value.detach().cpu().tolist() if torch.is_tensor(value) else value

def module_options(model):
    result = {}
    for name, module in model.named_modules():
        for attr in ('nc', 'nm', 'npr', 'reg_max', 'end2end', 'legacy',
                     'inplace', 'export', 'dynamic', 'xyxy', 'max_det'):
            if hasattr(module, attr):
                value = getattr(module, attr)
                if value is None or isinstance(value, (bool, int, float, str)):
                    result[f'{name}.{attr}'] = value
    return result

options_a, options_b = module_options(a), module_options(b)
yaml_equal = (json.dumps(getattr(a, 'yaml', None), sort_keys=True, default=str)
              == json.dumps(getattr(b, 'yaml', None), sort_keys=True, default=str))

state_equal = not missing and not extra and not different
basic_metadata_equal = (module_types(a) == module_types(b)
                        and left.names == right.names
                        and left.task == right.task
                        and stride(a) == stride(b)
                        and options_a == options_b
                        and yaml_equal)
report = {
    'versions': {name: version(name) for name in ('torch', 'ultralytics')},
    'source_sha256': source_sha, 'http_sha256': deployed_sha,
    'source_matches_frozen_test': True,
    'checkpoint_files_identical': deployed_sha == expected,
    'effective_state_equal_on_cpu_float32': state_equal,
    'basic_model_metadata_equal': basic_metadata_equal,
    'model_yaml_equal': yaml_equal,
    'different_module_options': {
        key: {'source': options_a.get(key), 'http': options_b.get(key)}
        for key in sorted(set(options_a) | set(options_b))
        if (key in options_a) != (key in options_b) or options_a.get(key) != options_b.get(key)
    },
    'state_entries_source': len(sa), 'state_entries_http': len(sb),
    'missing_keys': missing, 'extra_keys': extra,
    'different_entries_count': len(different), 'first_differences': different[:10],
    'scope': 'Loaded parameters and buffers, module types, class names, task and strides. Not output parity or a benchmark.',
}
print(json.dumps(report, indent=2, ensure_ascii=False))
del sa, sb, a, b, left, right
gc.collect()
print('Read-only comparison. No files changed. No training, prediction or Test evaluation.')


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# Run immediately after compare_m5_state_readonly.py in the SAME Colab session.
# Saves a new evidence file only. Does not change models, manifest or Test results.
from pathlib import Path
from datetime import datetime, timezone
import json

assert 'report' in globals() and isinstance(report, dict), 'Run the M5 state comparison cell first.'
required = ('source_matches_frozen_test', 'effective_state_equal_on_cpu_float32',
            'basic_model_metadata_equal', 'model_yaml_equal')
assert all(report.get(key) is True for key in required), 'Unexpected comparison result: review before saving.'
assert report.get('different_entries_count') == 0, 'Comparison contains differences.'
assert report.get('source_sha256') and report.get('http_sha256'), 'Missing model hashes.'
now = datetime.now(timezone.utc)
payload = dict(report)
payload['evidence_saved_at_utc'] = now.isoformat()
payload['note'] = ('Loaded state equality on CPU float32; file hashes differ. '
                   'This does not establish full output parity or replace the original HTTP benchmark provenance.')
folder = Path('/content/drive/MyDrive/Project9_Phase2/evaluation/release_audit')
folder.mkdir(parents=True, exist_ok=True)
saved = folder / ('M5_state_comparison_' + now.strftime('%Y%m%dT%H%M%S%fZ') + '.json')
with saved.open('x', encoding='utf-8') as stream:
    json.dump(payload, stream, indent=2, ensure_ascii=False)
print('Saved:', saved)
print('Models, release manifest and Test results were not changed.')


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

"""Colab cell: compare M5 masks on eight VAL images; save NEW audit evidence.
No training, Test evaluation, model export, or changes to existing artifacts.
"""

def audit_onnx_masks_val():
    from pathlib import Path
    from datetime import datetime, timezone
    from importlib.metadata import version
    import hashlib
    import json
    import platform
    import numpy as np
    from PIL import Image
    from scipy.optimize import linear_sum_assignment
    from ultralytics import YOLO

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    pt_path = root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt'
    onnx_path = root / 'deployment/onnx/segment.onnx'
    protocol_path = root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json'
    export_manifest_path = root / 'deployment/onnx/exports_sha256.json'
    val_dir = Path('/content/Project9_Phase2/zerowaste_yolo_grouped_v2/images/val')
    for path in (pt_path, onnx_path, protocol_path, export_manifest_path):
        assert path.is_file(), f'Missing: {path}'

    def sha256(path):
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    protocol = json.loads(protocol_path.read_text())
    export_manifest = json.loads(export_manifest_path.read_text())
    pt_sha, onnx_sha = sha256(pt_path), sha256(onnx_path)
    assert protocol.get('m5_checkpoint_sha256') == pt_sha, 'M5 does not match frozen Test checkpoint.'
    expected_onnx = export_manifest.get('segment', {}).get('sha256')
    assert expected_onnx and onnx_sha == expected_onnx, 'ONNX does not match saved export hash.'
    all_images = sorted(val_dir.glob('*.PNG'))
    assert len(all_images) == 451, f'Expected 451 VAL images; found {len(all_images)}. Restore VAL data first.'
    # Same filename sampling rule used in the notebook's earlier box-only audit.
    images = all_images[::max(1, len(all_images) // 8)][:8]
    assert len(images) == 8 and all(path.parent.name == 'val' for path in images)

    # Fix square letterboxing: PT rect=True could otherwise differ from static ONNX.
    options = dict(imgsz=640, rect=False, conf=0.25, iou=0.50,
                   max_det=300, agnostic_nms=False, augment=False,
                   half=False, device='cpu', retina_masks=True,
                   save=False, save_txt=False, save_crop=False, verbose=False)
    pt = YOLO(str(pt_path), task='segment')
    ort = YOLO(str(onnx_path), task='segment')
    names = {int(key): str(value) for key, value in pt.names.items()}

    def unpack(result, shape):
        assert tuple(result.orig_shape) == shape, 'Unexpected original image shape.'
        assert result.boxes is not None, 'Missing segmentation boxes.'
        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.int().cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        assert np.isfinite(boxes).all() and np.isfinite(scores).all()
        result_names = {int(key): str(value) for key, value in result.names.items()}
        assert result_names == names, 'Class names/order differ between backends.'
        if len(boxes) == 0:
            assert result.masks is None or len(result.masks.data) == 0
            masks = np.empty((0, *shape), dtype=bool)
        else:
            assert result.masks is not None, 'Boxes exist but masks are missing.'
            masks = result.masks.data.cpu().numpy()
            assert masks.shape == (len(boxes), *shape), (
                f'Mask shape {masks.shape} is not original image size {shape}. No silent resizing.')
            assert np.isfinite(masks).all(), 'Nonfinite mask values.'
            masks = masks > 0.5
        return boxes, classes, scores, masks

    def box_ious(left, right):
        if not len(left) or not len(right):
            return np.zeros((len(left), len(right)), dtype=np.float64)
        lo = np.maximum(left[:, None, :2], right[None, :, :2])
        hi = np.minimum(left[:, None, 2:], right[None, :, 2:])
        intersection = np.maximum(hi - lo, 0).prod(axis=2)
        area_left = np.maximum(left[:, 2:] - left[:, :2], 0).prod(axis=1)
        area_right = np.maximum(right[:, 2:] - right[:, :2], 0).prod(axis=1)
        union = area_left[:, None] + area_right[None, :] - intersection
        return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

    records, all_pairs = [], []
    total_pt = total_onnx = 0
    for number, path in enumerate(images, 1):
        with Image.open(path) as image:
            width, height = image.size
        shape = (height, width)
        ap = pt.predict(str(path), **options)[0]
        bp = ort.predict(str(path), **options)[0]
        boxes_a, cls_a, score_a, masks_a = unpack(ap, shape)
        boxes_b, cls_b, score_b, masks_b = unpack(bp, shape)
        overlaps = box_ious(boxes_a, boxes_b)
        valid = (cls_a[:, None] == cls_b[None, :]) & (overlaps >= 0.90)
        pairs = []
        if len(boxes_a) and len(boxes_b):
            # Maximize number of valid matches, then sum box IoU, one-to-one.
            # 1e6 invalid penalty dominates <=300 valid edge costs in [0,1].
            rows, columns = linear_sum_assignment(np.where(valid, 1.0 - overlaps, 1e6))
            for ia, ib in zip(rows, columns):
                if not valid[ia, ib]:
                    continue
                ma, mb = masks_a[ia], masks_b[ib]
                area_a, area_b = int(np.count_nonzero(ma)), int(np.count_nonzero(mb))
                intersection = int(np.count_nonzero(ma & mb))
                union = area_a + area_b - intersection
                pair = {
                    'pt_index': int(ia), 'onnx_index': int(ib),
                    'class_id': int(cls_a[ia]), 'class_name': names[int(cls_a[ia])],
                    'box_iou': float(overlaps[ia, ib]),
                    'pt_confidence': float(score_a[ia]), 'onnx_confidence': float(score_b[ib]),
                    'abs_confidence_difference': abs(float(score_a[ia]) - float(score_b[ib])),
                    'pt_mask_area': area_a, 'onnx_mask_area': area_b,
                    'mask_iou': intersection / union if union else None,
                    'mask_dice': 2 * intersection / (area_a + area_b) if area_a + area_b else None,
                }
                pairs.append(pair)

        matched_a = {pair['pt_index'] for pair in pairs}
        matched_b = {pair['onnx_index'] for pair in pairs}

        def unmatched(boxes, classes, scores, matched):
            return [{'index': int(i), 'class_id': int(classes[i]),
                     'confidence': float(scores[i]), 'box_xyxy': boxes[i].tolist()}
                    for i in range(len(boxes)) if i not in matched]

        record = {'image': path.name, 'sha256': sha256(path), 'shape_hw': list(shape),
                  'pt_total': len(boxes_a), 'onnx_total': len(boxes_b),
                  'matched_pairs': pairs,
                  'unmatched_pt': unmatched(boxes_a, cls_a, score_a, matched_a),
                  'unmatched_onnx': unmatched(boxes_b, cls_b, score_b, matched_b)}
        records.append(record)
        all_pairs.extend(pairs)
        total_pt += len(boxes_a)
        total_onnx += len(boxes_b)
        print(f'VAL image {number}/8: PT={len(boxes_a)}, ONNX={len(boxes_b)}, matched={len(pairs)}')
        del ap, bp, masks_a, masks_b

    def stat(key, operation):
        values = [pair[key] for pair in all_pairs if pair[key] is not None]
        return float(operation(values)) if values else None

    summary = {
        'split': 'val', 'image_count': len(images),
        'pt_total': total_pt, 'onnx_total': total_onnx, 'matched_pairs': len(all_pairs),
        'unmatched_pt': total_pt - len(all_pairs), 'unmatched_onnx': total_onnx - len(all_pairs),
        'matched_fraction_of_pt': len(all_pairs) / total_pt if total_pt else None,
        'matched_fraction_of_onnx': len(all_pairs) / total_onnx if total_onnx else None,
        'matched_pairs_with_defined_mask_metrics': sum(pair['mask_iou'] is not None for pair in all_pairs),
        'matched_mask_iou_mean': stat('mask_iou', np.mean),
        'matched_mask_iou_min': stat('mask_iou', np.min),
        'matched_mask_dice_mean': stat('mask_dice', np.mean),
        'matched_mask_dice_min': stat('mask_dice', np.min),
        'matched_confidence_difference_max': stat('abs_confidence_difference', np.max),
        'interpretation': 'Mask means are CONDITIONAL on matched boxes, not whole-system parity or GT accuracy.',
    }
    now = datetime.now(timezone.utc)
    audit = {
        'created_at_utc': now.isoformat(), 'python': platform.python_version(),
        'versions': {key: version(key) for key in ('torch', 'ultralytics', 'onnxruntime', 'scipy', 'numpy')},
        'source_pt': {'path': str(pt_path), 'sha256': pt_sha, 'matches_frozen_test': True},
        'onnx': {'path': str(onnx_path), 'sha256': onnx_sha, 'matches_export_manifest': True,
                 'original_checkpoint_provenance': 'Not established: old export manifest has no source checkpoint hash.'},
        'prediction_options': options, 'mask_binary_threshold': 0.5,
        'matching': 'Maximum-cardinality class-consistent box IoU>=0.90, then maximum total box IoU.',
        'sampling': 'Same deterministic filename rule as earlier eight-image box audit; square rect=False is explicit here.',
        'summary': summary, 'images': records,
        'limitations': 'Eight VAL images only. No raw-tensor parity, GT accuracy, Test, export or benchmark. No new acceptance threshold.',
    }
    folder = root / 'evaluation/release_audit'
    folder.mkdir(parents=True, exist_ok=True)
    saved = folder / ('M5_ONNX_masks_VAL_' + now.strftime('%Y%m%dT%H%M%S%fZ') + '.json')
    with saved.open('x', encoding='utf-8') as stream:
        json.dump(audit, stream, indent=2, ensure_ascii=False, allow_nan=False)
    print('\n=== ONNX MASK AUDIT: VALIDATION ONLY ===')
    print(json.dumps(summary, indent=2, ensure_ascii=False, allow_nan=False))
    print('Saved:', saved)
    print('No training, Test evaluation, export or changes to previous files.')
    return {'summary': summary, 'saved': str(saved)}


onnx_mask_audit_result = audit_onnx_masks_val()


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

"""Colab: collect existing results and figures; no model execution or Drive writes.
Creates a new ZIP in /content and downloads it. Excludes model binaries and datasets.
"""

def collect_report_evidence():
    from pathlib import Path
    from datetime import datetime, timezone
    from importlib.metadata import version, PackageNotFoundError
    import hashlib
    import json
    import platform
    import zipfile
    from google.colab import files

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    assert root.is_dir(), 'Mount Google Drive first.'
    now = datetime.now(timezone.utc)
    output = Path('/content') / ('Project9_Report_Evidence_' + now.strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
    metadata_suffixes = {'.json', '.csv', '.yaml', '.yml', '.md', '.txt'}
    figure_suffixes = {'.png', '.jpg', '.jpeg'}
    candidates = {}
    skipped = []

    def consider(path):
        if not path.is_file() or path.is_symlink():
            return
        relative = path.relative_to(root)
        if any(part.lower() in {'wandb', '.git', '__pycache__'} or part.startswith('.') for part in relative.parts):
            return
        if any(word in path.stem.lower() for word in ('secret', 'credential', 'token', 'api_key')):
            return
        if path.suffix.lower() not in metadata_suffixes | figure_suffixes:
            return
        if path.stat().st_size > 5 * 1024 * 1024:
            skipped.append({'path': relative.as_posix(), 'reason': 'larger than 5 MiB'})
            return
        candidates[relative.as_posix()] = path

    # Known experiment/evaluation outputs, not raw training/validation/test datasets.
    for folder in ('data_protocol', 'evaluation', 'open_world'):
        base = root / folder
        if base.is_dir():
            for path in base.rglob('*'):
                consider(path)
    runs = root / 'runs'
    if runs.is_dir():
        for run in sorted(runs.iterdir()):
            if run.is_dir() and run.name.startswith(('M4_', 'M5_')):
                for path in run.iterdir():
                    consider(path)
    for filename in ('exports_sha256.json', 'sampled_val_parity.json'):
        consider(root / 'deployment/onnx' / filename)
    for filename in ('release_manifest.json', 'MODEL_CARD_DRAFT.md', 'README.txt',
                     'requirements.txt', 'http_benchmark_VAL_preliminary.json'):
        consider(root / 'deployment/http_service' / filename)

    manifest = {
        'created_at_utc': now.isoformat(), 'project_root': str(root),
        'python': platform.python_version(), 'packages': {},
        'scope': 'Saved results and diagnostic figures only; no training, inference, Test evaluation or exports.',
        'files': [], 'skipped': skipped, 'model_inventory': [],
    }
    for name in ('torch', 'ultralytics', 'onnxruntime', 'fastapi', 'numpy'):
        try:
            manifest['packages'][name] = version(name)
        except PackageNotFoundError:
            manifest['packages'][name] = None

    def hash_file(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    for relative in (
        'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt',
        'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt',
        'deployment/http_service/best_det.pt', 'deployment/http_service/best_seg.pt',
        'deployment/onnx/detect.onnx', 'deployment/onnx/segment.onnx',
    ):
        path = root / relative
        item = {'path': relative, 'exists': path.is_file()}
        if path.is_file():
            item.update(bytes=path.stat().st_size, sha256=hash_file(path))
        manifest['model_inventory'].append(item)

    required = (
        'evaluation/FINAL_TEST_ONCE/results.json',
        'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'evaluation/acceptance_before_test.json',
        'evaluation/final_acceptance_comparison.json',
        'evaluation/M8/benchmark_summary.json',
        'evaluation/M8/benchmark_requests.csv',
        'deployment/onnx/exports_sha256.json',
    )
    manifest['missing_expected_files'] = [name for name in required if not (root / name).is_file()]
    total = 0
    with zipfile.ZipFile(output, 'x', zipfile.ZIP_DEFLATED) as archive:
        for relative, path in sorted(candidates.items()):
            size = path.stat().st_size
            if total + size > 80 * 1024 * 1024:
                skipped.append({'path': relative, 'reason': '80 MiB package budget'})
                continue
            data = path.read_bytes()
            archive.writestr('evidence/' + relative, data)
            total += len(data)
            manifest['files'].append({'path': relative, 'bytes': len(data),
                                      'sha256': hashlib.sha256(data).hexdigest()})
        archive.writestr('collection_manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False))

    included = [item['path'] for item in manifest['files']]
    summary = {
        'collected_files': len(included),
        'figures': sum(Path(name).suffix.lower() in figure_suffixes for name in included),
        'saved_M5_state_audits': sum('M5_state_comparison_' in name for name in included),
        'saved_ONNX_mask_audits': sum('M5_ONNX_masks_VAL_' in name for name in included),
        'zip_MiB': round(output.stat().st_size / 1024**2, 2),
        'missing_expected_files': manifest['missing_expected_files'],
        'skipped_files': len(skipped),
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print('ZIP:', output)
    print('Downloading saved evidence only. No models, full datasets or API credentials are requested.')
    files.download(str(output))
    return {'zip': str(output), 'summary': summary}


report_evidence_collection = collect_report_evidence()


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

%pip install -q "ultralytics==8.4.213"

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ultralytics

root = Path('/content/drive/MyDrive/Project9_Phase2')
val = Path('/content/Project9_Phase2/zerowaste_seg_grouped_v2/images/val')

print('Ultralytics:', ultralytics.__version__)
print('M5 موجود:', (root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt').is_file())
print('صور Validation:', len(list(val.glob('*.PNG'))))

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# Colab: restore frozen VAL files, then compare baseline and selected M5.
# May download the 7.52 GB original archive if local source/cache is unavailable.

"""Paste into one Colab cell: restore the frozen 451-image M5 VAL split only.

No model is loaded, trained, exported, or evaluated. Drive files are read only.
If neither raw source nor a cached archive is available, this cell downloads
the original 7.52 GB archive to Colab's temporary /content disk.
"""


def restore_m5_validation_only():
    from pathlib import Path, PurePosixPath
    from collections import Counter, defaultdict
    from datetime import datetime, timezone
    from importlib.metadata import version
    import csv
    import hashlib
    import io
    import json
    import math
    import shutil
    import subprocess
    import tempfile
    import traceback
    import zipfile

    import numpy as np
    from PIL import Image
    from ultralytics.data.converter import merge_multi_segment

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    work = Path('/content/Project9_Phase2')
    destination = work / 'zerowaste_seg_grouped_v2_VAL_ONLY_RESTORED'
    raw_train = Path('/content/ZeroWaste/splits_final_deblurred/train')
    local_archive = Path('/content/zerowaste-f-final.zip')
    cached_archive = root / 'cache/zerowaste-f-final.zip'
    protocol_path = root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json'
    results_path = root / 'evaluation/FINAL_TEST_ONCE/results.json'
    model_path = root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt'
    expected_manifest = 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50'
    archive_size = 7518242799
    archive_md5 = 'e26e31a58080bca6782dca0e56074c5d'
    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    password = 'REDACTED_DATASET_ZIP_PASSWORD'
    class_names = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    expected_objects = dict(zip(class_names, [1442, 34, 144, 622]))

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest(path, algorithm='sha256'):
        value = hashlib.new(algorithm)
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()

    def write_json(path, value):
        with path.open('x', encoding='utf-8') as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)

    print('=== فحص استعادة Validation فقط ===')
    require(version('ultralytics') == '8.4.213', 'يلزم إصدار ultralytics 8.4.213 المستخدم في المشروع.')
    missing = [str(p) for p in [protocol_path, results_path, model_path] if not p.is_file()]
    require(not missing, 'ملفات Drive المطلوبة غير موجودة؛ لم يبدأ تنزيل: ' + repr(missing))
    frozen = json.loads(protocol_path.read_text(encoding='utf-8'))
    require(frozen.get('manifest_sha256') == expected_manifest, 'بصمة تقسيم البروتوكول مختلفة؛ توقف للمراجعة.')
    manifest_path = root / 'data_protocol' / expected_manifest[:12] / 'split_manifest.csv'
    require(manifest_path.is_file(), 'ملف التقسيم المجمد غير موجود: ' + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    require(hashlib.sha256(manifest_bytes).hexdigest() == expected_manifest, 'بصمة ملف التقسيم غير مطابقة.')
    model_sha = digest(model_path)
    require(model_sha == frozen.get('m5_checkpoint_sha256'), 'أوزان M5 لا تطابق الأوزان المجمدة؛ لم يبدأ تنزيل.')
    before = {str(p): digest(p) for p in [protocol_path, results_path, model_path, manifest_path]}
    require(not destination.exists(),
            'مجلد البيانات موجود بالفعل. لن نستبدله. أرسل هذا المسار للمراجعة: ' + str(destination))
    require(not (destination / 'TRAIN_UNAVAILABLE_DO_NOT_TRAIN').exists(), 'مسار حارس التدريب موجود بشكل غير متوقع.')
    all_rows = list(csv.DictReader(io.StringIO(manifest_bytes.decode('utf-8-sig'))))
    require(len(all_rows) == 4503, 'عدد صفوف ملف التقسيم غير متوقع.')
    rows = [r for r in all_rows if r['split'] == 'val']
    require(len(rows) == 451, 'المطلوب 451 صورة Validation من التقسيم المحفوظ.')
    require(all(r['original_split'] == 'train' for r in rows), 'مصدر صفوف Validation مختلف.')
    require(Counter(r['sequence'].zfill(2) for r in rows) == {'07': 351, '11': 100}, 'مجموعات Validation مختلفة.')
    require({name: sum(int(r[name]) for r in rows) for name in class_names} == expected_objects,
            'إحصاءات الفئات في ملف التقسيم مختلفة.')
    require(len({r['filename'] for r in rows}) == 451, 'أسماء صور Validation مكررة.')
    require(len({int(r['image_id']) for r in rows}) == 451, 'معرفات الصور مكررة.')
    for row in rows:
        name = row['filename']
        require(name == PurePosixPath(name).name and '\\' not in name and name.endswith('.PNG'),
                'اسم صورة غير آمن في ملف التقسيم.')
        require(PurePosixPath(row['source_path']) == PurePosixPath(str(raw_train / 'data' / name)),
                'مسار مصدر صورة مختلف في ملف التقسيم.')
        require(PurePosixPath(row['annotation_path']) == PurePosixPath(str(raw_train / 'labels.json')),
                'مسار ملف التعليقات مختلف في ملف التقسيم.')
    work.mkdir(parents=True, exist_ok=True)

    # Reuse existing raw files when every required VAL source exists. This does
    # not claim that an unverified raw directory has the archive's MD5.
    raw_ready = (raw_train / 'labels.json').is_file() and all(
        (raw_train / 'data' / r['filename']).is_file() for r in rows)
    archive = None
    zip_handle = None
    source_report = {}
    member_for_image = {}
    if raw_ready:
        print('مصدر الصور موجود؛ لا حاجة لتنزيل الأرشيف.')
        annotations_bytes = (raw_train / 'labels.json').read_bytes()
        source_report = {'mode': 'existing_raw_files', 'raw_directory': str(raw_train),
                         'annotation_file_sha256': hashlib.sha256(annotations_bytes).hexdigest(),
                         'archive_md5_verified': False,
                         'notice': 'Raw file provenance is recorded; a full archive digest is not established.'}
        image_bytes_needed = sum((raw_train / 'data' / r['filename']).stat().st_size for r in rows)
    else:
        archive = local_archive if local_archive.is_file() else cached_archive if cached_archive.is_file() else local_archive
        current_size = archive.stat().st_size if archive.is_file() else 0
        require(current_size <= archive_size, 'حجم الأرشيف أكبر من الحجم المعتمد؛ توقف للمراجعة.')
        if current_size < archive_size:
            require(archive == local_archive, 'نسخة الأرشيف في Drive ناقصة؛ لم يتم تعديلها. أرسل هذا الناتج.')
            require(shutil.which('wget') is not None, 'أداة wget غير متاحة في الجلسة.')
            remaining = archive_size - current_size
            required_free = remaining + 2 * 1024**3
            free = shutil.disk_usage('/content').free
            require(free >= required_free,
                    f'مساحة Colab غير كافية: المتاح {free / 1024**3:.2f} GiB والمطلوب تقريبًا {required_free / 1024**3:.2f} GiB.')
            print(f'سيتم تنزيل/استكمال أرشيف ZeroWaste في Colab فقط: المتبقي {remaining / 10**9:.2f} GB من 7.52 GB.')
            subprocess.run(['wget', '-c', '--tries=5', '--timeout=60', '--progress=dot:giga',
                            '-O', str(local_archive), archive_url], check=True)
        require(archive.is_file() and archive.stat().st_size == archive_size, 'حجم تنزيل الأرشيف غير مكتمل.')
        print('التحقق من سلامة الأرشيف؛ قد يستغرق قليلًا...')
        actual_md5 = digest(archive, 'md5')
        require(actual_md5 == archive_md5, 'بصمة MD5 للأرشيف غير مطابقة؛ لن يتم استخراج البيانات.')
        require(shutil.which('unzip') is not None, 'أداة unzip غير متاحة في الجلسة.')
        zip_handle = zipfile.ZipFile(archive, 'r')

        def find_member(suffix):
            found = [entry for entry in zip_handle.infolist()
                     if not entry.is_dir()
                     and (entry.filename == suffix or entry.filename.endswith('/' + suffix))
                     and '..' not in PurePosixPath(entry.filename).parts]
            require(len(found) == 1, 'عنصر المصدر داخل الأرشيف غير محدد بشكل فريد: ' + suffix)
            return found[0]

        annotation_member = find_member('train/labels.json')
        annotations_bytes = subprocess.run(
            ['unzip', '-p', '-P', password, str(archive), annotation_member.filename],
            stdout=subprocess.PIPE, check=True).stdout
        for row in rows:
            member_for_image[row['filename']] = find_member('train/data/' + row['filename'])
        image_bytes_needed = sum(entry.file_size for entry in member_for_image.values())
        source_report = {'mode': 'verified_archive', 'archive_path': str(archive),
                         'archive_bytes': archive_size, 'archive_md5': actual_md5,
                         'archive_md5_verified': True,
                         'annotation_member': annotation_member.filename,
                         'annotation_file_sha256': hashlib.sha256(annotations_bytes).hexdigest()}

    require(shutil.disk_usage(work).free >= image_bytes_needed + 256 * 1024**2,
            'مساحة Colab لا تكفي لبناء نسخة Validation المؤقتة.')
    coco = json.loads(annotations_bytes.decode('utf-8-sig'))
    images_by_id = {int(i['id']): i for i in coco['images']}
    require(len(images_by_id) == len(coco['images']), 'معرفات COCO مكررة.')
    categories = {int(c['id']): c['name'] for c in coco['categories']}
    require(set(categories.values()) == set(class_names), 'فئات المصدر مختلفة.')
    selected_ids = {int(r['image_id']) for r in rows}
    all_annotations = defaultdict(list)
    for ann in coco['annotations']:
        if int(ann['image_id']) in selected_ids:
            all_annotations[int(ann['image_id'])].append(ann)

    temporary = Path(tempfile.mkdtemp(prefix='.m5_val_restore_', dir=work))
    (temporary / 'images/val').mkdir(parents=True)
    (temporary / 'labels/val').mkdir(parents=True)
    print('استعادة صور Validation وتحويل المضلعات وفق التقسيم المجمد...')
    try:
        counts = Counter()
        backgrounds = 0
        multi_count = 0
        inventory = []
        for index, row in enumerate(rows, 1):
            image_id = int(row['image_id'])
            require(image_id in images_by_id, 'معرف صورة غير موجود في المصدر.')
            info = images_by_id[image_id]
            require(PurePosixPath(info['file_name']).name == row['filename'], 'معرف COCO واسم الصورة غير متطابقين.')
            width, height = int(info['width']), int(info['height'])
            require(width > 0 and height > 0, 'أبعاد مصدر غير صالحة.')
            export_name = row['original_split'] + '__' + row['filename']
            image_out = temporary / 'images/val' / export_name
            with image_out.open('xb') as stream:
                if raw_ready:
                    with (raw_train / 'data' / row['filename']).open('rb') as source:
                        shutil.copyfileobj(source, stream, 1024 * 1024)
                else:
                    subprocess.run(['unzip', '-p', '-P', password, str(archive),
                                    member_for_image[row['filename']].filename], stdout=stream, check=True)
            with Image.open(image_out) as image:
                require(image.size == (width, height), 'أبعاد الصورة لا تطابق COCO: ' + export_name)
                image.verify()
            if not raw_ready:
                require(image_out.stat().st_size == member_for_image[row['filename']].file_size,
                        'حجم الصورة المستخرجة غير مطابق لعنصر الأرشيف.')
            output = []
            image_counts = Counter()
            for ann in all_annotations[image_id]:
                class_name = categories[int(ann['category_id'])]
                c = class_names.index(class_name)
                raw = ann.get('segmentation', [])
                require(isinstance(raw, list), 'وجد تعليق RLE؛ يلزم مراجعته يدويًا.')
                # Exact polygon filtering, joining, clipping and formatting
                # used by original notebook cell 24.
                polygons = [p for p in raw if len(p) >= 6 and len(p) % 2 == 0
                            and all(math.isfinite(float(v)) for v in p)]
                require(bool(polygons), 'مضلع مفقود: ' + str(ann['id']))
                if len(polygons) > 1:
                    multi_count += 1
                    points = np.concatenate(merge_multi_segment(polygons), axis=0).astype(float)
                else:
                    points = np.array(polygons[0], dtype=float).reshape(-1, 2)
                points[:, 0] = np.clip(points[:, 0] / width, 0, 1)
                points[:, 1] = np.clip(points[:, 1] / height, 0, 1)
                require(len(points) >= 3 and np.isfinite(points).all(), 'مضلع غير صالح.')
                output.append(str(c) + ' ' + ' '.join(f'{v:.8f}' for v in points.reshape(-1)))
                image_counts[class_name] += 1
                counts[class_name] += 1
            require(all(image_counts[name] == int(row[name]) for name in class_names),
                    'عدد أجسام الصورة مختلف عن ملف التقسيم: ' + export_name)
            backgrounds += int(not output)
            label_out = temporary / 'labels/val' / (Path(export_name).stem + '.txt')
            with label_out.open('x', encoding='utf-8', newline='\n') as stream:
                stream.write('\n'.join(output) + ('\n' if output else ''))
            inventory.append({'export_name': export_name, 'image_id': image_id,
                              'image_bytes': image_out.stat().st_size, 'image_sha256': digest(image_out),
                              'label_sha256': digest(label_out), 'objects': len(output)})
            if index % 50 == 0 or index == 451:
                print(f'تمت استعادة {index}/451 صورة')

        expected_images = {'train__' + row['filename'] for row in rows}
        require({p.name for p in (temporary / 'images/val').iterdir()} == expected_images, 'مجموعة الصور غير مطابقة.')
        require({p.name for p in (temporary / 'labels/val').iterdir()} ==
                {Path(n).stem + '.txt' for n in expected_images}, 'مجموعة ملفات التسميات غير مطابقة.')
        require(dict(counts) == expected_objects and sum(counts.values()) == 2242, 'إحصاءات الأجسام النهائية غير مطابقة.')
        require(backgrounds == 8, 'عدد صور الخلفية غير مطابق.')
        (temporary / 'split_manifest.csv').write_bytes(manifest_bytes)
        require(digest(temporary / 'split_manifest.csv') == expected_manifest, 'نسخة ملف التقسيم غير مطابقة.')
        yaml_text = (f'path: {destination}\ntrain: TRAIN_UNAVAILABLE_DO_NOT_TRAIN\nval: images/val\nnames:\n'
                     + ''.join(f'  {i}: {name}\n' for i, name in enumerate(class_names)))
        with (temporary / 'data.yaml').open('x', encoding='utf-8') as stream:
            stream.write(yaml_text)
        for path_text, original_sha in before.items():
            require(digest(Path(path_text)) == original_sha, 'ملف Drive تغير أثناء الاستعادة: ' + path_text)
        report = {'status': 'complete', 'created_at_utc': datetime.now(timezone.utc).isoformat(),
                  'scope': 'VAL_ONLY_RESTORE', 'not_a_full_train_val_test_dataset': True,
                  'dataset_path': str(destination),
                  'train_yaml_entry_is_unavailable_guard': True,
                  'manifest_sha256': expected_manifest, 'class_names': class_names,
                  'classes': class_names, 'images': 451, 'objects': 2242,
                  'objects_by_class': dict(counts), 'backgrounds': backgrounds,
                  'multi_polygon_annotations': multi_count, 'missing_polygon_annotations': 0,
                  'ultralytics_version': version('ultralytics'), 'source': source_report,
                  'drive_reference_hashes_unchanged': before,
                  'notice': 'Restoration only. No training, model inference, Test evaluation or new split selection.'}
        write_json(temporary / 'build_info.json', report)
        write_json(temporary / 'restoration_inventory.json', inventory)
        # Both paths are explicitly constrained to the same local workspace.
        require(temporary.resolve().parent == work.resolve() == destination.resolve().parent,
                'مسار اعتماد البيانات خارج مجلد العمل المحلي.')
        require(not destination.exists(), 'ظهر مجلد الهدف أثناء الاستعادة؛ لن نستبدله.')
        temporary.rename(destination)
        print('\n=== VALIDATION RESTORE COMPLETE ===')
        print('صور Validation: 451 | ملفات التسميات: 451 | الأجسام: 2242 | الخلفيات: 8')
        print('Dataset:', destination)
        print('تم الحفاظ على التقسيم والملفات الأصلية في Drive. لم ينفذ تدريب أو تقييم Test.')
        print('يمكن الآن تشغيل مقارنة M5 مع استخدام مسار dataset أعلاه.')
        return report
    except Exception as error:
        write_json(temporary / 'restoration_failure.json',
                   {'status': 'incomplete', 'error': str(error), 'traceback': traceback.format_exc()})
        print('لم تكتمل الاستعادة. أرسل الخطأ الظاهر. المجلد المؤقت محفوظ:', temporary)
        raise
    finally:
        if zip_handle is not None:
            zip_handle.close()


m5_val_restoration = restore_m5_validation_only()


"""Colab cell: reconstructed pre-finetuning baseline vs frozen M5, VAL only.
No optimizer/training, no Test, no W&B init/log, no replacement of prior results.
The baseline is reconstructed now; it is NOT the original historical epoch-0 file.
"""

def compare_m5_baseline_val(dataset_override=None):
    from pathlib import Path
    from datetime import datetime, timezone
    from importlib.metadata import version
    import contextlib
    import copy
    import csv
    import gc
    import hashlib
    import io
    import json
    import random
    import traceback
    import numpy as np
    import torch
    import yaml
    from ultralytics import YOLO
    from ultralytics.cfg import DEFAULT_CFG_DICT
    from ultralytics.models.yolo.segment.val import SegmentationValidator

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    dataset = Path(dataset_override or '/content/Project9_Phase2/zerowaste_seg_grouped_v2')
    weights = root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt'
    protocol_file = root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json'
    frozen_results = root / 'evaluation/FINAL_TEST_ONCE/results.json'
    dataset_yaml = dataset / 'data.yaml'
    split_file = dataset / 'split_manifest.csv'
    classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    names = dict(enumerate(classes))
    required = [weights, protocol_file, frozen_results, dataset_yaml, split_file]
    missing = [str(path) for path in required if not path.is_file()]
    print('=== M5 BASELINE READINESS ===')
    print('GPU:', torch.cuda.is_available())
    print('Missing required files:', missing)
    assert not missing, 'Required files are missing. Stop and share the readiness output; do not run training or Test.'
    assert callable(getattr(SegmentationValidator, 'run_callbacks', None)), 'Unexpected validator API; stop for review.'

    def file_hash(path):
        value = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()

    def json_write(path, value):
        with path.open('x', encoding='utf-8') as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)

    frozen = json.loads(protocol_file.read_text())
    original_hash = file_hash(weights)
    results_hash = file_hash(frozen_results)
    assert original_hash == frozen.get('m5_checkpoint_sha256'), 'M5 no longer matches the frozen checkpoint.'
    assert file_hash(split_file) == frozen.get('manifest_sha256'), 'Split manifest mismatch.'
    data_config = yaml.safe_load(dataset_yaml.read_text())
    configured_names = data_config['names']
    if isinstance(configured_names, list):
        configured_names = dict(enumerate(configured_names))
    assert {int(k): str(v) for k, v in configured_names.items()} == names, 'Class order mismatch.'
    images = sorted((dataset / 'images/val').glob('*.PNG'))
    labels = sorted((dataset / 'labels/val').glob('*.txt'))
    assert len(images) == len(labels) == 451, 'Expected the complete 451-image VAL split.'
    assert {p.stem for p in images} == {p.stem for p in labels}, 'VAL image/label mismatch.'

    def label_hash():
        digest = hashlib.sha256()
        for path in labels:
            digest.update(path.name.encode() + b'\0' + path.read_bytes() + b'\0')
        return digest.hexdigest()

    val_labels_sha = label_hash()
    now = datetime.now(timezone.utc)
    out = root / 'evaluation' / ('M5_BASELINE_VAL_' + now.strftime('%Y%m%dT%H%M%S%fZ'))
    out.mkdir(parents=True, exist_ok=False)
    # Validator schema requires a train key, but standalone VAL does not use it.
    # A nonexistent sentinel prevents accidental training on validation images.
    # There is deliberately no Test entry.
    val_only_yaml = out / 'VAL_ONLY.yaml'
    assert not (dataset / 'TRAIN_UNAVAILABLE_DO_NOT_TRAIN').exists(), 'Training sentinel unexpectedly exists.'
    with val_only_yaml.open('x', encoding='utf-8') as stream:
        yaml.safe_dump({'path': str(dataset), 'train': 'TRAIN_UNAVAILABLE_DO_NOT_TRAIN', 'val': 'images/val',
                        'names': names}, stream, sort_keys=False)
    device = 0 if torch.cuda.is_available() else 'cpu'
    options = dict(data=str(val_only_yaml), split='val', imgsz=640, batch=8,
                   device=device, conf=0.001, iou=0.50, rect=True, max_det=300,
                   augment=False, agnostic_nms=False, workers=0,
                   plots=True, save_json=False, save_txt=False,
                   project=str(out), exist_ok=False, verbose=False)
    # Use the supported full-precision selector for newer releases; older ones
    # use half=False. Both models always receive the exact same option.
    if 'quantize' in DEFAULT_CFG_DICT:
        options['quantize'] = None
    else:
        options['half'] = False
    plan = {
        'created_at_utc': now.isoformat(), 'split': 'val', 'images': 451,
        'purpose': 'Post-hoc VAL comparison of task-supervised M5 with reconstructed transferred initialization.',
        'baseline': 'COCO pretrained compatible weights; newly initialized material-specific classification tensors; zero optimization steps.',
        'not_historical_epoch_zero': True,
        'limitations': 'Weak untrained-target-head baseline; does not isolate backbone fine-tuning from target-head learning. Not direct semantic evaluation of COCO80.',
        'seed': 42, 'options': options, 'class_names': classes,
        'serialization': 'Baseline saved via Ultralytics.save (may store FP16) then reloaded; both evaluations execute FP32. Transfer provenance describes the pre-save state.',
        'versions': {name: version(name) for name in ('torch', 'ultralytics', 'numpy', 'PyYAML')},
        'frozen_m5_sha256': original_hash, 'frozen_results_sha256': results_hash,
        'validation_dataset_root': str(dataset),
        'dataset_build_info_sha256': file_hash(dataset / 'build_info.json') if (dataset / 'build_info.json').is_file() else None,
        'split_manifest_sha256': file_hash(split_file), 'val_label_tree_sha256': val_labels_sha,
        'tracking': 'Validation callbacks disabled for this local comparison; no W&B experiment is created.',
    }
    json_write(out / 'comparison_protocol.json', plan)
    print('New evidence directory:', out)
    print('Validation only; two evaluations, zero optimizer steps. Device:', device)

    try:
        selected = YOLO(str(weights), task='segment')
        assert {int(k): str(v) for k, v in selected.names.items()} == names
        cfg = copy.deepcopy(selected.model.yaml)
        cfg['nc'] = 4
        yaml_text = yaml.safe_dump(cfg, sort_keys=True)
        (out / 'reconstructed_architecture.yaml').write_text(yaml_text, encoding='utf-8')
        # Downloads the official small COCO checkpoint if it is not cached.
        pretrained = YOLO('yolo11n-seg.pt', task='segment')
        pretrained_path = Path(pretrained.ckpt_path).resolve()
        assert pretrained_path.is_file(), 'Could not locate the downloaded/cached pretrained checkpoint.'
        source_names = {int(k): str(v) for k, v in pretrained.names.items()}
        assert len(source_names) == 80, 'Expected COCO80 pretrained source, not a project checkpoint.'
        assert not set(classes) & set(source_names.values()), 'Unexpected class overlap: review transfer semantics.'
        random.seed(42)
        np.random.seed(42)
        torch.manual_seed(42)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42)
        initial = type(selected.model)(cfg=copy.deepcopy(cfg), ch=3, nc=4, verbose=False)
        initial.names = names.copy()
        transfer_log = io.StringIO()
        with contextlib.redirect_stdout(transfer_log):
            initial.load(pretrained.model, verbose=True)
        initial.names = names.copy()
        source_state, target_state = pretrained.model.state_dict(), initial.state_dict()
        equal_source_keys = []
        shape_mismatches = []
        for key, tensor in target_state.items():
            if key in source_state and tensor.shape == source_state[key].shape:
                if torch.equal(tensor.cpu(), source_state[key].cpu()):
                    equal_source_keys.append(key)
            elif key in source_state:
                shape_mismatches.append({'key': key, 'source': list(source_state[key].shape),
                                         'target': list(tensor.shape)})
        selected_shapes = {k: tuple(v.shape) for k, v in selected.model.state_dict().items()}
        initial_shapes = {k: tuple(v.shape) for k, v in target_state.items()}
        assert selected_shapes == initial_shapes, 'Baseline/fine-tuned architecture shapes differ; stop for review.'
        assert equal_source_keys, 'No pretrained weights transferred.'
        init_info = {
            'pretrained_file_sha256': file_hash(pretrained_path),
            'pretrained_file': str(pretrained_path), 'pretrained_names': source_names,
            'architecture_yaml_sha256': hashlib.sha256(yaml_text.encode()).hexdigest(),
            'load_stdout': transfer_log.getvalue(),
            'target_state_entries': len(target_state),
            'keys_equal_to_pretrained_after_loading': equal_source_keys,
            'source_target_shape_mismatches': shape_mismatches,
            'target_keys_not_equal_to_pretrained': sorted(set(target_state) - set(equal_source_keys)),
            'source_keys_not_in_target': sorted(set(source_state) - set(target_state)),
            'seed': 42, 'optimization_steps': 0,
        }
        json_write(out / 'initialization_provenance.json', init_info)
        del source_state, target_state

        # Use a wrapper whose checkpoint is COCO, then replace its module.
        # Clear inherited checkpoint metadata so EMA cannot override the new model.
        baseline = pretrained
        baseline.model = initial
        baseline.task = 'segment'
        baseline.predictor = None
        baseline.ckpt = {}
        baseline.overrides = {'task': 'segment', 'model': 'reconstructed_material_initialization'}
        initial.args = {**copy.deepcopy(DEFAULT_CFG_DICT), 'task': 'segment', 'mode': 'val',
                        'model': str(out / 'reconstructed_architecture.yaml'),
                        'data': str(val_only_yaml), 'epochs': 0, 'pretrained': str(pretrained_path)}
        baseline_path = out / 'baseline_unfinetuned_4class.pt'
        assert not baseline_path.exists()
        baseline.save(str(baseline_path))
        # Evaluate the serialized baseline, so a future rerun loads the same bytes.
        # Ultralytics save may store FP16; both evaluations request FP32 execution.
        baseline = YOLO(str(baseline_path), task='segment')
        assert {int(k): str(v) for k, v in baseline.names.items()} == names
        assert int(baseline.model.model[-1].nc) == int(selected.model.model[-1].nc) == 4
        assert file_hash(baseline_path) != original_hash

        observed = {}

        def validator_for(expected_model, tag):
            class CheckedVal(SegmentationValidator):
                def run_callbacks(self, event):
                    # Do not publish or alter existing tracking runs.
                    return None

                def init_metrics(self, model):
                    # The backend is passed here; validators need not retain
                    # it as self.model across Ultralytics releases.
                    assert hasattr(model, 'fp16'), 'Cannot verify runtime precision.'
                    self._comparison_fp16 = bool(model.fp16)
                    assert not self._comparison_fp16, 'Expected full-precision execution.'
                    assert {int(k): str(v) for k, v in model.names.items()} == names
                    actual_images = [Path(p).resolve() for p in self.dataloader.dataset.im_files]
                    assert len(actual_images) == 451
                    assert set(actual_images) == {p.resolve() for p in images}, 'VAL image paths differ.'
                    return super().init_metrics(model)

                def __call__(self, *args, **kwargs):
                    assert kwargs.get('model') is expected_model, 'Validator received a different model.'
                    assert self.args.split == 'val'
                    result = super().__call__(*args, **kwargs)
                    assert int(self.seen) == 451, 'Incomplete VAL evaluation.'
                    observed[tag] = {'images_seen': int(self.seen), 'split': self.args.split,
                                     'imgsz': self.args.imgsz, 'conf': self.args.conf,
                                     'iou': self.args.iou, 'rect': self.args.rect,
                                     'max_det': self.args.max_det,
                                     'runtime_fp16': self._comparison_fp16}
                    return result
            return CheckedVal

        measurements = []
        for tag, model in [('baseline_unfinetuned', baseline), ('selected_finetuned', selected)]:
            print('\nEvaluating:', tag, '| VAL 451 images')
            result = model.val(validator=validator_for(model.model, tag), name=tag, **options)
            row = {'model': tag, 'split': 'val', 'images': 451,
                   'box_mAP50': float(result.box.map50), 'box_mAP50_95': float(result.box.map),
                   'mask_mAP50': float(result.seg.map50), 'mask_mAP50_95': float(result.seg.map)}
            assert all(np.isfinite(v) for v in row.values() if isinstance(v, float))
            measurements.append(row)
        assert file_hash(weights) == original_hash, 'Frozen M5 file changed unexpectedly.'
        assert file_hash(frozen_results) == results_hash, 'Frozen Test results changed unexpectedly.'
        assert label_hash() == val_labels_sha, 'VAL labels changed unexpectedly.'
        with (out / 'comparison_VAL.csv').open('x', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(measurements[0]))
            writer.writeheader()
            writer.writerows(measurements)
        report = {'status': 'complete', 'protocol': plan,
                  'baseline_checkpoint_sha256': file_hash(baseline_path),
                  'measurements': measurements, 'observed_validation_conditions': observed,
                  'mask_mAP50_95_difference': measurements[1]['mask_mAP50_95'] - measurements[0]['mask_mAP50_95'],
                  'frozen_model_test_results_and_val_labels_unchanged': True}
        json_write(out / 'comparison_VAL.json', report)
        print('\n=== M5 BASELINE COMPARISON: VALIDATION ONLY ===')
        print(json.dumps(measurements, indent=2))
        print('Mask mAP50-95 difference:', report['mask_mAP50_95_difference'])
        print('Saved:', out / 'comparison_VAL.json')
        print('No training or Test evaluation. Previous experiment files preserved.')
        gc.collect()
        return {'measurements': measurements, 'saved': str(out / 'comparison_VAL.json')}
    except Exception as error:
        json_write(out / 'failure.json', {'status': 'incomplete', 'error_type': type(error).__name__,
                                         'message': str(error), 'traceback': traceback.format_exc()})
        print('Comparison incomplete. Share the error; do not run training or Test.')
        raise


m5_baseline_val_comparison = compare_m5_baseline_val(dataset_override=m5_val_restoration['dataset_path'])


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

"""Colab: preserve VAL restoration metadata and download the completed M5 audit.
No training, prediction, validation, export, or changes to previous results.
Only a new provenance subdirectory is added to the existing comparison folder.
"""


def collect_m5_baseline_evidence():
    from pathlib import Path
    from datetime import datetime, timezone
    import hashlib
    import json
    import zipfile
    from google.colab import files

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    run = root / 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z'
    dataset = Path('/content/Project9_Phase2/zerowaste_seg_grouped_v2_VAL_ONLY_RESTORED')

    def sha(path):
        value = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(block)
        return value.hexdigest()

    def read_json(path):
        return json.loads(path.read_text(encoding='utf-8'))

    required = [run / name for name in (
        'comparison_VAL.json', 'comparison_VAL.csv', 'comparison_protocol.json',
        'initialization_provenance.json', 'baseline_unfinetuned_4class.pt')]
    required += [dataset / name for name in (
        'build_info.json', 'restoration_inventory.json', 'data.yaml', 'split_manifest.csv')]
    missing = [str(path) for path in required if not path.is_file()]
    assert not missing, 'Missing evidence files; share this output: ' + repr(missing)

    report = read_json(run / 'comparison_VAL.json')
    plan = read_json(run / 'comparison_protocol.json')
    assert report['status'] == 'complete' and report['protocol'] == plan
    assert report['frozen_model_test_results_and_val_labels_unchanged'] is True
    assert plan['split'] == 'val' and plan['images'] == 451
    assert Path(plan['validation_dataset_root']).resolve() == dataset.resolve()
    assert sha(dataset / 'build_info.json') == plan['dataset_build_info_sha256']
    assert sha(run / 'baseline_unfinetuned_4class.pt') == report['baseline_checkpoint_sha256']
    assert {row['model'] for row in report['measurements']} == {'baseline_unfinetuned', 'selected_finetuned'}
    for conditions in report['observed_validation_conditions'].values():
        assert conditions['images_seen'] == 451 and conditions['split'] == 'val'
        assert conditions['runtime_fp16'] is False
        assert conditions['conf'] == 0.001 and conditions['iou'] == 0.5
        assert conditions['rect'] is True and conditions['max_det'] == 300

    frozen_dir = root / 'evaluation/FINAL_TEST_ONCE'
    model_path = root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt'
    frozen = read_json(frozen_dir / 'frozen_protocol.json')
    assert sha(model_path) == plan['frozen_m5_sha256'] == frozen['m5_checkpoint_sha256']
    assert sha(frozen_dir / 'results.json') == plan['frozen_results_sha256']
    assert sha(dataset / 'split_manifest.csv') == plan['split_manifest_sha256'] == frozen['manifest_sha256']

    restore = read_json(dataset / 'build_info.json')
    inventory = read_json(dataset / 'restoration_inventory.json')
    assert restore['scope'] == 'VAL_ONLY_RESTORE' and restore['images'] == 451
    assert restore['objects'] == 2242 and restore['backgrounds'] == 8
    assert len(inventory) == len({item['export_name'] for item in inventory}) == 451
    assert sum(item['objects'] for item in inventory) == 2242
    labels_digest = hashlib.sha256()
    labels = sorted((dataset / 'labels/val').glob('*.txt'))
    assert len(labels) == 451
    for path in labels:
        labels_digest.update(path.name.encode() + b'\0' + path.read_bytes() + b'\0')
    assert labels_digest.hexdigest() == plan['val_label_tree_sha256']

    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    provenance = run / ('val_restore_provenance_' + timestamp)
    provenance.mkdir(exist_ok=False)
    for name in ('build_info.json', 'restoration_inventory.json', 'data.yaml', 'split_manifest.csv'):
        source = dataset / name
        with (provenance / name).open('xb') as stream:
            stream.write(source.read_bytes())
        assert sha(provenance / name) == sha(source)

    allowed = {'.json', '.csv', '.yaml', '.yml', '.txt', '.png', '.jpg', '.jpeg'}
    checkpoint = run / 'baseline_unfinetuned_4class.pt'
    candidates = sorted(path for path in run.rglob('*')
                        if path.is_file() and not path.is_symlink()
                        and (path.suffix.lower() in allowed or path == checkpoint))
    assert sum(path.stat().st_size for path in candidates) < 200 * 1024**2, 'Unexpected evidence package size.'
    output = Path('/content') / ('Project9_M5_VAL_Evidence_' + timestamp + '.zip')
    manifest = {'created_at_utc': timestamp, 'comparison_folder': str(run),
                'scope': 'Completed post-hoc VAL comparison, reconstructed baseline checkpoint, plots, and restoration provenance; no full dataset.',
                'reference_checks_passed': True, 'files': []}
    with zipfile.ZipFile(output, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in candidates:
            relative = path.relative_to(run).as_posix()
            data = path.read_bytes()
            archive.writestr('M5_VAL/' + relative, data)
            manifest['files'].append({'path': relative, 'bytes': len(data),
                                      'sha256': hashlib.sha256(data).hexdigest()})
        archive.writestr('collection_manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False))
    with zipfile.ZipFile(output) as archive:
        assert archive.testzip() is None, 'ZIP integrity check failed.'
    print('تم التحقق من ملفات المقارنة وثبات أوزان M5 ونتائج Test.')
    print('حُفظت أدلة استعادة Validation في Drive:', provenance)
    print('ZIP:', output)
    print('حجم الحزمة MiB:', round(output.stat().st_size / 1024**2, 2))
    print('تنزيل الأدلة فقط؛ لم ينفذ تدريب أو تقييم جديد.')
    files.download(str(output))
    return {'zip': str(output), 'provenance': str(provenance), 'files': len(manifest['files'])}


m5_baseline_evidence = collect_m5_baseline_evidence()


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib.util

root = Path('/content/drive/MyDrive/Project9_Phase2')

for model, folder in [
    ('M4', 'M4_yolo11n_640_grouped_v2_seed42'),
    ('M5', 'M5_yolo11n_seg_640_grouped_v2_seed42')
]:
    path = root / 'runs' / folder / 'weights/best.pt'
    print(model, 'موجود:', path.is_file())

val = Path('/content/Project9_Phase2/zerowaste_seg_grouped_v2_VAL_ONLY_RESTORED/images/val')
print('صور Validation:', len(list(val.glob('*.PNG'))))
print('مكتبة YOLO موجودة:', importlib.util.find_spec('ultralytics') is not None)

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

%pip install -q "ultralytics==8.4.213"

import requests
import ultralytics
from pathlib import Path

print('Ultralytics:', ultralytics.__version__)
print('أرشيف محلي:', Path('/content/zerowaste-f-final.zip').is_file())
print('أرشيف Drive:', Path('/content/drive/MyDrive/Project9_Phase2/cache/zerowaste-f-final.zip').is_file())

url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'

try:
    with requests.get(
        url,
        headers={'Range': 'bytes=0-15', 'Accept-Encoding': 'identity'},
        stream=True,
        timeout=30
    ) as response:
        print('HTTP:', response.status_code)
        print('Content-Range:', response.headers.get('Content-Range'))
        supported = response.status_code == 206
        print('يدعم تنزيل أجزاء:', supported)
        if supported:
            print('بايتات الفحص:', len(response.raw.read(16)))
except requests.RequestException as error:
    print('تعذر فحص الاتصال:', str(error))

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# الصق هذا الملف كاملًا في خلية جديدة في جلسة Colab الحالية، ثم شغّل تلك الخلية فقط.
# يتطلب Drive متصلًا و ultralytics==8.4.213، كما في الفحص الناجح الأخير.
# أربع صور Validation، وثمانية تنبؤات على CPU. لا تدريب ولا تقييم Test.
# يجلب عناصر الأرشيف المطلوبة بقراءة جزئية؛ لا ينزّل الأرشيف كاملًا.


# ========================================================================
# prepare_failure_gallery_inputs.py
# ========================================================================
"""Colab preparation: restore four frozen-VAL inputs by verified HTTP ranges.
No model execution, Test evaluation, or complete-archive download.
"""


def prepare_failure_gallery_inputs():
    from collections import Counter, OrderedDict, defaultdict
    from datetime import datetime, timezone
    from importlib.metadata import version
    from pathlib import Path, PurePosixPath
    import csv
    import hashlib
    import io
    import json
    import math
    import re
    import time
    import zipfile
    import numpy as np
    import requests
    from PIL import Image
    from ultralytics.data.converter import merge_multi_segment

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    receipt = root / 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z'
    cache = Path('/content/Project9_Phase2/failure_inputs_VAL_v1')
    names = ['train__07_frame_000001.PNG', 'train__11_frame_000001.PNG',
             'train__07_frame_042800.PNG', 'train__07_frame_037900.PNG']
    classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    expected_manifest_sha = 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50'
    expected_inventory_sha = '86e913fa48c09c596f396ebfaeefa49ff2b720d3cf9983f6645bd3f911734835'
    expected_build_sha = '3e2561df6b6b5072de6273bab3e6cc6d93cb7e61f2d1a9c91a90376a2d182966'
    expected_annotation_sha = '242ba81def2ee6162f7341f0ee48e21e380411e9fac03df26a270d1a869f860a'
    expected_models = {
        'M4_yolo11n_640_grouped_v2_seed42': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
        'M5_yolo11n_seg_640_grouped_v2_seed42': '2cf68941dff6ee01b15ff8947d2f12a8737bdc26e5cc59ec8783e10204179c10',
    }
    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    archive_size = 7518242799
    password = b'REDACTED_DATASET_ZIP_PASSWORD'

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest_bytes(data):
        return hashlib.sha256(data).hexdigest()

    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()

    require(version('ultralytics') == '8.4.213', 'Run the pinned Ultralytics installation cell first.')
    files = {
        'manifest': root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv',
        'inventory': receipt / 'restoration_inventory.json',
        'build': receipt / 'build_info.json',
        'frozen': root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'results': root / 'evaluation/FINAL_TEST_ONCE/results.json',
    }
    missing = [str(path) for path in files.values() if not path.is_file()]
    require(not missing, 'Missing saved provenance; stop and share this output: ' + repr(missing))
    protected = dict(files)
    for run, expected_sha in expected_models.items():
        path = root / 'runs' / run / 'weights/best.pt'
        require(path.is_file() and digest(path) == expected_sha, 'Frozen checkpoint missing or changed: ' + run)
        protected[run] = path
    before = {name: digest(path) for name, path in protected.items()}
    require(before['manifest'] == expected_manifest_sha, 'Frozen manifest hash mismatch.')
    require(before['inventory'] == expected_inventory_sha, 'Saved image/label inventory hash mismatch.')
    require(before['build'] == expected_build_sha, 'Saved restoration record hash mismatch.')
    require(before['frozen'] == 'aca3f6749633875b719c47c8bddff3f79c0a9da28f35246384a79f6d4cf9faff',
            'Frozen protocol bytes changed.')
    require(before['results'] == 'b6f48bf11f862930c4a1d34a4290d2fea3ed3918f2ccc153fef169cb78cb91dc',
            'Original Test-results bytes changed.')
    build = json.loads(files['build'].read_text(encoding='utf-8'))
    require(build['source']['annotation_file_sha256'] == expected_annotation_sha,
            'Unexpected historical COCO annotation hash.')
    require(build['source']['archive_md5_verified'] is True and
            build['source']['archive_md5'] == 'e26e31a58080bca6782dca0e56074c5d', 'Historical source record mismatch.')
    inventory = {item['export_name']: item for item in json.loads(files['inventory'].read_text(encoding='utf-8'))}
    require(len(inventory) == 451 and all(name in inventory for name in names), 'VAL inventory is incomplete.')
    all_rows = list(csv.DictReader(io.StringIO(files['manifest'].read_text(encoding='utf-8-sig'))))
    rows = {}
    for name in names:
        found = [row for row in all_rows if row['original_split'] + '__' + row['filename'] == name]
        require(len(found) == 1 and found[0]['split'] == 'val' and found[0]['original_split'] == 'train',
                'Selected file is not uniquely in frozen VAL: ' + name)
        rows[name] = found[0]

    class HTTPRangeFile(io.RawIOBase):
        """Bounded read-only seekable HTTP file for Python's ZIP64 reader."""
        def __init__(self):
            super().__init__()
            self.position = 0
            self.size = archive_size
            self.block_size = 1024 * 1024
            self.blocks = OrderedDict()
            self.transferred = 0
            self.requests_count = 0
            self.session = requests.Session()

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.position

        def seek(self, offset, whence=0):
            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
            if position < 0:
                raise ValueError('Invalid HTTP ZIP seek.')
            self.position = int(position)
            return self.position

        def get_block(self, index):
            if index in self.blocks:
                self.blocks.move_to_end(index)
                return self.blocks[index]
            start = index * self.block_size
            end = min(self.size - 1, start + self.block_size - 1)
            needed = end - start + 1
            require(self.transferred + needed <= 128 * 1024**2,
                    'Partial-download budget exceeded; the full archive will not be downloaded automatically.')
            for attempt in range(3):
                try:
                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={
                        'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity', 'Cache-Control': 'no-cache'
                    }) as response:
                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
                            time.sleep(2 * (attempt + 1))
                            continue
                        require(response.status_code == 206,
                                f'HTTP {response.status_code}: requested range was not supported; full download prevented.')
                        content_range = response.headers.get('Content-Range', '').strip()
                        match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', content_range, flags=re.I)
                        require(match and tuple(map(int, match.groups())) == (start, end, self.size),
                                'Server returned an unexpected Content-Range: ' + content_range)
                        data = response.raw.read(needed + 1)
                        require(len(data) == needed, 'Incomplete or oversized HTTP range response.')
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
            self.transferred += len(data)
            self.requests_count += 1
            self.blocks[index] = data
            while len(self.blocks) > 16:
                self.blocks.popitem(last=False)
            if self.requests_count % 5 == 0:
                print(f'Partial ZIP transfer: {self.transferred / 1024**2:.1f} MiB')
            return data

        def read(self, count=-1):
            if self.closed:
                raise ValueError('HTTP ZIP stream is closed.')
            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))
            require(count <= 64 * 1024**2, 'Unexpected large ZIP read; complete-archive download prevented.')
            chunks = []
            remaining = count
            while remaining:
                index, offset = divmod(self.position, self.block_size)
                data = self.get_block(index)
                take = min(remaining, len(data) - offset)
                require(take > 0, 'Unexpected ZIP read position.')
                chunks.append(data[offset:offset + take])
                self.position += take
                remaining -= take
            return b''.join(chunks)

        def close(self):
            self.session.close()
            self.blocks.clear()
            super().close()

    cache.mkdir(parents=True, exist_ok=True)
    (cache / 'images').mkdir(exist_ok=True)
    (cache / 'labels').mkdir(exist_ok=True)
    range_file = None
    zip_source = None
    source_mode = 'verified_member_cache'

    def open_zip():
        nonlocal range_file, zip_source, source_mode
        if zip_source is None:
            existing = [Path('/content/zerowaste-f-final.zip'), root / 'cache/zerowaste-f-final.zip']
            local = next((path for path in existing if path.is_file() and path.stat().st_size == archive_size), None)
            if local is not None:
                zip_source = zipfile.ZipFile(local)
                source_mode = 'existing_local_archive_members_verified_by_sha256'
            else:
                print('Reading selected members through HTTP ranges; no full archive download.')
                range_file = HTTPRangeFile()
                zip_source = zipfile.ZipFile(range_file)
                source_mode = 'http_ranges_members_verified_by_sha256'
        return zip_source

    def member_bytes(suffix, expected_sha):
        archive = open_zip()
        choices = [item for item in archive.infolist() if not item.is_dir()
                   and (item.filename == suffix or item.filename.endswith('/' + suffix))
                   and '..' not in PurePosixPath(item.filename).parts]
        require(len(choices) == 1, 'Archive member is not unique: ' + suffix)
        require(choices[0].file_size <= 256 * 1024**2, 'Unexpectedly large selected member.')
        data = archive.read(choices[0], pwd=password)
        require(digest_bytes(data) == expected_sha, 'Selected original member hash mismatch: ' + suffix)
        return data

    def ensure_file(path, expected_sha, getter):
        if path.exists():
            require(path.is_file() and digest(path) == expected_sha, 'Existing cache bytes differ: ' + str(path))
        else:
            data = getter()
            require(digest_bytes(data) == expected_sha, 'Prepared bytes differ: ' + str(path))
            with path.open('xb') as stream:
                stream.write(data)

    try:
        annotation_path = cache / 'source_train_annotations.json'
        ensure_file(annotation_path, expected_annotation_sha,
                    lambda: member_bytes('train/labels.json', expected_annotation_sha))
        coco = json.loads(annotation_path.read_text(encoding='utf-8-sig'))
        image_infos = {int(item['id']): item for item in coco['images']}
        categories = {int(item['id']): item['name'] for item in coco['categories']}
        require(set(categories.values()) == set(classes), 'Source categories differ.')
        selected_ids = {int(rows[name]['image_id']) for name in names}
        annotations = defaultdict(list)
        for annotation in coco['annotations']:
            if int(annotation['image_id']) in selected_ids:
                annotations[int(annotation['image_id'])].append(annotation)
        records = []
        for name in names:
            row = rows[name]
            item = inventory[name]
            image_id = int(row['image_id'])
            require(item['image_id'] == image_id, 'Inventory image ID differs.')
            info = image_infos[image_id]
            require(PurePosixPath(info['file_name']).name == row['filename'], 'COCO filename/image ID mismatch.')
            width, height = int(info['width']), int(info['height'])
            path = cache / 'images' / name
            ensure_file(path, item['image_sha256'],
                        lambda row=row, item=item: member_bytes('train/data/' + row['filename'], item['image_sha256']))
            with Image.open(path) as image:
                require(image.format == 'PNG' and image.size == (width, height), 'Original image size/format mismatch.')
                image.verify()
            lines, boxes, cls_ids, polygons, ann_ids = [], [], [], [], []
            class_counts = Counter()
            for annotation in annotations[image_id]:
                class_name = categories[int(annotation['category_id'])]
                class_id = classes.index(class_name)
                raw = annotation.get('segmentation', [])
                require(isinstance(raw, list), 'Unexpected RLE annotation.')
                parts = [part for part in raw if len(part) >= 6 and len(part) % 2 == 0
                         and all(math.isfinite(float(value)) for value in part)]
                require(bool(parts), 'Missing valid source polygon.')
                points = np.concatenate(merge_multi_segment(parts), axis=0).astype(float) if len(parts) > 1 else np.array(parts[0], dtype=float).reshape(-1, 2)
                points[:, 0] = np.clip(points[:, 0] / width, 0, 1)
                points[:, 1] = np.clip(points[:, 1] / height, 0, 1)
                line = str(class_id) + ' ' + ' '.join(f'{value:.8f}' for value in points.reshape(-1))
                lines.append(line)
                decoded = np.asarray([float(value) for value in line.split()[1:]]).reshape(-1, 2)
                decoded *= np.array([width, height])
                polygons.append(decoded.tolist())
                x, y, w, h = map(float, annotation['bbox'])
                require(all(math.isfinite(value) for value in [x, y, w, h]) and w > 0 and h > 0,
                        'Invalid official COCO box.')
                box = [max(0, min(width, x)), max(0, min(height, y)),
                       max(0, min(width, x + w)), max(0, min(height, y + h))]
                require(box[2] > box[0] and box[3] > box[1], 'Empty clipped official COCO box.')
                boxes.append(box)
                cls_ids.append(class_id)
                ann_ids.append(int(annotation['id']))
                class_counts[class_name] += 1
            require(len(lines) == item['objects'] and all(class_counts[c] == int(row[c]) for c in classes),
                    'Selected annotation counts differ from frozen manifest.')
            label_data = ('\n'.join(lines) + ('\n' if lines else '')).encode('utf-8')
            require(digest_bytes(label_data) == item['label_sha256'], 'Reconstructed exact labels differ from prior inventory.')
            label_path = cache / 'labels' / (Path(name).stem + '.txt')
            ensure_file(label_path, item['label_sha256'], lambda data=label_data: data)
            records.append({'path': str(path), 'label_path': str(label_path), 'export_name': name,
                            'image_id': image_id, 'width': width, 'height': height,
                            'gt_boxes_xyxy': boxes, 'gt_classes': cls_ids, 'gt_polygons': polygons,
                            'annotation_ids': ann_ids, 'image_sha256': item['image_sha256'],
                            'label_sha256': item['label_sha256'], 'gt_yolo_lines': lines})
            print('Verified VAL input:', name, '| objects:', len(lines))
        transferred = range_file.transferred if range_file is not None else 0
        for record in records:
            record['provenance'] = {'source_mode': source_mode, 'archive_url': archive_url,
                'archive_total_bytes': archive_size, 'range_bytes_transferred_this_call': transferred,
                'whole_archive_md5_recomputed_this_call': False,
                'verification': 'Retrieved PNGs and full original train annotation JSON SHA256 match the earlier fully MD5-verified archive receipts; exact converted label hashes also match.',
                'annotation_source_sha256': expected_annotation_sha,
                'saved_inventory_sha256': expected_inventory_sha,
                'split_manifest_sha256': expected_manifest_sha,
                'selection': 'Four fixed purposive images from the prior VAL24 diagnostic; not a representative sample.'}
        require(before == {name: digest(path) for name, path in protected.items()}, 'A protected reference changed.')
        print('=== FOUR VAL INPUTS READY ===')
        print('HTTP data transferred this call (MiB):', round(transferred / 1024**2, 2))
        print('No model inference or Test evaluation in preparation.')
        return records
    finally:
        if zip_source is not None:
            zip_source.close()
        if range_file is not None:
            range_file.close()



# ========================================================================
# failure_gallery_val_core.py
# ========================================================================
"""Post hoc, four-image VAL diagnostic; importing this file performs no work.

Call only after the separate restoration step has verified the original PNGs,
their frozen-manifest VAL membership, and the original eight-decimal YOLO labels.
This module neither restores data nor downloads packages or model weights.

Input contract for ``run_failure_gallery_val(input_records, output_root)``:

``input_records`` is a list containing exactly the four fixed export names below.
Each record is a dictionary with these keys:

* path: local path to the verified, original PNG (not an annotated/resized JPEG).
* export_name: the exact frozen-manifest export filename.
* image_id, width, height: original COCO image metadata, as integers.
* gt_boxes_xyxy: one [x1, y1, x2, y2] pixel-coordinate box per annotation.
* gt_classes: one zero-based class ID per annotation, in CLASS_NAMES order.
* gt_polygons: one Nx2 pixel-coordinate merged polygon per annotation, aligned
  with the same original eight-decimal YOLO label used by the restoration step.
* annotation_ids: the original COCO annotation IDs in the same object order.
* image_sha256, label_sha256: verified original PNG and exact label-file hashes.

Optional keys:

* label_path: exact restored original label file; rehashed independently here.
* gt_yolo_lines: original eight-decimal YOLO label lines, retained verbatim for
  provenance. Their byte hash is NOT reconstructed from lines, because line
  endings and final-newline presence are part of the original file hash.
* provenance: JSON-serializable restoration metadata supplied by the caller.

Without label_path, label_sha256 is explicitly an upstream verification claim.
The caller must validate it against the frozen manifest before calling this.
output_root is the parent output directory and must be the fixed project's
evaluation directory. A fresh FAILURE_GALLERY_VAL_<UTCmicroseconds> child is made.
The function returns its JSON-serializable report and paths, and never initiates
a browser download. It does not call train(), val(), export(), or a Test loader.
"""


def run_failure_gallery_val(input_records, output_root):
    """Infer four preverified VAL images with each frozen FP32 CPU checkpoint.

    Produces eight two-panel JPEGs, protocol/GT/report JSON, per-image CSV and a
    scope README. Failures are written to the new output directory and reraised.
    No global accuracy/AP, deployment benchmark, or error-cause claim is made.
    """
    from datetime import datetime, timezone
    from importlib.metadata import version
    from pathlib import Path
    import csv
    import hashlib
    import json
    import platform
    import traceback

    project_root = Path('/content/drive/MyDrive/Project9_Phase2')
    expected_output_parent = (project_root / 'evaluation').resolve()
    output_parent = Path(output_root).resolve()
    if output_parent != expected_output_parent:
        raise ValueError(f'output_root must be {expected_output_parent}; got {output_parent}')
    if not output_parent.is_dir():
        raise FileNotFoundError(f'Missing existing evaluation directory: {output_parent}')
    started = datetime.now(timezone.utc)
    output_dir = output_parent / ('FAILURE_GALLERY_VAL_' + started.strftime('%Y%m%dT%H%M%S%fZ'))
    output_dir.mkdir(parents=False, exist_ok=False)

    class_names = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    selected_names = [
        'train__07_frame_000001.PNG',
        'train__11_frame_000001.PNG',
        'train__07_frame_042800.PNG',
        'train__07_frame_037900.PNG',
    ]
    expected_manifest_sha = 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50'
    model_specs = {
        'M4': {
            'path': project_root / 'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt',
            'sha256': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
            'task': 'detect',
            'metric': 'box_iou',
        },
        'M5': {
            'path': project_root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt',
            'sha256': '2cf68941dff6ee01b15ff8947d2f12a8737bdc26e5cc59ec8783e10204179c10',
            'task': 'segment',
            'metric': 'full_resolution_binary_mask_iou',
        },
    }
    protocol_path = project_root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json'
    results_path = project_root / 'evaluation/FINAL_TEST_ONCE/results.json'
    manifest_path = project_root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    protected = [protocol_path, results_path, manifest_path] + [spec['path'] for spec in model_specs.values()]
    before_hashes = {}
    forward_counts = {'M4': 0, 'M5': 0}
    rows = []

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha256(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    def save_json(name, value):
        path = output_dir / name
        with path.open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write('\n')
        return str(path)

    def snapshot_hashes():
        return {str(path): sha256(path) for path in protected}

    try:
        import cv2
        import numpy as np
        import torch
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        from matplotlib.lines import Line2D
        from matplotlib.patches import Rectangle
        from PIL import Image
        from scipy.optimize import linear_sum_assignment
        from ultralytics import YOLO
        from ultralytics.models.yolo.detect.predict import DetectionPredictor
        from ultralytics.models.yolo.segment.predict import SegmentationPredictor

        require(version('ultralytics') == '8.4.213', 'This reviewed diagnostic requires ultralytics==8.4.213.')
        for path in protected:
            require(path.is_file(), f'Missing protected source: {path}')
        before_hashes = snapshot_hashes()
        frozen = json.loads(protocol_path.read_text(encoding='utf-8'))
        # The saved Test results are read only as provenance, never evaluated.
        original_results = json.loads(results_path.read_text(encoding='utf-8'))
        require(isinstance(original_results, dict), 'Existing results.json must be a JSON object.')
        require(frozen.get('manifest_sha256') == expected_manifest_sha, 'Frozen manifest SHA mismatch.')
        require(before_hashes[str(manifest_path)] == expected_manifest_sha, 'Actual manifest SHA mismatch.')
        for key, expected in [('confidence', 0.25), ('nms_iou', 0.5), ('imgsz', 640)]:
            require(frozen.get(key) == expected, f'Unexpected frozen protocol {key}.')
        require(frozen.get('split') == 'test', 'Unexpected frozen provenance split metadata.')
        for model_name, spec in model_specs.items():
            require(frozen.get(model_name.lower() + '_checkpoint_sha256') == spec['sha256'],
                    f'{model_name} frozen checkpoint provenance mismatch.')
            require(before_hashes[str(spec['path'])] == spec['sha256'], f'{model_name} actual checkpoint SHA mismatch.')

        require(isinstance(input_records, (list, tuple)) and len(input_records) == 4,
                'Exactly four prepared VAL records are required.')
        require(all(isinstance(record, dict) for record in input_records), 'Every record must be a dictionary.')
        by_name = {record.get('export_name'): record for record in input_records}
        require(len(by_name) == 4 and set(by_name) == set(selected_names), 'The four fixed VAL export names must match exactly.')
        with manifest_path.open(encoding='utf-8', newline='') as stream:
            manifest_rows = list(csv.DictReader(stream))
        selected_manifest_rows = {}
        for name in selected_names:
            matches = [row for row in manifest_rows
                       if row.get('original_split', '') + '__' + row.get('filename', '') == name]
            require(len(matches) == 1 and matches[0].get('split') == 'val',
                    f'{name}: frozen manifest must contain exactly one VAL row.')
            selected_manifest_rows[name] = matches[0]
        records = []
        input_guard_hashes = {}
        for name in selected_names:
            source = by_name[name]
            required = ['path', 'export_name', 'image_id', 'width', 'height', 'gt_boxes_xyxy',
                        'gt_classes', 'gt_polygons', 'annotation_ids', 'image_sha256', 'label_sha256']
            require(all(key in source for key in required), f'{name}: missing required input fields.')
            path = Path(source['path']).resolve()
            require(path.is_file() and path.suffix.lower() == '.png', f'{name}: missing original PNG.')
            require(path.name == name, f'{name}: source basename must be the exact export name.')
            require('test' not in [part.lower() for part in path.parts], f'{name}: Test directory source is forbidden.')
            require(all(isinstance(source[key], int) and not isinstance(source[key], bool)
                        for key in ['image_id', 'width', 'height']), f'{name}: metadata must use integers.')
            require(int(selected_manifest_rows[name]['image_id']) == source['image_id'],
                    f'{name}: image_id differs from its frozen VAL manifest row.')
            width, height = source['width'], source['height']
            require(width > 0 and height > 0, f'{name}: invalid original dimensions.')
            for hash_key in ['image_sha256', 'label_sha256']:
                value = source[hash_key]
                require(isinstance(value, str) and len(value) == 64 and all(ch in '0123456789abcdef' for ch in value),
                        f'{name}: invalid lowercase {hash_key}.')
            image_hash = sha256(path)
            require(image_hash == source['image_sha256'], f'{name}: original PNG SHA mismatch.')
            input_guard_hashes[str(path)] = image_hash
            with Image.open(path) as image:
                require(image.format == 'PNG' and image.size == (width, height), f'{name}: original PNG dimensions/format mismatch.')
                require(getattr(image, 'n_frames', 1) == 1, f'{name}: animated PNG is unsupported.')
            boxes = np.asarray(source['gt_boxes_xyxy'], dtype=np.float64)
            classes = np.asarray(source['gt_classes'])
            annotation_ids = source['annotation_ids']
            polygons = source['gt_polygons']
            n_gt = len(annotation_ids)
            require(boxes.shape == (n_gt, 4), f'{name}: GT box array must be N by 4.')
            require(classes.shape == (n_gt,) and np.issubdtype(classes.dtype, np.integer), f'{name}: invalid GT classes.')
            require(len(polygons) == n_gt and len(set(annotation_ids)) == n_gt, f'{name}: annotation alignment/IDs invalid.')
            require(all(isinstance(value, int) and not isinstance(value, bool) for value in annotation_ids),
                    f'{name}: annotation IDs must be integers.')
            require(np.isfinite(boxes).all(), f'{name}: nonfinite GT boxes.')
            require(((classes >= 0) & (classes < 4)).all(), f'{name}: class outside frozen order.')
            require(((boxes[:, 0] >= 0) & (boxes[:, 1] >= 0) & (boxes[:, 2] <= width)
                     & (boxes[:, 3] <= height) & (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])).all(),
                    f'{name}: invalid or out-of-bounds GT box.')
            validated_polygons, gt_masks = [], []
            for index, polygon in enumerate(polygons):
                points = np.asarray(polygon, dtype=np.float64)
                require(points.ndim == 2 and points.shape[1] == 2 and len(points) >= 3
                        and np.isfinite(points).all(), f'{name}: invalid polygon {index}.')
                require(((points[:, 0] >= 0) & (points[:, 0] <= width)
                         & (points[:, 1] >= 0) & (points[:, 1] <= height)).all(), f'{name}: polygon outside original image.')
                mask = np.zeros((height, width), dtype=np.uint8)
                cv2.fillPoly(mask, [np.rint(points).astype(np.int32)], 1)
                require(bool(mask.any()), f'{name}: empty rasterized GT polygon {index}.')
                validated_polygons.append(points.tolist())
                gt_masks.append(mask.astype(bool))
            gt_masks = np.stack(gt_masks) if gt_masks else np.empty((0, height, width), dtype=bool)
            label_verification = 'upstream_restoration_verified_against_frozen_manifest'
            label_path = None
            if source.get('label_path') is not None:
                label_path = Path(source['label_path']).resolve()
                require(label_path.is_file(), f'{name}: missing original label file.')
                require(sha256(label_path) == source['label_sha256'], f'{name}: original label SHA mismatch.')
                input_guard_hashes[str(label_path)] = source['label_sha256']
                label_verification = 'independently_rehashed_here_after_upstream_frozen_manifest_verification'
            if 'gt_yolo_lines' in source:
                require(isinstance(source['gt_yolo_lines'], list) and len(source['gt_yolo_lines']) == n_gt
                        and all(isinstance(line, str) and '\n' not in line and '\r' not in line for line in source['gt_yolo_lines']),
                        f'{name}: optional original label lines must align with GT annotations.')
            public = {
                'path': str(path), 'export_name': name, 'image_id': source['image_id'],
                'width': width, 'height': height, 'split': 'val',
                'gt_boxes_xyxy': boxes.tolist(), 'gt_classes': classes.astype(int).tolist(),
                'gt_polygons': validated_polygons, 'annotation_ids': annotation_ids,
                'image_sha256': source['image_sha256'], 'label_sha256': source['label_sha256'],
                'label_sha256_verification': label_verification,
                'label_path': str(label_path) if label_path is not None else None,
                'frozen_manifest_row': selected_manifest_rows[name],
            }
            for optional in ['gt_yolo_lines', 'provenance']:
                if optional in source:
                    public[optional] = source[optional]
            records.append({'public': public, 'boxes': boxes, 'classes': classes.astype(int), 'masks': gt_masks})

        options = dict(imgsz=640, conf=0.25, iou=0.5, rect=True, max_det=300,
                       quantize=None, device='cpu', augment=False,
                       agnostic_nms=False, batch=1, save=False, save_txt=False,
                       save_conf=False, save_crop=False, show=False, verbose=False)
        scope = (
            'Four purposively selected original VAL images from the previous 24-image VAL audit, '
            'including known misses and a rare metal example. This is neither random nor representative. '
            'TP/FP/FN are per-image post hoc diagnostic counts under the explicit matching criteria only; '
            'no AP, global performance, Test result, deployment speed or HTTP benchmark is inferred. '
            'Error causes require later manual visual review. No errors are fabricated if none are observed.'
        )
        protocol = {
            'created_at_utc': started.isoformat(), 'status': 'prepared_before_inference',
            'split': 'val', 'input_image_count': 4, 'expected_total_forward_calls': 8,
            'selected_export_names_in_order': selected_names, 'selection': scope,
            'class_names_in_id_order': class_names,
            'versions': {key: version(key) for key in ['ultralytics', 'torch', 'numpy', 'scipy', 'matplotlib', 'Pillow']},
            'python': platform.python_version(),
            'sources': {key: {'path': str(spec['path']), 'sha256': spec['sha256'], 'task': spec['task']}
                        for key, spec in model_specs.items()},
            'checkpoint_scope': 'Frozen source .pt checkpoints; this new GT diagnostic is not an HTTP deployment benchmark.',
            'm5_http_note': (
                'The previously audited HTTP source.pt had a different file hash. A separate CPU state audit '
                'found all 561 loaded state tensors equal; that historical finding does not replace source hashes here.'
            ),
            'frozen_protocol_reference_only': {'path': str(protocol_path), 'sha256': before_hashes[str(protocol_path)],
                                               'content': frozen, 'split_is_not_used_for_inference': True},
            'existing_results_reference_only': {'path': str(results_path), 'sha256': before_hashes[str(results_path)]},
            'manifest': {'path': str(manifest_path), 'sha256': expected_manifest_sha,
                         'val_membership': 'independently checked here for all four export names and image IDs',
                         'original_label_validation': 'performed by caller restoration step'},
            'prediction_options_common': options, 'prediction_options_M5_extra': {'retina_masks': True},
            'actual_precision_requirement': 'predictor.model.fp16 must be False; CPU input and floating parameters are float32',
            'callbacks': 'Custom DetectionPredictor/SegmentationPredictor suppress all prediction callbacks.',
            'matching': {
                'M4': 'Same class AND box IoU >= 0.50.',
                'M5': 'Same class AND full-resolution binary mask IoU >= 0.50.',
                'assignment': 'Maximum cardinality, then maximum valid-pair total IoU; Hungarian score = where(valid, min(Ngt,Npred)+1+IoU, 0); invalid pairs discarded.',
                'iou_matrix_axes': 'rows = zero-based GT indices; columns = zero-based prediction indices',
                'gt_mask_rasterization': 'cv2.fillPoly(np.rint(pixel_points).astype(np.int32), value=1) at original H,W',
                'predicted_mask_binary_threshold': '> 0.5; full N,H,W shape required; no resizing',
                'separation_from_prior_audit': 'These IoU>=0.50 diagnostic matches are separate from the earlier VAL24 any-intersection rule and saved Test AP.',
            },
            'protected_sources_sha256_before': before_hashes,
            'inputs_sha256_before': input_guard_hashes,
            'output_directory': str(output_dir),
        }
        protocol_saved = save_json('protocol.json', protocol)
        gt_saved = save_json('selected_ground_truth.json', {
            'class_names_in_id_order': class_names, 'split': 'val', 'scope': scope,
            'input_contract': 'Caller verified membership, original files and aligned annotation/label conversion before this function.',
            'images': [record['public'] for record in records],
        })
        readme = (
            '# Four-image VAL diagnostic gallery\n\n' + scope + '\n\n'
            'Eight figures compare ground truth (top) with predictions (bottom): four for M4 boxes and four for M5 masks. '
            'Green outlines indicate matched instances; red outlines indicate an unmatched GT (FN) or prediction (FP). '
            'GT and prediction indices are zero-based and correspond to report.json. A label with no FN/FP has a valid match.\n\n'
            'Prediction settings: CPU FP32, imgsz=640, confidence=0.25, NMS IoU=0.50, rect=True, '
            'max_det=300, no augmentation, class-aware NMS, quantize=None; M5 retina_masks=True. '
            'M4 uses same-class box IoU>=0.50 and M5 uses same-class original-resolution binary-mask IoU>=0.50. '
            'Assignment maximizes valid one-to-one match count before summed IoU. GT masks use '
            'cv2.fillPoly with np.rint integer coordinates; predicted masks use >0.5 without resizing.\n\n'
            'The prior VAL24 audit used an any-intersection rule; its counts are not directly comparable. '
            'Frozen Test metadata is read solely to verify source provenance. No Test images are loaded or evaluated. '
            'The original weights, manifest, frozen protocol and saved results are hashed before and after.\n\n'
            'Files: protocol.json (written before inference), selected_ground_truth.json, report.json, '
            'per_image.csv, eight JPEGs and artifacts_sha256.json. Raw original PNGs and full mask pixels are not copied.\n'
        )
        with (output_dir / 'README.md').open('x', encoding='utf-8', newline='\n') as stream:
            stream.write(readme)

        class SilentDetectionPredictor(DetectionPredictor):
            def run_callbacks(self, event):
                return None

        class SilentSegmentationPredictor(SegmentationPredictor):
            def run_callbacks(self, event):
                return None

        def box_iou(gt, pred):
            if not len(gt) or not len(pred):
                return np.zeros((len(gt), len(pred)), dtype=np.float64)
            low = np.maximum(gt[:, None, :2], pred[None, :, :2])
            high = np.minimum(gt[:, None, 2:], pred[None, :, 2:])
            intersection = np.maximum(high - low, 0).prod(axis=2)
            areas_gt = np.maximum(gt[:, 2:] - gt[:, :2], 0).prod(axis=1)
            areas_pred = np.maximum(pred[:, 2:] - pred[:, :2], 0).prod(axis=1)
            union = areas_gt[:, None] + areas_pred[None, :] - intersection
            return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

        def mask_iou(gt, pred):
            overlaps = np.zeros((len(gt), len(pred)), dtype=np.float64)
            gt_areas = np.count_nonzero(gt, axis=(1, 2))
            pred_areas = np.count_nonzero(pred, axis=(1, 2))
            for gi in range(len(gt)):
                for pi in range(len(pred)):
                    intersection = int(np.count_nonzero(gt[gi] & pred[pi]))
                    union = int(gt_areas[gi] + pred_areas[pi]) - intersection
                    overlaps[gi, pi] = intersection / union if union else 0.0
            return overlaps

        def match_instances(overlaps, gt_classes, pred_classes):
            valid = (gt_classes[:, None] == pred_classes[None, :]) & (overlaps >= 0.5)
            pairs = []
            if overlaps.shape[0] and overlaps.shape[1]:
                # One added valid pair outweighs every possible IoU tie-break gain.
                score = np.where(valid, min(overlaps.shape) + 1 + overlaps, 0.0)
                gt_indices, pred_indices = linear_sum_assignment(-score)
                pairs = [(int(gi), int(pi), float(overlaps[gi, pi]))
                         for gi, pi in zip(gt_indices, pred_indices) if valid[gi, pi]]
            return pairs

        def names_in_order(names):
            if isinstance(names, dict):
                require(set(int(key) for key in names) == set(range(4)), 'Unexpected class-name keys.')
                return [str(names.get(index, names.get(str(index)))) for index in range(4)]
            return list(names)

        def unpack(result, record, segmentation):
            public = record['public']
            shape = (public['height'], public['width'])
            require(tuple(result.orig_shape) == shape, 'Result original dimensions changed.')
            require(names_in_order(result.names) == class_names, 'Prediction class names/order mismatch.')
            require(result.boxes is not None, 'Missing prediction boxes container.')
            pred_boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float64)
            raw_classes = result.boxes.cls.detach().cpu().numpy()
            confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float64)
            require(pred_boxes.shape == (len(confidences), 4) and raw_classes.shape == (len(confidences),),
                    'Prediction arrays are misaligned.')
            require(np.isfinite(pred_boxes).all() and np.isfinite(confidences).all() and np.isfinite(raw_classes).all(),
                    'Nonfinite prediction values.')
            require((raw_classes == np.rint(raw_classes)).all(), 'Nonintegral prediction class.')
            pred_classes = raw_classes.astype(int)
            require(((pred_classes >= 0) & (pred_classes < 4)).all(), 'Prediction class outside frozen order.')
            require(((confidences >= 0.25 - 1e-7) & (confidences <= 1)).all(), 'Confidence outside operating threshold/range.')
            pred_masks = None
            if segmentation:
                if not len(pred_boxes):
                    require(result.masks is None or len(result.masks.data) == 0, 'Masks exist without boxes.')
                    pred_masks = np.empty((0, *shape), dtype=bool)
                else:
                    require(result.masks is not None, 'M5 boxes exist without masks.')
                    data = result.masks.data.detach().cpu().numpy()
                    require(data.shape == (len(pred_boxes), *shape),
                            f'M5 mask shape {data.shape} must equal {(len(pred_boxes), *shape)}; no silent resizing.')
                    require(np.isfinite(data).all(), 'Nonfinite M5 mask values.')
                    pred_masks = data > 0.5
            return pred_boxes, pred_classes, confidences, pred_masks

        def draw_figure(record, model_name, pred_boxes, pred_classes, confidences, pred_masks, matched_gt, matched_pred, counts):
            public = record['public']
            with Image.open(public['path']) as image:
                background = np.asarray(image.convert('RGB'))
            green, red = '#13993f', '#e0292c'
            fig, axes = plt.subplots(2, 1, figsize=(12, 15.2), dpi=160)
            try:
                for ax in axes:
                    ax.imshow(background)
                    ax.set_xlim(-0.5, public['width'] - 0.5)
                    ax.set_ylim(public['height'] - 0.5, -0.5)
                    ax.set_aspect('equal')
                    ax.axis('off')

                def outline(ax, box, mask, label, color):
                    if mask is not None:
                        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
                        rgb = matplotlib.colors.to_rgb(color)
                        overlay[mask, :3] = rgb
                        overlay[mask, 3] = 0.18
                        ax.imshow(overlay)
                        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                        for contour in contours:
                            points = contour[:, 0, :]
                            if len(points) > 1:
                                points = np.vstack([points, points[0]])
                                ax.plot(points[:, 0], points[:, 1], color=color, linewidth=1.6)
                    else:
                        ax.add_patch(Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                               fill=False, edgecolor=color, linewidth=2))
                    x = float(np.clip(box[0], 0, max(0, public['width'] - 1)))
                    y = float(np.clip(box[1], 0, max(0, public['height'] - 1)))
                    ax.text(x, y, label, fontsize=14, color='white', va='top', ha='left',
                            bbox={'facecolor': color, 'alpha': 0.9, 'edgecolor': 'white', 'pad': 2},
                            clip_on=True)

                for index, (box, cls) in enumerate(zip(record['boxes'], record['classes'])):
                    matched = index in matched_gt
                    label = f'GT#{index}' + ('' if matched else ' FN') + f' {class_names[int(cls)]}'
                    outline(axes[0], box, record['masks'][index] if model_name == 'M5' else None,
                            label, green if matched else red)
                for index, (box, cls, confidence) in enumerate(zip(pred_boxes, pred_classes, confidences)):
                    matched = index in matched_pred
                    label = f'Pred#{index}' + ('' if matched else ' FP') + f' {class_names[int(cls)]} {confidence:.3f}'
                    outline(axes[1], box, pred_masks[index] if model_name == 'M5' else None,
                            label, green if matched else red)
                axes[0].set_title('Ground truth', fontsize=17)
                axes[1].set_title('Predictions', fontsize=17)
                criterion = 'box IoU' if model_name == 'M4' else 'full-resolution mask IoU'
                observed = 'No errors observed under this rule' if counts['fp'] == counts['fn'] == 0 else 'Observed unmatched instances under this rule'
                fig.suptitle(
                    f"{public['export_name']} | VAL | {model_name}\n"
                    f"conf >= 0.25 | NMS IoU = 0.50 | same-class {criterion} >= 0.50 | "
                    f"TP={counts['tp']}  FP={counts['fp']}  FN={counts['fn']}", fontsize=15, y=0.98)
                fig.legend(handles=[Line2D([0], [0], color=green, lw=3, label='Matched GT / TP prediction'),
                                    Line2D([0], [0], color=red, lw=3, label='Unmatched GT = FN / prediction = FP')],
                           loc='lower center', bbox_to_anchor=(0.5, 0.045), ncol=1, fontsize=13, frameon=False)
                fig.text(0.5, 0.022, observed + ' | Purposive four-image diagnostic; no global performance claim.',
                         ha='center', fontsize=10)
                fig.subplots_adjust(top=0.90, bottom=0.12, left=0.015, right=0.985, hspace=0.13)
                path = output_dir / f"{model_name}__{Path(public['export_name']).stem}.jpg"
                require(not path.exists(), f'Refusing existing figure path: {path}')
                fig.savefig(path, format='jpeg', dpi=160, facecolor='white', pil_kwargs={'quality': 93, 'subsampling': 0})
                return str(path)
            finally:
                plt.close(fig)

        for model_name, spec in model_specs.items():
            predictor_type = SilentDetectionPredictor if model_name == 'M4' else SilentSegmentationPredictor
            model = YOLO(str(spec['path']), task=spec['task'])
            require(names_in_order(model.names) == class_names, f'{model_name}: checkpoint class order differs.')
            require(next(model.model.parameters()).device.type == 'cpu', f'{model_name}: checkpoint must initially be on CPU.')
            # Suppress model-level callback collections too. The subclass stops dispatch.
            model.callbacks.clear()
            model_forward_count = [0]

            def count_forward(_module, _args):
                model_forward_count[0] += 1

            handle = model.model.register_forward_pre_hook(count_forward)
            try:
                for image_number, record in enumerate(records, 1):
                    call_options = dict(options)
                    if model_name == 'M5':
                        call_options['retina_masks'] = True
                    old_count = model_forward_count[0]
                    prediction_results = model.predict(source=record['public']['path'], predictor=predictor_type,
                                                       stream=False, **call_options)
                    require(model_forward_count[0] - old_count == 1,
                            f'{model_name}: expected one CPU forward per selected image; observed {model_forward_count[0] - old_count}.')
                    require(len(prediction_results) == 1, 'Each selected image must produce exactly one result.')
                    require(model.predictor.model.fp16 is False, f'{model_name}: actual predictor is not FP32.')
                    require(model.predictor.model.device.type == 'cpu', f'{model_name}: actual predictor is not on CPU.')
                    require(all(parameter.dtype == torch.float32 for parameter in model.predictor.model.model.parameters()
                                if parameter.is_floating_point()), f'{model_name}: non-FP32 floating parameter.')
                    pred_boxes, pred_classes, confidences, pred_masks = unpack(prediction_results[0], record, model_name == 'M5')
                    overlaps = box_iou(record['boxes'], pred_boxes) if model_name == 'M4' else mask_iou(record['masks'], pred_masks)
                    pairs = match_instances(overlaps, record['classes'], pred_classes)
                    matched_gt = {gi for gi, _, _ in pairs}
                    matched_pred = {pi for _, pi, _ in pairs}
                    counts = {'tp': len(pairs), 'fp': len(pred_boxes) - len(pairs), 'fn': len(record['boxes']) - len(pairs)}
                    figure_path = draw_figure(record, model_name, pred_boxes, pred_classes, confidences,
                                              pred_masks, matched_gt, matched_pred, counts)
                    gt_to_pred = {gi: (pi, iou) for gi, pi, iou in pairs}
                    pred_to_gt = {pi: (gi, iou) for gi, pi, iou in pairs}
                    public = record['public']
                    result_record = {
                        'model': model_name, 'image': public['export_name'], 'image_id': public['image_id'], 'split': 'val',
                        'width': public['width'], 'height': public['height'],
                        'image_sha256': public['image_sha256'], 'label_sha256': public['label_sha256'],
                        'checkpoint_sha256': spec['sha256'], 'counts': counts,
                        'observed_status': 'no_errors_observed_under_explicit_rule' if counts['fp'] == counts['fn'] == 0 else 'unmatched_instances_observed_under_explicit_rule',
                        'criterion': spec['metric'], 'iou_threshold': 0.5, 'confidence_threshold': 0.25,
                        'iou_matrix': overlaps.tolist(), 'iou_matrix_shape': list(overlaps.shape),
                        'iou_matrix_axes': {'rows': 'gt_index', 'columns': 'prediction_index'},
                        'matches': [{'gt_index': gi, 'prediction_index': pi, 'iou': iou,
                                     'annotation_id': public['annotation_ids'][gi], 'class_id': int(record['classes'][gi])}
                                    for gi, pi, iou in pairs],
                        'ground_truth': [
                            {'index': gi, 'annotation_id': public['annotation_ids'][gi], 'class_id': int(cls),
                             'class_name': class_names[int(cls)], 'box_xyxy': box.tolist(),
                             'status': 'matched' if gi in matched_gt else 'FN',
                             'matched_prediction_index': gt_to_pred[gi][0] if gi in gt_to_pred else None,
                             'matched_iou': gt_to_pred[gi][1] if gi in gt_to_pred else None,
                             'mask_area_pixels': int(np.count_nonzero(record['masks'][gi])) if model_name == 'M5' else None}
                            for gi, (box, cls) in enumerate(zip(record['boxes'], record['classes']))
                        ],
                        'predictions': [
                            {'index': pi, 'class_id': int(cls), 'class_name': class_names[int(cls)],
                             'confidence': float(confidence), 'box_xyxy': box.tolist(),
                             'status': 'TP' if pi in matched_pred else 'FP',
                             'matched_gt_index': pred_to_gt[pi][0] if pi in pred_to_gt else None,
                             'matched_annotation_id': public['annotation_ids'][pred_to_gt[pi][0]] if pi in pred_to_gt else None,
                             'matched_iou': pred_to_gt[pi][1] if pi in pred_to_gt else None,
                             'mask_area_pixels': int(np.count_nonzero(pred_masks[pi])) if model_name == 'M5' else None}
                            for pi, (box, cls, confidence) in enumerate(zip(pred_boxes, pred_classes, confidences))
                        ],
                        'figure_path': figure_path,
                        'actual_runtime': {'device': str(model.predictor.model.device), 'fp16': model.predictor.model.fp16,
                                           'floating_parameters_dtype': 'float32', 'forward_calls_this_image': 1},
                    }
                    rows.append(result_record)
                    print(f"{model_name} VAL {image_number}/4: {public['export_name']} TP={counts['tp']} FP={counts['fp']} FN={counts['fn']}")
                    del prediction_results, pred_masks
                forward_counts[model_name] = model_forward_count[0]
                require(model_forward_count[0] == 4, f'{model_name}: expected exactly four forward calls.')
            finally:
                forward_counts[model_name] = model_forward_count[0]
                handle.remove()
                del model

        require(len(rows) == 8 and sum(forward_counts.values()) == 8, 'Expected eight image/model diagnostics and eight forwards.')
        after_hashes = snapshot_hashes()
        require(before_hashes == after_hashes, 'Protected source changed during diagnostic; review immediately.')
        inputs_after = {path: sha256(path) for path in input_guard_hashes}
        require(input_guard_hashes == inputs_after, 'A selected original PNG or label changed during diagnostic.')
        counts_by_model = {
            model_name: {key: sum(row['counts'][key] for row in rows if row['model'] == model_name) for key in ['tp', 'fp', 'fn']}
            for model_name in model_specs
        }
        csv_path = output_dir / 'per_image.csv'
        with csv_path.open('x', encoding='utf-8', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=['model', 'image', 'split', 'criterion', 'confidence_threshold',
                                                        'iou_threshold', 'gt_count', 'prediction_count', 'tp', 'fp', 'fn',
                                                        'image_sha256', 'label_sha256', 'figure_path'])
            writer.writeheader()
            for row in rows:
                writer.writerow({key: row[key] for key in ['model', 'image', 'split', 'criterion', 'confidence_threshold',
                                                           'iou_threshold', 'image_sha256', 'label_sha256', 'figure_path']}
                                | {'gt_count': len(row['ground_truth']), 'prediction_count': len(row['predictions']), **row['counts']})
        report = {
            'status': 'complete', 'created_at_utc': started.isoformat(),
            'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            'output_directory': str(output_dir), 'scope': scope, 'split': 'val',
            'image_count': 4, 'model_image_diagnostics': 8, 'forward_calls': forward_counts,
            'counts_within_these_four_images_only': counts_by_model,
            'counts_are_not_global_performance_estimates': True,
            'manual_error_cause_review': 'pending; no causal labels inferred automatically',
            'protocol_path': protocol_saved, 'ground_truth_path': gt_saved, 'per_image_csv_path': str(csv_path),
            'figure_paths': [row['figure_path'] for row in rows], 'images': rows,
            'protected_sources_sha256_before': before_hashes, 'protected_sources_sha256_after': after_hashes,
            'protected_sources_unchanged': True,
            'inputs_sha256_after': inputs_after, 'selected_inputs_unchanged': True,
        }
        report_saved = save_json('report.json', report)
        artifacts = {path.name: sha256(path) for path in sorted(output_dir.iterdir()) if path.is_file()}
        artifacts_saved = save_json('artifacts_sha256.json', {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'note': 'This inventory excludes itself; original PNGs and checkpoints are not copied.',
            'files': artifacts,
        })
        return {'output_directory': str(output_dir), 'report_path': report_saved,
                'protocol_path': protocol_saved, 'ground_truth_path': gt_saved,
                'per_image_csv_path': str(csv_path), 'artifacts_manifest_path': artifacts_saved,
                'figure_paths': report['figure_paths'], 'counts': counts_by_model,
                'forward_calls': forward_counts, 'report': report}
    except Exception as error:
        failure = {
            'status': 'failed', 'started_at_utc': started.isoformat(),
            'failed_at_utc': datetime.now(timezone.utc).isoformat(), 'output_directory': str(output_dir),
            'error_type': type(error).__name__, 'error_message': str(error),
            'traceback': traceback.format_exc(), 'forward_calls': forward_counts,
            'completed_model_image_diagnostics': len(rows), 'protected_sources_sha256_before': before_hashes,
        }
        try:
            failure['protected_sources_sha256_at_failure'] = {
                str(path): sha256(path) if path.is_file() else None for path in protected
            }
            if before_hashes:
                failure['protected_sources_unchanged'] = before_hashes == failure['protected_sources_sha256_at_failure']
        except Exception as hash_error:
            failure['failure_hash_check_error'] = str(hash_error)
        save_json('failure.json', failure)
        raise



# ========================================================================
# run_failure_gallery_entry.py
# ========================================================================
"""Colab entry point: execute after the two reviewed function definitions."""

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import zipfile

print('=== M4 / M5: FOUR VALIDATION IMAGES ONLY ===')
print('CPU FP32 | no training | no Test evaluation | no full archive download')
fg_inputs = prepare_failure_gallery_inputs()
fg_result = run_failure_gallery_val(
    fg_inputs, Path('/content/drive/MyDrive/Project9_Phase2/evaluation')
)

fg_output = Path(fg_result['output_directory'])
fg_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
fg_zip_path = Path('/content') / ('Project9_FAILURE_GALLERY_VAL_' + fg_stamp + '.zip')
fg_files = sorted(fg_output.iterdir())
if not fg_files or not all(path.is_file() for path in fg_files):
    raise RuntimeError('Unexpected gallery output structure; packaging stopped.')
fg_receipts = []
for fg_path in fg_files:
    fg_data = fg_path.read_bytes()
    fg_receipts.append({'file': fg_path.name, 'bytes': len(fg_data),
                        'sha256': hashlib.sha256(fg_data).hexdigest()})
fg_collection = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'split': 'val', 'image_count': 4, 'model_image_diagnostics': 8,
    'files': fg_receipts,
    'note': 'The file inventory excludes this collection_manifest.json itself. '
            'Original PNGs, full source annotation JSON, and model weights are not packaged.',
}
with zipfile.ZipFile(fg_zip_path, 'x', compression=zipfile.ZIP_DEFLATED) as fg_archive:
    for fg_path in fg_files:
        fg_archive.write(fg_path, arcname=fg_path.name)
    fg_archive.writestr('collection_manifest.json',
                       json.dumps(fg_collection, indent=2, ensure_ascii=False, allow_nan=False) + '\n')
with zipfile.ZipFile(fg_zip_path) as fg_archive:
    if fg_archive.testzip() is not None:
        raise RuntimeError('ZIP integrity check failed; download stopped.')
    for fg_receipt in fg_receipts:
        fg_data = fg_archive.read(fg_receipt['file'])
        if (len(fg_data) != fg_receipt['bytes'] or
                hashlib.sha256(fg_data).hexdigest() != fg_receipt['sha256']):
            raise RuntimeError('Packaged evidence hash mismatch; download stopped.')

print('=== GALLERY COMPLETE ===')
print('Counts concern these four purposively selected images only:')
print(json.dumps(fg_result['counts'], indent=2, ensure_ascii=False))
print('Saved in Drive:', fg_output)
print('ZIP ready:', fg_zip_path)
print('Protected source files and saved Test results were unchanged.')
print('Downloading the ZIP. If no download appears, find it in the Colab Files panel.')
from google.colab import files
files.download(str(fg_zip_path))



In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# انسخ الملف كاملًا إلى خلية جديدة في جلسة Colab الحالية وشغّل تلك الخلية فقط.
# يتصل بـDrive ويجهز الجلسة ويستعيد صورة واحدة عند الحاجة دون تنزيل مجموعة البيانات كاملة.
# قياس HTTP على صورة Validation واحدة: طلبان للتهيئة و25 طلبًا للقياس.
# لا تدريب ولا تقييم Test.
# يستخدم ملفات خدمة HTTP الأصلية وأوزانها دون تعديلها.

_M8_TIMING_SUPPORT_SOURCE = '"""Read-only profiling support for the isolated Project9 Colab HTTP service.\n\nNo model imports, model execution, network calls, or automatic file writes occur\non import. The instrumented /predict route obtains ``trace_context.get()`` and\nsets three fields before returning its ORIGINAL response dictionary:\n\n    context[\'stages\'] = {\n        \'preprocess_ms\': ...,\n        \'m4_predict_call_ms\': ...,\n        \'m5_predict_call_ms\': ...,\n        \'application_postprocess_ms\': ...,\n    }\n    context[\'route_total_ms\'] = ...\n    context[\'runtime\'] = {... actual device / fp16 / parameter dtypes ...}\n\nWrap the FastAPI app with TimingASGI. Only POST /predict is profiled. Such\nrequests must carry x-project9-profile-id, by default beginning ``p9stage-``.\nAll response messages and body bytes are passed through unchanged.\n"""\n\nfrom contextvars import ContextVar\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom time import perf_counter_ns\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport traceback\n\n\ntrace_context = ContextVar(\'project9_stage_timing_context\', default=None)\n\nSTAGE_KEYS = (\n    \'preprocess_ms\',\n    \'m4_predict_call_ms\',\n    \'m5_predict_call_ms\',\n    \'application_postprocess_ms\',\n)\n\n\ndef _nonnegative_finite(value, name):\n    if isinstance(value, bool) or not isinstance(value, (int, float)):\n        raise RuntimeError(f\'{name} must be a finite numeric duration.\')\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise RuntimeError(f\'{name} must be a nonnegative finite duration.\')\n    return value\n\n\nclass TimingASGI:\n    """Trace one-process, serial profiling requests without changing responses.\n\n    The ASGI interval ends at the perf_counter_ns call immediately following\n    successful completion of await send(final_response_body). It does not end\n    after the inner app returns or after logging. Thus inner-app background\n    work after the final body is excluded. Timing bookkeeping before that point\n    is part of the measured path; final-body hashing and JSONL logging performed\n    after that timestamp are excluded. This is an instrumented-service timing.\n    """\n\n    def __init__(self, app, id_prefix=\'p9stage-\', log_path=None):\n        if (not isinstance(id_prefix, str) or\n                re.fullmatch(r\'[A-Za-z0-9_-]{1,48}\', id_prefix, flags=re.ASCII) is None):\n            raise ValueError(\'Profiling ID prefix must be 1-48 safe ASCII characters.\')\n        configured_path = log_path if log_path is not None else os.environ.get(\'PROJECT9_STAGE_LOG\')\n        if not configured_path:\n            raise RuntimeError(\'PROJECT9_STAGE_LOG or log_path must name the new local JSONL file.\')\n        path = Path(configured_path)\n        if not path.is_absolute() or not path.parent.is_dir():\n            raise RuntimeError(\'Profiling JSONL path must be absolute with an existing parent directory.\')\n        self.app = app\n        self.id_prefix = id_prefix\n        self.log_path = path\n\n    async def __call__(self, scope, receive, send):\n        entered_ns = perf_counter_ns()\n        if (scope.get(\'type\') != \'http\' or scope.get(\'method\') != \'POST\' or\n                scope.get(\'path\') != \'/predict\'):\n            await self.app(scope, receive, send)\n            return\n\n        # Reject missing/duplicate/unsafe IDs before allowing any model call.\n        ids = [value for name, value in scope.get(\'headers\', [])\n               if name.lower() == b\'x-project9-profile-id\']\n        if len(ids) != 1:\n            raise RuntimeError(\'A single x-project9-profile-id is required for profiling.\')\n        try:\n            request_id = ids[0].decode(\'ascii\')\n        except (UnicodeDecodeError, AttributeError) as error:\n            raise RuntimeError(\'Profiling request ID must be safe ASCII.\') from error\n        suffix = request_id[len(self.id_prefix):] if request_id.startswith(self.id_prefix) else \'\'\n        if re.fullmatch(r\'[A-Za-z0-9_.-]{1,96}\', suffix, flags=re.ASCII) is None:\n            raise RuntimeError(\'Profiling request ID has an invalid prefix or suffix.\')\n\n        context = {\'request_id\': request_id, \'stages\': {}, \'route_total_ms\': None, \'runtime\': {}}\n        token = trace_context.set(context)\n        response_status = None\n        response_hash = hashlib.sha256()\n        response_bytes = 0\n        final_send_ns = None\n        app_error = None\n        app_error_traceback = None\n        app_error_stack = None\n\n        async def traced_send(message):\n            nonlocal response_status, response_bytes, final_send_ns\n            message_type = message.get(\'type\')\n            body = message.get(\'body\', b\'\') if message_type == \'http.response.body\' else None\n            if body is not None and not isinstance(body, bytes):\n                raise RuntimeError(\'Expected ASGI response body bytes.\')\n            if message_type == \'http.response.body\' and final_send_ns is not None:\n                raise RuntimeError(\'Response body appeared after the final response body.\')\n            await send(message)\n            sent_ns = perf_counter_ns()\n            if message_type == \'http.response.start\':\n                if response_status is not None:\n                    raise RuntimeError(\'Duplicate ASGI response start.\')\n                response_status = int(message[\'status\'])\n            elif message_type == \'http.response.body\':\n                # The timestamp must be captured before hashing the final chunk.\n                if not message.get(\'more_body\', False):\n                    final_send_ns = sent_ns\n                response_hash.update(body)\n                response_bytes += len(body)\n\n        try:\n            await self.app(scope, receive, traced_send)\n        except BaseException as error:\n            app_error = error\n            app_error_traceback = error.__traceback__\n            app_error_stack = traceback.format_exc()\n        finally:\n            # Always restore the previous context, even if validation/logging fails.\n            trace_context.reset(token)\n\n        observed_after_app_ns = perf_counter_ns()\n        record = {\n            \'request_id\': request_id,\n            \'recorded_at_utc\': datetime.now(timezone.utc).isoformat(),\n            \'pid\': os.getpid(),\n            \'status_code\': response_status,\n            \'response_complete\': final_send_ns is not None,\n            \'response_body_sha256\': response_hash.hexdigest(),\n            \'response_body_bytes\': response_bytes,\n            \'success\': False,\n            \'error\': None,\n            \'stages\': {},\n            \'route_total_ms\': None,\n            \'server_asgi_ms\': ((final_send_ns - entered_ns) / 1_000_000\n                               if final_send_ns is not None else None),\n            \'framework_remainder_ms\': None,\n            \'runtime\': {},\n            \'instrumentation_note\': (\n                \'ASGI entry to successful final-body send completion; inner-app work after \'\n                \'final send, final-chunk hashing and JSONL logging are excluded. Other wrapper \'\n                \'bookkeeping before final send is included. Predictor-call timings include \'\n                \'Ultralytics internal preprocessing, inference and postprocessing.\'\n            ),\n        }\n        validation_error = None\n        validation_traceback = None\n        try:\n            if app_error is not None:\n                raise RuntimeError(\'Inner application raised an exception; profile is unsuccessful.\')\n            if response_status != 200 or final_send_ns is None:\n                raise RuntimeError(\'Profiling requires one complete HTTP 200 response.\')\n            stages = context.get(\'stages\')\n            if not isinstance(stages, dict) or set(stages) != set(STAGE_KEYS):\n                raise RuntimeError(\'Route did not supply exactly the four expected stage durations.\')\n            record[\'stages\'] = {key: _nonnegative_finite(stages[key], key) for key in STAGE_KEYS}\n            record[\'route_total_ms\'] = _nonnegative_finite(context.get(\'route_total_ms\'), \'route_total_ms\')\n            stage_sum = sum(record[\'stages\'].values())\n            if stage_sum > record[\'route_total_ms\'] + 0.001:\n                raise RuntimeError(\'Summed stages exceed the enclosing route duration.\')\n            if record[\'route_total_ms\'] > record[\'server_asgi_ms\'] + 0.001:\n                raise RuntimeError(\'Route duration exceeds the enclosing ASGI duration.\')\n            runtime = context.get(\'runtime\')\n            if not isinstance(runtime, dict) or not runtime:\n                raise RuntimeError(\'Route did not supply actual runtime metadata.\')\n            # Validate JSON data without silently stringifying unknown values.\n            json.dumps(runtime, allow_nan=False)\n            record[\'runtime\'] = runtime\n            record[\'framework_remainder_ms\'] = max(0.0, record[\'server_asgi_ms\'] - record[\'route_total_ms\'])\n            record[\'stage_sum_ms\'] = stage_sum\n            record[\'route_bookkeeping_ms\'] = max(0.0, record[\'route_total_ms\'] - stage_sum)\n            record[\'success\'] = True\n        except Exception as error:\n            validation_error = error\n            validation_traceback = error.__traceback__\n            record[\'error_type\'] = type(app_error or error).__name__\n            record[\'error_message\'] = str(app_error or error)\n            record[\'error\'] = {\'type\': record[\'error_type\'], \'message\': record[\'error_message\']}\n            record[\'traceback\'] = app_error_stack or traceback.format_exc()\n            record[\'elapsed_to_inner_app_return_or_error_ms\'] = (observed_after_app_ns - entered_ns) / 1_000_000\n            # repr retains invalid stage data while keeping failure JSON finite.\n            record[\'invalid_or_partial_context_repr\'] = repr(context)\n\n        # These observations occur after the captured interval, never within it.\n        process_times = os.times()\n        record[\'process_cpu_times_after_seconds\'] = {\n            \'user\': process_times.user, \'system\': process_times.system,\n            \'children_user\': process_times.children_user, \'children_system\': process_times.children_system,\n        }\n        try:\n            line = json.dumps(record, ensure_ascii=False, allow_nan=False)\n            with self.log_path.open(\'a\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n                stream.write(line + \'\\n\')\n                stream.flush()\n        except Exception as log_error:\n            if app_error is not None:\n                app_error.add_note(\'Profiling JSONL logging also failed: \' + repr(log_error))\n                raise app_error.with_traceback(app_error_traceback)\n            raise RuntimeError(\'Profiling JSONL logging failed; do not use this run.\') from log_error\n        if app_error is not None:\n            raise app_error.with_traceback(app_error_traceback)\n        if validation_error is not None:\n            raise validation_error.with_traceback(validation_traceback)\n'

"""Build a timing-only copy of the exact reviewed HTTP service AST.

Importing this helper neither loads models nor performs requests. The caller
must separately verify the original source bytes, manifest and checkpoints.
"""


def instrument_service(source_text, original_path):
    import ast
    import copy
    import hashlib
    import json
    import math
    from pathlib import Path

    baseline_ast_sha256 = 'ff88f5a91bce4bae194ff76398cf34396b3cd9b9b7c7e4b8a02f9c765b9b7c98'

    def canonical(value):
        if isinstance(value, ast.AST):
            return {'node_type': type(value).__name__,
                    'fields': {name: canonical(getattr(value, name)) for name in value._fields}}
        if isinstance(value, list):
            return [canonical(item) for item in value]
        if isinstance(value, bytes):
            return {'scalar_type': 'bytes', 'hex': value.hex()}
        if value is Ellipsis:
            return {'scalar_type': 'ellipsis'}
        if isinstance(value, float) and not math.isfinite(value):
            return {'scalar_type': 'nonfinite_float', 'repr': repr(value)}
        if value is None or isinstance(value, (str, bool, int, float)):
            return value
        raise TypeError('Unsupported AST scalar: ' + type(value).__name__)

    def dump(node):
        # ast.dump's default treatment of empty fields changes across Python
        # releases. Encode every _fields member explicitly for 3.12/3.13 parity.
        return json.dumps(canonical(node), sort_keys=True, separators=(',', ':'),
                          ensure_ascii=True, allow_nan=False)

    def sha(text):
        return hashlib.sha256(text.encode('utf-8')).hexdigest()

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    require(isinstance(source_text, str), 'Service source must be text.')
    original_path = Path(original_path).resolve()
    tree = ast.parse(source_text, filename=str(original_path))
    original_dump = dump(tree)
    require(sha(original_dump) == baseline_ast_sha256,
            'The service AST differs from the reviewed service; timing instrumentation stopped.')
    pristine = copy.deepcopy(tree)
    require('_m8_' not in original_dump, 'Timing names collide with original service names.')
    routes = [node for node in tree.body if isinstance(node, ast.AsyncFunctionDef) and node.name == 'predict']
    require(len(routes) == 1, 'Expected exactly one async predict route.')
    route = routes[0]

    def assigned_name(statement):
        if isinstance(statement, ast.Assign) and len(statement.targets) == 1 and isinstance(statement.targets[0], ast.Name):
            return statement.targets[0].id
        return None

    det_indices = [index for index, node in enumerate(route.body) if assigned_name(node) == 'detection']
    seg_indices = [index for index, node in enumerate(route.body) if assigned_name(node) == 'segmentation']
    require(len(det_indices) == len(seg_indices) == 1 and seg_indices[0] == det_indices[0] + 1,
            'Expected adjacent M4 and M5 prediction assignments.')
    require(dump(route.body[det_indices[0]]) == dump(ast.parse('detection=det.predict(bgr,**opts)[0]').body[0]),
            'Unexpected detector prediction statement.')
    require(dump(route.body[seg_indices[0]]) == dump(ast.parse('segmentation=seg.predict(bgr,retina_masks=True,**opts)[0]').body[0]),
            'Unexpected segmenter prediction statement.')
    require(isinstance(route.body[-1], ast.Return) and isinstance(route.body[-1].value, ast.Dict),
            'Expected the original final response dictionary.')
    original_statements = copy.deepcopy(route.body)
    response_value = copy.deepcopy(route.body[-1].value)
    t = lambda index: ast.parse(f'_m8_t{index} = perf_counter_ns()').body[0]
    injected_runtime = ast.parse('''
_m8_t4 = perf_counter_ns()
_m8_context = trace_context.get()
if _m8_context is None:
    raise RuntimeError('Timing context is missing; use the instrumented ASGI entry point.')
_m8_context['stages'] = {
    'preprocess_ms': (_m8_t1 - _m8_t0) / 1000000.0,
    'm4_predict_call_ms': (_m8_t2 - _m8_t1) / 1000000.0,
    'm5_predict_call_ms': (_m8_t3 - _m8_t2) / 1000000.0,
    'application_postprocess_ms': (_m8_t4 - _m8_t3) / 1000000.0,
}
_m8_context['route_total_ms'] = (_m8_t4 - _m8_t0) / 1000000.0
_m8_context['runtime'] = {
    'detector_fp16': bool(det.predictor.model.fp16),
    'segmenter_fp16': bool(seg.predictor.model.fp16),
    'detector_device': str(det.predictor.model.device),
    'segmenter_device': str(seg.predictor.model.device),
    'detector_parameter_dtypes': sorted({str(p.dtype) for p in det.predictor.model.model.parameters() if p.is_floating_point()}),
    'segmenter_parameter_dtypes': sorted({str(p.dtype) for p in seg.predictor.model.model.parameters() if p.is_floating_point()}),
    'input_original_bgr_shape': list(bgr.shape),
    'torch_num_threads': int(__import__('torch').get_num_threads()),
    'torch_num_interop_threads': int(__import__('torch').get_num_interop_threads()),
}
return _m8_payload
''').body
    changed = [t(0)]
    for index, statement in enumerate(original_statements[:-1]):
        if index == det_indices[0]:
            changed.append(t(1))
        if index == seg_indices[0]:
            changed.append(t(2))
        changed.append(statement)
        if index == seg_indices[0]:
            changed.append(t(3))
    changed.append(ast.Assign(targets=[ast.Name(id='_m8_payload', ctx=ast.Store())], value=response_value))
    changed.extend(injected_runtime)
    route.body = changed

    # Assert that all original route statements, including their order, remain.
    preserved = [node for node in route.body
                 if node in original_statements[:-1]]
    require([dump(node) for node in preserved] == [dump(node) for node in original_statements[:-1]],
            'An original route statement was removed, changed or reordered.')
    require(dump(response_value) == dump(original_statements[-1].value), 'Response dictionary changed.')
    original_route = next(node for node in pristine.body if isinstance(node, ast.AsyncFunctionDef) and node.name == 'predict')
    require(dump(route.args) == dump(original_route.args) and
            [dump(node) for node in route.decorator_list] == [dump(node) for node in original_route.decorator_list],
            'Route signature or decorators changed.')

    here_nodes = [node for node in tree.body if assigned_name(node) == 'HERE']
    require(len(here_nodes) == 1, 'Expected one original HERE assignment.')
    here_nodes[0].value = ast.Call(func=ast.Name(id='Path', ctx=ast.Load()),
                                 args=[ast.Constant(value=str(original_path.parent))], keywords=[])
    # Only HERE and predict may change in the original top-level module.
    for old, new in zip(pristine.body, tree.body):
        if old is original_route or assigned_name(old) == 'HERE':
            continue
        require(dump(old) == dump(new), 'Unexpected change outside the timing route/HERE anchor.')
    tree.body.insert(0, ast.parse('from m8_timing_support_source import trace_context, perf_counter_ns').body[0])
    tree.body.extend(ast.parse('from m8_timing_support_source import TimingASGI\nstage_app = TimingASGI(app)').body)
    tree = ast.fix_missing_locations(tree)
    code = ast.unparse(tree) + '\n'
    compile(code, 'instrumented_service.py', 'exec')
    metadata = {
        'source_text_sha256': sha(source_text),
        'original_service_ast_sha256': sha(original_dump),
        'reviewed_baseline_ast_sha256': baseline_ast_sha256,
        'ast_fingerprint_format': 'Canonical JSON: node_type and every _fields entry, recursively; sorted keys, compact separators, ASCII; no location attributes.',
        'instrumented_source_sha256': sha(code),
        'original_service_path': str(original_path),
        'HERE_points_to_original_folder': str(original_path.parent),
        'response_dictionary_ast_unchanged': True,
        'original_route_statements_and_order_preserved': True,
        'original_route_signature_and_decorators_preserved': True,
        'original_top_level_changes': ['HERE filesystem anchor only', 'predict timing-only statements'],
        'stage_boundaries': {
            'preprocess_ms': 'Route entry through upload read, validation, decode, RGB-to-BGR conversion and options; excludes FastAPI multipart parsing before route entry.',
            'm4_predict_call_ms': 'Complete det.predict call, including its internal preprocessing, neural execution and postprocessing; not pure forward time.',
            'm5_predict_call_ms': 'Complete seg.predict call with retina_masks=True, including internal preprocessing, neural execution and mask postprocessing.',
            'application_postprocess_ms': 'Detection/mask array and polygon extraction, greedy association, centroid calculation, counts and original response-dictionary construction.',
            'route_total_ms': 'Sum of four contiguous route stages. Runtime metadata inspection occurs afterward and contributes to framework remainder.',
        },
        'no_service_execution_during_instrumentation': True,
    }
    return code, metadata


"""Restore only the verified HTTP timing image after a Colab reset.
Defining this function performs no installation, network request or file write.
"""


def prepare_m8_session_after_reset():
    from collections import OrderedDict
    from importlib.metadata import PackageNotFoundError, version
    from pathlib import Path, PurePosixPath
    import csv
    import hashlib
    import io
    import json
    import re
    import subprocess
    import sys
    import time
    import zipfile
    from google.colab import drive

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()

    def installed(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None

    drive.mount('/content/drive')
    root = Path('/content/drive/MyDrive/Project9_Phase2')
    manifest = root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    receipt = root / 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json'
    protected = {
        'service': root / 'deployment/http_service/service.py',
        'release_manifest': root / 'deployment/http_service/release_manifest.json',
        'detector': root / 'deployment/http_service/best_det.pt',
        'segmenter': root / 'deployment/http_service/best_seg.pt',
        'frozen_protocol': root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'frozen_results': root / 'evaluation/FINAL_TEST_ONCE/results.json',
        'historical_benchmark_summary': root / 'evaluation/M8/benchmark_summary.json',
        'historical_benchmark_requests': root / 'evaluation/M8/benchmark_requests.csv',
        'manifest': manifest, 'inventory': receipt,
    }
    missing = [str(path) for path in protected.values() if not path.is_file()]
    require(not missing, 'Saved project files are missing in Drive; stop and share this output: ' + repr(missing))
    before = {name: digest(path) for name, path in protected.items()}
    expected_manifest_sha = 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50'
    expected_inventory_sha = '86e913fa48c09c596f396ebfaeefa49ff2b720d3cf9983f6645bd3f911734835'
    expected_image_sha = '122ac26da011334ab0dda881d8086b88657791f8afbf03a1fcb5185af7f6c722'
    expected = {
        'manifest': expected_manifest_sha, 'inventory': expected_inventory_sha,
        'detector': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
        'segmenter': '1ceaa151c3dc3d3503d2812216309322be62d25eebfcd57e35f513036412f178',
        'frozen_protocol': 'aca3f6749633875b719c47c8bddff3f79c0a9da28f35246384a79f6d4cf9faff',
        'frozen_results': 'b6f48bf11f862930c4a1d34a4290d2fea3ed3918f2ccc153fef169cb78cb91dc',
    }
    for name, expected_sha in expected.items():
        require(before[name] == expected_sha, 'Saved provenance/model hash mismatch: ' + name)
    export_name = 'train__07_frame_000001.PNG'
    rows = list(csv.DictReader(io.StringIO(manifest.read_text(encoding='utf-8-sig'))))
    matches = [row for row in rows if row['original_split'] == 'train' and row['filename'] == '07_frame_000001.PNG']
    require(len(matches) == 1 and matches[0]['split'] == 'val', 'The selected input is not unique in frozen Validation.')
    inventory = json.loads(receipt.read_text(encoding='utf-8'))
    inventory_matches = [item for item in inventory if item['export_name'] == export_name]
    require(len(inventory_matches) == 1 and inventory_matches[0]['image_sha256'] == expected_image_sha,
            'Historical image SHA anchor differs.')
    require(int(matches[0]['image_id']) == inventory_matches[0]['image_id'], 'Frozen input image identity differs.')

    torch_before = installed('torch')
    require(torch_before is not None, 'This Colab runtime has no preinstalled PyTorch; stop and share this output.')
    ultra_before = installed('ultralytics')
    if ultra_before != '8.4.213':
        print('Preparing Ultralytics 8.4.213 while retaining the current PyTorch package...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'ultralytics==8.4.213', 'torch==' + torch_before], check=True, timeout=300)
    require(installed('torch') == torch_before, 'PyTorch version changed unexpectedly; stop before profiling.')
    require(installed('ultralytics') == '8.4.213', 'Pinned Ultralytics installation did not complete.')
    import requests
    from PIL import Image

    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    archive_size = 7518242799
    password = b'REDACTED_DATASET_ZIP_PASSWORD'
    suffix = 'train/data/07_frame_000001.PNG'
    image_path = Path('/content/Project9_Phase2/failure_inputs_VAL_v1/images') / export_name

    class HTTPRangeFile(io.RawIOBase):
        """Bounded read-only seekable HTTP file for Python's ZIP64 reader."""
        def __init__(self):
            super().__init__()
            self.position = 0
            self.size = archive_size
            self.block_size = 1024 * 1024
            self.blocks = OrderedDict()
            self.transferred = 0
            self.requests_count = 0
            self.session = requests.Session()

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.position

        def seek(self, offset, whence=0):
            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
            if position < 0:
                raise ValueError('Invalid HTTP ZIP seek.')
            self.position = int(position)
            return self.position

        def get_block(self, index):
            if index in self.blocks:
                self.blocks.move_to_end(index)
                return self.blocks[index]
            start = index * self.block_size
            end = min(self.size - 1, start + self.block_size - 1)
            needed = end - start + 1
            require(self.transferred + needed <= 128 * 1024**2,
                    'Partial-download budget exceeded; the full archive will not be downloaded automatically.')
            for attempt in range(3):
                try:
                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={
                        'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity', 'Cache-Control': 'no-cache'
                    }) as response:
                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
                            time.sleep(2 * (attempt + 1))
                            continue
                        require(response.status_code == 206,
                                f'HTTP {response.status_code}: requested range was not supported; full download prevented.')
                        content_range = response.headers.get('Content-Range', '').strip()
                        match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', content_range, flags=re.I)
                        require(match and tuple(map(int, match.groups())) == (start, end, self.size),
                                'Server returned an unexpected Content-Range: ' + content_range)
                        data = response.raw.read(needed + 1)
                        require(len(data) == needed, 'Incomplete or oversized HTTP range response.')
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
            self.transferred += len(data)
            self.requests_count += 1
            self.blocks[index] = data
            while len(self.blocks) > 16:
                self.blocks.popitem(last=False)
            if self.requests_count % 5 == 0:
                print(f'Partial ZIP transfer: {self.transferred / 1024**2:.1f} MiB')
            return data

        def read(self, count=-1):
            if self.closed:
                raise ValueError('HTTP ZIP stream is closed.')
            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))
            require(count <= 64 * 1024**2, 'Unexpected large ZIP read; complete-archive download prevented.')
            chunks = []
            remaining = count
            while remaining:
                index, offset = divmod(self.position, self.block_size)
                data = self.get_block(index)
                take = min(remaining, len(data) - offset)
                require(take > 0, 'Unexpected ZIP read position.')
                chunks.append(data[offset:offset + take])
                self.position += take
                remaining -= take
            return b''.join(chunks)

        def close(self):
            self.session.close()
            self.blocks.clear()
            super().close()

    image_path.parent.mkdir(parents=True, exist_ok=True)
    mode = 'verified_existing_cache'
    transferred = 0
    requests_count = 0
    actual_member = None
    if image_path.exists():
        require(image_path.is_file() and digest(image_path) == expected_image_sha,
                'Existing image cache differs; it will not be overwritten: ' + str(image_path))
    else:
        local_candidates = [Path('/content/zerowaste-f-final.zip'), root / 'cache/zerowaste-f-final.zip']
        local = next((path for path in local_candidates if path.is_file() and path.stat().st_size == archive_size), None)
        range_file = None
        try:
            if local is not None:
                mode = 'existing_local_archive_member_sha256_verified'
                archive_source = local
            else:
                mode = 'http_ranges_single_member_sha256_verified'
                print('Restoring ONE Validation PNG through bounded HTTP ranges; no full archive download.')
                range_file = HTTPRangeFile()
                archive_source = range_file
            with zipfile.ZipFile(archive_source) as archive:
                choices = [item for item in archive.infolist() if not item.is_dir()
                           and (item.filename == suffix or item.filename.endswith('/' + suffix))
                           and '..' not in PurePosixPath(item.filename).parts]
                require(len(choices) == 1, 'The selected original PNG is not unique in the ZIP.')
                require(0 < choices[0].file_size <= 12 * 1024**2, 'Unexpected selected PNG size.')
                actual_member = choices[0].filename
                data = archive.read(choices[0], pwd=password)
            require(hashlib.sha256(data).hexdigest() == expected_image_sha, 'Retrieved original PNG hash mismatch.')
            with Image.open(io.BytesIO(data)) as image:
                require(image.format == 'PNG' and image.size == (1920, 1080), 'Retrieved PNG dimensions/format differ.')
                image.verify()
            with image_path.open('xb') as stream:
                stream.write(data)
            if range_file is not None:
                transferred = range_file.transferred
                requests_count = range_file.requests_count
        finally:
            if range_file is not None:
                range_file.close()
    require(digest(image_path) == expected_image_sha, 'Final original PNG hash mismatch.')
    with Image.open(image_path) as image:
        require(image.format == 'PNG' and image.size == (1920, 1080), 'Cached PNG dimensions/format differ.')
        image.verify()
    require(before == {name: digest(path) for name, path in protected.items()}, 'A saved Drive reference changed.')
    provenance = {
        'mode': mode, 'image_path': str(image_path), 'export_name': export_name,
        'image_sha256': expected_image_sha, 'width': 1920, 'height': 1080,
        'archive_url': archive_url, 'archive_total_bytes': archive_size,
        'requested_member_suffix': suffix, 'actual_member_this_call': actual_member,
        'range_bytes_transferred_this_call': transferred, 'range_requests_this_call': requests_count,
        'whole_archive_md5_recomputed_this_call': False,
        'image_hash_anchor': 'Previously saved inventory from the earlier full-archive verification; independently checked against its recorded SHA256 here.',
        'inventory_sha256': expected_inventory_sha, 'split_manifest_sha256': expected_manifest_sha,
        'frozen_manifest_row': matches[0], 'protected_drive_files_unchanged': True,
        'ultralytics_before': ultra_before, 'ultralytics_after': installed('ultralytics'),
        'torch_before': torch_before, 'torch_after': installed('torch'),
        'scope': 'Session setup and one verified Validation PNG only; no model execution, Test, labels or additional images.',
    }
    print('Session ready. Original HTTP service files verified; one Validation image ready.')
    print('Partial image retrieval this call (MiB):', round(transferred / 1024**2, 2))
    return provenance


_M8_INPUT_RESTORE_PROVENANCE = prepare_m8_session_after_reset()

"""Execute inside Colab after the reviewed instrumenter and support source."""


def run_m8_stage_timing():
    from collections import Counter
    from datetime import datetime, timezone
    from importlib.metadata import PackageNotFoundError, version
    from pathlib import Path
    import csv
    import hashlib
    import io
    import json
    import math
    import os
    import platform
    import shutil
    import socket
    import subprocess
    import sys
    import time
    import traceback
    import zipfile

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    service_dir = root / 'deployment/http_service'
    image_path = Path('/content/Project9_Phase2/failure_inputs_VAL_v1/images/train__07_frame_000001.PNG')
    image_sha = '122ac26da011334ab0dda881d8086b88657791f8afbf03a1fcb5185af7f6c722'
    manifest_sha = 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50'
    model_hashes = {
        'best_det.pt': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
        'best_seg.pt': '1ceaa151c3dc3d3503d2812216309322be62d25eebfcd57e35f513036412f178',
    }
    expected_classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for part in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(part)
        return value.hexdigest()

    def package_version(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None

    require(package_version('ultralytics') == '8.4.213',
            'Ultralytics 8.4.213 is required. Do not run the full notebook; share this error.')
    service_path = service_dir / 'service.py'
    release_path = service_dir / 'release_manifest.json'
    split_path = root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    protected = {
        'service_source': service_path, 'release_manifest': release_path,
        'split_manifest': split_path, 'selected_val_png': image_path,
        'original_test_protocol': root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'original_test_results': root / 'evaluation/FINAL_TEST_ONCE/results.json',
        'original_benchmark_summary': root / 'evaluation/M8/benchmark_summary.json',
        'original_benchmark_requests': root / 'evaluation/M8/benchmark_requests.csv',
        **{name: service_dir / name for name in model_hashes},
    }
    missing = [str(path) for path in protected.values() if not path.is_file()]
    require(not missing, 'Required saved files are missing; stop and share this output: ' + repr(missing))
    before = {name: sha(path) for name, path in protected.items()}
    require(before['selected_val_png'] == image_sha, 'Cached original Validation PNG hash mismatch.')
    require(before['split_manifest'] == manifest_sha, 'Frozen split bytes changed.')
    require(before['original_test_protocol'] == 'aca3f6749633875b719c47c8bddff3f79c0a9da28f35246384a79f6d4cf9faff',
            'Frozen Test protocol bytes changed.')
    require(before['original_test_results'] == 'b6f48bf11f862930c4a1d34a4290d2fea3ed3918f2ccc153fef169cb78cb91dc',
            'Frozen Test results bytes changed.')
    for name, expected in model_hashes.items():
        require(before[name] == expected, 'Original HTTP checkpoint hash differs: ' + name)
    config = json.loads(release_path.read_text(encoding='utf-8'))
    require(config['class_names'] == expected_classes and config['device'] == 'cpu'
            and config['backend'] == 'PyTorch' and config['imgsz'] == 640
            and config['confidence'] == 0.25 and config['nms_iou'] == 0.5,
            'Original service settings differ from the reviewed configuration.')
    for key, name in [('detector', 'best_det.pt'), ('segmenter', 'best_seg.pt')]:
        require(config['weights'][key] == {'file': name, 'sha256': model_hashes[name]},
                'Release manifest checkpoint mapping changed.')
    split_rows = list(csv.DictReader(io.StringIO(split_path.read_text(encoding='utf-8-sig'))))
    selected = [row for row in split_rows
                if row['original_split'] + '__' + row['filename'] == image_path.name]
    require(len(selected) == 1 and selected[0]['split'] == 'val', 'Input is not uniquely in frozen Validation.')
    from PIL import Image
    with Image.open(image_path) as image:
        require(image.format == 'PNG' and image.size == (1920, 1080), 'Original PNG dimensions/format differ.')
        image.verify()
    source = service_path.read_text(encoding='utf-8-sig')
    instrumented_source, instrumentation = instrument_service(source, service_path)
    require(instrumentation['source_text_sha256'] == hashlib.sha256(source.encode('utf-8')).hexdigest(),
            'Instrumenter source receipt mismatch.')

    required_packages = {'fastapi': '0.141.1', 'uvicorn': '0.54.0', 'python-multipart': '0.0.33'}
    installs = [name + '==' + target for name, target in required_packages.items()
                if package_version(name) != target]
    if package_version('requests') is None:
        installs.append('requests')
    if package_version('psutil') is None:
        installs.append('psutil')
    if installs:
        print('Installing HTTP measurement dependencies only; existing PyTorch is retained.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *installs], check=True, timeout=240)
    import numpy as np
    import psutil
    import requests

    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output = root / 'evaluation' / ('M8_STAGE_TIMING_VAL_' + stamp)
    runtime_dir = Path('/content/Project9_Phase2') / ('M8_STAGE_RUNTIME_' + stamp)
    output.mkdir(parents=False, exist_ok=False)
    runtime_dir.mkdir(parents=False, exist_ok=False)
    trace_path = runtime_dir / 'server_trace.jsonl'
    server_log_path = runtime_dir / 'uvicorn.log'
    (runtime_dir / 'm8_timing_support_source.py').write_text(_M8_TIMING_SUPPORT_SOURCE, encoding='utf-8')
    (runtime_dir / 'instrumented_service.py').write_text(instrumented_source, encoding='utf-8')
    shutil.copyfile(runtime_dir / 'm8_timing_support_source.py', output / 'm8_timing_support_source.py')
    shutil.copyfile(runtime_dir / 'instrumented_service.py', output / 'instrumented_service.py')
    client_rows = []
    process = None
    server_log = None
    session = requests.Session()
    session.trust_env = False

    def save_json(name, value):
        with (output / name).open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write('\n')

    def stop_owned_server():
        nonlocal process, server_log
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        if server_log is not None and not server_log.closed:
            server_log.close()

    try:
        protocol = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(), 'status': 'prepared_before_requests',
            'scope': 'New instrumented local CPU HTTP profile on one fixed Validation image; not Test, video, pure-forward timing or a replacement of the historical benchmark.',
            'session_preparation': _M8_INPUT_RESTORE_PROVENANCE,
            'image': {'file': image_path.name, 'sha256': image_sha, 'width': 1920, 'height': 1080,
                      'frozen_manifest_row': selected[0]},
            'source_service_path': str(service_path), 'original_release_manifest': config,
            'instrumentation': instrumentation,
            'instrumented_code_sha256': sha(runtime_dir / 'instrumented_service.py'),
            'timing_support_sha256': sha(runtime_dir / 'm8_timing_support_source.py'),
            'device': 'cpu', 'backend': 'PyTorch', 'workers': 1, 'concurrency': 1,
            'warmup_requests': 2, 'measured_requests': 25, 'client': 'requests.Session with HTTP keep-alive; trust_env=False',
            'input_transport': 'Multipart PNG upload to an owned Uvicorn server on 127.0.0.1',
            'percentiles': 'numpy.percentile, method=linear, from 25 individual measured requests',
            'memory': 'Server process RSS sampled by the client after each response; not continuous peak memory.',
            'latency_limits': 'No new threshold selection or causal before/after performance claim.',
            'timing_limits': 'Predictor calls include Ultralytics preprocessing, model forward and internal postprocessing. Framework remainder is computed per request, then summarized; percentiles are not additive. Instrumentation adds overhead.',
            'versions': {name: package_version(name) for name in
                         ['ultralytics', 'torch', 'fastapi', 'starlette', 'uvicorn', 'python-multipart',
                          'numpy', 'Pillow', 'requests', 'psutil']},
            'environment': {'python': platform.python_version(), 'platform': platform.platform(),
                            'logical_cpu_count': os.cpu_count(),
                            'cpu_affinity_count': len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else None,
                            'client_torch_already_imported': 'torch' in sys.modules},
            'historical_manifest_is_unchanged': True,
            'protected_sha256_before': before,
            'tracking_environment': {'WANDB_MODE': 'disabled'},
        }
        save_json('protocol.json', protocol)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_socket:
            port_socket.bind(('127.0.0.1', 0))
            port = port_socket.getsockname()[1]
        server_environment = os.environ.copy()
        server_environment['PROJECT9_STAGE_LOG'] = str(trace_path)
        server_environment['WANDB_MODE'] = 'disabled'
        server_environment['TOKENIZERS_PARALLELISM'] = 'false'
        server_log = server_log_path.open('xb')
        command = [sys.executable, '-m', 'uvicorn', 'instrumented_service:stage_app',
                   '--host', '127.0.0.1', '--port', str(port), '--workers', '1',
                   '--log-level', 'warning', '--no-access-log']
        process = subprocess.Popen(command, cwd=str(runtime_dir), env=server_environment,
                                   stdout=server_log, stderr=subprocess.STDOUT)
        address = 'http://127.0.0.1:' + str(port)
        print('Starting the original HTTP service with timing probes on CPU...')
        startup_deadline = time.monotonic() + 120
        while True:
            require(process.poll() is None, 'Measurement server exited. See uvicorn.log in the new evidence folder.')
            try:
                health = session.get(address + '/health', timeout=2)
                if health.status_code == 200:
                    health_json = health.json()
                    require(health_json['model_verified'] is True and health_json['device'] == 'cpu'
                            and health_json['classes'] == expected_classes, 'Unexpected health response.')
                    break
            except requests.RequestException:
                pass
            require(time.monotonic() < startup_deadline, 'Measurement server startup timed out.')
            time.sleep(0.5)
        server_process = psutil.Process(process.pid)
        raw_image = image_path.read_bytes()
        first_response = None
        print('Two warm-up requests, then 25 serial measured requests on one Validation image.')
        for index in range(27):
            phase = 'warmup' if index < 2 else 'measured'
            ordinal = index + 1 if phase == 'warmup' else index - 1
            request_id = 'p9stage-' + stamp + '-' + phase + '-' + str(ordinal).zfill(2)
            start_ns = time.perf_counter_ns()
            response = session.post(address + '/predict',
                headers={'X-Project9-Profile-ID': request_id},
                files={'file': (image_path.name, raw_image, 'image/png')}, timeout=(10, 180))
            stop_ns = time.perf_counter_ns()
            require(response.status_code == 200, 'HTTP prediction failed: ' + str(response.status_code) + ' ' + response.text[:500])
            payload = response.json()
            require(set(payload) == {'image_width', 'image_height', 'objects', 'counts', 'pick_point_units'}
                    and payload['image_width'] == 1920 and payload['image_height'] == 1080
                    and payload['pick_point_units'] == 'original_image_pixels', 'Original response schema changed.')
            require(set(payload['counts']) == set(expected_classes), 'Response class counts differ.')
            require(all(obj['material'] in expected_classes for obj in payload['objects']), 'Unexpected response material.')
            actual_counts = Counter(obj['material'] for obj in payload['objects'])
            require(all(payload['counts'][name] == actual_counts[name] for name in expected_classes),
                    'Response counts disagree with its object list.')
            row = {'request_id': request_id, 'phase': phase, 'ordinal': ordinal,
                   'http_status': response.status_code, 'client_http_ms': (stop_ns - start_ns) / 1e6,
                   'response_body_sha256': hashlib.sha256(response.content).hexdigest(),
                   'object_count': len(payload['objects']),
                   'server_rss_after_response_mib': server_process.memory_info().rss / 1024**2}
            client_rows.append(row)
            if phase == 'measured' and first_response is None:
                first_response = payload
            if phase == 'measured' and ordinal % 5 == 0:
                print('Measured requests:', ordinal, '/25')
        trace_deadline = time.monotonic() + 10
        while True:
            trace_lines = trace_path.read_text(encoding='utf-8').splitlines() if trace_path.exists() else []
            if len(trace_lines) >= 27:
                break
            require(time.monotonic() < trace_deadline, 'Incomplete timing sidecar; stop and share this output.')
            time.sleep(0.1)
        stop_owned_server()
        traces = [json.loads(line) for line in trace_path.read_text(encoding='utf-8').splitlines()]
        require(len(traces) == 27, 'Expected exactly 27 profiled prediction responses.')
        by_id = {row['request_id']: row for row in traces}
        require(len(by_id) == 27 and set(by_id) == {row['request_id'] for row in client_rows},
                'Server/client request correlation mismatch.')
        stage_keys = ['preprocess_ms', 'm4_predict_call_ms', 'm5_predict_call_ms',
                      'application_postprocess_ms']
        metric_keys = ['client_http_ms', 'server_asgi_ms', 'route_total_ms', *stage_keys,
                       'framework_remainder_ms']
        combined = []
        for client in client_rows:
            trace = by_id[client['request_id']]
            require(trace.get('status_code') == 200 and trace.get('error') is None,
                    'Server timing trace contains an unsuccessful request.')
            require(trace['response_body_sha256'] == client['response_body_sha256'], 'Server/client body hashes differ.')
            runtime = trace['runtime']
            require(runtime['detector_fp16'] is False and runtime['segmenter_fp16'] is False
                    and runtime['detector_device'] == 'cpu' and runtime['segmenter_device'] == 'cpu'
                    and runtime['detector_parameter_dtypes'] == ['torch.float32']
                    and runtime['segmenter_parameter_dtypes'] == ['torch.float32'],
                    'Actual predictor device or precision differs from CPU FP32.')
            require(abs(sum(trace['stages'][name] for name in stage_keys) - trace['route_total_ms']) < 1e-5,
                    'Timing stages do not sum to their route interval.')
            require(trace['framework_remainder_ms'] >= -1e-5, 'ASGI interval is shorter than the route interval.')
            row = {**client, **trace['stages'],
                   **{name: trace[name] for name in ['server_asgi_ms', 'route_total_ms', 'framework_remainder_ms']}}
            require(all(math.isfinite(row[key]) and row[key] >= -1e-5 for key in metric_keys), 'Invalid timing value.')
            combined.append(row)
        measured = [row for row in combined if row['phase'] == 'measured']
        require(len(measured) == 25, 'Expected 25 measured rows.')
        metrics = {}
        for key in metric_keys:
            values = np.asarray([row[key] for row in measured], dtype=np.float64)
            metrics[key] = {'mean_ms': float(values.mean()),
                            'p50_ms': float(np.percentile(values, 50, method='linear')),
                            'p95_ms': float(np.percentile(values, 95, method='linear')),
                            'p99_ms': float(np.percentile(values, 99, method='linear'))}
        after = {name: sha(path) for name, path in protected.items()}
        require(before == after, 'A protected source changed during profiling.')
        summary = {
            'status': 'complete', 'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            'split': 'val', 'unique_images': 1, 'warmup_requests': 2, 'measured_requests': 25,
            'successful_http_predictions': 27, 'expected_predictor_calls_per_model': 27,
            'metrics': metrics,
            'throughput_from_sum_of_client_request_durations_images_per_second':
                25 / (sum(row['client_http_ms'] for row in measured) / 1000),
            'max_server_rss_after_measured_responses_mib':
                max(row['server_rss_after_response_mib'] for row in measured),
            'rss_is_not_continuous_peak': True,
            'distinct_response_body_hashes_across_25_measured_requests':
                len({row['response_body_sha256'] for row in measured}),
            'actual_predictor_runtime_after_requests': traces[-1]['runtime'],
            'protected_sources_unchanged': True, 'protected_sha256_before': before,
            'protected_sha256_after': after,
            'interpretation': protocol['scope'], 'timing_limits': protocol['timing_limits'],
            'historical_benchmark_replaced': False, 'test_evaluated': False, 'training_run': False,
        }
        save_json('summary.json', summary)
        save_json('first_measured_response.json', first_response)
        with (output / 'requests_with_stages.csv').open('x', encoding='utf-8', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(combined[0]))
            writer.writeheader()
            writer.writerows(combined)
        shutil.copyfile(trace_path, output / 'server_trace.jsonl')
        shutil.copyfile(server_log_path, output / 'uvicorn.log')
        (output / 'README.md').write_text(
            '# One-image Validation HTTP stage profile\n\n' + protocol['scope'] + '\n\n' + protocol['timing_limits']
            + '\n\nThe service source and original deployment files were not edited. A temporary instrumented copy loads '
              'the original manifest-matching HTTP checkpoints. The M5 HTTP checkpoint has different file bytes '
              'from the training-source checkpoint; no replacement is performed. Warm-up rows are excluded from summaries. '
              'Logging is after the last response-body send and excluded from its ASGI interval; timing-wrapper overhead '
              'remains part of the measurements. Framework remainder is not pure JSON time. '
              'The historical benchmark and frozen Test files are unchanged. This is neither video FPS nor a diverse-load benchmark.\n',
            encoding='utf-8')
        print('=== HTTP STAGE TIMING COMPLETE ===')
        print('Milliseconds: p50 / p95 (25 measured requests; one Validation image)')
        for key, values in metrics.items():
            print(key, ':', round(values['p50_ms'], 3), '/', round(values['p95_ms'], 3))
        print('Saved in Drive:', output)
        zip_path = Path('/content') / ('Project9_M8_STAGE_TIMING_VAL_' + stamp + '.zip')
        receipts = []
        with zipfile.ZipFile(zip_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(output.iterdir()):
                require(path.is_file(), 'Unexpected output structure while packaging.')
                receipts.append({'file': path.name, 'bytes': path.stat().st_size, 'sha256': sha(path)})
                archive.write(path, arcname=path.name)
            archive.writestr('collection_manifest.json', json.dumps({'files': receipts, 'split': 'val',
                'note': 'Inventory excludes itself. Model binaries and original PNG are not packaged.'}, indent=2) + '\n')
        with zipfile.ZipFile(zip_path) as archive:
            require(archive.testzip() is None, 'ZIP CRC check failed.')
            for receipt in receipts:
                data = archive.read(receipt['file'])
                require(len(data) == receipt['bytes'] and hashlib.sha256(data).hexdigest() == receipt['sha256'],
                        'Packaged file integrity check failed.')
        print('ZIP ready:', zip_path)
        from google.colab import files
        files.download(str(zip_path))
        return summary
    except Exception as error:
        stop_owned_server()
        if trace_path.exists() and not (output / 'server_trace.jsonl').exists():
            shutil.copyfile(trace_path, output / 'server_trace.jsonl')
        if server_log_path.exists() and not (output / 'uvicorn.log').exists():
            shutil.copyfile(server_log_path, output / 'uvicorn.log')
        failure = {'status': 'failed', 'error': str(error), 'traceback': traceback.format_exc(),
                   'completed_client_predictions': len(client_rows), 'protected_sha256_before': before,
                   'protected_sha256_at_failure': {name: sha(path) if path.is_file() else None
                                                  for name, path in protected.items()}}
        save_json('failure.json', failure)
        print('Stopped. New evidence/log folder:', output)
        raise
    finally:
        stop_owned_server()
        session.close()


m8_stage_timing_result = run_m8_stage_timing()


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# انسخ الملف كاملًا إلى خلية جديدة في Colab وشغّل تلك الخلية فقط.
# يربط Drive ويجهّز الجلسة ويسترجع الصور المطلوبة فقط إذا اختفت.
# عرض من30صورة Validation متتابعة منمشهد07؛ لا تدريب ولا تقييم Test.
# سرعة عرض الفيديو3صور/ثانية اختيار للعرض، وسرعة المعالجة تقاس بصورة مستقلة.

_REPLAY_TIMING_SUPPORT_SOURCE = '"""Read-only profiling support for the isolated Project9 Colab HTTP service.\n\nNo model imports, model execution, network calls, or automatic file writes occur\non import. The instrumented /predict route obtains ``trace_context.get()`` and\nsets three fields before returning its ORIGINAL response dictionary:\n\n    context[\'stages\'] = {\n        \'preprocess_ms\': ...,\n        \'m4_predict_call_ms\': ...,\n        \'m5_predict_call_ms\': ...,\n        \'application_postprocess_ms\': ...,\n    }\n    context[\'route_total_ms\'] = ...\n    context[\'runtime\'] = {... actual device / fp16 / parameter dtypes ...}\n\nWrap the FastAPI app with TimingASGI. Only POST /predict is profiled. Such\nrequests must carry x-project9-profile-id, by default beginning ``p9stage-``.\nAll response messages and body bytes are passed through unchanged.\n"""\n\nfrom contextvars import ContextVar\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom time import perf_counter_ns\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport traceback\n\n\ntrace_context = ContextVar(\'project9_stage_timing_context\', default=None)\n\nSTAGE_KEYS = (\n    \'preprocess_ms\',\n    \'m4_predict_call_ms\',\n    \'m5_predict_call_ms\',\n    \'application_postprocess_ms\',\n)\n\n\ndef _nonnegative_finite(value, name):\n    if isinstance(value, bool) or not isinstance(value, (int, float)):\n        raise RuntimeError(f\'{name} must be a finite numeric duration.\')\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise RuntimeError(f\'{name} must be a nonnegative finite duration.\')\n    return value\n\n\nclass TimingASGI:\n    """Trace one-process, serial profiling requests without changing responses.\n\n    The ASGI interval ends at the perf_counter_ns call immediately following\n    successful completion of await send(final_response_body). It does not end\n    after the inner app returns or after logging. Thus inner-app background\n    work after the final body is excluded. Timing bookkeeping before that point\n    is part of the measured path; final-body hashing and JSONL logging performed\n    after that timestamp are excluded. This is an instrumented-service timing.\n    """\n\n    def __init__(self, app, id_prefix=\'p9stage-\', log_path=None):\n        if (not isinstance(id_prefix, str) or\n                re.fullmatch(r\'[A-Za-z0-9_-]{1,48}\', id_prefix, flags=re.ASCII) is None):\n            raise ValueError(\'Profiling ID prefix must be 1-48 safe ASCII characters.\')\n        configured_path = log_path if log_path is not None else os.environ.get(\'PROJECT9_STAGE_LOG\')\n        if not configured_path:\n            raise RuntimeError(\'PROJECT9_STAGE_LOG or log_path must name the new local JSONL file.\')\n        path = Path(configured_path)\n        if not path.is_absolute() or not path.parent.is_dir():\n            raise RuntimeError(\'Profiling JSONL path must be absolute with an existing parent directory.\')\n        self.app = app\n        self.id_prefix = id_prefix\n        self.log_path = path\n\n    async def __call__(self, scope, receive, send):\n        entered_ns = perf_counter_ns()\n        if (scope.get(\'type\') != \'http\' or scope.get(\'method\') != \'POST\' or\n                scope.get(\'path\') != \'/predict\'):\n            await self.app(scope, receive, send)\n            return\n\n        # Reject missing/duplicate/unsafe IDs before allowing any model call.\n        ids = [value for name, value in scope.get(\'headers\', [])\n               if name.lower() == b\'x-project9-profile-id\']\n        if len(ids) != 1:\n            raise RuntimeError(\'A single x-project9-profile-id is required for profiling.\')\n        try:\n            request_id = ids[0].decode(\'ascii\')\n        except (UnicodeDecodeError, AttributeError) as error:\n            raise RuntimeError(\'Profiling request ID must be safe ASCII.\') from error\n        suffix = request_id[len(self.id_prefix):] if request_id.startswith(self.id_prefix) else \'\'\n        if re.fullmatch(r\'[A-Za-z0-9_.-]{1,96}\', suffix, flags=re.ASCII) is None:\n            raise RuntimeError(\'Profiling request ID has an invalid prefix or suffix.\')\n\n        context = {\'request_id\': request_id, \'stages\': {}, \'route_total_ms\': None, \'runtime\': {}}\n        token = trace_context.set(context)\n        response_status = None\n        response_hash = hashlib.sha256()\n        response_bytes = 0\n        final_send_ns = None\n        app_error = None\n        app_error_traceback = None\n        app_error_stack = None\n\n        async def traced_send(message):\n            nonlocal response_status, response_bytes, final_send_ns\n            message_type = message.get(\'type\')\n            body = message.get(\'body\', b\'\') if message_type == \'http.response.body\' else None\n            if body is not None and not isinstance(body, bytes):\n                raise RuntimeError(\'Expected ASGI response body bytes.\')\n            if message_type == \'http.response.body\' and final_send_ns is not None:\n                raise RuntimeError(\'Response body appeared after the final response body.\')\n            await send(message)\n            sent_ns = perf_counter_ns()\n            if message_type == \'http.response.start\':\n                if response_status is not None:\n                    raise RuntimeError(\'Duplicate ASGI response start.\')\n                response_status = int(message[\'status\'])\n            elif message_type == \'http.response.body\':\n                # The timestamp must be captured before hashing the final chunk.\n                if not message.get(\'more_body\', False):\n                    final_send_ns = sent_ns\n                response_hash.update(body)\n                response_bytes += len(body)\n\n        try:\n            await self.app(scope, receive, traced_send)\n        except BaseException as error:\n            app_error = error\n            app_error_traceback = error.__traceback__\n            app_error_stack = traceback.format_exc()\n        finally:\n            # Always restore the previous context, even if validation/logging fails.\n            trace_context.reset(token)\n\n        observed_after_app_ns = perf_counter_ns()\n        record = {\n            \'request_id\': request_id,\n            \'recorded_at_utc\': datetime.now(timezone.utc).isoformat(),\n            \'pid\': os.getpid(),\n            \'status_code\': response_status,\n            \'response_complete\': final_send_ns is not None,\n            \'response_body_sha256\': response_hash.hexdigest(),\n            \'response_body_bytes\': response_bytes,\n            \'success\': False,\n            \'error\': None,\n            \'stages\': {},\n            \'route_total_ms\': None,\n            \'server_asgi_ms\': ((final_send_ns - entered_ns) / 1_000_000\n                               if final_send_ns is not None else None),\n            \'framework_remainder_ms\': None,\n            \'runtime\': {},\n            \'instrumentation_note\': (\n                \'ASGI entry to successful final-body send completion; inner-app work after \'\n                \'final send, final-chunk hashing and JSONL logging are excluded. Other wrapper \'\n                \'bookkeeping before final send is included. Predictor-call timings include \'\n                \'Ultralytics internal preprocessing, inference and postprocessing.\'\n            ),\n        }\n        validation_error = None\n        validation_traceback = None\n        try:\n            if app_error is not None:\n                raise RuntimeError(\'Inner application raised an exception; profile is unsuccessful.\')\n            if response_status != 200 or final_send_ns is None:\n                raise RuntimeError(\'Profiling requires one complete HTTP 200 response.\')\n            stages = context.get(\'stages\')\n            if not isinstance(stages, dict) or set(stages) != set(STAGE_KEYS):\n                raise RuntimeError(\'Route did not supply exactly the four expected stage durations.\')\n            record[\'stages\'] = {key: _nonnegative_finite(stages[key], key) for key in STAGE_KEYS}\n            record[\'route_total_ms\'] = _nonnegative_finite(context.get(\'route_total_ms\'), \'route_total_ms\')\n            stage_sum = sum(record[\'stages\'].values())\n            if stage_sum > record[\'route_total_ms\'] + 0.001:\n                raise RuntimeError(\'Summed stages exceed the enclosing route duration.\')\n            if record[\'route_total_ms\'] > record[\'server_asgi_ms\'] + 0.001:\n                raise RuntimeError(\'Route duration exceeds the enclosing ASGI duration.\')\n            runtime = context.get(\'runtime\')\n            if not isinstance(runtime, dict) or not runtime:\n                raise RuntimeError(\'Route did not supply actual runtime metadata.\')\n            # Validate JSON data without silently stringifying unknown values.\n            json.dumps(runtime, allow_nan=False)\n            record[\'runtime\'] = runtime\n            record[\'framework_remainder_ms\'] = max(0.0, record[\'server_asgi_ms\'] - record[\'route_total_ms\'])\n            record[\'stage_sum_ms\'] = stage_sum\n            record[\'route_bookkeeping_ms\'] = max(0.0, record[\'route_total_ms\'] - stage_sum)\n            record[\'success\'] = True\n        except Exception as error:\n            validation_error = error\n            validation_traceback = error.__traceback__\n            record[\'error_type\'] = type(app_error or error).__name__\n            record[\'error_message\'] = str(app_error or error)\n            record[\'error\'] = {\'type\': record[\'error_type\'], \'message\': record[\'error_message\']}\n            record[\'traceback\'] = app_error_stack or traceback.format_exc()\n            record[\'elapsed_to_inner_app_return_or_error_ms\'] = (observed_after_app_ns - entered_ns) / 1_000_000\n            # repr retains invalid stage data while keeping failure JSON finite.\n            record[\'invalid_or_partial_context_repr\'] = repr(context)\n\n        # These observations occur after the captured interval, never within it.\n        process_times = os.times()\n        record[\'process_cpu_times_after_seconds\'] = {\n            \'user\': process_times.user, \'system\': process_times.system,\n            \'children_user\': process_times.children_user, \'children_system\': process_times.children_system,\n        }\n        try:\n            line = json.dumps(record, ensure_ascii=False, allow_nan=False)\n            with self.log_path.open(\'a\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n                stream.write(line + \'\\n\')\n                stream.flush()\n        except Exception as log_error:\n            if app_error is not None:\n                app_error.add_note(\'Profiling JSONL logging also failed: \' + repr(log_error))\n                raise app_error.with_traceback(app_error_traceback)\n            raise RuntimeError(\'Profiling JSONL logging failed; do not use this run.\') from log_error\n        if app_error is not None:\n            raise app_error.with_traceback(app_error_traceback)\n        if validation_error is not None:\n            raise validation_error.with_traceback(validation_traceback)\n'

_REPLAY_RENDERER_SOURCE = '"""Scientific overlays for ordered ZeroWaste image replay.\n\nImporting this module performs no file writes, inference, or network access.\nOnly the supplied HTTP objects are drawn: M4 boxes and associated M5 polygons.\nNo tracking IDs, ground truth, or segmentation-only detections are invented.\n"""\n\n\ndef _replay_label_rectangle(anchor, text_size, occupied, bounds, padding=4):\n    """Place a label inside bounds, preferring candidates without text overlap."""\n    left, top, right, bottom = bounds\n    width = min(right - left, int(text_size[0]) + 2 * padding)\n    height = min(bottom - top, int(text_size[1]) + 2 * padding)\n    ax, ay = anchor\n    xs = [ax, ax - width, right - width, left]\n    ys = [ay - height - 2, ay + 2]\n    ys.extend(ay + 2 + n * (height + 2) for n in range(1, 8))\n    ys.extend(ay - height - 2 - n * (height + 2) for n in range(1, 5))\n    candidates = []\n    for raw_x in xs:\n        for raw_y in ys:\n            x = max(left, min(right - width, int(round(raw_x))))\n            y = max(top, min(bottom - height, int(round(raw_y))))\n            rect = (x, y, x + width, y + height)\n            overlap = sum(max(0, min(rect[2], old[2]) - max(rect[0], old[0]))\n                          * max(0, min(rect[3], old[3]) - max(rect[1], old[1]))\n                          for old in occupied)\n            distance = abs(x - ax) + abs(y + height - ay)\n            candidates.append((overlap, distance, rect))\n    return min(candidates, key=lambda item: (item[0], item[1]))[2]\n\n\ndef render_replay_frame(image_bgr, payload, metadata):\n    """Return one 1280x840 uint8 BGR overlay frame, with a 120-pixel header.\n\n    ``image_bgr`` is the original 1920x1080 image. ``payload`` is an unchanged\n    successful /predict JSON response. Metadata keys are export_name,\n    frame_number, ordinal, total, http_ms and display_fps. Display fps is a\n    presentation setting; source capture timing remains unknown.\n    """\n    import math\n    import numpy as np\n    from PIL import Image, ImageDraw, ImageFont\n\n    def require(condition, message):\n        if not condition:\n            raise ValueError(message)\n\n    require(isinstance(image_bgr, np.ndarray) and image_bgr.dtype == np.uint8\n            and image_bgr.shape == (1080, 1920, 3), \'Expected original 1920x1080 uint8 BGR image.\')\n    require(isinstance(payload, dict) and payload.get(\'image_width\') == 1920\n            and payload.get(\'image_height\') == 1080 and isinstance(payload.get(\'objects\'), list),\n            \'HTTP response dimensions or object list differ.\')\n    require(payload.get(\'pick_point_units\') == \'original_image_pixels\', \'Unexpected response-coordinate units.\')\n    require(isinstance(metadata, dict), \'Replay metadata must be a dictionary.\')\n    required = [\'export_name\', \'frame_number\', \'ordinal\', \'total\', \'http_ms\', \'display_fps\']\n    require(all(key in metadata for key in required), \'Incomplete replay metadata.\')\n    for key in [\'frame_number\', \'ordinal\', \'total\']:\n        require(isinstance(metadata[key], int) and not isinstance(metadata[key], bool),\n                \'Replay frame indices must be integers.\')\n    require(metadata[\'frame_number\'] >= 1 and 1 <= metadata[\'ordinal\'] <= metadata[\'total\'],\n            \'Invalid replay frame indices.\')\n    require(isinstance(metadata[\'export_name\'], str) and \'\\n\' not in metadata[\'export_name\']\n            and \'\\r\' not in metadata[\'export_name\'], \'Invalid export filename.\')\n    require(metadata[\'total\'] == 30, \'This reviewed demonstration uses exactly 30 ordered samples.\')\n    for key in [\'http_ms\', \'display_fps\']:\n        require(not isinstance(metadata[key], bool) and isinstance(metadata[key], (int, float))\n                and math.isfinite(float(metadata[key])) and float(metadata[key]) > 0,\n                \'Replay time/fps metadata must be finite and positive.\')\n\n    names = [\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\']\n    # Fixed RGB colors identify predicted material only, not correctness.\n    colors = {\'cardboard\': (245, 174, 55), \'metal\': (74, 181, 239),\n              \'rigid_plastic\': (135, 222, 102), \'soft_plastic\': (230, 119, 218)}\n    width, image_height, header = 1280, 720, 120\n    rgb = Image.fromarray(np.ascontiguousarray(image_bgr[:, :, ::-1]), mode=\'RGB\')\n    rgb = rgb.resize((width, image_height), resample=Image.Resampling.LANCZOS)\n    canvas = Image.new(\'RGB\', (width, image_height + header), (19, 26, 35))\n    canvas.paste(rgb, (0, header))\n\n    def font_at(size, bold=False):\n        candidates = ([\'DejaVuSans-Bold.ttf\', \'/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf\',\n                       r\'C:\\Windows\\Fonts\\arialbd.ttf\'] if bold else\n                      [\'DejaVuSans.ttf\', \'/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\',\n                       r\'C:\\Windows\\Fonts\\arial.ttf\'])\n        for candidate in candidates:\n            try:\n                return ImageFont.truetype(candidate, size=size)\n            except OSError:\n                pass\n        return ImageFont.load_default(size=size)\n\n    def finite_coordinates(value, shape, description):\n        try:\n            array = np.asarray(value, dtype=np.float64)\n        except (TypeError, ValueError) as error:\n            raise ValueError(\'Invalid \' + description) from error\n        require(array.shape == shape and np.isfinite(array).all(), \'Invalid \' + description)\n        return array\n\n    def xy(point):\n        # Clipping confines annotations to the original image area only.\n        return (max(0, min(width - 1, float(point[0]) * width / 1920)),\n                header + max(0, min(image_height - 1, float(point[1]) * image_height / 1080)))\n\n    validated = []\n    for obj in payload[\'objects\']:\n        require(isinstance(obj, dict) and obj.get(\'material\') in names, \'Unknown HTTP material.\')\n        require(isinstance(obj.get(\'requires_review\'), bool), \'Missing mask-association review flag.\')\n        confidence = obj.get(\'confidence\')\n        require(not isinstance(confidence, bool) and isinstance(confidence, (int, float))\n                and math.isfinite(float(confidence)) and 0 <= confidence <= 1, \'Invalid detector confidence.\')\n        box = finite_coordinates(obj.get(\'bbox_xyxy\'), (4,), \'M4 bounding box\')\n        require(box[2] > box[0] and box[3] > box[1], \'Empty M4 bounding box.\')\n        x1, y1 = xy(box[:2])\n        x2, y2 = xy(box[2:])\n        require(x2 > x1 and y2 > y1, \'M4 box lies outside the image.\')\n        polygon = obj.get(\'mask_polygon_xy\')\n        points = None\n        if polygon is not None:\n            try:\n                polygon = np.asarray(polygon, dtype=np.float64)\n            except (TypeError, ValueError) as error:\n                raise ValueError(\'Invalid associated M5 polygon.\') from error\n            require(polygon.ndim == 2 and polygon.shape[1] == 2 and len(polygon) >= 3\n                    and np.isfinite(polygon).all(), \'Invalid associated M5 polygon.\')\n            require(obj[\'requires_review\'] is False, \'Associated polygon disagrees with original review flag.\')\n            points = [xy(point) for point in polygon]\n        else:\n            require(obj[\'requires_review\'] is True, \'Absent associated mask disagrees with original review flag.\')\n        validated.append((obj, (x1, y1, x2, y2), points))\n\n    # Transparent fills preserve source imagery; they do not add predictions.\n    overlay = Image.new(\'RGBA\', canvas.size, (0, 0, 0, 0))\n    overlay_draw = ImageDraw.Draw(overlay)\n    for obj, box, polygon in validated:\n        if polygon is not None:\n            color = colors[obj[\'material\']]\n            overlay_draw.polygon(polygon, fill=(*color, 51), outline=(*color, 230), width=2)\n    canvas = Image.alpha_composite(canvas.convert(\'RGBA\'), overlay).convert(\'RGB\')\n    draw = ImageDraw.Draw(canvas)\n    label_font = font_at(18, bold=True)\n    occupied = []\n    bounds = (2, header + 2, width - 2, header + image_height - 2)\n    # Draw all geometry before labels so a later box cannot erase earlier text.\n    for obj, box, polygon in validated:\n        draw.rectangle(box, outline=colors[obj[\'material\']], width=3)\n    labels = []\n    for obj, box, polygon in validated:\n        color = colors[obj[\'material\']]\n        label = obj[\'material\'].replace(\'_\', \' \') + f" {float(obj[\'confidence\']):.3f}"\n        if obj[\'requires_review\']:\n            label += \' REVIEW\'\n        bbox = draw.textbbox((0, 0), label, font=label_font)\n        label_size = (bbox[2] - bbox[0], bbox[3] - bbox[1])\n        rect = _replay_label_rectangle((box[0], box[1]), label_size, occupied, bounds)\n        occupied.append(rect)\n        labels.append((label, bbox, rect, color))\n        # A short leader makes shifted labels traceable without tracking IDs.\n        leader_start = (max(rect[0], min(rect[2], box[0])), rect[3])\n        if abs(leader_start[0] - box[0]) + abs(leader_start[1] - box[1]) > 8:\n            draw.line([leader_start, (box[0], box[1])], fill=color, width=2)\n    for label, bbox, rect, color in labels:\n        draw.rectangle(rect, fill=(19, 26, 35), outline=color, width=2)\n        draw.text((rect[0] + 4 - bbox[0], rect[1] + 4 - bbox[1]), label,\n                  font=label_font, fill=(255, 255, 255))\n\n    def header_line(text, top, size=20, bold=False, fill=(234, 239, 245)):\n        # Font size is fitted before drawing; no header text is clipped.\n        font = font_at(size, bold=bold)\n        bbox = draw.textbbox((0, 0), text, font=font)\n        while bbox[2] - bbox[0] > width - 24 and size > 12:\n            size -= 1\n            font = font_at(size, bold=bold)\n            bbox = draw.textbbox((0, 0), text, font=font)\n        require(bbox[2] - bbox[0] <= width - 24, \'Replay header cannot fit.\')\n        require(top + bbox[3] - bbox[1] <= header - 3, \'Replay header height overflow.\')\n        draw.text((12 - bbox[0], top - bbox[1]), text, font=font, fill=fill)\n\n    header_line(\'ZeroWaste VAL - ordered image replay | source sequence 07 | \'\n                + f"sample {metadata[\'ordinal\']:02d}/{metadata[\'total\']:02d}", 8, size=23, bold=True)\n    header_line(f"{metadata[\'export_name\']} | source frame index {metadata[\'frame_number\']}", 39, size=19)\n    header_line(f"Display: {float(metadata[\'display_fps\']):g} fps (chosen) | capture fps: unknown | "\n                + f"HTTP: {float(metadata[\'http_ms\']):.1f} ms", 66, size=19)\n    header_line(\'Boxes: M4 | filled polygons: associated M5 | REVIEW: no valid mask association\',\n                94, size=18, fill=(198, 210, 224))\n    result_rgb = np.asarray(canvas, dtype=np.uint8)\n    return np.ascontiguousarray(result_rgb[:, :, ::-1])\n'

"""Build a timing-only copy of the exact reviewed HTTP service AST.

Importing this helper neither loads models nor performs requests. The caller
must separately verify the original source bytes, manifest and checkpoints.
"""


def instrument_service(source_text, original_path):
    import ast
    import copy
    import hashlib
    import json
    import math
    from pathlib import Path

    baseline_ast_sha256 = 'ff88f5a91bce4bae194ff76398cf34396b3cd9b9b7c7e4b8a02f9c765b9b7c98'

    def canonical(value):
        if isinstance(value, ast.AST):
            return {'node_type': type(value).__name__,
                    'fields': {name: canonical(getattr(value, name)) for name in value._fields}}
        if isinstance(value, list):
            return [canonical(item) for item in value]
        if isinstance(value, bytes):
            return {'scalar_type': 'bytes', 'hex': value.hex()}
        if value is Ellipsis:
            return {'scalar_type': 'ellipsis'}
        if isinstance(value, float) and not math.isfinite(value):
            return {'scalar_type': 'nonfinite_float', 'repr': repr(value)}
        if value is None or isinstance(value, (str, bool, int, float)):
            return value
        raise TypeError('Unsupported AST scalar: ' + type(value).__name__)

    def dump(node):
        # ast.dump's default treatment of empty fields changes across Python
        # releases. Encode every _fields member explicitly for 3.12/3.13 parity.
        return json.dumps(canonical(node), sort_keys=True, separators=(',', ':'),
                          ensure_ascii=True, allow_nan=False)

    def sha(text):
        return hashlib.sha256(text.encode('utf-8')).hexdigest()

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    require(isinstance(source_text, str), 'Service source must be text.')
    original_path = Path(original_path).resolve()
    tree = ast.parse(source_text, filename=str(original_path))
    original_dump = dump(tree)
    require(sha(original_dump) == baseline_ast_sha256,
            'The service AST differs from the reviewed service; timing instrumentation stopped.')
    pristine = copy.deepcopy(tree)
    require('_m8_' not in original_dump, 'Timing names collide with original service names.')
    routes = [node for node in tree.body if isinstance(node, ast.AsyncFunctionDef) and node.name == 'predict']
    require(len(routes) == 1, 'Expected exactly one async predict route.')
    route = routes[0]

    def assigned_name(statement):
        if isinstance(statement, ast.Assign) and len(statement.targets) == 1 and isinstance(statement.targets[0], ast.Name):
            return statement.targets[0].id
        return None

    det_indices = [index for index, node in enumerate(route.body) if assigned_name(node) == 'detection']
    seg_indices = [index for index, node in enumerate(route.body) if assigned_name(node) == 'segmentation']
    require(len(det_indices) == len(seg_indices) == 1 and seg_indices[0] == det_indices[0] + 1,
            'Expected adjacent M4 and M5 prediction assignments.')
    require(dump(route.body[det_indices[0]]) == dump(ast.parse('detection=det.predict(bgr,**opts)[0]').body[0]),
            'Unexpected detector prediction statement.')
    require(dump(route.body[seg_indices[0]]) == dump(ast.parse('segmentation=seg.predict(bgr,retina_masks=True,**opts)[0]').body[0]),
            'Unexpected segmenter prediction statement.')
    require(isinstance(route.body[-1], ast.Return) and isinstance(route.body[-1].value, ast.Dict),
            'Expected the original final response dictionary.')
    original_statements = copy.deepcopy(route.body)
    response_value = copy.deepcopy(route.body[-1].value)
    t = lambda index: ast.parse(f'_m8_t{index} = perf_counter_ns()').body[0]
    injected_runtime = ast.parse('''
_m8_t4 = perf_counter_ns()
_m8_context = trace_context.get()
if _m8_context is None:
    raise RuntimeError('Timing context is missing; use the instrumented ASGI entry point.')
_m8_context['stages'] = {
    'preprocess_ms': (_m8_t1 - _m8_t0) / 1000000.0,
    'm4_predict_call_ms': (_m8_t2 - _m8_t1) / 1000000.0,
    'm5_predict_call_ms': (_m8_t3 - _m8_t2) / 1000000.0,
    'application_postprocess_ms': (_m8_t4 - _m8_t3) / 1000000.0,
}
_m8_context['route_total_ms'] = (_m8_t4 - _m8_t0) / 1000000.0
_m8_context['runtime'] = {
    'detector_fp16': bool(det.predictor.model.fp16),
    'segmenter_fp16': bool(seg.predictor.model.fp16),
    'detector_device': str(det.predictor.model.device),
    'segmenter_device': str(seg.predictor.model.device),
    'detector_parameter_dtypes': sorted({str(p.dtype) for p in det.predictor.model.model.parameters() if p.is_floating_point()}),
    'segmenter_parameter_dtypes': sorted({str(p.dtype) for p in seg.predictor.model.model.parameters() if p.is_floating_point()}),
    'input_original_bgr_shape': list(bgr.shape),
    'torch_num_threads': int(__import__('torch').get_num_threads()),
    'torch_num_interop_threads': int(__import__('torch').get_num_interop_threads()),
}
return _m8_payload
''').body
    changed = [t(0)]
    for index, statement in enumerate(original_statements[:-1]):
        if index == det_indices[0]:
            changed.append(t(1))
        if index == seg_indices[0]:
            changed.append(t(2))
        changed.append(statement)
        if index == seg_indices[0]:
            changed.append(t(3))
    changed.append(ast.Assign(targets=[ast.Name(id='_m8_payload', ctx=ast.Store())], value=response_value))
    changed.extend(injected_runtime)
    route.body = changed

    # Assert that all original route statements, including their order, remain.
    preserved = [node for node in route.body
                 if node in original_statements[:-1]]
    require([dump(node) for node in preserved] == [dump(node) for node in original_statements[:-1]],
            'An original route statement was removed, changed or reordered.')
    require(dump(response_value) == dump(original_statements[-1].value), 'Response dictionary changed.')
    original_route = next(node for node in pristine.body if isinstance(node, ast.AsyncFunctionDef) and node.name == 'predict')
    require(dump(route.args) == dump(original_route.args) and
            [dump(node) for node in route.decorator_list] == [dump(node) for node in original_route.decorator_list],
            'Route signature or decorators changed.')

    here_nodes = [node for node in tree.body if assigned_name(node) == 'HERE']
    require(len(here_nodes) == 1, 'Expected one original HERE assignment.')
    here_nodes[0].value = ast.Call(func=ast.Name(id='Path', ctx=ast.Load()),
                                 args=[ast.Constant(value=str(original_path.parent))], keywords=[])
    # Only HERE and predict may change in the original top-level module.
    for old, new in zip(pristine.body, tree.body):
        if old is original_route or assigned_name(old) == 'HERE':
            continue
        require(dump(old) == dump(new), 'Unexpected change outside the timing route/HERE anchor.')
    tree.body.insert(0, ast.parse('from m8_timing_support_source import trace_context, perf_counter_ns').body[0])
    tree.body.extend(ast.parse('from m8_timing_support_source import TimingASGI\nstage_app = TimingASGI(app)').body)
    tree = ast.fix_missing_locations(tree)
    code = ast.unparse(tree) + '\n'
    compile(code, 'instrumented_service.py', 'exec')
    metadata = {
        'source_text_sha256': sha(source_text),
        'original_service_ast_sha256': sha(original_dump),
        'reviewed_baseline_ast_sha256': baseline_ast_sha256,
        'ast_fingerprint_format': 'Canonical JSON: node_type and every _fields entry, recursively; sorted keys, compact separators, ASCII; no location attributes.',
        'instrumented_source_sha256': sha(code),
        'original_service_path': str(original_path),
        'HERE_points_to_original_folder': str(original_path.parent),
        'response_dictionary_ast_unchanged': True,
        'original_route_statements_and_order_preserved': True,
        'original_route_signature_and_decorators_preserved': True,
        'original_top_level_changes': ['HERE filesystem anchor only', 'predict timing-only statements'],
        'stage_boundaries': {
            'preprocess_ms': 'Route entry through upload read, validation, decode, RGB-to-BGR conversion and options; excludes FastAPI multipart parsing before route entry.',
            'm4_predict_call_ms': 'Complete det.predict call, including its internal preprocessing, neural execution and postprocessing; not pure forward time.',
            'm5_predict_call_ms': 'Complete seg.predict call with retina_masks=True, including internal preprocessing, neural execution and mask postprocessing.',
            'application_postprocess_ms': 'Detection/mask array and polygon extraction, greedy association, centroid calculation, counts and original response-dictionary construction.',
            'route_total_ms': 'Sum of four contiguous route stages. Runtime metadata inspection occurs afterward and contributes to framework remainder.',
        },
        'no_service_execution_during_instrumentation': True,
    }
    return code, metadata


"""Scientific overlays for ordered ZeroWaste image replay.

Importing this module performs no file writes, inference, or network access.
Only the supplied HTTP objects are drawn: M4 boxes and associated M5 polygons.
No tracking IDs, ground truth, or segmentation-only detections are invented.
"""


def _replay_label_rectangle(anchor, text_size, occupied, bounds, padding=4):
    """Place a label inside bounds, preferring candidates without text overlap."""
    left, top, right, bottom = bounds
    width = min(right - left, int(text_size[0]) + 2 * padding)
    height = min(bottom - top, int(text_size[1]) + 2 * padding)
    ax, ay = anchor
    xs = [ax, ax - width, right - width, left]
    ys = [ay - height - 2, ay + 2]
    ys.extend(ay + 2 + n * (height + 2) for n in range(1, 8))
    ys.extend(ay - height - 2 - n * (height + 2) for n in range(1, 5))
    candidates = []
    for raw_x in xs:
        for raw_y in ys:
            x = max(left, min(right - width, int(round(raw_x))))
            y = max(top, min(bottom - height, int(round(raw_y))))
            rect = (x, y, x + width, y + height)
            overlap = sum(max(0, min(rect[2], old[2]) - max(rect[0], old[0]))
                          * max(0, min(rect[3], old[3]) - max(rect[1], old[1]))
                          for old in occupied)
            distance = abs(x - ax) + abs(y + height - ay)
            candidates.append((overlap, distance, rect))
    return min(candidates, key=lambda item: (item[0], item[1]))[2]


def render_replay_frame(image_bgr, payload, metadata):
    """Return one 1280x840 uint8 BGR overlay frame, with a 120-pixel header.

    ``image_bgr`` is the original 1920x1080 image. ``payload`` is an unchanged
    successful /predict JSON response. Metadata keys are export_name,
    frame_number, ordinal, total, http_ms and display_fps. Display fps is a
    presentation setting; source capture timing remains unknown.
    """
    import math
    import numpy as np
    from PIL import Image, ImageDraw, ImageFont

    def require(condition, message):
        if not condition:
            raise ValueError(message)

    require(isinstance(image_bgr, np.ndarray) and image_bgr.dtype == np.uint8
            and image_bgr.shape == (1080, 1920, 3), 'Expected original 1920x1080 uint8 BGR image.')
    require(isinstance(payload, dict) and payload.get('image_width') == 1920
            and payload.get('image_height') == 1080 and isinstance(payload.get('objects'), list),
            'HTTP response dimensions or object list differ.')
    require(payload.get('pick_point_units') == 'original_image_pixels', 'Unexpected response-coordinate units.')
    require(isinstance(metadata, dict), 'Replay metadata must be a dictionary.')
    required = ['export_name', 'frame_number', 'ordinal', 'total', 'http_ms', 'display_fps']
    require(all(key in metadata for key in required), 'Incomplete replay metadata.')
    for key in ['frame_number', 'ordinal', 'total']:
        require(isinstance(metadata[key], int) and not isinstance(metadata[key], bool),
                'Replay frame indices must be integers.')
    require(metadata['frame_number'] >= 1 and 1 <= metadata['ordinal'] <= metadata['total'],
            'Invalid replay frame indices.')
    require(isinstance(metadata['export_name'], str) and '\n' not in metadata['export_name']
            and '\r' not in metadata['export_name'], 'Invalid export filename.')
    require(metadata['total'] == 30, 'This reviewed demonstration uses exactly 30 ordered samples.')
    for key in ['http_ms', 'display_fps']:
        require(not isinstance(metadata[key], bool) and isinstance(metadata[key], (int, float))
                and math.isfinite(float(metadata[key])) and float(metadata[key]) > 0,
                'Replay time/fps metadata must be finite and positive.')

    names = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    # Fixed RGB colors identify predicted material only, not correctness.
    colors = {'cardboard': (245, 174, 55), 'metal': (74, 181, 239),
              'rigid_plastic': (135, 222, 102), 'soft_plastic': (230, 119, 218)}
    width, image_height, header = 1280, 720, 120
    rgb = Image.fromarray(np.ascontiguousarray(image_bgr[:, :, ::-1]), mode='RGB')
    rgb = rgb.resize((width, image_height), resample=Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (width, image_height + header), (19, 26, 35))
    canvas.paste(rgb, (0, header))

    def font_at(size, bold=False):
        candidates = (['DejaVuSans-Bold.ttf', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
                       r'C:\Windows\Fonts\arialbd.ttf'] if bold else
                      ['DejaVuSans.ttf', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                       r'C:\Windows\Fonts\arial.ttf'])
        for candidate in candidates:
            try:
                return ImageFont.truetype(candidate, size=size)
            except OSError:
                pass
        return ImageFont.load_default(size=size)

    def finite_coordinates(value, shape, description):
        try:
            array = np.asarray(value, dtype=np.float64)
        except (TypeError, ValueError) as error:
            raise ValueError('Invalid ' + description) from error
        require(array.shape == shape and np.isfinite(array).all(), 'Invalid ' + description)
        return array

    def xy(point):
        # Clipping confines annotations to the original image area only.
        return (max(0, min(width - 1, float(point[0]) * width / 1920)),
                header + max(0, min(image_height - 1, float(point[1]) * image_height / 1080)))

    validated = []
    for obj in payload['objects']:
        require(isinstance(obj, dict) and obj.get('material') in names, 'Unknown HTTP material.')
        require(isinstance(obj.get('requires_review'), bool), 'Missing mask-association review flag.')
        confidence = obj.get('confidence')
        require(not isinstance(confidence, bool) and isinstance(confidence, (int, float))
                and math.isfinite(float(confidence)) and 0 <= confidence <= 1, 'Invalid detector confidence.')
        box = finite_coordinates(obj.get('bbox_xyxy'), (4,), 'M4 bounding box')
        require(box[2] > box[0] and box[3] > box[1], 'Empty M4 bounding box.')
        x1, y1 = xy(box[:2])
        x2, y2 = xy(box[2:])
        require(x2 > x1 and y2 > y1, 'M4 box lies outside the image.')
        polygon = obj.get('mask_polygon_xy')
        points = None
        if polygon is not None:
            try:
                polygon = np.asarray(polygon, dtype=np.float64)
            except (TypeError, ValueError) as error:
                raise ValueError('Invalid associated M5 polygon.') from error
            require(polygon.ndim == 2 and polygon.shape[1] == 2 and len(polygon) >= 3
                    and np.isfinite(polygon).all(), 'Invalid associated M5 polygon.')
            require(obj['requires_review'] is False, 'Associated polygon disagrees with original review flag.')
            points = [xy(point) for point in polygon]
        else:
            require(obj['requires_review'] is True, 'Absent associated mask disagrees with original review flag.')
        validated.append((obj, (x1, y1, x2, y2), points))

    # Transparent fills preserve source imagery; they do not add predictions.
    overlay = Image.new('RGBA', canvas.size, (0, 0, 0, 0))
    overlay_draw = ImageDraw.Draw(overlay)
    for obj, box, polygon in validated:
        if polygon is not None:
            color = colors[obj['material']]
            overlay_draw.polygon(polygon, fill=(*color, 51), outline=(*color, 230), width=2)
    canvas = Image.alpha_composite(canvas.convert('RGBA'), overlay).convert('RGB')
    draw = ImageDraw.Draw(canvas)
    label_font = font_at(18, bold=True)
    occupied = []
    bounds = (2, header + 2, width - 2, header + image_height - 2)
    # Draw all geometry before labels so a later box cannot erase earlier text.
    for obj, box, polygon in validated:
        draw.rectangle(box, outline=colors[obj['material']], width=3)
    labels = []
    for obj, box, polygon in validated:
        color = colors[obj['material']]
        label = obj['material'].replace('_', ' ') + f" {float(obj['confidence']):.3f}"
        if obj['requires_review']:
            label += ' REVIEW'
        bbox = draw.textbbox((0, 0), label, font=label_font)
        label_size = (bbox[2] - bbox[0], bbox[3] - bbox[1])
        rect = _replay_label_rectangle((box[0], box[1]), label_size, occupied, bounds)
        occupied.append(rect)
        labels.append((label, bbox, rect, color))
        # A short leader makes shifted labels traceable without tracking IDs.
        leader_start = (max(rect[0], min(rect[2], box[0])), rect[3])
        if abs(leader_start[0] - box[0]) + abs(leader_start[1] - box[1]) > 8:
            draw.line([leader_start, (box[0], box[1])], fill=color, width=2)
    for label, bbox, rect, color in labels:
        draw.rectangle(rect, fill=(19, 26, 35), outline=color, width=2)
        draw.text((rect[0] + 4 - bbox[0], rect[1] + 4 - bbox[1]), label,
                  font=label_font, fill=(255, 255, 255))

    def header_line(text, top, size=20, bold=False, fill=(234, 239, 245)):
        # Font size is fitted before drawing; no header text is clipped.
        font = font_at(size, bold=bold)
        bbox = draw.textbbox((0, 0), text, font=font)
        while bbox[2] - bbox[0] > width - 24 and size > 12:
            size -= 1
            font = font_at(size, bold=bold)
            bbox = draw.textbbox((0, 0), text, font=font)
        require(bbox[2] - bbox[0] <= width - 24, 'Replay header cannot fit.')
        require(top + bbox[3] - bbox[1] <= header - 3, 'Replay header height overflow.')
        draw.text((12 - bbox[0], top - bbox[1]), text, font=font, fill=fill)

    header_line('ZeroWaste VAL - ordered image replay | source sequence 07 | '
                + f"sample {metadata['ordinal']:02d}/{metadata['total']:02d}", 8, size=23, bold=True)
    header_line(f"{metadata['export_name']} | source frame index {metadata['frame_number']}", 39, size=19)
    header_line(f"Display: {float(metadata['display_fps']):g} fps (chosen) | capture fps: unknown | "
                + f"HTTP: {float(metadata['http_ms']):.1f} ms", 66, size=19)
    header_line('Boxes: M4 | filled polygons: associated M5 | REVIEW: no valid mask association',
                94, size=18, fill=(198, 210, 224))
    result_rgb = np.asarray(canvas, dtype=np.uint8)
    return np.ascontiguousarray(result_rgb[:, :, ::-1])


"""Restore 30 fixed, ordered ZeroWaste Validation PNGs for a replay demonstration.
Defining this function performs no installation, network request or file write.
"""


def prepare_zerowaste_replay_inputs():
    from collections import OrderedDict
    from importlib.metadata import PackageNotFoundError, version
    from pathlib import Path, PurePosixPath
    import csv
    import hashlib
    import io
    import json
    import re
    import subprocess
    import sys
    import time
    import zipfile
    from google.colab import drive

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()

    def installed(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None

    drive.mount('/content/drive')
    root = Path('/content/drive/MyDrive/Project9_Phase2')
    manifest = root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    receipt = root / 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json'
    protected = {
        'service': root / 'deployment/http_service/service.py',
        'release_manifest': root / 'deployment/http_service/release_manifest.json',
        'detector': root / 'deployment/http_service/best_det.pt',
        'segmenter': root / 'deployment/http_service/best_seg.pt',
        'frozen_protocol': root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'frozen_results': root / 'evaluation/FINAL_TEST_ONCE/results.json',
        'historical_benchmark_summary': root / 'evaluation/M8/benchmark_summary.json',
        'historical_benchmark_requests': root / 'evaluation/M8/benchmark_requests.csv',
        'manifest': manifest, 'inventory': receipt,
    }
    missing = [str(path) for path in protected.values() if not path.is_file()]
    require(not missing, 'Saved project files are missing in Drive; stop and share this output: ' + repr(missing))
    before = {name: digest(path) for name, path in protected.items()}
    expected_manifest_sha = 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50'
    expected_inventory_sha = '86e913fa48c09c596f396ebfaeefa49ff2b720d3cf9983f6645bd3f911734835'
    expected = {
        'manifest': expected_manifest_sha, 'inventory': expected_inventory_sha,
        'detector': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
        'segmenter': '1ceaa151c3dc3d3503d2812216309322be62d25eebfcd57e35f513036412f178',
        'frozen_protocol': 'aca3f6749633875b719c47c8bddff3f79c0a9da28f35246384a79f6d4cf9faff',
        'frozen_results': 'b6f48bf11f862930c4a1d34a4290d2fea3ed3918f2ccc153fef169cb78cb91dc',
    }
    for name, expected_sha in expected.items():
        require(before[name] == expected_sha, 'Saved provenance/model hash mismatch: ' + name)
    rows = list(csv.DictReader(io.StringIO(manifest.read_text(encoding='utf-8-sig'))))
    selected = sorted([row for row in rows if row['split'] == 'val'
                       and row['original_split'] == 'train' and row['sequence'] == '07'],
                      key=lambda row: (int(row['frame_number']), row['filename']))[:30]
    require(len(selected) == 30 and [int(row['frame_number']) for row in selected] == list(range(1, 292, 10)),
            'The fixed first-30 Validation sequence-07 selection differs; stop before inference.')
    inventory_items = json.loads(receipt.read_text(encoding='utf-8'))
    require(isinstance(inventory_items, list) and len(inventory_items) == 451, 'Historical Validation inventory is incomplete.')
    inventory = {item['export_name']: item for item in inventory_items}
    require(len(inventory) == 451, 'Historical Validation inventory has duplicate names.')
    selected_names = ['train__' + row['filename'] for row in selected]
    require(len(set(selected_names)) == 30, 'The replay image selection has duplicate filenames.')
    for row, export_name in zip(selected, selected_names):
        require(export_name in inventory and int(row['image_id']) == inventory[export_name]['image_id'],
                'Frozen image identity differs: ' + export_name)
        require(len(inventory[export_name]['image_sha256']) == 64, 'Historical image hash is malformed.')

    torch_before = installed('torch')
    require(torch_before is not None, 'This Colab runtime has no preinstalled PyTorch; stop and share this output.')
    ultra_before = installed('ultralytics')
    if ultra_before != '8.4.213':
        print('Preparing Ultralytics 8.4.213 while retaining the current PyTorch package...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'ultralytics==8.4.213', 'torch==' + torch_before], check=True, timeout=300)
    require(installed('torch') == torch_before, 'PyTorch version changed unexpectedly; stop before profiling.')
    require(installed('ultralytics') == '8.4.213', 'Pinned Ultralytics installation did not complete.')
    import requests
    from PIL import Image

    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    archive_size = 7518242799
    password = b'REDACTED_DATASET_ZIP_PASSWORD'
    cache = Path('/content/Project9_Phase2/failure_inputs_VAL_v1/images')

    class HTTPRangeFile(io.RawIOBase):
        """Bounded read-only seekable HTTP file for Python's ZIP64 reader."""
        def __init__(self):
            super().__init__()
            self.position = 0
            self.size = archive_size
            self.block_size = 1024 * 1024
            self.blocks = OrderedDict()
            self.transferred = 0
            self.requests_count = 0
            self.session = requests.Session()

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.position

        def seek(self, offset, whence=0):
            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
            if position < 0:
                raise ValueError('Invalid HTTP ZIP seek.')
            self.position = int(position)
            return self.position

        def get_block(self, index):
            if index in self.blocks:
                self.blocks.move_to_end(index)
                return self.blocks[index]
            start = index * self.block_size
            end = min(self.size - 1, start + self.block_size - 1)
            needed = end - start + 1
            require(self.transferred + needed <= 128 * 1024**2,
                    'Partial-download budget exceeded; the full archive will not be downloaded automatically.')
            for attempt in range(3):
                try:
                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={
                        'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity', 'Cache-Control': 'no-cache'
                    }) as response:
                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
                            time.sleep(2 * (attempt + 1))
                            continue
                        require(response.status_code == 206,
                                f'HTTP {response.status_code}: requested range was not supported; full download prevented.')
                        content_range = response.headers.get('Content-Range', '').strip()
                        match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', content_range, flags=re.I)
                        require(match and tuple(map(int, match.groups())) == (start, end, self.size),
                                'Server returned an unexpected Content-Range: ' + content_range)
                        data = response.raw.read(needed + 1)
                        require(len(data) == needed, 'Incomplete or oversized HTTP range response.')
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
            self.transferred += len(data)
            self.requests_count += 1
            self.blocks[index] = data
            while len(self.blocks) > 16:
                self.blocks.popitem(last=False)
            if self.requests_count % 5 == 0:
                print(f'Partial ZIP transfer: {self.transferred / 1024**2:.1f} MiB')
            return data

        def read(self, count=-1):
            if self.closed:
                raise ValueError('HTTP ZIP stream is closed.')
            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))
            require(count <= 64 * 1024**2, 'Unexpected large ZIP read; complete-archive download prevented.')
            chunks = []
            remaining = count
            while remaining:
                index, offset = divmod(self.position, self.block_size)
                data = self.get_block(index)
                take = min(remaining, len(data) - offset)
                require(take > 0, 'Unexpected ZIP read position.')
                chunks.append(data[offset:offset + take])
                self.position += take
                remaining -= take
            return b''.join(chunks)

        def close(self):
            self.session.close()
            self.blocks.clear()
            super().close()

    cache.mkdir(parents=True, exist_ok=True)
    # Cache checks happen before opening any archive. A differing file is never replaced.
    for row, export_name in zip(selected, selected_names):
        path = cache / export_name
        if path.exists():
            require(path.is_file() and digest(path) == inventory[export_name]['image_sha256'],
                    'Existing image cache differs; it will not be overwritten: ' + str(path))
    range_file = None
    archive = None
    source_mode = 'verified_existing_cache'
    downloaded = 0
    reused = 0
    records = []
    member_names = {}

    def ensure_archive():
        nonlocal range_file, archive, source_mode
        if archive is None:
            candidates = [Path('/content/zerowaste-f-final.zip'), root / 'cache/zerowaste-f-final.zip']
            local = next((path for path in candidates if path.is_file() and path.stat().st_size == archive_size), None)
            if local is not None:
                source_mode = 'existing_local_archive_members_sha256_verified'
                archive = zipfile.ZipFile(local)
            else:
                source_mode = 'http_ranges_selected_members_sha256_verified'
                print('Restoring only missing replay PNGs through bounded HTTP ranges; no full archive download.')
                range_file = HTTPRangeFile()
                archive = zipfile.ZipFile(range_file)
        return archive

    try:
        for ordinal, (row, export_name) in enumerate(zip(selected, selected_names), 1):
            path = cache / export_name
            expected_sha = inventory[export_name]['image_sha256']
            restored_here = False
            if path.exists():
                reused += 1
            else:
                current_archive = ensure_archive()
                suffix = 'train/data/' + row['filename']
                choices = [item for item in current_archive.infolist() if not item.is_dir()
                           and (item.filename == suffix or item.filename.endswith('/' + suffix))
                           and '..' not in PurePosixPath(item.filename).parts]
                require(len(choices) == 1, 'Selected PNG is not unique in the original ZIP: ' + suffix)
                require(0 < choices[0].file_size <= 12 * 1024**2, 'Unexpected selected PNG size.')
                data = current_archive.read(choices[0], pwd=password)
                require(hashlib.sha256(data).hexdigest() == expected_sha,
                        'Retrieved original PNG hash mismatch: ' + export_name)
                with Image.open(io.BytesIO(data)) as image:
                    require(image.format == 'PNG' and image.size == (1920, 1080),
                            'Retrieved PNG dimensions/format differ: ' + export_name)
                    image.verify()
                with path.open('xb') as stream:
                    stream.write(data)
                downloaded += 1
                restored_here = True
                member_names[export_name] = choices[0].filename
            require(digest(path) == expected_sha, 'Final cached PNG hash differs: ' + export_name)
            with Image.open(path) as image:
                require(image.format == 'PNG' and image.size == (1920, 1080),
                        'Cached PNG dimensions/format differ: ' + export_name)
                image.verify()
            records.append({'ordinal': ordinal, 'export_name': export_name, 'filename': row['filename'],
                            'path': str(path), 'image_id': int(row['image_id']),
                            'width': 1920, 'height': 1080, 'frame_number': int(row['frame_number']),
                            'image_sha256': expected_sha, 'frozen_manifest_row': row,
                            'restored_this_call': restored_here})
            if ordinal % 5 == 0:
                print('Verified replay inputs:', ordinal, '/30')
        require(len(records) == 30 and downloaded + reused == 30, 'Replay preparation count mismatch.')
        after = {name: digest(path) for name, path in protected.items()}
        require(before == after, 'A protected Drive reference changed during replay preparation.')
        transferred = range_file.transferred if range_file is not None else 0
        requests_count = range_file.requests_count if range_file is not None else 0
        provenance = {
            'mode': source_mode, 'selected_images': 30, 'downloaded_images_this_call': downloaded,
            'reused_images_this_call': reused, 'original_split': 'train', 'project_split': 'val',
            'source_sequence': '07', 'selected_frame_numbers': [item['frame_number'] for item in records],
            'selection': 'First 30 frozen Validation sequence-07 rows in numeric frame order, fixed before any new predictions. These are sampled source frames, not a representative benchmark.',
            'source_frame_number_step': 10, 'original_capture_fps_unknown': True,
            'replay_is_native_capture_video': False,
            'archive_url': archive_url, 'archive_total_bytes': archive_size,
            'range_bytes_transferred_this_call': transferred, 'range_requests_this_call': requests_count,
            'archive_members_retrieved_this_call': member_names,
            'whole_archive_md5_recomputed_this_call': False,
            'image_hash_anchor': 'Retrieved PNG SHA256 values match the previously saved 451-image Validation inventory, whose complete bytes are independently hash checked. The full source archive is not rehashed here.',
            'inventory_sha256': expected_inventory_sha, 'split_manifest_sha256': expected_manifest_sha,
            'protected_drive_sha256_before': before, 'protected_drive_sha256_after': after,
            'protected_drive_files_unchanged': True,
            'ultralytics_before': ultra_before, 'ultralytics_after': installed('ultralytics'),
            'torch_before': torch_before, 'torch_after': installed('torch'),
            'scope': 'Session setup and 30 verified ordered Validation PNGs only; no model execution, ground-truth scoring, Test evaluation, labels or additional images.',
        }
        print('=== REPLAY INPUTS READY ===')
        print('Verified Validation PNGs:', len(records), '| restored:', downloaded, '| reused:', reused)
        print('Partial archive retrieval this call (MiB):', round(transferred / 1024**2, 2))
        return {'records': records, 'provenance': provenance,
                'project_root': str(root), 'cache_directory': str(cache)}
    finally:
        if archive is not None:
            archive.close()
        if range_file is not None:
            range_file.close()


_REPLAY_PREPARATION = prepare_zerowaste_replay_inputs()

"""Replay fixed, ordered Validation PNGs through the verified HTTP service."""


def run_zerowaste_replay(preparation):
    from collections import Counter
    from datetime import datetime, timezone
    from importlib.metadata import PackageNotFoundError, version
    from pathlib import Path
    import csv
    import hashlib
    import io
    import json
    import math
    import os
    import platform
    import shutil
    import socket
    import subprocess
    import sys
    import time
    import traceback
    import zipfile
    import numpy as np
    from PIL import Image

    root = Path('/content/drive/MyDrive/Project9_Phase2')
    source_path = root / 'deployment/http_service/service.py'
    release_path = root / 'deployment/http_service/release_manifest.json'
    records = preparation['records']
    display_fps = 3
    expected_classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()

    def pkg(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None

    require(len(records) == 30 and [row['frame_number'] for row in records] == list(range(1, 292, 10)),
            'The fixed ordered replay selection changed.')
    require(all(row['frozen_manifest_row']['split'] == 'val'
                and row['frozen_manifest_row']['sequence'] == '07' for row in records),
            'Every replay input must be in frozen Validation sequence 07.')
    require(pkg('ultralytics') == '8.4.213', 'Pinned Ultralytics setup is incomplete.')
    protected = {
        'service_source': source_path, 'release_manifest': release_path,
        'http_detector': root / 'deployment/http_service/best_det.pt',
        'http_segmenter': root / 'deployment/http_service/best_seg.pt',
        'split_manifest': root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv',
        'frozen_test_protocol': root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'frozen_test_results': root / 'evaluation/FINAL_TEST_ONCE/results.json',
        'original_benchmark_summary': root / 'evaluation/M8/benchmark_summary.json',
        'original_benchmark_requests': root / 'evaluation/M8/benchmark_requests.csv',
    }
    require(all(path.is_file() for path in protected.values()), 'A protected original project file is missing.')
    before = {key: sha(path) for key, path in protected.items()}
    expected_hashes = {
        'http_detector': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
        'http_segmenter': '1ceaa151c3dc3d3503d2812216309322be62d25eebfcd57e35f513036412f178',
        'split_manifest': 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50',
        'frozen_test_protocol': 'aca3f6749633875b719c47c8bddff3f79c0a9da28f35246384a79f6d4cf9faff',
        'frozen_test_results': 'b6f48bf11f862930c4a1d34a4290d2fea3ed3918f2ccc153fef169cb78cb91dc',
    }
    require(all(before[key] == expected for key, expected in expected_hashes.items()), 'Original provenance bytes differ.')
    input_before = {row['export_name']: sha(row['path']) for row in records}
    require(all(input_before[row['export_name']] == row['image_sha256'] for row in records),
            'A restored original image hash differs.')
    config = json.loads(release_path.read_text(encoding='utf-8'))
    require(config['device'] == 'cpu' and config['backend'] == 'PyTorch'
            and config['class_names'] == expected_classes and config['imgsz'] == 640
            and config['confidence'] == 0.25 and config['nms_iou'] == 0.5,
            'Original HTTP configuration differs.')
    for key, name, expected in [
        ('detector', 'best_det.pt', expected_hashes['http_detector']),
        ('segmenter', 'best_seg.pt', expected_hashes['http_segmenter']),
    ]:
        require(config['weights'][key] == {'file': name, 'sha256': expected},
                'Original HTTP checkpoint mapping differs: ' + key)
    code, source_receipt = instrument_service(source_path.read_text(encoding='utf-8-sig'), source_path)

    needed = {'fastapi': '0.141.1', 'uvicorn': '0.54.0', 'python-multipart': '0.0.33'}
    installs = [key + '==' + target for key, target in needed.items() if pkg(key) != target]
    installs += [name for name in ['requests', 'psutil'] if pkg(name) is None]
    if installs:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *installs], check=True, timeout=240)
    import psutil
    import requests
    import cv2

    ffmpeg = shutil.which('ffmpeg')

    def supports_h264(executable):
        if not executable:
            return False
        result = subprocess.run([executable, '-hide_banner', '-encoders'], capture_output=True,
                                text=True, timeout=30, check=False)
        return result.returncode == 0 and 'libx264' in result.stdout

    if not supports_h264(ffmpeg):
        if pkg('imageio-ffmpeg') != '0.6.0':
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imageio-ffmpeg==0.6.0'],
                           check=True, timeout=240)
        import imageio_ffmpeg
        ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    require(supports_h264(ffmpeg), 'An H.264 encoder is unavailable; stop and share this output.')

    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output = root / 'evaluation' / ('ZEROWASTE_ORDERED_REPLAY_VAL_' + stamp)
    runtime = Path('/content/Project9_Phase2') / ('REPLAY_RUNTIME_' + stamp)
    output.mkdir(parents=False, exist_ok=False)
    runtime.mkdir(parents=False, exist_ok=False)
    frame_dir = runtime / 'frames'
    response_dir = runtime / 'responses'
    frame_dir.mkdir()
    response_dir.mkdir()
    trace_path = runtime / 'server_trace.jsonl'
    log_path = runtime / 'uvicorn.log'
    (runtime / 'm8_timing_support_source.py').write_text(_REPLAY_TIMING_SUPPORT_SOURCE, encoding='utf-8')
    (runtime / 'instrumented_service.py').write_text(code, encoding='utf-8')
    (output / 'renderer_source.py').write_text(_REPLAY_RENDERER_SOURCE, encoding='utf-8')
    (output / 'm8_timing_support_source.py').write_text(_REPLAY_TIMING_SUPPORT_SOURCE, encoding='utf-8')
    (output / 'instrumented_service.py').write_text(code, encoding='utf-8')
    client_rows = []
    session = requests.Session()
    session.trust_env = False
    process = None
    log_stream = None

    def save_json(name, value):
        with (output / name).open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write('\n')

    def stop_owned_server():
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        if log_stream is not None and not log_stream.closed:
            log_stream.close()

    def validate_payload(payload):
        require(set(payload) == {'image_width', 'image_height', 'objects', 'counts', 'pick_point_units'}
                and (payload['image_width'], payload['image_height']) == (1920, 1080)
                and payload['pick_point_units'] == 'original_image_pixels', 'HTTP response schema differs.')
        require(set(payload['counts']) == set(expected_classes), 'HTTP response class names differ.')
        counts = Counter(item['material'] for item in payload['objects'])
        require(all(item['material'] in expected_classes for item in payload['objects'])
                and all(payload['counts'][name] == counts[name] for name in expected_classes),
                'Response material counts disagree with its objects.')

    def submit(raw, filename, phase, ordinal):
        request_id = 'p9stage-' + stamp + '-replay-' + phase + '-' + str(ordinal).zfill(2)
        start = time.perf_counter_ns()
        response = session.post(address + '/predict', headers={'X-Project9-Profile-ID': request_id},
            files={'file': (filename, raw, 'image/png')}, timeout=(10, 180))
        finished = time.perf_counter_ns()
        require(response.status_code == 200, 'HTTP replay request failed: ' + str(response.status_code)
                + ' ' + response.text[:500])
        row = {'request_id': request_id, 'phase': phase, 'ordinal': ordinal,
               'http_status': response.status_code, 'client_http_ms': (finished - start) / 1e6,
               'response_body_sha256': hashlib.sha256(response.content).hexdigest()}
        return response, row, start, finished

    try:
        protocol = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(), 'status': 'prepared_before_requests',
            'scope': 'Ordered sampled-image replay from frozen Validation sequence 07, not native captured video, tracking, Test scoring or a real-time conveyor claim.',
            'selection': preparation['provenance'], 'selected_inputs': records,
            'display': {'chosen_fps': display_fps, 'frames': 30, 'chosen_duration_seconds': 30 / display_fps,
                        'output_width': 1280, 'output_height': 840, 'original_capture_fps': None,
                        'source_frame_number_step': 10, 'audio': False,
                        'display_fps_is_measured_processing_fps': False},
            'pipeline': 'Serial POST /predict to owned local Uvicorn. Boxes/counts come from M4; filled polygons are only M5 instances associated by the original service. REVIEW means no valid mask association.',
            'model_settings': config, 'instrumentation': source_receipt,
            'server': {'host': '127.0.0.1', 'workers': 1, 'concurrency': 1, 'warmup_requests': 2,
                       'measured_unique_inputs': 30, 'device': 'cpu', 'tracking': 'WANDB_MODE=disabled'},
            'measurement_boundaries': {
                'client_http_ms': 'Immediately before requests.post through fully buffered HTTP response; excludes local image decode and overlay rendering.',
                'local_prepare_ms': 'Original PNG read, local decoding and BGR array preparation before sending that PNG.',
                'render_and_record_ms': 'Response JSON parsing/validation, overlays, annotated JPEG and exact response-byte file writes after response receipt.',
                'frame_pipeline_ms': 'Local preparation plus HTTP duration plus render-and-record. Excludes later RSS sampling and progress printing.',
                'loop_wall_seconds': 'Measured 30-frame loop, including local preparation, HTTP, rendering/local writes, RSS samples, file-hash bookkeeping and progress; no deliberate playback delay. Excludes setup/download, warm-up, encoding and final evidence copy/package.',
                'server_stage_times': 'Same reviewed instrumented predictor-call boundaries as the separate M8 profile; predictor calls include their own internal preprocessing/forward/postprocess.',
                'memory': 'Server RSS sampled after responses, not continuously sampled peak.',
            },
            'limits': 'Adjacent sampled frames are correlated. No AP/accuracy or track consistency is computed. Display speed is chosen separately from processing throughput; source capture timing is unknown. Instrumentation adds overhead. Historical benchmarks and Test files are not replaced.',
            'versions': {name: pkg(name) for name in ['ultralytics', 'torch', 'numpy', 'Pillow', 'fastapi',
                         'uvicorn', 'requests', 'psutil', 'opencv-python', 'imageio-ffmpeg']},
            'python': platform.python_version(), 'logical_cpu_count': os.cpu_count(),
            'ffmpeg_executable': ffmpeg,
            'ffmpeg_version': subprocess.run([ffmpeg, '-version'], capture_output=True, text=True,
                                              check=True, timeout=30).stdout.splitlines()[0],
            'code_sha256': {'service': sha(runtime / 'instrumented_service.py'),
                            'timing_support': sha(runtime / 'm8_timing_support_source.py'),
                            'renderer': sha(output / 'renderer_source.py')},
            'protected_sha256_before': before, 'input_sha256_before': input_before,
        }
        save_json('protocol.json', protocol)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as bound:
            bound.bind(('127.0.0.1', 0))
            port = bound.getsockname()[1]
        address = 'http://127.0.0.1:' + str(port)
        environment = os.environ.copy()
        environment['PROJECT9_STAGE_LOG'] = str(trace_path)
        environment['WANDB_MODE'] = 'disabled'
        environment['TOKENIZERS_PARALLELISM'] = 'false'
        log_stream = log_path.open('xb')
        process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'instrumented_service:stage_app',
                                    '--host', '127.0.0.1', '--port', str(port), '--workers', '1',
                                    '--log-level', 'warning', '--no-access-log'],
                                   cwd=str(runtime), env=environment, stdout=log_stream, stderr=subprocess.STDOUT)
        print('Starting the verified CPU HTTP service for the ordered replay...')
        deadline = time.monotonic() + 120
        while True:
            require(process.poll() is None, 'Replay server exited; share the new uvicorn.log.')
            try:
                response = session.get(address + '/health', timeout=2)
                if response.status_code == 200:
                    health = response.json()
                    require(health['device'] == 'cpu' and health['model_verified'] is True
                            and health['classes'] == expected_classes, 'Unexpected service health response.')
                    break
            except requests.RequestException:
                pass
            require(time.monotonic() < deadline, 'Replay service startup timed out.')
            time.sleep(0.5)
        child = psutil.Process(process.pid)
        warmup_raw = Path(records[0]['path']).read_bytes()
        for ordinal in [1, 2]:
            response, row, _, _ = submit(warmup_raw, records[0]['export_name'], 'warmup', ordinal)
            validate_payload(response.json())
            path = response_dir / ('warmup_' + str(ordinal).zfill(2) + '.json')
            path.write_bytes(response.content)
            row.update({'export_name': records[0]['export_name'], 'source_frame_number': 1,
                        'input_sha256': records[0]['image_sha256'], 'response_file': 'responses/' + path.name})
            client_rows.append(row)
        print('Processing 30 ordered Validation images; no artificial playback delay in the measured loop.')
        loop_start = time.perf_counter_ns()
        for index, record in enumerate(records):
            frame_start = time.perf_counter_ns()
            raw = Path(record['path']).read_bytes()
            with Image.open(io.BytesIO(raw)) as image:
                bgr = np.asarray(image.convert('RGB'))[:, :, ::-1].copy()
            response, row, http_started, received = submit(raw, record['export_name'], 'measured', index + 1)
            payload = response.json()
            validate_payload(payload)
            canvas = render_replay_frame(bgr, payload, {
                'export_name': record['export_name'], 'frame_number': record['frame_number'],
                'ordinal': index + 1, 'total': 30, 'http_ms': row['client_http_ms'], 'display_fps': display_fps,
            })
            require(canvas.shape == (840, 1280, 3) and canvas.dtype == np.uint8, 'Rendered frame dimensions/type differ.')
            frame_path = frame_dir / ('frame_' + str(index).zfill(3) + '.jpg')
            Image.fromarray(canvas[:, :, ::-1]).save(frame_path, format='JPEG', quality=93, subsampling=0)
            response_path = response_dir / ('frame_' + str(index).zfill(3) + '.json')
            response_path.write_bytes(response.content)
            frame_finished = time.perf_counter_ns()
            row.update({
                'export_name': record['export_name'], 'source_frame_number': record['frame_number'],
                'input_sha256': record['image_sha256'],
                'local_prepare_ms': (http_started - frame_start) / 1e6,
                'render_and_record_ms': (frame_finished - received) / 1e6,
                'frame_pipeline_ms': (frame_finished - frame_start) / 1e6,
                'object_count': len(payload['objects']),
                'objects_requiring_mask_review': sum(bool(obj['requires_review']) for obj in payload['objects']),
                'server_rss_after_response_mib': child.memory_info().rss / 1024**2,
                'frame_file': 'frames/' + frame_path.name, 'frame_file_sha256': sha(frame_path),
                'response_file': 'responses/' + response_path.name,
            })
            require(abs(row['local_prepare_ms'] + row['client_http_ms'] + row['render_and_record_ms']
                        - row['frame_pipeline_ms']) < 1e-5, 'Local replay intervals are not contiguous.')
            client_rows.append(row)
            if (index + 1) % 5 == 0:
                print('Rendered replay frames:', index + 1, '/30')
        loop_finished = time.perf_counter_ns()
        deadline = time.monotonic() + 10
        while True:
            lines = trace_path.read_text(encoding='utf-8').splitlines() if trace_path.exists() else []
            if len(lines) >= 32:
                break
            require(time.monotonic() < deadline, 'Incomplete replay server trace.')
            time.sleep(0.1)
        stop_owned_server()
        traces = [json.loads(line) for line in trace_path.read_text(encoding='utf-8').splitlines()]
        by_id = {item['request_id']: item for item in traces}
        require(len(traces) == len(by_id) == len(client_rows) == 32
                and set(by_id) == {row['request_id'] for row in client_rows}, 'Replay request correlation mismatch.')
        stage_keys = ['preprocess_ms', 'm4_predict_call_ms', 'm5_predict_call_ms', 'application_postprocess_ms']
        for row in client_rows:
            trace = by_id[row['request_id']]
            require(trace['success'] is True and trace['status_code'] == 200 and trace['error'] is None
                    and trace['response_complete'] is True, 'An unsuccessful replay trace was recorded.')
            require(trace['response_body_sha256'] == row['response_body_sha256']
                    and sha(runtime / row['response_file']) == row['response_body_sha256'], 'Response-byte hashes differ.')
            actual = trace['runtime']
            require(actual['detector_device'] == actual['segmenter_device'] == 'cpu'
                    and actual['detector_fp16'] is False and actual['segmenter_fp16'] is False
                    and actual['detector_parameter_dtypes'] == actual['segmenter_parameter_dtypes'] == ['torch.float32'],
                    'Actual replay model precision/device differ from CPU FP32.')
            require(set(trace['stages']) == set(stage_keys)
                    and abs(sum(trace['stages'].values()) - trace['route_total_ms']) < 1e-5,
                    'Server stage intervals disagree.')
            row.update({**trace['stages'], 'server_asgi_ms': trace['server_asgi_ms'],
                        'route_total_ms': trace['route_total_ms'], 'framework_remainder_ms': trace['framework_remainder_ms']})
        measured = [row for row in client_rows if row['phase'] == 'measured']
        require(len(measured) == len({row['export_name'] for row in measured}) == 30, 'Expected 30 distinct ordered input images.')
        csv_columns = list(dict.fromkeys(key for row in client_rows for key in row))
        with (output / 'requests_and_frames.csv').open('x', encoding='utf-8', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=csv_columns)
            writer.writeheader()
            writer.writerows(client_rows)
        metrics = {}
        metric_keys = ['client_http_ms', 'local_prepare_ms', 'render_and_record_ms', 'frame_pipeline_ms',
                       'server_asgi_ms', 'route_total_ms', 'framework_remainder_ms', *stage_keys]
        for key in metric_keys:
            values = np.asarray([row[key] for row in measured], dtype=np.float64)
            require(np.isfinite(values).all() and (values >= 0).all(), 'Invalid measured replay timing.')
            metrics[key] = {'mean_ms': float(values.mean()), 'p50_ms': float(np.percentile(values, 50, method='linear')),
                            'p95_ms': float(np.percentile(values, 95, method='linear')),
                            'p99_ms': float(np.percentile(values, 99, method='linear'))}

        print('Encoding the 30 annotated images as an H.264 replay at chosen display speed 3 fps...')
        video_path = runtime / 'ZeroWaste_VAL_ordered_replay.mp4'
        encoder_command = [ffmpeg, '-hide_banner', '-loglevel', 'error', '-nostdin', '-n',
                           '-framerate', str(display_fps), '-start_number', '0',
                           '-i', str(frame_dir / 'frame_%03d.jpg'), '-frames:v', '30', '-an',
                           '-c:v', 'libx264', '-preset', 'medium', '-crf', '20',
                           '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(video_path)]
        encoding_started = time.perf_counter()
        encoding = subprocess.run(encoder_command, capture_output=True, text=True, timeout=180)
        require(encoding.returncode == 0 and video_path.is_file(), 'Video encoding failed: ' + encoding.stderr[-1500:])
        encoding_seconds = time.perf_counter() - encoding_started
        capture = cv2.VideoCapture(str(video_path))
        require(capture.isOpened(), 'Encoded MP4 cannot be reopened for verification.')
        try:
            actual_fps = float(capture.get(cv2.CAP_PROP_FPS))
            width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
            decoded = 0
            while True:
                ok, frame = capture.read()
                if not ok:
                    break
                require(frame.shape == (840, 1280, 3), 'A decoded video frame has incorrect dimensions.')
                decoded += 1
            require(decoded == 30 and (width, height) == (1280, 840)
                    and abs(actual_fps - display_fps) < 0.01, 'Video frame count, resolution or display rate differs.')
        finally:
            capture.release()
        after = {key: sha(path) for key, path in protected.items()}
        input_after = {row['export_name']: sha(row['path']) for row in records}
        require(before == after and input_before == input_after, 'A protected project source or PNG changed.')
        loop_seconds = (loop_finished - loop_start) / 1e9
        summary = {
            'status': 'complete', 'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            'split': 'val', 'sequence': '07', 'unique_sampled_source_images': 30,
            'warmup_requests': 2, 'measured_requests': 30, 'successful_http_predictions': 32,
            'source_frame_numbers': [row['frame_number'] for row in records],
            'display_fps_chosen': display_fps, 'decoded_video_frames': decoded,
            'decoded_video_fps': actual_fps, 'encoded_replay_duration_seconds': decoded / actual_fps,
            'native_capture_fps': None, 'native_video_realtime_performance_claimed': False,
            'metrics': metrics,
            'http_only_images_per_second_from_summed_request_durations': 30 / (sum(row['client_http_ms'] for row in measured) / 1000),
            'local_pipeline_images_per_second_from_summed_frame_intervals': 30 / (sum(row['frame_pipeline_ms'] for row in measured) / 1000),
            'measured_loop_wall_seconds': loop_seconds, 'observed_replay_loop_frames_per_second': 30 / loop_seconds,
            'max_server_rss_after_measured_responses_mib': max(row['server_rss_after_response_mib'] for row in measured),
            'rss_is_continuous_peak': False, 'video_encoding_seconds_excluded_from_loop': encoding_seconds,
            'actual_predictor_runtime': traces[-1]['runtime'], 'protected_sources_unchanged': True,
            'protected_sha256_before': before, 'protected_sha256_after': after,
            'input_sha256_before': input_before, 'input_sha256_after': input_after,
            'video_file': video_path.name, 'video_sha256': sha(video_path),
            'limits': protocol['limits'], 'training_run': False, 'test_evaluated': False,
            'tracking_or_physical_grasp_success_measured': False,
        }
        save_json('summary.json', summary)
        save_json('encoder_and_video_check.json', {'command': encoder_command, 'returncode': encoding.returncode,
            'stderr': encoding.stderr, 'decoded_frames': decoded, 'decoded_fps': actual_fps,
            'decoded_resolution': [width, height], 'chosen_display_fps': display_fps,
            'audio': False, 'source_capture_fps': None})
        for name in ['server_trace.jsonl', 'uvicorn.log']:
            shutil.copyfile(runtime / name, output / name)
        shutil.copytree(frame_dir, output / 'frames')
        shutil.copytree(response_dir, output / 'responses')
        shutil.copyfile(video_path, output / video_path.name)
        shutil.copyfile(frame_dir / 'frame_000.jpg', output / 'replay_poster.jpg')
        (output / 'README.md').write_text(
            '# Ordered ZeroWaste Validation replay\n\n' + protocol['scope'] + '\n\n' + protocol['limits']
            + '\n\nThirty original images are from one source sequence in increasing frame-number order, sampled '
              'every10source-frame indices. The encoded3fps speed is chosen for viewing and does not reproduce known '
              'camera timing. Overlays show M4 boxes and only M5 masks associated by the original HTTP pipeline; '
              'REVIEW means absent valid mask association, not a ground-truth error verdict. No tracking is performed. '
              'Two warm-up rows are retained and excluded from all measured summaries. Raw response bytes, input '
              'image SHA anchors, per-frame request/timing records and all30annotatedJPEGs accompany the MP4. '
              'HTTP-only, localpipeline and loop-wall throughput have distinct boundaries. Setup/download, warm-up, '
              'video encoding and final Drive copy/ZIP are excluded from the measuredloop. No previous evidence was overwritten.\n',
            encoding='utf-8')
        zip_path = Path('/content') / ('Project9_ZEROWASTE_REPLAY_VAL_' + stamp + '.zip')
        receipts = []
        with zipfile.ZipFile(zip_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(output.rglob('*')):
                if path.is_file():
                    name = path.relative_to(output).as_posix()
                    receipts.append({'file': name, 'bytes': path.stat().st_size, 'sha256': sha(path)})
                    archive.write(path, arcname=name)
            archive.writestr('collection_manifest.json', json.dumps({'files': receipts, 'split': 'val',
                'note': 'Inventory excludes itself. Source PNGs and model binaries are not packaged.'}, indent=2) + '\n')
        with zipfile.ZipFile(zip_path) as archive:
            require(archive.testzip() is None, 'Replay ZIP CRC check failed.')
            for receipt in receipts:
                data = archive.read(receipt['file'])
                require(len(data) == receipt['bytes'] and hashlib.sha256(data).hexdigest() == receipt['sha256'],
                        'Packaged replay evidence hash mismatch.')
        print('=== ORDERED VALIDATION REPLAY COMPLETE ===')
        print('30 sampled images | chosen display3fps | replay10seconds | CPU FP32 realHTTP requests')
        print('ClientHTTP p50/p95(ms):', round(metrics['client_http_ms']['p50_ms'], 3), '/', round(metrics['client_http_ms']['p95_ms'], 3))
        print('Observed replay-loop throughput(frames/s):', round(summary['observed_replay_loop_frames_per_second'], 4))
        print('Display speed is chosen; source camera speed and native-video real-time performance are unknown.')
        print('Saved in Drive:', output)
        print('ZIP ready:', zip_path)
        try:
            from IPython.display import display, Video
            display(Video(str(video_path), embed=True, width=850))
        except Exception as preview_error:
            print('Inline preview unavailable; the verified MP4 remains in the ZIP:', type(preview_error).__name__)
        from google.colab import files
        files.download(str(zip_path))
        return summary
    except Exception as error:
        stop_owned_server()
        for name in ['server_trace.jsonl', 'uvicorn.log']:
            if (runtime / name).exists() and not (output / name).exists():
                shutil.copyfile(runtime / name, output / name)
        save_json('failure.json', {'status': 'failed', 'error': str(error), 'traceback': traceback.format_exc(),
                  'completed_client_requests': len(client_rows), 'protected_sha256_before': before,
                  'protected_sha256_at_failure': {key: sha(path) if path.is_file() else None for key, path in protected.items()}})
        print('Replay stopped. New diagnostic folder:', output)
        raise
    finally:
        stop_owned_server()
        session.close()


zerowaste_replay_result = run_zerowaste_replay(_REPLAY_PREPARATION)


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# انسخ الملف كاملًا إلى خلية جديدة في Colab وشغّل هذه الخلية فقط.
# يجمع حالة ملفات Visual Search وCLIP. لا تنزيل أوزان، ولا تدريب، ولا تنبؤ أو تقييم Test.
from google.colab import drive, files
drive.mount('/content/drive')

from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
import csv
import hashlib
import io
import json
import os
import shutil
import subprocess
import zipfile

ext_root = Path('/content/drive/MyDrive/Project9_Phase2')
ext_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

def ext_sha(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            result.update(chunk)
    return result.hexdigest()

def ext_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

ext_sources = {
    'visual_search_comparison': ext_root / 'open_world/visual_search/comparison_VAL.json',
    'gallery_index': ext_root / 'open_world/visual_search/gallery_index_train.npz',
    'clip_comparison': ext_root / 'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv',
    'split_manifest': ext_root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv',
    'restoration_inventory': ext_root / 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json',
    'restoration_build': ext_root / 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/build_info.json',
    'service_source': ext_root / 'deployment/http_service/service.py',
    'frozen_test_protocol': ext_root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
    'frozen_test_results': ext_root / 'evaluation/FINAL_TEST_ONCE/results.json',
}
if not ext_root.is_dir():
    raise RuntimeError('Project9_Phase2 folder is missing in the mounted Drive. Share this output.')
ext_before = {key: ext_sha(path) for key, path in ext_sources.items() if path.is_file()}
ext_output = ext_root / 'evaluation' / ('EXTENSIONS_READINESS_' + ext_stamp)
ext_output.mkdir(parents=False, exist_ok=False)
ext_info = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'scope': 'Read-only state collection for the two extension topics; writes new evidence only. No model weights loaded, model download, inference, training or Test evaluation.',
    'versions': {key: ext_version(key) for key in ['torch', 'torchvision', 'transformers', 'tokenizers',
                                                 'huggingface-hub', 'numpy', 'scikit-learn', 'Pillow']},
    'existing_files': {}, 'missing_files': [], 'notes': [],
}
ext_packaged = []
for ext_key, ext_path in ext_sources.items():
    if not ext_path.is_file():
        ext_info['missing_files'].append(ext_key)
        continue
    ext_info['existing_files'][ext_key] = {'path': str(ext_path), 'bytes': ext_path.stat().st_size,
                                          'sha256': ext_before[ext_key]}
    if ext_key in ['visual_search_comparison', 'gallery_index', 'clip_comparison',
                   'restoration_build', 'service_source']:
        if ext_path.stat().st_size <= 5 * 1024**2:
            ext_name = ext_key + ext_path.suffix
            shutil.copyfile(ext_path, ext_output / ext_name)
            ext_packaged.append(ext_name)
        else:
            ext_info['notes'].append(ext_key + ' exceeds the 5MiB collection limit; hash/metadata only.')

ext_gallery_path = ext_sources['gallery_index']
if ext_gallery_path.is_file():
    try:
        import numpy as np
        with np.load(ext_gallery_path, allow_pickle=False) as ext_index:
            ext_info['gallery_index'] = {'arrays': {}}
            for ext_name in ext_index.files:
                ext_array = ext_index[ext_name]
                ext_info['gallery_index']['arrays'][ext_name] = {'shape': list(ext_array.shape),
                                                               'dtype': str(ext_array.dtype)}
                if ext_name == 'labels':
                    ext_values, ext_counts = np.unique(ext_array, return_counts=True)
                    ext_info['gallery_index']['label_counts'] = {
                        str(int(key)): int(value) for key, value in zip(ext_values, ext_counts)}
                elif ext_name == 'source_paths':
                    ext_info['gallery_index']['source_image_basenames'] = [Path(str(value)).name for value in ext_array]
                elif ext_name == 'embeddings':
                    ext_info['gallery_index']['embeddings_are_finite'] = bool(np.isfinite(ext_array).all())
    except Exception as ext_error:
        ext_info['gallery_index_read_error'] = type(ext_error).__name__ + ': ' + str(ext_error)

if ext_sources['visual_search_comparison'].is_file():
    ext_info['saved_visual_search_result'] = json.loads(ext_sources['visual_search_comparison'].read_text(encoding='utf-8'))
if ext_sources['clip_comparison'].is_file():
    ext_info['saved_clip_result'] = list(csv.DictReader(io.StringIO(
        ext_sources['clip_comparison'].read_text(encoding='utf-8-sig'))))
if ext_sources['split_manifest'].is_file():
    ext_rows = list(csv.DictReader(io.StringIO(ext_sources['split_manifest'].read_text(encoding='utf-8-sig'))))
    ext_info['manifest_image_counts'] = {split: sum(row['split'] == split for row in ext_rows)
                                         for split in ['train', 'val', 'test', 'excluded']}
if ext_sources['restoration_inventory'].is_file():
    ext_inventory = json.loads(ext_sources['restoration_inventory'].read_text(encoding='utf-8'))
    ext_info['saved_validation_inventory_count'] = len(ext_inventory)

ext_torch_cache = Path(os.environ.get('TORCH_HOME', str(Path.home() / '.cache/torch'))) / 'hub/checkpoints'
ext_resnet_cache = ext_torch_cache / 'resnet18-f37072fd.pth'
ext_info['resnet18_weight_cache'] = {'path': str(ext_resnet_cache), 'exists': ext_resnet_cache.is_file(),
    'bytes': ext_resnet_cache.stat().st_size if ext_resnet_cache.is_file() else None}
ext_hf_cache = Path(os.environ.get('HF_HUB_CACHE', os.environ.get('HUGGINGFACE_HUB_CACHE',
    str(Path(os.environ.get('HF_HOME', str(Path.home() / '.cache/huggingface'))) / 'hub'))))
ext_clip_cache = ext_hf_cache / 'models--openai--clip-vit-base-patch32/snapshots'
ext_info['clip_cached_snapshots'] = []
if ext_clip_cache.is_dir():
    for ext_snapshot in sorted(ext_clip_cache.iterdir()):
        if not ext_snapshot.is_dir():
            continue
        ext_files_found = []
        for ext_name in ['config.json', 'preprocessor_config.json', 'tokenizer_config.json',
                         'tokenizer.json', 'vocab.json', 'merges.txt', 'pytorch_model.bin', 'model.safetensors']:
            ext_cached_file = ext_snapshot / ext_name
            if ext_cached_file.is_file():
                ext_files_found.append({'name': ext_name, 'bytes': ext_cached_file.stat().st_size})
        ext_info['clip_cached_snapshots'].append({'revision': ext_snapshot.name, 'files': ext_files_found})

ext_gpu_query = shutil.which('nvidia-smi')
if ext_gpu_query:
    try:
        ext_gpu_result = subprocess.run([ext_gpu_query, '--query-gpu=name,memory.total', '--format=csv,noheader'],
                                        capture_output=True, text=True, timeout=15, check=False)
        ext_info['gpu_inventory'] = {'exit_code': ext_gpu_result.returncode,
                                      'output': ext_gpu_result.stdout.strip()[:1000]}
    except (OSError, subprocess.TimeoutExpired) as ext_gpu_error:
        ext_info['gpu_inventory'] = {'query_error': type(ext_gpu_error).__name__}
else:
    ext_info['gpu_inventory'] = {'available_command': False}

ext_after = {key: ext_sha(ext_sources[key]) for key in ext_before}
if ext_before != ext_after:
    raise RuntimeError('A source reference changed during collection; stop and share this output.')
ext_info['protected_sha256_before'] = ext_before
ext_info['protected_sha256_after'] = ext_after
ext_info['source_files_unchanged'] = True
with (ext_output / 'readiness.json').open('x', encoding='utf-8') as ext_stream:
    json.dump(ext_info, ext_stream, indent=2, ensure_ascii=False, allow_nan=False)

ext_zip = Path('/content') / ('Project9_EXTENSIONS_READINESS_' + ext_stamp + '.zip')
ext_receipts = []
with zipfile.ZipFile(ext_zip, 'x', compression=zipfile.ZIP_DEFLATED) as ext_archive:
    for ext_path in sorted(ext_output.iterdir()):
        if ext_path.is_file():
            ext_receipts.append({'file': ext_path.name, 'bytes': ext_path.stat().st_size, 'sha256': ext_sha(ext_path)})
            ext_archive.write(ext_path, arcname=ext_path.name)
    ext_archive.writestr('collection_manifest.json', json.dumps({'files': ext_receipts,
        'note': 'Inventory excludes itself. No source images, model binaries or account tokens are packaged.'}, indent=2))
with zipfile.ZipFile(ext_zip) as ext_archive:
    if ext_archive.testzip() is not None:
        raise RuntimeError('ZIP integrity check failed.')
    for ext_receipt in ext_receipts:
        ext_bytes = ext_archive.read(ext_receipt['file'])
        if len(ext_bytes) != ext_receipt['bytes'] or hashlib.sha256(ext_bytes).hexdigest() != ext_receipt['sha256']:
            raise RuntimeError('Packaged evidence hash mismatch.')

print('=== EXTENSION READINESS COLLECTED ===')
print('Missing saved references:', ext_info['missing_files'])
print('Gallery arrays:', ext_info.get('gallery_index', {}).get('arrays',
    ext_info.get('gallery_index_read_error', 'not found')))
print('No model execution, training or Test evaluation. Original project files unchanged.')
print('ZIP:', ext_zip)
files.download(str(ext_zip))


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# انسخ هذا الملف كاملًا إلى خلية جديدة في Colab وشغّلها فقط.
# استعادة عينات البحث الأصلية + مقارنة Validation + خمسة طلبات فعلية لمساعدة المراجع.
# لا تدريب ولا تقييم Test ولا تعديل للملفات أو النتائج السابقة.
# قد ينزّل أوزان المساعدة الرسمية (~650 MB)، وأجزاء الصور اللازمة فقط.

_EXTENSION_SOURCES = {'prepare_extension_cohorts.py': '"""Reconstruct the original Train-gallery/Val-query extension cohorts.\n\nDefining the function does not execute models, install packages, or access data.\nOnly source Train/Val archive members can be retrieved; Test is never opened.\n"""\n\n\ndef prepare_extension_cohorts():\n    from collections import Counter, OrderedDict, defaultdict\n    from datetime import datetime, timezone\n    from importlib.metadata import version\n    from pathlib import Path, PurePosixPath\n    import csv\n    import hashlib\n    import io\n    import json\n    import math\n    import re\n    import time\n    import zipfile\n    import zlib\n    import numpy as np\n    import requests\n    from PIL import Image\n    from google.colab import drive\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def digest_bytes(data):\n        return hashlib.sha256(data).hexdigest()\n\n    def digest(path):\n        value = hashlib.sha256()\n        with Path(path).open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b\'\'):\n                value.update(chunk)\n        return value.hexdigest()\n\n    def write_new(path, data):\n        with Path(path).open(\'xb\') as stream:\n            stream.write(data)\n\n    drive.mount(\'/content/drive\')\n    root = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    local_root = Path(\'/content/Project9_Phase2\')\n    manifest_path = root / \'data_protocol/ef6b35d2ba7f/split_manifest.csv\'\n    inventory_path = root / \'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json\'\n    original_index_path = root / \'open_world/visual_search/gallery_index_train.npz\'\n    expected_manifest = \'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50\'\n    expected_inventory = \'86e913fa48c09c596f396ebfaeefa49ff2b720d3cf9983f6645bd3f911734835\'\n    expected_index = \'8162cf1b1f4eff5741bcad74230fc0bee3501e27ae89948ddc2c7af0c3532d57\'\n    expected_train_json = \'242ba81def2ee6162f7341f0ee48e21e380411e9fac03df26a270d1a869f860a\'\n    classes = [\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\']\n    protected = {\n        \'split_manifest\': manifest_path, \'validation_inventory\': inventory_path,\n        \'gallery_index\': original_index_path,\n        \'visual_search_result\': root / \'open_world/visual_search/comparison_VAL.json\',\n        \'clip_result\': root / \'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv\',\n        \'service_source\': root / \'deployment/http_service/service.py\',\n        \'service_manifest\': root / \'deployment/http_service/release_manifest.json\',\n        \'http_detector\': root / \'deployment/http_service/best_det.pt\',\n        \'http_segmenter\': root / \'deployment/http_service/best_seg.pt\',\n        \'source_detector\': root / \'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt\',\n        \'source_segmenter\': root / \'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt\',\n        \'test_protocol\': root / \'evaluation/FINAL_TEST_ONCE/frozen_protocol.json\',\n        \'test_results\': root / \'evaluation/FINAL_TEST_ONCE/results.json\',\n    }\n    expected_protected = {\n        \'split_manifest\': expected_manifest, \'validation_inventory\': expected_inventory,\n        \'gallery_index\': expected_index,\n        \'visual_search_result\': \'312d1739e7edfde1e4d3a6ac7b8022f12c135c129bdb2fac9988c47f80f9174d\',\n        \'clip_result\': \'7d770ea93ee9075b79491c759f7166fb95e2bb620578ebf9d49959085bacdf59\',\n        \'service_source\': \'6a10bbddd6e675f1b65a8c95f4bf2d3a439ab9a97a85b4caf80e9cd108753bd2\',\n        \'service_manifest\': \'c4c6c1b9bbb70ae17922e6c253ebf432482f92332728d0440555a1fd7a7b4903\',\n        \'http_detector\': \'90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7\',\n        \'http_segmenter\': \'1ceaa151c3dc3d3503d2812216309322be62d25eebfcd57e35f513036412f178\',\n        \'source_detector\': \'90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7\',\n        \'source_segmenter\': \'2cf68941dff6ee01b15ff8947d2f12a8737bdc26e5cc59ec8783e10204179c10\',\n        \'test_protocol\': \'aca3f6749633875b719c47c8bddff3f79c0a9da28f35246384a79f6d4cf9faff\',\n        \'test_results\': \'b6f48bf11f862930c4a1d34a4290d2fea3ed3918f2ccc153fef169cb78cb91dc\',\n    }\n    missing = [str(path) for path in protected.values() if not path.is_file()]\n    require(not missing, \'Missing saved project references; stop and share this output: \' + repr(missing))\n    before = {name: digest(path) for name, path in protected.items()}\n    for name, expected in expected_protected.items():\n        require(before[name] == expected, \'Saved reference/model bytes changed: \' + name)\n    rows = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding=\'utf-8-sig\'))))\n    active_rows = [row for row in rows if row[\'split\'] in (\'train\', \'val\')]\n    require(Counter(row[\'split\'] for row in active_rows) == {\'train\': 1851, \'val\': 451},\n            \'Frozen Train/Validation image counts differ.\')\n    require(all(row[\'original_split\'] in (\'train\', \'val\') for row in active_rows),\n            \'Train/Validation references unexpected source partitions.\')\n    by_export = {row[\'original_split\'] + \'__\' + row[\'filename\']: row for row in active_rows}\n    require(len(by_export) == len(active_rows), \'Active export filenames are not unique.\')\n    inventory_items = json.loads(inventory_path.read_text(encoding=\'utf-8\'))\n    inventory = {item[\'export_name\']: item for item in inventory_items}\n    require(len(inventory_items) == len(inventory) == 451, \'Saved Validation inventory differs.\')\n    with np.load(original_index_path, allow_pickle=False) as bank:\n        require(set(bank.files) == {\'embeddings\', \'labels\', \'source_paths\'}, \'Unexpected historical index arrays.\')\n        historical_labels = bank[\'labels\'].copy()\n        historical_names = [PurePosixPath(str(value).replace(\'\\\\\', \'/\')).name for value in bank[\'source_paths\']]\n        require(bank[\'embeddings\'].shape == (160, 512) and np.isfinite(bank[\'embeddings\']).all()\n                and historical_labels.shape == (160,) and len(historical_names) == 160,\n                \'Historical index dimensions/values differ.\')\n        require(Counter(int(value) for value in historical_labels) == {0: 40, 1: 40, 2: 40, 3: 40},\n                \'Historical gallery is not the recorded balanced 160-object bank.\')\n\n    archive_url = \'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1\'\n    archive_size = 7518242799\n    password = b\'REDACTED_DATASET_ZIP_PASSWORD\'\n\n    class HTTPRangeFile(io.RawIOBase):\n        """Bounded read-only seekable HTTP file for Python\'s ZIP64 reader."""\n        def __init__(self):\n            super().__init__()\n            self.position = 0\n            self.size = archive_size\n            self.block_size = 1024 * 1024\n            self.blocks = OrderedDict()\n            self.transferred = 0\n            self.requests_count = 0\n            self.session = requests.Session()\n\n        def readable(self):\n            return True\n\n        def seekable(self):\n            return True\n\n        def tell(self):\n            return self.position\n\n        def seek(self, offset, whence=0):\n            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1\n            if position < 0:\n                raise ValueError(\'Invalid HTTP ZIP seek.\')\n            self.position = int(position)\n            return self.position\n\n        def get_block(self, index):\n            if index in self.blocks:\n                self.blocks.move_to_end(index)\n                return self.blocks[index]\n            start = index * self.block_size\n            end = min(self.size - 1, start + self.block_size - 1)\n            needed = end - start + 1\n            require(self.transferred + needed <= 768 * 1024**2,\n                    \'Partial-download budget exceeded; the full archive will not be downloaded automatically.\')\n            for attempt in range(3):\n                try:\n                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={\n                        \'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\', \'Cache-Control\': \'no-cache\'\n                    }) as response:\n                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:\n                            time.sleep(2 * (attempt + 1))\n                            continue\n                        require(response.status_code == 206,\n                                f\'HTTP {response.status_code}: requested range was not supported; full download prevented.\')\n                        content_range = response.headers.get(\'Content-Range\', \'\').strip()\n                        match = re.fullmatch(r\'bytes (\\d+)-(\\d+)/(\\d+)\', content_range, flags=re.I)\n                        require(match and tuple(map(int, match.groups())) == (start, end, self.size),\n                                \'Server returned an unexpected Content-Range: \' + content_range)\n                        data = response.raw.read(needed + 1)\n                        require(len(data) == needed, \'Incomplete or oversized HTTP range response.\')\n                    break\n                except requests.RequestException:\n                    if attempt == 2:\n                        raise\n                    time.sleep(2 * (attempt + 1))\n            self.transferred += len(data)\n            self.requests_count += 1\n            self.blocks[index] = data\n            while len(self.blocks) > 16:\n                self.blocks.popitem(last=False)\n            if self.requests_count % 5 == 0:\n                print(f\'Partial ZIP transfer: {self.transferred / 1024**2:.1f} MiB\')\n            return data\n\n        def read(self, count=-1):\n            if self.closed:\n                raise ValueError(\'HTTP ZIP stream is closed.\')\n            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))\n            require(count <= 64 * 1024**2, \'Unexpected large ZIP read; complete-archive download prevented.\')\n            chunks = []\n            remaining = count\n            while remaining:\n                index, offset = divmod(self.position, self.block_size)\n                data = self.get_block(index)\n                take = min(remaining, len(data) - offset)\n                require(take > 0, \'Unexpected ZIP read position.\')\n                chunks.append(data[offset:offset + take])\n                self.position += take\n                remaining -= take\n            return b\'\'.join(chunks)\n\n        def close(self):\n            self.session.close()\n            self.blocks.clear()\n            super().close()\n\n    annotation_cache = local_root / \'extensions_annotation_cache\'\n    image_cache = local_root / \'extensions_source_images\'\n    annotation_cache.mkdir(parents=True, exist_ok=True)\n    image_cache.mkdir(parents=True, exist_ok=True)\n    range_file = None\n    archive = None\n    archive_mode = None\n    annotation_provenance = {}\n    image_provenance = []\n\n    def open_archive():\n        nonlocal range_file, archive, archive_mode\n        if archive is None:\n            candidates = [Path(\'/content/zerowaste-f-final.zip\'), root / \'cache/zerowaste-f-final.zip\']\n            local = next((path for path in candidates if path.is_file() and path.stat().st_size == archive_size), None)\n            if local is not None:\n                archive = zipfile.ZipFile(local)\n                archive_mode = \'existing_local_archive_selected_members\'\n            else:\n                print(\'Reading only required source Train/Val members by bounded HTTP ranges.\')\n                range_file = HTTPRangeFile()\n                archive = zipfile.ZipFile(range_file)\n                archive_mode = \'bounded_http_ranges_selected_members\'\n        return archive\n\n    def member_for(suffix):\n        require(suffix.startswith((\'train/\', \'val/\')) and not suffix.startswith(\'test/\'),\n                \'Only original source Train/Val members are permitted.\')\n        choices = [item for item in open_archive().infolist() if not item.is_dir()\n                   and (item.filename == suffix or item.filename.endswith(\'/\' + suffix))\n                   and \'..\' not in PurePosixPath(item.filename).parts]\n        require(len(choices) == 1, \'Source ZIP member is not unique: \' + suffix)\n        return choices[0]\n\n    def check_member_bytes(data, info, expected_sha=None):\n        require(len(data) == info.file_size and (zlib.crc32(data) & 0xffffffff) == info.CRC,\n                \'Retrieved/cached member size or CRC differs: \' + info.filename)\n        value = digest_bytes(data)\n        if expected_sha is not None:\n            require(value == expected_sha, \'Historical member SHA256 differs: \' + info.filename)\n        return value\n\n    def save_sidecar(path, data):\n        sidecar = Path(str(path) + \'.source.json\')\n        encoded = (json.dumps(data, sort_keys=True, indent=2) + \'\\n\').encode(\'utf-8\')\n        if sidecar.exists():\n            old = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n            for key in [\'sha256\', \'member\', \'member_crc32\', \'member_bytes\', \'archive_url\', \'archive_total_bytes\']:\n                require(old.get(key) == data[key], \'Existing cache provenance differs: \' + str(sidecar))\n        else:\n            write_new(sidecar, encoded)\n        return sidecar\n\n    def verify_cached(path, info, expected_sha=None):\n        require(path.is_file(), \'Cache entry is not a file: \' + str(path))\n        data = path.read_bytes()\n        value = check_member_bytes(data, info, expected_sha)\n        if expected_sha is None:\n            sidecar = Path(str(path) + \'.source.json\')\n            require(sidecar.is_file(), \'Unanchored cached Train/Val source requires its saved receipt: \' + str(path))\n            old = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n            require(old.get(\'sha256\') == value and old.get(\'member\') == info.filename\n                    and old.get(\'member_crc32\') == info.CRC and old.get(\'member_bytes\') == info.file_size\n                    and old.get(\'archive_url\') == archive_url and old.get(\'archive_total_bytes\') == archive_size,\n                    \'Existing source cache differs from its receipt: \' + str(path))\n        return data, value\n\n    try:\n        coco_sources = {}\n        for source_split in [\'train\', \'val\']:\n            info = member_for(source_split + \'/labels.json\')\n            require(0 < info.file_size <= 256 * 1024**2, \'Unexpected COCO source annotation size.\')\n            path = annotation_cache / (source_split + \'_labels.json\')\n            expected = expected_train_json if source_split == \'train\' else None\n            if path.exists():\n                data, annotation_sha = verify_cached(path, info, expected)\n                mode = \'existing_verified_annotation_cache\'\n            else:\n                old_train_cache = local_root / \'failure_inputs_VAL_v1/source_train_annotations.json\'\n                if source_split == \'train\' and old_train_cache.is_file():\n                    data = old_train_cache.read_bytes()\n                    mode = \'reused_previously_verified_train_json\'\n                else:\n                    data = open_archive().read(info, pwd=password)\n                    mode = \'retrieved_source_member\'\n                annotation_sha = check_member_bytes(data, info, expected)\n                write_new(path, data)\n            receipt = {\'sha256\': annotation_sha, \'member\': info.filename, \'member_crc32\': info.CRC,\n                       \'member_bytes\': info.file_size, \'archive_url\': archive_url,\n                       \'archive_total_bytes\': archive_size}\n            save_sidecar(path, receipt)\n            annotation_provenance[source_split] = {**receipt, \'mode\': mode, \'cache_path\': str(path),\n                \'historical_sha256_anchor_available\': source_split == \'train\',\n                \'historical_anchor_note\': (\'Matches the saved original Train COCO JSON hash.\' if source_split == \'train\'\n                                          else \'New source Val JSON hash recorded from a CRC-verified member; no historical JSON SHA anchor is claimed.\')}\n            coco = json.loads(data.decode(\'utf-8-sig\'))\n            images = {int(item[\'id\']): item for item in coco[\'images\']}\n            categories = {int(item[\'id\']): item[\'name\'] for item in coco[\'categories\']}\n            require(len(images) == len(coco[\'images\']) and set(categories.values()) == set(classes),\n                    \'COCO image identities or category names differ.\')\n            anns = defaultdict(list)\n            for ann in coco[\'annotations\']:\n                anns[int(ann[\'image_id\'])].append(ann)\n            coco_sources[source_split] = {\'images\': images, \'categories\': categories, \'annotations\': anns}\n            print(\'Verified source annotation JSON:\', source_split)\n\n        # This reproduces notebook cell10 followed by the file-order reading in cell51.\n        pools = {\'train\': defaultdict(list), \'val\': defaultdict(list)}\n        image_metadata = {}\n        pool_counts = {}\n        for export_name in sorted(by_export):\n            row = by_export[export_name]\n            source_split = row[\'original_split\']\n            source = coco_sources[source_split]\n            image_id = int(row[\'image_id\'])\n            require(image_id in source[\'images\'], \'COCO image ID missing: \' + export_name)\n            info = source[\'images\'][image_id]\n            require(PurePosixPath(info[\'file_name\']).name == row[\'filename\'], \'COCO filename differs: \' + export_name)\n            width, height = int(info[\'width\']), int(info[\'height\'])\n            require(width > 0 and height > 0, \'Invalid COCO image dimensions.\')\n            image_metadata[export_name] = {\'width\': width, \'height\': height,\n                                          \'image_id\': image_id, \'frozen_manifest_row\': row}\n            class_counts = Counter()\n            for ann_ordinal, ann in enumerate(source[\'annotations\'][image_id]):\n                class_name = source[\'categories\'][int(ann[\'category_id\'])]\n                class_id = classes.index(class_name)\n                x, y, w, h = map(float, ann[\'bbox\'])\n                require(all(math.isfinite(value) for value in (x, y, w, h)) and w > 0 and h > 0,\n                        \'Invalid original COCO box: \' + export_name)\n                x1, y1, x2, y2 = (max(0, min(width, x)), max(0, min(height, y)),\n                                  max(0, min(width, x + w)), max(0, min(height, y + h)))\n                require(x2 > x1 and y2 > y1, \'Empty clipped original COCO box.\')\n                xc, yc, bw, bh = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height\n                line = f\'{class_id} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}\'\n                decoded_class, xc, yc, bw, bh = map(float, line.split())\n                require(int(decoded_class) == class_id, \'Converted class identity differs.\')\n                class_counts[class_name] += 1\n                if bw > 0.003 and bh > 0.003:\n                    pools[row[\'split\']][class_id].append({\n                        \'export_name\': export_name, \'filename\': row[\'filename\'],\n                        \'original_split\': source_split, \'project_split\': row[\'split\'],\n                        \'image_id\': image_id, \'width\': width, \'height\': height,\n                        \'class_id\': class_id, \'class_name\': class_name,\n                        \'annotation_id\': int(ann[\'id\']), \'annotation_ordinal_in_image\': ann_ordinal,\n                        \'xywh_normalized\': [xc, yc, bw, bh], \'converted_yolo_line\': line,\n                        \'official_clipped_box_xyxy\': [x1, y1, x2, y2],\n                        \'frozen_manifest_row\': row,\n                    })\n            require(all(class_counts[name] == int(row[name]) for name in classes),\n                    \'COCO object counts differ from frozen manifest: \' + export_name)\n        for split in [\'train\', \'val\']:\n            pool_counts[split] = {classes[c]: len(pools[split][c]) for c in range(4)}\n            require(all(pools[split][c] for c in range(4)), \'A material pool is empty: \' + split)\n\n        rng = np.random.default_rng(42)\n\n        def choose_objects(split, per_class):\n            chosen = []\n            for class_id in range(len(classes)):\n                pool = pools[split][class_id]\n                indices = rng.choice(len(pool), size=min(per_class, len(pool)), replace=False)\n                chosen.extend(dict(pool[int(index)]) for index in indices)\n            return chosen\n\n        gallery = choose_objects(\'train\', 40)\n        queries = choose_objects(\'val\', 10)\n        require(len(gallery) == 160 and len(queries) == 40, \'Reconstructed balanced cohort sizes differ.\')\n        require([record[\'class_id\'] for record in gallery] == [int(value) for value in historical_labels]\n                and [record[\'export_name\'] for record in gallery] == historical_names,\n                \'Reconstructed gallery order/classes do not match the saved historical index; stop before image download.\')\n        print(\'Historical gallery order verified: all 160 source basenames and labels match.\')\n\n        operator_rows = []\n        for number in [1, 71, 141, 211, 291]:\n            found = [row for row in active_rows if row[\'split\'] == \'val\' and row[\'original_split\'] == \'train\'\n                     and row[\'sequence\'] == \'07\' and int(row[\'frame_number\']) == number]\n            require(len(found) == 1, \'Operator Validation selection differs.\')\n            operator_rows.append(found[0])\n        selected_names = sorted({record[\'export_name\'] for record in gallery + queries}\n                                | {\'train__\' + row[\'filename\'] for row in operator_rows})\n        member_infos = {}\n        for name in selected_names:\n            row = by_export[name]\n            require(row[\'split\'] in (\'train\', \'val\') and row[\'original_split\'] in (\'train\', \'val\'),\n                    \'Selected image is outside permitted project Train/Validation.\')\n            info = member_for(row[\'original_split\'] + \'/data/\' + row[\'filename\'])\n            require(0 < info.file_size <= 12 * 1024**2, \'Unexpected selected PNG size.\')\n            member_infos[name] = info\n        # Reading by header offset minimizes rereads in the bounded range cache.\n        download_order = sorted(selected_names, key=lambda name: member_infos[name].header_offset)\n        image_records = {}\n        downloaded, reused, borrowed = 0, 0, 0\n        print(\'Preparing unique source images:\', len(selected_names))\n        for ordinal, name in enumerate(download_order, 1):\n            row = by_export[name]\n            info = member_infos[name]\n            target = image_cache / name\n            expected_image_sha = None\n            if row[\'split\'] == \'val\':\n                require(name in inventory and inventory[name][\'image_id\'] == int(row[\'image_id\']),\n                        \'Project Validation image identity differs from historical inventory.\')\n                expected_image_sha = inventory[name][\'image_sha256\']\n            if target.exists():\n                data, image_sha = verify_cached(target, info, expected_image_sha)\n                mode = \'existing_verified_extension_image_cache\'\n                reused += 1\n            else:\n                old_val = local_root / \'failure_inputs_VAL_v1/images\' / name\n                if expected_image_sha is not None and old_val.is_file():\n                    data = old_val.read_bytes()\n                    mode = \'reused_historical_sha_verified_validation_cache\'\n                    borrowed += 1\n                else:\n                    data = open_archive().read(info, pwd=password)\n                    mode = \'retrieved_source_member\'\n                    downloaded += 1\n                image_sha = check_member_bytes(data, info, expected_image_sha)\n                with Image.open(io.BytesIO(data)) as image:\n                    expected_dimensions = (image_metadata[name][\'width\'], image_metadata[name][\'height\'])\n                    require(image.format == \'PNG\' and image.size == expected_dimensions,\n                            \'Original PNG dimensions/format differ: \' + name)\n                    image.verify()\n                write_new(target, data)\n            with Image.open(io.BytesIO(data)) as image:\n                require(image.format == \'PNG\' and image.size == (image_metadata[name][\'width\'], image_metadata[name][\'height\']),\n                        \'Cached original PNG dimensions/format differ: \' + name)\n                image.verify()\n            cache_receipt = {\'sha256\': image_sha, \'member\': info.filename, \'member_crc32\': info.CRC,\n                             \'member_bytes\': info.file_size, \'archive_url\': archive_url,\n                             \'archive_total_bytes\': archive_size}\n            save_sidecar(target, cache_receipt)\n            record = {\'export_name\': name, \'filename\': row[\'filename\'], \'path\': str(target),\n                      \'image_sha256\': image_sha, **image_metadata[name],\n                      \'original_split\': row[\'original_split\'], \'project_split\': row[\'split\']}\n            image_records[name] = record\n            image_provenance.append({**cache_receipt, \'export_name\': name, \'project_split\': row[\'split\'],\n                \'original_split\': row[\'original_split\'], \'mode\': mode,\n                \'historical_image_sha256_anchor_available\': expected_image_sha is not None,\n                \'hash_scope\': (\'Matches the earlier 451-image Validation inventory.\' if expected_image_sha is not None\n                               else \'New image SHA256 recorded from a source ZIP member verified by CRC/size; no historical Train-image SHA is claimed.\')})\n            if ordinal % 10 == 0 or ordinal == len(download_order):\n                print(\'Verified extension source images:\', ordinal, \'/\', len(download_order))\n\n        stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n        work_dir = local_root / (\'extension_inputs_\' + stamp)\n        (work_dir / \'gallery\').mkdir(parents=True, exist_ok=False)\n        (work_dir / \'queries\').mkdir(exist_ok=False)\n\n        def make_crops(objects, directory, prefix):\n            result = []\n            for ordinal, original in enumerate(objects):\n                record = dict(original)\n                source_image = image_records[record[\'export_name\']]\n                record[\'path\'] = source_image[\'path\']\n                record[\'image_sha256\'] = source_image[\'image_sha256\']\n                xc, yc, bw, bh = record[\'xywh_normalized\']\n                width, height = record[\'width\'], record[\'height\']\n                # Exact notebook cell51 crop_object arithmetic after 8-decimal readback.\n                x1 = max(0, int((xc - bw / 2) * width))\n                x2 = min(width, int(np.ceil((xc + bw / 2) * width)))\n                y1 = max(0, int((yc - bh / 2) * height))\n                y2 = min(height, int(np.ceil((yc + bh / 2) * height)))\n                crop_box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]\n                crop_file = directory + \'/\' + prefix + \'_\' + str(ordinal).zfill(3) + \'.png\'\n                crop_path = work_dir / crop_file\n                with Image.open(record[\'path\']) as image:\n                    rgb = image.convert(\'RGB\')\n                    require(rgb.size == (width, height), \'Source dimensions changed while cropping.\')\n                    crop = rgb.crop(tuple(crop_box))\n                    with crop_path.open(\'xb\') as stream:\n                        crop.save(stream, format=\'PNG\')\n                record.update({\'ordinal\': ordinal, \'crop_file\': crop_file, \'crop_path\': str(crop_path),\n                               \'crop_box_xyxy\': crop_box, \'crop_sha256\': digest(crop_path),\n                               \'crop_width\': crop_box[2] - crop_box[0], \'crop_height\': crop_box[3] - crop_box[1]})\n                result.append(record)\n            return result\n\n        gallery_records = make_crops(gallery, \'gallery\', \'g\')\n        query_records = make_crops(queries, \'queries\', \'q\')\n        operator_records = []\n        for ordinal, row in enumerate(operator_rows):\n            record = dict(image_records[\'train__\' + row[\'filename\']])\n            record.update({\'ordinal\': ordinal, \'frame_number\': int(row[\'frame_number\']), \'source_sequence\': \'07\'})\n            operator_records.append(record)\n        after = {name: digest(path) for name, path in protected.items()}\n        require(before == after, \'A protected saved project reference changed during preparation.\')\n        transferred = range_file.transferred if range_file is not None else 0\n        request_count = range_file.requests_count if range_file is not None else 0\n        provenance = {\n            \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n            \'scope\': \'Project Train gallery and Validation queries/operator images only; no Test data, inference or optimization.\',\n            \'class_names_in_id_order\': classes, \'manifest_sha256\': expected_manifest,\n            \'original_index_path\': str(original_index_path), \'historical_index_sha256\': expected_index,\n            \'historical_gallery_order_check\': {\'source_basenames_match\': True, \'class_ids_match\': True,\n                                             \'objects\': 160, \'before_any_png_download\': True},\n            \'historical_order_limit\': \'The saved index has source paths and labels, but no historical crop pixels, annotation IDs or boxes. Those are reconstructed from the source annotations and the recorded algorithm, not independently compared to unavailable historical crop files.\',\n            \'selection_algorithm\': \'Notebook cell10 clip/normalize bbox and format8dec; cell51 sorted export label order, original annotation order, strict w>0.003 and h>0.003; one NumPy default_rng(42), Train40perclass first, then Validation10perclass.\',\n            \'sampling_rng\': {\'seed\': 42, \'bit_generator\': type(rng.bit_generator).__name__,\n                             \'gallery_then_queries_share_one_rng\': True},\n            \'versions\': {name: version(name) for name in [\'numpy\', \'Pillow\', \'requests\']},\n            \'pool_counts_after_small_box_filter\': pool_counts,\n            \'gallery_size\': 160, \'query_size\': 40, \'operator_image_count\': 5,\n            \'operator_selection\': \'Frozen Validation source sequence07 frame indices1,71,141,211,291; fixed before model execution.\',\n            \'annotation_sources\': annotation_provenance, \'source_images\': image_provenance,\n            \'archive_url\': archive_url, \'archive_total_bytes\': archive_size, \'archive_mode\': archive_mode,\n            \'range_byte_budget\': 768 * 1024**2, \'range_bytes_transferred_this_call\': transferred,\n            \'range_requests_this_call\': request_count, \'whole_archive_md5_recomputed_this_call\': False,\n            \'unique_source_images\': len(image_records), \'downloaded_images_this_call\': downloaded,\n            \'reused_extension_image_cache\': reused, \'borrowed_historical_validation_cache\': borrowed,\n            \'source_train_images_have_new_sha_provenance\': True,\n            \'source_val_annotation_sha_is_new_not_historical\': True,\n            \'protected_sha256_before\': before, \'protected_sha256_after\': after,\n            \'protected_sources_unchanged\': True,\n            \'crop_format\': \'Lossless RGB PNG; exact recorded original crop_object integer/ceil arithmetic.\',\n        }\n        bundle = {\'gallery\': gallery_records, \'queries\': query_records, \'operator_images\': operator_records,\n                  \'provenance\': provenance, \'work_dir\': str(work_dir), \'project_root\': str(root),\n                  \'original_index_path\': str(original_index_path), \'historical_bank_sha256\': expected_index,\n                  \'source_image_cache\': str(image_cache)}\n        with (work_dir / \'cohort_provenance.json\').open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            json.dump(bundle, stream, indent=2, ensure_ascii=False, allow_nan=False)\n            stream.write(\'\\n\')\n        print(\'=== EXTENSION COHORTS READY ===\')\n        print(\'Gallery160 | Validation queries40 | operator images5 | unique source PNGs:\', len(image_records))\n        print(\'Partial source retrieval this call (MiB):\', round(transferred / 1024**2, 2))\n        print(\'Local lossless crop folder:\', work_dir)\n        return bundle\n    finally:\n        if archive is not None:\n            archive.close()\n        if range_file is not None:\n            range_file.close()\n', 'extension_assist_models.py': '"""CPU models and numeric helpers for the optional operator-assistance stage.\n\nImporting this module performs no model loading, downloads or inference.\nModel construction is explicit; all predictions are advisory and do not alter\nthe original detector/segmenter response or introduce a rejection threshold.\n"""\n\nCLASS_NAMES = (\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\')\nCLIP_MATERIAL_NAMES = (\'cardboard\', \'metal\', \'rigid plastic\', \'soft plastic\')\nPLAIN_PROMPTS = tuple(\'a photo of \' + name for name in CLIP_MATERIAL_NAMES)\nCLIP_MODEL_ID = \'openai/clip-vit-base-patch32\'\n\n\ndef _assist_require(condition, message):\n    if not condition:\n        raise RuntimeError(message)\n\n\ndef _assist_sha256(path):\n    from pathlib import Path\n    import hashlib\n    value = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b\'\'):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef unit(values):\n    """The original float32 L2 normalization, including its 1e-10 guard."""\n    import numpy as np\n    array = np.asarray(values, dtype=np.float32)\n    _assist_require(array.ndim == 2 and np.isfinite(array).all(), \'Expected finite 2D features.\')\n    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)\n\n\ndef rgb_hist(crops):\n    """Original 24-bin RGB histogram baseline; crops must be PIL images."""\n    import numpy as np\n    rows = []\n    for crop in crops:\n        array = np.asarray(crop.convert(\'RGB\').resize((64, 64)), dtype=np.uint8)\n        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,\n                                                range=(0, 256))[0] for channel in range(3)]))\n    _assist_require(bool(rows), \'At least one crop is required.\')\n    return unit(np.asarray(rows, dtype=np.float32))\n\n\ndef classification_metrics(true_labels, predicted_labels):\n    """Four-material accuracy and macro metrics with zero-division set to zero."""\n    import numpy as np\n    truth = np.asarray(true_labels)\n    prediction = np.asarray(predicted_labels)\n    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape\n                    and truth.size > 0, \'Class arrays must be nonempty, aligned 1D arrays.\')\n    _assist_require(np.issubdtype(truth.dtype, np.integer)\n                    and np.issubdtype(prediction.dtype, np.integer), \'Class IDs must be integers.\')\n    _assist_require(((truth >= 0) & (truth < 4)).all()\n                    and ((prediction >= 0) & (prediction < 4)).all(), \'Unknown material class ID.\')\n    matrix = np.zeros((4, 4), dtype=np.int64)\n    np.add.at(matrix, (truth, prediction), 1)\n    tp = np.diag(matrix).astype(np.float64)\n    predicted_support = matrix.sum(axis=0)\n    support = matrix.sum(axis=1)\n    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)\n    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)\n    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)\n    return {\n        \'accuracy\': float(np.mean(truth == prediction)), \'macro_f1\': float(f1.mean()),\n        \'macro_precision\': float(precision.mean()), \'macro_recall\': float(recall.mean()),\n        \'count\': int(truth.size), \'confusion_matrix_true_rows_predicted_columns\': matrix.tolist(),\n        \'per_class\': [{\'class_id\': index, \'material\': name, \'support\': int(support[index]),\n                       \'precision\': float(precision[index]), \'recall\': float(recall[index]),\n                       \'f1\': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],\n    }\n\n\ndef retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):\n    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""\n    import numpy as np\n    gallery = np.asarray(gallery_features, dtype=np.float32)\n    queries = np.asarray(query_features, dtype=np.float32)\n    gallery_y = np.asarray(gallery_labels)\n    query_y = np.asarray(query_labels)\n    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]\n                    and len(gallery) >= 5 and len(queries) > 0, \'Invalid gallery/query feature shape.\')\n    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)\n                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), \'Misaligned or nonfinite retrieval inputs.\')\n    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]\n    top1 = gallery_y[rankings[:, 0]]\n    top5 = gallery_y[rankings[:, :5]]\n    return ({\'top1_accuracy\': float(np.mean(top1 == query_y)),\n             \'hit_at_5\': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),\n             \'precision_at_5\': float(np.mean(top5 == query_y[:, None]))}, rankings)\n\n\nclass OperatorAssistModels:\n    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.\n\n    ``assets_dir`` may be None while building a new gallery. Then call\n    ``load_gallery`` after writing these files:\n\n    - gallery_index.npz: float32 embeddings [160,512], float32 histograms\n      [160,24], int64 labels [160].\n    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,\n      each with class_id and crop_file="gallery/g_000.png" etc.\n    - the corresponding 160 PNG reference crops.\n\n    local_only=True requires the ResNet checkpoint and the explicit immutable\n    CLIP revision to be present already; it performs no network request.\n    """\n\n    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):\n        from pathlib import Path\n        from importlib.metadata import version\n        import re\n        import torch\n        import torch.nn as nn\n        from torchvision.models import resnet18, ResNet18_Weights\n        from transformers import CLIPModel, CLIPProcessor\n        from huggingface_hub import hf_hub_download, model_info\n\n        self.torch = torch\n        self.assets_dir = None\n        self.gallery_embeddings = None\n        self.gallery_histograms = None\n        self.gallery_labels = None\n        self.gallery_meta = None\n        self.local_only = bool(local_only)\n        torch.set_num_threads(1)\n        try:\n            torch.set_num_interop_threads(1)\n        except RuntimeError:\n            _assist_require(torch.get_num_interop_threads() == 1,\n                            \'Torch inter-op threads are already initialized differently; use a fresh process.\')\n\n        weights = ResNet18_Weights.IMAGENET1K_V1\n        self.resnet_transform = weights.transforms()\n        resnet_file = Path(torch.hub.get_dir()) / \'checkpoints/resnet18-f37072fd.pth\'\n        if self.local_only:\n            _assist_require(resnet_file.is_file(), \'Cached ResNet18 checkpoint is required for local-only serving.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'Cached ResNet18 checkpoint checksum differs.\')\n            encoder = resnet18(weights=None)\n            state = torch.load(resnet_file, map_location=\'cpu\', weights_only=True)\n            encoder.load_state_dict(state, strict=True)\n            del state\n        else:\n            encoder = resnet18(weights=weights, progress=False)\n            _assist_require(resnet_file.is_file(), \'Downloaded ResNet18 checkpoint was not found at its standard cache path.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'ResNet18 checkpoint checksum differs from its official filename prefix.\')\n        encoder.fc = nn.Identity()\n        self.encoder = encoder.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.encoder.parameters():\n            parameter.requires_grad_(False)\n\n        hub_info = None\n        if clip_revision is None:\n            _assist_require(not self.local_only, \'An explicit cached CLIP revision is required in local-only mode.\')\n            hub_info = model_info(CLIP_MODEL_ID, revision=\'main\', files_metadata=True, token=False)\n            clip_revision = hub_info.sha\n        _assist_require(isinstance(clip_revision, str)\n                        and re.fullmatch(r\'[0-9a-f]{40}|[0-9a-f]{64}\', clip_revision) is not None,\n                        \'CLIP revision must be an immutable Hub commit hash.\')\n        self.clip_revision = clip_revision\n        common = {\'revision\': self.clip_revision, \'local_files_only\': self.local_only, \'token\': False}\n        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,\n                                             trust_remote_code=False, **common)\n        self.clip = self.clip.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.clip.parameters():\n            parameter.requires_grad_(False)\n        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,\n                                                       trust_remote_code=False, **common)\n        # The loader above explicitly selects the bin checkpoint. This cache-only\n        # lookup records its actual bytes without triggering a second download.\n        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename=\'pytorch_model.bin\',\n                                        revision=self.clip_revision, local_files_only=True, token=False))\n        clip_sha = _assist_sha256(clip_file)\n        expected_clip_sha = None\n        if hub_info is not None:\n            for sibling in hub_info.siblings or []:\n                if sibling.rfilename == \'pytorch_model.bin\':\n                    lfs = getattr(sibling, \'lfs\', None)\n                    expected_clip_sha = (lfs.get(\'sha256\') if isinstance(lfs, dict)\n                                         else getattr(lfs, \'sha256\', None))\n                    break\n        if expected_clip_sha is not None:\n            _assist_require(clip_sha == expected_clip_sha, \'CLIP binary differs from the Hub LFS SHA256.\')\n        config_commit = getattr(self.clip.config, \'_commit_hash\', None)\n        if config_commit is not None:\n            _assist_require(config_commit == self.clip_revision, \'Loaded CLIP configuration revision differs.\')\n        self._provenance = {\n            \'resnet_weights\': \'ResNet18_Weights.IMAGENET1K_V1\',\n            \'resnet_weights_url\': weights.url, \'resnet_checkpoint_path\': str(resnet_file),\n            \'resnet_checkpoint_sha256\': resnet_sha, \'resnet_transform\': repr(self.resnet_transform),\n            \'clip_model_id\': CLIP_MODEL_ID, \'clip_revision\': self.clip_revision,\n            \'clip_config_commit_hash\': config_commit, \'clip_checkpoint_path\': str(clip_file),\n            \'clip_checkpoint_sha256\': clip_sha, \'clip_hub_lfs_sha256\': expected_clip_sha,\n            \'clip_checkpoint_url\': \'https://huggingface.co/\' + CLIP_MODEL_ID + \'/resolve/\'\n                                   + self.clip_revision + \'/pytorch_model.bin\',\n            \'clip_processor_class\': type(self.processor).__name__,\n            \'clip_image_processor_class\': type(self.processor.image_processor).__name__,\n            \'clip_image_processor_config\': self.processor.image_processor.to_dict(),\n            \'clip_attention_implementation\': getattr(self.clip.config, \'_attn_implementation\', None),\n            \'versions\': {name: version(name) for name in [\'torch\', \'torchvision\', \'transformers\',\n                                                         \'huggingface-hub\', \'numpy\', \'Pillow\']},\n        }\n        self._assert_cpu_frozen()\n        if assets_dir is not None:\n            self.load_gallery(assets_dir)\n\n    def _assert_cpu_frozen(self):\n        for name, model in [(\'resnet\', self.encoder), (\'clip\', self.clip)]:\n            _assist_require(model.training is False, name + \' must be in eval mode.\')\n            _assist_require(all(parameter.device.type == \'cpu\' and not parameter.requires_grad\n                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)\n                                for parameter in model.parameters()), name + \' must be frozen CPU FP32.\')\n\n    def embed_resnet(self, crops, batch=16):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid ResNet crop batch.\')\n        self._assert_cpu_frozen()\n        vectors = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                tensor = self.torch.stack([self.resnet_transform(crop.convert(\'RGB\'))\n                                          for crop in crops[start:start + batch]])\n                output = self.encoder(tensor.to(device=\'cpu\', dtype=self.torch.float32))\n                vectors.append(output.detach().cpu().numpy())\n        array = np.concatenate(vectors).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 512), \'Unexpected ResNet feature dimensions.\')\n        return unit(array)\n\n    def score_clip(self, crops, batch=8):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid CLIP crop batch.\')\n        self._assert_cpu_frozen()\n        logits = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                inputs = self.processor(text=list(PLAIN_PROMPTS),\n                                        images=[crop.convert(\'RGB\') for crop in crops[start:start + batch]],\n                                        return_tensors=\'pt\', padding=True)\n                inputs = {key: value.to(device=\'cpu\', dtype=self.torch.float32)\n                          if key == \'pixel_values\' else value.to(\'cpu\') for key, value in inputs.items()}\n                output = self.clip(**inputs)\n                logits.append(output.logits_per_image.detach().cpu().numpy())\n        array = np.concatenate(logits).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), \'Unexpected or nonfinite CLIP logits.\')\n        return array\n\n    def rgb_hist(self, crops):\n        return rgb_hist(crops)\n\n    def load_gallery(self, assets_dir):\n        from pathlib import Path, PurePosixPath\n        import json\n        import numpy as np\n        directory = Path(assets_dir).resolve()\n        with np.load(directory / \'gallery_index.npz\', allow_pickle=False) as bank:\n            _assist_require(set(bank.files) == {\'embeddings\', \'histograms\', \'labels\'}, \'Unexpected assistance gallery keys.\')\n            embeddings = bank[\'embeddings\'].copy()\n            histograms = bank[\'histograms\'].copy()\n            labels = bank[\'labels\'].copy()\n        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32\n                        and histograms.shape == (160, 24) and histograms.dtype == np.float32\n                        and labels.shape == (160,) and labels.dtype == np.int64,\n                        \'Assistance gallery array shapes/dtypes differ.\')\n        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()\n                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),\n                        \'Assistance gallery features/classes differ.\')\n        for name, values in [(\'embedding\', embeddings), (\'histogram\', histograms)]:\n            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),\n                            \'Unnormalized gallery \' + name + \' features.\')\n        metadata = json.loads((directory / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n        _assist_require(isinstance(metadata, list) and len(metadata) == 160, \'Expected 160 gallery metadata records.\')\n        for index, item in enumerate(metadata):\n            expected_file = \'gallery/g_\' + str(index).zfill(3) + \'.png\'\n            _assist_require(isinstance(item, dict) and item.get(\'class_id\') == int(labels[index])\n                            and item.get(\'crop_file\') == expected_file, \'Gallery metadata/array alignment differs.\')\n            crop = PurePosixPath(item[\'crop_file\'])\n            path = (directory / crop).resolve()\n            _assist_require(not crop.is_absolute() and \'..\' not in crop.parts\n                            and path.is_relative_to(directory) and path.is_file(), \'Missing or unsafe reference crop.\')\n        self.assets_dir = directory\n        self.gallery_embeddings = embeddings\n        self.gallery_histograms = histograms\n        self.gallery_labels = labels\n        self.gallery_meta = metadata\n        return self\n\n    def query_crop(self, crop):\n        import numpy as np\n        _assist_require(self.gallery_embeddings is not None, \'Load the assistance gallery first.\')\n        feature = self.embed_resnet([crop], batch=1)\n        similarities = (feature @ self.gallery_embeddings.T)[0]\n        ranking = similarities.argsort()[::-1]\n        references = []\n        for raw_index in ranking[:5]:\n            index = int(raw_index)\n            class_id = int(self.gallery_labels[index])\n            references.append({\'gallery_index\': index, \'class_id\': class_id,\n                               \'material\': CLASS_NAMES[class_id], \'similarity\': float(similarities[index]),\n                               \'crop_file\': self.gallery_meta[index][\'crop_file\']})\n        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)\n        relative = np.exp(clip_logits - clip_logits.max())\n        relative /= relative.sum()\n        clip_class = int(clip_logits.argmax())\n        histogram = rgb_hist([crop])\n        hist_similarities = (histogram @ self.gallery_histograms.T)[0]\n        hist_index = int(hist_similarities.argmax())\n        hist_class = int(self.gallery_labels[hist_index])\n        return {\n            \'visual_search\': {\'predicted_class_id\': references[0][\'class_id\'],\n                              \'predicted_material\': references[0][\'material\'], \'references\': references,\n                              \'score_type\': \'Cosine similarity to frozen Train reference crops; not calibrated confidence.\'},\n            \'clip\': {\'predicted_class_id\': clip_class, \'predicted_material\': CLASS_NAMES[clip_class],\n                     \'logits\': clip_logits.tolist(), \'relative_scores\': relative.tolist(),\n                     \'prompts\': list(PLAIN_PROMPTS),\n                     \'score_type\': \'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.\'},\n            \'histogram\': {\'predicted_class_id\': hist_class, \'predicted_material\': CLASS_NAMES[hist_class],\n                          \'reference_gallery_index\': hist_index, \'similarity\': float(hist_similarities[hist_index]),\n                          \'crop_file\': self.gallery_meta[hist_index][\'crop_file\'],\n                          \'score_type\': \'Nearest Train RGB histogram by cosine similarity; argmax tie policy.\'},\n        }\n\n    def runtime_metadata(self):\n        self._assert_cpu_frozen()\n        return {\n            **self._provenance, \'clip_revision\': self.clip_revision, \'device\': \'cpu\', \'precision\': \'float32\',\n            \'resnet_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()\n                                              if parameter.is_floating_point()}),\n            \'clip_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.clip.parameters()\n                                            if parameter.is_floating_point()}),\n            \'torch_num_threads\': int(self.torch.get_num_threads()),\n            \'torch_num_interop_threads\': int(self.torch.get_num_interop_threads()),\n            \'all_parameters_frozen\': True, \'plain_prompts\': list(PLAIN_PROMPTS),\n            \'gallery_loaded\': self.gallery_embeddings is not None,\n            \'gallery_size\': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,\n            \'model_training_or_optimization\': False,\n            \'decision_role\': \'Advisory references and material suggestion; original detector outputs are not overwritten.\',\n        }\n', 'extension_assist_service.py': '"""Local operator-assistance endpoint; the original service file is unchanged."""\nfrom pathlib import Path\nimport base64\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport math\nimport os\n\nfrom fastapi import File, HTTPException, UploadFile\nfrom fastapi.responses import FileResponse, HTMLResponse\nfrom PIL import Image\n\nfrom extension_assist_models import OperatorAssistModels\n\nHERE = Path(__file__).resolve().parent\nMANIFEST = json.loads((HERE / \'assist_manifest.json\').read_text(encoding=\'utf-8\'))\nASSETS = HERE / \'assets\'\nfor relative, expected in MANIFEST[\'assets_sha256\'].items():\n    asset = (ASSETS / relative).resolve()\n    if not asset.is_relative_to(ASSETS.resolve()) or hashlib.sha256(asset.read_bytes()).hexdigest() != expected:\n        raise RuntimeError(\'Assistance asset checksum differs: \' + relative)\nORIGINAL = Path(os.environ.get(\'PROJECT9_BASE_SERVICE\', MANIFEST[\'original_service_path\']))\nif hashlib.sha256(ORIGINAL.read_bytes()).hexdigest() != MANIFEST[\'original_service_sha256\']:\n    raise RuntimeError(\'Original HTTP service source hash differs.\')\nif hashlib.sha256((ORIGINAL.parent / \'release_manifest.json\').read_bytes()).hexdigest() != MANIFEST[\'original_release_manifest_sha256\']:\n    raise RuntimeError(\'Original HTTP release manifest checksum differs.\')\nspec = importlib.util.spec_from_file_location(\'project9_original_service_for_assist\', ORIGINAL)\nbase = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(base)\nif base.CONFIG[\'weights\'] != MANIFEST[\'original_model_mapping\']:\n    raise RuntimeError(\'Original HTTP model mapping differs.\')\nif base.CONFIG[\'device\'] != \'cpu\' or base.CONFIG[\'class_names\'] != MANIFEST[\'class_names\']:\n    raise RuntimeError(\'Original HTTP configuration differs.\')\nif (base.CONFIG[\'confidence\'], base.CONFIG[\'nms_iou\'], base.CONFIG[\'imgsz\']) != (0.25, 0.5, 640):\n    raise RuntimeError(\'Original HTTP thresholds or resolution differ.\')\n\nengine = OperatorAssistModels(assets_dir=ASSETS, clip_revision=MANIFEST[\'clip_revision\'], local_only=True)\nfor key in [\'resnet_checkpoint_sha256\', \'clip_checkpoint_sha256\']:\n    if engine.runtime_metadata()[key] != MANIFEST[\'model_resources\'][key]:\n        raise RuntimeError(\'Assistance model resource checksum differs: \' + key)\napp = base.app\n\n\n@app.get(\'/assist-health\')\ndef assist_health():\n    return {\'status\': \'ok\', \'purpose\': \'operator assistance\', \'base_source_verified\': True,\n            \'gallery_items\': len(engine.gallery_meta), \'models\': engine.runtime_metadata(),\n            \'original_predict_route_retained\': True, \'automatic_material_override\': False}\n\n\n@app.get(\'/assist-gallery/{gallery_index}\')\ndef gallery_image(gallery_index: int):\n    if not 0 <= gallery_index < len(engine.gallery_meta):\n        raise HTTPException(404, \'Reference image not found\')\n    target = (ASSETS / engine.gallery_meta[gallery_index][\'crop_file\']).resolve()\n    if not target.is_relative_to(ASSETS.resolve()) or not target.is_file():\n        raise HTTPException(404, \'Reference image not found\')\n    return FileResponse(target, media_type=\'image/png\')\n\n\n@app.get(\'/operator\', response_class=HTMLResponse)\ndef operator_page():\n    return (HERE / \'operator_ui.html\').read_text(encoding=\'utf-8\')\n\n\n@app.post(\'/assist\')\nasync def assist(file: UploadFile = File(...)):\n    raw = await file.read(12 * 1024 * 1024 + 1)\n    if not raw:\n        raise HTTPException(400, \'Empty image\')\n    if len(raw) > 12 * 1024 * 1024:\n        raise HTTPException(413, \'Max upload 12 MiB\')\n    supplied = UploadFile(file=io.BytesIO(raw), filename=file.filename or \'image.PNG\')\n    try:\n        detected = await base.predict(file=supplied)\n    finally:\n        await supplied.close()\n    objects = detected[\'objects\']\n    selected_index = next((index for index, item in enumerate(objects) if item[\'requires_review\']),\n                          0 if objects else None)\n    runtime = {\n        \'detector_device\': str(base.det.predictor.model.device),\n        \'segmenter_device\': str(base.seg.predictor.model.device),\n        \'detector_fp16\': bool(base.det.predictor.model.fp16),\n        \'segmenter_fp16\': bool(base.seg.predictor.model.fp16),\n    }\n    output = {\'status\': \'no_detections\' if selected_index is None else \'assistance_ready\',\n              \'source_detection\': detected, \'selected_object_index\': selected_index,\n              \'selection_policy\': \'First object requiring mask-association review; otherwise first object.\',\n              \'base_runtime\': runtime, \'assistance_runtime\': engine.runtime_metadata(),\n              \'automatic_material_override\': False, \'ground_truth_accuracy_measured\': False}\n    if selected_index is None:\n        output[\'assistance\'] = None\n        return output\n    chosen = objects[selected_index]\n    with Image.open(io.BytesIO(raw)) as source:\n        source = source.convert(\'RGB\')\n        width, height = source.size\n        coordinates = chosen[\'bbox_xyxy\']\n        if len(coordinates) != 4 or not all(math.isfinite(float(value)) for value in coordinates):\n            raise HTTPException(500, \'Invalid detection coordinates\')\n        x1 = max(0, min(width - 1, math.floor(coordinates[0])))\n        y1 = max(0, min(height - 1, math.floor(coordinates[1])))\n        x2 = max(x1 + 1, min(width, math.ceil(coordinates[2])))\n        y2 = max(y1 + 1, min(height, math.ceil(coordinates[3])))\n        crop = source.crop((x1, y1, x2, y2))\n    result = engine.query_crop(crop)\n    thumbnail = crop.copy()\n    thumbnail.thumbnail((640, 480))\n    buffer = io.BytesIO()\n    thumbnail.save(buffer, format=\'PNG\')\n    output.update({\'selected_object\': chosen, \'crop_box_xyxy\': [x1, y1, x2, y2],\n                   \'crop_preview_png_base64\': base64.b64encode(buffer.getvalue()).decode(\'ascii\'),\n                   \'assistance\': result,\n                   \'interpretation\': \'Related labelled training crops and a second material suggestion for human review; similarity scores are not calibrated correctness probabilities.\'})\n    return output\n', 'extension_assist_start.py': '"""Run the saved operator assistance service using the original Drive models."""\nfrom pathlib import Path\nimport argparse\nimport gc\nimport json\nimport os\nimport sys\n\nfrom extension_assist_models import OperatorAssistModels\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--port\', type=int, default=8008)\n    args = parser.parse_args()\n    here = Path(__file__).resolve().parent\n    config = json.loads((here / \'assist_manifest.json\').read_text(encoding=\'utf-8\'))\n    # The immutable revision is already fixed. Missing official cache files may\n    # be downloaded, but their full hashes must equal the audited resource set.\n    engine = OperatorAssistModels(clip_revision=config[\'clip_revision\'])\n    meta = engine.runtime_metadata()\n    for key in [\'resnet_checkpoint_sha256\', \'clip_checkpoint_sha256\']:\n        if meta[key] != config[\'model_resources\'][key]:\n            raise RuntimeError(\'Downloaded/cached model resource checksum differs: \' + key)\n    del engine\n    gc.collect()\n    os.environ[\'WANDB_MODE\'] = \'disabled\'\n    os.chdir(here)\n    os.execv(sys.executable, [sys.executable, \'-m\', \'uvicorn\',\n                             \'extension_assist_service:app\', \'--host\', \'127.0.0.1\',\n                             \'--port\', str(args.port), \'--workers\', \'1\'])\n\n\nif __name__ == \'__main__\':\n    main()\n', 'extension_assist_evidence_render.py': '"""Pure-Pillow cards for recorded extension results and operator responses.\n\nNo model execution, package installation, network access or automatic writes.\nFunctions draw supplied data only and keep ground-truth crop experiments\nseparate from the detector-crop operator examples.\n"""\n\n_EXT_CLASS_NAMES = (\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\')\n_EXT_COLORS = ((201, 134, 32), (36, 122, 179), (66, 137, 58), (153, 71, 156))\n\n\ndef _ext_require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef _ext_font(size, bold=False):\n    from PIL import ImageFont\n    choices = ([\'DejaVuSans-Bold.ttf\', \'/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf\',\n                r\'C:\\Windows\\Fonts\\arialbd.ttf\'] if bold else\n               [\'DejaVuSans.ttf\', \'/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\',\n                r\'C:\\Windows\\Fonts\\arial.ttf\'])\n    for path in choices:\n        try:\n            return ImageFont.truetype(path, size=size)\n        except OSError:\n            pass\n    return ImageFont.load_default(size=size)\n\n\ndef _ext_lines(draw, text, font, width):\n    result = []\n    for paragraph in str(text).split(\'\\n\'):\n        current = \'\'\n        for word in paragraph.split(\' \'):\n            trial = (current + \' \' + word).strip()\n            if draw.textbbox((0, 0), trial, font=font)[2] <= width:\n                current = trial\n                continue\n            if current:\n                result.append(current)\n                current = \'\'\n            # Long filenames are split instead of clipped or silently omitted.\n            chunk = \'\'\n            for char in word:\n                if chunk and draw.textbbox((0, 0), chunk + char, font=font)[2] > width:\n                    result.append(chunk)\n                    chunk = char\n                else:\n                    chunk += char\n            current = chunk\n        result.append(current)\n    return result\n\n\ndef _ext_text(draw, text, box, size=20, bold=False, color=(32, 42, 53), min_size=12):\n    x, y, width, height = box\n    while size >= min_size:\n        font = _ext_font(size, bold)\n        lines = _ext_lines(draw, text, font, width)\n        spacing = max(2, int(size * .22))\n        line_height = size + spacing\n        if len(lines) * line_height <= height:\n            for index, line in enumerate(lines):\n                bb = draw.textbbox((0, 0), line, font=font)\n                draw.text((x - bb[0], y + index * line_height - bb[1]), line, font=font, fill=color)\n            return\n        size -= 1\n    raise ValueError(\'Evidence text does not fit its allocated card area.\')\n\n\ndef _ext_safe_crop(assets_dir, record):\n    from pathlib import Path, PurePosixPath\n    root = Path(assets_dir).resolve()\n    rel_value = record.get(\'crop_file\')\n    _ext_require(isinstance(rel_value, str) and \'\\\\\' not in rel_value, \'Invalid crop filename.\')\n    relative = PurePosixPath(rel_value)\n    _ext_require(not relative.is_absolute() and \'..\' not in relative.parts, \'Unsafe crop path.\')\n    path = (root / Path(*relative.parts)).resolve()\n    _ext_require(path.is_relative_to(root) and path.is_file(), \'Reference crop is absent or outside assets.\')\n    return path\n\n\ndef _ext_validate_records(assets_dir, records, expected):\n    _ext_require(isinstance(records, (list, tuple)) and len(records) == expected, \'Unexpected crop cohort size.\')\n    for item in records:\n        _ext_require(isinstance(item, dict) and isinstance(item.get(\'class_id\'), int)\n                     and not isinstance(item[\'class_id\'], bool) and 0 <= item[\'class_id\'] < 4,\n                     \'Invalid crop material class.\')\n        _ext_safe_crop(assets_dir, item)\n\n\ndef _ext_thumb(canvas, source, box):\n    from PIL import Image, ImageDraw\n    x, y, width, height = (int(value) for value in box)\n    ImageDraw.Draw(canvas).rectangle((x, y, x + width - 1, y + height - 1), fill=(235, 239, 244), outline=(206, 214, 224))\n    if isinstance(source, Image.Image):\n        image = source.convert(\'RGB\').copy()\n    else:\n        with Image.open(source) as opened:\n            image = opened.convert(\'RGB\')\n    image.thumbnail((width - 8, height - 8), Image.Resampling.LANCZOS)\n    canvas.paste(image, (x + (width - image.width) // 2, y + (height - image.height) // 2))\n\n\ndef _ext_new_output(output_dir):\n    from pathlib import Path\n    path = Path(output_dir).resolve()\n    path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef _ext_save_card(image, path):\n    with path.open(\'xb\') as stream:\n        image.save(stream, format=\'JPEG\', quality=94, subsampling=0)\n\n\ndef _ext_save_json(path, value):\n    import json\n    with path.open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n        stream.write(\'\\n\')\n\n\ndef _ext_html(path, title, note, cards):\n    import base64\n    import html\n    parts = [\'<!doctype html><html lang="en"><meta charset="utf-8">\',\n             \'<meta name="viewport" content="width=device-width, initial-scale=1">\',\n             \'<title>\' + html.escape(title) + \'</title>\',\n             \'<style>body{font:18px Arial,sans-serif;margin:24px;background:#eef2f6;color:#202a35}\'\n             \'main{max-width:1500px;margin:auto}img{width:100%;height:auto;display:block}\'\n             \'figure{margin:24px 0;padding:12px;background:white;border:1px solid #ccd5df}\'\n             \'figcaption{margin:10px 4px;line-height:1.4}.note{line-height:1.5}</style><main>\',\n             \'<h1>\' + html.escape(title) + \'</h1><p class="note">\' + html.escape(note) + \'</p>\']\n    for card in cards:\n        encoded = base64.b64encode(card[\'card_path\'].read_bytes()).decode(\'ascii\')\n        parts.extend([\'<figure><img alt="\' + html.escape(card[\'caption\'], quote=True)\n                      + \'" src="data:image/jpeg;base64,\' + encoded + \'">\',\n                      \'<figcaption>\' + html.escape(card[\'caption\']) + \'</figcaption></figure>\'])\n    if not cards:\n        parts.append(\'<p>No example card was produced because the stated error condition was absent.</p>\')\n    parts.append(\'</main></html>\')\n    with path.open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n        stream.write(\'\\n\'.join(parts) + \'\\n\')\n\n\ndef _ext_list(value):\n    return value.tolist() if hasattr(value, \'tolist\') else list(value)\n\n\ndef _ext_class(value):\n    _ext_require(isinstance(value, int) and not isinstance(value, bool) and 0 <= value < 4, \'Invalid prediction class.\')\n    return value\n\n\ndef _ext_relative(logits):\n    import math\n    values = [float(value) for value in logits]\n    _ext_require(len(values) == 4 and all(math.isfinite(value) for value in values), \'Invalid CLIP logits.\')\n    scaled = [math.exp(value - max(values)) for value in values]\n    return [value / sum(scaled) for value in scaled]\n\n\ndef render_extension_failures(assets_dir, gallery_records, query_records, rankings, clip_logits, hist_predictions, output_dir):\n    """Draw up to three observed errors per method from 40 annotated Val crops.\n\n    rankings is [40,160] in descending ResNet similarity order; clip_logits is\n    [40,4] for the plain prompts; hist_predictions contains 40 class IDs.\n    Returns JSON-serializable metadata plus a self-contained static HTML view.\n    """\n    from PIL import Image, ImageDraw\n    _ext_validate_records(assets_dir, gallery_records, 160)\n    _ext_validate_records(assets_dir, query_records, 40)\n    ranks = [_ext_list(row) for row in _ext_list(rankings)]\n    logits = [_ext_list(row) for row in _ext_list(clip_logits)]\n    hist = _ext_list(hist_predictions)\n    _ext_require(len(ranks) == len(logits) == len(hist) == 40, \'Unexpected result-array row count.\')\n    for row in ranks:\n        _ext_require(len(row) == 160 and all(isinstance(value, int) and not isinstance(value, bool) for value in row)\n                     and set(row) == set(range(160)), \'Rankings must be permutations of all 160 Train items.\')\n    hist = [_ext_class(value) for value in hist]\n    relative = [_ext_relative(row) for row in logits]\n    clip_pred = [max(range(4), key=lambda class_id: float(row[class_id])) for row in logits]\n    truths = [item[\'class_id\'] for item in query_records]\n    resnet_pred = [gallery_records[row[0]][\'class_id\'] for row in ranks]\n    resnet_wrong = [index for index in range(40) if resnet_pred[index] != truths[index]]\n    clip_wrong = [index for index in range(40) if clip_pred[index] != truths[index]]\n    output = _ext_new_output(output_dir)\n    cards = []\n    html_cards = []\n\n    def query_pane(canvas, draw, query_index, box):\n        record = query_records[query_index]\n        _ext_thumb(canvas, _ext_safe_crop(assets_dir, record), box)\n        _ext_text(draw, \'Annotated Validation query\', (box[0], box[1] - 35, box[2], 28), size=20, bold=True)\n        _ext_text(draw, \'Ground truth: \' + _EXT_CLASS_NAMES[truths[query_index]].replace(\'_\', \' \'),\n                  (box[0], box[1] + box[3] + 12, box[2], 60), size=20, bold=True)\n        _ext_text(draw, str(record.get(\'export_name\', record[\'crop_file\'])),\n                  (box[0], box[1] + box[3] + 78, box[2], 65), size=17)\n\n    for ordinal, index in enumerate(resnet_wrong[:3]):\n        canvas = Image.new(\'RGB\', (1500, 640), \'white\'); draw = ImageDraw.Draw(canvas)\n        _ext_text(draw, \'Visual Search: an incorrect top-1 material\', (24, 20, 1452, 40), 29, True)\n        _ext_text(draw, f\'Validation crop {index:02d} | Query label: {_EXT_CLASS_NAMES[truths[index]]} | \'\n                  f\'Top-1: {_EXT_CLASS_NAMES[resnet_pred[index]]} | RGB baseline: {_EXT_CLASS_NAMES[hist[index]]}\',\n                  (24, 68, 1452, 52), 20)\n        query_pane(canvas, draw, index, (24, 160, 252, 240))\n        references = []\n        for rank, gallery_index in enumerate(ranks[index][:5], 1):\n            record = gallery_records[gallery_index]; cls = record[\'class_id\']; x = 310 + (rank - 1) * 232\n            same = cls == truths[index]\n            _ext_text(draw, \'Train reference #\' + str(rank), (x, 126, 218, 28), 19, True)\n            _ext_thumb(canvas, _ext_safe_crop(assets_dir, record), (x, 160, 218, 240))\n            _ext_text(draw, _EXT_CLASS_NAMES[cls].replace(\'_\', \' \'), (x, 412, 218, 35), 20, True,\n                      _EXT_COLORS[cls])\n            _ext_text(draw, \'Same query material\' if same else \'Different query material\',\n                      (x, 450, 218, 48), 17)\n            _ext_text(draw, str(record.get(\'export_name\', record[\'crop_file\'])), (x, 500, 218, 64), 15)\n            references.append({\'rank\': rank, \'gallery_index\': gallery_index, \'class_id\': cls,\n                               \'crop_file\': record[\'crop_file\'], \'same_query_material\': same})\n        _ext_text(draw, \'GT-crop experiment: localization errors are excluded. The first observed errors are shown; \'\n                  \'this is not a representative failure rate or an explanation of the cause.\', (24, 582, 1452, 45), 18)\n        path = output / f\'failure_resnet_{ordinal:02d}.jpg\'; _ext_save_card(canvas, path)\n        caption = f\'Visual Search: query {index}, annotated {_EXT_CLASS_NAMES[truths[index]]}, top-1 {_EXT_CLASS_NAMES[resnet_pred[index]]}.\'\n        card = {\'kind\': \'visual_search_top1_error\', \'query_index\': index, \'query_crop_file\': query_records[index][\'crop_file\'],\n                \'ground_truth_class_id\': truths[index], \'prediction_class_id\': resnet_pred[index],\n                \'histogram_prediction_class_id\': hist[index], \'references\': references,\n                \'card_file\': path.name, \'card_path\': str(path), \'caption\': caption}\n        cards.append(card); html_cards.append({\'card_path\': path, \'caption\': caption})\n\n    for ordinal, index in enumerate(clip_wrong[:3]):\n        canvas = Image.new(\'RGB\', (1500, 640), \'white\'); draw = ImageDraw.Draw(canvas)\n        _ext_text(draw, \'CLIP plain prompts: an incorrect material suggestion\', (24, 20, 1452, 40), 29, True)\n        _ext_text(draw, f\'Validation crop {index:02d} | Query label: {_EXT_CLASS_NAMES[truths[index]]} | \'\n                  f\'CLIP top-1: {_EXT_CLASS_NAMES[clip_pred[index]]} | RGB baseline: {_EXT_CLASS_NAMES[hist[index]]}\',\n                  (24, 68, 1452, 52), 20)\n        query_pane(canvas, draw, index, (24, 160, 300, 260))\n        _ext_text(draw, \'Four-way relative text-image scores (uncalibrated)\', (390, 125, 1080, 35), 23, True)\n        for cls in range(4):\n            y = 183 + cls * 91\n            flags = (\' | ground truth\' if cls == truths[index] else \'\') + (\' | CLIP top-1\' if cls == clip_pred[index] else \'\')\n            _ext_text(draw, _EXT_CLASS_NAMES[cls].replace(\'_\', \' \') + flags, (390, y, 1075, 32), 20, True)\n            draw.rectangle((390, y + 38, 1190, y + 58), fill=(231, 236, 242))\n            draw.rectangle((390, y + 38, 390 + int(800 * relative[index][cls]), y + 58), fill=_EXT_COLORS[cls])\n            _ext_text(draw, f"score {relative[index][cls]:.4f} | logit {float(logits[index][cls]):.3f}",\n                      (1210, y + 30, 260, 48), 17)\n        _ext_text(draw, \'Plain prompts: "a photo of [material]". These four-way scores are not calibrated probabilities \'\n                  \'or unknown-category detection. GT crops exclude detector errors; no error cause is inferred.\',\n                  (24, 582, 1452, 45), 18)\n        path = output / f\'failure_clip_{ordinal:02d}.jpg\'; _ext_save_card(canvas, path)\n        caption = f\'CLIP: query {index}, annotated {_EXT_CLASS_NAMES[truths[index]]}, predicted {_EXT_CLASS_NAMES[clip_pred[index]]}.\'\n        card = {\'kind\': \'clip_plain_top1_error\', \'query_index\': index, \'query_crop_file\': query_records[index][\'crop_file\'],\n                \'ground_truth_class_id\': truths[index], \'prediction_class_id\': clip_pred[index],\n                \'histogram_prediction_class_id\': hist[index], \'logits\': [float(value) for value in logits[index]],\n                \'relative_scores\': relative[index], \'card_file\': path.name, \'card_path\': str(path), \'caption\': caption}\n        cards.append(card); html_cards.append({\'card_path\': path, \'caption\': caption})\n    metadata = {\'scope\': \'First up to three observed top-1 errors per method in the supplied 40 annotated Validation crops.\',\n                \'query_count\': 40, \'resnet_top1_error_count\': len(resnet_wrong), \'clip_plain_error_count\': len(clip_wrong),\n                \'resnet_error_examples_absent\': not resnet_wrong, \'clip_error_examples_absent\': not clip_wrong,\n                \'selection\': \'Ascending query index; no example is fabricated when no true/predicted class disagreement exists.\',\n                \'cards\': cards, \'output_dir\': str(output), \'metadata_file\': str(output / \'extension_failure_cards.json\'),\n                \'html_file\': str(output / \'extension_failures.html\'),\n                \'limits\': \'Ground-truth box crops only; no full-system accuracy, causal diagnosis or Test result is inferred.\'}\n    _ext_save_json(output / \'extension_failure_cards.json\', metadata)\n    _ext_html(output / \'extension_failures.html\', \'Validation crop failure examples\', metadata[\'scope\'] + \' \' + metadata[\'limits\'], html_cards)\n    return metadata\n\n\ndef render_operator_cases(responses, gallery_records, assets_dir, output_dir):\n    """Draw proof cards from five actual /assist responses; no GT is assigned.\n\n    Entries may be actual response dictionaries, or envelopes with ``payload``\n    (``response`` is also accepted), optional export_name and request_id.\n    """\n    import base64\n    import io\n    import math\n    from PIL import Image, ImageDraw\n    _ext_validate_records(assets_dir, gallery_records, 160)\n    _ext_require(isinstance(responses, (list, tuple)) and len(responses) == 5, \'Five actual operator responses are required.\')\n    output = _ext_new_output(output_dir); cards = []; html_cards = []\n    for ordinal, entry in enumerate(responses):\n        _ext_require(isinstance(entry, dict), \'Operator response must be a dictionary.\')\n        if isinstance(entry.get(\'payload\'), dict):\n            payload = entry[\'payload\']\n        elif isinstance(entry.get(\'response\'), dict):\n            payload = entry[\'response\']\n        else:\n            payload = entry\n        source_name = str(entry.get(\'export_name\', payload.get(\'export_name\', \'Validation operator case \' + str(ordinal + 1))))\n        _ext_require(payload.get(\'automatic_material_override\') is False\n                     and payload.get(\'ground_truth_accuracy_measured\') is False,\n                     \'Operator result must preserve the advisory/no-GT role.\')\n        detection = payload.get(\'source_detection\')\n        _ext_require(isinstance(detection, dict) and isinstance(detection.get(\'objects\'), list), \'Missing original detector response.\')\n        objects = detection[\'objects\']; selected_index = payload.get(\'selected_object_index\')\n        status = payload.get(\'status\'); assistance = payload.get(\'assistance\')\n        canvas = Image.new(\'RGB\', (1500, 850), \'white\'); draw = ImageDraw.Draw(canvas)\n        _ext_text(draw, \'Operator assistance: \' + source_name, (24, 20, 1452, 55), 29, True)\n        _ext_text(draw, \'Advisory references and a second material suggestion. The detector result is retained; \'\n                  \'no ground-truth accuracy or human-review outcome is measured.\', (24, 83, 1452, 56), 20)\n        metadata = {\'case_index\': ordinal, \'export_name\': source_name, \'status\': status,\n                    \'request_id\': entry.get(\'request_id\'), \'automatic_material_override\': False,\n                    \'ground_truth_accuracy_measured\': False, \'original_detected_object_count\': len(objects)}\n        if selected_index is None:\n            _ext_require(status == \'no_detections\' and not objects and assistance is None, \'Inconsistent no-detection response.\')\n            _ext_text(draw, \'No detected object was available for assistance.\', (40, 220, 1400, 90), 32, True)\n            _ext_text(draw, \'No reference ranking or CLIP material is invented for this case.\', (40, 340, 1400, 70), 23)\n            metadata[\'assistance_absent\'] = True\n        else:\n            _ext_require(status == \'assistance_ready\' and isinstance(selected_index, int) and not isinstance(selected_index, bool)\n                         and 0 <= selected_index < len(objects) and isinstance(assistance, dict), \'Invalid selected-object response.\')\n            chosen = payload.get(\'selected_object\')\n            _ext_require(isinstance(chosen, dict) and chosen == objects[selected_index]\n                         and chosen.get(\'material\') in _EXT_CLASS_NAMES and isinstance(chosen.get(\'requires_review\'), bool),\n                         \'Selected object differs from the original detector object.\')\n            confidence = float(chosen[\'confidence\'])\n            _ext_require(math.isfinite(confidence) and 0 <= confidence <= 1, \'Invalid original detector confidence.\')\n            preview = payload.get(\'crop_preview_png_base64\')\n            _ext_require(isinstance(preview, str) and len(preview) <= 12 * 1024**2, \'Missing/oversized selected-crop preview.\')\n            preview_bytes = base64.b64decode(preview, validate=True)\n            with Image.open(io.BytesIO(preview_bytes)) as opened:\n                _ext_require(opened.format == \'PNG\' and 0 < opened.width <= 640 and 0 < opened.height <= 480,\n                             \'Unexpected selected-crop preview format/size.\')\n                query = opened.convert(\'RGB\')\n            visual = assistance.get(\'visual_search\'); clip = assistance.get(\'clip\'); histogram = assistance.get(\'histogram\')\n            _ext_require(all(isinstance(value, dict) for value in [visual, clip, histogram]), \'Missing assistance component.\')\n            references = visual.get(\'references\')\n            _ext_require(isinstance(references, list) and len(references) == 5, \'Expected five actual Train references.\')\n            reference_metadata = []\n            for rank, ref in enumerate(references, 1):\n                index = ref.get(\'gallery_index\')\n                _ext_require(isinstance(index, int) and not isinstance(index, bool) and 0 <= index < 160, \'Invalid reference index.\')\n                expected = gallery_records[index]; cls = expected[\'class_id\']\n                _ext_require(ref.get(\'class_id\') == cls and ref.get(\'material\') == _EXT_CLASS_NAMES[cls]\n                             and ref.get(\'crop_file\') == expected[\'crop_file\'] and math.isfinite(float(ref[\'similarity\'])),\n                             \'Reference index, label or crop does not match the recorded Train bank.\')\n                reference_metadata.append({\'rank\': rank, \'gallery_index\': index, \'class_id\': cls,\n                                           \'crop_file\': ref[\'crop_file\'], \'similarity\': float(ref[\'similarity\'])})\n            _ext_require(len({ref[\'gallery_index\'] for ref in reference_metadata}) == 5,\n                         \'The five references must identify distinct gallery entries.\')\n            visual_class = _ext_class(visual.get(\'predicted_class_id\'))\n            clip_class = _ext_class(clip.get(\'predicted_class_id\')); histogram_class = _ext_class(histogram.get(\'predicted_class_id\'))\n            _ext_require(visual_class == reference_metadata[0][\'class_id\']\n                         and visual.get(\'predicted_material\') == _EXT_CLASS_NAMES[visual_class]\n                         and clip.get(\'predicted_material\') == _EXT_CLASS_NAMES[clip_class]\n                         and histogram.get(\'predicted_material\') == _EXT_CLASS_NAMES[histogram_class],\n                         \'Assistance material identities disagree.\')\n            clip_logits = [float(value) for value in clip.get(\'logits\', [])]\n            relative = _ext_relative(clip_logits)\n            recorded_relative = [float(value) for value in clip.get(\'relative_scores\', [])]\n            _ext_require(len(recorded_relative) == 4 and all(math.isfinite(value) and 0 <= value <= 1 for value in recorded_relative)\n                         and all(abs(a - b) <= 1e-5 for a, b in zip(relative, recorded_relative))\n                         and max(range(4), key=lambda index: clip_logits[index]) == clip_class,\n                         \'CLIP relative scores/logits/prediction disagree.\')\n            hist_index = histogram.get(\'reference_gallery_index\')\n            _ext_require(isinstance(hist_index, int) and not isinstance(hist_index, bool) and 0 <= hist_index < 160\n                         and gallery_records[hist_index][\'class_id\'] == histogram_class\n                         and histogram.get(\'crop_file\') == gallery_records[hist_index][\'crop_file\']\n                         and math.isfinite(float(histogram[\'similarity\'])), \'Invalid histogram reference.\')\n            _ext_thumb(canvas, query, (24, 210, 342, 300))\n            _ext_text(draw, \'Selected detector crop\', (24, 160, 342, 35), 23, True)\n            _ext_text(draw, f"M4: {chosen[\'material\'].replace(\'_\', \' \')}\\nDetector score: {confidence:.4f}\\n"\n                      + (\'Mask association: REVIEW\' if chosen[\'requires_review\'] else \'Mask association: available\'),\n                      (24, 528, 342, 110), 20, True)\n            _ext_text(draw, \'Five labelled Train references\', (404, 160, 1070, 35), 23, True)\n            for rank, ref in enumerate(reference_metadata):\n                record = gallery_records[ref[\'gallery_index\']]; x = 404 + rank * 214\n                _ext_thumb(canvas, _ext_safe_crop(assets_dir, record), (x, 210, 198, 200))\n                _ext_text(draw, f"#{rank + 1} {_EXT_CLASS_NAMES[ref[\'class_id\']].replace(\'_\', \' \')}",\n                          (x, 424, 198, 52), 18, True, _EXT_COLORS[ref[\'class_id\']])\n                _ext_text(draw, f"Cosine {ref[\'similarity\']:.4f}", (x, 482, 198, 35), 16)\n            _ext_text(draw, \'Visual Search top-1: \' + _EXT_CLASS_NAMES[visual_class].replace(\'_\', \' \')\n                      + \' | CLIP suggestion: \' + _EXT_CLASS_NAMES[clip_class].replace(\'_\', \' \')\n                      + \' | RGB baseline: \' + _EXT_CLASS_NAMES[histogram_class].replace(\'_\', \' \'),\n                      (404, 546, 1070, 58), 21, True)\n            score_text = \'CLIP relative scores: \' + \' | \'.join(_EXT_CLASS_NAMES[c].replace(\'_\', \' \')\n                         + f\' {recorded_relative[c]:.4f}\' for c in range(4))\n            _ext_text(draw, score_text, (404, 622, 1070, 70), 19)\n            _ext_text(draw, \'Similarity and four-way scores are uncalibrated. These suggestions are not automatic \'\n                      \'material overrides or ground-truth error decisions.\', (404, 708, 1070, 60), 19)\n            metadata.update({\'assistance_absent\': False, \'selected_object_index\': selected_index,\n                             \'selected_object\': chosen, \'crop_box_xyxy\': payload.get(\'crop_box_xyxy\'),\n                             \'visual_search_class_id\': visual_class, \'clip_class_id\': clip_class,\n                             \'histogram_class_id\': histogram_class, \'references\': reference_metadata,\n                             \'clip_logits\': clip_logits, \'clip_relative_scores\': recorded_relative,\n                             \'selection_policy\': payload.get(\'selection_policy\')})\n        _ext_text(draw, \'Functional case record only: no tracking, physical pick success, overall system accuracy \'\n                  \'or completed human decision is inferred.\', (24, 798, 1452, 38), 18)\n        path = output / f\'operator_case_{ordinal:02d}.jpg\'; _ext_save_card(canvas, path)\n        caption = source_name + \': \' + (\'no detections; assistance absent.\' if selected_index is None\n                                       else \'actual detector crop, related Train references and advisory material suggestions.\')\n        metadata.update({\'card_file\': path.name, \'card_path\': str(path), \'caption\': caption})\n        cards.append(metadata); html_cards.append({\'card_path\': path, \'caption\': caption})\n    result = {\'scope\': \'Five supplied local operator-assistance responses on Validation images; advisory integration evidence only.\',\n              \'case_count\': 5, \'cases_with_assistance\': sum(not item[\'assistance_absent\'] for item in cards),\n              \'cases_without_detections\': sum(item[\'assistance_absent\'] for item in cards),\n              \'automatic_material_override\': False, \'ground_truth_accuracy_measured\': False,\n              \'cards\': cards, \'output_dir\': str(output), \'metadata_file\': str(output / \'operator_case_cards.json\'),\n              \'html_file\': str(output / \'operator_cases.html\')}\n    _ext_save_json(output / \'operator_case_cards.json\', result)\n    _ext_html(output / \'operator_cases.html\', \'Operator assistance case records\', result[\'scope\'], html_cards)\n    return result\n', 'operator_ui.html': '<!doctype html>\n<html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Waste material review</title>\n<style>\nbody{font:16px system-ui,sans-serif;background:#f1f5f8;color:#172735;margin:0}main{max-width:1100px;margin:32px auto;padding:24px}h1{margin:0 0 8px}p{line-height:1.5}.panel{background:white;border:1px solid #d5dfe7;border-radius:12px;padding:22px;margin:18px 0}button{background:#08735e;color:white;border:0;border-radius:7px;padding:12px 22px;font-weight:700;cursor:pointer}button:disabled{opacity:.6}input{padding:12px}.row{display:flex;gap:24px;flex-wrap:wrap}.crop{max-width:360px;max-height:280px;object-fit:contain}.refs{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:14px}.ref img{width:100%;height:150px;object-fit:contain;background:#eef2f4}.ref{border:1px solid #dce4e9;border-radius:8px;padding:10px}.muted{color:#526576}.badge{display:inline-block;background:#e4f4ef;color:#095d4d;border-radius:5px;padding:5px 10px}.error{color:#a02626}\n</style>\n<main><h1>Waste material review</h1><p class="muted">Inspect a detected item with labelled reference images and a second material suggestion.</p>\n<section class="panel"><label for="image">Conveyor image</label><input id="image" type="file" accept="image/*"><button id="go">Review an item</button><p id="status" role="status"></p></section>\n<section id="result" class="panel" hidden><div class="row"><img id="crop" class="crop" alt="Detected item crop"><div><h2>Detected item</h2><p id="material"></p><p id="review"></p><h3>Second material suggestion</h3><p id="clip" class="badge"></p><p class="muted">The suggestion supports operator review. It does not replace the original system label.</p></div></div><h2>Related training references</h2><div id="refs" class="refs"></div><p class="muted">Reference similarity and text-image similarity describe model agreement; they are not calibrated probabilities of a correct label.</p></section>\n<section class="panel"><strong>Scope</strong><p>Four known materials: cardboard, metal, rigid plastic and soft plastic. The original detection and mask output is retained. Physical grasp success and unknown-material recognition are not established.</p></section></main>\n<script>\nconst el=id=>document.getElementById(id);\nel(\'go\').onclick=async()=>{\n const file=el(\'image\').files[0];if(!file){el(\'status\').textContent=\'Choose an image first.\';return;}\n el(\'go\').disabled=true;el(\'result\').hidden=true;el(\'status\').className=\'\';el(\'status\').textContent=\'Reviewing the image…\';\n try{const body=new FormData();body.append(\'file\',file);const response=await fetch(\'/assist\',{method:\'POST\',body});if(!response.ok)throw new Error(\'The image could not be processed.\');const data=await response.json();\n if(data.status===\'no_detections\'){el(\'status\').textContent=\'No item was detected in this image.\';return;}\n el(\'crop\').src=\'data:image/png;base64,\'+data.crop_preview_png_base64;\n el(\'material\').textContent=\'System material: \'+data.selected_object.material.replaceAll(\'_\',\' \');\n el(\'review\').textContent=data.selected_object.requires_review?\'This item has no valid associated mask.\':\'An associated mask is available.\';\n el(\'clip\').textContent=data.assistance.clip.predicted_material.replaceAll(\'_\',\' \');\n el(\'refs\').replaceChildren();\n for(const ref of data.assistance.visual_search.references){const card=document.createElement(\'div\');card.className=\'ref\';const img=document.createElement(\'img\');img.alt=\'Labelled training reference\';img.src=\'/assist-gallery/\'+ref.gallery_index;const label=document.createElement(\'p\');label.textContent=ref.material.replaceAll(\'_\',\' \');const score=document.createElement(\'small\');score.textContent=\'Similarity \'+ref.similarity.toFixed(3);card.append(img,label,score);el(\'refs\').append(card);}\n el(\'result\').hidden=false;el(\'status\').textContent=\'Suggestions ready for operator review.\';\n }catch(error){el(\'status\').className=\'error\';el(\'status\').textContent=error.message||\'The image could not be processed.\';}finally{el(\'go\').disabled=false;}\n};\n</script></html>\n', 'run_extension_assist_colab.py': '"""One explicit Validation-only extension reproduction and HTTP workflow run."""\n\n\ndef run_extension_assist():\n    from datetime import datetime, timezone\n    from importlib.metadata import PackageNotFoundError, version\n    from pathlib import Path\n    import base64\n    import csv\n    import gc\n    import hashlib\n    import io\n    import json\n    import os\n    import shutil\n    import socket\n    import subprocess\n    import sys\n    import time\n    import zipfile\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def sha(path):\n        value = hashlib.sha256()\n        with Path(path).open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n                value.update(chunk)\n        return value.hexdigest()\n\n    def save_json(path, value):\n        with Path(path).open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)\n            stream.write(\'\\n\')\n\n    def installed(name):\n        try:\n            return version(name)\n        except PackageNotFoundError:\n            return None\n\n    os.environ[\'WANDB_MODE\'] = \'disabled\'\n    os.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n    os.environ[\'HF_HUB_DISABLE_TELEMETRY\'] = \'1\'\n    torch_before, vision_before = installed(\'torch\'), installed(\'torchvision\')\n    require(torch_before and vision_before, \'Colab must have matching preinstalled PyTorch and torchvision.\')\n    required = {\'ultralytics\': \'8.4.213\', \'transformers\': \'5.22.1\',\n                \'fastapi\': \'0.141.1\'}\n    packages = [name + \'==\' + target for name, target in required.items() if installed(name) != target]\n    packages += [name for name in [\'uvicorn\', \'python-multipart\', \'requests\'] if installed(name) is None]\n    if packages:\n        print(\'Preparing missing packages; retaining the current PyTorch and torchvision...\')\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\',\n                        \'torch==\' + torch_before, \'torchvision==\' + vision_before, *packages],\n                       check=True, timeout=360)\n    require(installed(\'torch\') == torch_before and installed(\'torchvision\') == vision_before,\n            \'Core runtime changed unexpectedly. Stop before evaluation.\')\n    for name, target in required.items():\n        require(installed(name) == target, \'Required runtime version differs: \' + name)\n\n    import numpy as np\n    import requests\n    from PIL import Image\n    from google.colab import files\n\n    print(\'=== PREPARING ORIGINAL EXTENSION COHORTS ===\')\n    preparation = prepare_extension_cohorts()\n    project = Path(preparation[\'project_root\'])\n    inputs = Path(preparation[\'work_dir\'])\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    output = project / \'evaluation\' / (\'EXTENSIONS_OPERATOR_VAL_\' + stamp)\n    output.mkdir(parents=True, exist_ok=False)\n    deploy = output / \'deployment\'\n    assets = deploy / \'assets\'\n    assets.mkdir(parents=True, exist_ok=False)\n    protected_relative = {\n        \'split_manifest\': \'data_protocol/ef6b35d2ba7f/split_manifest.csv\',\n        \'validation_inventory\': \'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json\',\n        \'gallery_index\': \'open_world/visual_search/gallery_index_train.npz\',\n        \'visual_search_result\': \'open_world/visual_search/comparison_VAL.json\',\n        \'clip_result\': \'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv\',\n        \'service_source\': \'deployment/http_service/service.py\',\n        \'service_manifest\': \'deployment/http_service/release_manifest.json\',\n        \'http_detector\': \'deployment/http_service/best_det.pt\',\n        \'http_segmenter\': \'deployment/http_service/best_seg.pt\',\n        \'source_detector\': \'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt\',\n        \'source_segmenter\': \'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt\',\n        \'test_protocol\': \'evaluation/FINAL_TEST_ONCE/frozen_protocol.json\',\n        \'test_results\': \'evaluation/FINAL_TEST_ONCE/results.json\',\n    }\n    protected = {name: project / relative for name, relative in protected_relative.items()}\n    before = {name: sha(path) for name, path in protected.items()}\n    require(before == preparation[\'provenance\'][\'protected_sha256_before\'],\n            \'A protected reference changed after cohort preparation.\')\n    save_json(output / \'cohort_provenance.json\', preparation)\n    for directory in [\'gallery\', \'queries\']:\n        shutil.copytree(inputs / directory, assets / directory)\n    gallery, queries = preparation[\'gallery\'], preparation[\'queries\']\n    save_json(assets / \'gallery_meta.json\', gallery)\n    save_json(assets / \'query_meta.json\', queries)\n    code_dir = output / \'code\'\n    code_dir.mkdir()\n    for name, source in _EXTENSION_SOURCES.items():\n        with (code_dir / name).open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            stream.write(source)\n    for name in [\'extension_assist_models.py\', \'extension_assist_service.py\',\n                 \'extension_assist_start.py\', \'operator_ui.html\']:\n        shutil.copyfile(code_dir / name, deploy / name)\n\n    plan = {\n        \'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'val\',\n        \'gallery\': {\'split\': \'train\', \'objects\': 160, \'per_class\': 40},\n        \'queries\': {\'split\': \'val\', \'objects\': 40, \'per_class\': 10},\n        \'cohort_provenance_sha256\': sha(output / \'cohort_provenance.json\'),\n        \'encoder\': \'Frozen ImageNet ResNet18 IMAGENET1K_V1; CPU FP32\',\n        \'clip_model\': \'openai/clip-vit-base-patch32\', \'prompts\': list(PLAIN_PROMPTS),\n        \'model_or_prompt_selection_during_this_run\': False,\n        \'historical_prompt_choice\': \'Plain prompts already selected using earlier Validation comparison; conveyor prompt result remains historical.\',\n        \'histogram_baseline\': \'PIL RGB resize64x64, eight bins per RGB channel, concatenate24, L2 normalize.\',\n        \'retrieval_ranking\': \'descending numpy argsort of cosine similarity, as in the recorded notebook\',\n        \'histogram_classification_tie_policy\': \'numpy argmax first index, as in the recorded notebook\',\n        \'gallery_reconstruction_check\': \'All160 name/class rows; then minimum paired old/new embedding cosine >=0.999 before query evaluation.\',\n        \'operator_images\': [item[\'export_name\'] for item in preparation[\'operator_images\']],\n        \'operator_crop_selection\': \'First requires_review object, otherwise first object; none if no detections.\',\n        \'crop_rule\': \'Original response rounded bbox coordinates, floor starts/ceil ends, clipped to image.\',\n        \'failure_selection\': \'First up to3 wrong top1 predictions per extension in fixed query order; explicitly report zero if absent.\',\n        \'metric_scope\': \'40 fixed ground-truth object crops; no full-image detection accuracy, physical grasp, unknown-class or whole-system accuracy claim.\',\n        \'http_scope\': \'Five actual Validation requests and HTML/reference route checks; functional proof, not a latency benchmark.\',\n        \'test_images_or_test_model_calls\': 0, \'optimizer_steps\': 0,\n        \'original_project_files_sha256\': before,\n    }\n    save_json(output / \'plan_before_model_loading.json\', plan)\n    print(\'Loading official frozen assistance models on CPU; missing weights download once (~650 MB total).\')\n    engine = OperatorAssistModels()\n    resources = engine.runtime_metadata()\n    protocol = {**plan, \'frozen_before_any_new_model_inference_at_utc\': datetime.now(timezone.utc).isoformat(),\n                \'model_resources\': resources,\n                \'source_files_sha256\': {name: sha(code_dir / name) for name in _EXTENSION_SOURCES}}\n    save_json(output / \'frozen_operator_VAL_protocol.json\', protocol)\n    protocol_sha = sha(output / \'frozen_operator_VAL_protocol.json\')\n\n    def load_crops(records):\n        result = []\n        for record in records:\n            path = assets / record[\'crop_file\']\n            require(sha(path) == record[\'crop_sha256\'], \'Crop changed: \' + record[\'crop_file\'])\n            with Image.open(path) as image:\n                result.append(image.convert(\'RGB\'))\n        return result\n\n    gallery_images, query_images = load_crops(gallery), load_crops(queries)\n    gy = np.asarray([item[\'class_id\'] for item in gallery], dtype=np.int64)\n    qy = np.asarray([item[\'class_id\'] for item in queries], dtype=np.int64)\n    print(\'Encoding160 Train references and comparing against the saved historical bank...\')\n    g = engine.embed_resnet(gallery_images)\n    with np.load(preparation[\'original_index_path\'], allow_pickle=False) as old:\n        old_g = old[\'embeddings\'].copy().astype(np.float32)\n    old_g = unit(old_g)\n    paired_cosine = np.sum(g * old_g, axis=1)\n    bank_check = {\n        \'count\': 160, \'source_basenames_and_classes_aligned\': True,\n        \'paired_cosines\': paired_cosine.tolist(), \'minimum_paired_cosine\': float(paired_cosine.min()),\n        \'maximum_paired_cosine\': float(paired_cosine.max()),\n        \'max_absolute_feature_difference\': float(np.abs(g - old_g).max()),\n        \'acceptance_minimum_cosine_fixed_before_inference\': 0.999,\n        \'historical_bank_sha256\': preparation[\'historical_bank_sha256\'],\n        \'passed\': bool(np.all(paired_cosine >= 0.999)),\n        \'interpretation\': \'Numerical compatibility check for reconstructed reference crops, not byte-identical model output proof.\',\n    }\n    save_json(output / \'historical_bank_reconstruction_check.json\', bank_check)\n    require(bank_check[\'passed\'], \'Reconstructed reference features differ from the historical bank. Stop and share this output; no query/HTTP result is claimed.\')\n    print(\'Historical reference feature check passed. Evaluating40 fixed Validation crops...\')\n    q = engine.embed_resnet(query_images)\n    gh, qh = rgb_hist(gallery_images), rgb_hist(query_images)\n    resnet_metrics, rankings = retrieval_metrics(g, q, gy, qy)\n    histogram_retrieval, histogram_rankings = retrieval_metrics(gh, qh, gy, qy)\n    hist_predictions = gy[(qh @ gh.T).argmax(axis=1)]\n    logits = engine.score_clip(query_images)\n    clip_predictions = logits.argmax(axis=1)\n    clip_metrics = classification_metrics(qy, clip_predictions)\n    hist_classification = classification_metrics(qy, hist_predictions)\n    np.savez_compressed(assets / \'gallery_index.npz\', embeddings=g, histograms=gh, labels=gy)\n    np.savez_compressed(output / \'validation_arrays.npz\', query_embeddings=q,\n                        query_histograms=qh, clip_logits=logits, query_labels=qy,\n                        resnet_rankings=rankings, histogram_rankings=histogram_rankings,\n                        histogram_class_predictions=hist_predictions)\n    historical = output / \'historical_references\'\n    historical.mkdir()\n    shutil.copyfile(protected[\'visual_search_result\'], historical / \'visual_search_comparison_VAL.json\')\n    shutil.copyfile(protected[\'clip_result\'], historical / \'CLIP_vs_histogram_VAL.csv\')\n    comparison = {\n        \'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'val\',\n        \'gallery_objects\': 160, \'query_objects\': 40, \'class_names\': list(CLASS_NAMES),\n        \'visual_search_resnet\': resnet_metrics, \'visual_search_histogram\': histogram_retrieval,\n        \'clip_plain\': clip_metrics, \'histogram_classification_baseline\': hist_classification,\n        \'historical_results_replaced\': False, \'protocol_sha256\': protocol_sha,\n        \'limits\': [\'Fixed small balanced ground-truth crop cohort, not a full image or system accuracy evaluation.\',\n                   \'Reconstructed evaluation in the recorded current CPU runtime, not the original historical inference event.\',\n                   \'Similarities and four-way softmax are not calibrated confidence or unknown-material detection.\'],\n    }\n    save_json(output / \'comparison_VAL.json\', comparison)\n    query_rows = []\n    for index, item in enumerate(queries):\n        top5 = [int(value) for value in rankings[index, :5]]\n        query_rows.append({\'query_index\': index, \'query_crop_file\': item[\'crop_file\'],\n                           \'source_export_name\': item[\'export_name\'], \'annotation_id\': item[\'annotation_id\'],\n                           \'ground_truth_class_id\': int(qy[index]),\n                           \'resnet_top1_class_id\': int(gy[top5[0]]),\n                           \'resnet_top5_gallery_indices\': top5,\n                           \'resnet_top5_cosine_similarities\': [float(q[index] @ g[value]) for value in top5],\n                           \'clip_logits\': logits[index].tolist(), \'clip_class_id\': int(clip_predictions[index]),\n                           \'histogram_class_id\': int(hist_predictions[index])})\n    save_json(output / \'per_query_VAL.json\', query_rows)\n    failure_metadata = render_extension_failures(assets, gallery, queries, rankings, logits,\n                                                  hist_predictions, output / \'failure_examples\')\n    save_json(output / \'failure_examples_index.json\', failure_metadata)\n    print(json.dumps(comparison, indent=2))\n\n    asset_hashes = {path.relative_to(assets).as_posix(): sha(path)\n                    for path in sorted(assets.rglob(\'*\')) if path.is_file()}\n    base_config = json.loads(protected[\'service_manifest\'].read_text(encoding=\'utf-8\'))\n    require((base_config[\'confidence\'], base_config[\'nms_iou\'], base_config[\'imgsz\'], base_config[\'device\'])\n            == (0.25, 0.5, 640, \'cpu\'), \'Frozen original HTTP settings differ.\')\n    assist_manifest = {\n        \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'original_service_path\': str(protected[\'service_source\']),\n        \'original_service_sha256\': before[\'service_source\'],\n        \'original_release_manifest_sha256\': before[\'service_manifest\'],\n        \'original_model_mapping\': base_config[\'weights\'], \'class_names\': base_config[\'class_names\'],\n        \'clip_revision\': resources[\'clip_revision\'], \'model_resources\': resources,\n        \'assets_sha256\': asset_hashes, \'protocol_sha256\': protocol_sha,\n        \'role\': \'Operator assistance only; no automatic replacement of material label or rejection threshold.\',\n    }\n    save_json(deploy / \'assist_manifest.json\', assist_manifest)\n    # Free parent weights before the independent server process loads them.\n    del engine, gallery_images, query_images\n    gc.collect()\n    with socket.socket() as listener:\n        listener.bind((\'127.0.0.1\', 0))\n        port = listener.getsockname()[1]\n    env = os.environ.copy()\n    env.update({\'WANDB_MODE\': \'disabled\', \'OMP_NUM_THREADS\': \'1\', \'MKL_NUM_THREADS\': \'1\',\n                \'TOKENIZERS_PARALLELISM\': \'false\', \'HF_HUB_DISABLE_TELEMETRY\': \'1\'})\n    logfile = (output / \'assist_server.log\').open(\'x\', encoding=\'utf-8\')\n    process = subprocess.Popen([sys.executable, \'-m\', \'uvicorn\', \'extension_assist_service:app\',\n                                \'--host\', \'127.0.0.1\', \'--port\', str(port), \'--workers\', \'1\'],\n                               cwd=deploy, env=env, stdout=logfile, stderr=subprocess.STDOUT)\n    session = requests.Session()\n    session.trust_env = False\n    url = \'http://127.0.0.1:\' + str(port)\n    cases = []\n    try:\n        deadline = time.monotonic() + 240\n        print(\'Starting the local operator assistance service...\')\n        while True:\n            require(process.poll() is None, \'Assistance server stopped; inspect the saved assist_server.log.\')\n            try:\n                health_response = session.get(url + \'/assist-health\', timeout=3)\n                if health_response.status_code == 200:\n                    break\n            except requests.RequestException:\n                pass\n            require(time.monotonic() < deadline, \'Assistance server startup timed out; inspect assist_server.log.\')\n            time.sleep(2)\n        health = health_response.json()\n        require(health[\'gallery_items\'] == 160 and health[\'original_predict_route_retained\'] is True,\n                \'Assistance health contract differs.\')\n        save_json(output / \'assist_health.json\', health)\n        response_dir = output / \'http_responses\'\n        response_dir.mkdir()\n        for ordinal, image in enumerate(preparation[\'operator_images\']):\n            raw = Path(image[\'path\']).read_bytes()\n            require(hashlib.sha256(raw).hexdigest() == image[\'image_sha256\'], \'Operator input image bytes changed.\')\n            started = time.perf_counter()\n            response = session.post(url + \'/assist\', files={\'file\': (image[\'export_name\'], raw, \'image/png\')}, timeout=240)\n            seconds = time.perf_counter() - started\n            require(response.status_code == 200, \'Operator request failed: \' + str(response.status_code) + \' \' + response.text[:500])\n            payload = response.json()\n            require(payload[\'automatic_material_override\'] is False and payload[\'ground_truth_accuracy_measured\'] is False,\n                    \'Unexpected automatic override or accuracy claim.\')\n            require(payload[\'base_runtime\'][\'detector_device\'] == \'cpu\'\n                    and payload[\'base_runtime\'][\'segmenter_device\'] == \'cpu\'\n                    and not payload[\'base_runtime\'][\'detector_fp16\']\n                    and not payload[\'base_runtime\'][\'segmenter_fp16\'], \'Base service runtime differs from CPU FP32.\')\n            with (response_dir / (\'case_\' + str(ordinal).zfill(2) + \'.json\')).open(\'xb\') as stream:\n                stream.write(response.content)\n            cases.append({\'case_index\': ordinal, \'export_name\': image[\'export_name\'],\n                          \'image_sha256\': image[\'image_sha256\'], \'http_status\': response.status_code,\n                          \'response_sha256\': hashlib.sha256(response.content).hexdigest(),\n                          \'client_seconds_observed_functional_request_not_benchmark\': seconds, \'payload\': payload})\n            print(\'Actual /assist Validation request\', ordinal + 1, \'/5:\', payload[\'status\'])\n        page = session.get(url + \'/operator\', timeout=20)\n        require(page.status_code == 200 and page.content == (deploy / \'operator_ui.html\').read_bytes(),\n                \'Operator HTML route did not return the saved UI.\')\n        reference = session.get(url + \'/assist-gallery/0\', timeout=20)\n        require(reference.status_code == 200 and hashlib.sha256(reference.content).hexdigest() == asset_hashes[\'gallery/g_000.png\'],\n                \'Reference image route bytes differ.\')\n        routes = {\'operator_html_status\': page.status_code,\n                  \'operator_html_sha256\': hashlib.sha256(page.content).hexdigest(),\n                  \'gallery_0_status\': reference.status_code,\n                  \'gallery_0_sha256\': hashlib.sha256(reference.content).hexdigest(),\n                  \'scope\': \'Actual HTTP route checks; UI source was checked, but no human-operated browser review is claimed.\'}\n        save_json(output / \'operator_routes_check.json\', routes)\n    finally:\n        session.close()\n        if process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=20)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait(timeout=10)\n        logfile.close()\n    save_json(output / \'operator_cases.json\', cases)\n    operator_metadata = render_operator_cases(cases, gallery, assets, output / \'operator_examples\')\n    save_json(output / \'operator_examples_index.json\', operator_metadata)\n    after = {name: sha(path) for name, path in protected.items()}\n    require(before == after, \'Protected original project files changed during the extension run.\')\n    require(sha(output / \'frozen_operator_VAL_protocol.json\') == protocol_sha, \'Frozen protocol changed.\')\n    save_json(output / \'protected_original_files.json\', {\n        \'before\': before, \'after\': after, \'unchanged\': True,\n        \'paths\': {name: str(path) for name, path in protected.items()},\n        \'test_model_calls\': 0, \'training_or_optimizer_steps\': 0})\n    requirements = {name: installed(name) for name in [\'torch\', \'torchvision\', \'numpy\', \'Pillow\', \'requests\',\n                    \'ultralytics\', \'transformers\', \'huggingface-hub\', \'tokenizers\', \'fastapi\', \'uvicorn\', \'python-multipart\']}\n    save_json(deploy / \'runtime_versions.json\', requirements)\n    restart_cell = \'\'\'# Run this saved cell after mounting the same Drive and installing the recorded dependencies.\nfrom pathlib import Path\nimport os, socket, subprocess, sys, time\nimport requests\nfrom google.colab import drive, output\ndrive.mount(\'/content/drive\')\nDEPLOYMENT = Path(__DEPLOYMENT_PATH__)\nwith socket.socket() as listener:\n    listener.bind((\'127.0.0.1\', 0))\n    port = listener.getsockname()[1]\nenv = os.environ.copy()\nenv.update({\'WANDB_MODE\':\'disabled\', \'OMP_NUM_THREADS\':\'1\', \'MKL_NUM_THREADS\':\'1\'})\nlog = (DEPLOYMENT / (\'operator_ui_\' + str(time.time_ns()) + \'.log\')).open(\'x\')\nserver = subprocess.Popen([sys.executable, str(DEPLOYMENT / \'extension_assist_start.py\'), \'--port\', str(port)],\n                          cwd=DEPLOYMENT, env=env, stdout=log, stderr=subprocess.STDOUT)\nclient = requests.Session()\nclient.trust_env = False\ndeadline = time.monotonic() + 900\ntry:\n    while True:\n        if server.poll() is not None:\n            raise RuntimeError(\'Operator service stopped. Inspect the new log in the deployment folder.\')\n        try:\n            if client.get(\'http://127.0.0.1:\' + str(port) + \'/assist-health\', timeout=3).status_code == 200:\n                break\n        except requests.RequestException:\n            pass\n        if time.monotonic() > deadline:\n            server.terminate()\n            raise RuntimeError(\'Operator service startup timed out; inspect the new log.\')\n        time.sleep(2)\nfinally:\n    client.close()\n    log.close()\nprint(\'Local service is running for this Colab session. Closing the runtime stops it.\')\noutput.serve_kernel_port_as_iframe(port, path=\'/operator\', height=\'860\')\n# To stop only this process when finished: server.terminate()\n\'\'\'.replace(\'__DEPLOYMENT_PATH__\', repr(str(deploy)))\n    compile(restart_cell, \'START_OPERATOR_UI_COLAB.txt\', \'exec\')\n    with (deploy / \'START_OPERATOR_UI_COLAB.txt\').open(\'x\', encoding=\'utf-8\') as stream:\n        stream.write(restart_cell)\n    readme = \'\'\'# Saved operator assistance workflow\n\nThis new directory is separate from the original service and recorded results.\nThe local server was started for five real Validation requests and then stopped.\nNo external hosting or completed human review is claimed.\n\nTo run again in a Colab session:\n1. Mount the same Google Drive account containing Project9_Phase2.\n2. Keep a matching preinstalled torch/torchvision pair. Install ultralytics==8.4.213,\n   transformers==5.22.1, fastapi==0.141.1, uvicorn and python-multipart if missing.\n   The exact observed versions are in runtime_versions.json.\n3. Copy START_OPERATOR_UI_COLAB.txt into a new Colab cell and run it.\n   It starts the saved service in the background, waits for its health route,\n   and embeds /operator in the notebook using the official Colab port helper.\n   The official assistance model files download only if not cached. Their pinned\n   revision and full byte hashes must match assist_manifest.json before serving.\n4. Choose a conveyor image and click Review an item, or call POST /assist with multipart\n   file=<image>. GET /assist-health and /assist-gallery/0 verify service/reference access.\n   Original POST /predict remains available and retains its original output.\n   The embedded browser interaction itself has not been audited by the saved HTTP checks.\n\nFor a terminal environment: python extension_assist_start.py --port 8008\nand open http://127.0.0.1:8008/operator while that process runs.\nColab helper source: https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py\n\nOriginal YOLO checkpoints, service.py and release_manifest.json must remain at\nthe Drive path recorded in assist_manifest.json. They are not included in this ZIP.\nFour material suggestions are advisory. No automatic relabelling, confidence\nthreshold selection, unknown-class detection or physical grasp claim is made.\nSaved case HTML/cards are reproducible observations, not a human approval log.\nAll new crop metrics are Validation only and kept separate from historical metrics.\nThe model binaries, full source PNGs, source annotation JSON and Test images are\nexcluded from this evidence ZIP. Train reference crops and Val query crops are included.\nDo not publish these dataset crops or private project paths to a public repository.\n\'\'\'\n    with (deploy / \'README_RUN.txt\').open(\'x\', encoding=\'utf-8\') as stream:\n        stream.write(readme)\n    summary = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n               \'status\': \'completed_validation_and_functional_HTTP_workflow\',\n               \'comparison\': comparison, \'gallery_reconstruction_passed\': True,\n               \'failure_examples\': failure_metadata, \'operator_examples\': operator_metadata,\n               \'operator_requests\': len(cases), \'operator_requests_http200\': len(cases),\n               \'operator_requests_with_detections\': sum(case[\'payload\'][\'status\'] == \'assistance_ready\' for case in cases),\n               \'server_stopped_after_recording\': True, \'public_hosting_claimed\': False,\n               \'human_review_completed_claimed\': False, \'original_project_files_unchanged\': True,\n               \'test_model_calls\': 0, \'optimizer_steps\': 0}\n    save_json(output / \'summary.json\', summary)\n    inventory = [{\'file\': path.relative_to(output).as_posix(), \'bytes\': path.stat().st_size, \'sha256\': sha(path)}\n                 for path in sorted(output.rglob(\'*\')) if path.is_file()]\n    save_json(output / \'collection_manifest.json\', {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'files\': inventory})\n    archive_path = project / (\'Project9_EXTENSIONS_OPERATOR_VAL_\' + stamp + \'.zip\')\n    with zipfile.ZipFile(archive_path, \'x\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file():\n                archive.write(path, arcname=path.relative_to(output).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        require(archive.testzip() is None, \'Evidence ZIP failed CRC verification.\')\n        for item in inventory:\n            data = archive.read(item[\'file\'])\n            require(len(data) == item[\'bytes\'] and hashlib.sha256(data).hexdigest() == item[\'sha256\'],\n                    \'Packaged evidence hash differs: \' + item[\'file\'])\n    print(\'=== EXTENSIONS COMPLETE: VALIDATION + ACTUAL HTTP WORKFLOW ===\')\n    print(\'Original files unchanged. No training, optimizer step or Test evaluation.\')\n    print(\'Evidence directory:\', output)\n    print(\'ZIP:\', archive_path)\n    print(\'New crop metrics are separate from the original recorded results.\')\n    files.download(str(archive_path))\n    return {\'output_dir\': str(output), \'zip\': str(archive_path), \'summary\': summary}\n\n\n_EXTENSION_RUN_RESULT = run_extension_assist()\n'}

"""CPU models and numeric helpers for the optional operator-assistance stage.

Importing this module performs no model loading, downloads or inference.
Model construction is explicit; all predictions are advisory and do not alter
the original detector/segmenter response or introduce a rejection threshold.
"""

CLASS_NAMES = ('cardboard', 'metal', 'rigid_plastic', 'soft_plastic')
CLIP_MATERIAL_NAMES = ('cardboard', 'metal', 'rigid plastic', 'soft plastic')
PLAIN_PROMPTS = tuple('a photo of ' + name for name in CLIP_MATERIAL_NAMES)
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'


def _assist_require(condition, message):
    if not condition:
        raise RuntimeError(message)


def _assist_sha256(path):
    from pathlib import Path
    import hashlib
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()


def unit(values):
    """The original float32 L2 normalization, including its 1e-10 guard."""
    import numpy as np
    array = np.asarray(values, dtype=np.float32)
    _assist_require(array.ndim == 2 and np.isfinite(array).all(), 'Expected finite 2D features.')
    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)


def rgb_hist(crops):
    """Original 24-bin RGB histogram baseline; crops must be PIL images."""
    import numpy as np
    rows = []
    for crop in crops:
        array = np.asarray(crop.convert('RGB').resize((64, 64)), dtype=np.uint8)
        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,
                                                range=(0, 256))[0] for channel in range(3)]))
    _assist_require(bool(rows), 'At least one crop is required.')
    return unit(np.asarray(rows, dtype=np.float32))


def classification_metrics(true_labels, predicted_labels):
    """Four-material accuracy and macro metrics with zero-division set to zero."""
    import numpy as np
    truth = np.asarray(true_labels)
    prediction = np.asarray(predicted_labels)
    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape
                    and truth.size > 0, 'Class arrays must be nonempty, aligned 1D arrays.')
    _assist_require(np.issubdtype(truth.dtype, np.integer)
                    and np.issubdtype(prediction.dtype, np.integer), 'Class IDs must be integers.')
    _assist_require(((truth >= 0) & (truth < 4)).all()
                    and ((prediction >= 0) & (prediction < 4)).all(), 'Unknown material class ID.')
    matrix = np.zeros((4, 4), dtype=np.int64)
    np.add.at(matrix, (truth, prediction), 1)
    tp = np.diag(matrix).astype(np.float64)
    predicted_support = matrix.sum(axis=0)
    support = matrix.sum(axis=1)
    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)
    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)
    return {
        'accuracy': float(np.mean(truth == prediction)), 'macro_f1': float(f1.mean()),
        'macro_precision': float(precision.mean()), 'macro_recall': float(recall.mean()),
        'count': int(truth.size), 'confusion_matrix_true_rows_predicted_columns': matrix.tolist(),
        'per_class': [{'class_id': index, 'material': name, 'support': int(support[index]),
                       'precision': float(precision[index]), 'recall': float(recall[index]),
                       'f1': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],
    }


def retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):
    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""
    import numpy as np
    gallery = np.asarray(gallery_features, dtype=np.float32)
    queries = np.asarray(query_features, dtype=np.float32)
    gallery_y = np.asarray(gallery_labels)
    query_y = np.asarray(query_labels)
    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]
                    and len(gallery) >= 5 and len(queries) > 0, 'Invalid gallery/query feature shape.')
    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)
                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), 'Misaligned or nonfinite retrieval inputs.')
    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]
    top1 = gallery_y[rankings[:, 0]]
    top5 = gallery_y[rankings[:, :5]]
    return ({'top1_accuracy': float(np.mean(top1 == query_y)),
             'hit_at_5': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),
             'precision_at_5': float(np.mean(top5 == query_y[:, None]))}, rankings)


class OperatorAssistModels:
    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.

    ``assets_dir`` may be None while building a new gallery. Then call
    ``load_gallery`` after writing these files:

    - gallery_index.npz: float32 embeddings [160,512], float32 histograms
      [160,24], int64 labels [160].
    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,
      each with class_id and crop_file="gallery/g_000.png" etc.
    - the corresponding 160 PNG reference crops.

    local_only=True requires the ResNet checkpoint and the explicit immutable
    CLIP revision to be present already; it performs no network request.
    """

    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):
        from pathlib import Path
        from importlib.metadata import version
        import re
        import torch
        import torch.nn as nn
        from torchvision.models import resnet18, ResNet18_Weights
        from transformers import CLIPModel, CLIPProcessor
        from huggingface_hub import hf_hub_download, model_info

        self.torch = torch
        self.assets_dir = None
        self.gallery_embeddings = None
        self.gallery_histograms = None
        self.gallery_labels = None
        self.gallery_meta = None
        self.local_only = bool(local_only)
        torch.set_num_threads(1)
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            _assist_require(torch.get_num_interop_threads() == 1,
                            'Torch inter-op threads are already initialized differently; use a fresh process.')

        weights = ResNet18_Weights.IMAGENET1K_V1
        self.resnet_transform = weights.transforms()
        resnet_file = Path(torch.hub.get_dir()) / 'checkpoints/resnet18-f37072fd.pth'
        if self.local_only:
            _assist_require(resnet_file.is_file(), 'Cached ResNet18 checkpoint is required for local-only serving.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'Cached ResNet18 checkpoint checksum differs.')
            encoder = resnet18(weights=None)
            state = torch.load(resnet_file, map_location='cpu', weights_only=True)
            encoder.load_state_dict(state, strict=True)
            del state
        else:
            encoder = resnet18(weights=weights, progress=False)
            _assist_require(resnet_file.is_file(), 'Downloaded ResNet18 checkpoint was not found at its standard cache path.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'ResNet18 checkpoint checksum differs from its official filename prefix.')
        encoder.fc = nn.Identity()
        self.encoder = encoder.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.encoder.parameters():
            parameter.requires_grad_(False)

        hub_info = None
        if clip_revision is None:
            _assist_require(not self.local_only, 'An explicit cached CLIP revision is required in local-only mode.')
            hub_info = model_info(CLIP_MODEL_ID, revision='main', files_metadata=True, token=False)
            clip_revision = hub_info.sha
        _assist_require(isinstance(clip_revision, str)
                        and re.fullmatch(r'[0-9a-f]{40}|[0-9a-f]{64}', clip_revision) is not None,
                        'CLIP revision must be an immutable Hub commit hash.')
        self.clip_revision = clip_revision
        common = {'revision': self.clip_revision, 'local_files_only': self.local_only, 'token': False}
        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,
                                             trust_remote_code=False, **common)
        self.clip = self.clip.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.clip.parameters():
            parameter.requires_grad_(False)
        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,
                                                       trust_remote_code=False, **common)
        # The loader above explicitly selects the bin checkpoint. This cache-only
        # lookup records its actual bytes without triggering a second download.
        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename='pytorch_model.bin',
                                        revision=self.clip_revision, local_files_only=True, token=False))
        clip_sha = _assist_sha256(clip_file)
        expected_clip_sha = None
        if hub_info is not None:
            for sibling in hub_info.siblings or []:
                if sibling.rfilename == 'pytorch_model.bin':
                    lfs = getattr(sibling, 'lfs', None)
                    expected_clip_sha = (lfs.get('sha256') if isinstance(lfs, dict)
                                         else getattr(lfs, 'sha256', None))
                    break
        if expected_clip_sha is not None:
            _assist_require(clip_sha == expected_clip_sha, 'CLIP binary differs from the Hub LFS SHA256.')
        config_commit = getattr(self.clip.config, '_commit_hash', None)
        if config_commit is not None:
            _assist_require(config_commit == self.clip_revision, 'Loaded CLIP configuration revision differs.')
        self._provenance = {
            'resnet_weights': 'ResNet18_Weights.IMAGENET1K_V1',
            'resnet_weights_url': weights.url, 'resnet_checkpoint_path': str(resnet_file),
            'resnet_checkpoint_sha256': resnet_sha, 'resnet_transform': repr(self.resnet_transform),
            'clip_model_id': CLIP_MODEL_ID, 'clip_revision': self.clip_revision,
            'clip_config_commit_hash': config_commit, 'clip_checkpoint_path': str(clip_file),
            'clip_checkpoint_sha256': clip_sha, 'clip_hub_lfs_sha256': expected_clip_sha,
            'clip_checkpoint_url': 'https://huggingface.co/' + CLIP_MODEL_ID + '/resolve/'
                                   + self.clip_revision + '/pytorch_model.bin',
            'clip_processor_class': type(self.processor).__name__,
            'clip_image_processor_class': type(self.processor.image_processor).__name__,
            'clip_image_processor_config': self.processor.image_processor.to_dict(),
            'clip_attention_implementation': getattr(self.clip.config, '_attn_implementation', None),
            'versions': {name: version(name) for name in ['torch', 'torchvision', 'transformers',
                                                         'huggingface-hub', 'numpy', 'Pillow']},
        }
        self._assert_cpu_frozen()
        if assets_dir is not None:
            self.load_gallery(assets_dir)

    def _assert_cpu_frozen(self):
        for name, model in [('resnet', self.encoder), ('clip', self.clip)]:
            _assist_require(model.training is False, name + ' must be in eval mode.')
            _assist_require(all(parameter.device.type == 'cpu' and not parameter.requires_grad
                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)
                                for parameter in model.parameters()), name + ' must be frozen CPU FP32.')

    def embed_resnet(self, crops, batch=16):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid ResNet crop batch.')
        self._assert_cpu_frozen()
        vectors = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                tensor = self.torch.stack([self.resnet_transform(crop.convert('RGB'))
                                          for crop in crops[start:start + batch]])
                output = self.encoder(tensor.to(device='cpu', dtype=self.torch.float32))
                vectors.append(output.detach().cpu().numpy())
        array = np.concatenate(vectors).astype(np.float32)
        _assist_require(array.shape == (len(crops), 512), 'Unexpected ResNet feature dimensions.')
        return unit(array)

    def score_clip(self, crops, batch=8):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid CLIP crop batch.')
        self._assert_cpu_frozen()
        logits = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                inputs = self.processor(text=list(PLAIN_PROMPTS),
                                        images=[crop.convert('RGB') for crop in crops[start:start + batch]],
                                        return_tensors='pt', padding=True)
                inputs = {key: value.to(device='cpu', dtype=self.torch.float32)
                          if key == 'pixel_values' else value.to('cpu') for key, value in inputs.items()}
                output = self.clip(**inputs)
                logits.append(output.logits_per_image.detach().cpu().numpy())
        array = np.concatenate(logits).astype(np.float32)
        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), 'Unexpected or nonfinite CLIP logits.')
        return array

    def rgb_hist(self, crops):
        return rgb_hist(crops)

    def load_gallery(self, assets_dir):
        from pathlib import Path, PurePosixPath
        import json
        import numpy as np
        directory = Path(assets_dir).resolve()
        with np.load(directory / 'gallery_index.npz', allow_pickle=False) as bank:
            _assist_require(set(bank.files) == {'embeddings', 'histograms', 'labels'}, 'Unexpected assistance gallery keys.')
            embeddings = bank['embeddings'].copy()
            histograms = bank['histograms'].copy()
            labels = bank['labels'].copy()
        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32
                        and histograms.shape == (160, 24) and histograms.dtype == np.float32
                        and labels.shape == (160,) and labels.dtype == np.int64,
                        'Assistance gallery array shapes/dtypes differ.')
        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()
                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),
                        'Assistance gallery features/classes differ.')
        for name, values in [('embedding', embeddings), ('histogram', histograms)]:
            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),
                            'Unnormalized gallery ' + name + ' features.')
        metadata = json.loads((directory / 'gallery_meta.json').read_text(encoding='utf-8'))
        _assist_require(isinstance(metadata, list) and len(metadata) == 160, 'Expected 160 gallery metadata records.')
        for index, item in enumerate(metadata):
            expected_file = 'gallery/g_' + str(index).zfill(3) + '.png'
            _assist_require(isinstance(item, dict) and item.get('class_id') == int(labels[index])
                            and item.get('crop_file') == expected_file, 'Gallery metadata/array alignment differs.')
            crop = PurePosixPath(item['crop_file'])
            path = (directory / crop).resolve()
            _assist_require(not crop.is_absolute() and '..' not in crop.parts
                            and path.is_relative_to(directory) and path.is_file(), 'Missing or unsafe reference crop.')
        self.assets_dir = directory
        self.gallery_embeddings = embeddings
        self.gallery_histograms = histograms
        self.gallery_labels = labels
        self.gallery_meta = metadata
        return self

    def query_crop(self, crop):
        import numpy as np
        _assist_require(self.gallery_embeddings is not None, 'Load the assistance gallery first.')
        feature = self.embed_resnet([crop], batch=1)
        similarities = (feature @ self.gallery_embeddings.T)[0]
        ranking = similarities.argsort()[::-1]
        references = []
        for raw_index in ranking[:5]:
            index = int(raw_index)
            class_id = int(self.gallery_labels[index])
            references.append({'gallery_index': index, 'class_id': class_id,
                               'material': CLASS_NAMES[class_id], 'similarity': float(similarities[index]),
                               'crop_file': self.gallery_meta[index]['crop_file']})
        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)
        relative = np.exp(clip_logits - clip_logits.max())
        relative /= relative.sum()
        clip_class = int(clip_logits.argmax())
        histogram = rgb_hist([crop])
        hist_similarities = (histogram @ self.gallery_histograms.T)[0]
        hist_index = int(hist_similarities.argmax())
        hist_class = int(self.gallery_labels[hist_index])
        return {
            'visual_search': {'predicted_class_id': references[0]['class_id'],
                              'predicted_material': references[0]['material'], 'references': references,
                              'score_type': 'Cosine similarity to frozen Train reference crops; not calibrated confidence.'},
            'clip': {'predicted_class_id': clip_class, 'predicted_material': CLASS_NAMES[clip_class],
                     'logits': clip_logits.tolist(), 'relative_scores': relative.tolist(),
                     'prompts': list(PLAIN_PROMPTS),
                     'score_type': 'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.'},
            'histogram': {'predicted_class_id': hist_class, 'predicted_material': CLASS_NAMES[hist_class],
                          'reference_gallery_index': hist_index, 'similarity': float(hist_similarities[hist_index]),
                          'crop_file': self.gallery_meta[hist_index]['crop_file'],
                          'score_type': 'Nearest Train RGB histogram by cosine similarity; argmax tie policy.'},
        }

    def runtime_metadata(self):
        self._assert_cpu_frozen()
        return {
            **self._provenance, 'clip_revision': self.clip_revision, 'device': 'cpu', 'precision': 'float32',
            'resnet_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()
                                              if parameter.is_floating_point()}),
            'clip_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.clip.parameters()
                                            if parameter.is_floating_point()}),
            'torch_num_threads': int(self.torch.get_num_threads()),
            'torch_num_interop_threads': int(self.torch.get_num_interop_threads()),
            'all_parameters_frozen': True, 'plain_prompts': list(PLAIN_PROMPTS),
            'gallery_loaded': self.gallery_embeddings is not None,
            'gallery_size': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,
            'model_training_or_optimization': False,
            'decision_role': 'Advisory references and material suggestion; original detector outputs are not overwritten.',
        }


"""Pure-Pillow cards for recorded extension results and operator responses.

No model execution, package installation, network access or automatic writes.
Functions draw supplied data only and keep ground-truth crop experiments
separate from the detector-crop operator examples.
"""

_EXT_CLASS_NAMES = ('cardboard', 'metal', 'rigid_plastic', 'soft_plastic')
_EXT_COLORS = ((201, 134, 32), (36, 122, 179), (66, 137, 58), (153, 71, 156))


def _ext_require(condition, message):
    if not condition:
        raise ValueError(message)


def _ext_font(size, bold=False):
    from PIL import ImageFont
    choices = (['DejaVuSans-Bold.ttf', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
                r'C:\Windows\Fonts\arialbd.ttf'] if bold else
               ['DejaVuSans.ttf', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                r'C:\Windows\Fonts\arial.ttf'])
    for path in choices:
        try:
            return ImageFont.truetype(path, size=size)
        except OSError:
            pass
    return ImageFont.load_default(size=size)


def _ext_lines(draw, text, font, width):
    result = []
    for paragraph in str(text).split('\n'):
        current = ''
        for word in paragraph.split(' '):
            trial = (current + ' ' + word).strip()
            if draw.textbbox((0, 0), trial, font=font)[2] <= width:
                current = trial
                continue
            if current:
                result.append(current)
                current = ''
            # Long filenames are split instead of clipped or silently omitted.
            chunk = ''
            for char in word:
                if chunk and draw.textbbox((0, 0), chunk + char, font=font)[2] > width:
                    result.append(chunk)
                    chunk = char
                else:
                    chunk += char
            current = chunk
        result.append(current)
    return result


def _ext_text(draw, text, box, size=20, bold=False, color=(32, 42, 53), min_size=12):
    x, y, width, height = box
    while size >= min_size:
        font = _ext_font(size, bold)
        lines = _ext_lines(draw, text, font, width)
        spacing = max(2, int(size * .22))
        line_height = size + spacing
        if len(lines) * line_height <= height:
            for index, line in enumerate(lines):
                bb = draw.textbbox((0, 0), line, font=font)
                draw.text((x - bb[0], y + index * line_height - bb[1]), line, font=font, fill=color)
            return
        size -= 1
    raise ValueError('Evidence text does not fit its allocated card area.')


def _ext_safe_crop(assets_dir, record):
    from pathlib import Path, PurePosixPath
    root = Path(assets_dir).resolve()
    rel_value = record.get('crop_file')
    _ext_require(isinstance(rel_value, str) and '\\' not in rel_value, 'Invalid crop filename.')
    relative = PurePosixPath(rel_value)
    _ext_require(not relative.is_absolute() and '..' not in relative.parts, 'Unsafe crop path.')
    path = (root / Path(*relative.parts)).resolve()
    _ext_require(path.is_relative_to(root) and path.is_file(), 'Reference crop is absent or outside assets.')
    return path


def _ext_validate_records(assets_dir, records, expected):
    _ext_require(isinstance(records, (list, tuple)) and len(records) == expected, 'Unexpected crop cohort size.')
    for item in records:
        _ext_require(isinstance(item, dict) and isinstance(item.get('class_id'), int)
                     and not isinstance(item['class_id'], bool) and 0 <= item['class_id'] < 4,
                     'Invalid crop material class.')
        _ext_safe_crop(assets_dir, item)


def _ext_thumb(canvas, source, box):
    from PIL import Image, ImageDraw
    x, y, width, height = (int(value) for value in box)
    ImageDraw.Draw(canvas).rectangle((x, y, x + width - 1, y + height - 1), fill=(235, 239, 244), outline=(206, 214, 224))
    if isinstance(source, Image.Image):
        image = source.convert('RGB').copy()
    else:
        with Image.open(source) as opened:
            image = opened.convert('RGB')
    image.thumbnail((width - 8, height - 8), Image.Resampling.LANCZOS)
    canvas.paste(image, (x + (width - image.width) // 2, y + (height - image.height) // 2))


def _ext_new_output(output_dir):
    from pathlib import Path
    path = Path(output_dir).resolve()
    path.mkdir(parents=True, exist_ok=True)
    return path


def _ext_save_card(image, path):
    with path.open('xb') as stream:
        image.save(stream, format='JPEG', quality=94, subsampling=0)


def _ext_save_json(path, value):
    import json
    with path.open('x', encoding='utf-8', newline='\n') as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.write('\n')


def _ext_html(path, title, note, cards):
    import base64
    import html
    parts = ['<!doctype html><html lang="en"><meta charset="utf-8">',
             '<meta name="viewport" content="width=device-width, initial-scale=1">',
             '<title>' + html.escape(title) + '</title>',
             '<style>body{font:18px Arial,sans-serif;margin:24px;background:#eef2f6;color:#202a35}'
             'main{max-width:1500px;margin:auto}img{width:100%;height:auto;display:block}'
             'figure{margin:24px 0;padding:12px;background:white;border:1px solid #ccd5df}'
             'figcaption{margin:10px 4px;line-height:1.4}.note{line-height:1.5}</style><main>',
             '<h1>' + html.escape(title) + '</h1><p class="note">' + html.escape(note) + '</p>']
    for card in cards:
        encoded = base64.b64encode(card['card_path'].read_bytes()).decode('ascii')
        parts.extend(['<figure><img alt="' + html.escape(card['caption'], quote=True)
                      + '" src="data:image/jpeg;base64,' + encoded + '">',
                      '<figcaption>' + html.escape(card['caption']) + '</figcaption></figure>'])
    if not cards:
        parts.append('<p>No example card was produced because the stated error condition was absent.</p>')
    parts.append('</main></html>')
    with path.open('x', encoding='utf-8', newline='\n') as stream:
        stream.write('\n'.join(parts) + '\n')


def _ext_list(value):
    return value.tolist() if hasattr(value, 'tolist') else list(value)


def _ext_class(value):
    _ext_require(isinstance(value, int) and not isinstance(value, bool) and 0 <= value < 4, 'Invalid prediction class.')
    return value


def _ext_relative(logits):
    import math
    values = [float(value) for value in logits]
    _ext_require(len(values) == 4 and all(math.isfinite(value) for value in values), 'Invalid CLIP logits.')
    scaled = [math.exp(value - max(values)) for value in values]
    return [value / sum(scaled) for value in scaled]


def render_extension_failures(assets_dir, gallery_records, query_records, rankings, clip_logits, hist_predictions, output_dir):
    """Draw up to three observed errors per method from 40 annotated Val crops.

    rankings is [40,160] in descending ResNet similarity order; clip_logits is
    [40,4] for the plain prompts; hist_predictions contains 40 class IDs.
    Returns JSON-serializable metadata plus a self-contained static HTML view.
    """
    from PIL import Image, ImageDraw
    _ext_validate_records(assets_dir, gallery_records, 160)
    _ext_validate_records(assets_dir, query_records, 40)
    ranks = [_ext_list(row) for row in _ext_list(rankings)]
    logits = [_ext_list(row) for row in _ext_list(clip_logits)]
    hist = _ext_list(hist_predictions)
    _ext_require(len(ranks) == len(logits) == len(hist) == 40, 'Unexpected result-array row count.')
    for row in ranks:
        _ext_require(len(row) == 160 and all(isinstance(value, int) and not isinstance(value, bool) for value in row)
                     and set(row) == set(range(160)), 'Rankings must be permutations of all 160 Train items.')
    hist = [_ext_class(value) for value in hist]
    relative = [_ext_relative(row) for row in logits]
    clip_pred = [max(range(4), key=lambda class_id: float(row[class_id])) for row in logits]
    truths = [item['class_id'] for item in query_records]
    resnet_pred = [gallery_records[row[0]]['class_id'] for row in ranks]
    resnet_wrong = [index for index in range(40) if resnet_pred[index] != truths[index]]
    clip_wrong = [index for index in range(40) if clip_pred[index] != truths[index]]
    output = _ext_new_output(output_dir)
    cards = []
    html_cards = []

    def query_pane(canvas, draw, query_index, box):
        record = query_records[query_index]
        _ext_thumb(canvas, _ext_safe_crop(assets_dir, record), box)
        _ext_text(draw, 'Annotated Validation query', (box[0], box[1] - 35, box[2], 28), size=20, bold=True)
        _ext_text(draw, 'Ground truth: ' + _EXT_CLASS_NAMES[truths[query_index]].replace('_', ' '),
                  (box[0], box[1] + box[3] + 12, box[2], 60), size=20, bold=True)
        _ext_text(draw, str(record.get('export_name', record['crop_file'])),
                  (box[0], box[1] + box[3] + 78, box[2], 65), size=17)

    for ordinal, index in enumerate(resnet_wrong[:3]):
        canvas = Image.new('RGB', (1500, 640), 'white'); draw = ImageDraw.Draw(canvas)
        _ext_text(draw, 'Visual Search: an incorrect top-1 material', (24, 20, 1452, 40), 29, True)
        _ext_text(draw, f'Validation crop {index:02d} | Query label: {_EXT_CLASS_NAMES[truths[index]]} | '
                  f'Top-1: {_EXT_CLASS_NAMES[resnet_pred[index]]} | RGB baseline: {_EXT_CLASS_NAMES[hist[index]]}',
                  (24, 68, 1452, 52), 20)
        query_pane(canvas, draw, index, (24, 160, 252, 240))
        references = []
        for rank, gallery_index in enumerate(ranks[index][:5], 1):
            record = gallery_records[gallery_index]; cls = record['class_id']; x = 310 + (rank - 1) * 232
            same = cls == truths[index]
            _ext_text(draw, 'Train reference #' + str(rank), (x, 126, 218, 28), 19, True)
            _ext_thumb(canvas, _ext_safe_crop(assets_dir, record), (x, 160, 218, 240))
            _ext_text(draw, _EXT_CLASS_NAMES[cls].replace('_', ' '), (x, 412, 218, 35), 20, True,
                      _EXT_COLORS[cls])
            _ext_text(draw, 'Same query material' if same else 'Different query material',
                      (x, 450, 218, 48), 17)
            _ext_text(draw, str(record.get('export_name', record['crop_file'])), (x, 500, 218, 64), 15)
            references.append({'rank': rank, 'gallery_index': gallery_index, 'class_id': cls,
                               'crop_file': record['crop_file'], 'same_query_material': same})
        _ext_text(draw, 'GT-crop experiment: localization errors are excluded. The first observed errors are shown; '
                  'this is not a representative failure rate or an explanation of the cause.', (24, 582, 1452, 45), 18)
        path = output / f'failure_resnet_{ordinal:02d}.jpg'; _ext_save_card(canvas, path)
        caption = f'Visual Search: query {index}, annotated {_EXT_CLASS_NAMES[truths[index]]}, top-1 {_EXT_CLASS_NAMES[resnet_pred[index]]}.'
        card = {'kind': 'visual_search_top1_error', 'query_index': index, 'query_crop_file': query_records[index]['crop_file'],
                'ground_truth_class_id': truths[index], 'prediction_class_id': resnet_pred[index],
                'histogram_prediction_class_id': hist[index], 'references': references,
                'card_file': path.name, 'card_path': str(path), 'caption': caption}
        cards.append(card); html_cards.append({'card_path': path, 'caption': caption})

    for ordinal, index in enumerate(clip_wrong[:3]):
        canvas = Image.new('RGB', (1500, 640), 'white'); draw = ImageDraw.Draw(canvas)
        _ext_text(draw, 'CLIP plain prompts: an incorrect material suggestion', (24, 20, 1452, 40), 29, True)
        _ext_text(draw, f'Validation crop {index:02d} | Query label: {_EXT_CLASS_NAMES[truths[index]]} | '
                  f'CLIP top-1: {_EXT_CLASS_NAMES[clip_pred[index]]} | RGB baseline: {_EXT_CLASS_NAMES[hist[index]]}',
                  (24, 68, 1452, 52), 20)
        query_pane(canvas, draw, index, (24, 160, 300, 260))
        _ext_text(draw, 'Four-way relative text-image scores (uncalibrated)', (390, 125, 1080, 35), 23, True)
        for cls in range(4):
            y = 183 + cls * 91
            flags = (' | ground truth' if cls == truths[index] else '') + (' | CLIP top-1' if cls == clip_pred[index] else '')
            _ext_text(draw, _EXT_CLASS_NAMES[cls].replace('_', ' ') + flags, (390, y, 1075, 32), 20, True)
            draw.rectangle((390, y + 38, 1190, y + 58), fill=(231, 236, 242))
            draw.rectangle((390, y + 38, 390 + int(800 * relative[index][cls]), y + 58), fill=_EXT_COLORS[cls])
            _ext_text(draw, f"score {relative[index][cls]:.4f} | logit {float(logits[index][cls]):.3f}",
                      (1210, y + 30, 260, 48), 17)
        _ext_text(draw, 'Plain prompts: "a photo of [material]". These four-way scores are not calibrated probabilities '
                  'or unknown-category detection. GT crops exclude detector errors; no error cause is inferred.',
                  (24, 582, 1452, 45), 18)
        path = output / f'failure_clip_{ordinal:02d}.jpg'; _ext_save_card(canvas, path)
        caption = f'CLIP: query {index}, annotated {_EXT_CLASS_NAMES[truths[index]]}, predicted {_EXT_CLASS_NAMES[clip_pred[index]]}.'
        card = {'kind': 'clip_plain_top1_error', 'query_index': index, 'query_crop_file': query_records[index]['crop_file'],
                'ground_truth_class_id': truths[index], 'prediction_class_id': clip_pred[index],
                'histogram_prediction_class_id': hist[index], 'logits': [float(value) for value in logits[index]],
                'relative_scores': relative[index], 'card_file': path.name, 'card_path': str(path), 'caption': caption}
        cards.append(card); html_cards.append({'card_path': path, 'caption': caption})
    metadata = {'scope': 'First up to three observed top-1 errors per method in the supplied 40 annotated Validation crops.',
                'query_count': 40, 'resnet_top1_error_count': len(resnet_wrong), 'clip_plain_error_count': len(clip_wrong),
                'resnet_error_examples_absent': not resnet_wrong, 'clip_error_examples_absent': not clip_wrong,
                'selection': 'Ascending query index; no example is fabricated when no true/predicted class disagreement exists.',
                'cards': cards, 'output_dir': str(output), 'metadata_file': str(output / 'extension_failure_cards.json'),
                'html_file': str(output / 'extension_failures.html'),
                'limits': 'Ground-truth box crops only; no full-system accuracy, causal diagnosis or Test result is inferred.'}
    _ext_save_json(output / 'extension_failure_cards.json', metadata)
    _ext_html(output / 'extension_failures.html', 'Validation crop failure examples', metadata['scope'] + ' ' + metadata['limits'], html_cards)
    return metadata


def render_operator_cases(responses, gallery_records, assets_dir, output_dir):
    """Draw proof cards from five actual /assist responses; no GT is assigned.

    Entries may be actual response dictionaries, or envelopes with ``payload``
    (``response`` is also accepted), optional export_name and request_id.
    """
    import base64
    import io
    import math
    from PIL import Image, ImageDraw
    _ext_validate_records(assets_dir, gallery_records, 160)
    _ext_require(isinstance(responses, (list, tuple)) and len(responses) == 5, 'Five actual operator responses are required.')
    output = _ext_new_output(output_dir); cards = []; html_cards = []
    for ordinal, entry in enumerate(responses):
        _ext_require(isinstance(entry, dict), 'Operator response must be a dictionary.')
        if isinstance(entry.get('payload'), dict):
            payload = entry['payload']
        elif isinstance(entry.get('response'), dict):
            payload = entry['response']
        else:
            payload = entry
        source_name = str(entry.get('export_name', payload.get('export_name', 'Validation operator case ' + str(ordinal + 1))))
        _ext_require(payload.get('automatic_material_override') is False
                     and payload.get('ground_truth_accuracy_measured') is False,
                     'Operator result must preserve the advisory/no-GT role.')
        detection = payload.get('source_detection')
        _ext_require(isinstance(detection, dict) and isinstance(detection.get('objects'), list), 'Missing original detector response.')
        objects = detection['objects']; selected_index = payload.get('selected_object_index')
        status = payload.get('status'); assistance = payload.get('assistance')
        canvas = Image.new('RGB', (1500, 850), 'white'); draw = ImageDraw.Draw(canvas)
        _ext_text(draw, 'Operator assistance: ' + source_name, (24, 20, 1452, 55), 29, True)
        _ext_text(draw, 'Advisory references and a second material suggestion. The detector result is retained; '
                  'no ground-truth accuracy or human-review outcome is measured.', (24, 83, 1452, 56), 20)
        metadata = {'case_index': ordinal, 'export_name': source_name, 'status': status,
                    'request_id': entry.get('request_id'), 'automatic_material_override': False,
                    'ground_truth_accuracy_measured': False, 'original_detected_object_count': len(objects)}
        if selected_index is None:
            _ext_require(status == 'no_detections' and not objects and assistance is None, 'Inconsistent no-detection response.')
            _ext_text(draw, 'No detected object was available for assistance.', (40, 220, 1400, 90), 32, True)
            _ext_text(draw, 'No reference ranking or CLIP material is invented for this case.', (40, 340, 1400, 70), 23)
            metadata['assistance_absent'] = True
        else:
            _ext_require(status == 'assistance_ready' and isinstance(selected_index, int) and not isinstance(selected_index, bool)
                         and 0 <= selected_index < len(objects) and isinstance(assistance, dict), 'Invalid selected-object response.')
            chosen = payload.get('selected_object')
            _ext_require(isinstance(chosen, dict) and chosen == objects[selected_index]
                         and chosen.get('material') in _EXT_CLASS_NAMES and isinstance(chosen.get('requires_review'), bool),
                         'Selected object differs from the original detector object.')
            confidence = float(chosen['confidence'])
            _ext_require(math.isfinite(confidence) and 0 <= confidence <= 1, 'Invalid original detector confidence.')
            preview = payload.get('crop_preview_png_base64')
            _ext_require(isinstance(preview, str) and len(preview) <= 12 * 1024**2, 'Missing/oversized selected-crop preview.')
            preview_bytes = base64.b64decode(preview, validate=True)
            with Image.open(io.BytesIO(preview_bytes)) as opened:
                _ext_require(opened.format == 'PNG' and 0 < opened.width <= 640 and 0 < opened.height <= 480,
                             'Unexpected selected-crop preview format/size.')
                query = opened.convert('RGB')
            visual = assistance.get('visual_search'); clip = assistance.get('clip'); histogram = assistance.get('histogram')
            _ext_require(all(isinstance(value, dict) for value in [visual, clip, histogram]), 'Missing assistance component.')
            references = visual.get('references')
            _ext_require(isinstance(references, list) and len(references) == 5, 'Expected five actual Train references.')
            reference_metadata = []
            for rank, ref in enumerate(references, 1):
                index = ref.get('gallery_index')
                _ext_require(isinstance(index, int) and not isinstance(index, bool) and 0 <= index < 160, 'Invalid reference index.')
                expected = gallery_records[index]; cls = expected['class_id']
                _ext_require(ref.get('class_id') == cls and ref.get('material') == _EXT_CLASS_NAMES[cls]
                             and ref.get('crop_file') == expected['crop_file'] and math.isfinite(float(ref['similarity'])),
                             'Reference index, label or crop does not match the recorded Train bank.')
                reference_metadata.append({'rank': rank, 'gallery_index': index, 'class_id': cls,
                                           'crop_file': ref['crop_file'], 'similarity': float(ref['similarity'])})
            _ext_require(len({ref['gallery_index'] for ref in reference_metadata}) == 5,
                         'The five references must identify distinct gallery entries.')
            visual_class = _ext_class(visual.get('predicted_class_id'))
            clip_class = _ext_class(clip.get('predicted_class_id')); histogram_class = _ext_class(histogram.get('predicted_class_id'))
            _ext_require(visual_class == reference_metadata[0]['class_id']
                         and visual.get('predicted_material') == _EXT_CLASS_NAMES[visual_class]
                         and clip.get('predicted_material') == _EXT_CLASS_NAMES[clip_class]
                         and histogram.get('predicted_material') == _EXT_CLASS_NAMES[histogram_class],
                         'Assistance material identities disagree.')
            clip_logits = [float(value) for value in clip.get('logits', [])]
            relative = _ext_relative(clip_logits)
            recorded_relative = [float(value) for value in clip.get('relative_scores', [])]
            _ext_require(len(recorded_relative) == 4 and all(math.isfinite(value) and 0 <= value <= 1 for value in recorded_relative)
                         and all(abs(a - b) <= 1e-5 for a, b in zip(relative, recorded_relative))
                         and max(range(4), key=lambda index: clip_logits[index]) == clip_class,
                         'CLIP relative scores/logits/prediction disagree.')
            hist_index = histogram.get('reference_gallery_index')
            _ext_require(isinstance(hist_index, int) and not isinstance(hist_index, bool) and 0 <= hist_index < 160
                         and gallery_records[hist_index]['class_id'] == histogram_class
                         and histogram.get('crop_file') == gallery_records[hist_index]['crop_file']
                         and math.isfinite(float(histogram['similarity'])), 'Invalid histogram reference.')
            _ext_thumb(canvas, query, (24, 210, 342, 300))
            _ext_text(draw, 'Selected detector crop', (24, 160, 342, 35), 23, True)
            _ext_text(draw, f"M4: {chosen['material'].replace('_', ' ')}\nDetector score: {confidence:.4f}\n"
                      + ('Mask association: REVIEW' if chosen['requires_review'] else 'Mask association: available'),
                      (24, 528, 342, 110), 20, True)
            _ext_text(draw, 'Five labelled Train references', (404, 160, 1070, 35), 23, True)
            for rank, ref in enumerate(reference_metadata):
                record = gallery_records[ref['gallery_index']]; x = 404 + rank * 214
                _ext_thumb(canvas, _ext_safe_crop(assets_dir, record), (x, 210, 198, 200))
                _ext_text(draw, f"#{rank + 1} {_EXT_CLASS_NAMES[ref['class_id']].replace('_', ' ')}",
                          (x, 424, 198, 52), 18, True, _EXT_COLORS[ref['class_id']])
                _ext_text(draw, f"Cosine {ref['similarity']:.4f}", (x, 482, 198, 35), 16)
            _ext_text(draw, 'Visual Search top-1: ' + _EXT_CLASS_NAMES[visual_class].replace('_', ' ')
                      + ' | CLIP suggestion: ' + _EXT_CLASS_NAMES[clip_class].replace('_', ' ')
                      + ' | RGB baseline: ' + _EXT_CLASS_NAMES[histogram_class].replace('_', ' '),
                      (404, 546, 1070, 58), 21, True)
            score_text = 'CLIP relative scores: ' + ' | '.join(_EXT_CLASS_NAMES[c].replace('_', ' ')
                         + f' {recorded_relative[c]:.4f}' for c in range(4))
            _ext_text(draw, score_text, (404, 622, 1070, 70), 19)
            _ext_text(draw, 'Similarity and four-way scores are uncalibrated. These suggestions are not automatic '
                      'material overrides or ground-truth error decisions.', (404, 708, 1070, 60), 19)
            metadata.update({'assistance_absent': False, 'selected_object_index': selected_index,
                             'selected_object': chosen, 'crop_box_xyxy': payload.get('crop_box_xyxy'),
                             'visual_search_class_id': visual_class, 'clip_class_id': clip_class,
                             'histogram_class_id': histogram_class, 'references': reference_metadata,
                             'clip_logits': clip_logits, 'clip_relative_scores': recorded_relative,
                             'selection_policy': payload.get('selection_policy')})
        _ext_text(draw, 'Functional case record only: no tracking, physical pick success, overall system accuracy '
                  'or completed human decision is inferred.', (24, 798, 1452, 38), 18)
        path = output / f'operator_case_{ordinal:02d}.jpg'; _ext_save_card(canvas, path)
        caption = source_name + ': ' + ('no detections; assistance absent.' if selected_index is None
                                       else 'actual detector crop, related Train references and advisory material suggestions.')
        metadata.update({'card_file': path.name, 'card_path': str(path), 'caption': caption})
        cards.append(metadata); html_cards.append({'card_path': path, 'caption': caption})
    result = {'scope': 'Five supplied local operator-assistance responses on Validation images; advisory integration evidence only.',
              'case_count': 5, 'cases_with_assistance': sum(not item['assistance_absent'] for item in cards),
              'cases_without_detections': sum(item['assistance_absent'] for item in cards),
              'automatic_material_override': False, 'ground_truth_accuracy_measured': False,
              'cards': cards, 'output_dir': str(output), 'metadata_file': str(output / 'operator_case_cards.json'),
              'html_file': str(output / 'operator_cases.html')}
    _ext_save_json(output / 'operator_case_cards.json', result)
    _ext_html(output / 'operator_cases.html', 'Operator assistance case records', result['scope'], html_cards)
    return result


"""Reconstruct the original Train-gallery/Val-query extension cohorts.

Defining the function does not execute models, install packages, or access data.
Only source Train/Val archive members can be retrieved; Test is never opened.
"""


def prepare_extension_cohorts():
    from collections import Counter, OrderedDict, defaultdict
    from datetime import datetime, timezone
    from importlib.metadata import version
    from pathlib import Path, PurePosixPath
    import csv
    import hashlib
    import io
    import json
    import math
    import re
    import time
    import zipfile
    import zlib
    import numpy as np
    import requests
    from PIL import Image
    from google.colab import drive

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest_bytes(data):
        return hashlib.sha256(data).hexdigest()

    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                value.update(chunk)
        return value.hexdigest()

    def write_new(path, data):
        with Path(path).open('xb') as stream:
            stream.write(data)

    drive.mount('/content/drive')
    root = Path('/content/drive/MyDrive/Project9_Phase2')
    local_root = Path('/content/Project9_Phase2')
    manifest_path = root / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    inventory_path = root / 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json'
    original_index_path = root / 'open_world/visual_search/gallery_index_train.npz'
    expected_manifest = 'ef6b35d2ba7f6708319b824e29707b4d2001750974ac3c4da03c72c9340fdb50'
    expected_inventory = '86e913fa48c09c596f396ebfaeefa49ff2b720d3cf9983f6645bd3f911734835'
    expected_index = '8162cf1b1f4eff5741bcad74230fc0bee3501e27ae89948ddc2c7af0c3532d57'
    expected_train_json = '242ba81def2ee6162f7341f0ee48e21e380411e9fac03df26a270d1a869f860a'
    classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    protected = {
        'split_manifest': manifest_path, 'validation_inventory': inventory_path,
        'gallery_index': original_index_path,
        'visual_search_result': root / 'open_world/visual_search/comparison_VAL.json',
        'clip_result': root / 'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv',
        'service_source': root / 'deployment/http_service/service.py',
        'service_manifest': root / 'deployment/http_service/release_manifest.json',
        'http_detector': root / 'deployment/http_service/best_det.pt',
        'http_segmenter': root / 'deployment/http_service/best_seg.pt',
        'source_detector': root / 'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt',
        'source_segmenter': root / 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt',
        'test_protocol': root / 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'test_results': root / 'evaluation/FINAL_TEST_ONCE/results.json',
    }
    expected_protected = {
        'split_manifest': expected_manifest, 'validation_inventory': expected_inventory,
        'gallery_index': expected_index,
        'visual_search_result': '312d1739e7edfde1e4d3a6ac7b8022f12c135c129bdb2fac9988c47f80f9174d',
        'clip_result': '7d770ea93ee9075b79491c759f7166fb95e2bb620578ebf9d49959085bacdf59',
        'service_source': '6a10bbddd6e675f1b65a8c95f4bf2d3a439ab9a97a85b4caf80e9cd108753bd2',
        'service_manifest': 'c4c6c1b9bbb70ae17922e6c253ebf432482f92332728d0440555a1fd7a7b4903',
        'http_detector': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
        'http_segmenter': '1ceaa151c3dc3d3503d2812216309322be62d25eebfcd57e35f513036412f178',
        'source_detector': '90f2117bdad6b9c0cb3e8d176d056c344bc11a737f7b8871534d0e718d000fd7',
        'source_segmenter': '2cf68941dff6ee01b15ff8947d2f12a8737bdc26e5cc59ec8783e10204179c10',
        'test_protocol': 'aca3f6749633875b719c47c8bddff3f79c0a9da28f35246384a79f6d4cf9faff',
        'test_results': 'b6f48bf11f862930c4a1d34a4290d2fea3ed3918f2ccc153fef169cb78cb91dc',
    }
    missing = [str(path) for path in protected.values() if not path.is_file()]
    require(not missing, 'Missing saved project references; stop and share this output: ' + repr(missing))
    before = {name: digest(path) for name, path in protected.items()}
    for name, expected in expected_protected.items():
        require(before[name] == expected, 'Saved reference/model bytes changed: ' + name)
    rows = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding='utf-8-sig'))))
    active_rows = [row for row in rows if row['split'] in ('train', 'val')]
    require(Counter(row['split'] for row in active_rows) == {'train': 1851, 'val': 451},
            'Frozen Train/Validation image counts differ.')
    require(all(row['original_split'] in ('train', 'val') for row in active_rows),
            'Train/Validation references unexpected source partitions.')
    by_export = {row['original_split'] + '__' + row['filename']: row for row in active_rows}
    require(len(by_export) == len(active_rows), 'Active export filenames are not unique.')
    inventory_items = json.loads(inventory_path.read_text(encoding='utf-8'))
    inventory = {item['export_name']: item for item in inventory_items}
    require(len(inventory_items) == len(inventory) == 451, 'Saved Validation inventory differs.')
    with np.load(original_index_path, allow_pickle=False) as bank:
        require(set(bank.files) == {'embeddings', 'labels', 'source_paths'}, 'Unexpected historical index arrays.')
        historical_labels = bank['labels'].copy()
        historical_names = [PurePosixPath(str(value).replace('\\', '/')).name for value in bank['source_paths']]
        require(bank['embeddings'].shape == (160, 512) and np.isfinite(bank['embeddings']).all()
                and historical_labels.shape == (160,) and len(historical_names) == 160,
                'Historical index dimensions/values differ.')
        require(Counter(int(value) for value in historical_labels) == {0: 40, 1: 40, 2: 40, 3: 40},
                'Historical gallery is not the recorded balanced 160-object bank.')

    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    archive_size = 7518242799
    password = b'REDACTED_DATASET_ZIP_PASSWORD'

    class HTTPRangeFile(io.RawIOBase):
        """Bounded read-only seekable HTTP file for Python's ZIP64 reader."""
        def __init__(self):
            super().__init__()
            self.position = 0
            self.size = archive_size
            self.block_size = 1024 * 1024
            self.blocks = OrderedDict()
            self.transferred = 0
            self.requests_count = 0
            self.session = requests.Session()

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.position

        def seek(self, offset, whence=0):
            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
            if position < 0:
                raise ValueError('Invalid HTTP ZIP seek.')
            self.position = int(position)
            return self.position

        def get_block(self, index):
            if index in self.blocks:
                self.blocks.move_to_end(index)
                return self.blocks[index]
            start = index * self.block_size
            end = min(self.size - 1, start + self.block_size - 1)
            needed = end - start + 1
            require(self.transferred + needed <= 768 * 1024**2,
                    'Partial-download budget exceeded; the full archive will not be downloaded automatically.')
            for attempt in range(3):
                try:
                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={
                        'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity', 'Cache-Control': 'no-cache'
                    }) as response:
                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
                            time.sleep(2 * (attempt + 1))
                            continue
                        require(response.status_code == 206,
                                f'HTTP {response.status_code}: requested range was not supported; full download prevented.')
                        content_range = response.headers.get('Content-Range', '').strip()
                        match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', content_range, flags=re.I)
                        require(match and tuple(map(int, match.groups())) == (start, end, self.size),
                                'Server returned an unexpected Content-Range: ' + content_range)
                        data = response.raw.read(needed + 1)
                        require(len(data) == needed, 'Incomplete or oversized HTTP range response.')
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
            self.transferred += len(data)
            self.requests_count += 1
            self.blocks[index] = data
            while len(self.blocks) > 16:
                self.blocks.popitem(last=False)
            if self.requests_count % 5 == 0:
                print(f'Partial ZIP transfer: {self.transferred / 1024**2:.1f} MiB')
            return data

        def read(self, count=-1):
            if self.closed:
                raise ValueError('HTTP ZIP stream is closed.')
            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))
            require(count <= 64 * 1024**2, 'Unexpected large ZIP read; complete-archive download prevented.')
            chunks = []
            remaining = count
            while remaining:
                index, offset = divmod(self.position, self.block_size)
                data = self.get_block(index)
                take = min(remaining, len(data) - offset)
                require(take > 0, 'Unexpected ZIP read position.')
                chunks.append(data[offset:offset + take])
                self.position += take
                remaining -= take
            return b''.join(chunks)

        def close(self):
            self.session.close()
            self.blocks.clear()
            super().close()

    annotation_cache = local_root / 'extensions_annotation_cache'
    image_cache = local_root / 'extensions_source_images'
    annotation_cache.mkdir(parents=True, exist_ok=True)
    image_cache.mkdir(parents=True, exist_ok=True)
    range_file = None
    archive = None
    archive_mode = None
    annotation_provenance = {}
    image_provenance = []

    def open_archive():
        nonlocal range_file, archive, archive_mode
        if archive is None:
            candidates = [Path('/content/zerowaste-f-final.zip'), root / 'cache/zerowaste-f-final.zip']
            local = next((path for path in candidates if path.is_file() and path.stat().st_size == archive_size), None)
            if local is not None:
                archive = zipfile.ZipFile(local)
                archive_mode = 'existing_local_archive_selected_members'
            else:
                print('Reading only required source Train/Val members by bounded HTTP ranges.')
                range_file = HTTPRangeFile()
                archive = zipfile.ZipFile(range_file)
                archive_mode = 'bounded_http_ranges_selected_members'
        return archive

    def member_for(suffix):
        require(suffix.startswith(('train/', 'val/')) and not suffix.startswith('test/'),
                'Only original source Train/Val members are permitted.')
        choices = [item for item in open_archive().infolist() if not item.is_dir()
                   and (item.filename == suffix or item.filename.endswith('/' + suffix))
                   and '..' not in PurePosixPath(item.filename).parts]
        require(len(choices) == 1, 'Source ZIP member is not unique: ' + suffix)
        return choices[0]

    def check_member_bytes(data, info, expected_sha=None):
        require(len(data) == info.file_size and (zlib.crc32(data) & 0xffffffff) == info.CRC,
                'Retrieved/cached member size or CRC differs: ' + info.filename)
        value = digest_bytes(data)
        if expected_sha is not None:
            require(value == expected_sha, 'Historical member SHA256 differs: ' + info.filename)
        return value

    def save_sidecar(path, data):
        sidecar = Path(str(path) + '.source.json')
        encoded = (json.dumps(data, sort_keys=True, indent=2) + '\n').encode('utf-8')
        if sidecar.exists():
            old = json.loads(sidecar.read_text(encoding='utf-8'))
            for key in ['sha256', 'member', 'member_crc32', 'member_bytes', 'archive_url', 'archive_total_bytes']:
                require(old.get(key) == data[key], 'Existing cache provenance differs: ' + str(sidecar))
        else:
            write_new(sidecar, encoded)
        return sidecar

    def verify_cached(path, info, expected_sha=None):
        require(path.is_file(), 'Cache entry is not a file: ' + str(path))
        data = path.read_bytes()
        value = check_member_bytes(data, info, expected_sha)
        if expected_sha is None:
            sidecar = Path(str(path) + '.source.json')
            require(sidecar.is_file(), 'Unanchored cached Train/Val source requires its saved receipt: ' + str(path))
            old = json.loads(sidecar.read_text(encoding='utf-8'))
            require(old.get('sha256') == value and old.get('member') == info.filename
                    and old.get('member_crc32') == info.CRC and old.get('member_bytes') == info.file_size
                    and old.get('archive_url') == archive_url and old.get('archive_total_bytes') == archive_size,
                    'Existing source cache differs from its receipt: ' + str(path))
        return data, value

    try:
        coco_sources = {}
        for source_split in ['train', 'val']:
            info = member_for(source_split + '/labels.json')
            require(0 < info.file_size <= 256 * 1024**2, 'Unexpected COCO source annotation size.')
            path = annotation_cache / (source_split + '_labels.json')
            expected = expected_train_json if source_split == 'train' else None
            if path.exists():
                data, annotation_sha = verify_cached(path, info, expected)
                mode = 'existing_verified_annotation_cache'
            else:
                old_train_cache = local_root / 'failure_inputs_VAL_v1/source_train_annotations.json'
                if source_split == 'train' and old_train_cache.is_file():
                    data = old_train_cache.read_bytes()
                    mode = 'reused_previously_verified_train_json'
                else:
                    data = open_archive().read(info, pwd=password)
                    mode = 'retrieved_source_member'
                annotation_sha = check_member_bytes(data, info, expected)
                write_new(path, data)
            receipt = {'sha256': annotation_sha, 'member': info.filename, 'member_crc32': info.CRC,
                       'member_bytes': info.file_size, 'archive_url': archive_url,
                       'archive_total_bytes': archive_size}
            save_sidecar(path, receipt)
            annotation_provenance[source_split] = {**receipt, 'mode': mode, 'cache_path': str(path),
                'historical_sha256_anchor_available': source_split == 'train',
                'historical_anchor_note': ('Matches the saved original Train COCO JSON hash.' if source_split == 'train'
                                          else 'New source Val JSON hash recorded from a CRC-verified member; no historical JSON SHA anchor is claimed.')}
            coco = json.loads(data.decode('utf-8-sig'))
            images = {int(item['id']): item for item in coco['images']}
            categories = {int(item['id']): item['name'] for item in coco['categories']}
            require(len(images) == len(coco['images']) and set(categories.values()) == set(classes),
                    'COCO image identities or category names differ.')
            anns = defaultdict(list)
            for ann in coco['annotations']:
                anns[int(ann['image_id'])].append(ann)
            coco_sources[source_split] = {'images': images, 'categories': categories, 'annotations': anns}
            print('Verified source annotation JSON:', source_split)

        # This reproduces notebook cell10 followed by the file-order reading in cell51.
        pools = {'train': defaultdict(list), 'val': defaultdict(list)}
        image_metadata = {}
        pool_counts = {}
        for export_name in sorted(by_export):
            row = by_export[export_name]
            source_split = row['original_split']
            source = coco_sources[source_split]
            image_id = int(row['image_id'])
            require(image_id in source['images'], 'COCO image ID missing: ' + export_name)
            info = source['images'][image_id]
            require(PurePosixPath(info['file_name']).name == row['filename'], 'COCO filename differs: ' + export_name)
            width, height = int(info['width']), int(info['height'])
            require(width > 0 and height > 0, 'Invalid COCO image dimensions.')
            image_metadata[export_name] = {'width': width, 'height': height,
                                          'image_id': image_id, 'frozen_manifest_row': row}
            class_counts = Counter()
            for ann_ordinal, ann in enumerate(source['annotations'][image_id]):
                class_name = source['categories'][int(ann['category_id'])]
                class_id = classes.index(class_name)
                x, y, w, h = map(float, ann['bbox'])
                require(all(math.isfinite(value) for value in (x, y, w, h)) and w > 0 and h > 0,
                        'Invalid original COCO box: ' + export_name)
                x1, y1, x2, y2 = (max(0, min(width, x)), max(0, min(height, y)),
                                  max(0, min(width, x + w)), max(0, min(height, y + h)))
                require(x2 > x1 and y2 > y1, 'Empty clipped original COCO box.')
                xc, yc, bw, bh = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height
                line = f'{class_id} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}'
                decoded_class, xc, yc, bw, bh = map(float, line.split())
                require(int(decoded_class) == class_id, 'Converted class identity differs.')
                class_counts[class_name] += 1
                if bw > 0.003 and bh > 0.003:
                    pools[row['split']][class_id].append({
                        'export_name': export_name, 'filename': row['filename'],
                        'original_split': source_split, 'project_split': row['split'],
                        'image_id': image_id, 'width': width, 'height': height,
                        'class_id': class_id, 'class_name': class_name,
                        'annotation_id': int(ann['id']), 'annotation_ordinal_in_image': ann_ordinal,
                        'xywh_normalized': [xc, yc, bw, bh], 'converted_yolo_line': line,
                        'official_clipped_box_xyxy': [x1, y1, x2, y2],
                        'frozen_manifest_row': row,
                    })
            require(all(class_counts[name] == int(row[name]) for name in classes),
                    'COCO object counts differ from frozen manifest: ' + export_name)
        for split in ['train', 'val']:
            pool_counts[split] = {classes[c]: len(pools[split][c]) for c in range(4)}
            require(all(pools[split][c] for c in range(4)), 'A material pool is empty: ' + split)

        rng = np.random.default_rng(42)

        def choose_objects(split, per_class):
            chosen = []
            for class_id in range(len(classes)):
                pool = pools[split][class_id]
                indices = rng.choice(len(pool), size=min(per_class, len(pool)), replace=False)
                chosen.extend(dict(pool[int(index)]) for index in indices)
            return chosen

        gallery = choose_objects('train', 40)
        queries = choose_objects('val', 10)
        require(len(gallery) == 160 and len(queries) == 40, 'Reconstructed balanced cohort sizes differ.')
        require([record['class_id'] for record in gallery] == [int(value) for value in historical_labels]
                and [record['export_name'] for record in gallery] == historical_names,
                'Reconstructed gallery order/classes do not match the saved historical index; stop before image download.')
        print('Historical gallery order verified: all 160 source basenames and labels match.')

        operator_rows = []
        for number in [1, 71, 141, 211, 291]:
            found = [row for row in active_rows if row['split'] == 'val' and row['original_split'] == 'train'
                     and row['sequence'] == '07' and int(row['frame_number']) == number]
            require(len(found) == 1, 'Operator Validation selection differs.')
            operator_rows.append(found[0])
        selected_names = sorted({record['export_name'] for record in gallery + queries}
                                | {'train__' + row['filename'] for row in operator_rows})
        member_infos = {}
        for name in selected_names:
            row = by_export[name]
            require(row['split'] in ('train', 'val') and row['original_split'] in ('train', 'val'),
                    'Selected image is outside permitted project Train/Validation.')
            info = member_for(row['original_split'] + '/data/' + row['filename'])
            require(0 < info.file_size <= 12 * 1024**2, 'Unexpected selected PNG size.')
            member_infos[name] = info
        # Reading by header offset minimizes rereads in the bounded range cache.
        download_order = sorted(selected_names, key=lambda name: member_infos[name].header_offset)
        image_records = {}
        downloaded, reused, borrowed = 0, 0, 0
        print('Preparing unique source images:', len(selected_names))
        for ordinal, name in enumerate(download_order, 1):
            row = by_export[name]
            info = member_infos[name]
            target = image_cache / name
            expected_image_sha = None
            if row['split'] == 'val':
                require(name in inventory and inventory[name]['image_id'] == int(row['image_id']),
                        'Project Validation image identity differs from historical inventory.')
                expected_image_sha = inventory[name]['image_sha256']
            if target.exists():
                data, image_sha = verify_cached(target, info, expected_image_sha)
                mode = 'existing_verified_extension_image_cache'
                reused += 1
            else:
                old_val = local_root / 'failure_inputs_VAL_v1/images' / name
                if expected_image_sha is not None and old_val.is_file():
                    data = old_val.read_bytes()
                    mode = 'reused_historical_sha_verified_validation_cache'
                    borrowed += 1
                else:
                    data = open_archive().read(info, pwd=password)
                    mode = 'retrieved_source_member'
                    downloaded += 1
                image_sha = check_member_bytes(data, info, expected_image_sha)
                with Image.open(io.BytesIO(data)) as image:
                    expected_dimensions = (image_metadata[name]['width'], image_metadata[name]['height'])
                    require(image.format == 'PNG' and image.size == expected_dimensions,
                            'Original PNG dimensions/format differ: ' + name)
                    image.verify()
                write_new(target, data)
            with Image.open(io.BytesIO(data)) as image:
                require(image.format == 'PNG' and image.size == (image_metadata[name]['width'], image_metadata[name]['height']),
                        'Cached original PNG dimensions/format differ: ' + name)
                image.verify()
            cache_receipt = {'sha256': image_sha, 'member': info.filename, 'member_crc32': info.CRC,
                             'member_bytes': info.file_size, 'archive_url': archive_url,
                             'archive_total_bytes': archive_size}
            save_sidecar(target, cache_receipt)
            record = {'export_name': name, 'filename': row['filename'], 'path': str(target),
                      'image_sha256': image_sha, **image_metadata[name],
                      'original_split': row['original_split'], 'project_split': row['split']}
            image_records[name] = record
            image_provenance.append({**cache_receipt, 'export_name': name, 'project_split': row['split'],
                'original_split': row['original_split'], 'mode': mode,
                'historical_image_sha256_anchor_available': expected_image_sha is not None,
                'hash_scope': ('Matches the earlier 451-image Validation inventory.' if expected_image_sha is not None
                               else 'New image SHA256 recorded from a source ZIP member verified by CRC/size; no historical Train-image SHA is claimed.')})
            if ordinal % 10 == 0 or ordinal == len(download_order):
                print('Verified extension source images:', ordinal, '/', len(download_order))

        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        work_dir = local_root / ('extension_inputs_' + stamp)
        (work_dir / 'gallery').mkdir(parents=True, exist_ok=False)
        (work_dir / 'queries').mkdir(exist_ok=False)

        def make_crops(objects, directory, prefix):
            result = []
            for ordinal, original in enumerate(objects):
                record = dict(original)
                source_image = image_records[record['export_name']]
                record['path'] = source_image['path']
                record['image_sha256'] = source_image['image_sha256']
                xc, yc, bw, bh = record['xywh_normalized']
                width, height = record['width'], record['height']
                # Exact notebook cell51 crop_object arithmetic after 8-decimal readback.
                x1 = max(0, int((xc - bw / 2) * width))
                x2 = min(width, int(np.ceil((xc + bw / 2) * width)))
                y1 = max(0, int((yc - bh / 2) * height))
                y2 = min(height, int(np.ceil((yc + bh / 2) * height)))
                crop_box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]
                crop_file = directory + '/' + prefix + '_' + str(ordinal).zfill(3) + '.png'
                crop_path = work_dir / crop_file
                with Image.open(record['path']) as image:
                    rgb = image.convert('RGB')
                    require(rgb.size == (width, height), 'Source dimensions changed while cropping.')
                    crop = rgb.crop(tuple(crop_box))
                    with crop_path.open('xb') as stream:
                        crop.save(stream, format='PNG')
                record.update({'ordinal': ordinal, 'crop_file': crop_file, 'crop_path': str(crop_path),
                               'crop_box_xyxy': crop_box, 'crop_sha256': digest(crop_path),
                               'crop_width': crop_box[2] - crop_box[0], 'crop_height': crop_box[3] - crop_box[1]})
                result.append(record)
            return result

        gallery_records = make_crops(gallery, 'gallery', 'g')
        query_records = make_crops(queries, 'queries', 'q')
        operator_records = []
        for ordinal, row in enumerate(operator_rows):
            record = dict(image_records['train__' + row['filename']])
            record.update({'ordinal': ordinal, 'frame_number': int(row['frame_number']), 'source_sequence': '07'})
            operator_records.append(record)
        after = {name: digest(path) for name, path in protected.items()}
        require(before == after, 'A protected saved project reference changed during preparation.')
        transferred = range_file.transferred if range_file is not None else 0
        request_count = range_file.requests_count if range_file is not None else 0
        provenance = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'scope': 'Project Train gallery and Validation queries/operator images only; no Test data, inference or optimization.',
            'class_names_in_id_order': classes, 'manifest_sha256': expected_manifest,
            'original_index_path': str(original_index_path), 'historical_index_sha256': expected_index,
            'historical_gallery_order_check': {'source_basenames_match': True, 'class_ids_match': True,
                                             'objects': 160, 'before_any_png_download': True},
            'historical_order_limit': 'The saved index has source paths and labels, but no historical crop pixels, annotation IDs or boxes. Those are reconstructed from the source annotations and the recorded algorithm, not independently compared to unavailable historical crop files.',
            'selection_algorithm': 'Notebook cell10 clip/normalize bbox and format8dec; cell51 sorted export label order, original annotation order, strict w>0.003 and h>0.003; one NumPy default_rng(42), Train40perclass first, then Validation10perclass.',
            'sampling_rng': {'seed': 42, 'bit_generator': type(rng.bit_generator).__name__,
                             'gallery_then_queries_share_one_rng': True},
            'versions': {name: version(name) for name in ['numpy', 'Pillow', 'requests']},
            'pool_counts_after_small_box_filter': pool_counts,
            'gallery_size': 160, 'query_size': 40, 'operator_image_count': 5,
            'operator_selection': 'Frozen Validation source sequence07 frame indices1,71,141,211,291; fixed before model execution.',
            'annotation_sources': annotation_provenance, 'source_images': image_provenance,
            'archive_url': archive_url, 'archive_total_bytes': archive_size, 'archive_mode': archive_mode,
            'range_byte_budget': 768 * 1024**2, 'range_bytes_transferred_this_call': transferred,
            'range_requests_this_call': request_count, 'whole_archive_md5_recomputed_this_call': False,
            'unique_source_images': len(image_records), 'downloaded_images_this_call': downloaded,
            'reused_extension_image_cache': reused, 'borrowed_historical_validation_cache': borrowed,
            'source_train_images_have_new_sha_provenance': True,
            'source_val_annotation_sha_is_new_not_historical': True,
            'protected_sha256_before': before, 'protected_sha256_after': after,
            'protected_sources_unchanged': True,
            'crop_format': 'Lossless RGB PNG; exact recorded original crop_object integer/ceil arithmetic.',
        }
        bundle = {'gallery': gallery_records, 'queries': query_records, 'operator_images': operator_records,
                  'provenance': provenance, 'work_dir': str(work_dir), 'project_root': str(root),
                  'original_index_path': str(original_index_path), 'historical_bank_sha256': expected_index,
                  'source_image_cache': str(image_cache)}
        with (work_dir / 'cohort_provenance.json').open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(bundle, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write('\n')
        print('=== EXTENSION COHORTS READY ===')
        print('Gallery160 | Validation queries40 | operator images5 | unique source PNGs:', len(image_records))
        print('Partial source retrieval this call (MiB):', round(transferred / 1024**2, 2))
        print('Local lossless crop folder:', work_dir)
        return bundle
    finally:
        if archive is not None:
            archive.close()
        if range_file is not None:
            range_file.close()


"""One explicit Validation-only extension reproduction and HTTP workflow run."""


def run_extension_assist():
    from datetime import datetime, timezone
    from importlib.metadata import PackageNotFoundError, version
    from pathlib import Path
    import base64
    import csv
    import gc
    import hashlib
    import io
    import json
    import os
    import shutil
    import socket
    import subprocess
    import sys
    import time
    import zipfile

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024**2), b''):
                value.update(chunk)
        return value.hexdigest()

    def save_json(path, value):
        with Path(path).open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
            stream.write('\n')

    def installed(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None

    os.environ['WANDB_MODE'] = 'disabled'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    torch_before, vision_before = installed('torch'), installed('torchvision')
    require(torch_before and vision_before, 'Colab must have matching preinstalled PyTorch and torchvision.')
    required = {'ultralytics': '8.4.213', 'transformers': '5.22.1',
                'fastapi': '0.141.1'}
    packages = [name + '==' + target for name, target in required.items() if installed(name) != target]
    packages += [name for name in ['uvicorn', 'python-multipart', 'requests'] if installed(name) is None]
    if packages:
        print('Preparing missing packages; retaining the current PyTorch and torchvision...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch==' + torch_before, 'torchvision==' + vision_before, *packages],
                       check=True, timeout=360)
    require(installed('torch') == torch_before and installed('torchvision') == vision_before,
            'Core runtime changed unexpectedly. Stop before evaluation.')
    for name, target in required.items():
        require(installed(name) == target, 'Required runtime version differs: ' + name)

    import numpy as np
    import requests
    from PIL import Image
    from google.colab import files

    print('=== PREPARING ORIGINAL EXTENSION COHORTS ===')
    preparation = prepare_extension_cohorts()
    project = Path(preparation['project_root'])
    inputs = Path(preparation['work_dir'])
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output = project / 'evaluation' / ('EXTENSIONS_OPERATOR_VAL_' + stamp)
    output.mkdir(parents=True, exist_ok=False)
    deploy = output / 'deployment'
    assets = deploy / 'assets'
    assets.mkdir(parents=True, exist_ok=False)
    protected_relative = {
        'split_manifest': 'data_protocol/ef6b35d2ba7f/split_manifest.csv',
        'validation_inventory': 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json',
        'gallery_index': 'open_world/visual_search/gallery_index_train.npz',
        'visual_search_result': 'open_world/visual_search/comparison_VAL.json',
        'clip_result': 'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv',
        'service_source': 'deployment/http_service/service.py',
        'service_manifest': 'deployment/http_service/release_manifest.json',
        'http_detector': 'deployment/http_service/best_det.pt',
        'http_segmenter': 'deployment/http_service/best_seg.pt',
        'source_detector': 'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt',
        'source_segmenter': 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt',
        'test_protocol': 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json',
        'test_results': 'evaluation/FINAL_TEST_ONCE/results.json',
    }
    protected = {name: project / relative for name, relative in protected_relative.items()}
    before = {name: sha(path) for name, path in protected.items()}
    require(before == preparation['provenance']['protected_sha256_before'],
            'A protected reference changed after cohort preparation.')
    save_json(output / 'cohort_provenance.json', preparation)
    for directory in ['gallery', 'queries']:
        shutil.copytree(inputs / directory, assets / directory)
    gallery, queries = preparation['gallery'], preparation['queries']
    save_json(assets / 'gallery_meta.json', gallery)
    save_json(assets / 'query_meta.json', queries)
    code_dir = output / 'code'
    code_dir.mkdir()
    for name, source in _EXTENSION_SOURCES.items():
        with (code_dir / name).open('x', encoding='utf-8', newline='\n') as stream:
            stream.write(source)
    for name in ['extension_assist_models.py', 'extension_assist_service.py',
                 'extension_assist_start.py', 'operator_ui.html']:
        shutil.copyfile(code_dir / name, deploy / name)

    plan = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'val',
        'gallery': {'split': 'train', 'objects': 160, 'per_class': 40},
        'queries': {'split': 'val', 'objects': 40, 'per_class': 10},
        'cohort_provenance_sha256': sha(output / 'cohort_provenance.json'),
        'encoder': 'Frozen ImageNet ResNet18 IMAGENET1K_V1; CPU FP32',
        'clip_model': 'openai/clip-vit-base-patch32', 'prompts': list(PLAIN_PROMPTS),
        'model_or_prompt_selection_during_this_run': False,
        'historical_prompt_choice': 'Plain prompts already selected using earlier Validation comparison; conveyor prompt result remains historical.',
        'histogram_baseline': 'PIL RGB resize64x64, eight bins per RGB channel, concatenate24, L2 normalize.',
        'retrieval_ranking': 'descending numpy argsort of cosine similarity, as in the recorded notebook',
        'histogram_classification_tie_policy': 'numpy argmax first index, as in the recorded notebook',
        'gallery_reconstruction_check': 'All160 name/class rows; then minimum paired old/new embedding cosine >=0.999 before query evaluation.',
        'operator_images': [item['export_name'] for item in preparation['operator_images']],
        'operator_crop_selection': 'First requires_review object, otherwise first object; none if no detections.',
        'crop_rule': 'Original response rounded bbox coordinates, floor starts/ceil ends, clipped to image.',
        'failure_selection': 'First up to3 wrong top1 predictions per extension in fixed query order; explicitly report zero if absent.',
        'metric_scope': '40 fixed ground-truth object crops; no full-image detection accuracy, physical grasp, unknown-class or whole-system accuracy claim.',
        'http_scope': 'Five actual Validation requests and HTML/reference route checks; functional proof, not a latency benchmark.',
        'test_images_or_test_model_calls': 0, 'optimizer_steps': 0,
        'original_project_files_sha256': before,
    }
    save_json(output / 'plan_before_model_loading.json', plan)
    print('Loading official frozen assistance models on CPU; missing weights download once (~650 MB total).')
    engine = OperatorAssistModels()
    resources = engine.runtime_metadata()
    protocol = {**plan, 'frozen_before_any_new_model_inference_at_utc': datetime.now(timezone.utc).isoformat(),
                'model_resources': resources,
                'source_files_sha256': {name: sha(code_dir / name) for name in _EXTENSION_SOURCES}}
    save_json(output / 'frozen_operator_VAL_protocol.json', protocol)
    protocol_sha = sha(output / 'frozen_operator_VAL_protocol.json')

    def load_crops(records):
        result = []
        for record in records:
            path = assets / record['crop_file']
            require(sha(path) == record['crop_sha256'], 'Crop changed: ' + record['crop_file'])
            with Image.open(path) as image:
                result.append(image.convert('RGB'))
        return result

    gallery_images, query_images = load_crops(gallery), load_crops(queries)
    gy = np.asarray([item['class_id'] for item in gallery], dtype=np.int64)
    qy = np.asarray([item['class_id'] for item in queries], dtype=np.int64)
    print('Encoding160 Train references and comparing against the saved historical bank...')
    g = engine.embed_resnet(gallery_images)
    with np.load(preparation['original_index_path'], allow_pickle=False) as old:
        old_g = old['embeddings'].copy().astype(np.float32)
    old_g = unit(old_g)
    paired_cosine = np.sum(g * old_g, axis=1)
    bank_check = {
        'count': 160, 'source_basenames_and_classes_aligned': True,
        'paired_cosines': paired_cosine.tolist(), 'minimum_paired_cosine': float(paired_cosine.min()),
        'maximum_paired_cosine': float(paired_cosine.max()),
        'max_absolute_feature_difference': float(np.abs(g - old_g).max()),
        'acceptance_minimum_cosine_fixed_before_inference': 0.999,
        'historical_bank_sha256': preparation['historical_bank_sha256'],
        'passed': bool(np.all(paired_cosine >= 0.999)),
        'interpretation': 'Numerical compatibility check for reconstructed reference crops, not byte-identical model output proof.',
    }
    save_json(output / 'historical_bank_reconstruction_check.json', bank_check)
    require(bank_check['passed'], 'Reconstructed reference features differ from the historical bank. Stop and share this output; no query/HTTP result is claimed.')
    print('Historical reference feature check passed. Evaluating40 fixed Validation crops...')
    q = engine.embed_resnet(query_images)
    gh, qh = rgb_hist(gallery_images), rgb_hist(query_images)
    resnet_metrics, rankings = retrieval_metrics(g, q, gy, qy)
    histogram_retrieval, histogram_rankings = retrieval_metrics(gh, qh, gy, qy)
    hist_predictions = gy[(qh @ gh.T).argmax(axis=1)]
    logits = engine.score_clip(query_images)
    clip_predictions = logits.argmax(axis=1)
    clip_metrics = classification_metrics(qy, clip_predictions)
    hist_classification = classification_metrics(qy, hist_predictions)
    np.savez_compressed(assets / 'gallery_index.npz', embeddings=g, histograms=gh, labels=gy)
    np.savez_compressed(output / 'validation_arrays.npz', query_embeddings=q,
                        query_histograms=qh, clip_logits=logits, query_labels=qy,
                        resnet_rankings=rankings, histogram_rankings=histogram_rankings,
                        histogram_class_predictions=hist_predictions)
    historical = output / 'historical_references'
    historical.mkdir()
    shutil.copyfile(protected['visual_search_result'], historical / 'visual_search_comparison_VAL.json')
    shutil.copyfile(protected['clip_result'], historical / 'CLIP_vs_histogram_VAL.csv')
    comparison = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'val',
        'gallery_objects': 160, 'query_objects': 40, 'class_names': list(CLASS_NAMES),
        'visual_search_resnet': resnet_metrics, 'visual_search_histogram': histogram_retrieval,
        'clip_plain': clip_metrics, 'histogram_classification_baseline': hist_classification,
        'historical_results_replaced': False, 'protocol_sha256': protocol_sha,
        'limits': ['Fixed small balanced ground-truth crop cohort, not a full image or system accuracy evaluation.',
                   'Reconstructed evaluation in the recorded current CPU runtime, not the original historical inference event.',
                   'Similarities and four-way softmax are not calibrated confidence or unknown-material detection.'],
    }
    save_json(output / 'comparison_VAL.json', comparison)
    query_rows = []
    for index, item in enumerate(queries):
        top5 = [int(value) for value in rankings[index, :5]]
        query_rows.append({'query_index': index, 'query_crop_file': item['crop_file'],
                           'source_export_name': item['export_name'], 'annotation_id': item['annotation_id'],
                           'ground_truth_class_id': int(qy[index]),
                           'resnet_top1_class_id': int(gy[top5[0]]),
                           'resnet_top5_gallery_indices': top5,
                           'resnet_top5_cosine_similarities': [float(q[index] @ g[value]) for value in top5],
                           'clip_logits': logits[index].tolist(), 'clip_class_id': int(clip_predictions[index]),
                           'histogram_class_id': int(hist_predictions[index])})
    save_json(output / 'per_query_VAL.json', query_rows)
    failure_metadata = render_extension_failures(assets, gallery, queries, rankings, logits,
                                                  hist_predictions, output / 'failure_examples')
    save_json(output / 'failure_examples_index.json', failure_metadata)
    print(json.dumps(comparison, indent=2))

    asset_hashes = {path.relative_to(assets).as_posix(): sha(path)
                    for path in sorted(assets.rglob('*')) if path.is_file()}
    base_config = json.loads(protected['service_manifest'].read_text(encoding='utf-8'))
    require((base_config['confidence'], base_config['nms_iou'], base_config['imgsz'], base_config['device'])
            == (0.25, 0.5, 640, 'cpu'), 'Frozen original HTTP settings differ.')
    assist_manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'original_service_path': str(protected['service_source']),
        'original_service_sha256': before['service_source'],
        'original_release_manifest_sha256': before['service_manifest'],
        'original_model_mapping': base_config['weights'], 'class_names': base_config['class_names'],
        'clip_revision': resources['clip_revision'], 'model_resources': resources,
        'assets_sha256': asset_hashes, 'protocol_sha256': protocol_sha,
        'role': 'Operator assistance only; no automatic replacement of material label or rejection threshold.',
    }
    save_json(deploy / 'assist_manifest.json', assist_manifest)
    # Free parent weights before the independent server process loads them.
    del engine, gallery_images, query_images
    gc.collect()
    with socket.socket() as listener:
        listener.bind(('127.0.0.1', 0))
        port = listener.getsockname()[1]
    env = os.environ.copy()
    env.update({'WANDB_MODE': 'disabled', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1',
                'TOKENIZERS_PARALLELISM': 'false', 'HF_HUB_DISABLE_TELEMETRY': '1'})
    logfile = (output / 'assist_server.log').open('x', encoding='utf-8')
    process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'extension_assist_service:app',
                                '--host', '127.0.0.1', '--port', str(port), '--workers', '1'],
                               cwd=deploy, env=env, stdout=logfile, stderr=subprocess.STDOUT)
    session = requests.Session()
    session.trust_env = False
    url = 'http://127.0.0.1:' + str(port)
    cases = []
    try:
        deadline = time.monotonic() + 240
        print('Starting the local operator assistance service...')
        while True:
            require(process.poll() is None, 'Assistance server stopped; inspect the saved assist_server.log.')
            try:
                health_response = session.get(url + '/assist-health', timeout=3)
                if health_response.status_code == 200:
                    break
            except requests.RequestException:
                pass
            require(time.monotonic() < deadline, 'Assistance server startup timed out; inspect assist_server.log.')
            time.sleep(2)
        health = health_response.json()
        require(health['gallery_items'] == 160 and health['original_predict_route_retained'] is True,
                'Assistance health contract differs.')
        save_json(output / 'assist_health.json', health)
        response_dir = output / 'http_responses'
        response_dir.mkdir()
        for ordinal, image in enumerate(preparation['operator_images']):
            raw = Path(image['path']).read_bytes()
            require(hashlib.sha256(raw).hexdigest() == image['image_sha256'], 'Operator input image bytes changed.')
            started = time.perf_counter()
            response = session.post(url + '/assist', files={'file': (image['export_name'], raw, 'image/png')}, timeout=240)
            seconds = time.perf_counter() - started
            require(response.status_code == 200, 'Operator request failed: ' + str(response.status_code) + ' ' + response.text[:500])
            payload = response.json()
            require(payload['automatic_material_override'] is False and payload['ground_truth_accuracy_measured'] is False,
                    'Unexpected automatic override or accuracy claim.')
            require(payload['base_runtime']['detector_device'] == 'cpu'
                    and payload['base_runtime']['segmenter_device'] == 'cpu'
                    and not payload['base_runtime']['detector_fp16']
                    and not payload['base_runtime']['segmenter_fp16'], 'Base service runtime differs from CPU FP32.')
            with (response_dir / ('case_' + str(ordinal).zfill(2) + '.json')).open('xb') as stream:
                stream.write(response.content)
            cases.append({'case_index': ordinal, 'export_name': image['export_name'],
                          'image_sha256': image['image_sha256'], 'http_status': response.status_code,
                          'response_sha256': hashlib.sha256(response.content).hexdigest(),
                          'client_seconds_observed_functional_request_not_benchmark': seconds, 'payload': payload})
            print('Actual /assist Validation request', ordinal + 1, '/5:', payload['status'])
        page = session.get(url + '/operator', timeout=20)
        require(page.status_code == 200 and page.content == (deploy / 'operator_ui.html').read_bytes(),
                'Operator HTML route did not return the saved UI.')
        reference = session.get(url + '/assist-gallery/0', timeout=20)
        require(reference.status_code == 200 and hashlib.sha256(reference.content).hexdigest() == asset_hashes['gallery/g_000.png'],
                'Reference image route bytes differ.')
        routes = {'operator_html_status': page.status_code,
                  'operator_html_sha256': hashlib.sha256(page.content).hexdigest(),
                  'gallery_0_status': reference.status_code,
                  'gallery_0_sha256': hashlib.sha256(reference.content).hexdigest(),
                  'scope': 'Actual HTTP route checks; UI source was checked, but no human-operated browser review is claimed.'}
        save_json(output / 'operator_routes_check.json', routes)
    finally:
        session.close()
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        logfile.close()
    save_json(output / 'operator_cases.json', cases)
    operator_metadata = render_operator_cases(cases, gallery, assets, output / 'operator_examples')
    save_json(output / 'operator_examples_index.json', operator_metadata)
    after = {name: sha(path) for name, path in protected.items()}
    require(before == after, 'Protected original project files changed during the extension run.')
    require(sha(output / 'frozen_operator_VAL_protocol.json') == protocol_sha, 'Frozen protocol changed.')
    save_json(output / 'protected_original_files.json', {
        'before': before, 'after': after, 'unchanged': True,
        'paths': {name: str(path) for name, path in protected.items()},
        'test_model_calls': 0, 'training_or_optimizer_steps': 0})
    requirements = {name: installed(name) for name in ['torch', 'torchvision', 'numpy', 'Pillow', 'requests',
                    'ultralytics', 'transformers', 'huggingface-hub', 'tokenizers', 'fastapi', 'uvicorn', 'python-multipart']}
    save_json(deploy / 'runtime_versions.json', requirements)
    restart_cell = '''# Run this saved cell after mounting the same Drive and installing the recorded dependencies.
from pathlib import Path
import os, socket, subprocess, sys, time
import requests
from google.colab import drive, output
drive.mount('/content/drive')
DEPLOYMENT = Path(__DEPLOYMENT_PATH__)
with socket.socket() as listener:
    listener.bind(('127.0.0.1', 0))
    port = listener.getsockname()[1]
env = os.environ.copy()
env.update({'WANDB_MODE':'disabled', 'OMP_NUM_THREADS':'1', 'MKL_NUM_THREADS':'1'})
log = (DEPLOYMENT / ('operator_ui_' + str(time.time_ns()) + '.log')).open('x')
server = subprocess.Popen([sys.executable, str(DEPLOYMENT / 'extension_assist_start.py'), '--port', str(port)],
                          cwd=DEPLOYMENT, env=env, stdout=log, stderr=subprocess.STDOUT)
client = requests.Session()
client.trust_env = False
deadline = time.monotonic() + 900
try:
    while True:
        if server.poll() is not None:
            raise RuntimeError('Operator service stopped. Inspect the new log in the deployment folder.')
        try:
            if client.get('http://127.0.0.1:' + str(port) + '/assist-health', timeout=3).status_code == 200:
                break
        except requests.RequestException:
            pass
        if time.monotonic() > deadline:
            server.terminate()
            raise RuntimeError('Operator service startup timed out; inspect the new log.')
        time.sleep(2)
finally:
    client.close()
    log.close()
print('Local service is running for this Colab session. Closing the runtime stops it.')
output.serve_kernel_port_as_iframe(port, path='/operator', height='860')
# To stop only this process when finished: server.terminate()
'''.replace('__DEPLOYMENT_PATH__', repr(str(deploy)))
    compile(restart_cell, 'START_OPERATOR_UI_COLAB.txt', 'exec')
    with (deploy / 'START_OPERATOR_UI_COLAB.txt').open('x', encoding='utf-8') as stream:
        stream.write(restart_cell)
    readme = '''# Saved operator assistance workflow

This new directory is separate from the original service and recorded results.
The local server was started for five real Validation requests and then stopped.
No external hosting or completed human review is claimed.

To run again in a Colab session:
1. Mount the same Google Drive account containing Project9_Phase2.
2. Keep a matching preinstalled torch/torchvision pair. Install ultralytics==8.4.213,
   transformers==5.22.1, fastapi==0.141.1, uvicorn and python-multipart if missing.
   The exact observed versions are in runtime_versions.json.
3. Copy START_OPERATOR_UI_COLAB.txt into a new Colab cell and run it.
   It starts the saved service in the background, waits for its health route,
   and embeds /operator in the notebook using the official Colab port helper.
   The official assistance model files download only if not cached. Their pinned
   revision and full byte hashes must match assist_manifest.json before serving.
4. Choose a conveyor image and click Review an item, or call POST /assist with multipart
   file=<image>. GET /assist-health and /assist-gallery/0 verify service/reference access.
   Original POST /predict remains available and retains its original output.
   The embedded browser interaction itself has not been audited by the saved HTTP checks.

For a terminal environment: python extension_assist_start.py --port 8008
and open http://127.0.0.1:8008/operator while that process runs.
Colab helper source: https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py

Original YOLO checkpoints, service.py and release_manifest.json must remain at
the Drive path recorded in assist_manifest.json. They are not included in this ZIP.
Four material suggestions are advisory. No automatic relabelling, confidence
threshold selection, unknown-class detection or physical grasp claim is made.
Saved case HTML/cards are reproducible observations, not a human approval log.
All new crop metrics are Validation only and kept separate from historical metrics.
The model binaries, full source PNGs, source annotation JSON and Test images are
excluded from this evidence ZIP. Train reference crops and Val query crops are included.
Do not publish these dataset crops or private project paths to a public repository.
'''
    with (deploy / 'README_RUN.txt').open('x', encoding='utf-8') as stream:
        stream.write(readme)
    summary = {'created_at_utc': datetime.now(timezone.utc).isoformat(),
               'status': 'completed_validation_and_functional_HTTP_workflow',
               'comparison': comparison, 'gallery_reconstruction_passed': True,
               'failure_examples': failure_metadata, 'operator_examples': operator_metadata,
               'operator_requests': len(cases), 'operator_requests_http200': len(cases),
               'operator_requests_with_detections': sum(case['payload']['status'] == 'assistance_ready' for case in cases),
               'server_stopped_after_recording': True, 'public_hosting_claimed': False,
               'human_review_completed_claimed': False, 'original_project_files_unchanged': True,
               'test_model_calls': 0, 'optimizer_steps': 0}
    save_json(output / 'summary.json', summary)
    inventory = [{'file': path.relative_to(output).as_posix(), 'bytes': path.stat().st_size, 'sha256': sha(path)}
                 for path in sorted(output.rglob('*')) if path.is_file()]
    save_json(output / 'collection_manifest.json', {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'files': inventory})
    archive_path = project / ('Project9_EXTENSIONS_OPERATOR_VAL_' + stamp + '.zip')
    with zipfile.ZipFile(archive_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if path.is_file():
                archive.write(path, arcname=path.relative_to(output).as_posix())
    with zipfile.ZipFile(archive_path) as archive:
        require(archive.testzip() is None, 'Evidence ZIP failed CRC verification.')
        for item in inventory:
            data = archive.read(item['file'])
            require(len(data) == item['bytes'] and hashlib.sha256(data).hexdigest() == item['sha256'],
                    'Packaged evidence hash differs: ' + item['file'])
    print('=== EXTENSIONS COMPLETE: VALIDATION + ACTUAL HTTP WORKFLOW ===')
    print('Original files unchanged. No training, optimizer step or Test evaluation.')
    print('Evidence directory:', output)
    print('ZIP:', archive_path)
    print('New crop metrics are separate from the original recorded results.')
    files.download(str(archive_path))
    return {'output_dir': str(output), 'zip': str(archive_path), 'summary': summary}


_EXTENSION_RUN_RESULT = run_extension_assist()




In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# خلية واحدة جديدة في Colab: اختبار الإضافتين على عينة Test ثابتة مرة واحدة.
# حتى50عنصرًا لكل مادة، دون اختيار إعدادات أو إعادة Test لـM4/M5.
# اتركها تكمل. إذا ظهر خطأ أرسل المخرجات؛ لا تحذف مجلد Test ولا تغيّر البروتوكول.
# إذا اكتملت سابقًا فإن إعادة تشغيل الخلية تنزّل الحزمة المحفوظة فقط.

_EXT_TEST_SOURCES = {'extension_assist_models.py': '"""CPU models and numeric helpers for the optional operator-assistance stage.\n\nImporting this module performs no model loading, downloads or inference.\nModel construction is explicit; all predictions are advisory and do not alter\nthe original detector/segmenter response or introduce a rejection threshold.\n"""\n\nCLASS_NAMES = (\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\')\nCLIP_MATERIAL_NAMES = (\'cardboard\', \'metal\', \'rigid plastic\', \'soft plastic\')\nPLAIN_PROMPTS = tuple(\'a photo of \' + name for name in CLIP_MATERIAL_NAMES)\nCLIP_MODEL_ID = \'openai/clip-vit-base-patch32\'\n\n\ndef _assist_require(condition, message):\n    if not condition:\n        raise RuntimeError(message)\n\n\ndef _assist_sha256(path):\n    from pathlib import Path\n    import hashlib\n    value = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b\'\'):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef unit(values):\n    """The original float32 L2 normalization, including its 1e-10 guard."""\n    import numpy as np\n    array = np.asarray(values, dtype=np.float32)\n    _assist_require(array.ndim == 2 and np.isfinite(array).all(), \'Expected finite 2D features.\')\n    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)\n\n\ndef rgb_hist(crops):\n    """Original 24-bin RGB histogram baseline; crops must be PIL images."""\n    import numpy as np\n    rows = []\n    for crop in crops:\n        array = np.asarray(crop.convert(\'RGB\').resize((64, 64)), dtype=np.uint8)\n        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,\n                                                range=(0, 256))[0] for channel in range(3)]))\n    _assist_require(bool(rows), \'At least one crop is required.\')\n    return unit(np.asarray(rows, dtype=np.float32))\n\n\ndef classification_metrics(true_labels, predicted_labels):\n    """Four-material accuracy and macro metrics with zero-division set to zero."""\n    import numpy as np\n    truth = np.asarray(true_labels)\n    prediction = np.asarray(predicted_labels)\n    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape\n                    and truth.size > 0, \'Class arrays must be nonempty, aligned 1D arrays.\')\n    _assist_require(np.issubdtype(truth.dtype, np.integer)\n                    and np.issubdtype(prediction.dtype, np.integer), \'Class IDs must be integers.\')\n    _assist_require(((truth >= 0) & (truth < 4)).all()\n                    and ((prediction >= 0) & (prediction < 4)).all(), \'Unknown material class ID.\')\n    matrix = np.zeros((4, 4), dtype=np.int64)\n    np.add.at(matrix, (truth, prediction), 1)\n    tp = np.diag(matrix).astype(np.float64)\n    predicted_support = matrix.sum(axis=0)\n    support = matrix.sum(axis=1)\n    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)\n    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)\n    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)\n    return {\n        \'accuracy\': float(np.mean(truth == prediction)), \'macro_f1\': float(f1.mean()),\n        \'macro_precision\': float(precision.mean()), \'macro_recall\': float(recall.mean()),\n        \'count\': int(truth.size), \'confusion_matrix_true_rows_predicted_columns\': matrix.tolist(),\n        \'per_class\': [{\'class_id\': index, \'material\': name, \'support\': int(support[index]),\n                       \'precision\': float(precision[index]), \'recall\': float(recall[index]),\n                       \'f1\': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],\n    }\n\n\ndef retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):\n    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""\n    import numpy as np\n    gallery = np.asarray(gallery_features, dtype=np.float32)\n    queries = np.asarray(query_features, dtype=np.float32)\n    gallery_y = np.asarray(gallery_labels)\n    query_y = np.asarray(query_labels)\n    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]\n                    and len(gallery) >= 5 and len(queries) > 0, \'Invalid gallery/query feature shape.\')\n    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)\n                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), \'Misaligned or nonfinite retrieval inputs.\')\n    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]\n    top1 = gallery_y[rankings[:, 0]]\n    top5 = gallery_y[rankings[:, :5]]\n    return ({\'top1_accuracy\': float(np.mean(top1 == query_y)),\n             \'hit_at_5\': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),\n             \'precision_at_5\': float(np.mean(top5 == query_y[:, None]))}, rankings)\n\n\nclass OperatorAssistModels:\n    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.\n\n    ``assets_dir`` may be None while building a new gallery. Then call\n    ``load_gallery`` after writing these files:\n\n    - gallery_index.npz: float32 embeddings [160,512], float32 histograms\n      [160,24], int64 labels [160].\n    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,\n      each with class_id and crop_file="gallery/g_000.png" etc.\n    - the corresponding 160 PNG reference crops.\n\n    local_only=True requires the ResNet checkpoint and the explicit immutable\n    CLIP revision to be present already; it performs no network request.\n    """\n\n    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):\n        from pathlib import Path\n        from importlib.metadata import version\n        import re\n        import torch\n        import torch.nn as nn\n        from torchvision.models import resnet18, ResNet18_Weights\n        from transformers import CLIPModel, CLIPProcessor\n        from huggingface_hub import hf_hub_download, model_info\n\n        self.torch = torch\n        self.assets_dir = None\n        self.gallery_embeddings = None\n        self.gallery_histograms = None\n        self.gallery_labels = None\n        self.gallery_meta = None\n        self.local_only = bool(local_only)\n        torch.set_num_threads(1)\n        try:\n            torch.set_num_interop_threads(1)\n        except RuntimeError:\n            _assist_require(torch.get_num_interop_threads() == 1,\n                            \'Torch inter-op threads are already initialized differently; use a fresh process.\')\n\n        weights = ResNet18_Weights.IMAGENET1K_V1\n        self.resnet_transform = weights.transforms()\n        resnet_file = Path(torch.hub.get_dir()) / \'checkpoints/resnet18-f37072fd.pth\'\n        if self.local_only:\n            _assist_require(resnet_file.is_file(), \'Cached ResNet18 checkpoint is required for local-only serving.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'Cached ResNet18 checkpoint checksum differs.\')\n            encoder = resnet18(weights=None)\n            state = torch.load(resnet_file, map_location=\'cpu\', weights_only=True)\n            encoder.load_state_dict(state, strict=True)\n            del state\n        else:\n            encoder = resnet18(weights=weights, progress=False)\n            _assist_require(resnet_file.is_file(), \'Downloaded ResNet18 checkpoint was not found at its standard cache path.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'ResNet18 checkpoint checksum differs from its official filename prefix.\')\n        encoder.fc = nn.Identity()\n        self.encoder = encoder.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.encoder.parameters():\n            parameter.requires_grad_(False)\n\n        hub_info = None\n        if clip_revision is None:\n            _assist_require(not self.local_only, \'An explicit cached CLIP revision is required in local-only mode.\')\n            hub_info = model_info(CLIP_MODEL_ID, revision=\'main\', files_metadata=True, token=False)\n            clip_revision = hub_info.sha\n        _assist_require(isinstance(clip_revision, str)\n                        and re.fullmatch(r\'[0-9a-f]{40}|[0-9a-f]{64}\', clip_revision) is not None,\n                        \'CLIP revision must be an immutable Hub commit hash.\')\n        self.clip_revision = clip_revision\n        common = {\'revision\': self.clip_revision, \'local_files_only\': self.local_only, \'token\': False}\n        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,\n                                             trust_remote_code=False, **common)\n        self.clip = self.clip.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.clip.parameters():\n            parameter.requires_grad_(False)\n        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,\n                                                       trust_remote_code=False, **common)\n        # The loader above explicitly selects the bin checkpoint. This cache-only\n        # lookup records its actual bytes without triggering a second download.\n        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename=\'pytorch_model.bin\',\n                                        revision=self.clip_revision, local_files_only=True, token=False))\n        clip_sha = _assist_sha256(clip_file)\n        expected_clip_sha = None\n        if hub_info is not None:\n            for sibling in hub_info.siblings or []:\n                if sibling.rfilename == \'pytorch_model.bin\':\n                    lfs = getattr(sibling, \'lfs\', None)\n                    expected_clip_sha = (lfs.get(\'sha256\') if isinstance(lfs, dict)\n                                         else getattr(lfs, \'sha256\', None))\n                    break\n        if expected_clip_sha is not None:\n            _assist_require(clip_sha == expected_clip_sha, \'CLIP binary differs from the Hub LFS SHA256.\')\n        config_commit = getattr(self.clip.config, \'_commit_hash\', None)\n        if config_commit is not None:\n            _assist_require(config_commit == self.clip_revision, \'Loaded CLIP configuration revision differs.\')\n        self._provenance = {\n            \'resnet_weights\': \'ResNet18_Weights.IMAGENET1K_V1\',\n            \'resnet_weights_url\': weights.url, \'resnet_checkpoint_path\': str(resnet_file),\n            \'resnet_checkpoint_sha256\': resnet_sha, \'resnet_transform\': repr(self.resnet_transform),\n            \'clip_model_id\': CLIP_MODEL_ID, \'clip_revision\': self.clip_revision,\n            \'clip_config_commit_hash\': config_commit, \'clip_checkpoint_path\': str(clip_file),\n            \'clip_checkpoint_sha256\': clip_sha, \'clip_hub_lfs_sha256\': expected_clip_sha,\n            \'clip_checkpoint_url\': \'https://huggingface.co/\' + CLIP_MODEL_ID + \'/resolve/\'\n                                   + self.clip_revision + \'/pytorch_model.bin\',\n            \'clip_processor_class\': type(self.processor).__name__,\n            \'clip_image_processor_class\': type(self.processor.image_processor).__name__,\n            \'clip_image_processor_config\': self.processor.image_processor.to_dict(),\n            \'clip_attention_implementation\': getattr(self.clip.config, \'_attn_implementation\', None),\n            \'versions\': {name: version(name) for name in [\'torch\', \'torchvision\', \'transformers\',\n                                                         \'huggingface-hub\', \'numpy\', \'Pillow\']},\n        }\n        self._assert_cpu_frozen()\n        if assets_dir is not None:\n            self.load_gallery(assets_dir)\n\n    def _assert_cpu_frozen(self):\n        for name, model in [(\'resnet\', self.encoder), (\'clip\', self.clip)]:\n            _assist_require(model.training is False, name + \' must be in eval mode.\')\n            _assist_require(all(parameter.device.type == \'cpu\' and not parameter.requires_grad\n                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)\n                                for parameter in model.parameters()), name + \' must be frozen CPU FP32.\')\n\n    def embed_resnet(self, crops, batch=16):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid ResNet crop batch.\')\n        self._assert_cpu_frozen()\n        vectors = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                tensor = self.torch.stack([self.resnet_transform(crop.convert(\'RGB\'))\n                                          for crop in crops[start:start + batch]])\n                output = self.encoder(tensor.to(device=\'cpu\', dtype=self.torch.float32))\n                vectors.append(output.detach().cpu().numpy())\n        array = np.concatenate(vectors).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 512), \'Unexpected ResNet feature dimensions.\')\n        return unit(array)\n\n    def score_clip(self, crops, batch=8):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid CLIP crop batch.\')\n        self._assert_cpu_frozen()\n        logits = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                inputs = self.processor(text=list(PLAIN_PROMPTS),\n                                        images=[crop.convert(\'RGB\') for crop in crops[start:start + batch]],\n                                        return_tensors=\'pt\', padding=True)\n                inputs = {key: value.to(device=\'cpu\', dtype=self.torch.float32)\n                          if key == \'pixel_values\' else value.to(\'cpu\') for key, value in inputs.items()}\n                output = self.clip(**inputs)\n                logits.append(output.logits_per_image.detach().cpu().numpy())\n        array = np.concatenate(logits).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), \'Unexpected or nonfinite CLIP logits.\')\n        return array\n\n    def rgb_hist(self, crops):\n        return rgb_hist(crops)\n\n    def load_gallery(self, assets_dir):\n        from pathlib import Path, PurePosixPath\n        import json\n        import numpy as np\n        directory = Path(assets_dir).resolve()\n        with np.load(directory / \'gallery_index.npz\', allow_pickle=False) as bank:\n            _assist_require(set(bank.files) == {\'embeddings\', \'histograms\', \'labels\'}, \'Unexpected assistance gallery keys.\')\n            embeddings = bank[\'embeddings\'].copy()\n            histograms = bank[\'histograms\'].copy()\n            labels = bank[\'labels\'].copy()\n        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32\n                        and histograms.shape == (160, 24) and histograms.dtype == np.float32\n                        and labels.shape == (160,) and labels.dtype == np.int64,\n                        \'Assistance gallery array shapes/dtypes differ.\')\n        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()\n                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),\n                        \'Assistance gallery features/classes differ.\')\n        for name, values in [(\'embedding\', embeddings), (\'histogram\', histograms)]:\n            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),\n                            \'Unnormalized gallery \' + name + \' features.\')\n        metadata = json.loads((directory / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n        _assist_require(isinstance(metadata, list) and len(metadata) == 160, \'Expected 160 gallery metadata records.\')\n        for index, item in enumerate(metadata):\n            expected_file = \'gallery/g_\' + str(index).zfill(3) + \'.png\'\n            _assist_require(isinstance(item, dict) and item.get(\'class_id\') == int(labels[index])\n                            and item.get(\'crop_file\') == expected_file, \'Gallery metadata/array alignment differs.\')\n            crop = PurePosixPath(item[\'crop_file\'])\n            path = (directory / crop).resolve()\n            _assist_require(not crop.is_absolute() and \'..\' not in crop.parts\n                            and path.is_relative_to(directory) and path.is_file(), \'Missing or unsafe reference crop.\')\n        self.assets_dir = directory\n        self.gallery_embeddings = embeddings\n        self.gallery_histograms = histograms\n        self.gallery_labels = labels\n        self.gallery_meta = metadata\n        return self\n\n    def query_crop(self, crop):\n        import numpy as np\n        _assist_require(self.gallery_embeddings is not None, \'Load the assistance gallery first.\')\n        feature = self.embed_resnet([crop], batch=1)\n        similarities = (feature @ self.gallery_embeddings.T)[0]\n        ranking = similarities.argsort()[::-1]\n        references = []\n        for raw_index in ranking[:5]:\n            index = int(raw_index)\n            class_id = int(self.gallery_labels[index])\n            references.append({\'gallery_index\': index, \'class_id\': class_id,\n                               \'material\': CLASS_NAMES[class_id], \'similarity\': float(similarities[index]),\n                               \'crop_file\': self.gallery_meta[index][\'crop_file\']})\n        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)\n        relative = np.exp(clip_logits - clip_logits.max())\n        relative /= relative.sum()\n        clip_class = int(clip_logits.argmax())\n        histogram = rgb_hist([crop])\n        hist_similarities = (histogram @ self.gallery_histograms.T)[0]\n        hist_index = int(hist_similarities.argmax())\n        hist_class = int(self.gallery_labels[hist_index])\n        return {\n            \'visual_search\': {\'predicted_class_id\': references[0][\'class_id\'],\n                              \'predicted_material\': references[0][\'material\'], \'references\': references,\n                              \'score_type\': \'Cosine similarity to frozen Train reference crops; not calibrated confidence.\'},\n            \'clip\': {\'predicted_class_id\': clip_class, \'predicted_material\': CLASS_NAMES[clip_class],\n                     \'logits\': clip_logits.tolist(), \'relative_scores\': relative.tolist(),\n                     \'prompts\': list(PLAIN_PROMPTS),\n                     \'score_type\': \'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.\'},\n            \'histogram\': {\'predicted_class_id\': hist_class, \'predicted_material\': CLASS_NAMES[hist_class],\n                          \'reference_gallery_index\': hist_index, \'similarity\': float(hist_similarities[hist_index]),\n                          \'crop_file\': self.gallery_meta[hist_index][\'crop_file\'],\n                          \'score_type\': \'Nearest Train RGB histogram by cosine similarity; argmax tie policy.\'},\n        }\n\n    def runtime_metadata(self):\n        self._assert_cpu_frozen()\n        return {\n            **self._provenance, \'clip_revision\': self.clip_revision, \'device\': \'cpu\', \'precision\': \'float32\',\n            \'resnet_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()\n                                              if parameter.is_floating_point()}),\n            \'clip_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.clip.parameters()\n                                            if parameter.is_floating_point()}),\n            \'torch_num_threads\': int(self.torch.get_num_threads()),\n            \'torch_num_interop_threads\': int(self.torch.get_num_interop_threads()),\n            \'all_parameters_frozen\': True, \'plain_prompts\': list(PLAIN_PROMPTS),\n            \'gallery_loaded\': self.gallery_embeddings is not None,\n            \'gallery_size\': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,\n            \'model_training_or_optimization\': False,\n            \'decision_role\': \'Advisory references and material suggestion; original detector outputs are not overwritten.\',\n        }\n', 'prepare_extension_test_inputs.py': '"""Prepare a prespecified material-balanced crop subset of the frozen Test split."""\n\n\ndef prepare_extension_test_inputs(frozen_protocol, evidence_dir):\n    from collections import Counter, OrderedDict, defaultdict\n    from datetime import datetime, timezone\n    from pathlib import Path, PurePosixPath\n    import csv\n    import hashlib\n    import io\n    import json\n    import math\n    import re\n    import time\n    import zipfile\n    import zlib\n    import numpy as np\n    import requests\n    from PIL import Image\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def sha(raw):\n        return hashlib.sha256(raw).hexdigest()\n\n    evidence_dir = Path(evidence_dir)\n    protocol_file = evidence_dir / \'frozen_protocol.json\'\n    require(protocol_file.is_file() and json.loads(protocol_file.read_text(encoding=\'utf-8\')) == frozen_protocol,\n            \'A byte-hashed frozen protocol must exist before opening Test annotations.\')\n    require(frozen_protocol[\'split\'] == \'test\' and frozen_protocol[\'per_class\'] == 50\n            and frozen_protocol[\'seed\'] == 20261116, \'The prespecified Test sample design differs.\')\n    project = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    local = Path(\'/content/Project9_Phase2\')\n    manifest_path = project / \'data_protocol/ef6b35d2ba7f/split_manifest.csv\'\n    require(sha(manifest_path.read_bytes()) == frozen_protocol[\'manifest_sha256\'], \'Frozen split bytes differ.\')\n    manifest = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding=\'utf-8-sig\'))))\n    rows = [row for row in manifest if row[\'split\'] == \'test\']\n    require(len(rows) == 929 and all(row[\'original_split\'] == \'test\' for row in rows), \'Frozen Test membership differs.\')\n    gallery = json.loads((Path(frozen_protocol[\'gallery_assets_path\']) / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n    gallery_names = {item[\'export_name\'] for item in gallery}\n    classes = [\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\']\n    archive_url = \'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1\'\n    archive_size = 7518242799\n    password = b\'REDACTED_DATASET_ZIP_PASSWORD\'\n    class HTTPRangeFile(io.RawIOBase):\n        """Bounded read-only seekable HTTP file for Python\'s ZIP64 reader."""\n\n        def __init__(self):\n            super().__init__()\n            self.position = 0\n            self.size = archive_size\n            self.block_size = 1024 * 1024\n            self.blocks = OrderedDict()\n            self.transferred = 0\n            self.requests_count = 0\n            self.session = requests.Session()\n\n        def readable(self):\n            return True\n\n        def seekable(self):\n            return True\n\n        def tell(self):\n            return self.position\n\n        def seek(self, offset, whence=0):\n            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1\n            if position < 0:\n                raise ValueError(\'Invalid HTTP ZIP seek.\')\n            self.position = int(position)\n            return self.position\n\n        def get_block(self, index):\n            if index in self.blocks:\n                self.blocks.move_to_end(index)\n                return self.blocks[index]\n            start = index * self.block_size\n            end = min(self.size - 1, start + self.block_size - 1)\n            needed = end - start + 1\n            require(self.transferred + needed <= 768 * 1024 ** 2, \'Partial-download budget exceeded; the full archive will not be downloaded automatically.\')\n            for attempt in range(3):\n                try:\n                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={\'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\', \'Cache-Control\': \'no-cache\'}) as response:\n                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:\n                            time.sleep(2 * (attempt + 1))\n                            continue\n                        require(response.status_code == 206, f\'HTTP {response.status_code}: requested range was not supported; full download prevented.\')\n                        content_range = response.headers.get(\'Content-Range\', \'\').strip()\n                        match = re.fullmatch(\'bytes (\\\\d+)-(\\\\d+)/(\\\\d+)\', content_range, flags=re.I)\n                        require(match and tuple(map(int, match.groups())) == (start, end, self.size), \'Server returned an unexpected Content-Range: \' + content_range)\n                        data = response.raw.read(needed + 1)\n                        require(len(data) == needed, \'Incomplete or oversized HTTP range response.\')\n                    break\n                except requests.RequestException:\n                    if attempt == 2:\n                        raise\n                    time.sleep(2 * (attempt + 1))\n            self.transferred += len(data)\n            self.requests_count += 1\n            self.blocks[index] = data\n            while len(self.blocks) > 16:\n                self.blocks.popitem(last=False)\n            if self.requests_count % 5 == 0:\n                print(f\'Partial ZIP transfer: {self.transferred / 1024 ** 2:.1f} MiB\')\n            return data\n\n        def read(self, count=-1):\n            if self.closed:\n                raise ValueError(\'HTTP ZIP stream is closed.\')\n            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))\n            require(count <= 64 * 1024 ** 2, \'Unexpected large ZIP read; complete-archive download prevented.\')\n            chunks = []\n            remaining = count\n            while remaining:\n                index, offset = divmod(self.position, self.block_size)\n                data = self.get_block(index)\n                take = min(remaining, len(data) - offset)\n                require(take > 0, \'Unexpected ZIP read position.\')\n                chunks.append(data[offset:offset + take])\n                self.position += take\n                remaining -= take\n            return b\'\'.join(chunks)\n\n        def close(self):\n            self.session.close()\n            self.blocks.clear()\n            super().close()\n\n    cache = local / \'extensions_test_source_images\'\n    cache.mkdir(parents=True, exist_ok=True)\n    existing_archive = next((path for path in [Path(\'/content/zerowaste-f-final.zip\'), project / \'cache/zerowaste-f-final.zip\']\n                             if path.is_file() and path.stat().st_size == archive_size), None)\n    remote = None\n    if existing_archive is None:\n        remote = HTTPRangeFile()\n        archive = zipfile.ZipFile(remote)\n    else:\n        archive = zipfile.ZipFile(existing_archive)\n    protocol_sha = sha(protocol_file.read_bytes())\n    try:\n        def member(suffix):\n            require(suffix.startswith(\'test/\') and \'..\' not in PurePosixPath(suffix).parts,\n                    \'Only source Test members selected by the frozen protocol may be read.\')\n            matches = [item for item in archive.infolist() if not item.is_dir()\n                       and (item.filename == suffix or item.filename.endswith(\'/\' + suffix))\n                       and \'..\' not in PurePosixPath(item.filename).parts]\n            require(len(matches) == 1, \'Test member is not unique: \' + suffix)\n            return matches[0]\n\n        def fetch(info, filename):\n            path = cache / filename\n            sidecar = cache / (filename + \'.source.json\')\n            require(0 < info.file_size <= (256 * 1024**2 if filename.endswith(\'.json\') else 12 * 1024**2),\n                    \'Unexpected source member size.\')\n            if path.exists():\n                require(sidecar.is_file(), \'Existing source cache has no provenance receipt.\')\n                raw = path.read_bytes()\n                receipt = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n                reused = True\n            else:\n                raw = archive.read(info, pwd=password)\n                receipt = {\'sha256\': sha(raw), \'member\': info.filename,\n                           \'member_crc32\': info.CRC, \'member_bytes\': info.file_size,\n                           \'archive_url\': archive_url, \'archive_total_bytes\': archive_size}\n                with path.open(\'xb\') as stream:\n                    stream.write(raw)\n                with sidecar.open(\'x\', encoding=\'utf-8\') as stream:\n                    json.dump(receipt, stream, indent=2)\n                reused = False\n            require(len(raw) == info.file_size and zlib.crc32(raw) & 0xffffffff == info.CRC,\n                    \'Source member CRC/size differs.\')\n            expected = {\'sha256\': sha(raw), \'member\': info.filename, \'member_crc32\': info.CRC,\n                        \'member_bytes\': info.file_size, \'archive_url\': archive_url,\n                        \'archive_total_bytes\': archive_size}\n            require(receipt == expected, \'Source cache provenance differs.\')\n            return path, raw, {**receipt, \'reused_cache\': reused,\n                               \'historical_full_source_image_sha256_anchor_available\': False}\n\n        _, raw_annotations, annotation_receipt = fetch(member(\'test/labels.json\'), \'source_test_annotations.json\')\n        coco = json.loads(raw_annotations.decode(\'utf-8-sig\'))\n        images = {int(item[\'id\']): item for item in coco[\'images\']}\n        category = {int(item[\'id\']): item[\'name\'] for item in coco[\'categories\']}\n        require(set(category.values()) == set(classes), \'Source material categories differ.\')\n        annotations = defaultdict(list)\n        for ann in coco[\'annotations\']:\n            annotations[int(ann[\'image_id\'])].append(ann)\n        pools = defaultdict(list)\n        for row in sorted(rows, key=lambda item: \'test__\' + item[\'filename\']):\n            image_id = int(row[\'image_id\'])\n            info = images[image_id]\n            require(PurePosixPath(info[\'file_name\']).name == row[\'filename\'], \'Test COCO image identity differs.\')\n            width, height = int(info[\'width\']), int(info[\'height\'])\n            counts = Counter()\n            for annotation_ordinal, ann in enumerate(annotations[image_id]):\n                cls = classes.index(category[int(ann[\'category_id\'])])\n                x, y, bw, bh = map(float, ann[\'bbox\'])\n                require(all(math.isfinite(value) for value in [x, y, bw, bh]) and bw > 0 and bh > 0,\n                        \'Invalid Test annotation box.\')\n                x1, y1, x2, y2 = max(0, min(width, x)), max(0, min(height, y)), max(0, min(width, x + bw)), max(0, min(height, y + bh))\n                require(x2 > x1 and y2 > y1, \'Empty clipped Test box.\')\n                xc, yc, wn, hn = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height\n                line = f\'{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}\'\n                _, xc, yc, wn, hn = map(float, line.split())\n                counts[classes[cls]] += 1\n                if wn > 0.003 and hn > 0.003:\n                    pools[cls].append({\'class_id\': cls, \'class_name\': classes[cls], \'image_id\': image_id,\n                        \'annotation_id\': int(ann[\'id\']), \'annotation_ordinal_in_image\': annotation_ordinal,\n                        \'export_name\': \'test__\' + row[\'filename\'], \'filename\': row[\'filename\'],\n                        \'width\': width, \'height\': height, \'original_split\': \'test\', \'project_split\': \'test\',\n                        \'xywh_normalized\': [xc, yc, wn, hn], \'converted_yolo_line\': line,\n                        \'official_clipped_box_xyxy\': [x1, y1, x2, y2], \'frozen_manifest_row\': row})\n            require(all(counts[name] == int(row[name]) for name in classes), \'Test annotation counts differ from frozen manifest.\')\n        rng = np.random.default_rng(frozen_protocol[\'seed\'])\n        selected = []\n        for cls in range(4):\n            require(pools[cls], \'A prespecified material has no eligible Test crops.\')\n            indices = rng.choice(len(pools[cls]), size=min(frozen_protocol[\'per_class\'], len(pools[cls])), replace=False)\n            selected.extend(dict(pools[cls][int(index)]) for index in indices)\n        selected_names = {item[\'export_name\'] for item in selected}\n        require(not selected_names & gallery_names, \'Test queries overlap the Train gallery source images.\')\n        member_by_name = {name: member(\'test/data/\' + name.removeprefix(\'test__\')) for name in selected_names}\n        source_records = {}\n        for ordinal, name in enumerate(sorted(selected_names, key=lambda value: member_by_name[value].header_offset), 1):\n            path, raw, receipt = fetch(member_by_name[name], name)\n            with Image.open(io.BytesIO(raw)) as image:\n                record = next(item for item in selected if item[\'export_name\'] == name)\n                require(image.format == \'PNG\' and image.size == (record[\'width\'], record[\'height\']), \'Test PNG dimensions/format differ.\')\n                image.verify()\n            source_records[name] = {\'path\': str(path), **receipt}\n            if ordinal % 10 == 0 or ordinal == len(selected_names):\n                print(\'Verified selected Test source PNGs:\', ordinal, \'/\', len(selected_names))\n        stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n        work_dir = local / (\'extension_test_inputs_\' + stamp)\n        (work_dir / \'test_queries\').mkdir(parents=True, exist_ok=False)\n        for ordinal, record in enumerate(selected):\n            source = source_records[record[\'export_name\']]\n            width, height = record[\'width\'], record[\'height\']\n            xc, yc, bw, bh = record[\'xywh_normalized\']\n            x1 = max(0, int((xc - bw / 2) * width))\n            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))\n            y1 = max(0, int((yc - bh / 2) * height))\n            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))\n            box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]\n            relative = f\'test_queries/t_{ordinal:03d}.png\'\n            path = work_dir / relative\n            with Image.open(source[\'path\']) as image:\n                image.convert(\'RGB\').crop(tuple(box)).save(path, format=\'PNG\')\n            record.update({\'ordinal\': ordinal, \'source_image_sha256\': source[\'sha256\'],\n                           \'crop_file\': relative, \'crop_path\': str(path), \'crop_sha256\': sha(path.read_bytes()),\n                           \'crop_box_xyxy\': box, \'crop_width\': box[2] - box[0], \'crop_height\': box[3] - box[1]})\n        require(sha(protocol_file.read_bytes()) == protocol_sha, \'Frozen Test protocol changed during preparation.\')\n        provenance = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n            \'selection\': \'Fixed seed20261116, up to50 eligible objects per material without replacement; use all if fewer than50.\',\n            \'seed\': 20261116, \'bit_generator\': type(rng.bit_generator).__name__,\n            \'sample_objects\': len(selected), \'unique_source_images\': len(selected_names),\n            \'counts_per_class\': {classes[cls]: sum(item[\'class_id\'] == cls for item in selected) for cls in range(4)},\n            \'eligible_objects_per_class\': {classes[cls]: len(pools[cls]) for cls in range(4)},\n            \'full_Test_partition_images\': 929, \'scope\': \'Prespecified material-balanced GT-crop Test subset; not a full929-image system evaluation.\',\n            \'annotation_source\': annotation_receipt, \'source_images\': source_records,\n            \'historical_full_PNG_or_Test_COCO_hash_anchors_available\': False,\n            \'new_source_verification\': \'Selected source ZIP members checked by CRC/size and new SHA256 receipts; identities/counts checked against frozen manifest.\',\n            \'whole_archive_md5_recomputed_this_run\': False, \'archive_url\': archive_url, \'archive_total_bytes\': archive_size,\n            \'range_bytes_transferred\': remote.transferred if remote else 0,\n            \'protocol_sha256_before_Test_annotation_access\': protocol_sha,\n            \'gallery_source_images_disjoint\': True, \'new_model_execution_in_preparation\': False}\n        return {\'queries\': selected, \'work_dir\': str(work_dir), \'provenance\': provenance}\n    finally:\n        archive.close()\n        if remote is not None:\n            remote.close()\n', 'run_extension_test_once_colab.py': '"""One prespecified held-out GT-crop Test evaluation for the two extensions."""\n\n\ndef run_extensions_test_once():\n    from datetime import datetime, timezone\n    from importlib.metadata import version, PackageNotFoundError\n    from pathlib import Path\n    import hashlib\n    import json\n    import os\n    import shutil\n    import subprocess\n    import sys\n    import zipfile\n    from google.colab import drive, files\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def digest(path):\n        value = hashlib.sha256()\n        with Path(path).open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n                value.update(chunk)\n        return value.hexdigest()\n\n    def save_new(path, value):\n        with Path(path).open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n            stream.write(\'\\n\')\n\n    def installed(name):\n        try:\n            return version(name)\n        except PackageNotFoundError:\n            return None\n\n    drive.mount(\'/content/drive\')\n    project = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    output = project / \'evaluation/EXTENSIONS_FINAL_TEST_ONCE\'\n    archive_path = project / \'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip\'\n    if output.exists():\n        require((output / \'COMPLETED.json\').is_file() and archive_path.is_file(),\n                \'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.\')\n        inventory = json.loads((output / \'collection_manifest.json\').read_text(encoding=\'utf-8\'))[\'files\']\n        with zipfile.ZipFile(archive_path) as archive:\n            require(archive.testzip() is None, \'Saved Test evidence ZIP failed CRC verification.\')\n            for item in inventory:\n                raw = archive.read(item[\'file\'])\n                require(len(raw) == item[\'bytes\'] and hashlib.sha256(raw).hexdigest() == item[\'sha256\'],\n                        \'Saved Test archive bytes differ: \' + item[\'file\'])\n        print(\'Previously completed extension Test found; downloading its saved evidence without model execution.\')\n        files.download(str(archive_path))\n        return {\'already_completed\': True, \'zip\': str(archive_path)}\n\n    os.environ[\'WANDB_MODE\'] = \'disabled\'\n    os.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n    os.environ[\'HF_HUB_DISABLE_TELEMETRY\'] = \'1\'\n    torch_version, vision_version = installed(\'torch\'), installed(\'torchvision\')\n    require(torch_version and vision_version, \'A matching preinstalled PyTorch/torchvision runtime is required.\')\n    desired = {\'transformers\': \'5.22.1\', \'numpy\': \'2.4.6\', \'Pillow\': \'11.3.0\'}\n    changes = [name + \'==\' + value for name, value in desired.items() if installed(name) != value]\n    changes += [name for name in [\'requests\'] if installed(name) is None]\n    if changes:\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'torch==\' + torch_version,\n                        \'torchvision==\' + vision_version, *changes], check=True, timeout=360)\n    import numpy as np\n    import PIL\n    from PIL import Image\n    require(np.__version__ == \'2.4.6\' and PIL.__version__ == \'11.3.0\',\n            \'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.\')\n    require(installed(\'torch\') == torch_version and installed(\'torchvision\') == vision_version,\n            \'The preinstalled torch pair unexpectedly changed.\')\n\n    validation = project / \'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z\'\n    assets = validation / \'deployment/assets\'\n    frozen_val = json.loads((validation / \'frozen_operator_VAL_protocol.json\').read_text(encoding=\'utf-8\'))\n    manifest = json.loads((validation / \'deployment/assist_manifest.json\').read_text(encoding=\'utf-8\'))\n    require(digest(validation / \'frozen_operator_VAL_protocol.json\') == _EXT_TEST_ANCHORS[\'val_protocol\'],\n            \'The already-audited Validation protocol differs.\')\n    require(digest(assets / \'gallery_index.npz\') == _EXT_TEST_ANCHORS[\'gallery_index\'],\n            \'The fixed160-reference gallery differs.\')\n    require(digest(assets / \'gallery_meta.json\') == _EXT_TEST_ANCHORS[\'gallery_meta\'], \'Gallery identities differ.\')\n    require(hashlib.sha256(_EXT_TEST_SOURCES[\'extension_assist_models.py\'].encode()).hexdigest()\n            == frozen_val[\'source_files_sha256\'][\'extension_assist_models.py\'], \'The model helper source changed after Validation.\')\n    protected_relative = _EXT_TEST_PROTECTED_PATHS\n    protected = {name: project / relative for name, relative in protected_relative.items()}\n    before = {name: digest(path) for name, path in protected.items()}\n    require(before == frozen_val[\'original_project_files_sha256\'], \'An original protected project reference differs.\')\n    for relative, expected in manifest[\'assets_sha256\'].items():\n        require(digest(assets / relative) == expected, \'Validation gallery/crop asset differs: \' + relative)\n    print(\'Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...\')\n    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest[\'clip_revision\'])\n    resources = engine.runtime_metadata()\n    for key in [\'clip_revision\', \'clip_checkpoint_sha256\', \'resnet_checkpoint_sha256\']:\n        require(resources[key] == manifest[\'model_resources\'][key], \'Official assistance model resource differs: \' + key)\n    gallery_meta = json.loads((assets / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n    gallery_crops = []\n    for item in gallery_meta:\n        with Image.open(assets / item[\'crop_file\']) as image:\n            gallery_crops.append(image.convert(\'RGB\'))\n    recomputed = engine.embed_resnet(gallery_crops)\n    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)\n    require(bool(np.all(cosines >= 0.999)), \'Runtime reference embedding compatibility failed before Test.\')\n    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-7),\n            \'Runtime histogram preprocessing differs from the fixed Validation reference bank.\')\n    del gallery_crops, recomputed\n\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    protocol = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n        \'manifest_sha256\': before[\'split_manifest\'], \'per_class\': 50, \'seed\': 20261116,\n        \'sampling_design\': \'Up to50 eligible GT objects per material without replacement; all if fewer than50. Independent RNGseed20261116, before viewing model outputs.\',\n        \'target_maximum_query_objects\': 200, \'full_Test_partition_images\': 929,\n        \'scope\': \'Prespecified material-balanced GT-crop Test subset, not a full929-image or end-to-end system evaluation.\',\n        \'small_box_filter\': \'Normalized width>0.003 and height>0.003, eight-decimal readback from original conversion.\',\n        \'gallery_assets_path\': str(assets), \'gallery_objects\': 160, \'output_dir\': str(output),\n        \'validation_protocol_sha256\': _EXT_TEST_ANCHORS[\'val_protocol\'],\n        \'gallery_index_sha256\': _EXT_TEST_ANCHORS[\'gallery_index\'], \'gallery_metadata_sha256\': _EXT_TEST_ANCHORS[\'gallery_meta\'],\n        \'source_files_sha256\': {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()},\n        \'model_resources\': resources, \'class_names\': list(CLASS_NAMES), \'clip_prompts\': list(PLAIN_PROMPTS),\n        \'prompt_model_threshold_selection_on_Test\': False,\n        \'histogram_baseline\': \'PIL11.3 RGBresize64,8bins/channel,L2unit; retrievaldescendingargsort, classificationfirstargmax.\',\n        \'reference_compatibility_min_cosine_before_Test\': float(cosines.min()),\n        \'protected_original_files_before\': before, \'M4_M5_Test_model_calls\': 0,\n        \'new_training_or_optimizer_steps\': 0, \'once_only_policy\': \'Fixed folder; completed attempts are downloaded again without model execution; incomplete attempts require explicit recovery review.\'}\n    output.mkdir(parents=True, exist_ok=False)\n    save_new(output / \'frozen_protocol.json\', protocol)\n    protocol_sha = digest(output / \'frozen_protocol.json\')\n    save_new(output / \'STATUS.json\', {\'phase\': \'preparing_inputs_no_Test_model_calls_yet\', \'protocol_sha256\': protocol_sha})\n    code_dir = output / \'code\'\n    code_dir.mkdir()\n    for name, source in _EXT_TEST_SOURCES.items():\n        (code_dir / name).write_text(source, encoding=\'utf-8\', newline=\'\\n\')\n    print(\'Frozen protocol saved BEFORE source Test annotation access. Preparing only the selected Test crops...\')\n    preparation = prepare_extension_test_inputs(protocol, output)\n    records = preparation[\'queries\']\n    work = Path(preparation[\'work_dir\'])\n    save_new(output / \'query_provenance.json\', preparation)\n    shutil.copytree(work / \'test_queries\', output / \'test_queries\')\n    reference_dir = output / \'reference_gallery\'\n    reference_dir.mkdir()\n    shutil.copytree(assets / \'gallery\', reference_dir / \'gallery\')\n    shutil.copyfile(assets / \'gallery_index.npz\', reference_dir / \'gallery_index.npz\')\n    shutil.copyfile(assets / \'gallery_meta.json\', reference_dir / \'gallery_meta.json\')\n    batch_dir = output / \'inference_batches\'\n    batch_dir.mkdir()\n    (output / \'STATUS.json\').write_text(json.dumps({\'phase\': \'Test_inference_started\',\n        \'protocol_sha256\': protocol_sha, \'query_objects\': len(records)}, indent=2), encoding=\'utf-8\')\n    q_parts, h_parts, l_parts = [], [], []\n    print(\'One fixed Test evaluation:\', len(records), \'annotated crops; no detector/segmenter calls.\')\n    for start in range(0, len(records), 8):\n        crops = []\n        for item in records[start:start + 8]:\n            path = output / item[\'crop_file\']\n            require(digest(path) == item[\'crop_sha256\'], \'Selected Test crop changed.\')\n            with Image.open(path) as image:\n                crops.append(image.convert(\'RGB\'))\n        q, h, logits = engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8)\n        path = batch_dir / f\'batch_{start // 8:03d}.npz\'\n        with path.open(\'xb\') as stream:\n            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)),\n                                embeddings=q, histograms=h, clip_logits=logits)\n        q_parts.append(q); h_parts.append(h); l_parts.append(logits)\n        print(\'Recorded Test crops:\', start + len(crops), \'/\', len(records))\n    q, h, logits = np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts)\n    labels = np.asarray([item[\'class_id\'] for item in records], dtype=np.int64)\n    gy = engine.gallery_labels\n    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)\n    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)\n    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]\n    clip_pred = logits.argmax(axis=1)\n    comparison = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n        \'sample_objects\': len(records), \'gallery_objects\': 160,\n        \'sample_counts\': preparation[\'provenance\'][\'counts_per_class\'],\n        \'unique_source_images\': preparation[\'provenance\'][\'unique_source_images\'],\n        \'visual_search_resnet\': search, \'visual_search_histogram\': baseline_search,\n        \'clip_plain\': classification_metrics(labels, clip_pred),\n        \'histogram_classification_baseline\': classification_metrics(labels, hist_pred),\n        \'frozen_protocol_sha256\': protocol_sha,\n        \'limits\': [\'Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.\',\n                   \'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.\',\n                   \'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.\']}\n    save_new(output / \'comparison_TEST.json\', comparison)\n    with (output / \'test_arrays.npz\').open(\'xb\') as stream:\n        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits,\n                            query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank,\n                            histogram_class_predictions=hist_pred)\n    rows = [{\'query_index\': index, \'crop_file\': item[\'crop_file\'], \'source_export_name\': item[\'export_name\'],\n             \'annotation_id\': item[\'annotation_id\'], \'ground_truth_class_id\': int(labels[index]),\n             \'resnet_top1_class_id\': int(gy[rank[index, 0]]),\n             \'resnet_top5_gallery_indices\': [int(value) for value in rank[index, :5]],\n             \'clip_class_id\': int(clip_pred[index]), \'clip_logits\': logits[index].tolist(),\n             \'histogram_class_id\': int(hist_pred[index])} for index, item in enumerate(records)]\n    save_new(output / \'per_query_TEST.json\', rows)\n    after = {name: digest(path) for name, path in protected.items()}\n    require(before == after and digest(output / \'frozen_protocol.json\') == protocol_sha,\n            \'An original file or frozen Test protocol changed.\')\n    save_new(output / \'protected_original_files.json\', {\'before\': before, \'after\': after,\n        \'unchanged\': True, \'original_M4_M5_Test_calls\': 0, \'training_or_optimizer_steps\': 0})\n    save_new(output / \'COMPLETED.json\', {\'status\': \'completed_once\', \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'frozen_protocol_sha256\': protocol_sha, \'comparison_sha256\': digest(output / \'comparison_TEST.json\'),\n        \'zip_name\': archive_path.name, \'M4_M5_Test_reevaluated\': False})\n    (output / \'STATUS.json\').write_text(json.dumps({\'phase\': \'completed_once\', \'protocol_sha256\': protocol_sha}), encoding=\'utf-8\')\n    inventory = [{\'file\': path.relative_to(output).as_posix(), \'bytes\': path.stat().st_size, \'sha256\': digest(path)}\n                 for path in sorted(output.rglob(\'*\')) if path.is_file()]\n    save_new(output / \'collection_manifest.json\', {\'files\': inventory})\n    with zipfile.ZipFile(archive_path, \'x\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file():\n                archive.write(path, path.relative_to(output).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        require(archive.testzip() is None, \'Test evidence ZIP CRC check failed.\')\n        for item in inventory:\n            raw = archive.read(item[\'file\'])\n            require(len(raw) == item[\'bytes\'] and hashlib.sha256(raw).hexdigest() == item[\'sha256\'], \'Packaged Test evidence differs.\')\n    print(json.dumps(comparison, indent=2))\n    print(\'Extension Test complete. No M4/M5 Test evaluation or training was run.\')\n    print(\'Saved original results unchanged. ZIP:\', archive_path)\n    files.download(str(archive_path))\n    return {\'zip\': str(archive_path), \'output_dir\': str(output), \'comparison\': comparison}\n\n\n_EXTENSION_TEST_RESULT = run_extensions_test_once()\n'}
_EXT_TEST_PROTECTED_PATHS = {'split_manifest': 'data_protocol/ef6b35d2ba7f/split_manifest.csv', 'validation_inventory': 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json', 'gallery_index': 'open_world/visual_search/gallery_index_train.npz', 'visual_search_result': 'open_world/visual_search/comparison_VAL.json', 'clip_result': 'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv', 'service_source': 'deployment/http_service/service.py', 'service_manifest': 'deployment/http_service/release_manifest.json', 'http_detector': 'deployment/http_service/best_det.pt', 'http_segmenter': 'deployment/http_service/best_seg.pt', 'source_detector': 'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt', 'source_segmenter': 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt', 'test_protocol': 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json', 'test_results': 'evaluation/FINAL_TEST_ONCE/results.json'}
_EXT_TEST_ANCHORS = {'val_protocol': '7d2e20f397e4ee0d13485b6998dbda1acc113a88a8f1b9b52ba6ba2d9bbc1b7d', 'gallery_index': '4e2a9cc509e2c54f99186a04019d9f82b3d82096e2e7ff4532ab0017b4571442', 'gallery_meta': '889381006555a967d65f4af69eaa895a6356af48dd93a6d43e1311440691a564'}

"""CPU models and numeric helpers for the optional operator-assistance stage.

Importing this module performs no model loading, downloads or inference.
Model construction is explicit; all predictions are advisory and do not alter
the original detector/segmenter response or introduce a rejection threshold.
"""

CLASS_NAMES = ('cardboard', 'metal', 'rigid_plastic', 'soft_plastic')
CLIP_MATERIAL_NAMES = ('cardboard', 'metal', 'rigid plastic', 'soft plastic')
PLAIN_PROMPTS = tuple('a photo of ' + name for name in CLIP_MATERIAL_NAMES)
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'


def _assist_require(condition, message):
    if not condition:
        raise RuntimeError(message)


def _assist_sha256(path):
    from pathlib import Path
    import hashlib
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()


def unit(values):
    """The original float32 L2 normalization, including its 1e-10 guard."""
    import numpy as np
    array = np.asarray(values, dtype=np.float32)
    _assist_require(array.ndim == 2 and np.isfinite(array).all(), 'Expected finite 2D features.')
    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)


def rgb_hist(crops):
    """Original 24-bin RGB histogram baseline; crops must be PIL images."""
    import numpy as np
    rows = []
    for crop in crops:
        array = np.asarray(crop.convert('RGB').resize((64, 64)), dtype=np.uint8)
        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,
                                                range=(0, 256))[0] for channel in range(3)]))
    _assist_require(bool(rows), 'At least one crop is required.')
    return unit(np.asarray(rows, dtype=np.float32))


def classification_metrics(true_labels, predicted_labels):
    """Four-material accuracy and macro metrics with zero-division set to zero."""
    import numpy as np
    truth = np.asarray(true_labels)
    prediction = np.asarray(predicted_labels)
    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape
                    and truth.size > 0, 'Class arrays must be nonempty, aligned 1D arrays.')
    _assist_require(np.issubdtype(truth.dtype, np.integer)
                    and np.issubdtype(prediction.dtype, np.integer), 'Class IDs must be integers.')
    _assist_require(((truth >= 0) & (truth < 4)).all()
                    and ((prediction >= 0) & (prediction < 4)).all(), 'Unknown material class ID.')
    matrix = np.zeros((4, 4), dtype=np.int64)
    np.add.at(matrix, (truth, prediction), 1)
    tp = np.diag(matrix).astype(np.float64)
    predicted_support = matrix.sum(axis=0)
    support = matrix.sum(axis=1)
    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)
    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)
    return {
        'accuracy': float(np.mean(truth == prediction)), 'macro_f1': float(f1.mean()),
        'macro_precision': float(precision.mean()), 'macro_recall': float(recall.mean()),
        'count': int(truth.size), 'confusion_matrix_true_rows_predicted_columns': matrix.tolist(),
        'per_class': [{'class_id': index, 'material': name, 'support': int(support[index]),
                       'precision': float(precision[index]), 'recall': float(recall[index]),
                       'f1': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],
    }


def retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):
    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""
    import numpy as np
    gallery = np.asarray(gallery_features, dtype=np.float32)
    queries = np.asarray(query_features, dtype=np.float32)
    gallery_y = np.asarray(gallery_labels)
    query_y = np.asarray(query_labels)
    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]
                    and len(gallery) >= 5 and len(queries) > 0, 'Invalid gallery/query feature shape.')
    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)
                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), 'Misaligned or nonfinite retrieval inputs.')
    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]
    top1 = gallery_y[rankings[:, 0]]
    top5 = gallery_y[rankings[:, :5]]
    return ({'top1_accuracy': float(np.mean(top1 == query_y)),
             'hit_at_5': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),
             'precision_at_5': float(np.mean(top5 == query_y[:, None]))}, rankings)


class OperatorAssistModels:
    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.

    ``assets_dir`` may be None while building a new gallery. Then call
    ``load_gallery`` after writing these files:

    - gallery_index.npz: float32 embeddings [160,512], float32 histograms
      [160,24], int64 labels [160].
    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,
      each with class_id and crop_file="gallery/g_000.png" etc.
    - the corresponding 160 PNG reference crops.

    local_only=True requires the ResNet checkpoint and the explicit immutable
    CLIP revision to be present already; it performs no network request.
    """

    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):
        from pathlib import Path
        from importlib.metadata import version
        import re
        import torch
        import torch.nn as nn
        from torchvision.models import resnet18, ResNet18_Weights
        from transformers import CLIPModel, CLIPProcessor
        from huggingface_hub import hf_hub_download, model_info

        self.torch = torch
        self.assets_dir = None
        self.gallery_embeddings = None
        self.gallery_histograms = None
        self.gallery_labels = None
        self.gallery_meta = None
        self.local_only = bool(local_only)
        torch.set_num_threads(1)
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            _assist_require(torch.get_num_interop_threads() == 1,
                            'Torch inter-op threads are already initialized differently; use a fresh process.')

        weights = ResNet18_Weights.IMAGENET1K_V1
        self.resnet_transform = weights.transforms()
        resnet_file = Path(torch.hub.get_dir()) / 'checkpoints/resnet18-f37072fd.pth'
        if self.local_only:
            _assist_require(resnet_file.is_file(), 'Cached ResNet18 checkpoint is required for local-only serving.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'Cached ResNet18 checkpoint checksum differs.')
            encoder = resnet18(weights=None)
            state = torch.load(resnet_file, map_location='cpu', weights_only=True)
            encoder.load_state_dict(state, strict=True)
            del state
        else:
            encoder = resnet18(weights=weights, progress=False)
            _assist_require(resnet_file.is_file(), 'Downloaded ResNet18 checkpoint was not found at its standard cache path.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'ResNet18 checkpoint checksum differs from its official filename prefix.')
        encoder.fc = nn.Identity()
        self.encoder = encoder.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.encoder.parameters():
            parameter.requires_grad_(False)

        hub_info = None
        if clip_revision is None:
            _assist_require(not self.local_only, 'An explicit cached CLIP revision is required in local-only mode.')
            hub_info = model_info(CLIP_MODEL_ID, revision='main', files_metadata=True, token=False)
            clip_revision = hub_info.sha
        _assist_require(isinstance(clip_revision, str)
                        and re.fullmatch(r'[0-9a-f]{40}|[0-9a-f]{64}', clip_revision) is not None,
                        'CLIP revision must be an immutable Hub commit hash.')
        self.clip_revision = clip_revision
        common = {'revision': self.clip_revision, 'local_files_only': self.local_only, 'token': False}
        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,
                                             trust_remote_code=False, **common)
        self.clip = self.clip.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.clip.parameters():
            parameter.requires_grad_(False)
        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,
                                                       trust_remote_code=False, **common)
        # The loader above explicitly selects the bin checkpoint. This cache-only
        # lookup records its actual bytes without triggering a second download.
        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename='pytorch_model.bin',
                                        revision=self.clip_revision, local_files_only=True, token=False))
        clip_sha = _assist_sha256(clip_file)
        expected_clip_sha = None
        if hub_info is not None:
            for sibling in hub_info.siblings or []:
                if sibling.rfilename == 'pytorch_model.bin':
                    lfs = getattr(sibling, 'lfs', None)
                    expected_clip_sha = (lfs.get('sha256') if isinstance(lfs, dict)
                                         else getattr(lfs, 'sha256', None))
                    break
        if expected_clip_sha is not None:
            _assist_require(clip_sha == expected_clip_sha, 'CLIP binary differs from the Hub LFS SHA256.')
        config_commit = getattr(self.clip.config, '_commit_hash', None)
        if config_commit is not None:
            _assist_require(config_commit == self.clip_revision, 'Loaded CLIP configuration revision differs.')
        self._provenance = {
            'resnet_weights': 'ResNet18_Weights.IMAGENET1K_V1',
            'resnet_weights_url': weights.url, 'resnet_checkpoint_path': str(resnet_file),
            'resnet_checkpoint_sha256': resnet_sha, 'resnet_transform': repr(self.resnet_transform),
            'clip_model_id': CLIP_MODEL_ID, 'clip_revision': self.clip_revision,
            'clip_config_commit_hash': config_commit, 'clip_checkpoint_path': str(clip_file),
            'clip_checkpoint_sha256': clip_sha, 'clip_hub_lfs_sha256': expected_clip_sha,
            'clip_checkpoint_url': 'https://huggingface.co/' + CLIP_MODEL_ID + '/resolve/'
                                   + self.clip_revision + '/pytorch_model.bin',
            'clip_processor_class': type(self.processor).__name__,
            'clip_image_processor_class': type(self.processor.image_processor).__name__,
            'clip_image_processor_config': self.processor.image_processor.to_dict(),
            'clip_attention_implementation': getattr(self.clip.config, '_attn_implementation', None),
            'versions': {name: version(name) for name in ['torch', 'torchvision', 'transformers',
                                                         'huggingface-hub', 'numpy', 'Pillow']},
        }
        self._assert_cpu_frozen()
        if assets_dir is not None:
            self.load_gallery(assets_dir)

    def _assert_cpu_frozen(self):
        for name, model in [('resnet', self.encoder), ('clip', self.clip)]:
            _assist_require(model.training is False, name + ' must be in eval mode.')
            _assist_require(all(parameter.device.type == 'cpu' and not parameter.requires_grad
                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)
                                for parameter in model.parameters()), name + ' must be frozen CPU FP32.')

    def embed_resnet(self, crops, batch=16):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid ResNet crop batch.')
        self._assert_cpu_frozen()
        vectors = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                tensor = self.torch.stack([self.resnet_transform(crop.convert('RGB'))
                                          for crop in crops[start:start + batch]])
                output = self.encoder(tensor.to(device='cpu', dtype=self.torch.float32))
                vectors.append(output.detach().cpu().numpy())
        array = np.concatenate(vectors).astype(np.float32)
        _assist_require(array.shape == (len(crops), 512), 'Unexpected ResNet feature dimensions.')
        return unit(array)

    def score_clip(self, crops, batch=8):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid CLIP crop batch.')
        self._assert_cpu_frozen()
        logits = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                inputs = self.processor(text=list(PLAIN_PROMPTS),
                                        images=[crop.convert('RGB') for crop in crops[start:start + batch]],
                                        return_tensors='pt', padding=True)
                inputs = {key: value.to(device='cpu', dtype=self.torch.float32)
                          if key == 'pixel_values' else value.to('cpu') for key, value in inputs.items()}
                output = self.clip(**inputs)
                logits.append(output.logits_per_image.detach().cpu().numpy())
        array = np.concatenate(logits).astype(np.float32)
        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), 'Unexpected or nonfinite CLIP logits.')
        return array

    def rgb_hist(self, crops):
        return rgb_hist(crops)

    def load_gallery(self, assets_dir):
        from pathlib import Path, PurePosixPath
        import json
        import numpy as np
        directory = Path(assets_dir).resolve()
        with np.load(directory / 'gallery_index.npz', allow_pickle=False) as bank:
            _assist_require(set(bank.files) == {'embeddings', 'histograms', 'labels'}, 'Unexpected assistance gallery keys.')
            embeddings = bank['embeddings'].copy()
            histograms = bank['histograms'].copy()
            labels = bank['labels'].copy()
        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32
                        and histograms.shape == (160, 24) and histograms.dtype == np.float32
                        and labels.shape == (160,) and labels.dtype == np.int64,
                        'Assistance gallery array shapes/dtypes differ.')
        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()
                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),
                        'Assistance gallery features/classes differ.')
        for name, values in [('embedding', embeddings), ('histogram', histograms)]:
            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),
                            'Unnormalized gallery ' + name + ' features.')
        metadata = json.loads((directory / 'gallery_meta.json').read_text(encoding='utf-8'))
        _assist_require(isinstance(metadata, list) and len(metadata) == 160, 'Expected 160 gallery metadata records.')
        for index, item in enumerate(metadata):
            expected_file = 'gallery/g_' + str(index).zfill(3) + '.png'
            _assist_require(isinstance(item, dict) and item.get('class_id') == int(labels[index])
                            and item.get('crop_file') == expected_file, 'Gallery metadata/array alignment differs.')
            crop = PurePosixPath(item['crop_file'])
            path = (directory / crop).resolve()
            _assist_require(not crop.is_absolute() and '..' not in crop.parts
                            and path.is_relative_to(directory) and path.is_file(), 'Missing or unsafe reference crop.')
        self.assets_dir = directory
        self.gallery_embeddings = embeddings
        self.gallery_histograms = histograms
        self.gallery_labels = labels
        self.gallery_meta = metadata
        return self

    def query_crop(self, crop):
        import numpy as np
        _assist_require(self.gallery_embeddings is not None, 'Load the assistance gallery first.')
        feature = self.embed_resnet([crop], batch=1)
        similarities = (feature @ self.gallery_embeddings.T)[0]
        ranking = similarities.argsort()[::-1]
        references = []
        for raw_index in ranking[:5]:
            index = int(raw_index)
            class_id = int(self.gallery_labels[index])
            references.append({'gallery_index': index, 'class_id': class_id,
                               'material': CLASS_NAMES[class_id], 'similarity': float(similarities[index]),
                               'crop_file': self.gallery_meta[index]['crop_file']})
        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)
        relative = np.exp(clip_logits - clip_logits.max())
        relative /= relative.sum()
        clip_class = int(clip_logits.argmax())
        histogram = rgb_hist([crop])
        hist_similarities = (histogram @ self.gallery_histograms.T)[0]
        hist_index = int(hist_similarities.argmax())
        hist_class = int(self.gallery_labels[hist_index])
        return {
            'visual_search': {'predicted_class_id': references[0]['class_id'],
                              'predicted_material': references[0]['material'], 'references': references,
                              'score_type': 'Cosine similarity to frozen Train reference crops; not calibrated confidence.'},
            'clip': {'predicted_class_id': clip_class, 'predicted_material': CLASS_NAMES[clip_class],
                     'logits': clip_logits.tolist(), 'relative_scores': relative.tolist(),
                     'prompts': list(PLAIN_PROMPTS),
                     'score_type': 'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.'},
            'histogram': {'predicted_class_id': hist_class, 'predicted_material': CLASS_NAMES[hist_class],
                          'reference_gallery_index': hist_index, 'similarity': float(hist_similarities[hist_index]),
                          'crop_file': self.gallery_meta[hist_index]['crop_file'],
                          'score_type': 'Nearest Train RGB histogram by cosine similarity; argmax tie policy.'},
        }

    def runtime_metadata(self):
        self._assert_cpu_frozen()
        return {
            **self._provenance, 'clip_revision': self.clip_revision, 'device': 'cpu', 'precision': 'float32',
            'resnet_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()
                                              if parameter.is_floating_point()}),
            'clip_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.clip.parameters()
                                            if parameter.is_floating_point()}),
            'torch_num_threads': int(self.torch.get_num_threads()),
            'torch_num_interop_threads': int(self.torch.get_num_interop_threads()),
            'all_parameters_frozen': True, 'plain_prompts': list(PLAIN_PROMPTS),
            'gallery_loaded': self.gallery_embeddings is not None,
            'gallery_size': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,
            'model_training_or_optimization': False,
            'decision_role': 'Advisory references and material suggestion; original detector outputs are not overwritten.',
        }


"""Prepare a prespecified material-balanced crop subset of the frozen Test split."""


def prepare_extension_test_inputs(frozen_protocol, evidence_dir):
    from collections import Counter, OrderedDict, defaultdict
    from datetime import datetime, timezone
    from pathlib import Path, PurePosixPath
    import csv
    import hashlib
    import io
    import json
    import math
    import re
    import time
    import zipfile
    import zlib
    import numpy as np
    import requests
    from PIL import Image

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    evidence_dir = Path(evidence_dir)
    protocol_file = evidence_dir / 'frozen_protocol.json'
    require(protocol_file.is_file() and json.loads(protocol_file.read_text(encoding='utf-8')) == frozen_protocol,
            'A byte-hashed frozen protocol must exist before opening Test annotations.')
    require(frozen_protocol['split'] == 'test' and frozen_protocol['per_class'] == 50
            and frozen_protocol['seed'] == 20261116, 'The prespecified Test sample design differs.')
    project = Path('/content/drive/MyDrive/Project9_Phase2')
    local = Path('/content/Project9_Phase2')
    manifest_path = project / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    require(sha(manifest_path.read_bytes()) == frozen_protocol['manifest_sha256'], 'Frozen split bytes differ.')
    manifest = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding='utf-8-sig'))))
    rows = [row for row in manifest if row['split'] == 'test']
    require(len(rows) == 929 and all(row['original_split'] == 'test' for row in rows), 'Frozen Test membership differs.')
    gallery = json.loads((Path(frozen_protocol['gallery_assets_path']) / 'gallery_meta.json').read_text(encoding='utf-8'))
    gallery_names = {item['export_name'] for item in gallery}
    classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    archive_size = 7518242799
    password = b'REDACTED_DATASET_ZIP_PASSWORD'
    class HTTPRangeFile(io.RawIOBase):
        """Bounded read-only seekable HTTP file for Python's ZIP64 reader."""

        def __init__(self):
            super().__init__()
            self.position = 0
            self.size = archive_size
            self.block_size = 1024 * 1024
            self.blocks = OrderedDict()
            self.transferred = 0
            self.requests_count = 0
            self.session = requests.Session()

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.position

        def seek(self, offset, whence=0):
            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
            if position < 0:
                raise ValueError('Invalid HTTP ZIP seek.')
            self.position = int(position)
            return self.position

        def get_block(self, index):
            if index in self.blocks:
                self.blocks.move_to_end(index)
                return self.blocks[index]
            start = index * self.block_size
            end = min(self.size - 1, start + self.block_size - 1)
            needed = end - start + 1
            require(self.transferred + needed <= 768 * 1024 ** 2, 'Partial-download budget exceeded; the full archive will not be downloaded automatically.')
            for attempt in range(3):
                try:
                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity', 'Cache-Control': 'no-cache'}) as response:
                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
                            time.sleep(2 * (attempt + 1))
                            continue
                        require(response.status_code == 206, f'HTTP {response.status_code}: requested range was not supported; full download prevented.')
                        content_range = response.headers.get('Content-Range', '').strip()
                        match = re.fullmatch('bytes (\\d+)-(\\d+)/(\\d+)', content_range, flags=re.I)
                        require(match and tuple(map(int, match.groups())) == (start, end, self.size), 'Server returned an unexpected Content-Range: ' + content_range)
                        data = response.raw.read(needed + 1)
                        require(len(data) == needed, 'Incomplete or oversized HTTP range response.')
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
            self.transferred += len(data)
            self.requests_count += 1
            self.blocks[index] = data
            while len(self.blocks) > 16:
                self.blocks.popitem(last=False)
            if self.requests_count % 5 == 0:
                print(f'Partial ZIP transfer: {self.transferred / 1024 ** 2:.1f} MiB')
            return data

        def read(self, count=-1):
            if self.closed:
                raise ValueError('HTTP ZIP stream is closed.')
            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))
            require(count <= 64 * 1024 ** 2, 'Unexpected large ZIP read; complete-archive download prevented.')
            chunks = []
            remaining = count
            while remaining:
                index, offset = divmod(self.position, self.block_size)
                data = self.get_block(index)
                take = min(remaining, len(data) - offset)
                require(take > 0, 'Unexpected ZIP read position.')
                chunks.append(data[offset:offset + take])
                self.position += take
                remaining -= take
            return b''.join(chunks)

        def close(self):
            self.session.close()
            self.blocks.clear()
            super().close()

    cache = local / 'extensions_test_source_images'
    cache.mkdir(parents=True, exist_ok=True)
    existing_archive = next((path for path in [Path('/content/zerowaste-f-final.zip'), project / 'cache/zerowaste-f-final.zip']
                             if path.is_file() and path.stat().st_size == archive_size), None)
    remote = None
    if existing_archive is None:
        remote = HTTPRangeFile()
        archive = zipfile.ZipFile(remote)
    else:
        archive = zipfile.ZipFile(existing_archive)
    protocol_sha = sha(protocol_file.read_bytes())
    try:
        def member(suffix):
            require(suffix.startswith('test/') and '..' not in PurePosixPath(suffix).parts,
                    'Only source Test members selected by the frozen protocol may be read.')
            matches = [item for item in archive.infolist() if not item.is_dir()
                       and (item.filename == suffix or item.filename.endswith('/' + suffix))
                       and '..' not in PurePosixPath(item.filename).parts]
            require(len(matches) == 1, 'Test member is not unique: ' + suffix)
            return matches[0]

        def fetch(info, filename):
            path = cache / filename
            sidecar = cache / (filename + '.source.json')
            require(0 < info.file_size <= (256 * 1024**2 if filename.endswith('.json') else 12 * 1024**2),
                    'Unexpected source member size.')
            if path.exists():
                require(sidecar.is_file(), 'Existing source cache has no provenance receipt.')
                raw = path.read_bytes()
                receipt = json.loads(sidecar.read_text(encoding='utf-8'))
                reused = True
            else:
                raw = archive.read(info, pwd=password)
                receipt = {'sha256': sha(raw), 'member': info.filename,
                           'member_crc32': info.CRC, 'member_bytes': info.file_size,
                           'archive_url': archive_url, 'archive_total_bytes': archive_size}
                with path.open('xb') as stream:
                    stream.write(raw)
                with sidecar.open('x', encoding='utf-8') as stream:
                    json.dump(receipt, stream, indent=2)
                reused = False
            require(len(raw) == info.file_size and zlib.crc32(raw) & 0xffffffff == info.CRC,
                    'Source member CRC/size differs.')
            expected = {'sha256': sha(raw), 'member': info.filename, 'member_crc32': info.CRC,
                        'member_bytes': info.file_size, 'archive_url': archive_url,
                        'archive_total_bytes': archive_size}
            require(receipt == expected, 'Source cache provenance differs.')
            return path, raw, {**receipt, 'reused_cache': reused,
                               'historical_full_source_image_sha256_anchor_available': False}

        _, raw_annotations, annotation_receipt = fetch(member('test/labels.json'), 'source_test_annotations.json')
        coco = json.loads(raw_annotations.decode('utf-8-sig'))
        images = {int(item['id']): item for item in coco['images']}
        category = {int(item['id']): item['name'] for item in coco['categories']}
        require(set(category.values()) == set(classes), 'Source material categories differ.')
        annotations = defaultdict(list)
        for ann in coco['annotations']:
            annotations[int(ann['image_id'])].append(ann)
        pools = defaultdict(list)
        for row in sorted(rows, key=lambda item: 'test__' + item['filename']):
            image_id = int(row['image_id'])
            info = images[image_id]
            require(PurePosixPath(info['file_name']).name == row['filename'], 'Test COCO image identity differs.')
            width, height = int(info['width']), int(info['height'])
            counts = Counter()
            for annotation_ordinal, ann in enumerate(annotations[image_id]):
                cls = classes.index(category[int(ann['category_id'])])
                x, y, bw, bh = map(float, ann['bbox'])
                require(all(math.isfinite(value) for value in [x, y, bw, bh]) and bw > 0 and bh > 0,
                        'Invalid Test annotation box.')
                x1, y1, x2, y2 = max(0, min(width, x)), max(0, min(height, y)), max(0, min(width, x + bw)), max(0, min(height, y + bh))
                require(x2 > x1 and y2 > y1, 'Empty clipped Test box.')
                xc, yc, wn, hn = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height
                line = f'{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}'
                _, xc, yc, wn, hn = map(float, line.split())
                counts[classes[cls]] += 1
                if wn > 0.003 and hn > 0.003:
                    pools[cls].append({'class_id': cls, 'class_name': classes[cls], 'image_id': image_id,
                        'annotation_id': int(ann['id']), 'annotation_ordinal_in_image': annotation_ordinal,
                        'export_name': 'test__' + row['filename'], 'filename': row['filename'],
                        'width': width, 'height': height, 'original_split': 'test', 'project_split': 'test',
                        'xywh_normalized': [xc, yc, wn, hn], 'converted_yolo_line': line,
                        'official_clipped_box_xyxy': [x1, y1, x2, y2], 'frozen_manifest_row': row})
            require(all(counts[name] == int(row[name]) for name in classes), 'Test annotation counts differ from frozen manifest.')
        rng = np.random.default_rng(frozen_protocol['seed'])
        selected = []
        for cls in range(4):
            require(pools[cls], 'A prespecified material has no eligible Test crops.')
            indices = rng.choice(len(pools[cls]), size=min(frozen_protocol['per_class'], len(pools[cls])), replace=False)
            selected.extend(dict(pools[cls][int(index)]) for index in indices)
        selected_names = {item['export_name'] for item in selected}
        require(not selected_names & gallery_names, 'Test queries overlap the Train gallery source images.')
        member_by_name = {name: member('test/data/' + name.removeprefix('test__')) for name in selected_names}
        source_records = {}
        for ordinal, name in enumerate(sorted(selected_names, key=lambda value: member_by_name[value].header_offset), 1):
            path, raw, receipt = fetch(member_by_name[name], name)
            with Image.open(io.BytesIO(raw)) as image:
                record = next(item for item in selected if item['export_name'] == name)
                require(image.format == 'PNG' and image.size == (record['width'], record['height']), 'Test PNG dimensions/format differ.')
                image.verify()
            source_records[name] = {'path': str(path), **receipt}
            if ordinal % 10 == 0 or ordinal == len(selected_names):
                print('Verified selected Test source PNGs:', ordinal, '/', len(selected_names))
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        work_dir = local / ('extension_test_inputs_' + stamp)
        (work_dir / 'test_queries').mkdir(parents=True, exist_ok=False)
        for ordinal, record in enumerate(selected):
            source = source_records[record['export_name']]
            width, height = record['width'], record['height']
            xc, yc, bw, bh = record['xywh_normalized']
            x1 = max(0, int((xc - bw / 2) * width))
            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))
            y1 = max(0, int((yc - bh / 2) * height))
            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))
            box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]
            relative = f'test_queries/t_{ordinal:03d}.png'
            path = work_dir / relative
            with Image.open(source['path']) as image:
                image.convert('RGB').crop(tuple(box)).save(path, format='PNG')
            record.update({'ordinal': ordinal, 'source_image_sha256': source['sha256'],
                           'crop_file': relative, 'crop_path': str(path), 'crop_sha256': sha(path.read_bytes()),
                           'crop_box_xyxy': box, 'crop_width': box[2] - box[0], 'crop_height': box[3] - box[1]})
        require(sha(protocol_file.read_bytes()) == protocol_sha, 'Frozen Test protocol changed during preparation.')
        provenance = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test',
            'selection': 'Fixed seed20261116, up to50 eligible objects per material without replacement; use all if fewer than50.',
            'seed': 20261116, 'bit_generator': type(rng.bit_generator).__name__,
            'sample_objects': len(selected), 'unique_source_images': len(selected_names),
            'counts_per_class': {classes[cls]: sum(item['class_id'] == cls for item in selected) for cls in range(4)},
            'eligible_objects_per_class': {classes[cls]: len(pools[cls]) for cls in range(4)},
            'full_Test_partition_images': 929, 'scope': 'Prespecified material-balanced GT-crop Test subset; not a full929-image system evaluation.',
            'annotation_source': annotation_receipt, 'source_images': source_records,
            'historical_full_PNG_or_Test_COCO_hash_anchors_available': False,
            'new_source_verification': 'Selected source ZIP members checked by CRC/size and new SHA256 receipts; identities/counts checked against frozen manifest.',
            'whole_archive_md5_recomputed_this_run': False, 'archive_url': archive_url, 'archive_total_bytes': archive_size,
            'range_bytes_transferred': remote.transferred if remote else 0,
            'protocol_sha256_before_Test_annotation_access': protocol_sha,
            'gallery_source_images_disjoint': True, 'new_model_execution_in_preparation': False}
        return {'queries': selected, 'work_dir': str(work_dir), 'provenance': provenance}
    finally:
        archive.close()
        if remote is not None:
            remote.close()


"""One prespecified held-out GT-crop Test evaluation for the two extensions."""


def run_extensions_test_once():
    from datetime import datetime, timezone
    from importlib.metadata import version, PackageNotFoundError
    from pathlib import Path
    import hashlib
    import json
    import os
    import shutil
    import subprocess
    import sys
    import zipfile
    from google.colab import drive, files

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024**2), b''):
                value.update(chunk)
        return value.hexdigest()

    def save_new(path, value):
        with Path(path).open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write('\n')

    def installed(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None

    drive.mount('/content/drive')
    project = Path('/content/drive/MyDrive/Project9_Phase2')
    output = project / 'evaluation/EXTENSIONS_FINAL_TEST_ONCE'
    archive_path = project / 'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip'
    if output.exists():
        require((output / 'COMPLETED.json').is_file() and archive_path.is_file(),
                'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.')
        inventory = json.loads((output / 'collection_manifest.json').read_text(encoding='utf-8'))['files']
        with zipfile.ZipFile(archive_path) as archive:
            require(archive.testzip() is None, 'Saved Test evidence ZIP failed CRC verification.')
            for item in inventory:
                raw = archive.read(item['file'])
                require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'],
                        'Saved Test archive bytes differ: ' + item['file'])
        print('Previously completed extension Test found; downloading its saved evidence without model execution.')
        files.download(str(archive_path))
        return {'already_completed': True, 'zip': str(archive_path)}

    os.environ['WANDB_MODE'] = 'disabled'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    torch_version, vision_version = installed('torch'), installed('torchvision')
    require(torch_version and vision_version, 'A matching preinstalled PyTorch/torchvision runtime is required.')
    desired = {'transformers': '5.22.1', 'numpy': '2.4.6', 'Pillow': '11.3.0'}
    changes = [name + '==' + value for name, value in desired.items() if installed(name) != value]
    changes += [name for name in ['requests'] if installed(name) is None]
    if changes:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==' + torch_version,
                        'torchvision==' + vision_version, *changes], check=True, timeout=360)
    import numpy as np
    import PIL
    from PIL import Image
    require(np.__version__ == '2.4.6' and PIL.__version__ == '11.3.0',
            'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.')
    require(installed('torch') == torch_version and installed('torchvision') == vision_version,
            'The preinstalled torch pair unexpectedly changed.')

    validation = project / 'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z'
    assets = validation / 'deployment/assets'
    frozen_val = json.loads((validation / 'frozen_operator_VAL_protocol.json').read_text(encoding='utf-8'))
    manifest = json.loads((validation / 'deployment/assist_manifest.json').read_text(encoding='utf-8'))
    require(digest(validation / 'frozen_operator_VAL_protocol.json') == _EXT_TEST_ANCHORS['val_protocol'],
            'The already-audited Validation protocol differs.')
    require(digest(assets / 'gallery_index.npz') == _EXT_TEST_ANCHORS['gallery_index'],
            'The fixed160-reference gallery differs.')
    require(digest(assets / 'gallery_meta.json') == _EXT_TEST_ANCHORS['gallery_meta'], 'Gallery identities differ.')
    require(hashlib.sha256(_EXT_TEST_SOURCES['extension_assist_models.py'].encode()).hexdigest()
            == frozen_val['source_files_sha256']['extension_assist_models.py'], 'The model helper source changed after Validation.')
    protected_relative = _EXT_TEST_PROTECTED_PATHS
    protected = {name: project / relative for name, relative in protected_relative.items()}
    before = {name: digest(path) for name, path in protected.items()}
    require(before == frozen_val['original_project_files_sha256'], 'An original protected project reference differs.')
    for relative, expected in manifest['assets_sha256'].items():
        require(digest(assets / relative) == expected, 'Validation gallery/crop asset differs: ' + relative)
    print('Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...')
    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest['clip_revision'])
    resources = engine.runtime_metadata()
    for key in ['clip_revision', 'clip_checkpoint_sha256', 'resnet_checkpoint_sha256']:
        require(resources[key] == manifest['model_resources'][key], 'Official assistance model resource differs: ' + key)
    gallery_meta = json.loads((assets / 'gallery_meta.json').read_text(encoding='utf-8'))
    gallery_crops = []
    for item in gallery_meta:
        with Image.open(assets / item['crop_file']) as image:
            gallery_crops.append(image.convert('RGB'))
    recomputed = engine.embed_resnet(gallery_crops)
    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)
    require(bool(np.all(cosines >= 0.999)), 'Runtime reference embedding compatibility failed before Test.')
    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-7),
            'Runtime histogram preprocessing differs from the fixed Validation reference bank.')
    del gallery_crops, recomputed

    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    protocol = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test',
        'manifest_sha256': before['split_manifest'], 'per_class': 50, 'seed': 20261116,
        'sampling_design': 'Up to50 eligible GT objects per material without replacement; all if fewer than50. Independent RNGseed20261116, before viewing model outputs.',
        'target_maximum_query_objects': 200, 'full_Test_partition_images': 929,
        'scope': 'Prespecified material-balanced GT-crop Test subset, not a full929-image or end-to-end system evaluation.',
        'small_box_filter': 'Normalized width>0.003 and height>0.003, eight-decimal readback from original conversion.',
        'gallery_assets_path': str(assets), 'gallery_objects': 160, 'output_dir': str(output),
        'validation_protocol_sha256': _EXT_TEST_ANCHORS['val_protocol'],
        'gallery_index_sha256': _EXT_TEST_ANCHORS['gallery_index'], 'gallery_metadata_sha256': _EXT_TEST_ANCHORS['gallery_meta'],
        'source_files_sha256': {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()},
        'model_resources': resources, 'class_names': list(CLASS_NAMES), 'clip_prompts': list(PLAIN_PROMPTS),
        'prompt_model_threshold_selection_on_Test': False,
        'histogram_baseline': 'PIL11.3 RGBresize64,8bins/channel,L2unit; retrievaldescendingargsort, classificationfirstargmax.',
        'reference_compatibility_min_cosine_before_Test': float(cosines.min()),
        'protected_original_files_before': before, 'M4_M5_Test_model_calls': 0,
        'new_training_or_optimizer_steps': 0, 'once_only_policy': 'Fixed folder; completed attempts are downloaded again without model execution; incomplete attempts require explicit recovery review.'}
    output.mkdir(parents=True, exist_ok=False)
    save_new(output / 'frozen_protocol.json', protocol)
    protocol_sha = digest(output / 'frozen_protocol.json')
    save_new(output / 'STATUS.json', {'phase': 'preparing_inputs_no_Test_model_calls_yet', 'protocol_sha256': protocol_sha})
    code_dir = output / 'code'
    code_dir.mkdir()
    for name, source in _EXT_TEST_SOURCES.items():
        (code_dir / name).write_text(source, encoding='utf-8', newline='\n')
    print('Frozen protocol saved BEFORE source Test annotation access. Preparing only the selected Test crops...')
    preparation = prepare_extension_test_inputs(protocol, output)
    records = preparation['queries']
    work = Path(preparation['work_dir'])
    save_new(output / 'query_provenance.json', preparation)
    shutil.copytree(work / 'test_queries', output / 'test_queries')
    reference_dir = output / 'reference_gallery'
    reference_dir.mkdir()
    shutil.copytree(assets / 'gallery', reference_dir / 'gallery')
    shutil.copyfile(assets / 'gallery_index.npz', reference_dir / 'gallery_index.npz')
    shutil.copyfile(assets / 'gallery_meta.json', reference_dir / 'gallery_meta.json')
    batch_dir = output / 'inference_batches'
    batch_dir.mkdir()
    (output / 'STATUS.json').write_text(json.dumps({'phase': 'Test_inference_started',
        'protocol_sha256': protocol_sha, 'query_objects': len(records)}, indent=2), encoding='utf-8')
    q_parts, h_parts, l_parts = [], [], []
    print('One fixed Test evaluation:', len(records), 'annotated crops; no detector/segmenter calls.')
    for start in range(0, len(records), 8):
        crops = []
        for item in records[start:start + 8]:
            path = output / item['crop_file']
            require(digest(path) == item['crop_sha256'], 'Selected Test crop changed.')
            with Image.open(path) as image:
                crops.append(image.convert('RGB'))
        q, h, logits = engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8)
        path = batch_dir / f'batch_{start // 8:03d}.npz'
        with path.open('xb') as stream:
            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)),
                                embeddings=q, histograms=h, clip_logits=logits)
        q_parts.append(q); h_parts.append(h); l_parts.append(logits)
        print('Recorded Test crops:', start + len(crops), '/', len(records))
    q, h, logits = np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts)
    labels = np.asarray([item['class_id'] for item in records], dtype=np.int64)
    gy = engine.gallery_labels
    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)
    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)
    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]
    clip_pred = logits.argmax(axis=1)
    comparison = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test',
        'sample_objects': len(records), 'gallery_objects': 160,
        'sample_counts': preparation['provenance']['counts_per_class'],
        'unique_source_images': preparation['provenance']['unique_source_images'],
        'visual_search_resnet': search, 'visual_search_histogram': baseline_search,
        'clip_plain': classification_metrics(labels, clip_pred),
        'histogram_classification_baseline': classification_metrics(labels, hist_pred),
        'frozen_protocol_sha256': protocol_sha,
        'limits': ['Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.',
                   'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.',
                   'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.']}
    save_new(output / 'comparison_TEST.json', comparison)
    with (output / 'test_arrays.npz').open('xb') as stream:
        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits,
                            query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank,
                            histogram_class_predictions=hist_pred)
    rows = [{'query_index': index, 'crop_file': item['crop_file'], 'source_export_name': item['export_name'],
             'annotation_id': item['annotation_id'], 'ground_truth_class_id': int(labels[index]),
             'resnet_top1_class_id': int(gy[rank[index, 0]]),
             'resnet_top5_gallery_indices': [int(value) for value in rank[index, :5]],
             'clip_class_id': int(clip_pred[index]), 'clip_logits': logits[index].tolist(),
             'histogram_class_id': int(hist_pred[index])} for index, item in enumerate(records)]
    save_new(output / 'per_query_TEST.json', rows)
    after = {name: digest(path) for name, path in protected.items()}
    require(before == after and digest(output / 'frozen_protocol.json') == protocol_sha,
            'An original file or frozen Test protocol changed.')
    save_new(output / 'protected_original_files.json', {'before': before, 'after': after,
        'unchanged': True, 'original_M4_M5_Test_calls': 0, 'training_or_optimizer_steps': 0})
    save_new(output / 'COMPLETED.json', {'status': 'completed_once', 'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'frozen_protocol_sha256': protocol_sha, 'comparison_sha256': digest(output / 'comparison_TEST.json'),
        'zip_name': archive_path.name, 'M4_M5_Test_reevaluated': False})
    (output / 'STATUS.json').write_text(json.dumps({'phase': 'completed_once', 'protocol_sha256': protocol_sha}), encoding='utf-8')
    inventory = [{'file': path.relative_to(output).as_posix(), 'bytes': path.stat().st_size, 'sha256': digest(path)}
                 for path in sorted(output.rglob('*')) if path.is_file()]
    save_new(output / 'collection_manifest.json', {'files': inventory})
    with zipfile.ZipFile(archive_path, 'x', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(output).as_posix())
    with zipfile.ZipFile(archive_path) as archive:
        require(archive.testzip() is None, 'Test evidence ZIP CRC check failed.')
        for item in inventory:
            raw = archive.read(item['file'])
            require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Packaged Test evidence differs.')
    print(json.dumps(comparison, indent=2))
    print('Extension Test complete. No M4/M5 Test evaluation or training was run.')
    print('Saved original results unchanged. ZIP:', archive_path)
    files.download(str(archive_path))
    return {'zip': str(archive_path), 'output_dir': str(output), 'comparison': comparison}


_EXTENSION_TEST_RESULT = run_extensions_test_once()




In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# خلية استعادة فقط لفشل المقارنة قبل بدء Test؛ لا تحذف أي مجلد.
# تتحقق من البروتوكول والحالة وغياب الدفعات، وتستخدم نفس الإعدادات والعينة.
# انسخ الملف كاملًا إلى خلية جديدة وشغّلها فقط.

_EXT_TEST_SOURCES = {'extension_assist_models.py': '"""CPU models and numeric helpers for the optional operator-assistance stage.\n\nImporting this module performs no model loading, downloads or inference.\nModel construction is explicit; all predictions are advisory and do not alter\nthe original detector/segmenter response or introduce a rejection threshold.\n"""\n\nCLASS_NAMES = (\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\')\nCLIP_MATERIAL_NAMES = (\'cardboard\', \'metal\', \'rigid plastic\', \'soft plastic\')\nPLAIN_PROMPTS = tuple(\'a photo of \' + name for name in CLIP_MATERIAL_NAMES)\nCLIP_MODEL_ID = \'openai/clip-vit-base-patch32\'\n\n\ndef _assist_require(condition, message):\n    if not condition:\n        raise RuntimeError(message)\n\n\ndef _assist_sha256(path):\n    from pathlib import Path\n    import hashlib\n    value = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b\'\'):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef unit(values):\n    """The original float32 L2 normalization, including its 1e-10 guard."""\n    import numpy as np\n    array = np.asarray(values, dtype=np.float32)\n    _assist_require(array.ndim == 2 and np.isfinite(array).all(), \'Expected finite 2D features.\')\n    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)\n\n\ndef rgb_hist(crops):\n    """Original 24-bin RGB histogram baseline; crops must be PIL images."""\n    import numpy as np\n    rows = []\n    for crop in crops:\n        array = np.asarray(crop.convert(\'RGB\').resize((64, 64)), dtype=np.uint8)\n        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,\n                                                range=(0, 256))[0] for channel in range(3)]))\n    _assist_require(bool(rows), \'At least one crop is required.\')\n    return unit(np.asarray(rows, dtype=np.float32))\n\n\ndef classification_metrics(true_labels, predicted_labels):\n    """Four-material accuracy and macro metrics with zero-division set to zero."""\n    import numpy as np\n    truth = np.asarray(true_labels)\n    prediction = np.asarray(predicted_labels)\n    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape\n                    and truth.size > 0, \'Class arrays must be nonempty, aligned 1D arrays.\')\n    _assist_require(np.issubdtype(truth.dtype, np.integer)\n                    and np.issubdtype(prediction.dtype, np.integer), \'Class IDs must be integers.\')\n    _assist_require(((truth >= 0) & (truth < 4)).all()\n                    and ((prediction >= 0) & (prediction < 4)).all(), \'Unknown material class ID.\')\n    matrix = np.zeros((4, 4), dtype=np.int64)\n    np.add.at(matrix, (truth, prediction), 1)\n    tp = np.diag(matrix).astype(np.float64)\n    predicted_support = matrix.sum(axis=0)\n    support = matrix.sum(axis=1)\n    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)\n    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)\n    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)\n    return {\n        \'accuracy\': float(np.mean(truth == prediction)), \'macro_f1\': float(f1.mean()),\n        \'macro_precision\': float(precision.mean()), \'macro_recall\': float(recall.mean()),\n        \'count\': int(truth.size), \'confusion_matrix_true_rows_predicted_columns\': matrix.tolist(),\n        \'per_class\': [{\'class_id\': index, \'material\': name, \'support\': int(support[index]),\n                       \'precision\': float(precision[index]), \'recall\': float(recall[index]),\n                       \'f1\': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],\n    }\n\n\ndef retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):\n    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""\n    import numpy as np\n    gallery = np.asarray(gallery_features, dtype=np.float32)\n    queries = np.asarray(query_features, dtype=np.float32)\n    gallery_y = np.asarray(gallery_labels)\n    query_y = np.asarray(query_labels)\n    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]\n                    and len(gallery) >= 5 and len(queries) > 0, \'Invalid gallery/query feature shape.\')\n    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)\n                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), \'Misaligned or nonfinite retrieval inputs.\')\n    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]\n    top1 = gallery_y[rankings[:, 0]]\n    top5 = gallery_y[rankings[:, :5]]\n    return ({\'top1_accuracy\': float(np.mean(top1 == query_y)),\n             \'hit_at_5\': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),\n             \'precision_at_5\': float(np.mean(top5 == query_y[:, None]))}, rankings)\n\n\nclass OperatorAssistModels:\n    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.\n\n    ``assets_dir`` may be None while building a new gallery. Then call\n    ``load_gallery`` after writing these files:\n\n    - gallery_index.npz: float32 embeddings [160,512], float32 histograms\n      [160,24], int64 labels [160].\n    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,\n      each with class_id and crop_file="gallery/g_000.png" etc.\n    - the corresponding 160 PNG reference crops.\n\n    local_only=True requires the ResNet checkpoint and the explicit immutable\n    CLIP revision to be present already; it performs no network request.\n    """\n\n    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):\n        from pathlib import Path\n        from importlib.metadata import version\n        import re\n        import torch\n        import torch.nn as nn\n        from torchvision.models import resnet18, ResNet18_Weights\n        from transformers import CLIPModel, CLIPProcessor\n        from huggingface_hub import hf_hub_download, model_info\n\n        self.torch = torch\n        self.assets_dir = None\n        self.gallery_embeddings = None\n        self.gallery_histograms = None\n        self.gallery_labels = None\n        self.gallery_meta = None\n        self.local_only = bool(local_only)\n        torch.set_num_threads(1)\n        try:\n            torch.set_num_interop_threads(1)\n        except RuntimeError:\n            _assist_require(torch.get_num_interop_threads() == 1,\n                            \'Torch inter-op threads are already initialized differently; use a fresh process.\')\n\n        weights = ResNet18_Weights.IMAGENET1K_V1\n        self.resnet_transform = weights.transforms()\n        resnet_file = Path(torch.hub.get_dir()) / \'checkpoints/resnet18-f37072fd.pth\'\n        if self.local_only:\n            _assist_require(resnet_file.is_file(), \'Cached ResNet18 checkpoint is required for local-only serving.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'Cached ResNet18 checkpoint checksum differs.\')\n            encoder = resnet18(weights=None)\n            state = torch.load(resnet_file, map_location=\'cpu\', weights_only=True)\n            encoder.load_state_dict(state, strict=True)\n            del state\n        else:\n            encoder = resnet18(weights=weights, progress=False)\n            _assist_require(resnet_file.is_file(), \'Downloaded ResNet18 checkpoint was not found at its standard cache path.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'ResNet18 checkpoint checksum differs from its official filename prefix.\')\n        encoder.fc = nn.Identity()\n        self.encoder = encoder.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.encoder.parameters():\n            parameter.requires_grad_(False)\n\n        hub_info = None\n        if clip_revision is None:\n            _assist_require(not self.local_only, \'An explicit cached CLIP revision is required in local-only mode.\')\n            hub_info = model_info(CLIP_MODEL_ID, revision=\'main\', files_metadata=True, token=False)\n            clip_revision = hub_info.sha\n        _assist_require(isinstance(clip_revision, str)\n                        and re.fullmatch(r\'[0-9a-f]{40}|[0-9a-f]{64}\', clip_revision) is not None,\n                        \'CLIP revision must be an immutable Hub commit hash.\')\n        self.clip_revision = clip_revision\n        common = {\'revision\': self.clip_revision, \'local_files_only\': self.local_only, \'token\': False}\n        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,\n                                             trust_remote_code=False, **common)\n        self.clip = self.clip.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.clip.parameters():\n            parameter.requires_grad_(False)\n        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,\n                                                       trust_remote_code=False, **common)\n        # The loader above explicitly selects the bin checkpoint. This cache-only\n        # lookup records its actual bytes without triggering a second download.\n        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename=\'pytorch_model.bin\',\n                                        revision=self.clip_revision, local_files_only=True, token=False))\n        clip_sha = _assist_sha256(clip_file)\n        expected_clip_sha = None\n        if hub_info is not None:\n            for sibling in hub_info.siblings or []:\n                if sibling.rfilename == \'pytorch_model.bin\':\n                    lfs = getattr(sibling, \'lfs\', None)\n                    expected_clip_sha = (lfs.get(\'sha256\') if isinstance(lfs, dict)\n                                         else getattr(lfs, \'sha256\', None))\n                    break\n        if expected_clip_sha is not None:\n            _assist_require(clip_sha == expected_clip_sha, \'CLIP binary differs from the Hub LFS SHA256.\')\n        config_commit = getattr(self.clip.config, \'_commit_hash\', None)\n        if config_commit is not None:\n            _assist_require(config_commit == self.clip_revision, \'Loaded CLIP configuration revision differs.\')\n        self._provenance = {\n            \'resnet_weights\': \'ResNet18_Weights.IMAGENET1K_V1\',\n            \'resnet_weights_url\': weights.url, \'resnet_checkpoint_path\': str(resnet_file),\n            \'resnet_checkpoint_sha256\': resnet_sha, \'resnet_transform\': repr(self.resnet_transform),\n            \'clip_model_id\': CLIP_MODEL_ID, \'clip_revision\': self.clip_revision,\n            \'clip_config_commit_hash\': config_commit, \'clip_checkpoint_path\': str(clip_file),\n            \'clip_checkpoint_sha256\': clip_sha, \'clip_hub_lfs_sha256\': expected_clip_sha,\n            \'clip_checkpoint_url\': \'https://huggingface.co/\' + CLIP_MODEL_ID + \'/resolve/\'\n                                   + self.clip_revision + \'/pytorch_model.bin\',\n            \'clip_processor_class\': type(self.processor).__name__,\n            \'clip_image_processor_class\': type(self.processor.image_processor).__name__,\n            \'clip_image_processor_config\': self.processor.image_processor.to_dict(),\n            \'clip_attention_implementation\': getattr(self.clip.config, \'_attn_implementation\', None),\n            \'versions\': {name: version(name) for name in [\'torch\', \'torchvision\', \'transformers\',\n                                                         \'huggingface-hub\', \'numpy\', \'Pillow\']},\n        }\n        self._assert_cpu_frozen()\n        if assets_dir is not None:\n            self.load_gallery(assets_dir)\n\n    def _assert_cpu_frozen(self):\n        for name, model in [(\'resnet\', self.encoder), (\'clip\', self.clip)]:\n            _assist_require(model.training is False, name + \' must be in eval mode.\')\n            _assist_require(all(parameter.device.type == \'cpu\' and not parameter.requires_grad\n                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)\n                                for parameter in model.parameters()), name + \' must be frozen CPU FP32.\')\n\n    def embed_resnet(self, crops, batch=16):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid ResNet crop batch.\')\n        self._assert_cpu_frozen()\n        vectors = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                tensor = self.torch.stack([self.resnet_transform(crop.convert(\'RGB\'))\n                                          for crop in crops[start:start + batch]])\n                output = self.encoder(tensor.to(device=\'cpu\', dtype=self.torch.float32))\n                vectors.append(output.detach().cpu().numpy())\n        array = np.concatenate(vectors).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 512), \'Unexpected ResNet feature dimensions.\')\n        return unit(array)\n\n    def score_clip(self, crops, batch=8):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid CLIP crop batch.\')\n        self._assert_cpu_frozen()\n        logits = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                inputs = self.processor(text=list(PLAIN_PROMPTS),\n                                        images=[crop.convert(\'RGB\') for crop in crops[start:start + batch]],\n                                        return_tensors=\'pt\', padding=True)\n                inputs = {key: value.to(device=\'cpu\', dtype=self.torch.float32)\n                          if key == \'pixel_values\' else value.to(\'cpu\') for key, value in inputs.items()}\n                output = self.clip(**inputs)\n                logits.append(output.logits_per_image.detach().cpu().numpy())\n        array = np.concatenate(logits).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), \'Unexpected or nonfinite CLIP logits.\')\n        return array\n\n    def rgb_hist(self, crops):\n        return rgb_hist(crops)\n\n    def load_gallery(self, assets_dir):\n        from pathlib import Path, PurePosixPath\n        import json\n        import numpy as np\n        directory = Path(assets_dir).resolve()\n        with np.load(directory / \'gallery_index.npz\', allow_pickle=False) as bank:\n            _assist_require(set(bank.files) == {\'embeddings\', \'histograms\', \'labels\'}, \'Unexpected assistance gallery keys.\')\n            embeddings = bank[\'embeddings\'].copy()\n            histograms = bank[\'histograms\'].copy()\n            labels = bank[\'labels\'].copy()\n        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32\n                        and histograms.shape == (160, 24) and histograms.dtype == np.float32\n                        and labels.shape == (160,) and labels.dtype == np.int64,\n                        \'Assistance gallery array shapes/dtypes differ.\')\n        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()\n                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),\n                        \'Assistance gallery features/classes differ.\')\n        for name, values in [(\'embedding\', embeddings), (\'histogram\', histograms)]:\n            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),\n                            \'Unnormalized gallery \' + name + \' features.\')\n        metadata = json.loads((directory / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n        _assist_require(isinstance(metadata, list) and len(metadata) == 160, \'Expected 160 gallery metadata records.\')\n        for index, item in enumerate(metadata):\n            expected_file = \'gallery/g_\' + str(index).zfill(3) + \'.png\'\n            _assist_require(isinstance(item, dict) and item.get(\'class_id\') == int(labels[index])\n                            and item.get(\'crop_file\') == expected_file, \'Gallery metadata/array alignment differs.\')\n            crop = PurePosixPath(item[\'crop_file\'])\n            path = (directory / crop).resolve()\n            _assist_require(not crop.is_absolute() and \'..\' not in crop.parts\n                            and path.is_relative_to(directory) and path.is_file(), \'Missing or unsafe reference crop.\')\n        self.assets_dir = directory\n        self.gallery_embeddings = embeddings\n        self.gallery_histograms = histograms\n        self.gallery_labels = labels\n        self.gallery_meta = metadata\n        return self\n\n    def query_crop(self, crop):\n        import numpy as np\n        _assist_require(self.gallery_embeddings is not None, \'Load the assistance gallery first.\')\n        feature = self.embed_resnet([crop], batch=1)\n        similarities = (feature @ self.gallery_embeddings.T)[0]\n        ranking = similarities.argsort()[::-1]\n        references = []\n        for raw_index in ranking[:5]:\n            index = int(raw_index)\n            class_id = int(self.gallery_labels[index])\n            references.append({\'gallery_index\': index, \'class_id\': class_id,\n                               \'material\': CLASS_NAMES[class_id], \'similarity\': float(similarities[index]),\n                               \'crop_file\': self.gallery_meta[index][\'crop_file\']})\n        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)\n        relative = np.exp(clip_logits - clip_logits.max())\n        relative /= relative.sum()\n        clip_class = int(clip_logits.argmax())\n        histogram = rgb_hist([crop])\n        hist_similarities = (histogram @ self.gallery_histograms.T)[0]\n        hist_index = int(hist_similarities.argmax())\n        hist_class = int(self.gallery_labels[hist_index])\n        return {\n            \'visual_search\': {\'predicted_class_id\': references[0][\'class_id\'],\n                              \'predicted_material\': references[0][\'material\'], \'references\': references,\n                              \'score_type\': \'Cosine similarity to frozen Train reference crops; not calibrated confidence.\'},\n            \'clip\': {\'predicted_class_id\': clip_class, \'predicted_material\': CLASS_NAMES[clip_class],\n                     \'logits\': clip_logits.tolist(), \'relative_scores\': relative.tolist(),\n                     \'prompts\': list(PLAIN_PROMPTS),\n                     \'score_type\': \'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.\'},\n            \'histogram\': {\'predicted_class_id\': hist_class, \'predicted_material\': CLASS_NAMES[hist_class],\n                          \'reference_gallery_index\': hist_index, \'similarity\': float(hist_similarities[hist_index]),\n                          \'crop_file\': self.gallery_meta[hist_index][\'crop_file\'],\n                          \'score_type\': \'Nearest Train RGB histogram by cosine similarity; argmax tie policy.\'},\n        }\n\n    def runtime_metadata(self):\n        self._assert_cpu_frozen()\n        return {\n            **self._provenance, \'clip_revision\': self.clip_revision, \'device\': \'cpu\', \'precision\': \'float32\',\n            \'resnet_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()\n                                              if parameter.is_floating_point()}),\n            \'clip_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.clip.parameters()\n                                            if parameter.is_floating_point()}),\n            \'torch_num_threads\': int(self.torch.get_num_threads()),\n            \'torch_num_interop_threads\': int(self.torch.get_num_interop_threads()),\n            \'all_parameters_frozen\': True, \'plain_prompts\': list(PLAIN_PROMPTS),\n            \'gallery_loaded\': self.gallery_embeddings is not None,\n            \'gallery_size\': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,\n            \'model_training_or_optimization\': False,\n            \'decision_role\': \'Advisory references and material suggestion; original detector outputs are not overwritten.\',\n        }\n', 'prepare_extension_test_inputs.py': '"""Prepare a prespecified material-balanced crop subset of the frozen Test split."""\n\n\ndef prepare_extension_test_inputs(frozen_protocol, evidence_dir):\n    from collections import Counter, OrderedDict, defaultdict\n    from datetime import datetime, timezone\n    from pathlib import Path, PurePosixPath\n    import csv\n    import hashlib\n    import io\n    import json\n    import math\n    import re\n    import time\n    import zipfile\n    import zlib\n    import numpy as np\n    import requests\n    from PIL import Image\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def sha(raw):\n        return hashlib.sha256(raw).hexdigest()\n\n    evidence_dir = Path(evidence_dir)\n    protocol_file = evidence_dir / \'frozen_protocol.json\'\n    require(protocol_file.is_file() and json.loads(protocol_file.read_text(encoding=\'utf-8\')) == frozen_protocol,\n            \'A byte-hashed frozen protocol must exist before opening Test annotations.\')\n    require(frozen_protocol[\'split\'] == \'test\' and frozen_protocol[\'per_class\'] == 50\n            and frozen_protocol[\'seed\'] == 20261116, \'The prespecified Test sample design differs.\')\n    project = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    local = Path(\'/content/Project9_Phase2\')\n    manifest_path = project / \'data_protocol/ef6b35d2ba7f/split_manifest.csv\'\n    require(sha(manifest_path.read_bytes()) == frozen_protocol[\'manifest_sha256\'], \'Frozen split bytes differ.\')\n    manifest = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding=\'utf-8-sig\'))))\n    rows = [row for row in manifest if row[\'split\'] == \'test\']\n    require(len(rows) == 929 and all(row[\'original_split\'] == \'test\' for row in rows), \'Frozen Test membership differs.\')\n    gallery = json.loads((Path(frozen_protocol[\'gallery_assets_path\']) / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n    gallery_names = {item[\'export_name\'] for item in gallery}\n    classes = [\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\']\n    archive_url = \'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1\'\n    archive_size = 7518242799\n    password = b\'REDACTED_DATASET_ZIP_PASSWORD\'\n    class HTTPRangeFile(io.RawIOBase):\n        """Bounded read-only seekable HTTP file for Python\'s ZIP64 reader."""\n\n        def __init__(self):\n            super().__init__()\n            self.position = 0\n            self.size = archive_size\n            self.block_size = 1024 * 1024\n            self.blocks = OrderedDict()\n            self.transferred = 0\n            self.requests_count = 0\n            self.session = requests.Session()\n\n        def readable(self):\n            return True\n\n        def seekable(self):\n            return True\n\n        def tell(self):\n            return self.position\n\n        def seek(self, offset, whence=0):\n            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1\n            if position < 0:\n                raise ValueError(\'Invalid HTTP ZIP seek.\')\n            self.position = int(position)\n            return self.position\n\n        def get_block(self, index):\n            if index in self.blocks:\n                self.blocks.move_to_end(index)\n                return self.blocks[index]\n            start = index * self.block_size\n            end = min(self.size - 1, start + self.block_size - 1)\n            needed = end - start + 1\n            require(self.transferred + needed <= 768 * 1024 ** 2, \'Partial-download budget exceeded; the full archive will not be downloaded automatically.\')\n            for attempt in range(3):\n                try:\n                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={\'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\', \'Cache-Control\': \'no-cache\'}) as response:\n                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:\n                            time.sleep(2 * (attempt + 1))\n                            continue\n                        require(response.status_code == 206, f\'HTTP {response.status_code}: requested range was not supported; full download prevented.\')\n                        content_range = response.headers.get(\'Content-Range\', \'\').strip()\n                        match = re.fullmatch(\'bytes (\\\\d+)-(\\\\d+)/(\\\\d+)\', content_range, flags=re.I)\n                        require(match and tuple(map(int, match.groups())) == (start, end, self.size), \'Server returned an unexpected Content-Range: \' + content_range)\n                        data = response.raw.read(needed + 1)\n                        require(len(data) == needed, \'Incomplete or oversized HTTP range response.\')\n                    break\n                except requests.RequestException:\n                    if attempt == 2:\n                        raise\n                    time.sleep(2 * (attempt + 1))\n            self.transferred += len(data)\n            self.requests_count += 1\n            self.blocks[index] = data\n            while len(self.blocks) > 16:\n                self.blocks.popitem(last=False)\n            if self.requests_count % 5 == 0:\n                print(f\'Partial ZIP transfer: {self.transferred / 1024 ** 2:.1f} MiB\')\n            return data\n\n        def read(self, count=-1):\n            if self.closed:\n                raise ValueError(\'HTTP ZIP stream is closed.\')\n            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))\n            require(count <= 64 * 1024 ** 2, \'Unexpected large ZIP read; complete-archive download prevented.\')\n            chunks = []\n            remaining = count\n            while remaining:\n                index, offset = divmod(self.position, self.block_size)\n                data = self.get_block(index)\n                take = min(remaining, len(data) - offset)\n                require(take > 0, \'Unexpected ZIP read position.\')\n                chunks.append(data[offset:offset + take])\n                self.position += take\n                remaining -= take\n            return b\'\'.join(chunks)\n\n        def close(self):\n            self.session.close()\n            self.blocks.clear()\n            super().close()\n\n    cache = local / \'extensions_test_source_images\'\n    cache.mkdir(parents=True, exist_ok=True)\n    existing_archive = next((path for path in [Path(\'/content/zerowaste-f-final.zip\'), project / \'cache/zerowaste-f-final.zip\']\n                             if path.is_file() and path.stat().st_size == archive_size), None)\n    remote = None\n    if existing_archive is None:\n        remote = HTTPRangeFile()\n        archive = zipfile.ZipFile(remote)\n    else:\n        archive = zipfile.ZipFile(existing_archive)\n    protocol_sha = sha(protocol_file.read_bytes())\n    try:\n        def member(suffix):\n            require(suffix.startswith(\'test/\') and \'..\' not in PurePosixPath(suffix).parts,\n                    \'Only source Test members selected by the frozen protocol may be read.\')\n            matches = [item for item in archive.infolist() if not item.is_dir()\n                       and (item.filename == suffix or item.filename.endswith(\'/\' + suffix))\n                       and \'..\' not in PurePosixPath(item.filename).parts]\n            require(len(matches) == 1, \'Test member is not unique: \' + suffix)\n            return matches[0]\n\n        def fetch(info, filename):\n            path = cache / filename\n            sidecar = cache / (filename + \'.source.json\')\n            require(0 < info.file_size <= (256 * 1024**2 if filename.endswith(\'.json\') else 12 * 1024**2),\n                    \'Unexpected source member size.\')\n            if path.exists():\n                require(sidecar.is_file(), \'Existing source cache has no provenance receipt.\')\n                raw = path.read_bytes()\n                receipt = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n                reused = True\n            else:\n                raw = archive.read(info, pwd=password)\n                receipt = {\'sha256\': sha(raw), \'member\': info.filename,\n                           \'member_crc32\': info.CRC, \'member_bytes\': info.file_size,\n                           \'archive_url\': archive_url, \'archive_total_bytes\': archive_size}\n                with path.open(\'xb\') as stream:\n                    stream.write(raw)\n                with sidecar.open(\'x\', encoding=\'utf-8\') as stream:\n                    json.dump(receipt, stream, indent=2)\n                reused = False\n            require(len(raw) == info.file_size and zlib.crc32(raw) & 0xffffffff == info.CRC,\n                    \'Source member CRC/size differs.\')\n            expected = {\'sha256\': sha(raw), \'member\': info.filename, \'member_crc32\': info.CRC,\n                        \'member_bytes\': info.file_size, \'archive_url\': archive_url,\n                        \'archive_total_bytes\': archive_size}\n            require(receipt == expected, \'Source cache provenance differs.\')\n            return path, raw, {**receipt, \'reused_cache\': reused,\n                               \'historical_full_source_image_sha256_anchor_available\': False}\n\n        _, raw_annotations, annotation_receipt = fetch(member(\'test/labels.json\'), \'source_test_annotations.json\')\n        coco = json.loads(raw_annotations.decode(\'utf-8-sig\'))\n        images = {int(item[\'id\']): item for item in coco[\'images\']}\n        category = {int(item[\'id\']): item[\'name\'] for item in coco[\'categories\']}\n        require(set(category.values()) == set(classes), \'Source material categories differ.\')\n        annotations = defaultdict(list)\n        for ann in coco[\'annotations\']:\n            annotations[int(ann[\'image_id\'])].append(ann)\n        pools = defaultdict(list)\n        for row in sorted(rows, key=lambda item: \'test__\' + item[\'filename\']):\n            image_id = int(row[\'image_id\'])\n            info = images[image_id]\n            require(PurePosixPath(info[\'file_name\']).name == row[\'filename\'], \'Test COCO image identity differs.\')\n            width, height = int(info[\'width\']), int(info[\'height\'])\n            counts = Counter()\n            for annotation_ordinal, ann in enumerate(annotations[image_id]):\n                cls = classes.index(category[int(ann[\'category_id\'])])\n                x, y, bw, bh = map(float, ann[\'bbox\'])\n                require(all(math.isfinite(value) for value in [x, y, bw, bh]) and bw > 0 and bh > 0,\n                        \'Invalid Test annotation box.\')\n                x1, y1, x2, y2 = max(0, min(width, x)), max(0, min(height, y)), max(0, min(width, x + bw)), max(0, min(height, y + bh))\n                require(x2 > x1 and y2 > y1, \'Empty clipped Test box.\')\n                xc, yc, wn, hn = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height\n                line = f\'{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}\'\n                _, xc, yc, wn, hn = map(float, line.split())\n                counts[classes[cls]] += 1\n                if wn > 0.003 and hn > 0.003:\n                    pools[cls].append({\'class_id\': cls, \'class_name\': classes[cls], \'image_id\': image_id,\n                        \'annotation_id\': int(ann[\'id\']), \'annotation_ordinal_in_image\': annotation_ordinal,\n                        \'export_name\': \'test__\' + row[\'filename\'], \'filename\': row[\'filename\'],\n                        \'width\': width, \'height\': height, \'original_split\': \'test\', \'project_split\': \'test\',\n                        \'xywh_normalized\': [xc, yc, wn, hn], \'converted_yolo_line\': line,\n                        \'official_clipped_box_xyxy\': [x1, y1, x2, y2], \'frozen_manifest_row\': row})\n            require(all(counts[name] == int(row[name]) for name in classes), \'Test annotation counts differ from frozen manifest.\')\n        rng = np.random.default_rng(frozen_protocol[\'seed\'])\n        selected = []\n        for cls in range(4):\n            require(pools[cls], \'A prespecified material has no eligible Test crops.\')\n            indices = rng.choice(len(pools[cls]), size=min(frozen_protocol[\'per_class\'], len(pools[cls])), replace=False)\n            selected.extend(dict(pools[cls][int(index)]) for index in indices)\n        selected_names = {item[\'export_name\'] for item in selected}\n        require(not selected_names & gallery_names, \'Test queries overlap the Train gallery source images.\')\n        member_by_name = {name: member(\'test/data/\' + name.removeprefix(\'test__\')) for name in selected_names}\n        source_records = {}\n        for ordinal, name in enumerate(sorted(selected_names, key=lambda value: member_by_name[value].header_offset), 1):\n            path, raw, receipt = fetch(member_by_name[name], name)\n            with Image.open(io.BytesIO(raw)) as image:\n                record = next(item for item in selected if item[\'export_name\'] == name)\n                require(image.format == \'PNG\' and image.size == (record[\'width\'], record[\'height\']), \'Test PNG dimensions/format differ.\')\n                image.verify()\n            source_records[name] = {\'path\': str(path), **receipt}\n            if ordinal % 10 == 0 or ordinal == len(selected_names):\n                print(\'Verified selected Test source PNGs:\', ordinal, \'/\', len(selected_names))\n        stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n        work_dir = local / (\'extension_test_inputs_\' + stamp)\n        (work_dir / \'test_queries\').mkdir(parents=True, exist_ok=False)\n        for ordinal, record in enumerate(selected):\n            source = source_records[record[\'export_name\']]\n            width, height = record[\'width\'], record[\'height\']\n            xc, yc, bw, bh = record[\'xywh_normalized\']\n            x1 = max(0, int((xc - bw / 2) * width))\n            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))\n            y1 = max(0, int((yc - bh / 2) * height))\n            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))\n            box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]\n            relative = f\'test_queries/t_{ordinal:03d}.png\'\n            path = work_dir / relative\n            with Image.open(source[\'path\']) as image:\n                image.convert(\'RGB\').crop(tuple(box)).save(path, format=\'PNG\')\n            record.update({\'ordinal\': ordinal, \'source_image_sha256\': source[\'sha256\'],\n                           \'crop_file\': relative, \'crop_path\': str(path), \'crop_sha256\': sha(path.read_bytes()),\n                           \'crop_box_xyxy\': box, \'crop_width\': box[2] - box[0], \'crop_height\': box[3] - box[1]})\n        require(sha(protocol_file.read_bytes()) == protocol_sha, \'Frozen Test protocol changed during preparation.\')\n        provenance = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n            \'selection\': \'Fixed seed20261116, up to50 eligible objects per material without replacement; use all if fewer than50.\',\n            \'seed\': 20261116, \'bit_generator\': type(rng.bit_generator).__name__,\n            \'sample_objects\': len(selected), \'unique_source_images\': len(selected_names),\n            \'counts_per_class\': {classes[cls]: sum(item[\'class_id\'] == cls for item in selected) for cls in range(4)},\n            \'eligible_objects_per_class\': {classes[cls]: len(pools[cls]) for cls in range(4)},\n            \'full_Test_partition_images\': 929, \'scope\': \'Prespecified material-balanced GT-crop Test subset; not a full929-image system evaluation.\',\n            \'annotation_source\': annotation_receipt, \'source_images\': source_records,\n            \'historical_full_PNG_or_Test_COCO_hash_anchors_available\': False,\n            \'new_source_verification\': \'Selected source ZIP members checked by CRC/size and new SHA256 receipts; identities/counts checked against frozen manifest.\',\n            \'whole_archive_md5_recomputed_this_run\': False, \'archive_url\': archive_url, \'archive_total_bytes\': archive_size,\n            \'range_bytes_transferred\': remote.transferred if remote else 0,\n            \'protocol_sha256_before_Test_annotation_access\': protocol_sha,\n            \'gallery_source_images_disjoint\': True, \'new_model_execution_in_preparation\': False}\n        return {\'queries\': selected, \'work_dir\': str(work_dir), \'provenance\': provenance}\n    finally:\n        archive.close()\n        if remote is not None:\n            remote.close()\n', 'run_extension_test_once_colab.py': '"""One prespecified held-out GT-crop Test evaluation for the two extensions."""\n\n\ndef run_extensions_test_once():\n    from datetime import datetime, timezone\n    from importlib.metadata import version, PackageNotFoundError\n    from pathlib import Path\n    import hashlib\n    import json\n    import os\n    import shutil\n    import subprocess\n    import sys\n    import zipfile\n    from google.colab import drive, files\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def digest(path):\n        value = hashlib.sha256()\n        with Path(path).open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n                value.update(chunk)\n        return value.hexdigest()\n\n    def save_new(path, value):\n        with Path(path).open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n            stream.write(\'\\n\')\n\n    def installed(name):\n        try:\n            return version(name)\n        except PackageNotFoundError:\n            return None\n\n    drive.mount(\'/content/drive\')\n    project = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    output = project / \'evaluation/EXTENSIONS_FINAL_TEST_ONCE\'\n    archive_path = project / \'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip\'\n    if output.exists():\n        require((output / \'COMPLETED.json\').is_file() and archive_path.is_file(),\n                \'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.\')\n        inventory = json.loads((output / \'collection_manifest.json\').read_text(encoding=\'utf-8\'))[\'files\']\n        with zipfile.ZipFile(archive_path) as archive:\n            require(archive.testzip() is None, \'Saved Test evidence ZIP failed CRC verification.\')\n            for item in inventory:\n                raw = archive.read(item[\'file\'])\n                require(len(raw) == item[\'bytes\'] and hashlib.sha256(raw).hexdigest() == item[\'sha256\'],\n                        \'Saved Test archive bytes differ: \' + item[\'file\'])\n        print(\'Previously completed extension Test found; downloading its saved evidence without model execution.\')\n        files.download(str(archive_path))\n        return {\'already_completed\': True, \'zip\': str(archive_path)}\n\n    os.environ[\'WANDB_MODE\'] = \'disabled\'\n    os.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n    os.environ[\'HF_HUB_DISABLE_TELEMETRY\'] = \'1\'\n    torch_version, vision_version = installed(\'torch\'), installed(\'torchvision\')\n    require(torch_version and vision_version, \'A matching preinstalled PyTorch/torchvision runtime is required.\')\n    desired = {\'transformers\': \'5.22.1\', \'numpy\': \'2.4.6\', \'Pillow\': \'11.3.0\'}\n    changes = [name + \'==\' + value for name, value in desired.items() if installed(name) != value]\n    changes += [name for name in [\'requests\'] if installed(name) is None]\n    if changes:\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'torch==\' + torch_version,\n                        \'torchvision==\' + vision_version, *changes], check=True, timeout=360)\n    import numpy as np\n    import PIL\n    from PIL import Image\n    require(np.__version__ == \'2.4.6\' and PIL.__version__ == \'11.3.0\',\n            \'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.\')\n    require(installed(\'torch\') == torch_version and installed(\'torchvision\') == vision_version,\n            \'The preinstalled torch pair unexpectedly changed.\')\n\n    validation = project / \'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z\'\n    assets = validation / \'deployment/assets\'\n    frozen_val = json.loads((validation / \'frozen_operator_VAL_protocol.json\').read_text(encoding=\'utf-8\'))\n    manifest = json.loads((validation / \'deployment/assist_manifest.json\').read_text(encoding=\'utf-8\'))\n    require(digest(validation / \'frozen_operator_VAL_protocol.json\') == _EXT_TEST_ANCHORS[\'val_protocol\'],\n            \'The already-audited Validation protocol differs.\')\n    require(digest(assets / \'gallery_index.npz\') == _EXT_TEST_ANCHORS[\'gallery_index\'],\n            \'The fixed160-reference gallery differs.\')\n    require(digest(assets / \'gallery_meta.json\') == _EXT_TEST_ANCHORS[\'gallery_meta\'], \'Gallery identities differ.\')\n    require(hashlib.sha256(_EXT_TEST_SOURCES[\'extension_assist_models.py\'].encode()).hexdigest()\n            == frozen_val[\'source_files_sha256\'][\'extension_assist_models.py\'], \'The model helper source changed after Validation.\')\n    protected_relative = _EXT_TEST_PROTECTED_PATHS\n    protected = {name: project / relative for name, relative in protected_relative.items()}\n    before = {name: digest(path) for name, path in protected.items()}\n    require(before == frozen_val[\'original_project_files_sha256\'], \'An original protected project reference differs.\')\n    for relative, expected in manifest[\'assets_sha256\'].items():\n        require(digest(assets / relative) == expected, \'Validation gallery/crop asset differs: \' + relative)\n    print(\'Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...\')\n    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest[\'clip_revision\'])\n    resources = engine.runtime_metadata()\n    for key in [\'clip_revision\', \'clip_checkpoint_sha256\', \'resnet_checkpoint_sha256\']:\n        require(resources[key] == manifest[\'model_resources\'][key], \'Official assistance model resource differs: \' + key)\n    gallery_meta = json.loads((assets / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n    gallery_crops = []\n    for item in gallery_meta:\n        with Image.open(assets / item[\'crop_file\']) as image:\n            gallery_crops.append(image.convert(\'RGB\'))\n    recomputed = engine.embed_resnet(gallery_crops)\n    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)\n    require(bool(np.all(cosines >= 0.999)), \'Runtime reference embedding compatibility failed before Test.\')\n    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-7),\n            \'Runtime histogram preprocessing differs from the fixed Validation reference bank.\')\n    del gallery_crops, recomputed\n\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    protocol = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n        \'manifest_sha256\': before[\'split_manifest\'], \'per_class\': 50, \'seed\': 20261116,\n        \'sampling_design\': \'Up to50 eligible GT objects per material without replacement; all if fewer than50. Independent RNGseed20261116, before viewing model outputs.\',\n        \'target_maximum_query_objects\': 200, \'full_Test_partition_images\': 929,\n        \'scope\': \'Prespecified material-balanced GT-crop Test subset, not a full929-image or end-to-end system evaluation.\',\n        \'small_box_filter\': \'Normalized width>0.003 and height>0.003, eight-decimal readback from original conversion.\',\n        \'gallery_assets_path\': str(assets), \'gallery_objects\': 160, \'output_dir\': str(output),\n        \'validation_protocol_sha256\': _EXT_TEST_ANCHORS[\'val_protocol\'],\n        \'gallery_index_sha256\': _EXT_TEST_ANCHORS[\'gallery_index\'], \'gallery_metadata_sha256\': _EXT_TEST_ANCHORS[\'gallery_meta\'],\n        \'source_files_sha256\': {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()},\n        \'model_resources\': resources, \'class_names\': list(CLASS_NAMES), \'clip_prompts\': list(PLAIN_PROMPTS),\n        \'prompt_model_threshold_selection_on_Test\': False,\n        \'histogram_baseline\': \'PIL11.3 RGBresize64,8bins/channel,L2unit; retrievaldescendingargsort, classificationfirstargmax.\',\n        \'reference_compatibility_min_cosine_before_Test\': float(cosines.min()),\n        \'protected_original_files_before\': before, \'M4_M5_Test_model_calls\': 0,\n        \'new_training_or_optimizer_steps\': 0, \'once_only_policy\': \'Fixed folder; completed attempts are downloaded again without model execution; incomplete attempts require explicit recovery review.\'}\n    output.mkdir(parents=True, exist_ok=False)\n    save_new(output / \'frozen_protocol.json\', protocol)\n    protocol_sha = digest(output / \'frozen_protocol.json\')\n    save_new(output / \'STATUS.json\', {\'phase\': \'preparing_inputs_no_Test_model_calls_yet\', \'protocol_sha256\': protocol_sha})\n    code_dir = output / \'code\'\n    code_dir.mkdir()\n    for name, source in _EXT_TEST_SOURCES.items():\n        (code_dir / name).write_text(source, encoding=\'utf-8\', newline=\'\\n\')\n    print(\'Frozen protocol saved BEFORE source Test annotation access. Preparing only the selected Test crops...\')\n    preparation = prepare_extension_test_inputs(protocol, output)\n    records = preparation[\'queries\']\n    work = Path(preparation[\'work_dir\'])\n    save_new(output / \'query_provenance.json\', preparation)\n    shutil.copytree(work / \'test_queries\', output / \'test_queries\')\n    reference_dir = output / \'reference_gallery\'\n    reference_dir.mkdir()\n    shutil.copytree(assets / \'gallery\', reference_dir / \'gallery\')\n    shutil.copyfile(assets / \'gallery_index.npz\', reference_dir / \'gallery_index.npz\')\n    shutil.copyfile(assets / \'gallery_meta.json\', reference_dir / \'gallery_meta.json\')\n    batch_dir = output / \'inference_batches\'\n    batch_dir.mkdir()\n    (output / \'STATUS.json\').write_text(json.dumps({\'phase\': \'Test_inference_started\',\n        \'protocol_sha256\': protocol_sha, \'query_objects\': len(records)}, indent=2), encoding=\'utf-8\')\n    q_parts, h_parts, l_parts = [], [], []\n    print(\'One fixed Test evaluation:\', len(records), \'annotated crops; no detector/segmenter calls.\')\n    for start in range(0, len(records), 8):\n        crops = []\n        for item in records[start:start + 8]:\n            path = output / item[\'crop_file\']\n            require(digest(path) == item[\'crop_sha256\'], \'Selected Test crop changed.\')\n            with Image.open(path) as image:\n                crops.append(image.convert(\'RGB\'))\n        q, h, logits = engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8)\n        path = batch_dir / f\'batch_{start // 8:03d}.npz\'\n        with path.open(\'xb\') as stream:\n            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)),\n                                embeddings=q, histograms=h, clip_logits=logits)\n        q_parts.append(q); h_parts.append(h); l_parts.append(logits)\n        print(\'Recorded Test crops:\', start + len(crops), \'/\', len(records))\n    q, h, logits = np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts)\n    labels = np.asarray([item[\'class_id\'] for item in records], dtype=np.int64)\n    gy = engine.gallery_labels\n    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)\n    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)\n    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]\n    clip_pred = logits.argmax(axis=1)\n    comparison = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n        \'sample_objects\': len(records), \'gallery_objects\': 160,\n        \'sample_counts\': preparation[\'provenance\'][\'counts_per_class\'],\n        \'unique_source_images\': preparation[\'provenance\'][\'unique_source_images\'],\n        \'visual_search_resnet\': search, \'visual_search_histogram\': baseline_search,\n        \'clip_plain\': classification_metrics(labels, clip_pred),\n        \'histogram_classification_baseline\': classification_metrics(labels, hist_pred),\n        \'frozen_protocol_sha256\': protocol_sha,\n        \'limits\': [\'Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.\',\n                   \'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.\',\n                   \'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.\']}\n    save_new(output / \'comparison_TEST.json\', comparison)\n    with (output / \'test_arrays.npz\').open(\'xb\') as stream:\n        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits,\n                            query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank,\n                            histogram_class_predictions=hist_pred)\n    rows = [{\'query_index\': index, \'crop_file\': item[\'crop_file\'], \'source_export_name\': item[\'export_name\'],\n             \'annotation_id\': item[\'annotation_id\'], \'ground_truth_class_id\': int(labels[index]),\n             \'resnet_top1_class_id\': int(gy[rank[index, 0]]),\n             \'resnet_top5_gallery_indices\': [int(value) for value in rank[index, :5]],\n             \'clip_class_id\': int(clip_pred[index]), \'clip_logits\': logits[index].tolist(),\n             \'histogram_class_id\': int(hist_pred[index])} for index, item in enumerate(records)]\n    save_new(output / \'per_query_TEST.json\', rows)\n    after = {name: digest(path) for name, path in protected.items()}\n    require(before == after and digest(output / \'frozen_protocol.json\') == protocol_sha,\n            \'An original file or frozen Test protocol changed.\')\n    save_new(output / \'protected_original_files.json\', {\'before\': before, \'after\': after,\n        \'unchanged\': True, \'original_M4_M5_Test_calls\': 0, \'training_or_optimizer_steps\': 0})\n    save_new(output / \'COMPLETED.json\', {\'status\': \'completed_once\', \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'frozen_protocol_sha256\': protocol_sha, \'comparison_sha256\': digest(output / \'comparison_TEST.json\'),\n        \'zip_name\': archive_path.name, \'M4_M5_Test_reevaluated\': False})\n    (output / \'STATUS.json\').write_text(json.dumps({\'phase\': \'completed_once\', \'protocol_sha256\': protocol_sha}), encoding=\'utf-8\')\n    inventory = [{\'file\': path.relative_to(output).as_posix(), \'bytes\': path.stat().st_size, \'sha256\': digest(path)}\n                 for path in sorted(output.rglob(\'*\')) if path.is_file()]\n    save_new(output / \'collection_manifest.json\', {\'files\': inventory})\n    with zipfile.ZipFile(archive_path, \'x\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file():\n                archive.write(path, path.relative_to(output).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        require(archive.testzip() is None, \'Test evidence ZIP CRC check failed.\')\n        for item in inventory:\n            raw = archive.read(item[\'file\'])\n            require(len(raw) == item[\'bytes\'] and hashlib.sha256(raw).hexdigest() == item[\'sha256\'], \'Packaged Test evidence differs.\')\n    print(json.dumps(comparison, indent=2))\n    print(\'Extension Test complete. No M4/M5 Test evaluation or training was run.\')\n    print(\'Saved original results unchanged. ZIP:\', archive_path)\n    files.download(str(archive_path))\n    return {\'zip\': str(archive_path), \'output_dir\': str(output), \'comparison\': comparison}\n\n\n_EXTENSION_TEST_RESULT = run_extensions_test_once()\n'}
_EXT_TEST_PROTECTED_PATHS = {'split_manifest': 'data_protocol/ef6b35d2ba7f/split_manifest.csv', 'validation_inventory': 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json', 'gallery_index': 'open_world/visual_search/gallery_index_train.npz', 'visual_search_result': 'open_world/visual_search/comparison_VAL.json', 'clip_result': 'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv', 'service_source': 'deployment/http_service/service.py', 'service_manifest': 'deployment/http_service/release_manifest.json', 'http_detector': 'deployment/http_service/best_det.pt', 'http_segmenter': 'deployment/http_service/best_seg.pt', 'source_detector': 'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt', 'source_segmenter': 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt', 'test_protocol': 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json', 'test_results': 'evaluation/FINAL_TEST_ONCE/results.json'}
_EXT_TEST_ANCHORS = {'val_protocol': '7d2e20f397e4ee0d13485b6998dbda1acc113a88a8f1b9b52ba6ba2d9bbc1b7d', 'gallery_index': '4e2a9cc509e2c54f99186a04019d9f82b3d82096e2e7ff4532ab0017b4571442', 'gallery_meta': '889381006555a967d65f4af69eaa895a6356af48dd93a6d43e1311440691a564'}
_EXT_RECOVERY_FUNCTION_SOURCE = "def recover_extension_test_before_inference():\n    from datetime import datetime, timezone\n    from importlib.metadata import version, PackageNotFoundError\n    from pathlib import Path\n    import hashlib\n    import json\n    import os\n    import shutil\n    import subprocess\n    import sys\n    import zipfile\n    from google.colab import drive, files\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def digest(path):\n        value = hashlib.sha256()\n        with Path(path).open('rb') as stream:\n            for chunk in iter(lambda: stream.read(4 * 1024 ** 2), b''):\n                value.update(chunk)\n        return value.hexdigest()\n\n    def save_new(path, value):\n        with Path(path).open('x', encoding='utf-8', newline='\\n') as stream:\n            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n            stream.write('\\n')\n\n    def installed(name):\n        try:\n            return version(name)\n        except PackageNotFoundError:\n            return None\n    drive.mount('/content/drive')\n    project = Path('/content/drive/MyDrive/Project9_Phase2')\n    output = project / 'evaluation/EXTENSIONS_FINAL_TEST_ONCE'\n    archive_path = project / 'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip'\n    if (output / 'COMPLETED.json').is_file():\n        require((output / 'COMPLETED.json').is_file() and archive_path.is_file(), 'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.')\n        inventory = json.loads((output / 'collection_manifest.json').read_text(encoding='utf-8'))['files']\n        with zipfile.ZipFile(archive_path) as archive:\n            require(archive.testzip() is None, 'Saved Test evidence ZIP failed CRC verification.')\n            for item in inventory:\n                raw = archive.read(item['file'])\n                require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Saved Test archive bytes differ: ' + item['file'])\n        print('Previously completed extension Test found; downloading its saved evidence without model execution.')\n        files.download(str(archive_path))\n        return {'already_completed': True, 'zip': str(archive_path)}\n    require(output.is_dir(), 'The original reserved Test folder is missing. Stop and share this output.')\n    protocol_file = output / 'frozen_protocol.json'\n    status_file = output / 'STATUS.json'\n    require(protocol_file.is_file() and status_file.is_file(), 'Original frozen protocol/status is missing.')\n    protocol_bytes = protocol_file.read_bytes()\n    protocol_sha = hashlib.sha256(protocol_bytes).hexdigest()\n    protocol = json.loads(protocol_bytes.decode('utf-8'))\n    status = json.loads(status_file.read_text(encoding='utf-8'))\n    require(status == {'phase': 'preparing_inputs_no_Test_model_calls_yet', 'protocol_sha256': protocol_sha}, 'Recovery is restricted to the recorded failure BEFORE Test inference; status/hash differs.')\n    blocked = ['query_provenance.json', 'test_queries', 'reference_gallery', 'inference_batches', 'test_arrays.npz', 'comparison_TEST.json', 'per_query_TEST.json', 'COMPLETED.json', 'collection_manifest.json']\n    require(not any(((output / name).exists() for name in blocked)) and (not archive_path.exists()), 'Existing Test preparation/results found. Do not evaluate again; share this output for separate recovery review.')\n    require(protocol.get('split') == 'test' and protocol.get('per_class') == 50 and (protocol.get('seed') == 20261116) and (protocol.get('target_maximum_query_objects') == 200) and (protocol.get('full_Test_partition_images') == 929), 'Frozen Test sampling parameters differ.')\n    require(protocol.get('output_dir') == str(output) and protocol.get('gallery_objects') == 160 and (protocol.get('prompt_model_threshold_selection_on_Test') is False) and (protocol.get('M4_M5_Test_model_calls') == 0) and (protocol.get('new_training_or_optimizer_steps') == 0), 'Frozen Test scope/settings differ.')\n    expected_sources = {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()}\n    require(protocol.get('source_files_sha256') == expected_sources, 'Frozen source references differ from the original supplied cell.')\n    for name, expected in expected_sources.items():\n        require(digest(output / 'code' / name) == expected, 'Original saved code bytes differ: ' + name)\n    require({path.name for path in (output / 'code').iterdir()} == set(expected_sources), 'Unexpected files appeared in the original frozen code folder.')\n    memory_diagnostic = {'available': False, 'note': 'The saved protocol bytes, not traceback memory, are the recovery authority.'}\n    last_error = getattr(sys, 'last_exc', None) or getattr(sys, 'last_value', None)\n    trace = getattr(last_error, '__traceback__', None)\n    while trace is not None:\n        frame = trace.tb_frame\n        if frame.f_code.co_name == 'prepare_extension_test_inputs' and 'frozen_protocol' in frame.f_locals:\n            argument = frame.f_locals['frozen_protocol']\n            try:\n                normalized = json.loads(json.dumps(argument, ensure_ascii=False, allow_nan=False))\n                memory_diagnostic.update({'available': True, 'python_object_equal': argument == protocol, 'json_normalized_equal': normalized == protocol})\n                if normalized != protocol:\n                    raise RuntimeError('Original in-memory protocol normalizes differently from the saved frozen file.')\n            except (TypeError, ValueError) as error:\n                raise RuntimeError('Original protocol memory is inconsistent with its saved JSON.') from error\n            break\n        trace = trace.tb_next\n    print('Recovery gate passed: original protocol hash matches, status is before Test, no Test batches/results exist.')\n    os.environ['WANDB_MODE'] = 'disabled'\n    os.environ['TOKENIZERS_PARALLELISM'] = 'false'\n    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'\n    torch_version, vision_version = (installed('torch'), installed('torchvision'))\n    require(torch_version and vision_version, 'A matching preinstalled PyTorch/torchvision runtime is required.')\n    desired = {'transformers': '5.22.1', 'numpy': '2.4.6', 'Pillow': '11.3.0'}\n    changes = [name + '==' + value for name, value in desired.items() if installed(name) != value]\n    changes += [name for name in ['requests'] if installed(name) is None]\n    if changes:\n        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==' + torch_version, 'torchvision==' + vision_version, *changes], check=True, timeout=360)\n    import numpy as np\n    import PIL\n    from PIL import Image\n    require(np.__version__ == '2.4.6' and PIL.__version__ == '11.3.0', 'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.')\n    require(installed('torch') == torch_version and installed('torchvision') == vision_version, 'The preinstalled torch pair unexpectedly changed.')\n    validation = project / 'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z'\n    assets = validation / 'deployment/assets'\n    frozen_val = json.loads((validation / 'frozen_operator_VAL_protocol.json').read_text(encoding='utf-8'))\n    manifest = json.loads((validation / 'deployment/assist_manifest.json').read_text(encoding='utf-8'))\n    require(digest(validation / 'frozen_operator_VAL_protocol.json') == _EXT_TEST_ANCHORS['val_protocol'], 'The already-audited Validation protocol differs.')\n    require(digest(assets / 'gallery_index.npz') == _EXT_TEST_ANCHORS['gallery_index'], 'The fixed160-reference gallery differs.')\n    require(digest(assets / 'gallery_meta.json') == _EXT_TEST_ANCHORS['gallery_meta'], 'Gallery identities differ.')\n    require(hashlib.sha256(_EXT_TEST_SOURCES['extension_assist_models.py'].encode()).hexdigest() == frozen_val['source_files_sha256']['extension_assist_models.py'], 'The model helper source changed after Validation.')\n    protected_relative = _EXT_TEST_PROTECTED_PATHS\n    protected = {name: project / relative for name, relative in protected_relative.items()}\n    before = {name: digest(path) for name, path in protected.items()}\n    require(before == frozen_val['original_project_files_sha256'], 'An original protected project reference differs.')\n    for relative, expected in manifest['assets_sha256'].items():\n        require(digest(assets / relative) == expected, 'Validation gallery/crop asset differs: ' + relative)\n    print('Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...')\n    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest['clip_revision'])\n    resources = engine.runtime_metadata()\n    for key in ['clip_revision', 'clip_checkpoint_sha256', 'resnet_checkpoint_sha256']:\n        require(resources[key] == manifest['model_resources'][key], 'Official assistance model resource differs: ' + key)\n    gallery_meta = json.loads((assets / 'gallery_meta.json').read_text(encoding='utf-8'))\n    gallery_crops = []\n    for item in gallery_meta:\n        with Image.open(assets / item['crop_file']) as image:\n            gallery_crops.append(image.convert('RGB'))\n    recomputed = engine.embed_resnet(gallery_crops)\n    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)\n    require(bool(np.all(cosines >= 0.999)), 'Runtime reference embedding compatibility failed before Test.')\n    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-07), 'Runtime histogram preprocessing differs from the fixed Validation reference bank.')\n    del gallery_crops, recomputed\n    require(protocol['manifest_sha256'] == before['split_manifest'] and protocol['protected_original_files_before'] == before, 'Original protected references differ from frozen Test protocol.')\n    require(protocol['gallery_assets_path'] == str(assets) and protocol['validation_protocol_sha256'] == _EXT_TEST_ANCHORS['val_protocol'] and (protocol['gallery_index_sha256'] == _EXT_TEST_ANCHORS['gallery_index']) and (protocol['gallery_metadata_sha256'] == _EXT_TEST_ANCHORS['gallery_meta']), 'Frozen gallery/Validation anchors differ.')\n    require(protocol['class_names'] == list(CLASS_NAMES) and protocol['clip_prompts'] == list(PLAIN_PROMPTS), 'Frozen class names/prompts differ.')\n    current_resources = json.loads(json.dumps(resources, ensure_ascii=False, allow_nan=False))\n    require(current_resources == protocol['model_resources'], 'Current runtime/model processor metadata differs from frozen protocol; stop before Test.')\n    require(digest(protocol_file) == protocol_sha, 'The original frozen protocol changed during resource checks.')\n    recovery_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')\n    recovery_dir = output / ('recovery_before_inference_' + recovery_stamp)\n    recovery_dir.mkdir(exist_ok=False)\n    with (recovery_dir / 'recovery_function.py').open('x', encoding='utf-8', newline='\\n') as stream:\n        stream.write(_EXT_RECOVERY_FUNCTION_SOURCE)\n    save_new(recovery_dir / 'recovery_receipt.json', {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'reason': 'Original direct Python-object versus persisted-JSON equality failed before Test annotation access.', 'fix': 'Pass the original SHA-verified decoded JSON object to the unchanged original preparation function.', 'frozen_protocol_sha256_unchanged': protocol_sha, 'original_code_sha256_unchanged': expected_sources, 'recovery_function_sha256': hashlib.sha256(_EXT_RECOVERY_FUNCTION_SOURCE.encode()).hexdigest(), 'status_before_recovery': status, 'memory_diagnostic': memory_diagnostic, 'sampling_models_prompts_thresholds_and_preprocessing_changed': False, 'previous_Test_batches_or_results_present': False, 'current_model_resources_json_normalized_match': True, 'original_M4_M5_Test_calls': 0})\n    print('Continuing the same frozen attempt with the original protocol JSON; no frozen file was rewritten.')\n    preparation = prepare_extension_test_inputs(protocol, output)\n    records = preparation['queries']\n    work = Path(preparation['work_dir'])\n    save_new(output / 'query_provenance.json', preparation)\n    shutil.copytree(work / 'test_queries', output / 'test_queries')\n    reference_dir = output / 'reference_gallery'\n    reference_dir.mkdir()\n    shutil.copytree(assets / 'gallery', reference_dir / 'gallery')\n    shutil.copyfile(assets / 'gallery_index.npz', reference_dir / 'gallery_index.npz')\n    shutil.copyfile(assets / 'gallery_meta.json', reference_dir / 'gallery_meta.json')\n    batch_dir = output / 'inference_batches'\n    batch_dir.mkdir()\n    (output / 'STATUS.json').write_text(json.dumps({'phase': 'Test_inference_started', 'protocol_sha256': protocol_sha, 'query_objects': len(records)}, indent=2), encoding='utf-8')\n    q_parts, h_parts, l_parts = ([], [], [])\n    print('One fixed Test evaluation:', len(records), 'annotated crops; no detector/segmenter calls.')\n    for start in range(0, len(records), 8):\n        crops = []\n        for item in records[start:start + 8]:\n            path = output / item['crop_file']\n            require(digest(path) == item['crop_sha256'], 'Selected Test crop changed.')\n            with Image.open(path) as image:\n                crops.append(image.convert('RGB'))\n        q, h, logits = (engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8))\n        path = batch_dir / f'batch_{start // 8:03d}.npz'\n        with path.open('xb') as stream:\n            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)), embeddings=q, histograms=h, clip_logits=logits)\n        q_parts.append(q)\n        h_parts.append(h)\n        l_parts.append(logits)\n        print('Recorded Test crops:', start + len(crops), '/', len(records))\n    q, h, logits = (np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts))\n    labels = np.asarray([item['class_id'] for item in records], dtype=np.int64)\n    gy = engine.gallery_labels\n    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)\n    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)\n    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]\n    clip_pred = logits.argmax(axis=1)\n    comparison = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test', 'sample_objects': len(records), 'gallery_objects': 160, 'sample_counts': preparation['provenance']['counts_per_class'], 'unique_source_images': preparation['provenance']['unique_source_images'], 'visual_search_resnet': search, 'visual_search_histogram': baseline_search, 'clip_plain': classification_metrics(labels, clip_pred), 'histogram_classification_baseline': classification_metrics(labels, hist_pred), 'frozen_protocol_sha256': protocol_sha, 'limits': ['Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.', 'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.', 'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.']}\n    save_new(output / 'comparison_TEST.json', comparison)\n    with (output / 'test_arrays.npz').open('xb') as stream:\n        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits, query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank, histogram_class_predictions=hist_pred)\n    rows = [{'query_index': index, 'crop_file': item['crop_file'], 'source_export_name': item['export_name'], 'annotation_id': item['annotation_id'], 'ground_truth_class_id': int(labels[index]), 'resnet_top1_class_id': int(gy[rank[index, 0]]), 'resnet_top5_gallery_indices': [int(value) for value in rank[index, :5]], 'clip_class_id': int(clip_pred[index]), 'clip_logits': logits[index].tolist(), 'histogram_class_id': int(hist_pred[index])} for index, item in enumerate(records)]\n    save_new(output / 'per_query_TEST.json', rows)\n    after = {name: digest(path) for name, path in protected.items()}\n    require(before == after and digest(output / 'frozen_protocol.json') == protocol_sha, 'An original file or frozen Test protocol changed.')\n    save_new(output / 'protected_original_files.json', {'before': before, 'after': after, 'unchanged': True, 'original_M4_M5_Test_calls': 0, 'training_or_optimizer_steps': 0})\n    save_new(output / 'COMPLETED.json', {'status': 'completed_once', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'frozen_protocol_sha256': protocol_sha, 'comparison_sha256': digest(output / 'comparison_TEST.json'), 'zip_name': archive_path.name, 'M4_M5_Test_reevaluated': False})\n    (output / 'STATUS.json').write_text(json.dumps({'phase': 'completed_once', 'protocol_sha256': protocol_sha}), encoding='utf-8')\n    inventory = [{'file': path.relative_to(output).as_posix(), 'bytes': path.stat().st_size, 'sha256': digest(path)} for path in sorted(output.rglob('*')) if path.is_file()]\n    save_new(output / 'collection_manifest.json', {'files': inventory})\n    with zipfile.ZipFile(archive_path, 'x', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob('*')):\n            if path.is_file():\n                archive.write(path, path.relative_to(output).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        require(archive.testzip() is None, 'Test evidence ZIP CRC check failed.')\n        for item in inventory:\n            raw = archive.read(item['file'])\n            require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Packaged Test evidence differs.')\n    print(json.dumps(comparison, indent=2))\n    print('Extension Test complete. No M4/M5 Test evaluation or training was run.')\n    print('Saved original results unchanged. ZIP:', archive_path)\n    files.download(str(archive_path))\n    return {'zip': str(archive_path), 'output_dir': str(output), 'comparison': comparison}\n"

"""CPU models and numeric helpers for the optional operator-assistance stage.

Importing this module performs no model loading, downloads or inference.
Model construction is explicit; all predictions are advisory and do not alter
the original detector/segmenter response or introduce a rejection threshold.
"""

CLASS_NAMES = ('cardboard', 'metal', 'rigid_plastic', 'soft_plastic')
CLIP_MATERIAL_NAMES = ('cardboard', 'metal', 'rigid plastic', 'soft plastic')
PLAIN_PROMPTS = tuple('a photo of ' + name for name in CLIP_MATERIAL_NAMES)
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'


def _assist_require(condition, message):
    if not condition:
        raise RuntimeError(message)


def _assist_sha256(path):
    from pathlib import Path
    import hashlib
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()


def unit(values):
    """The original float32 L2 normalization, including its 1e-10 guard."""
    import numpy as np
    array = np.asarray(values, dtype=np.float32)
    _assist_require(array.ndim == 2 and np.isfinite(array).all(), 'Expected finite 2D features.')
    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)


def rgb_hist(crops):
    """Original 24-bin RGB histogram baseline; crops must be PIL images."""
    import numpy as np
    rows = []
    for crop in crops:
        array = np.asarray(crop.convert('RGB').resize((64, 64)), dtype=np.uint8)
        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,
                                                range=(0, 256))[0] for channel in range(3)]))
    _assist_require(bool(rows), 'At least one crop is required.')
    return unit(np.asarray(rows, dtype=np.float32))


def classification_metrics(true_labels, predicted_labels):
    """Four-material accuracy and macro metrics with zero-division set to zero."""
    import numpy as np
    truth = np.asarray(true_labels)
    prediction = np.asarray(predicted_labels)
    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape
                    and truth.size > 0, 'Class arrays must be nonempty, aligned 1D arrays.')
    _assist_require(np.issubdtype(truth.dtype, np.integer)
                    and np.issubdtype(prediction.dtype, np.integer), 'Class IDs must be integers.')
    _assist_require(((truth >= 0) & (truth < 4)).all()
                    and ((prediction >= 0) & (prediction < 4)).all(), 'Unknown material class ID.')
    matrix = np.zeros((4, 4), dtype=np.int64)
    np.add.at(matrix, (truth, prediction), 1)
    tp = np.diag(matrix).astype(np.float64)
    predicted_support = matrix.sum(axis=0)
    support = matrix.sum(axis=1)
    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)
    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)
    return {
        'accuracy': float(np.mean(truth == prediction)), 'macro_f1': float(f1.mean()),
        'macro_precision': float(precision.mean()), 'macro_recall': float(recall.mean()),
        'count': int(truth.size), 'confusion_matrix_true_rows_predicted_columns': matrix.tolist(),
        'per_class': [{'class_id': index, 'material': name, 'support': int(support[index]),
                       'precision': float(precision[index]), 'recall': float(recall[index]),
                       'f1': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],
    }


def retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):
    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""
    import numpy as np
    gallery = np.asarray(gallery_features, dtype=np.float32)
    queries = np.asarray(query_features, dtype=np.float32)
    gallery_y = np.asarray(gallery_labels)
    query_y = np.asarray(query_labels)
    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]
                    and len(gallery) >= 5 and len(queries) > 0, 'Invalid gallery/query feature shape.')
    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)
                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), 'Misaligned or nonfinite retrieval inputs.')
    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]
    top1 = gallery_y[rankings[:, 0]]
    top5 = gallery_y[rankings[:, :5]]
    return ({'top1_accuracy': float(np.mean(top1 == query_y)),
             'hit_at_5': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),
             'precision_at_5': float(np.mean(top5 == query_y[:, None]))}, rankings)


class OperatorAssistModels:
    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.

    ``assets_dir`` may be None while building a new gallery. Then call
    ``load_gallery`` after writing these files:

    - gallery_index.npz: float32 embeddings [160,512], float32 histograms
      [160,24], int64 labels [160].
    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,
      each with class_id and crop_file="gallery/g_000.png" etc.
    - the corresponding 160 PNG reference crops.

    local_only=True requires the ResNet checkpoint and the explicit immutable
    CLIP revision to be present already; it performs no network request.
    """

    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):
        from pathlib import Path
        from importlib.metadata import version
        import re
        import torch
        import torch.nn as nn
        from torchvision.models import resnet18, ResNet18_Weights
        from transformers import CLIPModel, CLIPProcessor
        from huggingface_hub import hf_hub_download, model_info

        self.torch = torch
        self.assets_dir = None
        self.gallery_embeddings = None
        self.gallery_histograms = None
        self.gallery_labels = None
        self.gallery_meta = None
        self.local_only = bool(local_only)
        torch.set_num_threads(1)
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            _assist_require(torch.get_num_interop_threads() == 1,
                            'Torch inter-op threads are already initialized differently; use a fresh process.')

        weights = ResNet18_Weights.IMAGENET1K_V1
        self.resnet_transform = weights.transforms()
        resnet_file = Path(torch.hub.get_dir()) / 'checkpoints/resnet18-f37072fd.pth'
        if self.local_only:
            _assist_require(resnet_file.is_file(), 'Cached ResNet18 checkpoint is required for local-only serving.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'Cached ResNet18 checkpoint checksum differs.')
            encoder = resnet18(weights=None)
            state = torch.load(resnet_file, map_location='cpu', weights_only=True)
            encoder.load_state_dict(state, strict=True)
            del state
        else:
            encoder = resnet18(weights=weights, progress=False)
            _assist_require(resnet_file.is_file(), 'Downloaded ResNet18 checkpoint was not found at its standard cache path.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'ResNet18 checkpoint checksum differs from its official filename prefix.')
        encoder.fc = nn.Identity()
        self.encoder = encoder.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.encoder.parameters():
            parameter.requires_grad_(False)

        hub_info = None
        if clip_revision is None:
            _assist_require(not self.local_only, 'An explicit cached CLIP revision is required in local-only mode.')
            hub_info = model_info(CLIP_MODEL_ID, revision='main', files_metadata=True, token=False)
            clip_revision = hub_info.sha
        _assist_require(isinstance(clip_revision, str)
                        and re.fullmatch(r'[0-9a-f]{40}|[0-9a-f]{64}', clip_revision) is not None,
                        'CLIP revision must be an immutable Hub commit hash.')
        self.clip_revision = clip_revision
        common = {'revision': self.clip_revision, 'local_files_only': self.local_only, 'token': False}
        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,
                                             trust_remote_code=False, **common)
        self.clip = self.clip.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.clip.parameters():
            parameter.requires_grad_(False)
        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,
                                                       trust_remote_code=False, **common)
        # The loader above explicitly selects the bin checkpoint. This cache-only
        # lookup records its actual bytes without triggering a second download.
        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename='pytorch_model.bin',
                                        revision=self.clip_revision, local_files_only=True, token=False))
        clip_sha = _assist_sha256(clip_file)
        expected_clip_sha = None
        if hub_info is not None:
            for sibling in hub_info.siblings or []:
                if sibling.rfilename == 'pytorch_model.bin':
                    lfs = getattr(sibling, 'lfs', None)
                    expected_clip_sha = (lfs.get('sha256') if isinstance(lfs, dict)
                                         else getattr(lfs, 'sha256', None))
                    break
        if expected_clip_sha is not None:
            _assist_require(clip_sha == expected_clip_sha, 'CLIP binary differs from the Hub LFS SHA256.')
        config_commit = getattr(self.clip.config, '_commit_hash', None)
        if config_commit is not None:
            _assist_require(config_commit == self.clip_revision, 'Loaded CLIP configuration revision differs.')
        self._provenance = {
            'resnet_weights': 'ResNet18_Weights.IMAGENET1K_V1',
            'resnet_weights_url': weights.url, 'resnet_checkpoint_path': str(resnet_file),
            'resnet_checkpoint_sha256': resnet_sha, 'resnet_transform': repr(self.resnet_transform),
            'clip_model_id': CLIP_MODEL_ID, 'clip_revision': self.clip_revision,
            'clip_config_commit_hash': config_commit, 'clip_checkpoint_path': str(clip_file),
            'clip_checkpoint_sha256': clip_sha, 'clip_hub_lfs_sha256': expected_clip_sha,
            'clip_checkpoint_url': 'https://huggingface.co/' + CLIP_MODEL_ID + '/resolve/'
                                   + self.clip_revision + '/pytorch_model.bin',
            'clip_processor_class': type(self.processor).__name__,
            'clip_image_processor_class': type(self.processor.image_processor).__name__,
            'clip_image_processor_config': self.processor.image_processor.to_dict(),
            'clip_attention_implementation': getattr(self.clip.config, '_attn_implementation', None),
            'versions': {name: version(name) for name in ['torch', 'torchvision', 'transformers',
                                                         'huggingface-hub', 'numpy', 'Pillow']},
        }
        self._assert_cpu_frozen()
        if assets_dir is not None:
            self.load_gallery(assets_dir)

    def _assert_cpu_frozen(self):
        for name, model in [('resnet', self.encoder), ('clip', self.clip)]:
            _assist_require(model.training is False, name + ' must be in eval mode.')
            _assist_require(all(parameter.device.type == 'cpu' and not parameter.requires_grad
                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)
                                for parameter in model.parameters()), name + ' must be frozen CPU FP32.')

    def embed_resnet(self, crops, batch=16):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid ResNet crop batch.')
        self._assert_cpu_frozen()
        vectors = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                tensor = self.torch.stack([self.resnet_transform(crop.convert('RGB'))
                                          for crop in crops[start:start + batch]])
                output = self.encoder(tensor.to(device='cpu', dtype=self.torch.float32))
                vectors.append(output.detach().cpu().numpy())
        array = np.concatenate(vectors).astype(np.float32)
        _assist_require(array.shape == (len(crops), 512), 'Unexpected ResNet feature dimensions.')
        return unit(array)

    def score_clip(self, crops, batch=8):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid CLIP crop batch.')
        self._assert_cpu_frozen()
        logits = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                inputs = self.processor(text=list(PLAIN_PROMPTS),
                                        images=[crop.convert('RGB') for crop in crops[start:start + batch]],
                                        return_tensors='pt', padding=True)
                inputs = {key: value.to(device='cpu', dtype=self.torch.float32)
                          if key == 'pixel_values' else value.to('cpu') for key, value in inputs.items()}
                output = self.clip(**inputs)
                logits.append(output.logits_per_image.detach().cpu().numpy())
        array = np.concatenate(logits).astype(np.float32)
        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), 'Unexpected or nonfinite CLIP logits.')
        return array

    def rgb_hist(self, crops):
        return rgb_hist(crops)

    def load_gallery(self, assets_dir):
        from pathlib import Path, PurePosixPath
        import json
        import numpy as np
        directory = Path(assets_dir).resolve()
        with np.load(directory / 'gallery_index.npz', allow_pickle=False) as bank:
            _assist_require(set(bank.files) == {'embeddings', 'histograms', 'labels'}, 'Unexpected assistance gallery keys.')
            embeddings = bank['embeddings'].copy()
            histograms = bank['histograms'].copy()
            labels = bank['labels'].copy()
        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32
                        and histograms.shape == (160, 24) and histograms.dtype == np.float32
                        and labels.shape == (160,) and labels.dtype == np.int64,
                        'Assistance gallery array shapes/dtypes differ.')
        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()
                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),
                        'Assistance gallery features/classes differ.')
        for name, values in [('embedding', embeddings), ('histogram', histograms)]:
            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),
                            'Unnormalized gallery ' + name + ' features.')
        metadata = json.loads((directory / 'gallery_meta.json').read_text(encoding='utf-8'))
        _assist_require(isinstance(metadata, list) and len(metadata) == 160, 'Expected 160 gallery metadata records.')
        for index, item in enumerate(metadata):
            expected_file = 'gallery/g_' + str(index).zfill(3) + '.png'
            _assist_require(isinstance(item, dict) and item.get('class_id') == int(labels[index])
                            and item.get('crop_file') == expected_file, 'Gallery metadata/array alignment differs.')
            crop = PurePosixPath(item['crop_file'])
            path = (directory / crop).resolve()
            _assist_require(not crop.is_absolute() and '..' not in crop.parts
                            and path.is_relative_to(directory) and path.is_file(), 'Missing or unsafe reference crop.')
        self.assets_dir = directory
        self.gallery_embeddings = embeddings
        self.gallery_histograms = histograms
        self.gallery_labels = labels
        self.gallery_meta = metadata
        return self

    def query_crop(self, crop):
        import numpy as np
        _assist_require(self.gallery_embeddings is not None, 'Load the assistance gallery first.')
        feature = self.embed_resnet([crop], batch=1)
        similarities = (feature @ self.gallery_embeddings.T)[0]
        ranking = similarities.argsort()[::-1]
        references = []
        for raw_index in ranking[:5]:
            index = int(raw_index)
            class_id = int(self.gallery_labels[index])
            references.append({'gallery_index': index, 'class_id': class_id,
                               'material': CLASS_NAMES[class_id], 'similarity': float(similarities[index]),
                               'crop_file': self.gallery_meta[index]['crop_file']})
        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)
        relative = np.exp(clip_logits - clip_logits.max())
        relative /= relative.sum()
        clip_class = int(clip_logits.argmax())
        histogram = rgb_hist([crop])
        hist_similarities = (histogram @ self.gallery_histograms.T)[0]
        hist_index = int(hist_similarities.argmax())
        hist_class = int(self.gallery_labels[hist_index])
        return {
            'visual_search': {'predicted_class_id': references[0]['class_id'],
                              'predicted_material': references[0]['material'], 'references': references,
                              'score_type': 'Cosine similarity to frozen Train reference crops; not calibrated confidence.'},
            'clip': {'predicted_class_id': clip_class, 'predicted_material': CLASS_NAMES[clip_class],
                     'logits': clip_logits.tolist(), 'relative_scores': relative.tolist(),
                     'prompts': list(PLAIN_PROMPTS),
                     'score_type': 'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.'},
            'histogram': {'predicted_class_id': hist_class, 'predicted_material': CLASS_NAMES[hist_class],
                          'reference_gallery_index': hist_index, 'similarity': float(hist_similarities[hist_index]),
                          'crop_file': self.gallery_meta[hist_index]['crop_file'],
                          'score_type': 'Nearest Train RGB histogram by cosine similarity; argmax tie policy.'},
        }

    def runtime_metadata(self):
        self._assert_cpu_frozen()
        return {
            **self._provenance, 'clip_revision': self.clip_revision, 'device': 'cpu', 'precision': 'float32',
            'resnet_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()
                                              if parameter.is_floating_point()}),
            'clip_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.clip.parameters()
                                            if parameter.is_floating_point()}),
            'torch_num_threads': int(self.torch.get_num_threads()),
            'torch_num_interop_threads': int(self.torch.get_num_interop_threads()),
            'all_parameters_frozen': True, 'plain_prompts': list(PLAIN_PROMPTS),
            'gallery_loaded': self.gallery_embeddings is not None,
            'gallery_size': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,
            'model_training_or_optimization': False,
            'decision_role': 'Advisory references and material suggestion; original detector outputs are not overwritten.',
        }


"""Prepare a prespecified material-balanced crop subset of the frozen Test split."""


def prepare_extension_test_inputs(frozen_protocol, evidence_dir):
    from collections import Counter, OrderedDict, defaultdict
    from datetime import datetime, timezone
    from pathlib import Path, PurePosixPath
    import csv
    import hashlib
    import io
    import json
    import math
    import re
    import time
    import zipfile
    import zlib
    import numpy as np
    import requests
    from PIL import Image

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()

    evidence_dir = Path(evidence_dir)
    protocol_file = evidence_dir / 'frozen_protocol.json'
    require(protocol_file.is_file() and json.loads(protocol_file.read_text(encoding='utf-8')) == frozen_protocol,
            'A byte-hashed frozen protocol must exist before opening Test annotations.')
    require(frozen_protocol['split'] == 'test' and frozen_protocol['per_class'] == 50
            and frozen_protocol['seed'] == 20261116, 'The prespecified Test sample design differs.')
    project = Path('/content/drive/MyDrive/Project9_Phase2')
    local = Path('/content/Project9_Phase2')
    manifest_path = project / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    require(sha(manifest_path.read_bytes()) == frozen_protocol['manifest_sha256'], 'Frozen split bytes differ.')
    manifest = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding='utf-8-sig'))))
    rows = [row for row in manifest if row['split'] == 'test']
    require(len(rows) == 929 and all(row['original_split'] == 'test' for row in rows), 'Frozen Test membership differs.')
    gallery = json.loads((Path(frozen_protocol['gallery_assets_path']) / 'gallery_meta.json').read_text(encoding='utf-8'))
    gallery_names = {item['export_name'] for item in gallery}
    classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    archive_size = 7518242799
    password = b'REDACTED_DATASET_ZIP_PASSWORD'
    class HTTPRangeFile(io.RawIOBase):
        """Bounded read-only seekable HTTP file for Python's ZIP64 reader."""

        def __init__(self):
            super().__init__()
            self.position = 0
            self.size = archive_size
            self.block_size = 1024 * 1024
            self.blocks = OrderedDict()
            self.transferred = 0
            self.requests_count = 0
            self.session = requests.Session()

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.position

        def seek(self, offset, whence=0):
            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
            if position < 0:
                raise ValueError('Invalid HTTP ZIP seek.')
            self.position = int(position)
            return self.position

        def get_block(self, index):
            if index in self.blocks:
                self.blocks.move_to_end(index)
                return self.blocks[index]
            start = index * self.block_size
            end = min(self.size - 1, start + self.block_size - 1)
            needed = end - start + 1
            require(self.transferred + needed <= 768 * 1024 ** 2, 'Partial-download budget exceeded; the full archive will not be downloaded automatically.')
            for attempt in range(3):
                try:
                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity', 'Cache-Control': 'no-cache'}) as response:
                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
                            time.sleep(2 * (attempt + 1))
                            continue
                        require(response.status_code == 206, f'HTTP {response.status_code}: requested range was not supported; full download prevented.')
                        content_range = response.headers.get('Content-Range', '').strip()
                        match = re.fullmatch('bytes (\\d+)-(\\d+)/(\\d+)', content_range, flags=re.I)
                        require(match and tuple(map(int, match.groups())) == (start, end, self.size), 'Server returned an unexpected Content-Range: ' + content_range)
                        data = response.raw.read(needed + 1)
                        require(len(data) == needed, 'Incomplete or oversized HTTP range response.')
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
                    time.sleep(2 * (attempt + 1))
            self.transferred += len(data)
            self.requests_count += 1
            self.blocks[index] = data
            while len(self.blocks) > 16:
                self.blocks.popitem(last=False)
            if self.requests_count % 5 == 0:
                print(f'Partial ZIP transfer: {self.transferred / 1024 ** 2:.1f} MiB')
            return data

        def read(self, count=-1):
            if self.closed:
                raise ValueError('HTTP ZIP stream is closed.')
            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))
            require(count <= 64 * 1024 ** 2, 'Unexpected large ZIP read; complete-archive download prevented.')
            chunks = []
            remaining = count
            while remaining:
                index, offset = divmod(self.position, self.block_size)
                data = self.get_block(index)
                take = min(remaining, len(data) - offset)
                require(take > 0, 'Unexpected ZIP read position.')
                chunks.append(data[offset:offset + take])
                self.position += take
                remaining -= take
            return b''.join(chunks)

        def close(self):
            self.session.close()
            self.blocks.clear()
            super().close()

    cache = local / 'extensions_test_source_images'
    cache.mkdir(parents=True, exist_ok=True)
    existing_archive = next((path for path in [Path('/content/zerowaste-f-final.zip'), project / 'cache/zerowaste-f-final.zip']
                             if path.is_file() and path.stat().st_size == archive_size), None)
    remote = None
    if existing_archive is None:
        remote = HTTPRangeFile()
        archive = zipfile.ZipFile(remote)
    else:
        archive = zipfile.ZipFile(existing_archive)
    protocol_sha = sha(protocol_file.read_bytes())
    try:
        def member(suffix):
            require(suffix.startswith('test/') and '..' not in PurePosixPath(suffix).parts,
                    'Only source Test members selected by the frozen protocol may be read.')
            matches = [item for item in archive.infolist() if not item.is_dir()
                       and (item.filename == suffix or item.filename.endswith('/' + suffix))
                       and '..' not in PurePosixPath(item.filename).parts]
            require(len(matches) == 1, 'Test member is not unique: ' + suffix)
            return matches[0]

        def fetch(info, filename):
            path = cache / filename
            sidecar = cache / (filename + '.source.json')
            require(0 < info.file_size <= (256 * 1024**2 if filename.endswith('.json') else 12 * 1024**2),
                    'Unexpected source member size.')
            if path.exists():
                require(sidecar.is_file(), 'Existing source cache has no provenance receipt.')
                raw = path.read_bytes()
                receipt = json.loads(sidecar.read_text(encoding='utf-8'))
                reused = True
            else:
                raw = archive.read(info, pwd=password)
                receipt = {'sha256': sha(raw), 'member': info.filename,
                           'member_crc32': info.CRC, 'member_bytes': info.file_size,
                           'archive_url': archive_url, 'archive_total_bytes': archive_size}
                with path.open('xb') as stream:
                    stream.write(raw)
                with sidecar.open('x', encoding='utf-8') as stream:
                    json.dump(receipt, stream, indent=2)
                reused = False
            require(len(raw) == info.file_size and zlib.crc32(raw) & 0xffffffff == info.CRC,
                    'Source member CRC/size differs.')
            expected = {'sha256': sha(raw), 'member': info.filename, 'member_crc32': info.CRC,
                        'member_bytes': info.file_size, 'archive_url': archive_url,
                        'archive_total_bytes': archive_size}
            require(receipt == expected, 'Source cache provenance differs.')
            return path, raw, {**receipt, 'reused_cache': reused,
                               'historical_full_source_image_sha256_anchor_available': False}

        _, raw_annotations, annotation_receipt = fetch(member('test/labels.json'), 'source_test_annotations.json')
        coco = json.loads(raw_annotations.decode('utf-8-sig'))
        images = {int(item['id']): item for item in coco['images']}
        category = {int(item['id']): item['name'] for item in coco['categories']}
        require(set(category.values()) == set(classes), 'Source material categories differ.')
        annotations = defaultdict(list)
        for ann in coco['annotations']:
            annotations[int(ann['image_id'])].append(ann)
        pools = defaultdict(list)
        for row in sorted(rows, key=lambda item: 'test__' + item['filename']):
            image_id = int(row['image_id'])
            info = images[image_id]
            require(PurePosixPath(info['file_name']).name == row['filename'], 'Test COCO image identity differs.')
            width, height = int(info['width']), int(info['height'])
            counts = Counter()
            for annotation_ordinal, ann in enumerate(annotations[image_id]):
                cls = classes.index(category[int(ann['category_id'])])
                x, y, bw, bh = map(float, ann['bbox'])
                require(all(math.isfinite(value) for value in [x, y, bw, bh]) and bw > 0 and bh > 0,
                        'Invalid Test annotation box.')
                x1, y1, x2, y2 = max(0, min(width, x)), max(0, min(height, y)), max(0, min(width, x + bw)), max(0, min(height, y + bh))
                require(x2 > x1 and y2 > y1, 'Empty clipped Test box.')
                xc, yc, wn, hn = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height
                line = f'{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}'
                _, xc, yc, wn, hn = map(float, line.split())
                counts[classes[cls]] += 1
                if wn > 0.003 and hn > 0.003:
                    pools[cls].append({'class_id': cls, 'class_name': classes[cls], 'image_id': image_id,
                        'annotation_id': int(ann['id']), 'annotation_ordinal_in_image': annotation_ordinal,
                        'export_name': 'test__' + row['filename'], 'filename': row['filename'],
                        'width': width, 'height': height, 'original_split': 'test', 'project_split': 'test',
                        'xywh_normalized': [xc, yc, wn, hn], 'converted_yolo_line': line,
                        'official_clipped_box_xyxy': [x1, y1, x2, y2], 'frozen_manifest_row': row})
            require(all(counts[name] == int(row[name]) for name in classes), 'Test annotation counts differ from frozen manifest.')
        rng = np.random.default_rng(frozen_protocol['seed'])
        selected = []
        for cls in range(4):
            require(pools[cls], 'A prespecified material has no eligible Test crops.')
            indices = rng.choice(len(pools[cls]), size=min(frozen_protocol['per_class'], len(pools[cls])), replace=False)
            selected.extend(dict(pools[cls][int(index)]) for index in indices)
        selected_names = {item['export_name'] for item in selected}
        require(not selected_names & gallery_names, 'Test queries overlap the Train gallery source images.')
        member_by_name = {name: member('test/data/' + name.removeprefix('test__')) for name in selected_names}
        source_records = {}
        for ordinal, name in enumerate(sorted(selected_names, key=lambda value: member_by_name[value].header_offset), 1):
            path, raw, receipt = fetch(member_by_name[name], name)
            with Image.open(io.BytesIO(raw)) as image:
                record = next(item for item in selected if item['export_name'] == name)
                require(image.format == 'PNG' and image.size == (record['width'], record['height']), 'Test PNG dimensions/format differ.')
                image.verify()
            source_records[name] = {'path': str(path), **receipt}
            if ordinal % 10 == 0 or ordinal == len(selected_names):
                print('Verified selected Test source PNGs:', ordinal, '/', len(selected_names))
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        work_dir = local / ('extension_test_inputs_' + stamp)
        (work_dir / 'test_queries').mkdir(parents=True, exist_ok=False)
        for ordinal, record in enumerate(selected):
            source = source_records[record['export_name']]
            width, height = record['width'], record['height']
            xc, yc, bw, bh = record['xywh_normalized']
            x1 = max(0, int((xc - bw / 2) * width))
            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))
            y1 = max(0, int((yc - bh / 2) * height))
            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))
            box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]
            relative = f'test_queries/t_{ordinal:03d}.png'
            path = work_dir / relative
            with Image.open(source['path']) as image:
                image.convert('RGB').crop(tuple(box)).save(path, format='PNG')
            record.update({'ordinal': ordinal, 'source_image_sha256': source['sha256'],
                           'crop_file': relative, 'crop_path': str(path), 'crop_sha256': sha(path.read_bytes()),
                           'crop_box_xyxy': box, 'crop_width': box[2] - box[0], 'crop_height': box[3] - box[1]})
        require(sha(protocol_file.read_bytes()) == protocol_sha, 'Frozen Test protocol changed during preparation.')
        provenance = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test',
            'selection': 'Fixed seed20261116, up to50 eligible objects per material without replacement; use all if fewer than50.',
            'seed': 20261116, 'bit_generator': type(rng.bit_generator).__name__,
            'sample_objects': len(selected), 'unique_source_images': len(selected_names),
            'counts_per_class': {classes[cls]: sum(item['class_id'] == cls for item in selected) for cls in range(4)},
            'eligible_objects_per_class': {classes[cls]: len(pools[cls]) for cls in range(4)},
            'full_Test_partition_images': 929, 'scope': 'Prespecified material-balanced GT-crop Test subset; not a full929-image system evaluation.',
            'annotation_source': annotation_receipt, 'source_images': source_records,
            'historical_full_PNG_or_Test_COCO_hash_anchors_available': False,
            'new_source_verification': 'Selected source ZIP members checked by CRC/size and new SHA256 receipts; identities/counts checked against frozen manifest.',
            'whole_archive_md5_recomputed_this_run': False, 'archive_url': archive_url, 'archive_total_bytes': archive_size,
            'range_bytes_transferred': remote.transferred if remote else 0,
            'protocol_sha256_before_Test_annotation_access': protocol_sha,
            'gallery_source_images_disjoint': True, 'new_model_execution_in_preparation': False}
        return {'queries': selected, 'work_dir': str(work_dir), 'provenance': provenance}
    finally:
        archive.close()
        if remote is not None:
            remote.close()


def recover_extension_test_before_inference():
    from datetime import datetime, timezone
    from importlib.metadata import version, PackageNotFoundError
    from pathlib import Path
    import hashlib
    import json
    import os
    import shutil
    import subprocess
    import sys
    import zipfile
    from google.colab import drive, files

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 ** 2), b''):
                value.update(chunk)
        return value.hexdigest()

    def save_new(path, value):
        with Path(path).open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write('\n')

    def installed(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None
    drive.mount('/content/drive')
    project = Path('/content/drive/MyDrive/Project9_Phase2')
    output = project / 'evaluation/EXTENSIONS_FINAL_TEST_ONCE'
    archive_path = project / 'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip'
    if (output / 'COMPLETED.json').is_file():
        require((output / 'COMPLETED.json').is_file() and archive_path.is_file(), 'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.')
        inventory = json.loads((output / 'collection_manifest.json').read_text(encoding='utf-8'))['files']
        with zipfile.ZipFile(archive_path) as archive:
            require(archive.testzip() is None, 'Saved Test evidence ZIP failed CRC verification.')
            for item in inventory:
                raw = archive.read(item['file'])
                require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Saved Test archive bytes differ: ' + item['file'])
        print('Previously completed extension Test found; downloading its saved evidence without model execution.')
        files.download(str(archive_path))
        return {'already_completed': True, 'zip': str(archive_path)}
    require(output.is_dir(), 'The original reserved Test folder is missing. Stop and share this output.')
    protocol_file = output / 'frozen_protocol.json'
    status_file = output / 'STATUS.json'
    require(protocol_file.is_file() and status_file.is_file(), 'Original frozen protocol/status is missing.')
    protocol_bytes = protocol_file.read_bytes()
    protocol_sha = hashlib.sha256(protocol_bytes).hexdigest()
    protocol = json.loads(protocol_bytes.decode('utf-8'))
    status = json.loads(status_file.read_text(encoding='utf-8'))
    require(status == {'phase': 'preparing_inputs_no_Test_model_calls_yet', 'protocol_sha256': protocol_sha}, 'Recovery is restricted to the recorded failure BEFORE Test inference; status/hash differs.')
    blocked = ['query_provenance.json', 'test_queries', 'reference_gallery', 'inference_batches', 'test_arrays.npz', 'comparison_TEST.json', 'per_query_TEST.json', 'COMPLETED.json', 'collection_manifest.json']
    require(not any(((output / name).exists() for name in blocked)) and (not archive_path.exists()), 'Existing Test preparation/results found. Do not evaluate again; share this output for separate recovery review.')
    require(protocol.get('split') == 'test' and protocol.get('per_class') == 50 and (protocol.get('seed') == 20261116) and (protocol.get('target_maximum_query_objects') == 200) and (protocol.get('full_Test_partition_images') == 929), 'Frozen Test sampling parameters differ.')
    require(protocol.get('output_dir') == str(output) and protocol.get('gallery_objects') == 160 and (protocol.get('prompt_model_threshold_selection_on_Test') is False) and (protocol.get('M4_M5_Test_model_calls') == 0) and (protocol.get('new_training_or_optimizer_steps') == 0), 'Frozen Test scope/settings differ.')
    expected_sources = {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()}
    require(protocol.get('source_files_sha256') == expected_sources, 'Frozen source references differ from the original supplied cell.')
    for name, expected in expected_sources.items():
        require(digest(output / 'code' / name) == expected, 'Original saved code bytes differ: ' + name)
    require({path.name for path in (output / 'code').iterdir()} == set(expected_sources), 'Unexpected files appeared in the original frozen code folder.')
    memory_diagnostic = {'available': False, 'note': 'The saved protocol bytes, not traceback memory, are the recovery authority.'}
    last_error = getattr(sys, 'last_exc', None) or getattr(sys, 'last_value', None)
    trace = getattr(last_error, '__traceback__', None)
    while trace is not None:
        frame = trace.tb_frame
        if frame.f_code.co_name == 'prepare_extension_test_inputs' and 'frozen_protocol' in frame.f_locals:
            argument = frame.f_locals['frozen_protocol']
            try:
                normalized = json.loads(json.dumps(argument, ensure_ascii=False, allow_nan=False))
                memory_diagnostic.update({'available': True, 'python_object_equal': argument == protocol, 'json_normalized_equal': normalized == protocol})
                if normalized != protocol:
                    raise RuntimeError('Original in-memory protocol normalizes differently from the saved frozen file.')
            except (TypeError, ValueError) as error:
                raise RuntimeError('Original protocol memory is inconsistent with its saved JSON.') from error
            break
        trace = trace.tb_next
    print('Recovery gate passed: original protocol hash matches, status is before Test, no Test batches/results exist.')
    os.environ['WANDB_MODE'] = 'disabled'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    torch_version, vision_version = (installed('torch'), installed('torchvision'))
    require(torch_version and vision_version, 'A matching preinstalled PyTorch/torchvision runtime is required.')
    desired = {'transformers': '5.22.1', 'numpy': '2.4.6', 'Pillow': '11.3.0'}
    changes = [name + '==' + value for name, value in desired.items() if installed(name) != value]
    changes += [name for name in ['requests'] if installed(name) is None]
    if changes:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==' + torch_version, 'torchvision==' + vision_version, *changes], check=True, timeout=360)
    import numpy as np
    import PIL
    from PIL import Image
    require(np.__version__ == '2.4.6' and PIL.__version__ == '11.3.0', 'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.')
    require(installed('torch') == torch_version and installed('torchvision') == vision_version, 'The preinstalled torch pair unexpectedly changed.')
    validation = project / 'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z'
    assets = validation / 'deployment/assets'
    frozen_val = json.loads((validation / 'frozen_operator_VAL_protocol.json').read_text(encoding='utf-8'))
    manifest = json.loads((validation / 'deployment/assist_manifest.json').read_text(encoding='utf-8'))
    require(digest(validation / 'frozen_operator_VAL_protocol.json') == _EXT_TEST_ANCHORS['val_protocol'], 'The already-audited Validation protocol differs.')
    require(digest(assets / 'gallery_index.npz') == _EXT_TEST_ANCHORS['gallery_index'], 'The fixed160-reference gallery differs.')
    require(digest(assets / 'gallery_meta.json') == _EXT_TEST_ANCHORS['gallery_meta'], 'Gallery identities differ.')
    require(hashlib.sha256(_EXT_TEST_SOURCES['extension_assist_models.py'].encode()).hexdigest() == frozen_val['source_files_sha256']['extension_assist_models.py'], 'The model helper source changed after Validation.')
    protected_relative = _EXT_TEST_PROTECTED_PATHS
    protected = {name: project / relative for name, relative in protected_relative.items()}
    before = {name: digest(path) for name, path in protected.items()}
    require(before == frozen_val['original_project_files_sha256'], 'An original protected project reference differs.')
    for relative, expected in manifest['assets_sha256'].items():
        require(digest(assets / relative) == expected, 'Validation gallery/crop asset differs: ' + relative)
    print('Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...')
    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest['clip_revision'])
    resources = engine.runtime_metadata()
    for key in ['clip_revision', 'clip_checkpoint_sha256', 'resnet_checkpoint_sha256']:
        require(resources[key] == manifest['model_resources'][key], 'Official assistance model resource differs: ' + key)
    gallery_meta = json.loads((assets / 'gallery_meta.json').read_text(encoding='utf-8'))
    gallery_crops = []
    for item in gallery_meta:
        with Image.open(assets / item['crop_file']) as image:
            gallery_crops.append(image.convert('RGB'))
    recomputed = engine.embed_resnet(gallery_crops)
    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)
    require(bool(np.all(cosines >= 0.999)), 'Runtime reference embedding compatibility failed before Test.')
    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-07), 'Runtime histogram preprocessing differs from the fixed Validation reference bank.')
    del gallery_crops, recomputed
    require(protocol['manifest_sha256'] == before['split_manifest'] and protocol['protected_original_files_before'] == before, 'Original protected references differ from frozen Test protocol.')
    require(protocol['gallery_assets_path'] == str(assets) and protocol['validation_protocol_sha256'] == _EXT_TEST_ANCHORS['val_protocol'] and (protocol['gallery_index_sha256'] == _EXT_TEST_ANCHORS['gallery_index']) and (protocol['gallery_metadata_sha256'] == _EXT_TEST_ANCHORS['gallery_meta']), 'Frozen gallery/Validation anchors differ.')
    require(protocol['class_names'] == list(CLASS_NAMES) and protocol['clip_prompts'] == list(PLAIN_PROMPTS), 'Frozen class names/prompts differ.')
    current_resources = json.loads(json.dumps(resources, ensure_ascii=False, allow_nan=False))
    require(current_resources == protocol['model_resources'], 'Current runtime/model processor metadata differs from frozen protocol; stop before Test.')
    require(digest(protocol_file) == protocol_sha, 'The original frozen protocol changed during resource checks.')
    recovery_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    recovery_dir = output / ('recovery_before_inference_' + recovery_stamp)
    recovery_dir.mkdir(exist_ok=False)
    with (recovery_dir / 'recovery_function.py').open('x', encoding='utf-8', newline='\n') as stream:
        stream.write(_EXT_RECOVERY_FUNCTION_SOURCE)
    save_new(recovery_dir / 'recovery_receipt.json', {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'reason': 'Original direct Python-object versus persisted-JSON equality failed before Test annotation access.', 'fix': 'Pass the original SHA-verified decoded JSON object to the unchanged original preparation function.', 'frozen_protocol_sha256_unchanged': protocol_sha, 'original_code_sha256_unchanged': expected_sources, 'recovery_function_sha256': hashlib.sha256(_EXT_RECOVERY_FUNCTION_SOURCE.encode()).hexdigest(), 'status_before_recovery': status, 'memory_diagnostic': memory_diagnostic, 'sampling_models_prompts_thresholds_and_preprocessing_changed': False, 'previous_Test_batches_or_results_present': False, 'current_model_resources_json_normalized_match': True, 'original_M4_M5_Test_calls': 0})
    print('Continuing the same frozen attempt with the original protocol JSON; no frozen file was rewritten.')
    preparation = prepare_extension_test_inputs(protocol, output)
    records = preparation['queries']
    work = Path(preparation['work_dir'])
    save_new(output / 'query_provenance.json', preparation)
    shutil.copytree(work / 'test_queries', output / 'test_queries')
    reference_dir = output / 'reference_gallery'
    reference_dir.mkdir()
    shutil.copytree(assets / 'gallery', reference_dir / 'gallery')
    shutil.copyfile(assets / 'gallery_index.npz', reference_dir / 'gallery_index.npz')
    shutil.copyfile(assets / 'gallery_meta.json', reference_dir / 'gallery_meta.json')
    batch_dir = output / 'inference_batches'
    batch_dir.mkdir()
    (output / 'STATUS.json').write_text(json.dumps({'phase': 'Test_inference_started', 'protocol_sha256': protocol_sha, 'query_objects': len(records)}, indent=2), encoding='utf-8')
    q_parts, h_parts, l_parts = ([], [], [])
    print('One fixed Test evaluation:', len(records), 'annotated crops; no detector/segmenter calls.')
    for start in range(0, len(records), 8):
        crops = []
        for item in records[start:start + 8]:
            path = output / item['crop_file']
            require(digest(path) == item['crop_sha256'], 'Selected Test crop changed.')
            with Image.open(path) as image:
                crops.append(image.convert('RGB'))
        q, h, logits = (engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8))
        path = batch_dir / f'batch_{start // 8:03d}.npz'
        with path.open('xb') as stream:
            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)), embeddings=q, histograms=h, clip_logits=logits)
        q_parts.append(q)
        h_parts.append(h)
        l_parts.append(logits)
        print('Recorded Test crops:', start + len(crops), '/', len(records))
    q, h, logits = (np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts))
    labels = np.asarray([item['class_id'] for item in records], dtype=np.int64)
    gy = engine.gallery_labels
    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)
    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)
    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]
    clip_pred = logits.argmax(axis=1)
    comparison = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test', 'sample_objects': len(records), 'gallery_objects': 160, 'sample_counts': preparation['provenance']['counts_per_class'], 'unique_source_images': preparation['provenance']['unique_source_images'], 'visual_search_resnet': search, 'visual_search_histogram': baseline_search, 'clip_plain': classification_metrics(labels, clip_pred), 'histogram_classification_baseline': classification_metrics(labels, hist_pred), 'frozen_protocol_sha256': protocol_sha, 'limits': ['Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.', 'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.', 'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.']}
    save_new(output / 'comparison_TEST.json', comparison)
    with (output / 'test_arrays.npz').open('xb') as stream:
        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits, query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank, histogram_class_predictions=hist_pred)
    rows = [{'query_index': index, 'crop_file': item['crop_file'], 'source_export_name': item['export_name'], 'annotation_id': item['annotation_id'], 'ground_truth_class_id': int(labels[index]), 'resnet_top1_class_id': int(gy[rank[index, 0]]), 'resnet_top5_gallery_indices': [int(value) for value in rank[index, :5]], 'clip_class_id': int(clip_pred[index]), 'clip_logits': logits[index].tolist(), 'histogram_class_id': int(hist_pred[index])} for index, item in enumerate(records)]
    save_new(output / 'per_query_TEST.json', rows)
    after = {name: digest(path) for name, path in protected.items()}
    require(before == after and digest(output / 'frozen_protocol.json') == protocol_sha, 'An original file or frozen Test protocol changed.')
    save_new(output / 'protected_original_files.json', {'before': before, 'after': after, 'unchanged': True, 'original_M4_M5_Test_calls': 0, 'training_or_optimizer_steps': 0})
    save_new(output / 'COMPLETED.json', {'status': 'completed_once', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'frozen_protocol_sha256': protocol_sha, 'comparison_sha256': digest(output / 'comparison_TEST.json'), 'zip_name': archive_path.name, 'M4_M5_Test_reevaluated': False})
    (output / 'STATUS.json').write_text(json.dumps({'phase': 'completed_once', 'protocol_sha256': protocol_sha}), encoding='utf-8')
    inventory = [{'file': path.relative_to(output).as_posix(), 'bytes': path.stat().st_size, 'sha256': digest(path)} for path in sorted(output.rglob('*')) if path.is_file()]
    save_new(output / 'collection_manifest.json', {'files': inventory})
    with zipfile.ZipFile(archive_path, 'x', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(output).as_posix())
    with zipfile.ZipFile(archive_path) as archive:
        require(archive.testzip() is None, 'Test evidence ZIP CRC check failed.')
        for item in inventory:
            raw = archive.read(item['file'])
            require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Packaged Test evidence differs.')
    print(json.dumps(comparison, indent=2))
    print('Extension Test complete. No M4/M5 Test evaluation or training was run.')
    print('Saved original results unchanged. ZIP:', archive_path)
    files.download(str(archive_path))
    return {'zip': str(archive_path), 'output_dir': str(output), 'comparison': comparison}

_EXTENSION_TEST_RECOVERY_RESULT = recover_extension_test_before_inference()


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# فحص حالة محفوظة فقط: لا تدريب، لا تحميل نماذج، لا تشغيل أو استكمال Test.
from pathlib import Path
import hashlib
import json
import sys
import zipfile


def check_saved_extension_test(project):
    project = Path(project)
    if not project.is_dir():
        print('Drive غير متصل في هذه الجلسة. لا تشغّل Test؛ أرسل هذه النتيجة.')
        return
    folder = project / 'evaluation/EXTENSIONS_FINAL_TEST_ONCE'
    archive = project / 'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip'
    print('=== فحص الحالة فقط ===')
    print('مجلد المحاولة موجود:', folder.is_dir())
    if not folder.is_dir():
        print('لم يظهر المجلد في Drive المتصل. لا تبدأ محاولة جديدة قبل مراجعة الحالة.')
        return
    status_file = folder / 'STATUS.json'
    protocol_file = folder / 'frozen_protocol.json'
    status = json.loads(status_file.read_text(encoding='utf-8')) if status_file.is_file() else {}
    print('الحالة المحفوظة:', json.dumps(status, ensure_ascii=False))
    if protocol_file.is_file():
        actual = hashlib.sha256(protocol_file.read_bytes()).hexdigest()
        print('بصمة البروتوكول تطابق الحالة:', actual == status.get('protocol_sha256'))
    batches = sorted((folder / 'inference_batches').glob('batch_*.npz'))
    valid_batches = 0
    for batch in batches:
        try:
            with zipfile.ZipFile(batch) as stream:
                valid_batches += stream.testzip() is None
        except (OSError, zipfile.BadZipFile):
            pass
    print('الدفعات المحفوظة:', len(batches), '| السليمة بفحص CRC:', valid_batches)
    markers = {name: (folder / name).is_file() for name in
               ['query_provenance.json', 'comparison_TEST.json', 'test_arrays.npz',
                'per_query_TEST.json', 'COMPLETED.json', 'collection_manifest.json']}
    print('ملفات الحالة:', json.dumps(markers, ensure_ascii=False))
    print('ZIP موجود:', archive.is_file())
    if archive.is_file():
        print('حجم ZIP بالميجابايت:', round(archive.stat().st_size / 1024**2, 2))
    packaged = False
    if archive.is_file() and markers['COMPLETED.json'] and markers['collection_manifest.json']:
        try:
            inventory = json.loads((folder / 'collection_manifest.json').read_text(encoding='utf-8'))['files']
            with zipfile.ZipFile(archive) as stream:
                packaged = stream.testzip() is None
                for item in inventory:
                    raw = stream.read(item['file'])
                    packaged = packaged and len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256']
            print('فحص ZIP والأحجام والبصمات:', 'PASS' if packaged else 'لم ينجح')
        except (OSError, KeyError, ValueError, zipfile.BadZipFile) as error:
            print('تعذّر التحقق من ZIP:', type(error).__name__)
    if packaged:
        print('الحزمة مكتملة ومحفوظة؛ نزّلها من Google Drive دون إعادة الاختبار:')
        print('My Drive / Project9_Phase2 / Project9_EXTENSIONS_FINAL_TEST_ONCE.zip')
    else:
        print('أرسل هذا الناتج لتحديد الخطوة التالية؛ لا تُعد تشغيل خلية Test أو الاستعادة الآن.')
    last = getattr(sys, 'last_exc', None) or getattr(sys, 'last_value', None)
    if last is not None:
        print('آخر خطأ Python محفوظ:', type(last).__name__, str(last)[:500])
    print('لم يُشغّل هذا الفحص نماذج أو تدريبًا أو Test، ولم يغيّر الملفات.')


check_saved_extension_test('/content/drive/MyDrive/Project9_Phase2')


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

# استعادة تنزيل ZeroWaste بعدHTTP429، مع الحفاظ على البروتوكول والإعدادات نفسها.
# خلية جديدة واحدة؛ لا تحذف مجلدات ولا تشغّل الخلايا القديمة.
# يبدأ بانتظار60ث ويطبع تقدم التنزيل؛ اتركه يكمل. لا يعيد دفعاتTestسابقة.

_EXT_TEST_SOURCES = {'extension_assist_models.py': '"""CPU models and numeric helpers for the optional operator-assistance stage.\n\nImporting this module performs no model loading, downloads or inference.\nModel construction is explicit; all predictions are advisory and do not alter\nthe original detector/segmenter response or introduce a rejection threshold.\n"""\n\nCLASS_NAMES = (\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\')\nCLIP_MATERIAL_NAMES = (\'cardboard\', \'metal\', \'rigid plastic\', \'soft plastic\')\nPLAIN_PROMPTS = tuple(\'a photo of \' + name for name in CLIP_MATERIAL_NAMES)\nCLIP_MODEL_ID = \'openai/clip-vit-base-patch32\'\n\n\ndef _assist_require(condition, message):\n    if not condition:\n        raise RuntimeError(message)\n\n\ndef _assist_sha256(path):\n    from pathlib import Path\n    import hashlib\n    value = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b\'\'):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef unit(values):\n    """The original float32 L2 normalization, including its 1e-10 guard."""\n    import numpy as np\n    array = np.asarray(values, dtype=np.float32)\n    _assist_require(array.ndim == 2 and np.isfinite(array).all(), \'Expected finite 2D features.\')\n    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)\n\n\ndef rgb_hist(crops):\n    """Original 24-bin RGB histogram baseline; crops must be PIL images."""\n    import numpy as np\n    rows = []\n    for crop in crops:\n        array = np.asarray(crop.convert(\'RGB\').resize((64, 64)), dtype=np.uint8)\n        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,\n                                                range=(0, 256))[0] for channel in range(3)]))\n    _assist_require(bool(rows), \'At least one crop is required.\')\n    return unit(np.asarray(rows, dtype=np.float32))\n\n\ndef classification_metrics(true_labels, predicted_labels):\n    """Four-material accuracy and macro metrics with zero-division set to zero."""\n    import numpy as np\n    truth = np.asarray(true_labels)\n    prediction = np.asarray(predicted_labels)\n    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape\n                    and truth.size > 0, \'Class arrays must be nonempty, aligned 1D arrays.\')\n    _assist_require(np.issubdtype(truth.dtype, np.integer)\n                    and np.issubdtype(prediction.dtype, np.integer), \'Class IDs must be integers.\')\n    _assist_require(((truth >= 0) & (truth < 4)).all()\n                    and ((prediction >= 0) & (prediction < 4)).all(), \'Unknown material class ID.\')\n    matrix = np.zeros((4, 4), dtype=np.int64)\n    np.add.at(matrix, (truth, prediction), 1)\n    tp = np.diag(matrix).astype(np.float64)\n    predicted_support = matrix.sum(axis=0)\n    support = matrix.sum(axis=1)\n    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)\n    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)\n    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)\n    return {\n        \'accuracy\': float(np.mean(truth == prediction)), \'macro_f1\': float(f1.mean()),\n        \'macro_precision\': float(precision.mean()), \'macro_recall\': float(recall.mean()),\n        \'count\': int(truth.size), \'confusion_matrix_true_rows_predicted_columns\': matrix.tolist(),\n        \'per_class\': [{\'class_id\': index, \'material\': name, \'support\': int(support[index]),\n                       \'precision\': float(precision[index]), \'recall\': float(recall[index]),\n                       \'f1\': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],\n    }\n\n\ndef retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):\n    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""\n    import numpy as np\n    gallery = np.asarray(gallery_features, dtype=np.float32)\n    queries = np.asarray(query_features, dtype=np.float32)\n    gallery_y = np.asarray(gallery_labels)\n    query_y = np.asarray(query_labels)\n    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]\n                    and len(gallery) >= 5 and len(queries) > 0, \'Invalid gallery/query feature shape.\')\n    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)\n                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), \'Misaligned or nonfinite retrieval inputs.\')\n    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]\n    top1 = gallery_y[rankings[:, 0]]\n    top5 = gallery_y[rankings[:, :5]]\n    return ({\'top1_accuracy\': float(np.mean(top1 == query_y)),\n             \'hit_at_5\': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),\n             \'precision_at_5\': float(np.mean(top5 == query_y[:, None]))}, rankings)\n\n\nclass OperatorAssistModels:\n    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.\n\n    ``assets_dir`` may be None while building a new gallery. Then call\n    ``load_gallery`` after writing these files:\n\n    - gallery_index.npz: float32 embeddings [160,512], float32 histograms\n      [160,24], int64 labels [160].\n    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,\n      each with class_id and crop_file="gallery/g_000.png" etc.\n    - the corresponding 160 PNG reference crops.\n\n    local_only=True requires the ResNet checkpoint and the explicit immutable\n    CLIP revision to be present already; it performs no network request.\n    """\n\n    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):\n        from pathlib import Path\n        from importlib.metadata import version\n        import re\n        import torch\n        import torch.nn as nn\n        from torchvision.models import resnet18, ResNet18_Weights\n        from transformers import CLIPModel, CLIPProcessor\n        from huggingface_hub import hf_hub_download, model_info\n\n        self.torch = torch\n        self.assets_dir = None\n        self.gallery_embeddings = None\n        self.gallery_histograms = None\n        self.gallery_labels = None\n        self.gallery_meta = None\n        self.local_only = bool(local_only)\n        torch.set_num_threads(1)\n        try:\n            torch.set_num_interop_threads(1)\n        except RuntimeError:\n            _assist_require(torch.get_num_interop_threads() == 1,\n                            \'Torch inter-op threads are already initialized differently; use a fresh process.\')\n\n        weights = ResNet18_Weights.IMAGENET1K_V1\n        self.resnet_transform = weights.transforms()\n        resnet_file = Path(torch.hub.get_dir()) / \'checkpoints/resnet18-f37072fd.pth\'\n        if self.local_only:\n            _assist_require(resnet_file.is_file(), \'Cached ResNet18 checkpoint is required for local-only serving.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'Cached ResNet18 checkpoint checksum differs.\')\n            encoder = resnet18(weights=None)\n            state = torch.load(resnet_file, map_location=\'cpu\', weights_only=True)\n            encoder.load_state_dict(state, strict=True)\n            del state\n        else:\n            encoder = resnet18(weights=weights, progress=False)\n            _assist_require(resnet_file.is_file(), \'Downloaded ResNet18 checkpoint was not found at its standard cache path.\')\n            resnet_sha = _assist_sha256(resnet_file)\n            _assist_require(resnet_sha.startswith(\'f37072fd\'), \'ResNet18 checkpoint checksum differs from its official filename prefix.\')\n        encoder.fc = nn.Identity()\n        self.encoder = encoder.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.encoder.parameters():\n            parameter.requires_grad_(False)\n\n        hub_info = None\n        if clip_revision is None:\n            _assist_require(not self.local_only, \'An explicit cached CLIP revision is required in local-only mode.\')\n            hub_info = model_info(CLIP_MODEL_ID, revision=\'main\', files_metadata=True, token=False)\n            clip_revision = hub_info.sha\n        _assist_require(isinstance(clip_revision, str)\n                        and re.fullmatch(r\'[0-9a-f]{40}|[0-9a-f]{64}\', clip_revision) is not None,\n                        \'CLIP revision must be an immutable Hub commit hash.\')\n        self.clip_revision = clip_revision\n        common = {\'revision\': self.clip_revision, \'local_files_only\': self.local_only, \'token\': False}\n        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,\n                                             trust_remote_code=False, **common)\n        self.clip = self.clip.to(device=\'cpu\', dtype=torch.float32).eval()\n        for parameter in self.clip.parameters():\n            parameter.requires_grad_(False)\n        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,\n                                                       trust_remote_code=False, **common)\n        # The loader above explicitly selects the bin checkpoint. This cache-only\n        # lookup records its actual bytes without triggering a second download.\n        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename=\'pytorch_model.bin\',\n                                        revision=self.clip_revision, local_files_only=True, token=False))\n        clip_sha = _assist_sha256(clip_file)\n        expected_clip_sha = None\n        if hub_info is not None:\n            for sibling in hub_info.siblings or []:\n                if sibling.rfilename == \'pytorch_model.bin\':\n                    lfs = getattr(sibling, \'lfs\', None)\n                    expected_clip_sha = (lfs.get(\'sha256\') if isinstance(lfs, dict)\n                                         else getattr(lfs, \'sha256\', None))\n                    break\n        if expected_clip_sha is not None:\n            _assist_require(clip_sha == expected_clip_sha, \'CLIP binary differs from the Hub LFS SHA256.\')\n        config_commit = getattr(self.clip.config, \'_commit_hash\', None)\n        if config_commit is not None:\n            _assist_require(config_commit == self.clip_revision, \'Loaded CLIP configuration revision differs.\')\n        self._provenance = {\n            \'resnet_weights\': \'ResNet18_Weights.IMAGENET1K_V1\',\n            \'resnet_weights_url\': weights.url, \'resnet_checkpoint_path\': str(resnet_file),\n            \'resnet_checkpoint_sha256\': resnet_sha, \'resnet_transform\': repr(self.resnet_transform),\n            \'clip_model_id\': CLIP_MODEL_ID, \'clip_revision\': self.clip_revision,\n            \'clip_config_commit_hash\': config_commit, \'clip_checkpoint_path\': str(clip_file),\n            \'clip_checkpoint_sha256\': clip_sha, \'clip_hub_lfs_sha256\': expected_clip_sha,\n            \'clip_checkpoint_url\': \'https://huggingface.co/\' + CLIP_MODEL_ID + \'/resolve/\'\n                                   + self.clip_revision + \'/pytorch_model.bin\',\n            \'clip_processor_class\': type(self.processor).__name__,\n            \'clip_image_processor_class\': type(self.processor.image_processor).__name__,\n            \'clip_image_processor_config\': self.processor.image_processor.to_dict(),\n            \'clip_attention_implementation\': getattr(self.clip.config, \'_attn_implementation\', None),\n            \'versions\': {name: version(name) for name in [\'torch\', \'torchvision\', \'transformers\',\n                                                         \'huggingface-hub\', \'numpy\', \'Pillow\']},\n        }\n        self._assert_cpu_frozen()\n        if assets_dir is not None:\n            self.load_gallery(assets_dir)\n\n    def _assert_cpu_frozen(self):\n        for name, model in [(\'resnet\', self.encoder), (\'clip\', self.clip)]:\n            _assist_require(model.training is False, name + \' must be in eval mode.\')\n            _assist_require(all(parameter.device.type == \'cpu\' and not parameter.requires_grad\n                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)\n                                for parameter in model.parameters()), name + \' must be frozen CPU FP32.\')\n\n    def embed_resnet(self, crops, batch=16):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid ResNet crop batch.\')\n        self._assert_cpu_frozen()\n        vectors = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                tensor = self.torch.stack([self.resnet_transform(crop.convert(\'RGB\'))\n                                          for crop in crops[start:start + batch]])\n                output = self.encoder(tensor.to(device=\'cpu\', dtype=self.torch.float32))\n                vectors.append(output.detach().cpu().numpy())\n        array = np.concatenate(vectors).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 512), \'Unexpected ResNet feature dimensions.\')\n        return unit(array)\n\n    def score_clip(self, crops, batch=8):\n        import numpy as np\n        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, \'Invalid CLIP crop batch.\')\n        self._assert_cpu_frozen()\n        logits = []\n        with self.torch.inference_mode():\n            for start in range(0, len(crops), batch):\n                inputs = self.processor(text=list(PLAIN_PROMPTS),\n                                        images=[crop.convert(\'RGB\') for crop in crops[start:start + batch]],\n                                        return_tensors=\'pt\', padding=True)\n                inputs = {key: value.to(device=\'cpu\', dtype=self.torch.float32)\n                          if key == \'pixel_values\' else value.to(\'cpu\') for key, value in inputs.items()}\n                output = self.clip(**inputs)\n                logits.append(output.logits_per_image.detach().cpu().numpy())\n        array = np.concatenate(logits).astype(np.float32)\n        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), \'Unexpected or nonfinite CLIP logits.\')\n        return array\n\n    def rgb_hist(self, crops):\n        return rgb_hist(crops)\n\n    def load_gallery(self, assets_dir):\n        from pathlib import Path, PurePosixPath\n        import json\n        import numpy as np\n        directory = Path(assets_dir).resolve()\n        with np.load(directory / \'gallery_index.npz\', allow_pickle=False) as bank:\n            _assist_require(set(bank.files) == {\'embeddings\', \'histograms\', \'labels\'}, \'Unexpected assistance gallery keys.\')\n            embeddings = bank[\'embeddings\'].copy()\n            histograms = bank[\'histograms\'].copy()\n            labels = bank[\'labels\'].copy()\n        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32\n                        and histograms.shape == (160, 24) and histograms.dtype == np.float32\n                        and labels.shape == (160,) and labels.dtype == np.int64,\n                        \'Assistance gallery array shapes/dtypes differ.\')\n        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()\n                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),\n                        \'Assistance gallery features/classes differ.\')\n        for name, values in [(\'embedding\', embeddings), (\'histogram\', histograms)]:\n            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),\n                            \'Unnormalized gallery \' + name + \' features.\')\n        metadata = json.loads((directory / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n        _assist_require(isinstance(metadata, list) and len(metadata) == 160, \'Expected 160 gallery metadata records.\')\n        for index, item in enumerate(metadata):\n            expected_file = \'gallery/g_\' + str(index).zfill(3) + \'.png\'\n            _assist_require(isinstance(item, dict) and item.get(\'class_id\') == int(labels[index])\n                            and item.get(\'crop_file\') == expected_file, \'Gallery metadata/array alignment differs.\')\n            crop = PurePosixPath(item[\'crop_file\'])\n            path = (directory / crop).resolve()\n            _assist_require(not crop.is_absolute() and \'..\' not in crop.parts\n                            and path.is_relative_to(directory) and path.is_file(), \'Missing or unsafe reference crop.\')\n        self.assets_dir = directory\n        self.gallery_embeddings = embeddings\n        self.gallery_histograms = histograms\n        self.gallery_labels = labels\n        self.gallery_meta = metadata\n        return self\n\n    def query_crop(self, crop):\n        import numpy as np\n        _assist_require(self.gallery_embeddings is not None, \'Load the assistance gallery first.\')\n        feature = self.embed_resnet([crop], batch=1)\n        similarities = (feature @ self.gallery_embeddings.T)[0]\n        ranking = similarities.argsort()[::-1]\n        references = []\n        for raw_index in ranking[:5]:\n            index = int(raw_index)\n            class_id = int(self.gallery_labels[index])\n            references.append({\'gallery_index\': index, \'class_id\': class_id,\n                               \'material\': CLASS_NAMES[class_id], \'similarity\': float(similarities[index]),\n                               \'crop_file\': self.gallery_meta[index][\'crop_file\']})\n        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)\n        relative = np.exp(clip_logits - clip_logits.max())\n        relative /= relative.sum()\n        clip_class = int(clip_logits.argmax())\n        histogram = rgb_hist([crop])\n        hist_similarities = (histogram @ self.gallery_histograms.T)[0]\n        hist_index = int(hist_similarities.argmax())\n        hist_class = int(self.gallery_labels[hist_index])\n        return {\n            \'visual_search\': {\'predicted_class_id\': references[0][\'class_id\'],\n                              \'predicted_material\': references[0][\'material\'], \'references\': references,\n                              \'score_type\': \'Cosine similarity to frozen Train reference crops; not calibrated confidence.\'},\n            \'clip\': {\'predicted_class_id\': clip_class, \'predicted_material\': CLASS_NAMES[clip_class],\n                     \'logits\': clip_logits.tolist(), \'relative_scores\': relative.tolist(),\n                     \'prompts\': list(PLAIN_PROMPTS),\n                     \'score_type\': \'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.\'},\n            \'histogram\': {\'predicted_class_id\': hist_class, \'predicted_material\': CLASS_NAMES[hist_class],\n                          \'reference_gallery_index\': hist_index, \'similarity\': float(hist_similarities[hist_index]),\n                          \'crop_file\': self.gallery_meta[hist_index][\'crop_file\'],\n                          \'score_type\': \'Nearest Train RGB histogram by cosine similarity; argmax tie policy.\'},\n        }\n\n    def runtime_metadata(self):\n        self._assert_cpu_frozen()\n        return {\n            **self._provenance, \'clip_revision\': self.clip_revision, \'device\': \'cpu\', \'precision\': \'float32\',\n            \'resnet_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()\n                                              if parameter.is_floating_point()}),\n            \'clip_parameter_dtypes\': sorted({str(parameter.dtype) for parameter in self.clip.parameters()\n                                            if parameter.is_floating_point()}),\n            \'torch_num_threads\': int(self.torch.get_num_threads()),\n            \'torch_num_interop_threads\': int(self.torch.get_num_interop_threads()),\n            \'all_parameters_frozen\': True, \'plain_prompts\': list(PLAIN_PROMPTS),\n            \'gallery_loaded\': self.gallery_embeddings is not None,\n            \'gallery_size\': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,\n            \'model_training_or_optimization\': False,\n            \'decision_role\': \'Advisory references and material suggestion; original detector outputs are not overwritten.\',\n        }\n', 'prepare_extension_test_inputs.py': '"""Prepare a prespecified material-balanced crop subset of the frozen Test split."""\n\n\ndef prepare_extension_test_inputs(frozen_protocol, evidence_dir):\n    from collections import Counter, OrderedDict, defaultdict\n    from datetime import datetime, timezone\n    from pathlib import Path, PurePosixPath\n    import csv\n    import hashlib\n    import io\n    import json\n    import math\n    import re\n    import time\n    import zipfile\n    import zlib\n    import numpy as np\n    import requests\n    from PIL import Image\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def sha(raw):\n        return hashlib.sha256(raw).hexdigest()\n\n    evidence_dir = Path(evidence_dir)\n    protocol_file = evidence_dir / \'frozen_protocol.json\'\n    require(protocol_file.is_file() and json.loads(protocol_file.read_text(encoding=\'utf-8\')) == frozen_protocol,\n            \'A byte-hashed frozen protocol must exist before opening Test annotations.\')\n    require(frozen_protocol[\'split\'] == \'test\' and frozen_protocol[\'per_class\'] == 50\n            and frozen_protocol[\'seed\'] == 20261116, \'The prespecified Test sample design differs.\')\n    project = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    local = Path(\'/content/Project9_Phase2\')\n    manifest_path = project / \'data_protocol/ef6b35d2ba7f/split_manifest.csv\'\n    require(sha(manifest_path.read_bytes()) == frozen_protocol[\'manifest_sha256\'], \'Frozen split bytes differ.\')\n    manifest = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding=\'utf-8-sig\'))))\n    rows = [row for row in manifest if row[\'split\'] == \'test\']\n    require(len(rows) == 929 and all(row[\'original_split\'] == \'test\' for row in rows), \'Frozen Test membership differs.\')\n    gallery = json.loads((Path(frozen_protocol[\'gallery_assets_path\']) / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n    gallery_names = {item[\'export_name\'] for item in gallery}\n    classes = [\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\']\n    archive_url = \'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1\'\n    archive_size = 7518242799\n    password = b\'REDACTED_DATASET_ZIP_PASSWORD\'\n    class HTTPRangeFile(io.RawIOBase):\n        """Bounded read-only seekable HTTP file for Python\'s ZIP64 reader."""\n\n        def __init__(self):\n            super().__init__()\n            self.position = 0\n            self.size = archive_size\n            self.block_size = 1024 * 1024\n            self.blocks = OrderedDict()\n            self.transferred = 0\n            self.requests_count = 0\n            self.session = requests.Session()\n\n        def readable(self):\n            return True\n\n        def seekable(self):\n            return True\n\n        def tell(self):\n            return self.position\n\n        def seek(self, offset, whence=0):\n            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1\n            if position < 0:\n                raise ValueError(\'Invalid HTTP ZIP seek.\')\n            self.position = int(position)\n            return self.position\n\n        def get_block(self, index):\n            if index in self.blocks:\n                self.blocks.move_to_end(index)\n                return self.blocks[index]\n            start = index * self.block_size\n            end = min(self.size - 1, start + self.block_size - 1)\n            needed = end - start + 1\n            require(self.transferred + needed <= 768 * 1024 ** 2, \'Partial-download budget exceeded; the full archive will not be downloaded automatically.\')\n            for attempt in range(3):\n                try:\n                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={\'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\', \'Cache-Control\': \'no-cache\'}) as response:\n                        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:\n                            time.sleep(2 * (attempt + 1))\n                            continue\n                        require(response.status_code == 206, f\'HTTP {response.status_code}: requested range was not supported; full download prevented.\')\n                        content_range = response.headers.get(\'Content-Range\', \'\').strip()\n                        match = re.fullmatch(\'bytes (\\\\d+)-(\\\\d+)/(\\\\d+)\', content_range, flags=re.I)\n                        require(match and tuple(map(int, match.groups())) == (start, end, self.size), \'Server returned an unexpected Content-Range: \' + content_range)\n                        data = response.raw.read(needed + 1)\n                        require(len(data) == needed, \'Incomplete or oversized HTTP range response.\')\n                    break\n                except requests.RequestException:\n                    if attempt == 2:\n                        raise\n                    time.sleep(2 * (attempt + 1))\n            self.transferred += len(data)\n            self.requests_count += 1\n            self.blocks[index] = data\n            while len(self.blocks) > 16:\n                self.blocks.popitem(last=False)\n            if self.requests_count % 5 == 0:\n                print(f\'Partial ZIP transfer: {self.transferred / 1024 ** 2:.1f} MiB\')\n            return data\n\n        def read(self, count=-1):\n            if self.closed:\n                raise ValueError(\'HTTP ZIP stream is closed.\')\n            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))\n            require(count <= 64 * 1024 ** 2, \'Unexpected large ZIP read; complete-archive download prevented.\')\n            chunks = []\n            remaining = count\n            while remaining:\n                index, offset = divmod(self.position, self.block_size)\n                data = self.get_block(index)\n                take = min(remaining, len(data) - offset)\n                require(take > 0, \'Unexpected ZIP read position.\')\n                chunks.append(data[offset:offset + take])\n                self.position += take\n                remaining -= take\n            return b\'\'.join(chunks)\n\n        def close(self):\n            self.session.close()\n            self.blocks.clear()\n            super().close()\n\n    cache = local / \'extensions_test_source_images\'\n    cache.mkdir(parents=True, exist_ok=True)\n    existing_archive = next((path for path in [Path(\'/content/zerowaste-f-final.zip\'), project / \'cache/zerowaste-f-final.zip\']\n                             if path.is_file() and path.stat().st_size == archive_size), None)\n    remote = None\n    if existing_archive is None:\n        remote = HTTPRangeFile()\n        archive = zipfile.ZipFile(remote)\n    else:\n        archive = zipfile.ZipFile(existing_archive)\n    protocol_sha = sha(protocol_file.read_bytes())\n    try:\n        def member(suffix):\n            require(suffix.startswith(\'test/\') and \'..\' not in PurePosixPath(suffix).parts,\n                    \'Only source Test members selected by the frozen protocol may be read.\')\n            matches = [item for item in archive.infolist() if not item.is_dir()\n                       and (item.filename == suffix or item.filename.endswith(\'/\' + suffix))\n                       and \'..\' not in PurePosixPath(item.filename).parts]\n            require(len(matches) == 1, \'Test member is not unique: \' + suffix)\n            return matches[0]\n\n        def fetch(info, filename):\n            path = cache / filename\n            sidecar = cache / (filename + \'.source.json\')\n            require(0 < info.file_size <= (256 * 1024**2 if filename.endswith(\'.json\') else 12 * 1024**2),\n                    \'Unexpected source member size.\')\n            if path.exists():\n                require(sidecar.is_file(), \'Existing source cache has no provenance receipt.\')\n                raw = path.read_bytes()\n                receipt = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n                reused = True\n            else:\n                raw = archive.read(info, pwd=password)\n                receipt = {\'sha256\': sha(raw), \'member\': info.filename,\n                           \'member_crc32\': info.CRC, \'member_bytes\': info.file_size,\n                           \'archive_url\': archive_url, \'archive_total_bytes\': archive_size}\n                with path.open(\'xb\') as stream:\n                    stream.write(raw)\n                with sidecar.open(\'x\', encoding=\'utf-8\') as stream:\n                    json.dump(receipt, stream, indent=2)\n                reused = False\n            require(len(raw) == info.file_size and zlib.crc32(raw) & 0xffffffff == info.CRC,\n                    \'Source member CRC/size differs.\')\n            expected = {\'sha256\': sha(raw), \'member\': info.filename, \'member_crc32\': info.CRC,\n                        \'member_bytes\': info.file_size, \'archive_url\': archive_url,\n                        \'archive_total_bytes\': archive_size}\n            require(receipt == expected, \'Source cache provenance differs.\')\n            return path, raw, {**receipt, \'reused_cache\': reused,\n                               \'historical_full_source_image_sha256_anchor_available\': False}\n\n        _, raw_annotations, annotation_receipt = fetch(member(\'test/labels.json\'), \'source_test_annotations.json\')\n        coco = json.loads(raw_annotations.decode(\'utf-8-sig\'))\n        images = {int(item[\'id\']): item for item in coco[\'images\']}\n        category = {int(item[\'id\']): item[\'name\'] for item in coco[\'categories\']}\n        require(set(category.values()) == set(classes), \'Source material categories differ.\')\n        annotations = defaultdict(list)\n        for ann in coco[\'annotations\']:\n            annotations[int(ann[\'image_id\'])].append(ann)\n        pools = defaultdict(list)\n        for row in sorted(rows, key=lambda item: \'test__\' + item[\'filename\']):\n            image_id = int(row[\'image_id\'])\n            info = images[image_id]\n            require(PurePosixPath(info[\'file_name\']).name == row[\'filename\'], \'Test COCO image identity differs.\')\n            width, height = int(info[\'width\']), int(info[\'height\'])\n            counts = Counter()\n            for annotation_ordinal, ann in enumerate(annotations[image_id]):\n                cls = classes.index(category[int(ann[\'category_id\'])])\n                x, y, bw, bh = map(float, ann[\'bbox\'])\n                require(all(math.isfinite(value) for value in [x, y, bw, bh]) and bw > 0 and bh > 0,\n                        \'Invalid Test annotation box.\')\n                x1, y1, x2, y2 = max(0, min(width, x)), max(0, min(height, y)), max(0, min(width, x + bw)), max(0, min(height, y + bh))\n                require(x2 > x1 and y2 > y1, \'Empty clipped Test box.\')\n                xc, yc, wn, hn = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height\n                line = f\'{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}\'\n                _, xc, yc, wn, hn = map(float, line.split())\n                counts[classes[cls]] += 1\n                if wn > 0.003 and hn > 0.003:\n                    pools[cls].append({\'class_id\': cls, \'class_name\': classes[cls], \'image_id\': image_id,\n                        \'annotation_id\': int(ann[\'id\']), \'annotation_ordinal_in_image\': annotation_ordinal,\n                        \'export_name\': \'test__\' + row[\'filename\'], \'filename\': row[\'filename\'],\n                        \'width\': width, \'height\': height, \'original_split\': \'test\', \'project_split\': \'test\',\n                        \'xywh_normalized\': [xc, yc, wn, hn], \'converted_yolo_line\': line,\n                        \'official_clipped_box_xyxy\': [x1, y1, x2, y2], \'frozen_manifest_row\': row})\n            require(all(counts[name] == int(row[name]) for name in classes), \'Test annotation counts differ from frozen manifest.\')\n        rng = np.random.default_rng(frozen_protocol[\'seed\'])\n        selected = []\n        for cls in range(4):\n            require(pools[cls], \'A prespecified material has no eligible Test crops.\')\n            indices = rng.choice(len(pools[cls]), size=min(frozen_protocol[\'per_class\'], len(pools[cls])), replace=False)\n            selected.extend(dict(pools[cls][int(index)]) for index in indices)\n        selected_names = {item[\'export_name\'] for item in selected}\n        require(not selected_names & gallery_names, \'Test queries overlap the Train gallery source images.\')\n        member_by_name = {name: member(\'test/data/\' + name.removeprefix(\'test__\')) for name in selected_names}\n        source_records = {}\n        for ordinal, name in enumerate(sorted(selected_names, key=lambda value: member_by_name[value].header_offset), 1):\n            path, raw, receipt = fetch(member_by_name[name], name)\n            with Image.open(io.BytesIO(raw)) as image:\n                record = next(item for item in selected if item[\'export_name\'] == name)\n                require(image.format == \'PNG\' and image.size == (record[\'width\'], record[\'height\']), \'Test PNG dimensions/format differ.\')\n                image.verify()\n            source_records[name] = {\'path\': str(path), **receipt}\n            if ordinal % 10 == 0 or ordinal == len(selected_names):\n                print(\'Verified selected Test source PNGs:\', ordinal, \'/\', len(selected_names))\n        stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n        work_dir = local / (\'extension_test_inputs_\' + stamp)\n        (work_dir / \'test_queries\').mkdir(parents=True, exist_ok=False)\n        for ordinal, record in enumerate(selected):\n            source = source_records[record[\'export_name\']]\n            width, height = record[\'width\'], record[\'height\']\n            xc, yc, bw, bh = record[\'xywh_normalized\']\n            x1 = max(0, int((xc - bw / 2) * width))\n            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))\n            y1 = max(0, int((yc - bh / 2) * height))\n            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))\n            box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]\n            relative = f\'test_queries/t_{ordinal:03d}.png\'\n            path = work_dir / relative\n            with Image.open(source[\'path\']) as image:\n                image.convert(\'RGB\').crop(tuple(box)).save(path, format=\'PNG\')\n            record.update({\'ordinal\': ordinal, \'source_image_sha256\': source[\'sha256\'],\n                           \'crop_file\': relative, \'crop_path\': str(path), \'crop_sha256\': sha(path.read_bytes()),\n                           \'crop_box_xyxy\': box, \'crop_width\': box[2] - box[0], \'crop_height\': box[3] - box[1]})\n        require(sha(protocol_file.read_bytes()) == protocol_sha, \'Frozen Test protocol changed during preparation.\')\n        provenance = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n            \'selection\': \'Fixed seed20261116, up to50 eligible objects per material without replacement; use all if fewer than50.\',\n            \'seed\': 20261116, \'bit_generator\': type(rng.bit_generator).__name__,\n            \'sample_objects\': len(selected), \'unique_source_images\': len(selected_names),\n            \'counts_per_class\': {classes[cls]: sum(item[\'class_id\'] == cls for item in selected) for cls in range(4)},\n            \'eligible_objects_per_class\': {classes[cls]: len(pools[cls]) for cls in range(4)},\n            \'full_Test_partition_images\': 929, \'scope\': \'Prespecified material-balanced GT-crop Test subset; not a full929-image system evaluation.\',\n            \'annotation_source\': annotation_receipt, \'source_images\': source_records,\n            \'historical_full_PNG_or_Test_COCO_hash_anchors_available\': False,\n            \'new_source_verification\': \'Selected source ZIP members checked by CRC/size and new SHA256 receipts; identities/counts checked against frozen manifest.\',\n            \'whole_archive_md5_recomputed_this_run\': False, \'archive_url\': archive_url, \'archive_total_bytes\': archive_size,\n            \'range_bytes_transferred\': remote.transferred if remote else 0,\n            \'protocol_sha256_before_Test_annotation_access\': protocol_sha,\n            \'gallery_source_images_disjoint\': True, \'new_model_execution_in_preparation\': False}\n        return {\'queries\': selected, \'work_dir\': str(work_dir), \'provenance\': provenance}\n    finally:\n        archive.close()\n        if remote is not None:\n            remote.close()\n', 'run_extension_test_once_colab.py': '"""One prespecified held-out GT-crop Test evaluation for the two extensions."""\n\n\ndef run_extensions_test_once():\n    from datetime import datetime, timezone\n    from importlib.metadata import version, PackageNotFoundError\n    from pathlib import Path\n    import hashlib\n    import json\n    import os\n    import shutil\n    import subprocess\n    import sys\n    import zipfile\n    from google.colab import drive, files\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def digest(path):\n        value = hashlib.sha256()\n        with Path(path).open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n                value.update(chunk)\n        return value.hexdigest()\n\n    def save_new(path, value):\n        with Path(path).open(\'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n            stream.write(\'\\n\')\n\n    def installed(name):\n        try:\n            return version(name)\n        except PackageNotFoundError:\n            return None\n\n    drive.mount(\'/content/drive\')\n    project = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    output = project / \'evaluation/EXTENSIONS_FINAL_TEST_ONCE\'\n    archive_path = project / \'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip\'\n    if output.exists():\n        require((output / \'COMPLETED.json\').is_file() and archive_path.is_file(),\n                \'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.\')\n        inventory = json.loads((output / \'collection_manifest.json\').read_text(encoding=\'utf-8\'))[\'files\']\n        with zipfile.ZipFile(archive_path) as archive:\n            require(archive.testzip() is None, \'Saved Test evidence ZIP failed CRC verification.\')\n            for item in inventory:\n                raw = archive.read(item[\'file\'])\n                require(len(raw) == item[\'bytes\'] and hashlib.sha256(raw).hexdigest() == item[\'sha256\'],\n                        \'Saved Test archive bytes differ: \' + item[\'file\'])\n        print(\'Previously completed extension Test found; downloading its saved evidence without model execution.\')\n        files.download(str(archive_path))\n        return {\'already_completed\': True, \'zip\': str(archive_path)}\n\n    os.environ[\'WANDB_MODE\'] = \'disabled\'\n    os.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n    os.environ[\'HF_HUB_DISABLE_TELEMETRY\'] = \'1\'\n    torch_version, vision_version = installed(\'torch\'), installed(\'torchvision\')\n    require(torch_version and vision_version, \'A matching preinstalled PyTorch/torchvision runtime is required.\')\n    desired = {\'transformers\': \'5.22.1\', \'numpy\': \'2.4.6\', \'Pillow\': \'11.3.0\'}\n    changes = [name + \'==\' + value for name, value in desired.items() if installed(name) != value]\n    changes += [name for name in [\'requests\'] if installed(name) is None]\n    if changes:\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'torch==\' + torch_version,\n                        \'torchvision==\' + vision_version, *changes], check=True, timeout=360)\n    import numpy as np\n    import PIL\n    from PIL import Image\n    require(np.__version__ == \'2.4.6\' and PIL.__version__ == \'11.3.0\',\n            \'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.\')\n    require(installed(\'torch\') == torch_version and installed(\'torchvision\') == vision_version,\n            \'The preinstalled torch pair unexpectedly changed.\')\n\n    validation = project / \'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z\'\n    assets = validation / \'deployment/assets\'\n    frozen_val = json.loads((validation / \'frozen_operator_VAL_protocol.json\').read_text(encoding=\'utf-8\'))\n    manifest = json.loads((validation / \'deployment/assist_manifest.json\').read_text(encoding=\'utf-8\'))\n    require(digest(validation / \'frozen_operator_VAL_protocol.json\') == _EXT_TEST_ANCHORS[\'val_protocol\'],\n            \'The already-audited Validation protocol differs.\')\n    require(digest(assets / \'gallery_index.npz\') == _EXT_TEST_ANCHORS[\'gallery_index\'],\n            \'The fixed160-reference gallery differs.\')\n    require(digest(assets / \'gallery_meta.json\') == _EXT_TEST_ANCHORS[\'gallery_meta\'], \'Gallery identities differ.\')\n    require(hashlib.sha256(_EXT_TEST_SOURCES[\'extension_assist_models.py\'].encode()).hexdigest()\n            == frozen_val[\'source_files_sha256\'][\'extension_assist_models.py\'], \'The model helper source changed after Validation.\')\n    protected_relative = _EXT_TEST_PROTECTED_PATHS\n    protected = {name: project / relative for name, relative in protected_relative.items()}\n    before = {name: digest(path) for name, path in protected.items()}\n    require(before == frozen_val[\'original_project_files_sha256\'], \'An original protected project reference differs.\')\n    for relative, expected in manifest[\'assets_sha256\'].items():\n        require(digest(assets / relative) == expected, \'Validation gallery/crop asset differs: \' + relative)\n    print(\'Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...\')\n    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest[\'clip_revision\'])\n    resources = engine.runtime_metadata()\n    for key in [\'clip_revision\', \'clip_checkpoint_sha256\', \'resnet_checkpoint_sha256\']:\n        require(resources[key] == manifest[\'model_resources\'][key], \'Official assistance model resource differs: \' + key)\n    gallery_meta = json.loads((assets / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n    gallery_crops = []\n    for item in gallery_meta:\n        with Image.open(assets / item[\'crop_file\']) as image:\n            gallery_crops.append(image.convert(\'RGB\'))\n    recomputed = engine.embed_resnet(gallery_crops)\n    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)\n    require(bool(np.all(cosines >= 0.999)), \'Runtime reference embedding compatibility failed before Test.\')\n    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-7),\n            \'Runtime histogram preprocessing differs from the fixed Validation reference bank.\')\n    del gallery_crops, recomputed\n\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    protocol = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n        \'manifest_sha256\': before[\'split_manifest\'], \'per_class\': 50, \'seed\': 20261116,\n        \'sampling_design\': \'Up to50 eligible GT objects per material without replacement; all if fewer than50. Independent RNGseed20261116, before viewing model outputs.\',\n        \'target_maximum_query_objects\': 200, \'full_Test_partition_images\': 929,\n        \'scope\': \'Prespecified material-balanced GT-crop Test subset, not a full929-image or end-to-end system evaluation.\',\n        \'small_box_filter\': \'Normalized width>0.003 and height>0.003, eight-decimal readback from original conversion.\',\n        \'gallery_assets_path\': str(assets), \'gallery_objects\': 160, \'output_dir\': str(output),\n        \'validation_protocol_sha256\': _EXT_TEST_ANCHORS[\'val_protocol\'],\n        \'gallery_index_sha256\': _EXT_TEST_ANCHORS[\'gallery_index\'], \'gallery_metadata_sha256\': _EXT_TEST_ANCHORS[\'gallery_meta\'],\n        \'source_files_sha256\': {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()},\n        \'model_resources\': resources, \'class_names\': list(CLASS_NAMES), \'clip_prompts\': list(PLAIN_PROMPTS),\n        \'prompt_model_threshold_selection_on_Test\': False,\n        \'histogram_baseline\': \'PIL11.3 RGBresize64,8bins/channel,L2unit; retrievaldescendingargsort, classificationfirstargmax.\',\n        \'reference_compatibility_min_cosine_before_Test\': float(cosines.min()),\n        \'protected_original_files_before\': before, \'M4_M5_Test_model_calls\': 0,\n        \'new_training_or_optimizer_steps\': 0, \'once_only_policy\': \'Fixed folder; completed attempts are downloaded again without model execution; incomplete attempts require explicit recovery review.\'}\n    output.mkdir(parents=True, exist_ok=False)\n    save_new(output / \'frozen_protocol.json\', protocol)\n    protocol_sha = digest(output / \'frozen_protocol.json\')\n    save_new(output / \'STATUS.json\', {\'phase\': \'preparing_inputs_no_Test_model_calls_yet\', \'protocol_sha256\': protocol_sha})\n    code_dir = output / \'code\'\n    code_dir.mkdir()\n    for name, source in _EXT_TEST_SOURCES.items():\n        (code_dir / name).write_text(source, encoding=\'utf-8\', newline=\'\\n\')\n    print(\'Frozen protocol saved BEFORE source Test annotation access. Preparing only the selected Test crops...\')\n    preparation = prepare_extension_test_inputs(protocol, output)\n    records = preparation[\'queries\']\n    work = Path(preparation[\'work_dir\'])\n    save_new(output / \'query_provenance.json\', preparation)\n    shutil.copytree(work / \'test_queries\', output / \'test_queries\')\n    reference_dir = output / \'reference_gallery\'\n    reference_dir.mkdir()\n    shutil.copytree(assets / \'gallery\', reference_dir / \'gallery\')\n    shutil.copyfile(assets / \'gallery_index.npz\', reference_dir / \'gallery_index.npz\')\n    shutil.copyfile(assets / \'gallery_meta.json\', reference_dir / \'gallery_meta.json\')\n    batch_dir = output / \'inference_batches\'\n    batch_dir.mkdir()\n    (output / \'STATUS.json\').write_text(json.dumps({\'phase\': \'Test_inference_started\',\n        \'protocol_sha256\': protocol_sha, \'query_objects\': len(records)}, indent=2), encoding=\'utf-8\')\n    q_parts, h_parts, l_parts = [], [], []\n    print(\'One fixed Test evaluation:\', len(records), \'annotated crops; no detector/segmenter calls.\')\n    for start in range(0, len(records), 8):\n        crops = []\n        for item in records[start:start + 8]:\n            path = output / item[\'crop_file\']\n            require(digest(path) == item[\'crop_sha256\'], \'Selected Test crop changed.\')\n            with Image.open(path) as image:\n                crops.append(image.convert(\'RGB\'))\n        q, h, logits = engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8)\n        path = batch_dir / f\'batch_{start // 8:03d}.npz\'\n        with path.open(\'xb\') as stream:\n            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)),\n                                embeddings=q, histograms=h, clip_logits=logits)\n        q_parts.append(q); h_parts.append(h); l_parts.append(logits)\n        print(\'Recorded Test crops:\', start + len(crops), \'/\', len(records))\n    q, h, logits = np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts)\n    labels = np.asarray([item[\'class_id\'] for item in records], dtype=np.int64)\n    gy = engine.gallery_labels\n    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)\n    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)\n    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]\n    clip_pred = logits.argmax(axis=1)\n    comparison = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\',\n        \'sample_objects\': len(records), \'gallery_objects\': 160,\n        \'sample_counts\': preparation[\'provenance\'][\'counts_per_class\'],\n        \'unique_source_images\': preparation[\'provenance\'][\'unique_source_images\'],\n        \'visual_search_resnet\': search, \'visual_search_histogram\': baseline_search,\n        \'clip_plain\': classification_metrics(labels, clip_pred),\n        \'histogram_classification_baseline\': classification_metrics(labels, hist_pred),\n        \'frozen_protocol_sha256\': protocol_sha,\n        \'limits\': [\'Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.\',\n                   \'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.\',\n                   \'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.\']}\n    save_new(output / \'comparison_TEST.json\', comparison)\n    with (output / \'test_arrays.npz\').open(\'xb\') as stream:\n        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits,\n                            query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank,\n                            histogram_class_predictions=hist_pred)\n    rows = [{\'query_index\': index, \'crop_file\': item[\'crop_file\'], \'source_export_name\': item[\'export_name\'],\n             \'annotation_id\': item[\'annotation_id\'], \'ground_truth_class_id\': int(labels[index]),\n             \'resnet_top1_class_id\': int(gy[rank[index, 0]]),\n             \'resnet_top5_gallery_indices\': [int(value) for value in rank[index, :5]],\n             \'clip_class_id\': int(clip_pred[index]), \'clip_logits\': logits[index].tolist(),\n             \'histogram_class_id\': int(hist_pred[index])} for index, item in enumerate(records)]\n    save_new(output / \'per_query_TEST.json\', rows)\n    after = {name: digest(path) for name, path in protected.items()}\n    require(before == after and digest(output / \'frozen_protocol.json\') == protocol_sha,\n            \'An original file or frozen Test protocol changed.\')\n    save_new(output / \'protected_original_files.json\', {\'before\': before, \'after\': after,\n        \'unchanged\': True, \'original_M4_M5_Test_calls\': 0, \'training_or_optimizer_steps\': 0})\n    save_new(output / \'COMPLETED.json\', {\'status\': \'completed_once\', \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'frozen_protocol_sha256\': protocol_sha, \'comparison_sha256\': digest(output / \'comparison_TEST.json\'),\n        \'zip_name\': archive_path.name, \'M4_M5_Test_reevaluated\': False})\n    (output / \'STATUS.json\').write_text(json.dumps({\'phase\': \'completed_once\', \'protocol_sha256\': protocol_sha}), encoding=\'utf-8\')\n    inventory = [{\'file\': path.relative_to(output).as_posix(), \'bytes\': path.stat().st_size, \'sha256\': digest(path)}\n                 for path in sorted(output.rglob(\'*\')) if path.is_file()]\n    save_new(output / \'collection_manifest.json\', {\'files\': inventory})\n    with zipfile.ZipFile(archive_path, \'x\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file():\n                archive.write(path, path.relative_to(output).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        require(archive.testzip() is None, \'Test evidence ZIP CRC check failed.\')\n        for item in inventory:\n            raw = archive.read(item[\'file\'])\n            require(len(raw) == item[\'bytes\'] and hashlib.sha256(raw).hexdigest() == item[\'sha256\'], \'Packaged Test evidence differs.\')\n    print(json.dumps(comparison, indent=2))\n    print(\'Extension Test complete. No M4/M5 Test evaluation or training was run.\')\n    print(\'Saved original results unchanged. ZIP:\', archive_path)\n    files.download(str(archive_path))\n    return {\'zip\': str(archive_path), \'output_dir\': str(output), \'comparison\': comparison}\n\n\n_EXTENSION_TEST_RESULT = run_extensions_test_once()\n'}
_EXT_TEST_PROTECTED_PATHS = {'split_manifest': 'data_protocol/ef6b35d2ba7f/split_manifest.csv', 'validation_inventory': 'evaluation/M5_BASELINE_VAL_20261115T154613464861Z/val_restore_provenance_20261115T155630452928Z/restoration_inventory.json', 'gallery_index': 'open_world/visual_search/gallery_index_train.npz', 'visual_search_result': 'open_world/visual_search/comparison_VAL.json', 'clip_result': 'open_world/zero_shot_clip/CLIP_vs_histogram_VAL.csv', 'service_source': 'deployment/http_service/service.py', 'service_manifest': 'deployment/http_service/release_manifest.json', 'http_detector': 'deployment/http_service/best_det.pt', 'http_segmenter': 'deployment/http_service/best_seg.pt', 'source_detector': 'runs/M4_yolo11n_640_grouped_v2_seed42/weights/best.pt', 'source_segmenter': 'runs/M5_yolo11n_seg_640_grouped_v2_seed42/weights/best.pt', 'test_protocol': 'evaluation/FINAL_TEST_ONCE/frozen_protocol.json', 'test_results': 'evaluation/FINAL_TEST_ONCE/results.json'}
_EXT_TEST_ANCHORS = {'val_protocol': '7d2e20f397e4ee0d13485b6998dbda1acc113a88a8f1b9b52ba6ba2d9bbc1b7d', 'gallery_index': '4e2a9cc509e2c54f99186a04019d9f82b3d82096e2e7ff4532ab0017b4571442', 'gallery_meta': '889381006555a967d65f4af69eaa895a6356af48dd93a6d43e1311440691a564'}
_EXT_RECOVERY_FUNCTION_SOURCE = "def recover_extension_test_after_rate_limit():\n    from datetime import datetime, timezone\n    from importlib.metadata import version, PackageNotFoundError\n    from pathlib import Path\n    import hashlib\n    import json\n    import os\n    import shutil\n    import subprocess\n    import sys\n    import zipfile\n    from google.colab import drive, files\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def digest(path):\n        value = hashlib.sha256()\n        with Path(path).open('rb') as stream:\n            for chunk in iter(lambda: stream.read(4 * 1024 ** 2), b''):\n                value.update(chunk)\n        return value.hexdigest()\n\n    def save_new(path, value):\n        with Path(path).open('x', encoding='utf-8', newline='\\n') as stream:\n            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n            stream.write('\\n')\n\n    def installed(name):\n        try:\n            return version(name)\n        except PackageNotFoundError:\n            return None\n    drive.mount('/content/drive')\n    project = Path('/content/drive/MyDrive/Project9_Phase2')\n    output = project / 'evaluation/EXTENSIONS_FINAL_TEST_ONCE'\n    archive_path = project / 'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip'\n    if (output / 'COMPLETED.json').is_file():\n        require((output / 'COMPLETED.json').is_file() and archive_path.is_file(), 'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.')\n        inventory = json.loads((output / 'collection_manifest.json').read_text(encoding='utf-8'))['files']\n        with zipfile.ZipFile(archive_path) as archive:\n            require(archive.testzip() is None, 'Saved Test evidence ZIP failed CRC verification.')\n            for item in inventory:\n                raw = archive.read(item['file'])\n                require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Saved Test archive bytes differ: ' + item['file'])\n        print('Previously completed extension Test found; downloading its saved evidence without model execution.')\n        files.download(str(archive_path))\n        return {'already_completed': True, 'zip': str(archive_path)}\n    require(output.is_dir(), 'The original reserved Test folder is missing. Stop and share this output.')\n    protocol_file = output / 'frozen_protocol.json'\n    status_file = output / 'STATUS.json'\n    require(protocol_file.is_file() and status_file.is_file(), 'Original frozen protocol/status is missing.')\n    protocol_bytes = protocol_file.read_bytes()\n    protocol_sha = hashlib.sha256(protocol_bytes).hexdigest()\n    protocol = json.loads(protocol_bytes.decode('utf-8'))\n    status = json.loads(status_file.read_text(encoding='utf-8'))\n    require(status == {'phase': 'preparing_inputs_no_Test_model_calls_yet', 'protocol_sha256': protocol_sha}, 'Recovery is restricted to the recorded failure BEFORE Test inference; status/hash differs.')\n    blocked = ['query_provenance.json', 'test_queries', 'reference_gallery', 'inference_batches', 'test_arrays.npz', 'comparison_TEST.json', 'per_query_TEST.json', 'COMPLETED.json', 'collection_manifest.json']\n    require(not any(((output / name).exists() for name in blocked)) and (not archive_path.exists()), 'Existing Test preparation/results found. Do not evaluate again; share this output for separate recovery review.')\n    require(protocol.get('split') == 'test' and protocol.get('per_class') == 50 and (protocol.get('seed') == 20261116) and (protocol.get('target_maximum_query_objects') == 200) and (protocol.get('full_Test_partition_images') == 929), 'Frozen Test sampling parameters differ.')\n    require(protocol.get('output_dir') == str(output) and protocol.get('gallery_objects') == 160 and (protocol.get('prompt_model_threshold_selection_on_Test') is False) and (protocol.get('M4_M5_Test_model_calls') == 0) and (protocol.get('new_training_or_optimizer_steps') == 0), 'Frozen Test scope/settings differ.')\n    expected_sources = {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()}\n    require(protocol.get('source_files_sha256') == expected_sources, 'Frozen source references differ from the original supplied cell.')\n    for name, expected in expected_sources.items():\n        require(digest(output / 'code' / name) == expected, 'Original saved code bytes differ: ' + name)\n    require({path.name for path in (output / 'code').iterdir()} == set(expected_sources), 'Unexpected files appeared in the original frozen code folder.')\n    memory_diagnostic = {'available': False, 'note': 'The saved protocol bytes, not traceback memory, are the recovery authority.'}\n    last_error = getattr(sys, 'last_exc', None) or getattr(sys, 'last_value', None)\n    trace = getattr(last_error, '__traceback__', None)\n    while trace is not None:\n        frame = trace.tb_frame\n        if frame.f_code.co_name == 'prepare_extension_test_inputs' and 'frozen_protocol' in frame.f_locals:\n            argument = frame.f_locals['frozen_protocol']\n            try:\n                normalized = json.loads(json.dumps(argument, ensure_ascii=False, allow_nan=False))\n                memory_diagnostic.update({'available': True, 'python_object_equal': argument == protocol, 'json_normalized_equal': normalized == protocol})\n                if normalized != protocol:\n                    raise RuntimeError('Original in-memory protocol normalizes differently from the saved frozen file.')\n            except (TypeError, ValueError) as error:\n                raise RuntimeError('Original protocol memory is inconsistent with its saved JSON.') from error\n            break\n        trace = trace.tb_next\n    print('Recovery gate passed: original protocol hash matches, status is before Test, no Test batches/results exist.')\n    os.environ['WANDB_MODE'] = 'disabled'\n    os.environ['TOKENIZERS_PARALLELISM'] = 'false'\n    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'\n    torch_version, vision_version = (installed('torch'), installed('torchvision'))\n    require(torch_version and vision_version, 'A matching preinstalled PyTorch/torchvision runtime is required.')\n    desired = {'transformers': '5.22.1', 'numpy': '2.4.6', 'Pillow': '11.3.0'}\n    changes = [name + '==' + value for name, value in desired.items() if installed(name) != value]\n    changes += [name for name in ['requests'] if installed(name) is None]\n    if changes:\n        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==' + torch_version, 'torchvision==' + vision_version, *changes], check=True, timeout=360)\n    import numpy as np\n    import PIL\n    from PIL import Image\n    require(np.__version__ == '2.4.6' and PIL.__version__ == '11.3.0', 'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.')\n    require(installed('torch') == torch_version and installed('torchvision') == vision_version, 'The preinstalled torch pair unexpectedly changed.')\n    validation = project / 'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z'\n    assets = validation / 'deployment/assets'\n    frozen_val = json.loads((validation / 'frozen_operator_VAL_protocol.json').read_text(encoding='utf-8'))\n    manifest = json.loads((validation / 'deployment/assist_manifest.json').read_text(encoding='utf-8'))\n    require(digest(validation / 'frozen_operator_VAL_protocol.json') == _EXT_TEST_ANCHORS['val_protocol'], 'The already-audited Validation protocol differs.')\n    require(digest(assets / 'gallery_index.npz') == _EXT_TEST_ANCHORS['gallery_index'], 'The fixed160-reference gallery differs.')\n    require(digest(assets / 'gallery_meta.json') == _EXT_TEST_ANCHORS['gallery_meta'], 'Gallery identities differ.')\n    require(hashlib.sha256(_EXT_TEST_SOURCES['extension_assist_models.py'].encode()).hexdigest() == frozen_val['source_files_sha256']['extension_assist_models.py'], 'The model helper source changed after Validation.')\n    protected_relative = _EXT_TEST_PROTECTED_PATHS\n    protected = {name: project / relative for name, relative in protected_relative.items()}\n    before = {name: digest(path) for name, path in protected.items()}\n    require(before == frozen_val['original_project_files_sha256'], 'An original protected project reference differs.')\n    for relative, expected in manifest['assets_sha256'].items():\n        require(digest(assets / relative) == expected, 'Validation gallery/crop asset differs: ' + relative)\n    print('Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...')\n    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest['clip_revision'])\n    resources = engine.runtime_metadata()\n    for key in ['clip_revision', 'clip_checkpoint_sha256', 'resnet_checkpoint_sha256']:\n        require(resources[key] == manifest['model_resources'][key], 'Official assistance model resource differs: ' + key)\n    gallery_meta = json.loads((assets / 'gallery_meta.json').read_text(encoding='utf-8'))\n    gallery_crops = []\n    for item in gallery_meta:\n        with Image.open(assets / item['crop_file']) as image:\n            gallery_crops.append(image.convert('RGB'))\n    recomputed = engine.embed_resnet(gallery_crops)\n    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)\n    require(bool(np.all(cosines >= 0.999)), 'Runtime reference embedding compatibility failed before Test.')\n    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-07), 'Runtime histogram preprocessing differs from the fixed Validation reference bank.')\n    del gallery_crops, recomputed\n    require(protocol['manifest_sha256'] == before['split_manifest'] and protocol['protected_original_files_before'] == before, 'Original protected references differ from frozen Test protocol.')\n    require(protocol['gallery_assets_path'] == str(assets) and protocol['validation_protocol_sha256'] == _EXT_TEST_ANCHORS['val_protocol'] and (protocol['gallery_index_sha256'] == _EXT_TEST_ANCHORS['gallery_index']) and (protocol['gallery_metadata_sha256'] == _EXT_TEST_ANCHORS['gallery_meta']), 'Frozen gallery/Validation anchors differ.')\n    require(protocol['class_names'] == list(CLASS_NAMES) and protocol['clip_prompts'] == list(PLAIN_PROMPTS), 'Frozen class names/prompts differ.')\n    current_resources = json.loads(json.dumps(resources, ensure_ascii=False, allow_nan=False))\n    require(current_resources == protocol['model_resources'], 'Current runtime/model processor metadata differs from frozen protocol; stop before Test.')\n    require(digest(protocol_file) == protocol_sha, 'The original frozen protocol changed during resource checks.')\n    recovery_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')\n    recovery_dir = output / ('recovery_before_inference_' + recovery_stamp)\n    recovery_dir.mkdir(exist_ok=False)\n    with (recovery_dir / 'recovery_function.py').open('x', encoding='utf-8', newline='\\n') as stream:\n        stream.write(_EXT_RECOVERY_FUNCTION_SOURCE)\n    save_new(recovery_dir / 'recovery_receipt.json', {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'reason': 'HTTP429 while retrieving source ZIP ranges, before any Test model batch was run.', 'fix': 'Keep original frozen protocol and sampling/model logic; patch only bounded archive transport to respect server rate-limit waits and reduce requests.', 'frozen_protocol_sha256_unchanged': protocol_sha, 'original_code_sha256_unchanged': expected_sources, 'recovery_function_sha256': hashlib.sha256(_EXT_RECOVERY_FUNCTION_SOURCE.encode()).hexdigest(), 'status_before_recovery': status, 'memory_diagnostic': memory_diagnostic, 'sampling_models_prompts_thresholds_and_preprocessing_changed': False, 'previous_Test_batches_or_results_present': False, 'current_model_resources_json_normalized_match': True, 'original_M4_M5_Test_calls': 0})\n    print('Continuing the same frozen attempt with the original protocol JSON; no frozen file was rewritten.')\n    globals()['_EXT_RATE_TRANSPORT_LOG_PATH'] = str(recovery_dir / 'zenodo_transport.jsonl')\n    with (recovery_dir / 'transport_preparer.py').open('x', encoding='utf-8', newline='\\n') as stream:\n        stream.write(_EXT_RATE_TRANSPORT_PREPARER_SOURCE)\n    with (recovery_dir / 'transport_reader.py').open('x', encoding='utf-8', newline='\\n') as stream:\n        stream.write(_EXT_RATE_TRANSPORT_READER_SOURCE)\n    save_new(recovery_dir / 'transport_patch_receipt.json', {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'reason': 'Zenodo HTTP429 during input preparation; saved status still has zero Test inference batches.', 'official_rate_limit_documentation': 'https://developers.zenodo.org/#rate-limiting', 'frozen_protocol_sha256_preserved': protocol_sha, 'original_preparer_sha256': expected_sources['prepare_extension_test_inputs.py'], 'patched_preparer_sha256': hashlib.sha256(_EXT_RATE_TRANSPORT_PREPARER_SOURCE.encode()).hexdigest(), 'transport_reader_sha256': hashlib.sha256(_EXT_RATE_TRANSPORT_READER_SOURCE.encode()).hexdigest(), 'change_boundary': 'Only the nested HTTPRangeFile class differs; outer sampler/crop/CRC/SHA logic AST-identical.', 'sampling_models_prompts_thresholds_and_preprocessing_changed': False, 'transport': 'Up to3 adjacent1MiB blocks per HTTP request;16-blockLRU;min3s betweenGETs;initial60s cooldown;honorRetry-After/reset;900s wait budget;768MiB transfer cap;strict206Content-Range.', 'full_archive_download_fallback': False, 'original_M4_M5_Test_calls': 0})\n    print('Zenodo download recovery: respecting server cooldown; using cached members where verified.')\n    print('Only required source members will be retrieved. Test sampling/settings and the original protocol are unchanged.')\n    preparation = prepare_extension_test_inputs(protocol, output)\n    records = preparation['queries']\n    work = Path(preparation['work_dir'])\n    save_new(output / 'query_provenance.json', preparation)\n    shutil.copytree(work / 'test_queries', output / 'test_queries')\n    reference_dir = output / 'reference_gallery'\n    reference_dir.mkdir()\n    shutil.copytree(assets / 'gallery', reference_dir / 'gallery')\n    shutil.copyfile(assets / 'gallery_index.npz', reference_dir / 'gallery_index.npz')\n    shutil.copyfile(assets / 'gallery_meta.json', reference_dir / 'gallery_meta.json')\n    batch_dir = output / 'inference_batches'\n    batch_dir.mkdir()\n    (output / 'STATUS.json').write_text(json.dumps({'phase': 'Test_inference_started', 'protocol_sha256': protocol_sha, 'query_objects': len(records)}, indent=2), encoding='utf-8')\n    q_parts, h_parts, l_parts = ([], [], [])\n    print('One fixed Test evaluation:', len(records), 'annotated crops; no detector/segmenter calls.')\n    for start in range(0, len(records), 8):\n        crops = []\n        for item in records[start:start + 8]:\n            path = output / item['crop_file']\n            require(digest(path) == item['crop_sha256'], 'Selected Test crop changed.')\n            with Image.open(path) as image:\n                crops.append(image.convert('RGB'))\n        q, h, logits = (engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8))\n        path = batch_dir / f'batch_{start // 8:03d}.npz'\n        with path.open('xb') as stream:\n            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)), embeddings=q, histograms=h, clip_logits=logits)\n        q_parts.append(q)\n        h_parts.append(h)\n        l_parts.append(logits)\n        print('Recorded Test crops:', start + len(crops), '/', len(records))\n    q, h, logits = (np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts))\n    labels = np.asarray([item['class_id'] for item in records], dtype=np.int64)\n    gy = engine.gallery_labels\n    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)\n    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)\n    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]\n    clip_pred = logits.argmax(axis=1)\n    comparison = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test', 'sample_objects': len(records), 'gallery_objects': 160, 'sample_counts': preparation['provenance']['counts_per_class'], 'unique_source_images': preparation['provenance']['unique_source_images'], 'visual_search_resnet': search, 'visual_search_histogram': baseline_search, 'clip_plain': classification_metrics(labels, clip_pred), 'histogram_classification_baseline': classification_metrics(labels, hist_pred), 'frozen_protocol_sha256': protocol_sha, 'limits': ['Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.', 'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.', 'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.']}\n    save_new(output / 'comparison_TEST.json', comparison)\n    with (output / 'test_arrays.npz').open('xb') as stream:\n        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits, query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank, histogram_class_predictions=hist_pred)\n    rows = [{'query_index': index, 'crop_file': item['crop_file'], 'source_export_name': item['export_name'], 'annotation_id': item['annotation_id'], 'ground_truth_class_id': int(labels[index]), 'resnet_top1_class_id': int(gy[rank[index, 0]]), 'resnet_top5_gallery_indices': [int(value) for value in rank[index, :5]], 'clip_class_id': int(clip_pred[index]), 'clip_logits': logits[index].tolist(), 'histogram_class_id': int(hist_pred[index])} for index, item in enumerate(records)]\n    save_new(output / 'per_query_TEST.json', rows)\n    after = {name: digest(path) for name, path in protected.items()}\n    require(before == after and digest(output / 'frozen_protocol.json') == protocol_sha, 'An original file or frozen Test protocol changed.')\n    save_new(output / 'protected_original_files.json', {'before': before, 'after': after, 'unchanged': True, 'original_M4_M5_Test_calls': 0, 'training_or_optimizer_steps': 0})\n    save_new(output / 'COMPLETED.json', {'status': 'completed_once', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'frozen_protocol_sha256': protocol_sha, 'comparison_sha256': digest(output / 'comparison_TEST.json'), 'zip_name': archive_path.name, 'M4_M5_Test_reevaluated': False})\n    (output / 'STATUS.json').write_text(json.dumps({'phase': 'completed_once', 'protocol_sha256': protocol_sha}), encoding='utf-8')\n    inventory = [{'file': path.relative_to(output).as_posix(), 'bytes': path.stat().st_size, 'sha256': digest(path)} for path in sorted(output.rglob('*')) if path.is_file()]\n    save_new(output / 'collection_manifest.json', {'files': inventory})\n    with zipfile.ZipFile(archive_path, 'x', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob('*')):\n            if path.is_file():\n                archive.write(path, path.relative_to(output).as_posix())\n    with zipfile.ZipFile(archive_path) as archive:\n        require(archive.testzip() is None, 'Test evidence ZIP CRC check failed.')\n        for item in inventory:\n            raw = archive.read(item['file'])\n            require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Packaged Test evidence differs.')\n    print(json.dumps(comparison, indent=2))\n    print('Extension Test complete. No M4/M5 Test evaluation or training was run.')\n    print('Saved original results unchanged. ZIP:', archive_path)\n    files.download(str(archive_path))\n    return {'zip': str(archive_path), 'output_dir': str(output), 'comparison': comparison}\n"
_EXT_RATE_TRANSPORT_PREPARER_SOURCE = '"""Prepare a prespecified material-balanced crop subset of the frozen Test split."""\n\ndef prepare_extension_test_inputs(frozen_protocol, evidence_dir):\n    from collections import Counter, OrderedDict, defaultdict\n    from datetime import datetime, timezone\n    from pathlib import Path, PurePosixPath\n    import csv\n    import hashlib\n    import io\n    import json\n    import math\n    import re\n    import time\n    import zipfile\n    import zlib\n    import numpy as np\n    import requests\n    from PIL import Image\n\n    def require(condition, message):\n        if not condition:\n            raise RuntimeError(message)\n\n    def sha(raw):\n        return hashlib.sha256(raw).hexdigest()\n    evidence_dir = Path(evidence_dir)\n    protocol_file = evidence_dir / \'frozen_protocol.json\'\n    require(protocol_file.is_file() and json.loads(protocol_file.read_text(encoding=\'utf-8\')) == frozen_protocol, \'A byte-hashed frozen protocol must exist before opening Test annotations.\')\n    require(frozen_protocol[\'split\'] == \'test\' and frozen_protocol[\'per_class\'] == 50 and (frozen_protocol[\'seed\'] == 20261116), \'The prespecified Test sample design differs.\')\n    project = Path(\'/content/drive/MyDrive/Project9_Phase2\')\n    local = Path(\'/content/Project9_Phase2\')\n    manifest_path = project / \'data_protocol/ef6b35d2ba7f/split_manifest.csv\'\n    require(sha(manifest_path.read_bytes()) == frozen_protocol[\'manifest_sha256\'], \'Frozen split bytes differ.\')\n    manifest = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding=\'utf-8-sig\'))))\n    rows = [row for row in manifest if row[\'split\'] == \'test\']\n    require(len(rows) == 929 and all((row[\'original_split\'] == \'test\' for row in rows)), \'Frozen Test membership differs.\')\n    gallery = json.loads((Path(frozen_protocol[\'gallery_assets_path\']) / \'gallery_meta.json\').read_text(encoding=\'utf-8\'))\n    gallery_names = {item[\'export_name\'] for item in gallery}\n    classes = [\'cardboard\', \'metal\', \'rigid_plastic\', \'soft_plastic\']\n    archive_url = \'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1\'\n    archive_size = 7518242799\n    password = b\'REDACTED_DATASET_ZIP_PASSWORD\'\n\n    class HTTPRangeFile(io.RawIOBase):\n        """Strict, paced, bounded seekable Zenodo reader; no whole-file fallback.\n\n    Globals are the original preparer\'s io/OrderedDict/requests/require/re/time,\n    archive_url and archive_size. Imports stay inside methods so this class can\n    replace only the original reader AST, without changing the sampler.\n    """\n\n        def __init__(self):\n            super().__init__()\n            self.position = 0\n            self.size = archive_size\n            self.block_size = 1024 * 1024\n            self.blocks = OrderedDict()\n            self.transferred = 0\n            self.requests_count = 0\n            self.http_gets = 0\n            self.waited_seconds = 0.0\n            self.wait_budget = 900.0\n            self.transfer_budget = 768 * 1024 ** 2\n            self._last_get_at = None\n            self._server_not_before = 0.0\n            self._initial_wait_pending = True\n            self._log_created = False\n            self._log_path = globals().get(\'_EXT_RATE_TRANSPORT_LOG_PATH\')\n            self.session = requests.Session()\n            self.rate_policy = {\'source_url\': \'https://developers.zenodo.org/#rate-limiting\', \'user_agent\': \'Project9EducationalArchiveReader/1.1 (+https://github.com/sultan1443/KAUST-AI251P-Project9-Public)\', \'minimum_seconds_between_GET_starts\': 3, \'initial_cooldown_seconds\': 60, \'automatic_wait_budget_seconds\': 900, \'transfer_budget_bytes\': self.transfer_budget, \'block_bytes\': self.block_size, \'maximum_blocks_per_GET\': 3, \'maximum_cached_blocks\': 16, \'maximum_read_bytes\': 64 * 1024 ** 2, \'maximum_attempts_per_range\': 12, \'maximum_network_or_5xx_failures\': 6, \'whole_archive_fallback\': False}\n            self._event(\'reader_policy\', policy=self.rate_policy)\n\n        def _event(self, event, **detail):\n            if self._log_path is None:\n                return\n            from pathlib import Path\n            import json\n            row = {\'event\': event, \'utc_epoch\': time.time(), \'http_gets\': self.http_gets, \'successful_GETs\': self.requests_count, \'accounted_body_bytes\': self.transferred, \'automatic_wait_seconds\': self.waited_seconds, **detail}\n            with Path(self._log_path).open(\'a\' if self._log_created else \'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n                stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + \'\\n\')\n            self._log_created = True\n\n        def _wait(self, seconds, reason, status=None):\n            seconds = max(0.0, float(seconds))\n            remaining = max(0.0, self.wait_budget - self.waited_seconds)\n            if seconds > remaining + 1e-08:\n                self._event(\'wait_budget_stop\', required_wait_seconds=seconds, remaining_wait_budget_seconds=remaining, reason=reason, http_status=status)\n                raise RuntimeError((\'HTTP 429 rate limit: \' if status == 429 else \'HTTP transport: \') + f\'{seconds:.1f} seconds of waiting are required; only {remaining:.1f} \' + \'seconds remain in the automatic wait budget. No earlier retry was made. \' + \'Keep the frozen attempt and share this output; do not restart Test inference.\')\n            if seconds <= 0:\n                return\n            self._event(\'wait_start\', seconds=seconds, reason=reason, http_status=status)\n            left = seconds\n            while left > 1e-08:\n                chunk = min(30.0, left)\n                print(f\'Archive transport: waiting {chunk:.1f}s ({reason}); {left:.1f}s remaining.\')\n                time.sleep(chunk)\n                self.waited_seconds += chunk\n                left -= chunk\n            self._event(\'wait_end\', seconds=seconds, reason=reason, http_status=status)\n\n        def _header_deadline(self, headers):\n            from email.utils import parsedate_to_datetime\n            from datetime import timezone\n            import math\n            now = time.time()\n            deadlines = []\n            retry_after = headers.get(\'Retry-After\')\n            if retry_after is not None:\n                text = str(retry_after).strip()\n                try:\n                    seconds = float(text)\n                    if math.isfinite(seconds) and seconds >= 0:\n                        deadlines.append(now + seconds)\n                except ValueError:\n                    try:\n                        value = parsedate_to_datetime(text)\n                        if value.tzinfo is None:\n                            value = value.replace(tzinfo=timezone.utc)\n                        deadlines.append(value.timestamp())\n                    except (TypeError, ValueError, OverflowError):\n                        pass\n            try:\n                remaining = float(headers.get(\'X-RateLimit-Remaining\', headers.get(\'XRateLimitRemaining\', \'nan\')))\n                reset = float(headers.get(\'X-RateLimit-Reset\', headers.get(\'XRateLimitReset\', \'nan\')))\n                if math.isfinite(remaining) and remaining <= 0 and math.isfinite(reset):\n                    deadlines.append(reset)\n            except (TypeError, ValueError):\n                pass\n            if deadlines:\n                deadline = max(now, *deadlines)\n                self._server_not_before = max(self._server_not_before, deadline)\n                return deadline\n            return None\n\n        def _pace(self):\n            if self._initial_wait_pending:\n                self._wait(60, \'initial cooldown after the recorded HTTP 429\', 429)\n                self._initial_wait_pending = False\n            server_delay = max(0.0, self._server_not_before - time.time())\n            interval_delay = 0.0 if self._last_get_at is None else max(0.0, 3.0 - (time.monotonic() - self._last_get_at))\n            self._wait(max(server_delay, interval_delay), \'server rate-limit deadline\' if server_delay >= interval_delay and server_delay > 0 else \'minimum GET spacing\', 429 if server_delay > 0 else None)\n\n        def readable(self):\n            return True\n\n        def seekable(self):\n            return True\n\n        def tell(self):\n            return self.position\n\n        def seek(self, offset, whence=0):\n            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1\n            if position < 0 or int(position) != position:\n                raise ValueError(\'Invalid HTTP ZIP seek.\')\n            self.position = int(position)\n            return self.position\n\n        def get_block(self, index):\n            from urllib3.exceptions import HTTPError as HTTPTransportError\n            require(not self.closed, \'HTTP ZIP stream is closed.\')\n            require(isinstance(index, int) and 0 <= index < (self.size + self.block_size - 1) // self.block_size, \'Invalid HTTP ZIP block index.\')\n            if index in self.blocks:\n                self.blocks.move_to_end(index)\n                return self.blocks[index]\n            last = index\n            while last + 1 < index + 3 and (last + 1) * self.block_size < self.size and (last + 1 not in self.blocks):\n                last += 1\n            start = index * self.block_size\n            end = min(self.size - 1, (last + 1) * self.block_size - 1)\n            needed = end - start + 1\n            rate_failures = network_failures = server_failures = 0\n            for attempt in range(12):\n                require(self.transferred + needed + 1 <= self.transfer_budget, \'Partial-download 768 MiB budget would be exceeded; no whole-archive fallback or further request.\')\n                self._pace()\n                self._last_get_at = time.monotonic()\n                self.http_gets += 1\n                self._event(\'GET_start\', first_byte=start, last_byte=end, attempt=attempt + 1)\n                try:\n                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={\'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\', \'User-Agent\': self.rate_policy[\'user_agent\']}) as response:\n                        status = response.status_code\n                        deadline = self._header_deadline(response.headers)\n                        self._event(\'GET_headers\', http_status=status, retry_after=response.headers.get(\'Retry-After\'), rate_limit_remaining=response.headers.get(\'X-RateLimit-Remaining\'), rate_limit_reset=response.headers.get(\'X-RateLimit-Reset\'))\n                        if status == 429:\n                            rate_failures += 1\n                            if attempt == 11:\n                                raise RuntimeError(\'HTTP 429: maximum12 range attempts reached; keep the frozen attempt. No early retry or full download.\')\n                            delay = max(0.0, deadline - time.time()) if deadline is not None else min(300.0, 60.0 * 2 ** (rate_failures - 1))\n                            response.close()\n                            self._wait(delay, \'HTTP 429 Retry-After/reset\' if deadline is not None else \'HTTP 429 bounded backoff\', 429)\n                            continue\n                        if status in (500, 502, 503, 504):\n                            server_failures += 1\n                            require(server_failures < 6 and attempt < 11, f\'HTTP {status}: bounded range retries exhausted; no full download.\')\n                            response.close()\n                            self._wait(max(0.0 if deadline is None else deadline - time.time(), min(60.0, 5.0 * 2 ** (server_failures - 1))), f\'HTTP {status} bounded retry\', status)\n                            continue\n                        require(status == 206, f\'HTTP {status}: requested range was not supported; full download prevented (body not read).\')\n                        content_range = response.headers.get(\'Content-Range\', \'\').strip()\n                        match = re.fullmatch(\'bytes (\\\\d+)-(\\\\d+)/(\\\\d+)\', content_range, flags=re.I)\n                        require(match and tuple(map(int, match.groups())) == (start, end, self.size), \'Server returned an unexpected Content-Range: \' + content_range)\n                        encoding = response.headers.get(\'Content-Encoding\', \'identity\').strip().lower()\n                        require(encoding in (\'\', \'identity\'), \'Encoded HTTP range rejected; exact archive bytes are required.\')\n                        length = response.headers.get(\'Content-Length\')\n                        require(length is None or str(length).strip() == str(needed), \'HTTP range Content-Length differs.\')\n                        try:\n                            data = response.raw.read(needed + 1)\n                        except (requests.RequestException, HTTPTransportError, OSError):\n                            self.transferred += needed + 1\n                            raise\n                        self.transferred += len(data)\n                        require(len(data) == needed, \'Incomplete or oversized HTTP range response.\')\n                    self.requests_count += 1\n                    for block in range(index, last + 1):\n                        lo = (block - index) * self.block_size\n                        hi = min(needed, lo + self.block_size)\n                        self.blocks[block] = data[lo:hi]\n                        self.blocks.move_to_end(block)\n                    while len(self.blocks) > 16:\n                        self.blocks.popitem(last=False)\n                    self._event(\'GET_success\', first_byte=start, last_byte=end, received_bytes=len(data), cached_blocks=len(self.blocks))\n                    if self.requests_count % 5 == 0:\n                        print(f\'Partial ZIP transport: {self.transferred / 1024 ** 2:.1f} MiB; {self.requests_count} successful GETs.\')\n                    return self.blocks[index]\n                except (requests.RequestException, HTTPTransportError, OSError) as error:\n                    network_failures += 1\n                    self._event(\'GET_network_error\', error_type=type(error).__name__, failures=network_failures)\n                    require(network_failures < 6 and attempt < 11, \'HTTP range network timeouts/errors exhausted the bounded retries; no full download.\')\n                    self._wait(min(60.0, 5.0 * 2 ** (network_failures - 1)), \'bounded network-error retry\')\n            raise RuntimeError(\'HTTP range attempt limit reached; no full download.\')\n\n        def read(self, count=-1):\n            if self.closed:\n                raise ValueError(\'HTTP ZIP stream is closed.\')\n            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))\n            require(count <= 64 * 1024 ** 2, \'Unexpected large ZIP read; complete-archive download prevented.\')\n            chunks = []\n            remaining = count\n            while remaining:\n                index, offset = divmod(self.position, self.block_size)\n                data = self.get_block(index)\n                take = min(remaining, len(data) - offset)\n                require(take > 0, \'Unexpected ZIP read position.\')\n                chunks.append(data[offset:offset + take])\n                self.position += take\n                remaining -= take\n            return b\'\'.join(chunks)\n\n        def close(self):\n            self.session.close()\n            self.blocks.clear()\n            super().close()\n    cache = local / \'extensions_test_source_images\'\n    cache.mkdir(parents=True, exist_ok=True)\n    existing_archive = next((path for path in [Path(\'/content/zerowaste-f-final.zip\'), project / \'cache/zerowaste-f-final.zip\'] if path.is_file() and path.stat().st_size == archive_size), None)\n    remote = None\n    if existing_archive is None:\n        remote = HTTPRangeFile()\n        archive = zipfile.ZipFile(remote)\n    else:\n        archive = zipfile.ZipFile(existing_archive)\n    protocol_sha = sha(protocol_file.read_bytes())\n    try:\n\n        def member(suffix):\n            require(suffix.startswith(\'test/\') and \'..\' not in PurePosixPath(suffix).parts, \'Only source Test members selected by the frozen protocol may be read.\')\n            matches = [item for item in archive.infolist() if not item.is_dir() and (item.filename == suffix or item.filename.endswith(\'/\' + suffix)) and (\'..\' not in PurePosixPath(item.filename).parts)]\n            require(len(matches) == 1, \'Test member is not unique: \' + suffix)\n            return matches[0]\n\n        def fetch(info, filename):\n            path = cache / filename\n            sidecar = cache / (filename + \'.source.json\')\n            require(0 < info.file_size <= (256 * 1024 ** 2 if filename.endswith(\'.json\') else 12 * 1024 ** 2), \'Unexpected source member size.\')\n            if path.exists():\n                require(sidecar.is_file(), \'Existing source cache has no provenance receipt.\')\n                raw = path.read_bytes()\n                receipt = json.loads(sidecar.read_text(encoding=\'utf-8\'))\n                reused = True\n            else:\n                raw = archive.read(info, pwd=password)\n                receipt = {\'sha256\': sha(raw), \'member\': info.filename, \'member_crc32\': info.CRC, \'member_bytes\': info.file_size, \'archive_url\': archive_url, \'archive_total_bytes\': archive_size}\n                with path.open(\'xb\') as stream:\n                    stream.write(raw)\n                with sidecar.open(\'x\', encoding=\'utf-8\') as stream:\n                    json.dump(receipt, stream, indent=2)\n                reused = False\n            require(len(raw) == info.file_size and zlib.crc32(raw) & 4294967295 == info.CRC, \'Source member CRC/size differs.\')\n            expected = {\'sha256\': sha(raw), \'member\': info.filename, \'member_crc32\': info.CRC, \'member_bytes\': info.file_size, \'archive_url\': archive_url, \'archive_total_bytes\': archive_size}\n            require(receipt == expected, \'Source cache provenance differs.\')\n            return (path, raw, {**receipt, \'reused_cache\': reused, \'historical_full_source_image_sha256_anchor_available\': False})\n        _, raw_annotations, annotation_receipt = fetch(member(\'test/labels.json\'), \'source_test_annotations.json\')\n        coco = json.loads(raw_annotations.decode(\'utf-8-sig\'))\n        images = {int(item[\'id\']): item for item in coco[\'images\']}\n        category = {int(item[\'id\']): item[\'name\'] for item in coco[\'categories\']}\n        require(set(category.values()) == set(classes), \'Source material categories differ.\')\n        annotations = defaultdict(list)\n        for ann in coco[\'annotations\']:\n            annotations[int(ann[\'image_id\'])].append(ann)\n        pools = defaultdict(list)\n        for row in sorted(rows, key=lambda item: \'test__\' + item[\'filename\']):\n            image_id = int(row[\'image_id\'])\n            info = images[image_id]\n            require(PurePosixPath(info[\'file_name\']).name == row[\'filename\'], \'Test COCO image identity differs.\')\n            width, height = (int(info[\'width\']), int(info[\'height\']))\n            counts = Counter()\n            for annotation_ordinal, ann in enumerate(annotations[image_id]):\n                cls = classes.index(category[int(ann[\'category_id\'])])\n                x, y, bw, bh = map(float, ann[\'bbox\'])\n                require(all((math.isfinite(value) for value in [x, y, bw, bh])) and bw > 0 and (bh > 0), \'Invalid Test annotation box.\')\n                x1, y1, x2, y2 = (max(0, min(width, x)), max(0, min(height, y)), max(0, min(width, x + bw)), max(0, min(height, y + bh)))\n                require(x2 > x1 and y2 > y1, \'Empty clipped Test box.\')\n                xc, yc, wn, hn = ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height)\n                line = f\'{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}\'\n                _, xc, yc, wn, hn = map(float, line.split())\n                counts[classes[cls]] += 1\n                if wn > 0.003 and hn > 0.003:\n                    pools[cls].append({\'class_id\': cls, \'class_name\': classes[cls], \'image_id\': image_id, \'annotation_id\': int(ann[\'id\']), \'annotation_ordinal_in_image\': annotation_ordinal, \'export_name\': \'test__\' + row[\'filename\'], \'filename\': row[\'filename\'], \'width\': width, \'height\': height, \'original_split\': \'test\', \'project_split\': \'test\', \'xywh_normalized\': [xc, yc, wn, hn], \'converted_yolo_line\': line, \'official_clipped_box_xyxy\': [x1, y1, x2, y2], \'frozen_manifest_row\': row})\n            require(all((counts[name] == int(row[name]) for name in classes)), \'Test annotation counts differ from frozen manifest.\')\n        rng = np.random.default_rng(frozen_protocol[\'seed\'])\n        selected = []\n        for cls in range(4):\n            require(pools[cls], \'A prespecified material has no eligible Test crops.\')\n            indices = rng.choice(len(pools[cls]), size=min(frozen_protocol[\'per_class\'], len(pools[cls])), replace=False)\n            selected.extend((dict(pools[cls][int(index)]) for index in indices))\n        selected_names = {item[\'export_name\'] for item in selected}\n        require(not selected_names & gallery_names, \'Test queries overlap the Train gallery source images.\')\n        member_by_name = {name: member(\'test/data/\' + name.removeprefix(\'test__\')) for name in selected_names}\n        source_records = {}\n        for ordinal, name in enumerate(sorted(selected_names, key=lambda value: member_by_name[value].header_offset), 1):\n            path, raw, receipt = fetch(member_by_name[name], name)\n            with Image.open(io.BytesIO(raw)) as image:\n                record = next((item for item in selected if item[\'export_name\'] == name))\n                require(image.format == \'PNG\' and image.size == (record[\'width\'], record[\'height\']), \'Test PNG dimensions/format differ.\')\n                image.verify()\n            source_records[name] = {\'path\': str(path), **receipt}\n            if ordinal % 10 == 0 or ordinal == len(selected_names):\n                print(\'Verified selected Test source PNGs:\', ordinal, \'/\', len(selected_names))\n        stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n        work_dir = local / (\'extension_test_inputs_\' + stamp)\n        (work_dir / \'test_queries\').mkdir(parents=True, exist_ok=False)\n        for ordinal, record in enumerate(selected):\n            source = source_records[record[\'export_name\']]\n            width, height = (record[\'width\'], record[\'height\'])\n            xc, yc, bw, bh = record[\'xywh_normalized\']\n            x1 = max(0, int((xc - bw / 2) * width))\n            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))\n            y1 = max(0, int((yc - bh / 2) * height))\n            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))\n            box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]\n            relative = f\'test_queries/t_{ordinal:03d}.png\'\n            path = work_dir / relative\n            with Image.open(source[\'path\']) as image:\n                image.convert(\'RGB\').crop(tuple(box)).save(path, format=\'PNG\')\n            record.update({\'ordinal\': ordinal, \'source_image_sha256\': source[\'sha256\'], \'crop_file\': relative, \'crop_path\': str(path), \'crop_sha256\': sha(path.read_bytes()), \'crop_box_xyxy\': box, \'crop_width\': box[2] - box[0], \'crop_height\': box[3] - box[1]})\n        require(sha(protocol_file.read_bytes()) == protocol_sha, \'Frozen Test protocol changed during preparation.\')\n        provenance = {\'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'split\': \'test\', \'selection\': \'Fixed seed20261116, up to50 eligible objects per material without replacement; use all if fewer than50.\', \'seed\': 20261116, \'bit_generator\': type(rng.bit_generator).__name__, \'sample_objects\': len(selected), \'unique_source_images\': len(selected_names), \'counts_per_class\': {classes[cls]: sum((item[\'class_id\'] == cls for item in selected)) for cls in range(4)}, \'eligible_objects_per_class\': {classes[cls]: len(pools[cls]) for cls in range(4)}, \'full_Test_partition_images\': 929, \'scope\': \'Prespecified material-balanced GT-crop Test subset; not a full929-image system evaluation.\', \'annotation_source\': annotation_receipt, \'source_images\': source_records, \'historical_full_PNG_or_Test_COCO_hash_anchors_available\': False, \'new_source_verification\': \'Selected source ZIP members checked by CRC/size and new SHA256 receipts; identities/counts checked against frozen manifest.\', \'whole_archive_md5_recomputed_this_run\': False, \'archive_url\': archive_url, \'archive_total_bytes\': archive_size, \'range_bytes_transferred\': remote.transferred if remote else 0, \'protocol_sha256_before_Test_annotation_access\': protocol_sha, \'gallery_source_images_disjoint\': True, \'new_model_execution_in_preparation\': False}\n        return {\'queries\': selected, \'work_dir\': str(work_dir), \'provenance\': provenance}\n    finally:\n        archive.close()\n        if remote is not None:\n            remote.close()\n'
_EXT_RATE_TRANSPORT_READER_SOURCE = 'class HTTPRangeFile(io.RawIOBase):\n    """Strict, paced, bounded seekable Zenodo reader; no whole-file fallback.\n\n    Globals are the original preparer\'s io/OrderedDict/requests/require/re/time,\n    archive_url and archive_size. Imports stay inside methods so this class can\n    replace only the original reader AST, without changing the sampler.\n    """\n\n    def __init__(self):\n        super().__init__()\n        self.position = 0\n        self.size = archive_size\n        self.block_size = 1024 * 1024\n        self.blocks = OrderedDict()\n        self.transferred = 0\n        self.requests_count = 0\n        self.http_gets = 0\n        self.waited_seconds = 0.0\n        self.wait_budget = 900.0\n        self.transfer_budget = 768 * 1024 ** 2\n        self._last_get_at = None\n        self._server_not_before = 0.0\n        self._initial_wait_pending = True\n        self._log_created = False\n        self._log_path = globals().get(\'_EXT_RATE_TRANSPORT_LOG_PATH\')\n        self.session = requests.Session()\n        self.rate_policy = {\'source_url\': \'https://developers.zenodo.org/#rate-limiting\', \'user_agent\': \'Project9EducationalArchiveReader/1.1 (+https://github.com/sultan1443/KAUST-AI251P-Project9-Public)\', \'minimum_seconds_between_GET_starts\': 3, \'initial_cooldown_seconds\': 60, \'automatic_wait_budget_seconds\': 900, \'transfer_budget_bytes\': self.transfer_budget, \'block_bytes\': self.block_size, \'maximum_blocks_per_GET\': 3, \'maximum_cached_blocks\': 16, \'maximum_read_bytes\': 64 * 1024 ** 2, \'maximum_attempts_per_range\': 12, \'maximum_network_or_5xx_failures\': 6, \'whole_archive_fallback\': False}\n        self._event(\'reader_policy\', policy=self.rate_policy)\n\n    def _event(self, event, **detail):\n        if self._log_path is None:\n            return\n        from pathlib import Path\n        import json\n        row = {\'event\': event, \'utc_epoch\': time.time(), \'http_gets\': self.http_gets, \'successful_GETs\': self.requests_count, \'accounted_body_bytes\': self.transferred, \'automatic_wait_seconds\': self.waited_seconds, **detail}\n        with Path(self._log_path).open(\'a\' if self._log_created else \'x\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + \'\\n\')\n        self._log_created = True\n\n    def _wait(self, seconds, reason, status=None):\n        seconds = max(0.0, float(seconds))\n        remaining = max(0.0, self.wait_budget - self.waited_seconds)\n        if seconds > remaining + 1e-08:\n            self._event(\'wait_budget_stop\', required_wait_seconds=seconds, remaining_wait_budget_seconds=remaining, reason=reason, http_status=status)\n            raise RuntimeError((\'HTTP 429 rate limit: \' if status == 429 else \'HTTP transport: \') + f\'{seconds:.1f} seconds of waiting are required; only {remaining:.1f} \' + \'seconds remain in the automatic wait budget. No earlier retry was made. \' + \'Keep the frozen attempt and share this output; do not restart Test inference.\')\n        if seconds <= 0:\n            return\n        self._event(\'wait_start\', seconds=seconds, reason=reason, http_status=status)\n        left = seconds\n        while left > 1e-08:\n            chunk = min(30.0, left)\n            print(f\'Archive transport: waiting {chunk:.1f}s ({reason}); {left:.1f}s remaining.\')\n            time.sleep(chunk)\n            self.waited_seconds += chunk\n            left -= chunk\n        self._event(\'wait_end\', seconds=seconds, reason=reason, http_status=status)\n\n    def _header_deadline(self, headers):\n        from email.utils import parsedate_to_datetime\n        from datetime import timezone\n        import math\n        now = time.time()\n        deadlines = []\n        retry_after = headers.get(\'Retry-After\')\n        if retry_after is not None:\n            text = str(retry_after).strip()\n            try:\n                seconds = float(text)\n                if math.isfinite(seconds) and seconds >= 0:\n                    deadlines.append(now + seconds)\n            except ValueError:\n                try:\n                    value = parsedate_to_datetime(text)\n                    if value.tzinfo is None:\n                        value = value.replace(tzinfo=timezone.utc)\n                    deadlines.append(value.timestamp())\n                except (TypeError, ValueError, OverflowError):\n                    pass\n        try:\n            remaining = float(headers.get(\'X-RateLimit-Remaining\', headers.get(\'XRateLimitRemaining\', \'nan\')))\n            reset = float(headers.get(\'X-RateLimit-Reset\', headers.get(\'XRateLimitReset\', \'nan\')))\n            if math.isfinite(remaining) and remaining <= 0 and math.isfinite(reset):\n                deadlines.append(reset)\n        except (TypeError, ValueError):\n            pass\n        if deadlines:\n            deadline = max(now, *deadlines)\n            self._server_not_before = max(self._server_not_before, deadline)\n            return deadline\n        return None\n\n    def _pace(self):\n        if self._initial_wait_pending:\n            self._wait(60, \'initial cooldown after the recorded HTTP 429\', 429)\n            self._initial_wait_pending = False\n        server_delay = max(0.0, self._server_not_before - time.time())\n        interval_delay = 0.0 if self._last_get_at is None else max(0.0, 3.0 - (time.monotonic() - self._last_get_at))\n        self._wait(max(server_delay, interval_delay), \'server rate-limit deadline\' if server_delay >= interval_delay and server_delay > 0 else \'minimum GET spacing\', 429 if server_delay > 0 else None)\n\n    def readable(self):\n        return True\n\n    def seekable(self):\n        return True\n\n    def tell(self):\n        return self.position\n\n    def seek(self, offset, whence=0):\n        position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1\n        if position < 0 or int(position) != position:\n            raise ValueError(\'Invalid HTTP ZIP seek.\')\n        self.position = int(position)\n        return self.position\n\n    def get_block(self, index):\n        from urllib3.exceptions import HTTPError as HTTPTransportError\n        require(not self.closed, \'HTTP ZIP stream is closed.\')\n        require(isinstance(index, int) and 0 <= index < (self.size + self.block_size - 1) // self.block_size, \'Invalid HTTP ZIP block index.\')\n        if index in self.blocks:\n            self.blocks.move_to_end(index)\n            return self.blocks[index]\n        last = index\n        while last + 1 < index + 3 and (last + 1) * self.block_size < self.size and (last + 1 not in self.blocks):\n            last += 1\n        start = index * self.block_size\n        end = min(self.size - 1, (last + 1) * self.block_size - 1)\n        needed = end - start + 1\n        rate_failures = network_failures = server_failures = 0\n        for attempt in range(12):\n            require(self.transferred + needed + 1 <= self.transfer_budget, \'Partial-download 768 MiB budget would be exceeded; no whole-archive fallback or further request.\')\n            self._pace()\n            self._last_get_at = time.monotonic()\n            self.http_gets += 1\n            self._event(\'GET_start\', first_byte=start, last_byte=end, attempt=attempt + 1)\n            try:\n                with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={\'Range\': f\'bytes={start}-{end}\', \'Accept-Encoding\': \'identity\', \'User-Agent\': self.rate_policy[\'user_agent\']}) as response:\n                    status = response.status_code\n                    deadline = self._header_deadline(response.headers)\n                    self._event(\'GET_headers\', http_status=status, retry_after=response.headers.get(\'Retry-After\'), rate_limit_remaining=response.headers.get(\'X-RateLimit-Remaining\'), rate_limit_reset=response.headers.get(\'X-RateLimit-Reset\'))\n                    if status == 429:\n                        rate_failures += 1\n                        if attempt == 11:\n                            raise RuntimeError(\'HTTP 429: maximum12 range attempts reached; keep the frozen attempt. No early retry or full download.\')\n                        delay = max(0.0, deadline - time.time()) if deadline is not None else min(300.0, 60.0 * 2 ** (rate_failures - 1))\n                        response.close()\n                        self._wait(delay, \'HTTP 429 Retry-After/reset\' if deadline is not None else \'HTTP 429 bounded backoff\', 429)\n                        continue\n                    if status in (500, 502, 503, 504):\n                        server_failures += 1\n                        require(server_failures < 6 and attempt < 11, f\'HTTP {status}: bounded range retries exhausted; no full download.\')\n                        response.close()\n                        self._wait(max(0.0 if deadline is None else deadline - time.time(), min(60.0, 5.0 * 2 ** (server_failures - 1))), f\'HTTP {status} bounded retry\', status)\n                        continue\n                    require(status == 206, f\'HTTP {status}: requested range was not supported; full download prevented (body not read).\')\n                    content_range = response.headers.get(\'Content-Range\', \'\').strip()\n                    match = re.fullmatch(\'bytes (\\\\d+)-(\\\\d+)/(\\\\d+)\', content_range, flags=re.I)\n                    require(match and tuple(map(int, match.groups())) == (start, end, self.size), \'Server returned an unexpected Content-Range: \' + content_range)\n                    encoding = response.headers.get(\'Content-Encoding\', \'identity\').strip().lower()\n                    require(encoding in (\'\', \'identity\'), \'Encoded HTTP range rejected; exact archive bytes are required.\')\n                    length = response.headers.get(\'Content-Length\')\n                    require(length is None or str(length).strip() == str(needed), \'HTTP range Content-Length differs.\')\n                    try:\n                        data = response.raw.read(needed + 1)\n                    except (requests.RequestException, HTTPTransportError, OSError):\n                        self.transferred += needed + 1\n                        raise\n                    self.transferred += len(data)\n                    require(len(data) == needed, \'Incomplete or oversized HTTP range response.\')\n                self.requests_count += 1\n                for block in range(index, last + 1):\n                    lo = (block - index) * self.block_size\n                    hi = min(needed, lo + self.block_size)\n                    self.blocks[block] = data[lo:hi]\n                    self.blocks.move_to_end(block)\n                while len(self.blocks) > 16:\n                    self.blocks.popitem(last=False)\n                self._event(\'GET_success\', first_byte=start, last_byte=end, received_bytes=len(data), cached_blocks=len(self.blocks))\n                if self.requests_count % 5 == 0:\n                    print(f\'Partial ZIP transport: {self.transferred / 1024 ** 2:.1f} MiB; {self.requests_count} successful GETs.\')\n                return self.blocks[index]\n            except (requests.RequestException, HTTPTransportError, OSError) as error:\n                network_failures += 1\n                self._event(\'GET_network_error\', error_type=type(error).__name__, failures=network_failures)\n                require(network_failures < 6 and attempt < 11, \'HTTP range network timeouts/errors exhausted the bounded retries; no full download.\')\n                self._wait(min(60.0, 5.0 * 2 ** (network_failures - 1)), \'bounded network-error retry\')\n        raise RuntimeError(\'HTTP range attempt limit reached; no full download.\')\n\n    def read(self, count=-1):\n        if self.closed:\n            raise ValueError(\'HTTP ZIP stream is closed.\')\n        count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))\n        require(count <= 64 * 1024 ** 2, \'Unexpected large ZIP read; complete-archive download prevented.\')\n        chunks = []\n        remaining = count\n        while remaining:\n            index, offset = divmod(self.position, self.block_size)\n            data = self.get_block(index)\n            take = min(remaining, len(data) - offset)\n            require(take > 0, \'Unexpected ZIP read position.\')\n            chunks.append(data[offset:offset + take])\n            self.position += take\n            remaining -= take\n        return b\'\'.join(chunks)\n\n    def close(self):\n        self.session.close()\n        self.blocks.clear()\n        super().close()\n'

"""CPU models and numeric helpers for the optional operator-assistance stage.

Importing this module performs no model loading, downloads or inference.
Model construction is explicit; all predictions are advisory and do not alter
the original detector/segmenter response or introduce a rejection threshold.
"""

CLASS_NAMES = ('cardboard', 'metal', 'rigid_plastic', 'soft_plastic')
CLIP_MATERIAL_NAMES = ('cardboard', 'metal', 'rigid plastic', 'soft plastic')
PLAIN_PROMPTS = tuple('a photo of ' + name for name in CLIP_MATERIAL_NAMES)
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'


def _assist_require(condition, message):
    if not condition:
        raise RuntimeError(message)


def _assist_sha256(path):
    from pathlib import Path
    import hashlib
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()


def unit(values):
    """The original float32 L2 normalization, including its 1e-10 guard."""
    import numpy as np
    array = np.asarray(values, dtype=np.float32)
    _assist_require(array.ndim == 2 and np.isfinite(array).all(), 'Expected finite 2D features.')
    return array / (np.linalg.norm(array, axis=-1, keepdims=True) + 1e-10)


def rgb_hist(crops):
    """Original 24-bin RGB histogram baseline; crops must be PIL images."""
    import numpy as np
    rows = []
    for crop in crops:
        array = np.asarray(crop.convert('RGB').resize((64, 64)), dtype=np.uint8)
        rows.append(np.concatenate([np.histogram(array[:, :, channel], bins=8,
                                                range=(0, 256))[0] for channel in range(3)]))
    _assist_require(bool(rows), 'At least one crop is required.')
    return unit(np.asarray(rows, dtype=np.float32))


def classification_metrics(true_labels, predicted_labels):
    """Four-material accuracy and macro metrics with zero-division set to zero."""
    import numpy as np
    truth = np.asarray(true_labels)
    prediction = np.asarray(predicted_labels)
    _assist_require(truth.ndim == prediction.ndim == 1 and truth.shape == prediction.shape
                    and truth.size > 0, 'Class arrays must be nonempty, aligned 1D arrays.')
    _assist_require(np.issubdtype(truth.dtype, np.integer)
                    and np.issubdtype(prediction.dtype, np.integer), 'Class IDs must be integers.')
    _assist_require(((truth >= 0) & (truth < 4)).all()
                    and ((prediction >= 0) & (prediction < 4)).all(), 'Unknown material class ID.')
    matrix = np.zeros((4, 4), dtype=np.int64)
    np.add.at(matrix, (truth, prediction), 1)
    tp = np.diag(matrix).astype(np.float64)
    predicted_support = matrix.sum(axis=0)
    support = matrix.sum(axis=1)
    precision = np.divide(tp, predicted_support, out=np.zeros(4), where=predicted_support != 0)
    recall = np.divide(tp, support, out=np.zeros(4), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(4), where=(precision + recall) != 0)
    return {
        'accuracy': float(np.mean(truth == prediction)), 'macro_f1': float(f1.mean()),
        'macro_precision': float(precision.mean()), 'macro_recall': float(recall.mean()),
        'count': int(truth.size), 'confusion_matrix_true_rows_predicted_columns': matrix.tolist(),
        'per_class': [{'class_id': index, 'material': name, 'support': int(support[index]),
                       'precision': float(precision[index]), 'recall': float(recall[index]),
                       'f1': float(f1[index])} for index, name in enumerate(CLASS_NAMES)],
    }


def retrieval_metrics(gallery_features, query_features, gallery_labels, query_labels):
    """Original descending argsort ranking and top-1/Hit@5/Precision@5 metrics."""
    import numpy as np
    gallery = np.asarray(gallery_features, dtype=np.float32)
    queries = np.asarray(query_features, dtype=np.float32)
    gallery_y = np.asarray(gallery_labels)
    query_y = np.asarray(query_labels)
    _assist_require(gallery.ndim == queries.ndim == 2 and gallery.shape[1] == queries.shape[1]
                    and len(gallery) >= 5 and len(queries) > 0, 'Invalid gallery/query feature shape.')
    _assist_require(gallery_y.shape == (len(gallery),) and query_y.shape == (len(queries),)
                    and np.isfinite(gallery).all() and np.isfinite(queries).all(), 'Misaligned or nonfinite retrieval inputs.')
    rankings = (queries @ gallery.T).argsort(axis=1)[:, ::-1]
    top1 = gallery_y[rankings[:, 0]]
    top5 = gallery_y[rankings[:, :5]]
    return ({'top1_accuracy': float(np.mean(top1 == query_y)),
             'hit_at_5': float(np.mean(np.any(top5 == query_y[:, None], axis=1))),
             'precision_at_5': float(np.mean(top5 == query_y[:, None]))}, rankings)


class OperatorAssistModels:
    """Frozen ResNet18 references and plain-prompt CLIP suggestions on CPU.

    ``assets_dir`` may be None while building a new gallery. Then call
    ``load_gallery`` after writing these files:

    - gallery_index.npz: float32 embeddings [160,512], float32 histograms
      [160,24], int64 labels [160].
    - gallery_meta.json: a list of 160 dictionaries aligned to those arrays,
      each with class_id and crop_file="gallery/g_000.png" etc.
    - the corresponding 160 PNG reference crops.

    local_only=True requires the ResNet checkpoint and the explicit immutable
    CLIP revision to be present already; it performs no network request.
    """

    def __init__(self, assets_dir=None, clip_revision=None, local_only=False):
        from pathlib import Path
        from importlib.metadata import version
        import re
        import torch
        import torch.nn as nn
        from torchvision.models import resnet18, ResNet18_Weights
        from transformers import CLIPModel, CLIPProcessor
        from huggingface_hub import hf_hub_download, model_info

        self.torch = torch
        self.assets_dir = None
        self.gallery_embeddings = None
        self.gallery_histograms = None
        self.gallery_labels = None
        self.gallery_meta = None
        self.local_only = bool(local_only)
        torch.set_num_threads(1)
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            _assist_require(torch.get_num_interop_threads() == 1,
                            'Torch inter-op threads are already initialized differently; use a fresh process.')

        weights = ResNet18_Weights.IMAGENET1K_V1
        self.resnet_transform = weights.transforms()
        resnet_file = Path(torch.hub.get_dir()) / 'checkpoints/resnet18-f37072fd.pth'
        if self.local_only:
            _assist_require(resnet_file.is_file(), 'Cached ResNet18 checkpoint is required for local-only serving.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'Cached ResNet18 checkpoint checksum differs.')
            encoder = resnet18(weights=None)
            state = torch.load(resnet_file, map_location='cpu', weights_only=True)
            encoder.load_state_dict(state, strict=True)
            del state
        else:
            encoder = resnet18(weights=weights, progress=False)
            _assist_require(resnet_file.is_file(), 'Downloaded ResNet18 checkpoint was not found at its standard cache path.')
            resnet_sha = _assist_sha256(resnet_file)
            _assist_require(resnet_sha.startswith('f37072fd'), 'ResNet18 checkpoint checksum differs from its official filename prefix.')
        encoder.fc = nn.Identity()
        self.encoder = encoder.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.encoder.parameters():
            parameter.requires_grad_(False)

        hub_info = None
        if clip_revision is None:
            _assist_require(not self.local_only, 'An explicit cached CLIP revision is required in local-only mode.')
            hub_info = model_info(CLIP_MODEL_ID, revision='main', files_metadata=True, token=False)
            clip_revision = hub_info.sha
        _assist_require(isinstance(clip_revision, str)
                        and re.fullmatch(r'[0-9a-f]{40}|[0-9a-f]{64}', clip_revision) is not None,
                        'CLIP revision must be an immutable Hub commit hash.')
        self.clip_revision = clip_revision
        common = {'revision': self.clip_revision, 'local_files_only': self.local_only, 'token': False}
        self.clip = CLIPModel.from_pretrained(CLIP_MODEL_ID, use_safetensors=False,
                                             trust_remote_code=False, **common)
        self.clip = self.clip.to(device='cpu', dtype=torch.float32).eval()
        for parameter in self.clip.parameters():
            parameter.requires_grad_(False)
        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False,
                                                       trust_remote_code=False, **common)
        # The loader above explicitly selects the bin checkpoint. This cache-only
        # lookup records its actual bytes without triggering a second download.
        clip_file = Path(hf_hub_download(CLIP_MODEL_ID, filename='pytorch_model.bin',
                                        revision=self.clip_revision, local_files_only=True, token=False))
        clip_sha = _assist_sha256(clip_file)
        expected_clip_sha = None
        if hub_info is not None:
            for sibling in hub_info.siblings or []:
                if sibling.rfilename == 'pytorch_model.bin':
                    lfs = getattr(sibling, 'lfs', None)
                    expected_clip_sha = (lfs.get('sha256') if isinstance(lfs, dict)
                                         else getattr(lfs, 'sha256', None))
                    break
        if expected_clip_sha is not None:
            _assist_require(clip_sha == expected_clip_sha, 'CLIP binary differs from the Hub LFS SHA256.')
        config_commit = getattr(self.clip.config, '_commit_hash', None)
        if config_commit is not None:
            _assist_require(config_commit == self.clip_revision, 'Loaded CLIP configuration revision differs.')
        self._provenance = {
            'resnet_weights': 'ResNet18_Weights.IMAGENET1K_V1',
            'resnet_weights_url': weights.url, 'resnet_checkpoint_path': str(resnet_file),
            'resnet_checkpoint_sha256': resnet_sha, 'resnet_transform': repr(self.resnet_transform),
            'clip_model_id': CLIP_MODEL_ID, 'clip_revision': self.clip_revision,
            'clip_config_commit_hash': config_commit, 'clip_checkpoint_path': str(clip_file),
            'clip_checkpoint_sha256': clip_sha, 'clip_hub_lfs_sha256': expected_clip_sha,
            'clip_checkpoint_url': 'https://huggingface.co/' + CLIP_MODEL_ID + '/resolve/'
                                   + self.clip_revision + '/pytorch_model.bin',
            'clip_processor_class': type(self.processor).__name__,
            'clip_image_processor_class': type(self.processor.image_processor).__name__,
            'clip_image_processor_config': self.processor.image_processor.to_dict(),
            'clip_attention_implementation': getattr(self.clip.config, '_attn_implementation', None),
            'versions': {name: version(name) for name in ['torch', 'torchvision', 'transformers',
                                                         'huggingface-hub', 'numpy', 'Pillow']},
        }
        self._assert_cpu_frozen()
        if assets_dir is not None:
            self.load_gallery(assets_dir)

    def _assert_cpu_frozen(self):
        for name, model in [('resnet', self.encoder), ('clip', self.clip)]:
            _assist_require(model.training is False, name + ' must be in eval mode.')
            _assist_require(all(parameter.device.type == 'cpu' and not parameter.requires_grad
                                and (not parameter.is_floating_point() or parameter.dtype == self.torch.float32)
                                for parameter in model.parameters()), name + ' must be frozen CPU FP32.')

    def embed_resnet(self, crops, batch=16):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid ResNet crop batch.')
        self._assert_cpu_frozen()
        vectors = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                tensor = self.torch.stack([self.resnet_transform(crop.convert('RGB'))
                                          for crop in crops[start:start + batch]])
                output = self.encoder(tensor.to(device='cpu', dtype=self.torch.float32))
                vectors.append(output.detach().cpu().numpy())
        array = np.concatenate(vectors).astype(np.float32)
        _assist_require(array.shape == (len(crops), 512), 'Unexpected ResNet feature dimensions.')
        return unit(array)

    def score_clip(self, crops, batch=8):
        import numpy as np
        _assist_require(isinstance(batch, int) and batch > 0 and len(crops) > 0, 'Invalid CLIP crop batch.')
        self._assert_cpu_frozen()
        logits = []
        with self.torch.inference_mode():
            for start in range(0, len(crops), batch):
                inputs = self.processor(text=list(PLAIN_PROMPTS),
                                        images=[crop.convert('RGB') for crop in crops[start:start + batch]],
                                        return_tensors='pt', padding=True)
                inputs = {key: value.to(device='cpu', dtype=self.torch.float32)
                          if key == 'pixel_values' else value.to('cpu') for key, value in inputs.items()}
                output = self.clip(**inputs)
                logits.append(output.logits_per_image.detach().cpu().numpy())
        array = np.concatenate(logits).astype(np.float32)
        _assist_require(array.shape == (len(crops), 4) and np.isfinite(array).all(), 'Unexpected or nonfinite CLIP logits.')
        return array

    def rgb_hist(self, crops):
        return rgb_hist(crops)

    def load_gallery(self, assets_dir):
        from pathlib import Path, PurePosixPath
        import json
        import numpy as np
        directory = Path(assets_dir).resolve()
        with np.load(directory / 'gallery_index.npz', allow_pickle=False) as bank:
            _assist_require(set(bank.files) == {'embeddings', 'histograms', 'labels'}, 'Unexpected assistance gallery keys.')
            embeddings = bank['embeddings'].copy()
            histograms = bank['histograms'].copy()
            labels = bank['labels'].copy()
        _assist_require(embeddings.shape == (160, 512) and embeddings.dtype == np.float32
                        and histograms.shape == (160, 24) and histograms.dtype == np.float32
                        and labels.shape == (160,) and labels.dtype == np.int64,
                        'Assistance gallery array shapes/dtypes differ.')
        _assist_require(np.isfinite(embeddings).all() and np.isfinite(histograms).all()
                        and all(int(np.sum(labels == index)) == 40 for index in range(4)),
                        'Assistance gallery features/classes differ.')
        for name, values in [('embedding', embeddings), ('histogram', histograms)]:
            _assist_require(np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=2e-5, rtol=0),
                            'Unnormalized gallery ' + name + ' features.')
        metadata = json.loads((directory / 'gallery_meta.json').read_text(encoding='utf-8'))
        _assist_require(isinstance(metadata, list) and len(metadata) == 160, 'Expected 160 gallery metadata records.')
        for index, item in enumerate(metadata):
            expected_file = 'gallery/g_' + str(index).zfill(3) + '.png'
            _assist_require(isinstance(item, dict) and item.get('class_id') == int(labels[index])
                            and item.get('crop_file') == expected_file, 'Gallery metadata/array alignment differs.')
            crop = PurePosixPath(item['crop_file'])
            path = (directory / crop).resolve()
            _assist_require(not crop.is_absolute() and '..' not in crop.parts
                            and path.is_relative_to(directory) and path.is_file(), 'Missing or unsafe reference crop.')
        self.assets_dir = directory
        self.gallery_embeddings = embeddings
        self.gallery_histograms = histograms
        self.gallery_labels = labels
        self.gallery_meta = metadata
        return self

    def query_crop(self, crop):
        import numpy as np
        _assist_require(self.gallery_embeddings is not None, 'Load the assistance gallery first.')
        feature = self.embed_resnet([crop], batch=1)
        similarities = (feature @ self.gallery_embeddings.T)[0]
        ranking = similarities.argsort()[::-1]
        references = []
        for raw_index in ranking[:5]:
            index = int(raw_index)
            class_id = int(self.gallery_labels[index])
            references.append({'gallery_index': index, 'class_id': class_id,
                               'material': CLASS_NAMES[class_id], 'similarity': float(similarities[index]),
                               'crop_file': self.gallery_meta[index]['crop_file']})
        clip_logits = self.score_clip([crop], batch=1)[0].astype(np.float64)
        relative = np.exp(clip_logits - clip_logits.max())
        relative /= relative.sum()
        clip_class = int(clip_logits.argmax())
        histogram = rgb_hist([crop])
        hist_similarities = (histogram @ self.gallery_histograms.T)[0]
        hist_index = int(hist_similarities.argmax())
        hist_class = int(self.gallery_labels[hist_index])
        return {
            'visual_search': {'predicted_class_id': references[0]['class_id'],
                              'predicted_material': references[0]['material'], 'references': references,
                              'score_type': 'Cosine similarity to frozen Train reference crops; not calibrated confidence.'},
            'clip': {'predicted_class_id': clip_class, 'predicted_material': CLASS_NAMES[clip_class],
                     'logits': clip_logits.tolist(), 'relative_scores': relative.tolist(),
                     'prompts': list(PLAIN_PROMPTS),
                     'score_type': 'Four-way CLIP similarity softmax; not calibrated probability or unknown detection.'},
            'histogram': {'predicted_class_id': hist_class, 'predicted_material': CLASS_NAMES[hist_class],
                          'reference_gallery_index': hist_index, 'similarity': float(hist_similarities[hist_index]),
                          'crop_file': self.gallery_meta[hist_index]['crop_file'],
                          'score_type': 'Nearest Train RGB histogram by cosine similarity; argmax tie policy.'},
        }

    def runtime_metadata(self):
        self._assert_cpu_frozen()
        return {
            **self._provenance, 'clip_revision': self.clip_revision, 'device': 'cpu', 'precision': 'float32',
            'resnet_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.encoder.parameters()
                                              if parameter.is_floating_point()}),
            'clip_parameter_dtypes': sorted({str(parameter.dtype) for parameter in self.clip.parameters()
                                            if parameter.is_floating_point()}),
            'torch_num_threads': int(self.torch.get_num_threads()),
            'torch_num_interop_threads': int(self.torch.get_num_interop_threads()),
            'all_parameters_frozen': True, 'plain_prompts': list(PLAIN_PROMPTS),
            'gallery_loaded': self.gallery_embeddings is not None,
            'gallery_size': int(len(self.gallery_labels)) if self.gallery_labels is not None else 0,
            'model_training_or_optimization': False,
            'decision_role': 'Advisory references and material suggestion; original detector outputs are not overwritten.',
        }


"""Prepare a prespecified material-balanced crop subset of the frozen Test split."""

def prepare_extension_test_inputs(frozen_protocol, evidence_dir):
    from collections import Counter, OrderedDict, defaultdict
    from datetime import datetime, timezone
    from pathlib import Path, PurePosixPath
    import csv
    import hashlib
    import io
    import json
    import math
    import re
    import time
    import zipfile
    import zlib
    import numpy as np
    import requests
    from PIL import Image

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha(raw):
        return hashlib.sha256(raw).hexdigest()
    evidence_dir = Path(evidence_dir)
    protocol_file = evidence_dir / 'frozen_protocol.json'
    require(protocol_file.is_file() and json.loads(protocol_file.read_text(encoding='utf-8')) == frozen_protocol, 'A byte-hashed frozen protocol must exist before opening Test annotations.')
    require(frozen_protocol['split'] == 'test' and frozen_protocol['per_class'] == 50 and (frozen_protocol['seed'] == 20261116), 'The prespecified Test sample design differs.')
    project = Path('/content/drive/MyDrive/Project9_Phase2')
    local = Path('/content/Project9_Phase2')
    manifest_path = project / 'data_protocol/ef6b35d2ba7f/split_manifest.csv'
    require(sha(manifest_path.read_bytes()) == frozen_protocol['manifest_sha256'], 'Frozen split bytes differ.')
    manifest = list(csv.DictReader(io.StringIO(manifest_path.read_text(encoding='utf-8-sig'))))
    rows = [row for row in manifest if row['split'] == 'test']
    require(len(rows) == 929 and all((row['original_split'] == 'test' for row in rows)), 'Frozen Test membership differs.')
    gallery = json.loads((Path(frozen_protocol['gallery_assets_path']) / 'gallery_meta.json').read_text(encoding='utf-8'))
    gallery_names = {item['export_name'] for item in gallery}
    classes = ['cardboard', 'metal', 'rigid_plastic', 'soft_plastic']
    archive_url = 'https://zenodo.org/records/6412647/files/zerowaste-f-final.zip?download=1'
    archive_size = 7518242799
    password = b'REDACTED_DATASET_ZIP_PASSWORD'

    class HTTPRangeFile(io.RawIOBase):
        """Strict, paced, bounded seekable Zenodo reader; no whole-file fallback.

    Globals are the original preparer's io/OrderedDict/requests/require/re/time,
    archive_url and archive_size. Imports stay inside methods so this class can
    replace only the original reader AST, without changing the sampler.
    """

        def __init__(self):
            super().__init__()
            self.position = 0
            self.size = archive_size
            self.block_size = 1024 * 1024
            self.blocks = OrderedDict()
            self.transferred = 0
            self.requests_count = 0
            self.http_gets = 0
            self.waited_seconds = 0.0
            self.wait_budget = 900.0
            self.transfer_budget = 768 * 1024 ** 2
            self._last_get_at = None
            self._server_not_before = 0.0
            self._initial_wait_pending = True
            self._log_created = False
            self._log_path = globals().get('_EXT_RATE_TRANSPORT_LOG_PATH')
            self.session = requests.Session()
            self.rate_policy = {'source_url': 'https://developers.zenodo.org/#rate-limiting', 'user_agent': 'Project9EducationalArchiveReader/1.1 (+https://github.com/sultan1443/KAUST-AI251P-Project9-Public)', 'minimum_seconds_between_GET_starts': 3, 'initial_cooldown_seconds': 60, 'automatic_wait_budget_seconds': 900, 'transfer_budget_bytes': self.transfer_budget, 'block_bytes': self.block_size, 'maximum_blocks_per_GET': 3, 'maximum_cached_blocks': 16, 'maximum_read_bytes': 64 * 1024 ** 2, 'maximum_attempts_per_range': 12, 'maximum_network_or_5xx_failures': 6, 'whole_archive_fallback': False}
            self._event('reader_policy', policy=self.rate_policy)

        def _event(self, event, **detail):
            if self._log_path is None:
                return
            from pathlib import Path
            import json
            row = {'event': event, 'utc_epoch': time.time(), 'http_gets': self.http_gets, 'successful_GETs': self.requests_count, 'accounted_body_bytes': self.transferred, 'automatic_wait_seconds': self.waited_seconds, **detail}
            with Path(self._log_path).open('a' if self._log_created else 'x', encoding='utf-8', newline='\n') as stream:
                stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
            self._log_created = True

        def _wait(self, seconds, reason, status=None):
            seconds = max(0.0, float(seconds))
            remaining = max(0.0, self.wait_budget - self.waited_seconds)
            if seconds > remaining + 1e-08:
                self._event('wait_budget_stop', required_wait_seconds=seconds, remaining_wait_budget_seconds=remaining, reason=reason, http_status=status)
                raise RuntimeError(('HTTP 429 rate limit: ' if status == 429 else 'HTTP transport: ') + f'{seconds:.1f} seconds of waiting are required; only {remaining:.1f} ' + 'seconds remain in the automatic wait budget. No earlier retry was made. ' + 'Keep the frozen attempt and share this output; do not restart Test inference.')
            if seconds <= 0:
                return
            self._event('wait_start', seconds=seconds, reason=reason, http_status=status)
            left = seconds
            while left > 1e-08:
                chunk = min(30.0, left)
                print(f'Archive transport: waiting {chunk:.1f}s ({reason}); {left:.1f}s remaining.')
                time.sleep(chunk)
                self.waited_seconds += chunk
                left -= chunk
            self._event('wait_end', seconds=seconds, reason=reason, http_status=status)

        def _header_deadline(self, headers):
            from email.utils import parsedate_to_datetime
            from datetime import timezone
            import math
            now = time.time()
            deadlines = []
            retry_after = headers.get('Retry-After')
            if retry_after is not None:
                text = str(retry_after).strip()
                try:
                    seconds = float(text)
                    if math.isfinite(seconds) and seconds >= 0:
                        deadlines.append(now + seconds)
                except ValueError:
                    try:
                        value = parsedate_to_datetime(text)
                        if value.tzinfo is None:
                            value = value.replace(tzinfo=timezone.utc)
                        deadlines.append(value.timestamp())
                    except (TypeError, ValueError, OverflowError):
                        pass
            try:
                remaining = float(headers.get('X-RateLimit-Remaining', headers.get('XRateLimitRemaining', 'nan')))
                reset = float(headers.get('X-RateLimit-Reset', headers.get('XRateLimitReset', 'nan')))
                if math.isfinite(remaining) and remaining <= 0 and math.isfinite(reset):
                    deadlines.append(reset)
            except (TypeError, ValueError):
                pass
            if deadlines:
                deadline = max(now, *deadlines)
                self._server_not_before = max(self._server_not_before, deadline)
                return deadline
            return None

        def _pace(self):
            if self._initial_wait_pending:
                self._wait(60, 'initial cooldown after the recorded HTTP 429', 429)
                self._initial_wait_pending = False
            server_delay = max(0.0, self._server_not_before - time.time())
            interval_delay = 0.0 if self._last_get_at is None else max(0.0, 3.0 - (time.monotonic() - self._last_get_at))
            self._wait(max(server_delay, interval_delay), 'server rate-limit deadline' if server_delay >= interval_delay and server_delay > 0 else 'minimum GET spacing', 429 if server_delay > 0 else None)

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.position

        def seek(self, offset, whence=0):
            position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
            if position < 0 or int(position) != position:
                raise ValueError('Invalid HTTP ZIP seek.')
            self.position = int(position)
            return self.position

        def get_block(self, index):
            from urllib3.exceptions import HTTPError as HTTPTransportError
            require(not self.closed, 'HTTP ZIP stream is closed.')
            require(isinstance(index, int) and 0 <= index < (self.size + self.block_size - 1) // self.block_size, 'Invalid HTTP ZIP block index.')
            if index in self.blocks:
                self.blocks.move_to_end(index)
                return self.blocks[index]
            last = index
            while last + 1 < index + 3 and (last + 1) * self.block_size < self.size and (last + 1 not in self.blocks):
                last += 1
            start = index * self.block_size
            end = min(self.size - 1, (last + 1) * self.block_size - 1)
            needed = end - start + 1
            rate_failures = network_failures = server_failures = 0
            for attempt in range(12):
                require(self.transferred + needed + 1 <= self.transfer_budget, 'Partial-download 768 MiB budget would be exceeded; no whole-archive fallback or further request.')
                self._pace()
                self._last_get_at = time.monotonic()
                self.http_gets += 1
                self._event('GET_start', first_byte=start, last_byte=end, attempt=attempt + 1)
                try:
                    with self.session.get(archive_url, stream=True, timeout=(15, 60), headers={'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity', 'User-Agent': self.rate_policy['user_agent']}) as response:
                        status = response.status_code
                        deadline = self._header_deadline(response.headers)
                        self._event('GET_headers', http_status=status, retry_after=response.headers.get('Retry-After'), rate_limit_remaining=response.headers.get('X-RateLimit-Remaining'), rate_limit_reset=response.headers.get('X-RateLimit-Reset'))
                        if status == 429:
                            rate_failures += 1
                            if attempt == 11:
                                raise RuntimeError('HTTP 429: maximum12 range attempts reached; keep the frozen attempt. No early retry or full download.')
                            delay = max(0.0, deadline - time.time()) if deadline is not None else min(300.0, 60.0 * 2 ** (rate_failures - 1))
                            response.close()
                            self._wait(delay, 'HTTP 429 Retry-After/reset' if deadline is not None else 'HTTP 429 bounded backoff', 429)
                            continue
                        if status in (500, 502, 503, 504):
                            server_failures += 1
                            require(server_failures < 6 and attempt < 11, f'HTTP {status}: bounded range retries exhausted; no full download.')
                            response.close()
                            self._wait(max(0.0 if deadline is None else deadline - time.time(), min(60.0, 5.0 * 2 ** (server_failures - 1))), f'HTTP {status} bounded retry', status)
                            continue
                        require(status == 206, f'HTTP {status}: requested range was not supported; full download prevented (body not read).')
                        content_range = response.headers.get('Content-Range', '').strip()
                        match = re.fullmatch('bytes (\\d+)-(\\d+)/(\\d+)', content_range, flags=re.I)
                        require(match and tuple(map(int, match.groups())) == (start, end, self.size), 'Server returned an unexpected Content-Range: ' + content_range)
                        encoding = response.headers.get('Content-Encoding', 'identity').strip().lower()
                        require(encoding in ('', 'identity'), 'Encoded HTTP range rejected; exact archive bytes are required.')
                        length = response.headers.get('Content-Length')
                        require(length is None or str(length).strip() == str(needed), 'HTTP range Content-Length differs.')
                        try:
                            data = response.raw.read(needed + 1)
                        except (requests.RequestException, HTTPTransportError, OSError):
                            self.transferred += needed + 1
                            raise
                        self.transferred += len(data)
                        require(len(data) == needed, 'Incomplete or oversized HTTP range response.')
                    self.requests_count += 1
                    for block in range(index, last + 1):
                        lo = (block - index) * self.block_size
                        hi = min(needed, lo + self.block_size)
                        self.blocks[block] = data[lo:hi]
                        self.blocks.move_to_end(block)
                    while len(self.blocks) > 16:
                        self.blocks.popitem(last=False)
                    self._event('GET_success', first_byte=start, last_byte=end, received_bytes=len(data), cached_blocks=len(self.blocks))
                    if self.requests_count % 5 == 0:
                        print(f'Partial ZIP transport: {self.transferred / 1024 ** 2:.1f} MiB; {self.requests_count} successful GETs.')
                    return self.blocks[index]
                except (requests.RequestException, HTTPTransportError, OSError) as error:
                    network_failures += 1
                    self._event('GET_network_error', error_type=type(error).__name__, failures=network_failures)
                    require(network_failures < 6 and attempt < 11, 'HTTP range network timeouts/errors exhausted the bounded retries; no full download.')
                    self._wait(min(60.0, 5.0 * 2 ** (network_failures - 1)), 'bounded network-error retry')
            raise RuntimeError('HTTP range attempt limit reached; no full download.')

        def read(self, count=-1):
            if self.closed:
                raise ValueError('HTTP ZIP stream is closed.')
            count = max(0, self.size - self.position) if count is None or count < 0 else min(count, max(0, self.size - self.position))
            require(count <= 64 * 1024 ** 2, 'Unexpected large ZIP read; complete-archive download prevented.')
            chunks = []
            remaining = count
            while remaining:
                index, offset = divmod(self.position, self.block_size)
                data = self.get_block(index)
                take = min(remaining, len(data) - offset)
                require(take > 0, 'Unexpected ZIP read position.')
                chunks.append(data[offset:offset + take])
                self.position += take
                remaining -= take
            return b''.join(chunks)

        def close(self):
            self.session.close()
            self.blocks.clear()
            super().close()
    cache = local / 'extensions_test_source_images'
    cache.mkdir(parents=True, exist_ok=True)
    existing_archive = next((path for path in [Path('/content/zerowaste-f-final.zip'), project / 'cache/zerowaste-f-final.zip'] if path.is_file() and path.stat().st_size == archive_size), None)
    remote = None
    if existing_archive is None:
        remote = HTTPRangeFile()
        archive = zipfile.ZipFile(remote)
    else:
        archive = zipfile.ZipFile(existing_archive)
    protocol_sha = sha(protocol_file.read_bytes())
    try:

        def member(suffix):
            require(suffix.startswith('test/') and '..' not in PurePosixPath(suffix).parts, 'Only source Test members selected by the frozen protocol may be read.')
            matches = [item for item in archive.infolist() if not item.is_dir() and (item.filename == suffix or item.filename.endswith('/' + suffix)) and ('..' not in PurePosixPath(item.filename).parts)]
            require(len(matches) == 1, 'Test member is not unique: ' + suffix)
            return matches[0]

        def fetch(info, filename):
            path = cache / filename
            sidecar = cache / (filename + '.source.json')
            require(0 < info.file_size <= (256 * 1024 ** 2 if filename.endswith('.json') else 12 * 1024 ** 2), 'Unexpected source member size.')
            if path.exists():
                require(sidecar.is_file(), 'Existing source cache has no provenance receipt.')
                raw = path.read_bytes()
                receipt = json.loads(sidecar.read_text(encoding='utf-8'))
                reused = True
            else:
                raw = archive.read(info, pwd=password)
                receipt = {'sha256': sha(raw), 'member': info.filename, 'member_crc32': info.CRC, 'member_bytes': info.file_size, 'archive_url': archive_url, 'archive_total_bytes': archive_size}
                with path.open('xb') as stream:
                    stream.write(raw)
                with sidecar.open('x', encoding='utf-8') as stream:
                    json.dump(receipt, stream, indent=2)
                reused = False
            require(len(raw) == info.file_size and zlib.crc32(raw) & 4294967295 == info.CRC, 'Source member CRC/size differs.')
            expected = {'sha256': sha(raw), 'member': info.filename, 'member_crc32': info.CRC, 'member_bytes': info.file_size, 'archive_url': archive_url, 'archive_total_bytes': archive_size}
            require(receipt == expected, 'Source cache provenance differs.')
            return (path, raw, {**receipt, 'reused_cache': reused, 'historical_full_source_image_sha256_anchor_available': False})
        _, raw_annotations, annotation_receipt = fetch(member('test/labels.json'), 'source_test_annotations.json')
        coco = json.loads(raw_annotations.decode('utf-8-sig'))
        images = {int(item['id']): item for item in coco['images']}
        category = {int(item['id']): item['name'] for item in coco['categories']}
        require(set(category.values()) == set(classes), 'Source material categories differ.')
        annotations = defaultdict(list)
        for ann in coco['annotations']:
            annotations[int(ann['image_id'])].append(ann)
        pools = defaultdict(list)
        for row in sorted(rows, key=lambda item: 'test__' + item['filename']):
            image_id = int(row['image_id'])
            info = images[image_id]
            require(PurePosixPath(info['file_name']).name == row['filename'], 'Test COCO image identity differs.')
            width, height = (int(info['width']), int(info['height']))
            counts = Counter()
            for annotation_ordinal, ann in enumerate(annotations[image_id]):
                cls = classes.index(category[int(ann['category_id'])])
                x, y, bw, bh = map(float, ann['bbox'])
                require(all((math.isfinite(value) for value in [x, y, bw, bh])) and bw > 0 and (bh > 0), 'Invalid Test annotation box.')
                x1, y1, x2, y2 = (max(0, min(width, x)), max(0, min(height, y)), max(0, min(width, x + bw)), max(0, min(height, y + bh)))
                require(x2 > x1 and y2 > y1, 'Empty clipped Test box.')
                xc, yc, wn, hn = ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height)
                line = f'{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}'
                _, xc, yc, wn, hn = map(float, line.split())
                counts[classes[cls]] += 1
                if wn > 0.003 and hn > 0.003:
                    pools[cls].append({'class_id': cls, 'class_name': classes[cls], 'image_id': image_id, 'annotation_id': int(ann['id']), 'annotation_ordinal_in_image': annotation_ordinal, 'export_name': 'test__' + row['filename'], 'filename': row['filename'], 'width': width, 'height': height, 'original_split': 'test', 'project_split': 'test', 'xywh_normalized': [xc, yc, wn, hn], 'converted_yolo_line': line, 'official_clipped_box_xyxy': [x1, y1, x2, y2], 'frozen_manifest_row': row})
            require(all((counts[name] == int(row[name]) for name in classes)), 'Test annotation counts differ from frozen manifest.')
        rng = np.random.default_rng(frozen_protocol['seed'])
        selected = []
        for cls in range(4):
            require(pools[cls], 'A prespecified material has no eligible Test crops.')
            indices = rng.choice(len(pools[cls]), size=min(frozen_protocol['per_class'], len(pools[cls])), replace=False)
            selected.extend((dict(pools[cls][int(index)]) for index in indices))
        selected_names = {item['export_name'] for item in selected}
        require(not selected_names & gallery_names, 'Test queries overlap the Train gallery source images.')
        member_by_name = {name: member('test/data/' + name.removeprefix('test__')) for name in selected_names}
        source_records = {}
        for ordinal, name in enumerate(sorted(selected_names, key=lambda value: member_by_name[value].header_offset), 1):
            path, raw, receipt = fetch(member_by_name[name], name)
            with Image.open(io.BytesIO(raw)) as image:
                record = next((item for item in selected if item['export_name'] == name))
                require(image.format == 'PNG' and image.size == (record['width'], record['height']), 'Test PNG dimensions/format differ.')
                image.verify()
            source_records[name] = {'path': str(path), **receipt}
            if ordinal % 10 == 0 or ordinal == len(selected_names):
                print('Verified selected Test source PNGs:', ordinal, '/', len(selected_names))
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        work_dir = local / ('extension_test_inputs_' + stamp)
        (work_dir / 'test_queries').mkdir(parents=True, exist_ok=False)
        for ordinal, record in enumerate(selected):
            source = source_records[record['export_name']]
            width, height = (record['width'], record['height'])
            xc, yc, bw, bh = record['xywh_normalized']
            x1 = max(0, int((xc - bw / 2) * width))
            x2 = min(width, int(np.ceil((xc + bw / 2) * width)))
            y1 = max(0, int((yc - bh / 2) * height))
            y2 = min(height, int(np.ceil((yc + bh / 2) * height)))
            box = [x1, y1, max(x1 + 1, x2), max(y1 + 1, y2)]
            relative = f'test_queries/t_{ordinal:03d}.png'
            path = work_dir / relative
            with Image.open(source['path']) as image:
                image.convert('RGB').crop(tuple(box)).save(path, format='PNG')
            record.update({'ordinal': ordinal, 'source_image_sha256': source['sha256'], 'crop_file': relative, 'crop_path': str(path), 'crop_sha256': sha(path.read_bytes()), 'crop_box_xyxy': box, 'crop_width': box[2] - box[0], 'crop_height': box[3] - box[1]})
        require(sha(protocol_file.read_bytes()) == protocol_sha, 'Frozen Test protocol changed during preparation.')
        provenance = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test', 'selection': 'Fixed seed20261116, up to50 eligible objects per material without replacement; use all if fewer than50.', 'seed': 20261116, 'bit_generator': type(rng.bit_generator).__name__, 'sample_objects': len(selected), 'unique_source_images': len(selected_names), 'counts_per_class': {classes[cls]: sum((item['class_id'] == cls for item in selected)) for cls in range(4)}, 'eligible_objects_per_class': {classes[cls]: len(pools[cls]) for cls in range(4)}, 'full_Test_partition_images': 929, 'scope': 'Prespecified material-balanced GT-crop Test subset; not a full929-image system evaluation.', 'annotation_source': annotation_receipt, 'source_images': source_records, 'historical_full_PNG_or_Test_COCO_hash_anchors_available': False, 'new_source_verification': 'Selected source ZIP members checked by CRC/size and new SHA256 receipts; identities/counts checked against frozen manifest.', 'whole_archive_md5_recomputed_this_run': False, 'archive_url': archive_url, 'archive_total_bytes': archive_size, 'range_bytes_transferred': remote.transferred if remote else 0, 'protocol_sha256_before_Test_annotation_access': protocol_sha, 'gallery_source_images_disjoint': True, 'new_model_execution_in_preparation': False}
        return {'queries': selected, 'work_dir': str(work_dir), 'provenance': provenance}
    finally:
        archive.close()
        if remote is not None:
            remote.close()


def recover_extension_test_after_rate_limit():
    from datetime import datetime, timezone
    from importlib.metadata import version, PackageNotFoundError
    from pathlib import Path
    import hashlib
    import json
    import os
    import shutil
    import subprocess
    import sys
    import zipfile
    from google.colab import drive, files

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def digest(path):
        value = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(4 * 1024 ** 2), b''):
                value.update(chunk)
        return value.hexdigest()

    def save_new(path, value):
        with Path(path).open('x', encoding='utf-8', newline='\n') as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write('\n')

    def installed(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None
    drive.mount('/content/drive')
    project = Path('/content/drive/MyDrive/Project9_Phase2')
    output = project / 'evaluation/EXTENSIONS_FINAL_TEST_ONCE'
    archive_path = project / 'Project9_EXTENSIONS_FINAL_TEST_ONCE.zip'
    if (output / 'COMPLETED.json').is_file():
        require((output / 'COMPLETED.json').is_file() and archive_path.is_file(), 'An earlier Test attempt reserved this folder. Do not evaluate again; share the output/status for recovery.')
        inventory = json.loads((output / 'collection_manifest.json').read_text(encoding='utf-8'))['files']
        with zipfile.ZipFile(archive_path) as archive:
            require(archive.testzip() is None, 'Saved Test evidence ZIP failed CRC verification.')
            for item in inventory:
                raw = archive.read(item['file'])
                require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Saved Test archive bytes differ: ' + item['file'])
        print('Previously completed extension Test found; downloading its saved evidence without model execution.')
        files.download(str(archive_path))
        return {'already_completed': True, 'zip': str(archive_path)}
    require(output.is_dir(), 'The original reserved Test folder is missing. Stop and share this output.')
    protocol_file = output / 'frozen_protocol.json'
    status_file = output / 'STATUS.json'
    require(protocol_file.is_file() and status_file.is_file(), 'Original frozen protocol/status is missing.')
    protocol_bytes = protocol_file.read_bytes()
    protocol_sha = hashlib.sha256(protocol_bytes).hexdigest()
    protocol = json.loads(protocol_bytes.decode('utf-8'))
    status = json.loads(status_file.read_text(encoding='utf-8'))
    require(status == {'phase': 'preparing_inputs_no_Test_model_calls_yet', 'protocol_sha256': protocol_sha}, 'Recovery is restricted to the recorded failure BEFORE Test inference; status/hash differs.')
    blocked = ['query_provenance.json', 'test_queries', 'reference_gallery', 'inference_batches', 'test_arrays.npz', 'comparison_TEST.json', 'per_query_TEST.json', 'COMPLETED.json', 'collection_manifest.json']
    require(not any(((output / name).exists() for name in blocked)) and (not archive_path.exists()), 'Existing Test preparation/results found. Do not evaluate again; share this output for separate recovery review.')
    require(protocol.get('split') == 'test' and protocol.get('per_class') == 50 and (protocol.get('seed') == 20261116) and (protocol.get('target_maximum_query_objects') == 200) and (protocol.get('full_Test_partition_images') == 929), 'Frozen Test sampling parameters differ.')
    require(protocol.get('output_dir') == str(output) and protocol.get('gallery_objects') == 160 and (protocol.get('prompt_model_threshold_selection_on_Test') is False) and (protocol.get('M4_M5_Test_model_calls') == 0) and (protocol.get('new_training_or_optimizer_steps') == 0), 'Frozen Test scope/settings differ.')
    expected_sources = {name: hashlib.sha256(source.encode()).hexdigest() for name, source in _EXT_TEST_SOURCES.items()}
    require(protocol.get('source_files_sha256') == expected_sources, 'Frozen source references differ from the original supplied cell.')
    for name, expected in expected_sources.items():
        require(digest(output / 'code' / name) == expected, 'Original saved code bytes differ: ' + name)
    require({path.name for path in (output / 'code').iterdir()} == set(expected_sources), 'Unexpected files appeared in the original frozen code folder.')
    memory_diagnostic = {'available': False, 'note': 'The saved protocol bytes, not traceback memory, are the recovery authority.'}
    last_error = getattr(sys, 'last_exc', None) or getattr(sys, 'last_value', None)
    trace = getattr(last_error, '__traceback__', None)
    while trace is not None:
        frame = trace.tb_frame
        if frame.f_code.co_name == 'prepare_extension_test_inputs' and 'frozen_protocol' in frame.f_locals:
            argument = frame.f_locals['frozen_protocol']
            try:
                normalized = json.loads(json.dumps(argument, ensure_ascii=False, allow_nan=False))
                memory_diagnostic.update({'available': True, 'python_object_equal': argument == protocol, 'json_normalized_equal': normalized == protocol})
                if normalized != protocol:
                    raise RuntimeError('Original in-memory protocol normalizes differently from the saved frozen file.')
            except (TypeError, ValueError) as error:
                raise RuntimeError('Original protocol memory is inconsistent with its saved JSON.') from error
            break
        trace = trace.tb_next
    print('Recovery gate passed: original protocol hash matches, status is before Test, no Test batches/results exist.')
    os.environ['WANDB_MODE'] = 'disabled'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    torch_version, vision_version = (installed('torch'), installed('torchvision'))
    require(torch_version and vision_version, 'A matching preinstalled PyTorch/torchvision runtime is required.')
    desired = {'transformers': '5.22.1', 'numpy': '2.4.6', 'Pillow': '11.3.0'}
    changes = [name + '==' + value for name, value in desired.items() if installed(name) != value]
    changes += [name for name in ['requests'] if installed(name) is None]
    if changes:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==' + torch_version, 'torchvision==' + vision_version, *changes], check=True, timeout=360)
    import numpy as np
    import PIL
    from PIL import Image
    require(np.__version__ == '2.4.6' and PIL.__version__ == '11.3.0', 'Runtime libraries are already loaded at different versions. Restart the runtime before any Test attempt.')
    require(installed('torch') == torch_version and installed('torchvision') == vision_version, 'The preinstalled torch pair unexpectedly changed.')
    validation = project / 'evaluation/EXTENSIONS_OPERATOR_VAL_20261116T183113877417Z'
    assets = validation / 'deployment/assets'
    frozen_val = json.loads((validation / 'frozen_operator_VAL_protocol.json').read_text(encoding='utf-8'))
    manifest = json.loads((validation / 'deployment/assist_manifest.json').read_text(encoding='utf-8'))
    require(digest(validation / 'frozen_operator_VAL_protocol.json') == _EXT_TEST_ANCHORS['val_protocol'], 'The already-audited Validation protocol differs.')
    require(digest(assets / 'gallery_index.npz') == _EXT_TEST_ANCHORS['gallery_index'], 'The fixed160-reference gallery differs.')
    require(digest(assets / 'gallery_meta.json') == _EXT_TEST_ANCHORS['gallery_meta'], 'Gallery identities differ.')
    require(hashlib.sha256(_EXT_TEST_SOURCES['extension_assist_models.py'].encode()).hexdigest() == frozen_val['source_files_sha256']['extension_assist_models.py'], 'The model helper source changed after Validation.')
    protected_relative = _EXT_TEST_PROTECTED_PATHS
    protected = {name: project / relative for name, relative in protected_relative.items()}
    before = {name: digest(path) for name, path in protected.items()}
    require(before == frozen_val['original_project_files_sha256'], 'An original protected project reference differs.')
    for relative, expected in manifest['assets_sha256'].items():
        require(digest(assets / relative) == expected, 'Validation gallery/crop asset differs: ' + relative)
    print('Verifying the already-selected frozen models and Train gallery BEFORE opening Test annotations...')
    engine = OperatorAssistModels(assets_dir=assets, clip_revision=manifest['clip_revision'])
    resources = engine.runtime_metadata()
    for key in ['clip_revision', 'clip_checkpoint_sha256', 'resnet_checkpoint_sha256']:
        require(resources[key] == manifest['model_resources'][key], 'Official assistance model resource differs: ' + key)
    gallery_meta = json.loads((assets / 'gallery_meta.json').read_text(encoding='utf-8'))
    gallery_crops = []
    for item in gallery_meta:
        with Image.open(assets / item['crop_file']) as image:
            gallery_crops.append(image.convert('RGB'))
    recomputed = engine.embed_resnet(gallery_crops)
    cosines = np.sum(recomputed * engine.gallery_embeddings, axis=1)
    require(bool(np.all(cosines >= 0.999)), 'Runtime reference embedding compatibility failed before Test.')
    require(np.allclose(rgb_hist(gallery_crops), engine.gallery_histograms, rtol=0, atol=1e-07), 'Runtime histogram preprocessing differs from the fixed Validation reference bank.')
    del gallery_crops, recomputed
    require(protocol['manifest_sha256'] == before['split_manifest'] and protocol['protected_original_files_before'] == before, 'Original protected references differ from frozen Test protocol.')
    require(protocol['gallery_assets_path'] == str(assets) and protocol['validation_protocol_sha256'] == _EXT_TEST_ANCHORS['val_protocol'] and (protocol['gallery_index_sha256'] == _EXT_TEST_ANCHORS['gallery_index']) and (protocol['gallery_metadata_sha256'] == _EXT_TEST_ANCHORS['gallery_meta']), 'Frozen gallery/Validation anchors differ.')
    require(protocol['class_names'] == list(CLASS_NAMES) and protocol['clip_prompts'] == list(PLAIN_PROMPTS), 'Frozen class names/prompts differ.')
    current_resources = json.loads(json.dumps(resources, ensure_ascii=False, allow_nan=False))
    require(current_resources == protocol['model_resources'], 'Current runtime/model processor metadata differs from frozen protocol; stop before Test.')
    require(digest(protocol_file) == protocol_sha, 'The original frozen protocol changed during resource checks.')
    recovery_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    recovery_dir = output / ('recovery_before_inference_' + recovery_stamp)
    recovery_dir.mkdir(exist_ok=False)
    with (recovery_dir / 'recovery_function.py').open('x', encoding='utf-8', newline='\n') as stream:
        stream.write(_EXT_RECOVERY_FUNCTION_SOURCE)
    save_new(recovery_dir / 'recovery_receipt.json', {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'reason': 'HTTP429 while retrieving source ZIP ranges, before any Test model batch was run.', 'fix': 'Keep original frozen protocol and sampling/model logic; patch only bounded archive transport to respect server rate-limit waits and reduce requests.', 'frozen_protocol_sha256_unchanged': protocol_sha, 'original_code_sha256_unchanged': expected_sources, 'recovery_function_sha256': hashlib.sha256(_EXT_RECOVERY_FUNCTION_SOURCE.encode()).hexdigest(), 'status_before_recovery': status, 'memory_diagnostic': memory_diagnostic, 'sampling_models_prompts_thresholds_and_preprocessing_changed': False, 'previous_Test_batches_or_results_present': False, 'current_model_resources_json_normalized_match': True, 'original_M4_M5_Test_calls': 0})
    print('Continuing the same frozen attempt with the original protocol JSON; no frozen file was rewritten.')
    globals()['_EXT_RATE_TRANSPORT_LOG_PATH'] = str(recovery_dir / 'zenodo_transport.jsonl')
    with (recovery_dir / 'transport_preparer.py').open('x', encoding='utf-8', newline='\n') as stream:
        stream.write(_EXT_RATE_TRANSPORT_PREPARER_SOURCE)
    with (recovery_dir / 'transport_reader.py').open('x', encoding='utf-8', newline='\n') as stream:
        stream.write(_EXT_RATE_TRANSPORT_READER_SOURCE)
    save_new(recovery_dir / 'transport_patch_receipt.json', {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'reason': 'Zenodo HTTP429 during input preparation; saved status still has zero Test inference batches.', 'official_rate_limit_documentation': 'https://developers.zenodo.org/#rate-limiting', 'frozen_protocol_sha256_preserved': protocol_sha, 'original_preparer_sha256': expected_sources['prepare_extension_test_inputs.py'], 'patched_preparer_sha256': hashlib.sha256(_EXT_RATE_TRANSPORT_PREPARER_SOURCE.encode()).hexdigest(), 'transport_reader_sha256': hashlib.sha256(_EXT_RATE_TRANSPORT_READER_SOURCE.encode()).hexdigest(), 'change_boundary': 'Only the nested HTTPRangeFile class differs; outer sampler/crop/CRC/SHA logic AST-identical.', 'sampling_models_prompts_thresholds_and_preprocessing_changed': False, 'transport': 'Up to3 adjacent1MiB blocks per HTTP request;16-blockLRU;min3s betweenGETs;initial60s cooldown;honorRetry-After/reset;900s wait budget;768MiB transfer cap;strict206Content-Range.', 'full_archive_download_fallback': False, 'original_M4_M5_Test_calls': 0})
    print('Zenodo download recovery: respecting server cooldown; using cached members where verified.')
    print('Only required source members will be retrieved. Test sampling/settings and the original protocol are unchanged.')
    preparation = prepare_extension_test_inputs(protocol, output)
    records = preparation['queries']
    work = Path(preparation['work_dir'])
    save_new(output / 'query_provenance.json', preparation)
    shutil.copytree(work / 'test_queries', output / 'test_queries')
    reference_dir = output / 'reference_gallery'
    reference_dir.mkdir()
    shutil.copytree(assets / 'gallery', reference_dir / 'gallery')
    shutil.copyfile(assets / 'gallery_index.npz', reference_dir / 'gallery_index.npz')
    shutil.copyfile(assets / 'gallery_meta.json', reference_dir / 'gallery_meta.json')
    batch_dir = output / 'inference_batches'
    batch_dir.mkdir()
    (output / 'STATUS.json').write_text(json.dumps({'phase': 'Test_inference_started', 'protocol_sha256': protocol_sha, 'query_objects': len(records)}, indent=2), encoding='utf-8')
    q_parts, h_parts, l_parts = ([], [], [])
    print('One fixed Test evaluation:', len(records), 'annotated crops; no detector/segmenter calls.')
    for start in range(0, len(records), 8):
        crops = []
        for item in records[start:start + 8]:
            path = output / item['crop_file']
            require(digest(path) == item['crop_sha256'], 'Selected Test crop changed.')
            with Image.open(path) as image:
                crops.append(image.convert('RGB'))
        q, h, logits = (engine.embed_resnet(crops, batch=8), rgb_hist(crops), engine.score_clip(crops, batch=8))
        path = batch_dir / f'batch_{start // 8:03d}.npz'
        with path.open('xb') as stream:
            np.savez_compressed(stream, query_indices=np.arange(start, start + len(crops)), embeddings=q, histograms=h, clip_logits=logits)
        q_parts.append(q)
        h_parts.append(h)
        l_parts.append(logits)
        print('Recorded Test crops:', start + len(crops), '/', len(records))
    q, h, logits = (np.concatenate(q_parts), np.concatenate(h_parts), np.concatenate(l_parts))
    labels = np.asarray([item['class_id'] for item in records], dtype=np.int64)
    gy = engine.gallery_labels
    search, rank = retrieval_metrics(engine.gallery_embeddings, q, gy, labels)
    baseline_search, hist_rank = retrieval_metrics(engine.gallery_histograms, h, gy, labels)
    hist_pred = gy[(h @ engine.gallery_histograms.T).argmax(axis=1)]
    clip_pred = logits.argmax(axis=1)
    comparison = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'split': 'test', 'sample_objects': len(records), 'gallery_objects': 160, 'sample_counts': preparation['provenance']['counts_per_class'], 'unique_source_images': preparation['provenance']['unique_source_images'], 'visual_search_resnet': search, 'visual_search_histogram': baseline_search, 'clip_plain': classification_metrics(labels, clip_pred), 'histogram_classification_baseline': classification_metrics(labels, hist_pred), 'frozen_protocol_sha256': protocol_sha, 'limits': ['Prespecified material-balanced GT-crop subset of the frozen929-image Test partition, not whole-split performance.', 'GT crops exclude detector/mask errors; no application accuracy, physical grasp or unknown-class claim.', 'No model/prompt/threshold tuning on Test; do not use these results to select a new configuration.']}
    save_new(output / 'comparison_TEST.json', comparison)
    with (output / 'test_arrays.npz').open('xb') as stream:
        np.savez_compressed(stream, query_embeddings=q, query_histograms=h, clip_logits=logits, query_labels=labels, resnet_rankings=rank, histogram_rankings=hist_rank, histogram_class_predictions=hist_pred)
    rows = [{'query_index': index, 'crop_file': item['crop_file'], 'source_export_name': item['export_name'], 'annotation_id': item['annotation_id'], 'ground_truth_class_id': int(labels[index]), 'resnet_top1_class_id': int(gy[rank[index, 0]]), 'resnet_top5_gallery_indices': [int(value) for value in rank[index, :5]], 'clip_class_id': int(clip_pred[index]), 'clip_logits': logits[index].tolist(), 'histogram_class_id': int(hist_pred[index])} for index, item in enumerate(records)]
    save_new(output / 'per_query_TEST.json', rows)
    after = {name: digest(path) for name, path in protected.items()}
    require(before == after and digest(output / 'frozen_protocol.json') == protocol_sha, 'An original file or frozen Test protocol changed.')
    save_new(output / 'protected_original_files.json', {'before': before, 'after': after, 'unchanged': True, 'original_M4_M5_Test_calls': 0, 'training_or_optimizer_steps': 0})
    save_new(output / 'COMPLETED.json', {'status': 'completed_once', 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'frozen_protocol_sha256': protocol_sha, 'comparison_sha256': digest(output / 'comparison_TEST.json'), 'zip_name': archive_path.name, 'M4_M5_Test_reevaluated': False})
    (output / 'STATUS.json').write_text(json.dumps({'phase': 'completed_once', 'protocol_sha256': protocol_sha}), encoding='utf-8')
    inventory = [{'file': path.relative_to(output).as_posix(), 'bytes': path.stat().st_size, 'sha256': digest(path)} for path in sorted(output.rglob('*')) if path.is_file()]
    save_new(output / 'collection_manifest.json', {'files': inventory})
    with zipfile.ZipFile(archive_path, 'x', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(output).as_posix())
    with zipfile.ZipFile(archive_path) as archive:
        require(archive.testzip() is None, 'Test evidence ZIP CRC check failed.')
        for item in inventory:
            raw = archive.read(item['file'])
            require(len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256'], 'Packaged Test evidence differs.')
    print(json.dumps(comparison, indent=2))
    print('Extension Test complete. No M4/M5 Test evaluation or training was run.')
    print('Saved original results unchanged. ZIP:', archive_path)
    files.download(str(archive_path))
    return {'zip': str(archive_path), 'output_dir': str(output), 'comparison': comparison}

_EXTENSION_TEST_RATE_RECOVERY_RESULT = recover_extension_test_after_rate_limit()


In [ ]:
raise RuntimeError('REVIEW-ONLY ARCHIVE: cells are intentionally disabled. Use the approved private Colab workflow; do not Run all.')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
print('مجلد المشروع موجود:', Path('/content/drive/MyDrive/Project9_Phase2').is_dir())

## Submission evidence and known limitations (not yet all finished)
- **M4 real:** 34 epochs completed; saved PyTorch checkpoint, validation AP and corrected threshold sweep. Still needed: compare a baseline, FP/FN gallery, per-class detection error analysis, video-like stream example, and final once-only Test result.
- **M5 real but interrupted:** best val mask mAP50–95 ~0.2888 at epoch 21, completed through epoch 26. Resume if GPU returns; compute full validation mask AP, separate mask IoU/Dice audit, compare baseline and show failures. Train/val mask audit visualized. No final Test value exists.
- **M8 code-only (not yet executed):** model/manifest packaging, ONNX export, small validation parity audit, locally runnable FastAPI, CPU prototype benchmark. Hosted service, full same-deployment benchmarks including memory, p50/p95/p99, throughput at actual belt speed, end-to-end per-stage timing remain to be demonstrated.
- **Open-world two topics:** runnable experiments added for image retrieval and zero-shot CLIP; not yet executed. Each requires a baseline comparison, meaningful *system* demonstration, failure cases, frozen settings and a once-only Test evaluation.
- **Split limitation:** 1,272 frames were excluded conservatively by filename-derived sequence grouping; explain lost training data, class imbalance, and remaining cross-sequence visual similarity risk. The current original Test set was preserved, but do not claim optical duplicate-freedom based solely on filename prefixes.
- **Acceptance criteria / report:** write pre-result thresholds or disclose if criteria were set after Phase 1; use the official 5-member Overleaf template; replace every placeholder; include real repo commit/W&B URLs; ensure every team member checks all claims against code. No original data or secrets in public GitHub.
